# Object RAG manual evaluation

Standalone evaluation of the current category-reranked object retriever. The first two manual suites were relocated from [the application notebook](05.application.rag.object.ipynb); their saved outputs are **historical**, not results of the new full evaluation. The appended CSV suites and final results section identify the independent run.

**828 cases**: 38 existing general-operation + 30 existing arc-welding + 760 new sampled cases. The production index and source/summary checkpoints are not modified. The configured model endpoints receive questions and retrieved manual evidence only when a batch cell is explicitly executed. References and expected facts are reviewer-only data.

## Configuration

Same model, index, limits and category boost as the application.

In [ ]:
# | notest
import math
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from ribosome.rag.object import ObjectRAG

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
load_dotenv(PROJ_ROOT / ".env", override=False)
# INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261022-152558-151254"))
INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683"))
if not INDEX_DIR.is_absolute(): INDEX_DIR = PROJ_ROOT / INDEX_DIR
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
HOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
REQUEST_TIMEOUT = 300
MAX_OUTPUT_TOKENS = 2048
CATEGORY_SCORE_BOOST = float(os.getenv("OBJECT_RAG_CATEGORY_BOOST", "0.15"))
if not math.isfinite(CATEGORY_SCORE_BOOST) or CATEGORY_SCORE_BOOST < 0:
    raise ValueError("OBJECT_RAG_CATEGORY_BOOST must be finite and nonnegative")

def rerank_by_category(hits, category):
    """Boost matching manual labels while retaining semantic similarity."""
    scored = []
    for hit in hits:
        similarity = 1.0 - hit["distance"]
        category_match = hit["manual"]["category"] == category
        boost = CATEGORY_SCORE_BOOST if category_match else 0.0
        scored.append(dict(hit, similarity=similarity, category_match=category_match,
                           category_boost=boost, score=similarity + boost))
    return sorted(scored, key=lambda hit: (-hit["score"], hit["distance"], hit["record"]["id"]))



In [ ]:
# | notest
import sys
if str(PROJ_ROOT / "utils") not in sys.path: sys.path.insert(0, str(PROJ_ROOT / "utils"))
from object_rag_evaluation import EvaluationRun, QueryEmbeddingCache, cache_readonly_snapshots, grouped_metrics
EVALUATION_TOP_K = 8
EVALUATION_CONCURRENCY = 2
USE_PERSISTENT_EVALUATION = True
def select_cases(bank, selected_ids):
    if selected_ids is None: return list(bank)
    known = {case["id"] for case in bank}
    if not selected_ids or set(selected_ids) - known: raise ValueError("Unknown or empty case selection")
    return [case for case in bank if case["id"] in selected_ids]

EVALUATION_CONFIG_SOURCE = '# | notest\nimport math\nimport os\nfrom pathlib import Path\nimport pandas as pd\nfrom dotenv import load_dotenv\nfrom IPython.display import Markdown, display\nfrom ribosome.rag.object import ObjectRAG\n\nPROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()\nwhile not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:\n    PROJ_ROOT = PROJ_ROOT.parent\nload_dotenv(PROJ_ROOT / ".env", override=False)\n# INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261022-152558-151254"))\nINDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683"))\nif not INDEX_DIR.is_absolute(): INDEX_DIR = PROJ_ROOT / INDEX_DIR\nEMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")\nHOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))\nRAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")\nRAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")\nMAX_CONTEXT_CHARS = 24000\nREQUEST_TIMEOUT = 300\nMAX_OUTPUT_TOKENS = 2048\nCATEGORY_SCORE_BOOST = float(os.getenv("OBJECT_RAG_CATEGORY_BOOST", "0.15"))\nif not math.isfinite(CATEGORY_SCORE_BOOST) or CATEGORY_SCORE_BOOST < 0:\n    raise ValueError("OBJECT_RAG_CATEGORY_BOOST must be finite and nonnegative")\n\ndef rerank_by_category(hits, category):\n    """Boost matching manual labels while retaining semantic similarity."""\n    scored = []\n    for hit in hits:\n        similarity = 1.0 - hit["distance"]\n        category_match = hit["manual"]["category"] == category\n        boost = CATEGORY_SCORE_BOOST if category_match else 0.0\n        scored.append(dict(hit, similarity=similarity, category_match=category_match,\n                           category_boost=boost, score=similarity + boost))\n    return sorted(scored, key=lambda hit: (-hit["score"], hit["distance"], hit["record"]["id"]))\n\n'


## Shared evidence and answer helpers

Legacy helpers are retained for source-suite diagnostics and offline regressions. Full evaluation uses persistent stage results.

In [ ]:
# | notest
import html
import re
from copy import deepcopy

def coverage_plain(text):
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', ' ', text)).split())

def coverage_heading_title(heading):
    title = re.sub(r'^\s*\d+(?:\.\d+)*\s+', '', heading.strip('* '))
    return re.sub(r'\s+', '', title).translate(str.maketrans({'—': '-', '–': '-', '－': '-'}))

def coverage_heading_path_matches(expected_path, headings):
    remaining = iter(coverage_heading_title(heading) for heading in headings)
    return all(any(title == coverage_heading_title(expected) for title in remaining)
               for expected in expected_path)

def coverage_case_metrics(case, result, error=''):
    hits = result.get('hits', [])
    # Version-specific cases must find their own corpus and manual, even though
    # production retrieval is category-reranked and may return both revisions.
    matching = [hit for hit in hits if not case['profile'] or (
        hit['manual']['profile'] == case['profile'] and
        Path(hit['record']['source_file']).stem == Path(case['references'][0]['docx']).stem)]
    headings = [heading.split(maxsplit=1)[0] for hit in matching
                for heading in hit['record'].get('section_headings', []) if heading.strip()]
    if case.get('heading_path'):
        section_hit = any(coverage_heading_path_matches(
            case['heading_path'], hit['record'].get('section_headings', [])) for hit in matching)
    else:
        section_hit = case['section'] in headings if case['section'] else None
    source_text = '\n'.join(coverage_plain(hit['record']['content']) for hit in matching)
    anchors_found = all(re.search(r'(?<![A-Za-z0-9_])'+re.escape(term)+r'(?![A-Za-z0-9_])',
                                 source_text, flags=re.IGNORECASE) for term in case['anchors'])
    return dict(case_id=case['id'], category=case['category'], kind=case['kind'],
                expected_revision=case['revision'], expected_section=case['section'],
                hits=len(hits), expected_manual_hit=bool(matching) if case['profile'] else None,
                expected_section_hit=section_hit,
                anchors_in_matching_hits=anchors_found if case['anchors'] else None,
                context_chars=len(result.get('context', '')), omissions=len(result.get('omissions', [])),
                missing_manuals=len(result.get('coverage', {}).get('missing', [])), error=error)

def show_coverage_retrieval(case_id, *, cases=None, results=None, errors=None):
    cases = coverage_cases if cases is None else cases
    results = coverage_retrieval_results if results is None else results
    errors = coverage_retrieval_errors if errors is None else errors
    case = next(case for case in cases if case['id'] == case_id)
    display(Markdown(f"### {case_id}\n\n{case['question']}"))
    if case_id in errors:
        print(errors[case_id])
        return
    result = results[case_id]
    display(pd.DataFrame([dict(score=hit.get('score'), similarity=hit.get('similarity'),
                               category_match=hit.get('category_match'),
                               category_boost=hit.get('category_boost'),
                               distance=hit['distance'], manual=hit['manual'],
                               lines=(hit['record']['attributes'].get('line_start'),
                                      hit['record']['attributes'].get('line_end')),
                               summary=hit['record'].get('summary', ''),
                               source=hit['record']['source_file']) for hit in result['hits']]))
    print(result['context'])
    print('Omitted:', result['omissions'])
    # Reference excerpts remain reviewer data and are never added to the prompt.
    if case['references']:
        display(pd.DataFrame(case['references']))

class CoverageAnswerReplay(ObjectRAG):
    """Run the production answer method on the exact retrieved test evidence."""
    def __init__(self, application, evidence, *, top_k=8):
        self.generator = application.generator
        self.rag_model = application.rag_model
        self.max_output_tokens = application.max_output_tokens
        self.evidence = evidence
        self.top_k = top_k
        self.last_result = None

    async def retrieve(self, question, category, *, top_k=8):
        if (question, category, top_k) != (
            self.evidence['question'], self.evidence['category'], self.top_k
        ):
            raise ValueError('Coverage replay requires the same question, category and top-k')
        self.last_result = deepcopy(self.evidence)
        return self.last_result

async def run_coverage_retrievals(application, cases, *, top_k, results, errors):
    """Retrieve into this suite's dictionaries; retain completed cases on interruption."""
    for number, case in enumerate(cases, 1):
        print(f"[{number}/{len(cases)}] Retrieve {case['id']}", flush=True)
        try:
            results[case['id']] = await application.retrieve(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case['id']] = f'{type(error).__name__}: {error}'
    return pd.DataFrame([
        coverage_case_metrics(case, results.get(case['id'], {}),
                              errors.get(case['id'], ''))
        for case in cases])

async def run_coverage_answers(application, cases, *, top_k, retrieval_results,
                               retrieval_errors, results, errors):
    """Answer from this suite's frozen evidence; never make another retrieval request."""
    for number, case in enumerate(cases, 1):
        case_id = case['id']
        display(Markdown(f"## [{number}/{len(cases)}] {case_id}\n\n{case['question']}"))
        if case_id not in retrieval_results:
            errors[case_id] = retrieval_errors.get(case_id, 'Retrieve this case first')
            display(Markdown('Retrieval failed: ' + errors[case_id]))
            continue
        replay = CoverageAnswerReplay(application, retrieval_results[case_id], top_k=top_k)
        try:
            results[case_id] = await replay.answer(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case_id] = f'{type(error).__name__}: {error}'
            results[case_id] = replay.last_result or {}
        finally:
            if case_id not in results and replay.last_result is not None:
                results[case_id] = replay.last_result
                errors[case_id] = 'Generation interrupted before completion'
            case_result = results.get(case_id, {})
            if 'retrieved_summary' in case_result:
                display(Markdown('### Retrieved content summary\n\n' + case_result['retrieved_summary']))
        if case_id in errors:
            display(Markdown('Generation failed: ' + errors[case_id]))
        else:
            display(Markdown('### Coding or diagnostic answer\n\n' + case_result['answer']))
    return pd.DataFrame([
        dict(**coverage_case_metrics(case, results.get(case['id'], {}),
                                     errors.get(case['id'], '')),
             example_status=results.get(case['id'], {}).get('example_status', 'not generated'))
        for case in cases])


## Open existing index and services

No embedding or generation occurs until a batch cell is executed.

In [ ]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / 'catalog.sqlite3').is_file() or not (INDEX_DIR / 'chroma').is_dir():
    raise FileNotFoundError(INDEX_DIR)
headers = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
embedding_client = AsyncClient(host=HOSTS[0], headers=headers, timeout=REQUEST_TIMEOUT)
generation_client = AsyncClient(host=RAG_HOST, headers=headers, timeout=REQUEST_TIMEOUT)
collection = chromadb.PersistentClient(path=str(INDEX_DIR/'chroma')).get_collection('object_summaries', embedding_function=None)
app = ObjectRAG(
    INDEX_DIR, collection, embedding_client, generation_client,
    mapping_dir=PROJ_ROOT / 'assets/coding_categories',
    embed_model=EMBED_MODEL,
    embedding_config=dict(provider='ollama-pool', hosts=HOSTS),
    rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
    max_output_tokens=MAX_OUTPUT_TOKENS, rerank=rerank_by_category,
)
async def retrieve(question, category, *, top_k=8):
    return await app.retrieve(question, category, top_k=top_k)
async def answer(question, category, *, top_k=8):
    return await app.answer(question, category, top_k=top_k)

# Settings and active-generation identities isolate compatible resumable runs.
evaluation = EvaluationRun(app, [], PROJ_ROOT / "db/object_rag_tests/results",
    top_k=EVALUATION_TOP_K, settings=dict(category_boost=CATEGORY_SCORE_BOOST,
    rag_host=RAG_HOST, rerank_source=EVALUATION_CONFIG_SOURCE))

query_embeddings = QueryEmbeddingCache(embedding_client, EMBED_MODEL, app.embedding_config, app.dimension, PROJ_ROOT / "db/object_rag_tests/query_embeddings")
app.embedder = query_embeddings

cache_readonly_snapshots(app)


## General-operation manual version tests

This separate `general_operation` suite tests the two versions of
《新松机器人通用操作手册》:
[B-5](../assets/code-docs.4.0/06%20产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx)
and [C-1](../assets/code-docs.5.0/06%20产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx).

There are **38 prompts**: two different instruction questions for each of B-5's
eight subsections and C-1's six subsections in 9.1 (**28 questions**), plus
**10 alarm-code questions**, sampled reproducibly as five per manual from the
报警 tables, excluding 警告 and 提示信息. Original DOCX row locators, hashes,
reference excerpts and sampling seeds are preserved below. `profile` identifies
corpus 4.0 or 5.0, which differs from the manuals' V5.0 label.

Run this notebook's configuration, imports, shared-helper and service cells
before this section; other test batches are optional.
Set `GENERAL_OPERATION_CASE_IDS` to a list of manual case IDs or leave it `None`
for all 38. This suite has its own top-k, selected cases, inspection ID and result
dictionaries, so it does not replace the category suite's results. No model request
occurs until its retrieval or answer cell is run. Run cleanup after all chosen suites.


In [9]:
# | notest
# Source locators are zero-based word/document.xml body blocks and table rows.
ALARM_SAMPLING = {'4.0': {'seed': 20261031,
         'population': 1229,
         'count': 5,
         'source_docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
         'sha256': '4e5f7dae024d703e27dd56a75c97606d0e9945e7c2206b148d492685469031db',
         'body_block': 3859,
         'rows': [150, 496, 627, 770, 1050]},
 '5.0': {'seed': 20261032,
         'population': 600,
         'count': 5,
         'source_docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
         'sha256': 'bfc19c6b11737c0a49c67565a4477d0d31e731fb26167a08b8e4800346aad286',
         'body_block': 3603,
         'rows': [85, 118, 154, 216, 580]}}

GENERAL_OPERATION_CASES = [{'id': 'general_40_9_1_1_single_output',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.1',
  'section_title': 'I/O类指令',
  'anchors': ['OUT'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3834,
                  'rows': [0, 1],
                  'excerpt': 'OUT OT# | 功能 | IO输出，单个IO\n | 格式 | OUT #<参数1>=<参数2>'}]},
 {'id': 'general_40_9_1_1_wait_input',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.1',
  'section_title': 'I/O类指令',
  'anchors': ['WAIT'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用WAIT等待输入信号满足条件，并设置等待时间？请说明输入号、状态值和超时参数。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3834,
                  'rows': [90, 91],
                  'excerpt': 'WAIT | 功能 | 等待条件满足。\n | 格式 | WAIT <参数1>=<参数2>  T=<参数3>'}]},
 {'id': 'general_40_9_1_2_goto_label',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.2',
  'section_title': '控制类1',
  'anchors': ['GOTO'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样用GOTO和标签L实现无条件跳转，说明被跳过的指令及跳转后的执行顺序？',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [0, 1],
                  'excerpt': 'GOTO | 功能 | 无条件跳转到标签处执行\n | 格式 | GOTO  L<参数项>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [4, 5],
                  'excerpt': 'LABEL | 功能 | 标签，标明指令条件满足时，要跳转到的指令位置\n | 格式 | L<参数项>'}]},
 {'id': 'general_40_9_1_2_conditional_integer',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.2',
  'section_title': '控制类1',
  'anchors': ['IF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样依据整型变量比较结果执行IF I条件转移？请解释比较参数和目标标签，并给出分支例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3836,
                  'rows': [23, 24],
                  'excerpt': 'IF I | 功能 | 如果整型变量满足比较条件，则作业跳转到标签处执行，否则继续执行IF指令的下一行指令。\n'
                             ' | 格式 | IF  <参数1><比较符><参数2>  L<参数3>'}]},
 {'id': 'general_40_9_1_3_subjob',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.3',
  'section_title': '控制类2',
  'anchors': ['CALL'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用CALL调用子作业，并在子作业中用RET返回母作业？请区分无条件和输入IO条件调用。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [0, 1],
                  'excerpt': 'CALL | 功能 | 调用子作业\n | 格式 | CALL <调用条件><子作业名>'}]},
 {'id': 'general_40_9_1_3_position_shift',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.3',
  'section_title': '控制类2',
  'anchors': ['SHIFTON', 'SHIFTOFF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用SHIFTON按位置变量在指定坐标系内偏移运动，并用SHIFTOFF取消偏移？请解释F#的坐标系类型。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [4, 5],
                  'excerpt': 'SHIFT P | 功能 | 以设定的坐标系按位置变量中的值进行偏移\n | 格式 | SHIFTON  PR<参数1>  F#<参数2>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3838,
                  'rows': [9, 10],
                  'excerpt': 'SHIFT OFF | 功能 | 停止所有偏移。\n | 格式 | SHIFTOFF'}]},
 {'id': 'general_40_9_1_4_linear_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.4',
  'section_title': '运动类指令',
  'anchors': ['MOVL'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVL使TCP以指定线速度运动到P1？请解释VL、ACC、CNT的含义和单位，并说明轨迹连续参数。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3840,
                  'rows': [7, 8],
                  'excerpt': 'MOVL P/PR | 功能 | 两点之间以直线插补方式进行运动。\n'
                             ' | 格式 | MOVL  P<参数1>  VL=<参数2> ACC=<参数3> CNT=<参数4>'}]},
 {'id': 'general_40_9_1_4_circular_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.4',
  'section_title': '运动类指令',
  'anchors': ['MOVC'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVC组织三点圆弧运动？请说明点位、VC、ACC、CNT参数以及圆弧示教要求。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3840,
                  'rows': [14, 15],
                  'excerpt': 'MOVC P/PR | 功能 | 三点之间以圆弧插补方式进行运动。\n'
                             ' | 格式 | MOVC P<参数1> VC=<参数2> ACC=<参数3> CNT=<参数4>'}]},
 {'id': 'general_40_9_1_5_user_frame',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.5',
  'section_title': '运算类指令',
  'anchors': ['SET UF'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何用SET UF设置用户坐标系号？请说明文件号范围及对自动运行和示教轴操作的影响。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3842,
                  'rows': [4, 5],
                  'excerpt': 'SET UF | 功能 | '
                             '设置用户坐标系号。该指令执行后，系统的当前用户坐标系号则被改变，用户坐标系号的改变不仅对自动执行的作业有影响，示教模式下的轴操作也使用的是新设定的用户坐标系。\n'
                             ' | 格式 | SET UF #<参数1>'}]},
 {'id': 'general_40_9_1_5_interrupt_position',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.5',
  'section_title': '运算类指令',
  'anchors': ['IRQP'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何在中断程序中获取中断发生瞬间的位置并存入PR10？请比较GET '
              'IRQP的规划值、实际值选择和中断等级要求。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3842,
                  'rows': [59, 60],
                  'excerpt': 'GET IRQP | 功能 | 获取中断发生瞬间的机器人当前位置的规划值或实际值赋值给位置\n'
                             '变量PR（需要在程序中做中断，然后在中断程序中使用此指令）\n'
                             ' | 格式 | IRQP<参数1> LEV=<参数2> P<参数3>'}]},
 {'id': 'general_40_9_1_6_integer_assignment',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.6',
  'section_title': '计算类指令',
  'anchors': ['SET'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.6节“计算类指令”回答：如何将整型变量I2的值赋给I1，再使用该值？请说明整型SET的参数类型和赋值方向。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3844,
                  'rows': [0, 1],
                  'excerpt': '整型 SET | 功能 | 赋值运算，将参数2的值赋给参数1\n | 格式 | SET  <参数1><参数2>'}]},
 {'id': 'general_40_9_1_6_byte_logic',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.6',
  'section_title': '计算类指令',
  'anchors': ['AND', 'XOR'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.6节“计算类指令”回答：字节变量的AND与XOR运算有什么区别？请说明结果保存到哪个参数，并分别给出有初始值的例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3850,
                  'rows': [5, 6],
                  'excerpt': '字节型\nAND | 功能 | 按位与运算，参数1与参数2按位求与，结果存在参数1中。\n | 格式 | AND  B<参数1>  B<参数2>'},
                 {'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3850,
                  'rows': [20, 21],
                  'excerpt': '字节型\nXOR | 功能 | 按位异或运算，参数1与参数2按位求异或，结果存在参数1中。\n | 格式 | XOR  B<参数1>  B<参数2>'}]},
 {'id': 'general_40_9_1_7_tcp_connect',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.7',
  'section_title': '通讯指令',
  'anchors': ['TCPCREATE'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.7节“通讯指令”回答：如何用TCPCREATE创建TCP连接？请解释NAME、IP、PORT、TIMEOUT、RET参数及如何判断连接结果。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3853,
                  'rows': [0, 1],
                  'excerpt': 'TCPCREATE  | 功能 | 创建一个 TCP 连接\n'
                             ' | 格式 | TCPCREATE NAME=<参数1> IP=<参数2> PORT=<参数3> TIMEOUT=<参数4> RET=<参数5>'}]},
 {'id': 'general_40_9_1_7_substring',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.7',
  'section_title': '通讯指令',
  'anchors': ['STRSUB'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.7节“通讯指令”回答：如何用STRSUB从字符串提取子串？请说明源字符串、起始位置、长度及目标字符串参数，并给出初始化和结果使用的例子。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3853,
                  'rows': [61, 62],
                  'excerpt': 'STRSUB | 功能 | 截取字符串的子串\n'
                             ' | 格式 | STRSUB STR=LS#<参数1> BEGIN=<参数2> END=<参数3> RET=<参数4>'}]},
 {'id': 'general_40_9_1_8_home_job',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.8',
  'section_title': '指令集',
  'anchors': ['A01_CMN HomePos'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.8节“指令集”回答：如何使用A01_CMN '
              'HomePos调用原位作业？请说明指令功能；若证据不足以组成完整返回原位程序，请明确缺少哪些条件。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3855,
                  'rows': [0, 1],
                  'excerpt': 'A01_CMN\nHomePos | 功能 | 调用指令集作业\n | 格式 | A01_CMN HomePos（原位作业）'}]},
 {'id': 'general_40_9_1_8_zone_request',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '9.1.8',
  'section_title': '指令集',
  'anchors': ['A01_CMN UserRequest'],
  'question': '请依据《新松机器人通用操作手册》(B-5)第9.1.8节“指令集”回答：如何使用A01_CMN '
              'UserRequest申请和释放区域？请比较AllocateZone、ReleaseZone与Request、Finish，并说明UserNo参数和关联整型变量。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3855,
                  'rows': [2, 3],
                  'excerpt': 'A01_CMN\n'
                             'UserRequest | 功能 | 调用不同类型的指令集作业\n'
                             ' | 格式 | A01_CMN UserRequest<参数1> UserNo<参数2>'}]},
 {'id': 'general_50_9_1_1_joint_and_linear',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.1',
  'section_title': '运动类',
  'anchors': ['MOVJ', 'MOVL'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.1节“运动类”回答：MOVJ与MOVL分别采用什么插补方式，速度参数及单位有何不同？请用P[]格式说明两种运动的参数并组织一个例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3387,
                  'rows': [0, 1],
                  'excerpt': 'MOVJ | 功能 | 关节运动指令\n'
                             ' | 格式 | MOVJ P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3389,
                  'rows': [0, 1],
                  'excerpt': 'MOVL | 功能 | 直线运动指令\n'
                             ' | 格式 | MOVL P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'}]},
 {'id': 'general_50_9_1_1_circular_motion',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.1',
  'section_title': '运动类',
  'anchors': ['MOVC'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.1节“运动类”回答：如何用MOVC组织圆弧运动？请说明示教点、速度、加速度和轨迹连续参数，保持使用本版本的点位语法。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3391,
                  'rows': [0, 1],
                  'excerpt': 'MOVC | 功能 | 圆弧运动指令\n'
                             ' | 格式 | MOVL P[值]\\PR[值] V = <参数2> ACC = <参数3> CNT = <参数4> \\ FINE'}]},
 {'id': 'general_50_9_1_2_typed_assignment',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.2',
  'section_title': '基础指令',
  'anchors': ['SET'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.2节“基础指令”回答：如何用SET将I[2]赋给I[1]？请解释本版本支持的变量类型，并给出初始化、赋值和后续使用的例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3394,
                  'rows': [0, 1],
                  'excerpt': 'SET | 功能 | 赋值指令\n | 格式 | SET  <参数1>  <参数2>'}]},
 {'id': 'general_50_9_1_2_substring',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.2',
  'section_title': '基础指令',
  'anchors': ['STRSUB'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.2节“基础指令”回答：如何使用STRSUB从S[]字符串中提取一段子串？请说明参数含义、索引约定及结果保存位置。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3421,
                  'rows': [0, 1],
                  'excerpt': 'STRSUB | 功能 | 获取部分字符串，从参数1中获取参数2与参数3长度的字符串，结果放参数4中。\n'
                             ' | 格式 | STRSUB <参数1> <参数2> <参数3> <参数4>'}]},
 {'id': 'general_50_9_1_3_compound_expression',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.3',
  'section_title': '运算指令',
  'anchors': ['COP'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.3节“运算指令”回答：如何使用COP组合加减乘除运算，把计算结果写入整型或实型变量？请说明可用运算符及对P变量运算的限制。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3450,
                  'rows': [0, 1],
                  'excerpt': 'COP | 功能 | 运算类符合指令，支持各种运算的组合，组合没有长度限制。\n'
                             ' | 格式 | <参数1>  =  <参数2>  [运算符] <参数3> …'}]},
 {'id': 'general_50_9_1_3_square_root',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.3',
  'section_title': '运算指令',
  'anchors': ['SQRT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.3节“运算指令”回答：如何使用SQRT计算平方根并保存结果？请说明输入、输出参数类型并给出初始化及后续使用的例子。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3464,
                  'rows': [0, 1],
                  'excerpt': 'SQRT | 功能 | 开平方指令，对参数2进行开方运算，结果存入参数1中。\n | 格式 | R[值] = SQRT <参数2>'}]},
 {'id': 'general_50_9_1_4_digital_io',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.4',
  'section_title': 'I/O指令',
  'anchors': ['IN', 'OUT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.4节“I/O指令”回答：如何用IN读取数字输入、用OUT设置数字输出？请说明DI[]、DO[]及组IO参数的含义，保持使用本版本语法。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3488,
                  'rows': [0, 1],
                  'excerpt': 'IN | 功能 | 将I/O赋给变量I[值]\n | 格式 | IN I[值] = <参数2> '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3491,
                  'rows': [0, 1],
                  'excerpt': 'OUT | 功能 | 对I/O和组I/O进行输出。\n | 格式 | OUT <参数1> = <参数2> '}]},
 {'id': 'general_50_9_1_4_analog_io',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.4',
  'section_title': 'I/O指令',
  'anchors': ['ANIN', 'ANOUT'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.4节“I/O指令”回答：如何用ANIN读取模拟量并用ANOUT输出模拟量？请说明通道、数值和变量类型参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3497,
                  'rows': [0, 1],
                  'excerpt': 'ANIN | 功能 | 模拟量输出信号，是连续的信号，一般表示电压、电流等数据。将模拟量输入I/O赋值给I变量。\n'
                             ' | 格式 | ANIN <参数1> = <参数2> '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3500,
                  'rows': [0, 1],
                  'excerpt': 'ANOUT | 功能 | 模拟量输出信号，是连续的信号，一般表示电压、电流等数据。将变量值转换成模拟量输出到指定的I/0中。\n'
                             ' | 格式 | ANOUT <参数1> = <参数2> '}]},
 {'id': 'general_50_9_1_5_conditional_branch',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.5',
  'section_title': '控制类',
  'anchors': ['IF', 'ELSEIF', 'ELSE', 'ENDIF'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.5节“控制类”回答：如何用IF、ELSEIF、ELSE、ENDIF组织多分支判断？请说明比较运算符及AND、OR的作用，并给出变量条件示例。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3502,
                  'rows': [0, 1],
                  'excerpt': 'IF OP | 功能 | 条件判断指令，对条件进行判断，并根据结果进行后续指令的执行。\n'
                             ' | 格式 | IF <参数1> 操作符 <参数2> <附加指令>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3507,
                  'rows': [0, 1],
                  'excerpt': 'ELSEIF OP | 功能 | 满足条件执行区间内指令。\n'
                             ' | 格式 | ELSEIF <参数1> 操作符 <参数2> AND <参数1> 操作符 <参数2> <附加指令>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3510,
                  'rows': [0, 1],
                  'excerpt': 'ELSE | 功能 | 与IF指令一起使用，IF条件未满足，执行区间内指令。\n | 格式 | ELSE'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3512,
                  'rows': [0, 1],
                  'excerpt': 'ENDIF | 功能 | 与IF指令一起使用，条件跳转指令结束。\n | 格式 | ENDIF'}]},
 {'id': 'general_50_9_1_5_for_loop',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.5',
  'section_title': '控制类',
  'anchors': ['FOR', 'ENDFOR', 'BREAK', 'CONTINUE'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.5节“控制类”回答：如何用FOR和ENDFOR组织带STEP的计数循环？请说明循环变量类型、条件和步长，并区分BREAK与CONTINUE。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3556,
                  'rows': [0, 1],
                  'excerpt': 'FOR...ENDFOR | 功能 | FOR循环指令。\n | 格式 | FOR<参数1>=<参数2><参数3><参数4><参数5><参数6>'},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3561,
                  'rows': [0, 1],
                  'excerpt': 'BREAK | 功能 | BREAK语句。\n | 格式 | BREAK '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3563,
                  'rows': [0, 1],
                  'excerpt': 'CONTINUE | 功能 | CONTINUE语句。\n | 格式 | CONTINUE '}]},
 {'id': 'general_50_9_1_6_interrupt_enable',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.6',
  'section_title': '操作类',
  'anchors': ['IRQON LEV', 'IRQOFF'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.6节“操作类”回答：如何使用IRQON LEV启用输入信号触发的中断并用IRQOFF关闭？请说明等级、比较条件和作业名参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3568,
                  'rows': [0, 1],
                  'excerpt': 'IRQON LEV | 功能 | 中断开启指令。\n | 格式 | IRQON LEV = <参数1> <参数2> [运算符] <参数3> [作业名]  '},
                 {'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3571,
                  'rows': [0, 1],
                  'excerpt': 'IRQOFF | 功能 | 中断结束指令。\n | 格式 | IRQOFF LEV = <参数1> '}]},
 {'id': 'general_50_9_1_6_tcp_receive',
  'category': 'general_operation',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '9.1.6',
  'section_title': '操作类',
  'anchors': ['TCPRECV'],
  'question': '请依据《新松机器人通用操作手册》(C-1)第9.1.6节“操作类”回答：如何使用TCPRECV接收服务端字符串？请说明连接名称、目标S[]变量、最大长度、TIMEOUT及接收结果参数。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3586,
                  'rows': [0, 1],
                  'excerpt': 'TCPRECV | 功能 | TCP接收指令并，接受服务端字符串数据，并存储至参数2中。\n'
                             ' | 格式 | TCPRECV <参数1> <参数2> <参数3> TIMEOUT = <参数4> <参数5>'}]},
 {'id': 'general_40_alarm_1253',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1253',
  'anchors': ['1253'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1253的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [150],
                  'excerpt': '1253 | PROT0连续接收到无效帧错误:SV   | 山洋驱动器报警，[参数]为报警驱动器对应轴号 | '
                             '[清除报警]可复位，对应驱动器报警代码10，反复出现请联系新松售后人员'}],
  'reference_alarm': {'message': 'PROT0连续接收到无效帧错误:SV  ',
                      'cause': '山洋驱动器报警，[参数]为报警驱动器对应轴号',
                      'remedy': '[清除报警]可复位，对应驱动器报警代码10，反复出现请联系新松售后人员'}},
 {'id': 'general_40_alarm_1613',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1613',
  'anchors': ['1613'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1613的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [496],
                  'excerpt': '1613 | 限位开关异常:SV  | 电机正向运行时，检测到负限位开关触发\n'
                             '电机反向运行时，检测到正限位开关触发 | 检测接线和参数设置，确保限位开关的极性定义与接线正确'}],
  'reference_alarm': {'message': '限位开关异常:SV ',
                      'cause': '电机正向运行时，检测到负限位开关触发\n电机反向运行时，检测到正限位开关触发',
                      'remedy': '检测接线和参数设置，确保限位开关的极性定义与接线正确'}},
 {'id': 'general_40_alarm_1744',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1744',
  'anchors': ['1744'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码1744的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [627],
                  'excerpt': '1744 | 运行统计信息初始化失败:SV | 初始化失败 | 请进行复位，或重启系统，反复出现请联系新松售后人员'}],
  'reference_alarm': {'message': '运行统计信息初始化失败:SV', 'cause': '初始化失败', 'remedy': '请进行复位，或重启系统，反复出现请联系新松售后人员'}},
 {'id': 'general_40_alarm_8150',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '8150',
  'anchors': ['8150'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码8150的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [770],
                  'excerpt': '8150 | 绝对位置遗失:SV  | 条件：绝对型编码器因为电池低电压或供电中断而遗失内部所记录的圈数。原因：1. 电池电压过低。2. '
                             '在驱动器控制电源Off 的状况下更换电池。3. 启动绝对型功能后，未安装电池。4. 电池供电线路接触不良或断线。 | 1. '
                             '检查电池电压是否低于2.9V；更换电池后，重新建立绝对型原点坐标。2. 请勿在驱动器控制电源Off 的状况下更换或移除电池。3. 建议检查事项如下：(a) '
                             '安装电池。(b) 检查电池外接盒跟驱动器的电池电源接线。(c) 检查编码器配线。4. 确认接线是否可让电池电力正常供给编码器，重新建立绝对型原点坐标。'}],
  'reference_alarm': {'message': '绝对位置遗失:SV ',
                      'cause': '条件：绝对型编码器因为电池低电压或供电中断而遗失内部所记录的圈数。原因：1. 电池电压过低。2. 在驱动器控制电源Off 的状况下更换电池。3. '
                               '启动绝对型功能后，未安装电池。4. 电池供电线路接触不良或断线。',
                      'remedy': '1. 检查电池电压是否低于2.9V；更换电池后，重新建立绝对型原点坐标。2. 请勿在驱动器控制电源Off 的状况下更换或移除电池。3. '
                                '建议检查事项如下：(a) 安装电池。(b) 检查电池外接盒跟驱动器的电池电源接线。(c) 检查编码器配线。4. '
                                '确认接线是否可让电池电力正常供给编码器，重新建立绝对型原点坐标。'}},
 {'id': 'general_40_alarm_8438',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '8438',
  'anchors': ['8438'],
  'question': '请依据《新松机器人通用操作手册》(B-5)的“报警信息→报警”表说明报警码8438的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx',
                  'body_block': 3859,
                  'rows': [1050],
                  'excerpt': '8438 | 输出对地短路:SV | 1. 驱动器动力线缆(U V W)对地发生短路2. 电机对地短路3. 驱动器故障 | '
                             '1、拔掉电机线缆，分别测量驱动器动力线缆U V W是否对地(PE) '
                             '短路。重新接线或更换驱动器动力线缆。2、确保驱动器动力线缆、电机线缆紧固连接后，分别测量驱动器U V W 端与接地线(PE) '
                             '之间的绝缘电阻是否为兆欧姆(MΩ) 级数值。更换电机。3、将驱动器动力线缆从伺服驱动器上卸下，多次接通电源后仍报故障。更换伺服驱动器。'}],
  'reference_alarm': {'message': '输出对地短路:SV',
                      'cause': '1. 驱动器动力线缆(U V W)对地发生短路2. 电机对地短路3. 驱动器故障',
                      'remedy': '1、拔掉电机线缆，分别测量驱动器动力线缆U V W是否对地(PE) '
                                '短路。重新接线或更换驱动器动力线缆。2、确保驱动器动力线缆、电机线缆紧固连接后，分别测量驱动器U V W 端与接地线(PE) '
                                '之间的绝缘电阻是否为兆欧姆(MΩ) 级数值。更换电机。3、将驱动器动力线缆从伺服驱动器上卸下，多次接通电源后仍报故障。更换伺服驱动器。'}},
 {'id': 'general_50_alarm_1372',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1372',
  'anchors': ['1372'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1372的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [85],
                  'excerpt': '1372 | CPU过载:SV   | 1.控制指令超过CPU负载能力2.驱动器损坏。 | 处理方法：降低控制指令操作频率；更换或维修驱动器'}],
  'reference_alarm': {'message': 'CPU过载:SV  ',
                      'cause': '1.控制指令超过CPU负载能力2.驱动器损坏。',
                      'remedy': '处理方法：降低控制指令操作频率；更换或维修驱动器'}},
 {'id': 'general_50_alarm_1405',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1405',
  'anchors': ['1405'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1405的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [118],
                  'excerpt': '1405 | 位置硬超限:SV   | 驱动器外接的位置限位开关触发； | 处理方法：检查位置指令规划范围'}],
  'reference_alarm': {'message': '位置硬超限:SV  ', 'cause': '驱动器外接的位置限位开关触发；', 'remedy': '处理方法：检查位置指令规划范围'}},
 {'id': 'general_50_alarm_1507',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1507',
  'anchors': ['1507'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1507的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [154],
                  'excerpt': '1507 | 轴内部数据记录初始化:SV | 对应轴号初始化 | 处理方法：联系技术人员。'}],
  'reference_alarm': {'message': '轴内部数据记录初始化:SV', 'cause': '对应轴号初始化', 'remedy': '处理方法：联系技术人员。'}},
 {'id': 'general_50_alarm_1569',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '1569',
  'anchors': ['1569'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码1569的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [216],
                  'excerpt': '1569 | 伺服参数内部转换初始化:SV  | 对应轴号初始化 | 处理方法：联系技术人员。'}],
  'reference_alarm': {'message': '伺服参数内部转换初始化:SV ', 'cause': '对应轴号初始化', 'remedy': '处理方法：联系技术人员。'}},
 {'id': 'general_50_alarm_6035',
  'category': 'general_operation',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'section': '10.1',
  'section_title': '报警',
  'alarm_code': '6035',
  'anchors': ['6035'],
  'question': '请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码6035的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx',
                  'body_block': 3603,
                  'rows': [580],
                  'excerpt': '6035 | 运动队列运动属运动类别错误 |  | '}],
  'reference_alarm': {'message': '运动队列运动属运动类别错误', 'cause': '', 'remedy': ''}}]


In [10]:
# | notest
GENERAL_OPERATION_TOP_K = 8
GENERAL_OPERATION_CASE_IDS = None  # Example: ['general_40_9_1_6_integer_assignment', 'general_50_alarm_6035']
known_general_operation_ids = {case['id'] for case in GENERAL_OPERATION_CASES}
assert {case['category'] for case in GENERAL_OPERATION_CASES} == {'general_operation'}
if GENERAL_OPERATION_CASE_IDS is not None:
    if not GENERAL_OPERATION_CASE_IDS or set(GENERAL_OPERATION_CASE_IDS) - known_general_operation_ids:
        raise ValueError('Select at least one known general-operation case ID')
general_operation_cases = [case for case in GENERAL_OPERATION_CASES
                          if GENERAL_OPERATION_CASE_IDS is None or case['id'] in GENERAL_OPERATION_CASE_IDS]
GENERAL_OPERATION_INSPECT_CASE_ID = general_operation_cases[0]['id']
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(general_operation_cases)[
        ['id', 'kind', 'profile', 'revision', 'section', 'section_title', 'question']])
display(app.snapshot('general_operation')[2])


,id,kind,profile,revision,section,section_title,question
0,general_40_9_1_1_single_output,instruction,4.0,B-5,9.1.1,I/O类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？
1,general_40_9_1_1_wait_input,instruction,4.0,B-5,9.1.1,I/O类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用WAIT等待输入信号满足条件，并设置等待时间？请说明输入号、状态值和超时参数。
2,general_40_9_1_2_goto_label,instruction,4.0,B-5,9.1.2,控制类1,请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样用GOTO和标签L实现无条件跳转，说明被跳过的指令及跳转后的执行顺序？
3,general_40_9_1_2_conditional_integer,instruction,4.0,B-5,9.1.2,控制类1,请依据《新松机器人通用操作手册》(B-5)第9.1.2节“控制类1”回答：怎样依据整型变量比较结果执行IF I条件转移？请解释比较参数和目标标签，并给出分支例子。
4,general_40_9_1_3_subjob,instruction,4.0,B-5,9.1.3,控制类2,请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用CALL调用子作业，并在子作业中用RET返回母作业？请区分无条件和输入IO条件调用。
5,general_40_9_1_3_position_shift,instruction,4.0,B-5,9.1.3,控制类2,请依据《新松机器人通用操作手册》(B-5)第9.1.3节“控制类2”回答：怎样用SHIFTON按位置变量在指定坐标系内偏移运动，并用SHIFTOFF取消偏移？请解释F#的坐标系类型。
6,general_40_9_1_4_linear_motion,instruction,4.0,B-5,9.1.4,运动类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVL使TCP以指定线速度运动到P1？请解释VL、ACC、CNT的含义和单位，并说明轨迹连续参数。
7,general_40_9_1_4_circular_motion,instruction,4.0,B-5,9.1.4,运动类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.4节“运动类指令”回答：怎样用MOVC组织三点圆弧运动？请说明点位、VC、ACC、CNT参数以及圆弧示教要求。
8,general_40_9_1_5_user_frame,instruction,4.0,B-5,9.1.5,运算类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何用SET UF设置用户坐标系号？请说明文件号范围及对自动运行和示教轴操作的影响。
9,general_40_9_1_5_interrupt_position,instruction,4.0,B-5,9.1.5,运算类指令,请依据《新松机器人通用操作手册》(B-5)第9.1.5节“运算类指令”回答：如何在中断程序中获取中断发生瞬间的位置并存入PR10？请比较GET IRQP的规划值、实际值选择和中断等级要求。


{'indexed': [{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5)',
   'profile': '4.0',
   'revision': 'V5.0 / B-5',
   'category': 'general_operation'},
  {'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1)',
   'profile': '5.0',
   'revision': 'V5.0 / C-1',
   'category': 'general_operation'}],
 'missing': []}

### Retrieve and inspect manual evidence

Retrieve the selected version-specific questions and report expected manual,
subsection and command/alarm anchors separately from the returned hit count.
Production retrieval still searches both mapped revisions; the diagnostics check
whether the intended revision was found. These checks are not answer correctness
scores. The inspection call takes this suite's cases and result dictionaries
explicitly; change `GENERAL_OPERATION_INSPECT_CASE_ID` to inspect another case.


In [11]:
# | notest
if USE_PERSISTENT_EVALUATION:
    general_operation_evaluation = evaluation.for_top_k(GENERAL_OPERATION_TOP_K)
    general_operation_retrieval_metrics = await general_operation_evaluation.retrieve_cases(general_operation_cases)
    general_operation_retrieval_results = {case["id"]: general_operation_evaluation.load(case).get("retrieval", {}) for case in general_operation_cases if general_operation_evaluation.load(case).get("retrieval_status") == "complete"}
    general_operation_retrieval_errors = {case["id"]: general_operation_evaluation.load(case)["retrieval_error"] for case in general_operation_cases if general_operation_evaluation.load(case).get("retrieval_error")}
    display(general_operation_retrieval_metrics)
else:
    # | notest
    general_operation_retrieval_results, general_operation_retrieval_errors = {}, {}
    general_operation_retrieval_metrics = await run_coverage_retrievals(
        app, general_operation_cases, top_k=GENERAL_OPERATION_TOP_K,
        results=general_operation_retrieval_results, errors=general_operation_retrieval_errors)
    display(general_operation_retrieval_metrics)
    show_coverage_retrieval(
        GENERAL_OPERATION_INSPECT_CASE_ID, cases=general_operation_cases,
        results=general_operation_retrieval_results, errors=general_operation_retrieval_errors)


[1/38] Retrieve general_40_9_1_1_single_output
[2/38] Retrieve general_40_9_1_1_wait_input
[3/38] Retrieve general_40_9_1_2_goto_label
[4/38] Retrieve general_40_9_1_2_conditional_integer
[5/38] Retrieve general_40_9_1_3_subjob
[6/38] Retrieve general_40_9_1_3_position_shift
[7/38] Retrieve general_40_9_1_4_linear_motion
[8/38] Retrieve general_40_9_1_4_circular_motion
[9/38] Retrieve general_40_9_1_5_user_frame
[10/38] Retrieve general_40_9_1_5_interrupt_position
[11/38] Retrieve general_40_9_1_6_integer_assignment
[12/38] Retrieve general_40_9_1_6_byte_logic
[13/38] Retrieve general_40_9_1_7_tcp_connect
[14/38] Retrieve general_40_9_1_7_substring
[15/38] Retrieve general_40_9_1_8_home_job
[16/38] Retrieve general_40_9_1_8_zone_request
[17/38] Retrieve general_50_9_1_1_joint_and_linear
[18/38] Retrieve general_50_9_1_1_circular_motion
[19/38] Retrieve general_50_9_1_2_typed_assignment
[20/38] Retrieve general_50_9_1_2_substring
[21/38] Retrieve general_50_9_1_3_compound_expression
[22

,case_id,category,kind,expected_revision,expected_section,hits,expected_manual_hit,expected_section_hit,anchors_in_matching_hits,context_chars,omissions,missing_manuals,error
0,general_40_9_1_1_single_output,general_operation,instruction,B-5,9.1.1,8,True,True,True,22639,1,0,
1,general_40_9_1_1_wait_input,general_operation,instruction,B-5,9.1.1,8,True,True,True,23637,3,0,
2,general_40_9_1_2_goto_label,general_operation,instruction,B-5,9.1.2,8,True,True,True,23683,1,0,
3,general_40_9_1_2_conditional_integer,general_operation,instruction,B-5,9.1.2,8,True,True,True,23459,1,0,
4,general_40_9_1_3_subjob,general_operation,instruction,B-5,9.1.3,8,True,True,True,22143,0,0,
5,general_40_9_1_3_position_shift,general_operation,instruction,B-5,9.1.3,8,True,True,True,23985,0,0,
6,general_40_9_1_4_linear_motion,general_operation,instruction,B-5,9.1.4,8,True,True,True,18869,0,0,
7,general_40_9_1_4_circular_motion,general_operation,instruction,B-5,9.1.4,8,True,True,True,20327,0,0,
8,general_40_9_1_5_user_frame,general_operation,instruction,B-5,9.1.5,8,True,True,True,18479,0,0,
9,general_40_9_1_5_interrupt_position,general_operation,instruction,B-5,9.1.5,8,True,True,True,22285,0,0,


### general_40_9_1_1_single_output

请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？

,score,similarity,category_match,category_boost,distance,manual,lines,summary,source
0,0.914331,0.764331,True,0.15,0.235669,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(12655, 12658)",OUT OT# 举例：OUT OT#1 =ON，OUT OT#2 =OFF,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
1,0.904675,0.754675,True,0.15,0.245325,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(12651, 12654)",**OUT OT# 参数2（说明）**：设置为 `<IO状态>`，其中 OFF 代表断开机器...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
2,0.892867,0.742867,True,0.15,0.257133,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(7555, 7558)",该指令为 **OUT OT#**，其格式为 `OUT #<参数1>=<参数2>`。,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
3,0.890519,0.740519,True,0.15,0.259481,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(7549, 7801)",该表格详细定义了机器人 IO 输出指令集，主要涵盖**静态状态输出**、**变量赋值输出**...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
4,0.886424,0.736424,True,0.15,0.263576,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(7586, 7589)",该行为新松机器人通用操作手册中 **OUT OG** 功能块的说明行，具体描述其 **参数2...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
5,0.883749,0.733749,True,0.15,0.266251,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(7564, 7567)",在“OUT OT#”的“说明”中，“参数2”定义为<IO状态>，其中OFF表示断开机器人输出...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
6,0.882969,0.732969,True,0.15,0.267031,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(12880, 12883)","该行示例了输出指令（如 `OUT_P OT#1`，所属组包括 `OG`, `OGH` 等）的...",/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
7,0.882869,0.732869,True,0.15,0.267131,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通...,"(7666, 7669)",该表行来自《新松机器人通用操作手册》，定义了指令 **OUT OG#** 的语法格式：`OU...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...


[E1] {'key': '06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5)', 'profile': '4.0', 'revision': 'V5.0 / B-5', 'category': 'general_operation'} | /d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md | lines 12655-12658 | 1a8a05fa04b3c6ecd41ef4898e04478d25278fe2caafad0e633ce0469a256433
Parent table context, source lines 12636-13124:
<table>
<tr>
<td rowspan="5"><p>OUT OT&#35;</p></td>
<td><p>功能</p></td>
<td colspan="7"><p>IO输出，单个IO</p></td>
</tr>
<tr>
<td><p>格式</p></td>
<td colspan="7"><p>OUT &#35;&lt;参数1&gt;=&lt;参数2&gt;</p></td>
</tr>
<tr>
<td rowspan="2"><p>说明</p></td>
<td colspan="4"><p>参数1</p></td>
<td colspan="3"><p>&lt;IO号&gt;，IO号范围1&#45;4096</p></td>
</tr>
<tr>
<td colspan="4"><p>参数2</p></td>
<td colspan="3"><p>&lt;IO状态&gt;，OFF代表断开机器人输出信号，ON代表接通机器人输出信号</p></td>
</tr>
<tr>
<td><p>举例</p></td>
<td colspan="7"><p>OUT OT&#35;1 =ON</p><p>OUT OT&#35;2 =OFF</p></td>
</tr>
<tr>
<td rowspan="5

,docx,body_block,rows,excerpt
0,assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX3...,3834,"[0, 1]",OUT OT# | 功能 | IO输出，单个IO\n | 格式 | OUT #<参数1>=<...


### Retrieved summaries and manual answers

Answers use the exact evidence from this section's retrieval pass through the same
production answer method as the category suite. Completed results and available
summaries remain inspectable after failure or interruption. Reference excerpts are
reviewer data and are never appended to model context.

Alarm questions ask for meaning, causes and remedies without invented programs.
Empty cause/remedy fields remain explicitly unknown. The coding-answer validator
may return `unsupported` for a diagnostic question; review the explanation and
references independently of that status and the retrieval coverage checks.


In [12]:
# | notest
if USE_PERSISTENT_EVALUATION:
    general_operation_evaluation = evaluation.for_top_k(GENERAL_OPERATION_TOP_K)
    general_operation_answer_metrics = await general_operation_evaluation.run(general_operation_cases, concurrency=EVALUATION_CONCURRENCY)
    general_operation_answer_results = {case["id"]: general_operation_evaluation.load(case).get("answer", {}) for case in general_operation_cases}
    general_operation_answer_errors = {case["id"]: general_operation_evaluation.load(case)["answer_error"] for case in general_operation_cases if general_operation_evaluation.load(case).get("answer_error")}
    display(general_operation_answer_metrics)
    general_operation_evaluation.show_answers(general_operation_cases)
else:
    # | notest
    general_operation_answer_results, general_operation_answer_errors = {}, {}
    general_operation_answer_metrics = await run_coverage_answers(
        app, general_operation_cases, top_k=GENERAL_OPERATION_TOP_K,
        retrieval_results=general_operation_retrieval_results,
        retrieval_errors=general_operation_retrieval_errors,
        results=general_operation_answer_results, errors=general_operation_answer_errors)
    display(general_operation_answer_metrics)


## [1/38] general_40_9_1_1_single_output

请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？

### Retrieved content summary

- OUT OT# 举例：OUT OT#1 =ON，OUT OT#2 =OFF
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12655–12658; corpus 4.0, revision V5.0 / B-5; object `1a8a05fa04b3c6ecd41ef4898e04478d25278fe2caafad0e633ce0469a256433`.

- **OUT OT# 参数2（说明）**：设置为 `<IO状态>`，其中 OFF 代表断开机器人输出信号，ON 代表接通机器人输出信号。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12651–12654; corpus 4.0, revision V5.0 / B-5; object `8fc02a9902d7114bc456071da455def98c8316c968c52935fab104a14b0fa85c`.

- 该指令为 **OUT OT#**，其格式为 `OUT #<参数1>=<参数2>`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7555–7558; corpus 4.0, revision V5.0 / B-5; object `b3a954aebae14cafa282199d3c78ed562c7796768bc7702c2c89194a2ce01020`.

- 该表格详细定义了机器人 IO 输出指令集，主要涵盖**静态状态输出**、**变量赋值输出**及**路径触发输出**三大类。

**1. 静态状态与常数输出**
*   **OUT OT#**：控制单个 IO（1-4096），状态为 ON/OFF。
*   **OUT OG#**：控制 8 位组 IO（1-8），数值范围 0-255，支持常数赋值。
*   **OUT OGH#**：控制 16 位组 IO（1-256），数值范围 0-65535，支持常数赋值。
*   **OUT OGFA#**：控制 32 位组 IO（1-128），支持直接赋值整数或实数。

**2. 变量赋值输出**
此类指令将变量值写入对应位宽的组 IO：
*   **8 位组 (OG)**：支持整型变量 I (1-520)。
*   **16 位组 (OGH)**：支持整型变量 I (1-520)。
*   **32 位组 (OGFA)**：支持整型变量 I (1-520) 或实型变量 R (1-520)。

**3. 路径触发输出（动态）**
基于机器人运动轨迹，按特定条件触发单个 IO 状态：
*   **OUT_P**：**按距离**触发。参数 D 为距离阈值（-1000 至 1000 mm），正值表示向下一目标行进达到距离时触发，负值表示剩余距离时触发。
*   **OUT_T**：**按时间**触发。参数 T 为时间阈值（-30 至 30 s），正值表示向下一目标行进达到时间时触发，负值表示剩余时间时触发。

所有指令均遵循 `指令 目标#<组/点号> = <值/变量>` 的基本结构，旨在灵活控制机器人数字输出信号的时机与状态。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7549–7801; corpus 4.0, revision V5.0 / B-5; object `18b4d47778f56b34d79647b489f7ff08f5ba1fe5ed5a37f64f1279a68debc905`.

- 该行为新松机器人通用操作手册中 **OUT OG** 功能块的说明行，具体描述其 **参数2** 为 **<组IO数值>**，取值范围为 **0-255** 的常数，且组IO信号可通过界面配置（详见3.1.2节组I/O说明）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7586–7589; corpus 4.0, revision V5.0 / B-5; object `a37debc31b329605f068aa249e535a3d8e7851c50c073ed4e56c481a07eaf1e5`.

- 在“OUT OT#”的“说明”中，“参数2”定义为<IO状态>，其中OFF表示断开机器人输出信号，ON表示接通机器人输出信号。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7564–7567; corpus 4.0, revision V5.0 / B-5; object `aa80b1c233e3422560840f57de8f03ff62225d36907fd103259f50e088075ac0`.

- 该行示例了输出指令（如 `OUT_P OT#1`，所属组包括 `OG`, `OGH` 等）的用法：通过 `MOVL` 定位至 P1（速度 600，加速度 20，计数 100），执行向下一点移动 1mm 并置输出为 1，最后移动至 P2。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12880–12883; corpus 4.0, revision V5.0 / B-5; object `6db461dbc44754400d20b9a45c0c39fe2000a07378dc87006ba4ecc3c89f9052`.

- 该表行来自《新松机器人通用操作手册》，定义了指令 **OUT OG#** 的语法格式：`OUT OG#<参数1>=I<参数2>`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7666–7669; corpus 4.0, revision V5.0 / B-5; object `8f4cbe06efb8960bda957154cba375d529494cf068c56f72a4cc95ac4a77f6d2`.

### Coding or diagnostic answer

依据《新松机器人通用操作手册》(B-5)第9.1.1节，OUT OT# 用于控制单个数字IO输出。格式为 OUT OT#<IO号> = <IO状态>。参数1为IO号，范围1-4096；参数2为IO状态，ON代表接通机器人输出信号（置位），OFF代表断开机器人输出信号（复位）。先置位再复位的例子即先执行 OUT OT#1 = ON 接通信号，再执行 OUT OT#1 = OFF 断开信号。

### Example — general_operation

机器人在搬运工位需要控制一个电磁阀（IO号1）：先移动到目标位置，然后接通电磁阀（置位），完成动作后再断开电磁阀（复位）。

**Environment and prerequisites:** 前提：机器人处于手动/自动模式，IO号1已在控制器中配置为数字输出。变量：IO号=1（用户指定的输出通道）。初始值：IO#1 当前状态为 OFF。

**Expected result:** 执行 OUT OT#1 = ON 后，IO#1 输出信号接通（高电平）；执行 OUT OT#1 = OFF 后，IO#1 输出信号断开（低电平），完成一次置位-复位循环。

**Manual:** SX322001《新松机器人通用操作手册》(B-5) V5.0 [E1] [E2] [E3] [E5]

```text
MOVL P1 VL=600 ACC=20 CNT=100
OUT OT#1 = ON
OUT OT#1 = OFF
```

## [2/38] general_40_9_1_1_wait_input

请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用WAIT等待输入信号满足条件，并设置等待时间？请说明输入号、状态值和超时参数。

### Retrieved content summary

- 在 **WAIT** 指令中，举例说明其用法为：`IF I[1] < I[2] : WAIT DI[1] = 1 T = 10`，即当整数变量 `I[1]` 小于 `I[2]` 时，等待数字输入 `DI[1]` 变为 1，超时时间设为 10。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 12392–12395; corpus 5.0, revision V5.0 / C-1; object `293327c0bf0c29c6113f7a2abce34607a912ef7b994bf34e72a5ed5f69259517`.

- WAIT指令示例：展示多种等待条件，包括输入/输出信号状态（如IN#1=ON, OT#1=OFF）、计数器值（如IG#1=100, OGH#1=10000）、寄存器引用（如IGH#1=I1, IGFA#1=R1）及互锁条件（INTERLOCK IGH#1=I1），对应不同超时时间（T=-1或T=10）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7926–7929; corpus 4.0, revision V5.0 / B-5; object `5d78f5ab58f4b83c1cddcf4f8ee778e04c917fcd1d50218c088281279de00be0`.

- WAIT指令举例：涵盖数字量(I/O)、模拟量(IG/OG)、高速计数器(IGH/OGH)及互锁(INTERLOCK)等信号的条件等待，支持设定触发状态、计数/阈值条件及超时时间(T)。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 13054–13057; corpus 4.0, revision V5.0 / B-5; object `7c2b340fa197235f61ac803a6e68905702fd8cf909d4cfb83c403cb985057634`.

- WAIT指令参数1定义输入/输出IO组地址格式：DI/DO支持GIB(8位)、GIH(16位)、GII/GIF/GOI/GOF(32位)
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 12379–12383; corpus 5.0, revision V5.0 / C-1; object `1fcd7501541312695c3fc49c325f6013e41357e4fffa4da116665f430a0f12bb`.

- WAIT指令中“参数1”的“说明”列定义了<等待条件>，支持多种信号类型及其取值范围，包括：IN#<输入号>(1-4096)、OUT#<输出号>(1-4096)、IG<8位组输入号>(1-8)、OG<8位组输出号>(1-8)、IGH<16位组输入号>(1-256)、OGH<16位组输出号>(1-256)、IGH_I<16位组输出号>(1-256)、OGH_I<16位组输出号>(1-256)、IGFA_R<32位组输出号>(1-128)、IG_I<8位组输出号>(1-8)、OG_I<8位组输出号>(1-8)、IGFA_I<32位组输出号>(1-128)、IGH_I_INTERLOCK<16位组输出号>(1-256)以及I<整形变量号>(1-520)。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 13041–13045; corpus 4.0, revision V5.0 / B-5; object `17b9e441d218c270049ceaa6c56f7cd023e36b344d300f299d26fca9cfd45ffc`.

- 在WAIT指令下，示例用法为 `WAIT DI[0] = 1 T = 10`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 7115–7118; corpus 5.0, revision V5.0 / C-1; object `681ff2e44d102e0b50676dc3f19d067ebf273e5b17a2a0f3ea2bc69f012b8ecb`.

- WAIT指令第一参数说明：支持多种等待条件，包括单点IO（IN/OUT#，范围1-4096）、8位组（IG/OG，1-8）、16位组（IGH/OGH及交互变体，1-256）和32位组（IGFA变体，1-128）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7913–7917; corpus 4.0, revision V5.0 / B-5; object `b1de57c71ada50b541206bdcd5e6eeaf66761509a7b45035e73077f32b6582f3`.

- **WAIT指令 - 参数1说明**：指定需监测的DI/DO信号，支持8位（GIB/GOB）、16位（GIH/GOH）及32位（GII/GIF/GOI/GOF）组IO。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 7102–7106; corpus 5.0, revision V5.0 / C-1; object `9040c83775df5c251ea8193fa5106a1f2b5dfcc9fc40658e9654a056185eafdf`.

...

### Coding or diagnostic answer

依据《新松机器人通用操作手册》(C-1)（V5.0 / C-1）

## [38/38] general_50_alarm_6035

请依据《新松机器人通用操作手册》(C-1)的“报警信息→报警”表说明报警码6035的含义、报警原因和解决措施，并引用来源。不要把警告码、提示码或另一版本的内容混入答案；手册未填写的原因或措施请明确说明缺失。本题是报警诊断，不需要编写机器人程序。

### Retrieved content summary

- 这是一份整理好的**新松机器人（Siasun）报警码故障排除指南**（基于通用操作手册 V5.0）。

为了便于查阅，我将报警码分为 **1500-1900系列（驱动器/硬件底层故障）**、**3000系列（系统/IO/通信配置故障）**、**6000系列（运动学/应用层故障）** 和 **9000系列（系统初始化/管理器故障）**。

> **通用建议**：
> 1. 若报警涉及“初始化失败”、“内部错误”、“EEPROM异常”等，通常建议 **断电重启**。若重启无效，需 **联系新松售后技术人员** 或更换驱动器。
> 2. 若涉及“过流”、“过热”、“编码器异常”，请优先检查 **负载机械状态**、**编码器接线** 及 **散热环境**。

---

### 一、 驱动器底层与硬件故障 (报警码 1500 - 1900)
*此类报警多由伺服驱动器硬件、电机、编码器或参数初始化引起。*

#### 1. 初始化与参数类 (1549-1786, 1793-1823)
*特征：大量报警提示“初始化失败”或“对应轴号初始化异常”，解决措施高度一致。*

| 报警码 | 报警信息/描述 | 可能原因 | 建议解决措施 |
| :--- | :--- | :--- | :--- |
| **1549-1571** | AD校正/系数/限值/调节器/中断等初始化:SV | 对应轴号初始化异常/失败 | **联系技术人员处理** |
| **1572** | 主电源输入异常:SV | 三相交流电缺相 | 检查三相电源输入，确保不缺相 |
| **1573-1574** | CPU1/CPU2 看门狗溢出:SV | 驱动器内部故障 | **重启驱动器** 或 **更换驱动器** |
| **1575-1576** | 编码器数据溢出:SV | 编码器多圈实际值变化超出允许范围 | 限制编码器多圈变化在允许范围内 |
| **1577-1581** | CPU1/CPU2 工作异常/内存异常/冲突:SV | 驱动器内部故障 | **重启** 或 **更换驱动器** |
| **1582-1589** | 编码器数据/通信/内部异常:SV | 编码器故障、接线错误/断线、强干扰、内部状态异常 | 1. 检查编码器接线（屏蔽/接地/磁环）<br>2. 执行**编码器清零**<br>3. **重启驱动器** |
| **1590** | 控制编码器超速:SV | 参数设置不合理、信号受干扰、编码器损坏 | 优化调节器参数，加强抗干扰，更换编码器 |
| **1591** | 驱动器输出缺相:SV | U/V/W断线/接触不良、电机阻抗过大、采样电路异常 | 检查电机U/V/W接线，更换电机或驱动器 |
| **1592** | 电机失速:SV | 调节器参数不合理、参数0x20A0过小、编码器/电路故障 | 优化调节器参数，增大参数 0x20A0 |
| **1593** | 协处理器通讯异常:SV | 地线未可靠连接、强干扰源 | 增加磁环，确保驱动器接地可靠 |
| **1594-1595** | AB信号异常/电流跟随误差过大:SV | 接线断线/虚焊、干扰、机械卡死、参数不合理 | 1. 检查编码器接线，加磁环<br>2. 检查机械负载，优化参数 |
| **1596** | 位置目标值异常:SV | 目标值过大或最大加速度设置过小 | 合理规划位置目标值及加速度参数 |
| **1597-1598** | 编码器上电数据溢出/位置目标值溢出:SV | 上电时电机位置变化超出允许范围 | 1. 确保上电时电机位置在允许范围内<br>2. 执行**编码器多圈清零** |
| **1599** | 电机抱闸异常:SV | 抱闸本体故障、高速停机、线路短路 | 更换抱闸，检查抱闸线路 |
| **1600** | 控制电源欠压:SV | 控制电源异常、接线错误、负载过大 | 检查接线与负载，更换电源或驱动器 |
| **1601-1603** | 硬限位/急停触发:SV | 限位开关或DI输入被触发 | 检查限位开关状态，确保未触发；确认DI未触发 |
| **1604** | 转矩监测饱和故障:SV | 实际转矩电流超过故障限制值 | 适当增加参数 0x20A7 和 0x20A8 |
| **1605-1606** | 速度/全闭环位置偏差过大:SV | 机械卡死、上位机加速度过高、伺服参数(0x2012/0x20A3/0x20A4)过小、调节器不合理 | 1. 检查机械负载<br>2. 降低上位机规划加速度<br>3. 优化调节器参数 |
| **1607** | 寻原点失败:SV | 寻原点参数(0x6098-0x609A)不合理、启动时触碰限位、模式切换错误 | 重新设置寻原点参数 0x6098/0x6099/0x609A |
| **1608** | EtherCAT通讯周期错误:SV | 通讯周期小于控制周期，或未满足250μs的2的整数次幂 | 调整EtherCAT周期，确保大于控制周期且满足幂次规则 |
| **1609** | 位置规划运行错误:SV | 位置规划参数设置不合理 | 正确设置位置规划参数 |
| **1610** | EtherCAT非法同步模式:SV | 驱动器与上位机同步模式不匹配(如DC/Free Run) | 确保驱动器和上位机同步模式设置一致 |
| **1611** | 位置目标值超出设定范围:SV | 指令值超出参数 0x607B 范围 | 确保位置指令值不超出参数 0x607B |
| **1612** | 散热器过热:SV | 负载过大、温度采样异常、环境温度超限 | 减小负载，改善散热，更换驱动器 |
| **1613** | 限位开关异常:SV | 正向运行触发负限位，或反向触发正限位 | 检查接线极性定义与接线是否正确 |
| **1614** | 接口编码器分辨率变更:SV | 当前分辨率与上次掉电保存值不一致 | **重启驱动器** |
| **1615/1617** | 编码器过热:SV | 温度采样异常、环境温度超限 | 改善机柜散热，更换驱动器 |
| **1616/1618** | 编码器电池欠电压:SV | 电池未接/接线不良/电压低 | 检查电池接线，更换电池，执行多圈清零 |
| **1619** | 控制模式设定错误:SV | 使能时设定了不支持的模式，或PP模式误开无限位置 | 使能前由控制器设定正确控制模式 |
| **1620** | 上电位置偏差过大:SV | 掉电后位置偏移，或编码器电池问题 | 确保编码器电池连接正常且电压充足 |
| **1621/1646** | 编码器加速度异常:SV | 加速度超出允许范围 | 合理规划加速度，设置合理伺服参数 0x2037 |
| **1622** | 电机堵转:SV | 机械负载卡死、抱闸未释放 | 排查机械卡死，检测抱闸回路 |
| **1623** | 电机过热:SV | 温度采样异常、环境温度超限 | 降低环境温度，更换电机/驱动器 |
| **1624** | 增量式编码器Z信号异常:SV | 接线断线/虚焊、受干扰 | 检查接线，增加磁环，动力线分开布线 |
| **1625-1626** | 读写EEPROM数据异常:SV | 内部电路异常、受干扰 | **更换驱动器**，重新上电，加强抗干扰 |
| **1627** | 能耗制动电路异常:SV | 电路异常或受干扰 | **更换驱动器**，加强抗干扰 |
| **1628** | 抱闸控制电路异常:SV | 抱闸开路/短路、24V欠压、输出过载、内部故障 | 检查抱闸接线及24V供电，必要时更换驱动器 |
| **16
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 13149–16760; corpus 5.0, revision V5.0 / C-1; object `aa202746a927d3876a7f3c11aad6684e18c4365469ae973148fcdb32e8c2922d`.

- 新松机器人报警码8305（加密芯片校验故障:SV），原因为未烧录加密软件；解决措施为断电重启测试，若故障持续则联系厂家重新烧录加密软件。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 22541–22546; corpus 4.0, revision V5.0 / B-5; object `00db98f57187c85e6484d02e79ff819ddc3e350465119da3497916d2add36b90`.

- 报警码 6005 表示“关节位置超限”（报警信息提示 axis 超限），原因为关节位置超限；解决措施为进行复位或重启系统，若反复出现请联系新松售后。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 16453–16458; corpus 5.0, revision V5.0 / C-1; object `c6019b89bd96332578695668c59cd7053a5c947d2e599a21851abf5496a9f8eb`.

- 新松机器人报警码 **6023**：圆弧中心计算错误！（报警原因及解决措施未提供）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 16561–16566; corpus 5.0, revision V5.0 / C-1; object `66775c8931bdfe6edb76a3e359fe8bc66866d8c057b9a93523ab6a463602b263`.

- 这是新松机器人通用操作手册中的报警处理表头，包含四列：报警码、报警信息、报警原因及解决措施。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 16989–16994; corpus 4.0, revision V5.0 / B-5; object `4894cfd0e51c38df05ee4223b7574a92ae6eaf1dce749751be53eb3c42396247`.

- 新松机器人报警码 6016 表示“运动出现奇异点”；该行的“报警原因”和“解决措施”均为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 16519–16524; corpus 5.0, revision V5.0 / C-1; object `59e9ea1f13b8030c752a54b25ad25331f8de94ee5460ac1fcd40e8e7736603d9`.

- 这是《新松机器人通用操作手册》中报警信息表格的**表头行**，定义了四列结构：**报警码**、**提示信息**、**报警原因**、**解决措施**，用于后续行列出具体故障代码及其对应的诊断与处理方法。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 26395–26400; corpus 4.0, revision V5.0 / B-5; object `5fe9ea44c7ab279f95c05a1e173689ccad518a4bd88fa0c620488e0723bb83da`.

- 报警码 **6003** 表示外部轴描述文件名错误，原因为外部轴配置错误；解决措施为查看外部轴配置，执行复位或重启系统，若反复出现请联系新松售后人员。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 16441–16446; corpus 5.0, revision V5.0 / C-1; object `015e489d6c983eb715636b58c77ec0fed5cb044831a2ec594214168423aa03ff`.

### Coding or diagnostic answer

在所提供的检索证据中，C-1版本（V5.0 / C-1）的报警表仅包含报警码6003、6005、6016、6023，未出现报警码6035。B-5版本（V5.0 / B-5）中出现的报警码为8305，亦非6035。因此，现有证据无法说明报警码6035的含义、报警原因及解决措施，无法依据手册给出诊断结论。

,case_id,category,kind,expected_revision,expected_section,hits,expected_manual_hit,expected_section_hit,anchors_in_matching_hits,context_chars,omissions,missing_manuals,error,example_status
0,general_40_9_1_1_single_output,general_operation,instruction,B-5,9.1.1,8,True,True,True,22639,1,0,,included
1,general_40_9_1_1_wait_input,general_operation,instruction,B-5,9.1.1,8,True,True,True,23637,3,0,,included
2,general_40_9_1_2_goto_label,general_operation,instruction,B-5,9.1.2,8,True,True,True,23683,1,0,,included
3,general_40_9_1_2_conditional_integer,general_operation,instruction,B-5,9.1.2,8,True,True,True,23459,1,0,,included
4,general_40_9_1_3_subjob,general_operation,instruction,B-5,9.1.3,8,True,True,True,22143,0,0,,included
5,general_40_9_1_3_position_shift,general_operation,instruction,B-5,9.1.3,8,True,True,True,23985,0,0,,included
6,general_40_9_1_4_linear_motion,general_operation,instruction,B-5,9.1.4,8,True,True,True,18869,0,0,,included
7,general_40_9_1_4_circular_motion,general_operation,instruction,B-5,9.1.4,8,True,True,True,20327,0,0,,included
8,general_40_9_1_5_user_frame,general_operation,instruction,B-5,9.1.5,8,True,True,True,18479,0,0,,included
9,general_40_9_1_5_interrupt_position,general_operation,instruction,B-5,9.1.5,8,True,True,True,22285,0,0,,included


## Arc-welding manual version tests

This separate `arc_welding` suite contains **30 cases** from
[B-5](../assets/code-docs.4.0/06%20产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx)
and [C-1](../assets/code-docs.5.0/06%20产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx):
**20 sampled instruction entries** (ten per revision) and **ten alarm-code
questions** (two B-5 and eight C-1). Sampling is stratified across chapters with fixed seeds;
command entries are deduplicated within each revision. The population consists of
named 功能 rows with a following 格式 row, excluding a combined three-alias heading and an instantiated
MOVL example. The generic point-laser entry is identified by its actual format,
`PLSS SEARCH`. DOCX hashes, exact table rows and full entry excerpts are retained.

The requested chapter numbers follow B-5's chapter order. C-1 reorganizes the
material and has eight chapters; its actual headings are used as follows:

| Requested B-5 section | Topic | C-1 section |
|---|---|---|
| 2 | 辅助焊接功能 | 3 |
| 3 | 智能焊接—电弧篇 | 4 |
| 4 | 智能焊接—激光篇 | 5 |
| 5 | 智能焊接—点激光篇 | No separate counterpart |
| 6 | 多道焊 | 6 |
| 7 | 自适应功能 | No separate counterpart |
| 10 | 附录 | 7 |
| 11 | 报警信息 | 8 |

B-5 contributes instruction samples from all seven requested instruction chapters;
C-1 contributes two per available counterpart chapter. B-5's 报警信息 table has
only **two strict alarm rows**. Both are included, alongside eight sampled codes
from C-1's nine-row 报警信息 table. This keeps all ten questions within the actual
alarm tables. Each question preserves the listed classification and any subcode.
警告信息, 提示信息 and revision-history tables are excluded.

B-5's converted index has inconsistent numeric heading labels, so the new cases
check the **chapter title and subheading path**, in addition to corpus and manual
identity. `requested_section` records the B-5 target; `section` records the source
chapter in that revision. These are retrieval diagnostics, not correctness scores.

Run this notebook's configuration, imports, shared-helper and service cells
first; the other test batches are optional. Set `ARC_WELDING_CASE_IDS` to selected
IDs or leave it `None` for all 30. This suite owns its selections, top-k and result
dictionaries. Defining or displaying its bank makes no model requests. Run cleanup
after all chosen suites. No index rebuild or summary/checkpoint change is needed.


In [13]:
# | notest
# Source locators are zero-based word/document.xml body blocks and table rows.
ARC_WELDING_SAMPLING = {'4.0': {'source_docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
         'sha256': 'a2803d821f8d46ef3010b299733441a015a63a284f06cf848ca549c4c180372d',
         'instruction_strata': [{'section': '2',
                                 'requested_section': '2',
                                 'seed': 20261429,
                                 'population': 13,
                                 'count': 2,
                                 'selected': [{'body_block': 540, 'row': 4, 'instruction': 'WAVE ON#'},
                                              {'body_block': 593, 'row': 0, 'instruction': 'ARCCIRA'}]},
                                {'section': '3',
                                 'requested_section': '3',
                                 'seed': 20261430,
                                 'population': 17,
                                 'count': 1,
                                 'selected': [{'body_block': 1041,
                                               'row': 0,
                                               'instruction': 'DETECT 1D INIT'}]},
                                {'section': '4',
                                 'requested_section': '4',
                                 'seed': 20261431,
                                 'population': 9,
                                 'count': 1,
                                 'selected': [{'body_block': 1560, 'row': 0, 'instruction': 'LASERSES_P'}]},
                                {'section': '5',
                                 'requested_section': '5',
                                 'seed': 20261432,
                                 'population': 1,
                                 'count': 1,
                                 'selected': [{'body_block': 1941, 'row': 0, 'instruction': 'PLSS SEARCH'}]},
                                {'section': '6',
                                 'requested_section': '6',
                                 'seed': 20261433,
                                 'population': 4,
                                 'count': 2,
                                 'selected': [{'body_block': 2368, 'row': 0, 'instruction': 'MLSC ON'},
                                              {'body_block': 2368, 'row': 4, 'instruction': 'MLSC OFF'}]},
                                {'section': '7',
                                 'requested_section': '7',
                                 'seed': 20261434,
                                 'population': 8,
                                 'count': 1,
                                 'selected': [{'body_block': 2523, 'row': 8, 'instruction': 'EDSCAN ON'}]},
                                {'section': '10',
                                 'requested_section': '10',
                                 'seed': 20261437,
                                 'population': 76,
                                 'count': 2,
                                 'selected': [{'body_block': 2904, 'row': 22, 'instruction': 'ARCON_UI'},
                                              {'body_block': 2908, 'row': 75, 'instruction': 'TS SAVE'}]}],
         'alarm_strata': [{'classification': '报警信息',
                           'seed': 20261507,
                           'population': 2,
                           'count': 2,
                           'body_block': 2915,
                           'rows': [1, 2]}]},
 '5.0': {'source_docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
         'sha256': 'c0e81c46578cbbffb53d8f173f41708d06b116f0694a4645d994f896b0d5172d',
         'instruction_strata': [{'section': '3',
                                 'requested_section': '2',
                                 'seed': 20261530,
                                 'population': 9,
                                 'count': 2,
                                 'selected': [{'body_block': 567, 'row': 10, 'instruction': 'WAVEON SINE'},
                                              {'body_block': 744, 'row': 0, 'instruction': 'POTCAL'}]},
                                {'section': '4',
                                 'requested_section': '3',
                                 'seed': 20261531,
                                 'population': 12,
                                 'count': 2,
                                 'selected': [{'body_block': 852, 'row': 0, 'instruction': 'SEARCHON 3D_3R'},
                                              {'body_block': 1041, 'row': 0, 'instruction': 'ARCTRACKOFF'}]},
                                {'section': '5',
                                 'requested_section': '4',
                                 'seed': 20261532,
                                 'population': 6,
                                 'count': 2,
                                 'selected': [{'body_block': 1175,
                                               'row': 0,
                                               'instruction': 'LSENSORTRACKOFF'},
                                              {'body_block': 1182, 'row': 0, 'instruction': 'ACCURATEEND'}]},
                                {'section': '6',
                                 'requested_section': '6',
                                 'seed': 20261533,
                                 'population': 4,
                                 'count': 2,
                                 'selected': [{'body_block': 1220,
                                               'row': 6,
                                               'instruction': 'MLOFFSETON PR RMM'},
                                              {'body_block': 1220, 'row': 10, 'instruction': 'MLOFFSETOFF'}]},
                                {'section': '7',
                                 'requested_section': '10',
                                 'seed': 20261534,
                                 'population': 24,
                                 'count': 2,
                                 'selected': [{'body_block': 1238, 'row': 0, 'instruction': 'SEARCHON 3D'},
                                              {'body_block': 1239, 'row': 4, 'instruction': 'SEARCHON C'}]}],
         'alarm_strata': [{'classification': '报警信息',
                           'seed': 20261607,
                           'population': 9,
                           'count': 8,
                           'body_block': 1253,
                           'rows': [1, 2, 3, 4, 5, 6, 8, 9]}]}}

ARC_WELDING_CASES = [{'id': 'arc_40_s2_wave_on',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '2',
  'section': '2',
  'section_title': '辅助焊接功能',
  'heading_path': ['辅助焊接功能', '摆动库', '指令说明'],
  'instruction': 'WAVE ON#',
  'source_label': 'WAVE ON#',
  'alarm_code': '',
  'classification': '',
  'anchors': ['WAVE ON'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第2章“辅助焊接功能→摆动库→指令说明”说明指令条目“WAVE '
              'ON#”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 540,
                  'rows': [4, 5, 6, 7],
                  'excerpt': 'WAVE ON# | 功能 | 调用#对应的摆动文件（1-32）中的参数进行后续摆动运动\n'
                             ' | 格式 | WAVE ON #<参数1>\n'
                             ' | 说明 | 参数1 | 摆动文件。\n'
                             '范围：1-32\n'
                             ' | 举例 | WAVE ON #1'}]},
 {'id': 'arc_40_s2_arccira',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '2',
  'section': '2',
  'section_title': '辅助焊接功能',
  'heading_path': ['辅助焊接功能', '小圆功能', '指令说明'],
  'instruction': 'ARCCIRA',
  'source_label': 'ARCCIRA',
  'alarm_code': '',
  'classification': '',
  'anchors': ['ARCCIRA'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第2章“辅助焊接功能→小圆功能→指令说明”说明指令条目“ARCCIRA”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 593,
                  'rows': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
                  'excerpt': 'ARCCIRA | 功能 | 通过小圆指令，示教圆心点可以实现整圆的焊接。\n'
                             ' | 格式 | ARCCIRA P/PR参数1 VC=参数2  ACC=参数3  CNT=参数4 R=参数5  OFFSET=参数6 OVER=参数7  '
                             'UF=参数8 FIFL=参数9\n'
                             ' | 说明 | 参数1 | 圆心点位置，局部/全局位置变量号。\n'
                             ' |  | 参数2 | 圆弧的速度，最大限值为2000，单位mm。\n'
                             ' |  | 参数3 | 最大加速度百分比，单位：%，范围：1-120\n'
                             ' |  | 参数4 | 逼近等级，在该指令中默认是0，不接受其他值\n'
                             ' |  | 参数5 | 目标圆的半径，单位是mm，范围：0.1-100\n'
                             ' |  | 参数6 | '
                             '起点偏移量（单位是mm）：OFFSET>0，表示起始点在圆外；OFFSET==0，表示起始点在圆上；OFFSET<0，表示起始点在圆内；(-100,100)\n'
                             ' |  | 参数7 | 终点偏移量；过焊量，单位是mm '
                             '；范围：(-100,100)；通过R，将OVER的单位转换成角度，如果偏移角度大于60度，则应该报警，即OVER（°）<60°,\n'
                             ' |  | 参数8 | 用户坐标系号，1-128，用来确认圆平面；\n'
                             ' |  | 参数9 | 焊接文件号，1-32；\n'
                             ' | 举例 | ARCCIRA P1 VC=100 ACC=100 CNT=0 R=100 FFSET=0 OVER=80 UF=1 FIFL=1'}]},
 {'id': 'arc_40_s3_detect_1d_init',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '3',
  'section': '3',
  'section_title': '智能焊接—电弧篇',
  'heading_path': ['智能焊接—电弧篇', '三维传感', '指令说明'],
  'instruction': 'DETECT 1D INIT',
  'source_label': 'DETECT 1D INIT',
  'alarm_code': '',
  'classification': '',
  'anchors': ['DETECT 1D INIT'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第3章“智能焊接—电弧篇→三维传感→指令说明”说明指令条目“DETECT 1D '
              'INIT”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 1041,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'DETECT 1D INIT | 功能 | 一维传感初始化且读取文件参数。\n'
                             ' | 格式 | DETECT 1D INIT #<参数1>\n'
                             ' | 说明 | 参数1 | 三维传感文件号。\n'
                             ' | 举例 | DETECT 1D INIT #1'}]},
 {'id': 'arc_40_s4_laserses_p',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '4',
  'section': '4',
  'section_title': '智能焊接—激光篇',
  'heading_path': ['智能焊接—激光篇', '激光搜索', '静态搜索', '指令说明'],
  'instruction': 'LASERSES_P',
  'source_label': 'LASERSES_P',
  'alarm_code': '',
  'classification': '',
  'anchors': ['LASERSES_P'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第4章“智能焊接—激光篇→激光搜索→静态搜索→指令说明”说明指令条目“LASERSES_P”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 1560,
                  'rows': [0, 1, 2, 3, 4, 5],
                  'excerpt': 'LASERSES_P | 功能 | 线激光定点传感。\n'
                             ' | 格式 | LASERSES_P 参数1 参数2 参数3\n'
                             ' | 说明 | 参数1 | 传感点位保存变量号：PR（1-5630）\n'
                             ' |  | 参数2 | 是否加补偿值：ΔPR（加补偿值）；Δ0（不加补偿）\n'
                             ' |  | 参数3 | 补偿值位置变量号：ΔPR（1-5630）\n'
                             ' |  | 参数4 | 补偿方向：MODE=（0-6（0:XYZ,1:X,2:Y,3:Z,4:XY,5:XZ,6:YZ））'}]},
 {'id': 'arc_40_s5_plss_search',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '5',
  'section': '5',
  'section_title': '智能焊接-点激光篇',
  'heading_path': ['智能焊接-点激光篇', '点激光传感', '点激光传感指令'],
  'instruction': 'PLSS SEARCH',
  'source_label': '点激光传感指令',
  'alarm_code': '',
  'classification': '',
  'anchors': ['PLSS SEARCH'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第5章“智能焊接-点激光篇→点激光传感→点激光传感指令”说明指令条目“PLSS '
              'SEARCH”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 1941,
                  'rows': [0, 1, 2, 3],
                  'excerpt': '点激光传感指令 | 功能 | 点激光传感指令\n'
                             'PLSS SEARCH | 格式 | PLSS SEARCH  # <参数1>\n'
                             ' | 说明 | 参数1：调用PLSS文件参数界面对应的文件号，范围为[1-32]。\n'
                             ' | 举例 | PLSS SEARCH  # 10'}]},
 {'id': 'arc_40_s6_mlsc_on',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '6',
  'section': '6',
  'section_title': '多道焊',
  'heading_path': ['多道焊', '辅助功能', '扫描区间', '指令说明'],
  'instruction': 'MLSC ON',
  'source_label': 'MLSC ON',
  'alarm_code': '',
  'classification': '',
  'anchors': ['MLSC ON'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第6章“多道焊→辅助功能→扫描区间→指令说明”说明指令条目“MLSC '
              'ON”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2368,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'MLSC ON  | 功能 | 扫描区间开始。\n'
                             ' | 格式 | MLSC ON TYPE=<参数1>\n'
                             ' | 说明 | 参数1 | TYPE=1为跟踪扫描；TYPE=2为示教扫描。\n'
                             ' | 举例 | MLSC ON TYPE=1'}]},
 {'id': 'arc_40_s6_mlsc_off',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '6',
  'section': '6',
  'section_title': '多道焊',
  'heading_path': ['多道焊', '辅助功能', '扫描区间', '指令说明'],
  'instruction': 'MLSC OFF',
  'source_label': 'MLSC OFF',
  'alarm_code': '',
  'classification': '',
  'anchors': ['MLSC OFF'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第6章“多道焊→辅助功能→扫描区间→指令说明”说明指令条目“MLSC '
              'OFF”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2368,
                  'rows': [4, 5, 6, 7],
                  'excerpt': 'MLSC OFF \n'
                             ' | 功能 | 扫描区间结束。\n'
                             ' | 格式 | MLSC OFF \n'
                             ' | 说明 | 扫描区间结束，跳出区间。\n'
                             ' | 举例 | MLSC OFF'}]},
 {'id': 'arc_40_s7_edscan_on',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '7',
  'section': '7',
  'section_title': '自适应功能',
  'heading_path': ['自适应功能', '多道焊坡口自适应', '焊缝尺寸获取', '自动扫描'],
  'instruction': 'EDSCAN ON',
  'source_label': 'EDSCAN ON',
  'alarm_code': '',
  'classification': '',
  'anchors': ['EDSCAN ON'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第7章“自适应功能→多道焊坡口自适应→焊缝尺寸获取→自动扫描”说明指令条目“EDSCAN '
              'ON”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2523,
                  'rows': [8, 9, 10, 11],
                  'excerpt': 'EDSCAN ON | 功能 | 终点不扫描区间开始指令\n'
                             ' | 格式 | EDSCAN ON\n'
                             ' | 说明 | 无参数\n'
                             ' | 举例 | EDSCAN ON'}]},
 {'id': 'arc_40_s10_arcon_ui',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '10',
  'section': '10',
  'section_title': '附录',
  'heading_path': ['附录', '弧焊逻辑指令', '焊接指令'],
  'instruction': 'ARCON_UI',
  'source_label': 'ARCON_UI',
  'alarm_code': '',
  'classification': '',
  'anchors': ['ARCON_UI'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第10章“附录→弧焊逻辑指令→焊接指令”说明指令条目“ARCON_UI”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2904,
                  'rows': [22, 23, 24, 25, 26, 27, 28],
                  'excerpt': 'ARCON_UI  | 功能 | 起弧指令可通过指令参数设置起弧电流和电压。\n'
                             ' | 格式 | ARCON_UI #参数1 #参数2 I=参数3 U=参数4\n'
                             ' | 说明 | 参数1 | 起弧文件号，范围1-32。\n'
                             ' |  | 参数2 | 焊接文件号，范围1-32。\n'
                             ' |  | 参数3 | 起弧电流值。\n'
                             ' |  | 参数4 | 起弧电压值。\n'
                             ' | 举例 | ARCON_UI #1 #1 I=200 U=20'}]},
 {'id': 'arc_40_s10_ts_save',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '10',
  'section': '10',
  'section_title': '附录',
  'heading_path': ['附录', '弧焊逻辑指令', '寻位、跟踪指令'],
  'instruction': 'TS SAVE',
  'source_label': 'TS SAVE',
  'alarm_code': '',
  'classification': '',
  'anchors': ['TS SAVE'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第10章“附录→弧焊逻辑指令→寻位、跟踪指令”说明指令条目“TS '
              'SAVE”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2908,
                  'rows': [75, 76, 77, 78],
                  'excerpt': 'TS SAVE | 功能 | 记录当前接触传感偏移状态与各偏移量。\n'
                             ' | 格式 | TS SAVE N=<参数1>\n'
                             ' | 说明 | 参数1 | 存储位置。\n'
                             ' | 举例 | TS SAVE N=1，存当前三维位姿传感偏移到通道1。'}]},
 {'id': 'arc_40_alarm_6490',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '11',
  'section': '11.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '6490',
  'classification': '报警信息',
  'anchors': ['6490'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第11章“报警信息→报警信息”表解释报警码6490（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2915,
                  'rows': [1],
                  'excerpt': '6490 | 数据存储CF卡：校验失败，数据已恢复默认值，请注意！ | - | - | 系统错误，请联系技术人员。'}],
  'reference_alarm': {'报警码': '6490',
                      '报警信息': '数据存储CF卡：校验失败，数据已恢复默认值，请注意！',
                      '子码': '-',
                      '子码信息': '-',
                      '解决方法': '系统错误，请联系技术人员。'}},
 {'id': 'arc_40_alarm_6491',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '4.0',
  'revision': 'B-5',
  'requested_section': '11',
  'section': '11.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '6491',
  'classification': '报警信息',
  'anchors': ['6491'],
  'question': '请依据《新松机器人弧焊应用操作手册》(B-5)第11章“报警信息→报警信息”表解释报警码6491（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx',
                  'body_block': 2915,
                  'rows': [2],
                  'excerpt': '6491 | 机器人焊枪发生碰撞，4、5、6松抱闸。 | - | - | 请移动机器人到安全位置。'}],
  'reference_alarm': {'报警码': '6491',
                      '报警信息': '机器人焊枪发生碰撞，4、5、6松抱闸。',
                      '子码': '-',
                      '子码信息': '-',
                      '解决方法': '请移动机器人到安全位置。'}},
 {'id': 'arc_50_s3_waveon_sine',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '2',
  'section': '3',
  'section_title': '辅助焊接功能',
  'heading_path': ['辅助焊接功能', '摆焊功能', '指令说明'],
  'instruction': 'WAVEON SINE',
  'source_label': 'WAVEON SINE',
  'alarm_code': '',
  'classification': '',
  'anchors': ['WAVEON SINE'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第3章“辅助焊接功能→摆焊功能→指令说明”说明指令条目“WAVEON '
              'SINE”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 567,
                  'rows': [10, 11, 12, 13, 14, 15, 16],
                  'excerpt': 'WAVEON SINE | 功能 | 以正弦型摆动的方式进行焊接\n'
                             ' | 格式 | WAVEON SINE F=<参数1> A=<参数2> LSEC=<参数3> RSEC=<参数4>\n'
                             ' | 说明 | 参数1 | 摆动频率，摆动一个周期所需时间的倒数。取值范围：[0.00，100.00]Hz\n'
                             ' |  | 参数2 | 摆幅：指左右两端点距离，范围0.1-99.99mm。\n'
                             ' |  | 参数3 | LSEC：左停留时间，范围0-100s；\n'
                             ' |  | 参数4 | RSEC：右停留时间，范围0-100s；。\n'
                             ' | 举例 | WAVEON SINE F=1.00 A=2.00 LSEC=0.6 RSEC=0.5'}]},
 {'id': 'arc_50_s3_potcal',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '2',
  'section': '3',
  'section_title': '辅助焊接功能',
  'heading_path': ['辅助焊接功能', '交点计算', '指令说明'],
  'instruction': 'POTCAL',
  'source_label': 'POTCAL',
  'alarm_code': '',
  'classification': '',
  'anchors': ['POTCAL'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第3章“辅助焊接功能→交点计算→指令说明”说明指令条目“POTCAL”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 744,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'POTCAL  | 功能 | 根据对应文件的设置参数对各P变量进行交点计算。\n'
                             ' | 格式 | POTCAL [参数1]\n'
                             ' | 说明 | 参数1 | 文件号，对应参数界面。\n'
                             ' | 举例 | POTCAL  [1]   '}]},
 {'id': 'arc_50_s4_searchon_3d_3r',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '3',
  'section': '4',
  'section_title': '智能焊接-电弧篇',
  'heading_path': ['智能焊接-电弧篇', '三维寻位', '三维寻位', '三维寻位指令说明'],
  'instruction': 'SEARCHON 3D_3R',
  'source_label': 'SEARCHON 3D_3R',
  'alarm_code': '',
  'classification': '',
  'anchors': ['SEARCHON 3D_3R'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第4章“智能焊接-电弧篇→三维寻位→三维寻位→三维寻位指令说明”说明指令条目“SEARCHON '
              '3D_3R”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 852,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'SEARCHON 3D_3R | 功能 | 三维传感初始化且读取文件参数。\n'
                             ' | 格式 | SEARCHON 3D_3R [参数1]\n'
                             ' | 说明 | 参数1 | 三维寻位文件号。\n'
                             ' | 举例 | SEARCHON 3D_3R [1]'}]},
 {'id': 'arc_50_s4_arctrackoff',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '3',
  'section': '4',
  'section_title': '智能焊接-电弧篇',
  'heading_path': ['智能焊接-电弧篇', '电弧跟踪', '指令说明'],
  'instruction': 'ARCTRACKOFF',
  'source_label': 'ARCTRACKOFF',
  'alarm_code': '',
  'classification': '',
  'anchors': ['ARCTRACKOFF'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第4章“智能焊接-电弧篇→电弧跟踪→指令说明”说明指令条目“ARCTRACKOFF”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1041,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'ARCTRACKOFF | 功能 | 关闭电弧跟踪\n'
                             ' | 格式 | ARCTRACKOFF\n'
                             ' | 说明 | 无 | \n'
                             ' | 举例 | ARCTRACKOFF'}]},
 {'id': 'arc_50_s5_lsensortrackoff',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '4',
  'section': '5',
  'section_title': '智能焊接-激光篇',
  'heading_path': ['智能焊接-激光篇', '激光跟踪', '指令说明'],
  'instruction': 'LSENSORTRACKOFF',
  'source_label': 'LSENSORTRACKOFF',
  'alarm_code': '',
  'classification': '',
  'anchors': ['LSENSORTRACKOFF'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第5章“智能焊接-激光篇→激光跟踪→指令说明”说明指令条目“LSENSORTRACKOFF”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1175,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'LSENSORTRACKOFF | 功能 | 跟踪结束\n'
                             ' | 格式 | LSENSORTRACKOFF\n'
                             ' | 说明 | 参数 | 该指令无参数\n'
                             ' | 举例 | LSENSORTRACKOFF'}]},
 {'id': 'arc_50_s5_accurateend',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '4',
  'section': '5',
  'section_title': '智能焊接-激光篇',
  'heading_path': ['智能焊接-激光篇', '激光跟踪', '指令说明'],
  'instruction': 'ACCURATEEND',
  'source_label': 'ACCURATEEND',
  'alarm_code': '',
  'classification': '',
  'anchors': ['ACCURATEEND'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第5章“智能焊接-激光篇→激光跟踪→指令说明”说明指令条目“ACCURATEEND”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1182,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'ACCURATEEND  | 功能 | 跟踪示教终点\n'
                             ' | 格式 | MOVL P[1] V=20 ACC=100 CNT=0:  ACCURATEEND\n'
                             ' | 说明 | 参数 | -\n'
                             ' | 举例 | MOVL P[1] V=20 ACC=100 CNT=0:  ACCURATEEND :ARCOFF [1] '}]},
 {'id': 'arc_50_s6_mloffseton_pr_rmm',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '6',
  'section': '6',
  'section_title': '多道焊',
  'heading_path': ['多道焊', '循环焊接', '指令说明'],
  'instruction': 'MLOFFSETON PR RMM',
  'source_label': 'MLOFFSETON PR RMM',
  'alarm_code': '',
  'classification': '',
  'anchors': ['MLOFFSETON PR RMM'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第6章“多道焊→循环焊接→指令说明”说明指令条目“MLOFFSETON PR '
              'RMM”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1220,
                  'rows': [6, 7, 8, 9],
                  'excerpt': 'MLOFFSETON PR RMM | 功能 | 覆盖道偏移开始\n'
                             ' | 格式 | MLOFFSETON PR [参数1]  RMM [参数2]\n'
                             ' | 说明 | 参数1 | 全局变量（此全局变量为焊缝偏移量）\n'
                             ' |  | 参数2 | 文件号'}]},
 {'id': 'arc_50_s6_mloffsetoff',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '6',
  'section': '6',
  'section_title': '多道焊',
  'heading_path': ['多道焊', '循环焊接', '指令说明'],
  'instruction': 'MLOFFSETOFF',
  'source_label': 'MLOFFSETOFF',
  'alarm_code': '',
  'classification': '',
  'anchors': ['MLOFFSETOFF'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第6章“多道焊→循环焊接→指令说明”说明指令条目“MLOFFSETOFF”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1220,
                  'rows': [10, 11, 12],
                  'excerpt': 'MLOFFSETOFF | 功能 | 覆盖道偏移结束\n | 格式 | MLOFFSETOFF\n | 说明 |  | '}]},
 {'id': 'arc_50_s7_searchon_3d',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '10',
  'section': '7',
  'section_title': '附录',
  'heading_path': ['附录', '弧焊逻辑指令', '寻位指令'],
  'instruction': 'SEARCHON 3D',
  'source_label': 'SEARCHON 3D',
  'alarm_code': '',
  'classification': '',
  'anchors': ['SEARCHON 3D'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第7章“附录→弧焊逻辑指令→寻位指令”说明指令条目“SEARCHON '
              '3D”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1238,
                  'rows': [0, 1, 2, 3],
                  'excerpt': 'SEARCHON 3D | 功能 | 一维传感初始化且读取文件参数。\n'
                             ' | 格式 | SEARCHON 3D[参数1]\n'
                             ' | 说明 | 参数1 | 三维寻位文件号。\n'
                             ' | 举例 | SEARCHON 3D[1]'}]},
 {'id': 'arc_50_s7_searchon_c',
  'category': 'arc_welding',
  'kind': 'instruction',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '10',
  'section': '7',
  'section_title': '附录',
  'heading_path': ['附录', '弧焊逻辑指令', '寻位指令'],
  'instruction': 'SEARCHON C',
  'source_label': 'SEARCHON C',
  'alarm_code': '',
  'classification': '',
  'anchors': ['SEARCHON C'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第7章“附录→弧焊逻辑指令→寻位指令”说明指令条目“SEARCHON '
              'C”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1239,
                  'rows': [4, 5, 6, 7],
                  'excerpt': 'SEARCHON C | 功能 | 圆弧寻位指令。\n'
                             ' | 格式 | SEARCHON C[参数1]\n'
                             ' |  | 参数1 | 圆弧寻位文件号。\n'
                             ' | 举例 | SEARCHON C[1]'}]},
 {'id': 'arc_50_alarm_20081',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20081',
  'classification': '报警信息',
  'anchors': ['20081'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20081（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [1],
                  'excerpt': '20081 | 焊机通讯断开！ | 请检查通讯配置界面，也可能存在干扰等问题'}],
  'reference_alarm': {'报警码': '20081', '报警信息': '焊机通讯断开！', '解决方法': '请检查通讯配置界面，也可能存在干扰等问题'}},
 {'id': 'arc_50_alarm_20082',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20082',
  'classification': '报警信息',
  'anchors': ['20082'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20082（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [2],
                  'excerpt': '20082 | 焊枪发生碰撞! | 解除碰撞，检查焊机对应的焊枪碰撞信号状态'}],
  'reference_alarm': {'报警码': '20082', '报警信息': '焊枪发生碰撞!', '解决方法': '解除碰撞，检查焊机对应的焊枪碰撞信号状态'}},
 {'id': 'arc_50_alarm_20083',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20083',
  'classification': '报警信息',
  'anchors': ['20083'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20083（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [3],
                  'excerpt': '20083 | 焊机异常! | 焊机发生故障，请复位焊机故障'}],
  'reference_alarm': {'报警码': '20083', '报警信息': '焊机异常!', '解决方法': '焊机发生故障，请复位焊机故障'}},
 {'id': 'arc_50_alarm_20084',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20084',
  'classification': '报警信息',
  'anchors': ['20084'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20084（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [4],
                  'excerpt': '20084 | 焊枪碰撞! | 解除碰撞，检查焊机对应的焊枪碰撞信号状态'}],
  'reference_alarm': {'报警码': '20084', '报警信息': '焊枪碰撞!', '解决方法': '解除碰撞，检查焊机对应的焊枪碰撞信号状态'}},
 {'id': 'arc_50_alarm_20085',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20085',
  'classification': '报警信息',
  'anchors': ['20085'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20085（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [5],
                  'excerpt': '20085 | 压缩空气异常! | 无压缩空气，检查焊接状态监控中压缩空气信号状态'}],
  'reference_alarm': {'报警码': '20085', '报警信息': '压缩空气异常!', '解决方法': '无压缩空气，检查焊接状态监控中压缩空气信号状态'}},
 {'id': 'arc_50_alarm_20086',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20086',
  'classification': '报警信息',
  'anchors': ['20086'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20086（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [6],
                  'excerpt': '20086 | 水流量异常! | 水流量低，检查焊接状态监控中水流量信号状态'}],
  'reference_alarm': {'报警码': '20086', '报警信息': '水流量异常!', '解决方法': '水流量低，检查焊接状态监控中水流量信号状态'}},
 {'id': 'arc_50_alarm_20088',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20088',
  'classification': '报警信息',
  'anchors': ['20088'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20088（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [8],
                  'excerpt': '20088 | 缺丝! | 送丝异常，检查焊接状态监控中送丝信号状态'}],
  'reference_alarm': {'报警码': '20088', '报警信息': '缺丝!', '解决方法': '送丝异常，检查焊接状态监控中送丝信号状态'}},
 {'id': 'arc_50_alarm_20089',
  'category': 'arc_welding',
  'kind': 'alarm',
  'profile': '5.0',
  'revision': 'C-1',
  'requested_section': '11',
  'section': '8.1',
  'section_title': '报警信息',
  'heading_path': ['报警信息', '报警信息'],
  'instruction': '',
  'source_label': '',
  'alarm_code': '20089',
  'classification': '报警信息',
  'anchors': ['20089'],
  'question': '请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20089（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。',
  'references': [{'docx': 'assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx',
                  'body_block': 1253,
                  'rows': [9],
                  'excerpt': '20089 | 机器人焊枪发生碰撞，4、5、6松抱闸 | 请移动机器人到安全位置'}],
  'reference_alarm': {'报警码': '20089', '报警信息': '机器人焊枪发生碰撞，4、5、6松抱闸', '解决方法': '请移动机器人到安全位置'}}]


In [14]:
# | notest
ARC_WELDING_TOP_K = 8
ARC_WELDING_CASE_IDS = None  # Set a list of IDs from the table below to run a subset.
known_arc_welding_ids = {case['id'] for case in ARC_WELDING_CASES}
assert {case['category'] for case in ARC_WELDING_CASES} == {'arc_welding'}
if ARC_WELDING_CASE_IDS is not None:
    if not ARC_WELDING_CASE_IDS or set(ARC_WELDING_CASE_IDS) - known_arc_welding_ids:
        raise ValueError('Select at least one known arc-welding case ID')
arc_welding_cases = [case for case in ARC_WELDING_CASES
                     if ARC_WELDING_CASE_IDS is None or case['id'] in ARC_WELDING_CASE_IDS]
ARC_WELDING_INSPECT_CASE_ID = arc_welding_cases[0]['id']
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(arc_welding_cases)[
        ['id', 'kind', 'profile', 'revision', 'requested_section', 'section',
         'section_title', 'instruction', 'alarm_code', 'classification', 'question']])
display(app.snapshot('arc_welding')[2])


,id,kind,profile,revision,requested_section,section,section_title,instruction,alarm_code,classification,question
0,arc_40_s2_wave_on,instruction,4.0,B-5,2,2,辅助焊接功能,WAVE ON#,,,请依据《新松机器人弧焊应用操作手册》(B-5)第2章“辅助焊接功能→摆动库→指令说明”说明指令条目“WAVE ON#”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
1,arc_40_s2_arccira,instruction,4.0,B-5,2,2,辅助焊接功能,ARCCIRA,,,请依据《新松机器人弧焊应用操作手册》(B-5)第2章“辅助焊接功能→小圆功能→指令说明”说明指令条目“ARCCIRA”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
2,arc_40_s3_detect_1d_init,instruction,4.0,B-5,3,3,智能焊接—电弧篇,DETECT 1D INIT,,,请依据《新松机器人弧焊应用操作手册》(B-5)第3章“智能焊接—电弧篇→三维传感→指令说明”说明指令条目“DETECT 1D INIT”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
3,arc_40_s4_laserses_p,instruction,4.0,B-5,4,4,智能焊接—激光篇,LASERSES_P,,,请依据《新松机器人弧焊应用操作手册》(B-5)第4章“智能焊接—激光篇→激光搜索→静态搜索→指令说明”说明指令条目“LASERSES_P”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
4,arc_40_s5_plss_search,instruction,4.0,B-5,5,5,智能焊接-点激光篇,PLSS SEARCH,,,请依据《新松机器人弧焊应用操作手册》(B-5)第5章“智能焊接-点激光篇→点激光传感→点激光传感指令”说明指令条目“PLSS SEARCH”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
5,arc_40_s6_mlsc_on,instruction,4.0,B-5,6,6,多道焊,MLSC ON,,,请依据《新松机器人弧焊应用操作手册》(B-5)第6章“多道焊→辅助功能→扫描区间→指令说明”说明指令条目“MLSC ON”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
6,arc_40_s6_mlsc_off,instruction,4.0,B-5,6,6,多道焊,MLSC OFF,,,请依据《新松机器人弧焊应用操作手册》(B-5)第6章“多道焊→辅助功能→扫描区间→指令说明”说明指令条目“MLSC OFF”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
7,arc_40_s7_edscan_on,instruction,4.0,B-5,7,7,自适应功能,EDSCAN ON,,,请依据《新松机器人弧焊应用操作手册》(B-5)第7章“自适应功能→多道焊坡口自适应→焊缝尺寸获取→自动扫描”说明指令条目“EDSCAN ON”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
8,arc_40_s10_arcon_ui,instruction,4.0,B-5,10,10,附录,ARCON_UI,,,请依据《新松机器人弧焊应用操作手册》(B-5)第10章“附录→弧焊逻辑指令→焊接指令”说明指令条目“ARCON_UI”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。
9,arc_40_s10_ts_save,instruction,4.0,B-5,10,10,附录,TS SAVE,,,请依据《新松机器人弧焊应用操作手册》(B-5)第10章“附录→弧焊逻辑指令→寻位、跟踪指令”说明指令条目“TS SAVE”的功能、实际书写格式、参数范围或单位，以及使用前提和调用顺序。请区分表中条目名称与实际指令格式，以及独立指令与运动指令附加项；仅在同一版本的证据充分时给出准备、该指令及后续动作的完整示例，缺少的条件请明确说明。


{'indexed': [{'key': '06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5)',
   'profile': '4.0',
   'revision': 'V5.0 / B-5',
   'category': 'arc_welding'},
  {'key': '06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1)',
   'profile': '5.0',
   'revision': 'V5.0 / C-1',
   'category': 'arc_welding'}],
 'missing': []}

### Retrieve and inspect arc-welding evidence

Use the production category-reranked retriever and report expected corpus/manual,
chapter/subheading path and command/code anchors. Chapter titles guard against the
numbering changes above. Inspect a selected case's exact context and DOCX reference
with the call below; references remain reviewer data and are never appended to
model context. Production retrieval can return both revisions, so an intended
revision missing from the returned evidence is reported explicitly.


In [ ]:
# | notest
if USE_PERSISTENT_EVALUATION:
    arc_welding_evaluation = evaluation.for_top_k(ARC_WELDING_TOP_K)
    arc_welding_retrieval_metrics = await arc_welding_evaluation.retrieve_cases(arc_welding_cases)
    arc_welding_retrieval_results = {case["id"]: arc_welding_evaluation.load(case).get("retrieval", {}) for case in arc_welding_cases if arc_welding_evaluation.load(case).get("retrieval_status") == "complete"}
    arc_welding_retrieval_errors = {case["id"]: arc_welding_evaluation.load(case)["retrieval_error"] for case in arc_welding_cases if arc_welding_evaluation.load(case).get("retrieval_error")}
    display(arc_welding_retrieval_metrics)
else:
    # | notest
    arc_welding_retrieval_results, arc_welding_retrieval_errors = {}, {}
    arc_welding_retrieval_metrics = await run_coverage_retrievals(
        app, arc_welding_cases, top_k=ARC_WELDING_TOP_K,
        results=arc_welding_retrieval_results, errors=arc_welding_retrieval_errors)
    display(arc_welding_retrieval_metrics)
    show_coverage_retrieval(
        ARC_WELDING_INSPECT_CASE_ID, cases=arc_welding_cases,
        results=arc_welding_retrieval_results, errors=arc_welding_retrieval_errors)


[1/30] Retrieve arc_40_s2_wave_on
[2/30] Retrieve arc_40_s2_arccira
[3/30] Retrieve arc_40_s3_detect_1d_init
[4/30] Retrieve arc_40_s4_laserses_p
[5/30] Retrieve arc_40_s5_plss_search
[6/30] Retrieve arc_40_s6_mlsc_on
[7/30] Retrieve arc_40_s6_mlsc_off
[8/30] Retrieve arc_40_s7_edscan_on
[9/30] Retrieve arc_40_s10_arcon_ui
[10/30] Retrieve arc_40_s10_ts_save
[11/30] Retrieve arc_40_alarm_6490
[12/30] Retrieve arc_40_alarm_6491
[13/30] Retrieve arc_50_s3_waveon_sine
[14/30] Retrieve arc_50_s3_potcal
[15/30] Retrieve arc_50_s4_searchon_3d_3r
[16/30] Retrieve arc_50_s4_arctrackoff
[17/30] Retrieve arc_50_s5_lsensortrackoff
[18/30] Retrieve arc_50_s5_accurateend
[19/30] Retrieve arc_50_s6_mloffseton_pr_rmm
[20/30] Retrieve arc_50_s6_mloffsetoff


### Retrieved summaries and arc-welding answers

Reuse this suite's exact retrieval evidence through the production answer method.
Each case displays the stored-content summary and its answer; completed cases and
available summaries remain inspectable after failure or interruption. Review
version-specific syntax, motion attachments, prerequisites, alarm severity and
subcodes against the source excerpts. Unknown causes or remedies must remain
unknown. Diagnostic answers need no invented program; the coding validator's
`unsupported` status is separate from retrieval coverage and source correctness.


In [ ]:
# | notest
if USE_PERSISTENT_EVALUATION:
    arc_welding_evaluation = evaluation.for_top_k(ARC_WELDING_TOP_K)
    arc_welding_answer_metrics = await arc_welding_evaluation.run(arc_welding_cases, concurrency=EVALUATION_CONCURRENCY)
    arc_welding_answer_results = {case["id"]: arc_welding_evaluation.load(case).get("answer", {}) for case in arc_welding_cases}
    arc_welding_answer_errors = {case["id"]: arc_welding_evaluation.load(case)["answer_error"] for case in arc_welding_cases if arc_welding_evaluation.load(case).get("answer_error")}
    display(arc_welding_answer_metrics)
    arc_welding_evaluation.show_answers(arc_welding_cases)
else:
    # | notest
    arc_welding_answer_results, arc_welding_answer_errors = {}, {}
    arc_welding_answer_metrics = await run_coverage_answers(
        app, arc_welding_cases, top_k=ARC_WELDING_TOP_K,
        retrieval_results=arc_welding_retrieval_results, retrieval_errors=arc_welding_retrieval_errors,
        results=arc_welding_answer_results, errors=arc_welding_answer_errors)
    display(arc_welding_answer_metrics)


## CSV manual sampling tests

The following 18 families contain **760 new cases**, **20 per CSV file**. Revisions are grouped by family. Samples span substantive chapters and available task types. Original prose and HTML rows are retained as reviewer references. References and expected facts never enter the answer prompt. Manual revision and RC corpus profile are separate: CSV cases match canonical source paths across either corpus.

Questions were authored locally from the fixed source samples; expected facts and anchors are literal source text. The shortest manual has 16 source units and uses 20 curated, distinct tasks within those units. Its manifest records that selection explicitly.

In [ ]:
# | notest
SAMPLED_CASES = []
SAMPLING_MANIFEST = []
SAMPLED_BANKS = {}
def register_sample_bank(category, cases, manifest):
    SAMPLED_BANKS[category] = (cases, manifest)
    SAMPLED_CASES[:] = [case for bank, _ in SAMPLED_BANKS.values() for case in bank]
    SAMPLING_MANIFEST[:] = [entry for _, entries in SAMPLED_BANKS.values() for entry in entries]


## General Operation — sampled manual revisions

60 cases from 3 CSV files. Set `GENERAL_OPERATION_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
GENERAL_OPERATION_SAMPLE_CASES = [{'id': 'sample_general_operation_e8c48bad9b_01', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '1.3.3', 'section_title': '1.3.3 维修、保养时的安全', 'heading_path': ['1 安全', '1.3 作业人员的安全', '1.3.3 维修、保养时的安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“维修、保养时的安全”中，应如何理解或操作“更换机器人部件时”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['9. 更换机器人部件时，不应使异物进入机器人'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 719, 'line_end': 719, 'excerpt': '9. 更换机器人部件时，不应使异物进入机器人。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_02', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '5.2.2', 'section_title': '5.2.2 3种运行模式', 'heading_path': ['5 程序的示教', '5.2 开机并手动控制', '5.2.2 3种运行模式'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），操作“3种运行模式”时，涉及“在T1或者T2模式”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['在T1或者T2模式，可以在安全门开启的情况下手动操作机器人', '必须在安全开关被按下的情况下，才可以运动机器人，安全按钮没有按下时，机器人不能运动'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 7752, 'line_end': 7752, 'excerpt': '在T1或者T2模式，可以在安全门开启的情况下手动操作机器人。必须在安全开关被按下的情况下，才可以运动机器人，安全按钮没有按下时，机器人不能运动。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_03', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '5.3.3', 'section_title': '5.3.3 示教动作指令', 'heading_path': ['5 程序的示教', '5.3 示教程序', '5.3.3 示教动作指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），“示教动作指令”中“起始点P0”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['MOVC'], 'expected_facts': ['起始点P0，因未添加相同示教点P1，故机器人从P0以直线运动形式运动到P1', '机器人从P1点走到P2点的轨迹由P1、P2、P3三点共同规划获得', '由于有连续多条MOVC轨迹，机器人从P2点走到P3点的轨迹重新规划，由P2、P3、P4三点共同规划获得'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 8295, 'line_end': 8295, 'excerpt': '起始点P0，因未添加相同示教点P1，故机器人从P0以直线运动形式运动到P1。机器人从P1点走到P2点的轨迹由P1、P2、P3三点共同规划获得。由于有连续多条MOVC轨迹，机器人从P2点走到P3点的轨迹重新规划，由P2、P3、P4三点共同规划获得。机器人最终的运行轨迹如图5-25实线所示，虚线轨迹为规划过但未执行的轨迹。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_04', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '9.1.7', 'section_title': '9.1.7 视觉通讯', 'heading_path': ['9 应用与操作', '9.1 通讯设备的设置', '9.1.7 视觉通讯'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“视觉通讯”时，“连接LAN2”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['连接LAN2，设置机器人IP，配置出视觉应用，配置视觉应用参数，根据不同的视觉品牌，查看不同的视觉相关指导文件'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 9790, 'line_end': 9790, 'excerpt': '连接LAN2，设置机器人IP，配置出视觉应用，配置视觉应用参数，根据不同的视觉品牌，查看不同的视觉相关指导文件。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_05', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '9.3.5', 'section_title': '9.3.5 变1、2、3标定', 'heading_path': ['9 应用与操作', '9.3 外部轴配置', '9.3.5 变1、2、3标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），操作“变1、2、3标定”时，涉及“变位机标定方法”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. 变位机标定方法：将工具中心点对准变位机上的参考点，标定第一点，该点的状态随即切换至ON', '切换为关节坐标和外部轴系统，正方向运动改变当前标定节点变位机角度（其他非标定节点禁止运动），旋转角度大于20度，切换回直角坐标和本体轴系统再移动机器人本体，使用直角坐标改变机器人XYZ坐标值，（不可改变机器人Rx，Ry，Rz坐标值），使工具中心点仍对准参考点，依次标定其余四点，按保存，变位机被标定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 10163, 'line_end': 10163, 'excerpt': '1. 变位机标定方法：将工具中心点对准变位机上的参考点，标定第一点，该点的状态随即切换至ON。切换为关节坐标和外部轴系统，正方向运动改变当前标定节点变位机角度（其他非标定节点禁止运动），旋转角度大于20度，切换回直角坐标和本体轴系统再移动机器人本体，使用直角坐标改变机器人XYZ坐标值，（不可改变机器人Rx，Ry，Rz坐标值），使工具中心点仍对准参考点，依次标定其余四点，按保存，变位机被标定。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_06', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '8.3', 'section_title': '8.3 作业上传/下载', 'heading_path': ['8 备份、恢复资料', '8.3 作业上传/下载'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“作业上传/下载”中，应如何理解或操作“点击主菜单【备份恢复】-【作业下载】”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 点击主菜单【备份恢复】-【作业下载】，进入作业恢复界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 9717, 'line_end': 9717, 'excerpt': '3. 点击主菜单【备份恢复】-【作业下载】，进入作业恢复界面。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_07', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '10.1.6', 'section_title': '10.1.6 计算类指令', 'heading_path': ['10 附录', '10.1 基础指令表', '10.1.6 计算类指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），“计算类指令”中“参数2”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['参数2 ·常数，范围0-255 ·B<变量号>：字节型用户变量，范围1-100 ·<<常数，左移位数，范围1-8 ·>>常数，右移位数，范围1-8'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 13107, 'line_end': 13110, 'excerpt': '<tr>\n<td colspan="2"><p>参数2</p></td>\n<td><p>·常数，范围0&#45;255</p><p>·B&lt;变量号&gt;：字节型用户变量，范围1&#45;100</p><p>·&lt;&lt;常数，左移位数，范围1&#45;8</p><p>·&gt;&gt;常数，右移位数，范围1&#45;8</p></td>\n</tr>', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_08', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '4', 'section_title': '4 程序的说明', 'heading_path': ['4 程序的说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），“程序的说明”中“还有一些机器人的程序指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['还有一些机器人的程序指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 4645, 'line_end': 4645, 'excerpt': '还有一些机器人的程序指令。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_09', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '9.2', 'section_title': '9.2 预约功能', 'heading_path': ['9 应用与操作', '9.2 预约功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“预约功能”中，应如何理解或操作“可按照工位实际编号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['可按照工位实际编号，输入作业名、启动作业的输入、输出信号、原位号', '设置完成后，按【保存】按钮保存'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 9878, 'line_end': 9878, 'excerpt': '可按照工位实际编号，输入作业名、启动作业的输入、输出信号、原位号。设置完成后，按【保存】按钮保存**。**', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_10', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '4.3.2', 'section_title': '4.3.2 位置信息', 'heading_path': ['4 程序的说明', '4.3 运动指令', '4.3.2 位置信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“位置信息”时，“参数3”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['参数3 R<变量号>，范围1-520'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 5247, 'line_end': 5250, 'excerpt': '<tr>\n<td><p>参数3</p></td>\n<td><p>R&lt;变量号&gt;，范围1&#45;520</p></td>\n</tr>', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_11', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '3.9', 'section_title': '3.9 执行开关设定', 'heading_path': ['3 机器人各项设定', '3.9 执行开关设定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），操作“执行开关设定”时，涉及“释放硬限位开关并解除故障后”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['释放硬限位开关并解除故障后，必须再次进入执行开关，将硬限位设置为ON', '以免硬限位开关无效导致机器人碰撞造成的人身伤害或设备损坏'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 3912, 'line_end': 3912, 'excerpt': '**释放硬限位开关并解除故障后，必须再次进入执行开关，将硬限位设置为ON。以免硬限位开关无效导致机器人碰撞造成的人身伤害或设备损坏。**', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_12', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '10.1.3', 'section_title': '10.1.3 控制类2', 'heading_path': ['10 附录', '10.1 基础指令表', '10.1.3 控制类2'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“控制类2”中，应如何理解或操作“TIMER GET”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['TIMER GET 功能 获取计时器时间，将其赋值给实型用户变量'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 11996, 'line_end': 12000, 'excerpt': '<tr>\n<td rowspan="6"><p>TIMER GET</p></td>\n<td><p>功能</p></td>\n<td colspan="4"><p>获取计时器时间，将其赋值给实型用户变量</p></td>\n</tr>', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_13', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '3.1.2', 'section_title': '3.1.2 模拟I\\O', 'heading_path': ['3 机器人各项设定', '3.1 I/O', '3.1.2 模拟I\\O'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“模拟I\\O”时，“注”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['注：修改完参数后，先点【保存】按钮，然后再点【刷新】按钮，物理量实时显示值才是修改完参数后对应的数值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 2179, 'line_end': 2179, 'excerpt': '注：修改完参数后，先点【保存】按钮，然后再点【刷新】按钮，物理量实时显示值才是修改完参数后对应的数值。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_14', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '6.1', 'section_title': '6.1 程序的停止和恢复', 'heading_path': ['6 程序的运行', '6.1 程序的停止和恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“程序的停止和恢复”中，应如何理解或操作“瞬间停止”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['瞬间停止：机器人迅速停止动作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 8679, 'line_end': 8679, 'excerpt': '瞬间停止：机器人迅速停止动作。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_15', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '2.4.2', 'section_title': '2.4.2 示教器窗口', 'heading_path': ['2 概述', '2.4 控制器', '2.4.2 示教器窗口'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“示教器窗口”中，应如何理解或操作“菜单树结构如下表2-7所示”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['菜单树结构如下表2-7所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 1370, 'line_end': 1370, 'excerpt': '菜单树结构如下表2-7所示。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_16', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '3.8.2', 'section_title': '3.8.2 空间形状标定', 'heading_path': ['3 机器人各项设定', '3.8 干涉区', '3.8.2 空间形状标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“空间形状标定”中，应如何理解或操作“点击【用户配置】-【干涉区】-【空间形状标定】”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 点击【用户配置】-【干涉区】-【空间形状标定】，进入空间干涉区形状配置界面，形状类型选择立方体', '上、下一页选择干涉区号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 3424, 'line_end': 3424, 'excerpt': '1. 点击【用户配置】-【干涉区】-【空间形状标定】，进入空间干涉区形状配置界面，形状类型选择立方体。上、下一页选择干涉区号。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_17', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '6.4.1', 'section_title': '6.4.1 测试前需要确定的设置', 'heading_path': ['6 程序的运行', '6.4 对程序进行测试', '6.4.1 测试前需要确定的设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），“测试前需要确定的设置”中“联锁状态，代表机器人在手动运行时，是否跳过逻辑指令。 o/o 是联锁指令开启状态，逻辑指令都执行”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['联锁 联锁状态，代表机器人在手动运行时，是否跳过逻辑指令', 'o/o 是联锁指令开启状态，逻辑指令都执行', 'o/x 代表逻辑指令不执行'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 8989, 'line_end': 8992, 'excerpt': '<tr>\n<td><p>联锁</p><p><img src="img/ad3699e748b4c3730b7d1a40faf65777ec16eef539443c1701fe652134540afc.png"/></p></td>\n<td><p>联锁状态，代表机器人在手动运行时，是否跳过逻辑指令。</p><p><strong>o/o</strong>是联锁指令开启状态，逻辑指令都执行。</p><p><strong>o/x</strong>代表逻辑指令不执行。</p></td>\n</tr>', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_18', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '10.1.3', 'section_title': '10.1.3 控制类2', 'heading_path': ['10 附录', '10.1 基础指令表', '10.1.3 控制类2'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“控制类2”时，“参数4”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['参数4 P变量，范围（1-5630）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 12098, 'line_end': 12101, 'excerpt': '<tr>\n<td><p>参数4</p></td>\n<td colspan="3"><p>P变量，范围（1&#45;5630）</p></td>\n</tr>', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_19', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '6.4.3', 'section_title': '6.4.3 连续运行测试程序', 'heading_path': ['6 程序的运行', '6.4 对程序进行测试', '6.4.3 连续运行测试程序'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“连续运行测试程序”时，“确定机器人运行范围内没有人”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 确定机器人运行范围内没有人'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 9122, 'line_end': 9122, 'excerpt': '1. 确定机器人运行范围内没有人。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_e8c48bad9b_20', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.4', 'section': '8.1', 'section_title': '8.1 系统参数、用户参数、动态参数', 'heading_path': ['8 备份、恢复资料', '8.1 系统参数、用户参数、动态参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'source_version': '', 'question': '根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），配置“系统参数、用户参数、动态参数”时，“对系统参数、用户参数、动态参数进行备份和恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['对系统参数、用户参数、动态参数进行备份和恢复'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'line_start': 9637, 'line_end': 9637, 'excerpt': '对系统参数、用户参数、动态参数进行备份和恢复。', 'sha256': '55f7d91923261eac727eecb8ee752f152017f9775575a94221d29dc405bc75d2'}]}, {'id': 'sample_general_operation_719be35c91_01', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '7.1', 'section_title': '7.1 参数备份恢复', 'heading_path': ['7 备份、恢复资料', '7.1 参数备份恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“参数备份恢复”时，“对参数进行备份和恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 对参数进行备份和恢复'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10924, 'line_end': 10924, 'excerpt': '2. 对参数进行备份和恢复。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_02', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '2.25.2', 'section_title': '2.25.2 STR指令', 'heading_path': ['2 机器人各项设定', '2.25 SOCKET通讯', '2.25.2 STR指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），“STR指令”中“STRCPY”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['STRCPY', 'DEST'], 'expected_facts': ['STRCPY 复制字符串 STRCPY DEST=LC#1 SRC=LC#2'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 5399, 'line_end': 5403, 'excerpt': '<tr>\n<td><p>STRCPY</p></td>\n<td><p>复制字符串</p></td>\n<td><p>STRCPY DEST=LC&#35;1 SRC=LC&#35;2</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_03', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.6', 'section_title': '4.6 作业变换', 'heading_path': ['4 程序的示教', '4.6 作业变换'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“作业变换”中，应如何理解或操作“可选择偏移、镜像、关节三种方式进行变换”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['可选择偏移、镜像、关节三种方式进行变换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 9797, 'line_end': 9797, 'excerpt': '可选择偏移、镜像、关节三种方式进行变换。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_04', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '8.1.3', 'section_title': '8.1.3 ModbusTCP', 'heading_path': ['8 应用与操作', '8.1 通讯设备的设置', '8.1.3 ModbusTCP'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“ModbusTCP”中，应如何理解或操作“通讯周期间隔”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. 通讯周期间隔：每经过设定时间进行请求数据，每次请求数据进行一个或多个开启的功能码命令请求'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 11482, 'line_end': 11482, 'excerpt': '5. 通讯周期间隔：每经过设定时间进行请求数据，每次请求数据进行一个或多个开启的功能码命令请求。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_05', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '5.6.1', 'section_title': '5.6.1 路径恢复的条件', 'heading_path': ['5 程序的运行', '5.6 路径恢复', '5.6.1 路径恢复的条件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“路径恢复的条件”时，“路径恢复最小距离”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['路径恢复最小距离：机器人实际位置与暂停处位置小于这个距离（TCP距离）时，机器人会直接从当前位置启动，不回到暂停处在继续运行'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10451, 'line_end': 10451, 'excerpt': '**路径恢复最小距离：**机器人实际位置与暂停处位置小于这个距离（TCP距离）时，机器人会直接从当前位置启动，不回到暂停处在继续运行。单位mm。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_06', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '2.26.2', 'section_title': '2.26.2 本体方位', 'heading_path': ['2 机器人各项设定', '2.26 其他设定', '2.26.2 本体方位'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“本体方位”时，“点击主菜单【用户配置】-【本体方位】”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['点击主菜单【用户配置】-【本体方位】，进入底座安装方向设置页面', '可设置机器人底座的安装角度，设置好相关参数后，点击【保存】键，设置成功后，会改变机器人动力学补偿结果，如图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 5536, 'line_end': 5536, 'excerpt': '点击主菜单【用户配置】-【本体方位】，进入底座安装方向设置页面。可设置机器人底座的安装角度，设置好相关参数后，点击【保存】键，设置成功后，会改变机器人动力学补偿结果，如图所示。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_07', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.2.2', 'section_title': '4.2.2 3种运行模式', 'heading_path': ['4 程序的示教', '4.2 开机并手动控制', '4.2.2 3种运行模式'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），操作“3种运行模式”时，涉及“在T1或者T2模式”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['在T1或者T2模式，可以在安全门开启的情况下手动操作机器人', '必须在DEADMAN被按下的情况下，才可以运动机器人，DEADMAN没有按下时，机器人不能运动'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 9069, 'line_end': 9069, 'excerpt': '在T1或者T2模式，可以在安全门开启的情况下手动操作机器人。必须在DEADMAN被按下的情况下，才可以运动机器人，DEADMAN没有按下时，机器人不能运动。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_08', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '9.1.2', 'section_title': '9.1.2 控制类1', 'heading_path': ['9 附录', '9.1 基础指令表', '9.1.2 控制类1'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“控制类1”时，“参数1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 参数1 B<字节型变量号>，范围1-100'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 13300, 'line_end': 13304, 'excerpt': '<tr>\n<td rowspan="4"><p>说明</p></td>\n<td colspan="3"><p>参数1</p></td>\n<td><p>B&lt;字节型变量号&gt;，范围1&#45;100</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_09', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '8.7', 'section_title': '8.7 更换CFast卡', 'heading_path': ['8 应用与操作', '8.7 更换CFast卡'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），“更换CFast卡”中“按这里弹出 CFast卡 USB接口”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['USB'], 'expected_facts': ['按这里弹出 CFast卡 USB接口'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 12557, 'line_end': 12559, 'excerpt': '<tr>\n<td><p>按这里弹出</p><p>CFast卡</p><p><img src="img/78017bf2026a609d7f04fabc499306f3e130176881b58dabfb96888dc014bd5a.emf"/></p><p>USB接口</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_10', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.4.2', 'section_title': '1.4.2 示教器窗口', 'heading_path': ['1 概述', '1.4 控制器', '1.4.2 示教器窗口'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“示教器窗口”时，“备份恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['备份恢复 参数 高级用户'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 1938, 'line_end': 1943, 'excerpt': '<tr>\n<td rowspan="5"><p>备份恢复</p></td>\n<td><p>参数</p></td>\n<td></td>\n<td><p>高级用户</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_11', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '5.2.3', 'section_title': '5.2.3 从暂停恢复启动', 'heading_path': ['5 程序的运行', '5.2 程序的执行', '5.2.3 从暂停恢复启动'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“从暂停恢复启动”中，应如何理解或操作“当机器人自动运行暂停后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['当机器人自动运行暂停后，重新按启动按钮，或外部启动，即可继续按照程序运动'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10154, 'line_end': 10154, 'excerpt': '当机器人自动运行暂停后，重新按启动按钮，或外部启动，即可继续按照程序运动。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_12', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.4.1', 'section_title': '1.4.1 示教器硬件', 'heading_path': ['1 概述', '1.4 控制器', '1.4.1 示教器硬件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“示教器硬件”中，应如何理解或操作“机器人在执行模式下,启动选定程序的按钮。”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['机器人在执行模式下,启动选定程序的按钮'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 1106, 'line_end': 1109, 'excerpt': '<tr>\n<td rowspan="2"><p><img src="img/f0e24ec6cd754b9cd4064c8eec1f78c5cfe24bd8349c48d6a3748ae9ad1b4d9b.emf"/></p></td>\n<td><p>机器人在执行模式下,启动选定程序的按钮。</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_13', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '7.3', 'section_title': '7.3 作业上传/下载', 'heading_path': ['7 备份、恢复资料', '7.3 作业上传/下载'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“作业上传/下载”中，应如何理解或操作“选择作业名”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 选择作业名，点击【上传】对单个作业进行备份', '（点击上传后会有提示，是否上传该作业，点击确认，会提示文件上传成功', '如U盘中已有该文件，会提示是否覆盖，点击确认后会提示成功覆盖上传'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10960, 'line_end': 10960, 'excerpt': '3. 选择作业名，点击【上传】对单个作业进行备份。（点击上传后会有提示，是否上传该作业，点击确认，会提示文件上传成功。如U盘中已有该文件，会提示是否覆盖，点击确认后会提示成功覆盖上传。）', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_14', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '8.1.1', 'section_title': '8.1.1 Devicenet', 'heading_path': ['8 应用与操作', '8.1 通讯设备的设置', '8.1.1 Devicenet'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“Devicenet”时，“输入长度”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['6. 输入长度： DeviceNet从站->机器人控制器的数据长度，单位为字节'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 11124, 'line_end': 11124, 'excerpt': '6. 输入长度： DeviceNet从站->机器人控制器的数据长度，单位为字节。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_15', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '7.3', 'section_title': '7.3 作业上传/下载', 'heading_path': ['7 备份、恢复资料', '7.3 作业上传/下载'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），操作“作业上传/下载”时，涉及“注意： 作业下载需要满足以下条件，应用、型号、轴数相同才能下载成功。”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意： 作业下载需要满足以下条件，应用、型号、轴数相同才能下载成功'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10982, 'line_end': 10984, 'excerpt': '<tr>\n<td><p><strong>注意：</strong></p><p><strong>作业下载需要满足以下条件，应用、型号、轴数相同才能下载成功。</strong></p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_16', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.4.3', 'section_title': '1.4.3 通讯', 'heading_path': ['1 概述', '1.4 控制器', '1.4.3 通讯'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），“通讯”中“为总线通讯而配置的接口有”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['为总线通讯而配置的接口有：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 2187, 'line_end': 2187, 'excerpt': '为总线通讯而配置的接口有：', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_17', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.6.2', 'section_title': '4.6.2 作业镜像变换', 'heading_path': ['4 程序的示教', '4.6 作业变换', '4.6.2 作业镜像变换'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），配置“作业镜像变换”时，“用户坐标系”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['用户坐标系：输入范围为1-8，选择参照的用户坐标系'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 9877, 'line_end': 9877, 'excerpt': '用户坐标系：输入范围为1-8，选择参照的用户坐标系。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_18', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '2.8', 'section_title': '2.8 执行开关', 'heading_path': ['2 机器人各项设定', '2.8 执行开关'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“执行开关”中，应如何理解或操作“系统实型5：”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['系统实型5： 世界坐标系姿态值R.y'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 4382, 'line_end': 4385, 'excerpt': '<tr>\n<td><p>系统实型5：</p></td>\n<td><p>世界坐标系姿态值R&#46;y</p></td>\n</tr>', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_19', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '6.1', 'section_title': '6.1 示教器的LED', 'heading_path': ['6 状态显示', '6.1 示教器的LED'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），在“示教器的LED”中，应如何理解或操作“表格 6-1 LED状态”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['表格 6-1 LED状态'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 10578, 'line_end': 10578, 'excerpt': '表格 6-1 LED状态', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_719be35c91_20', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '10.1', 'section_title': '10.1 报警', 'heading_path': ['10 报警信息', '10.1 报警'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322001《新松机器人通用操作手册》(B-5)》（V5.0 / B-5），操作“报警”时，涉及“下面列出的报警发生时”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['下面列出的报警发生时，伺服驱动器会掉电，不允许直接启动', '如果想伺服上电，必须保证系统没有报警，并按取消键取消报警', '报警会列入到错误信息中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'line_start': 16985, 'line_end': 16985, 'excerpt': '下面列出的报警发生时，伺服驱动器会掉电，不允许直接启动。如果想伺服上电，必须保证系统没有报警，并按取消键取消报警。报警会列入到错误信息中。', 'sha256': '62dccc1a6083cb35c2e09d2dd63bb151056a310f8b73cfc55fd8ebd0cecc7630'}]}, {'id': 'sample_general_operation_ea92c45f1d_01', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.5', 'section_title': '6.5 实时显示', 'heading_path': ['6 状态显示', '6.5 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“实时显示”时，“世界位置的T6阵”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 世界位置的T6阵 ：在世界坐标系下，机器人位置的T6阵，N、O、A为矩阵，P代表位置，单位是m'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9123, 'line_end': 9123, 'excerpt': '1. 世界位置的T6阵 ：在世界坐标系下，机器人位置的T6阵，N、O、A为矩阵，P代表位置，单位是m。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_02', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.1.2.1', 'section_title': '2.1.2.1 8位组I\\O', 'heading_path': ['2 机器人各项设定', '2.1 I/O', '2.1.2 组I/O', '2.1.2.1 8位组I\\O'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），“8位组I\\O”中“指令不检测组I\\O输出范围超界，示教指令为OUT GOB[1]=25,25转换为二进制00011”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'GOB'], 'expected_facts': ['指令不检测组I\\O输出范围超界，示教指令为OUT GOB[1]=25,25转换为二进制00011001，后4位IO的状态正常输出为1001，前4位不做任何处理'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 2023, 'line_end': 2025, 'excerpt': '<tr>\n<td colspan="9"><p>指令不检测组I&#92;O输出范围超界，示教指令为OUT GOB&#91;1&#93;=25,25转换为二进制00011001，后4位IO的状态正常输出为1001，前4位不做任何处理。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_03', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.4.2', 'section_title': '1.4.2 示教器窗口', 'heading_path': ['1 概述', '1.4 控制器', '1.4.2 示教器窗口'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“示教器窗口”时，“对机器人的作业、参数等数据进行备份或恢复。”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['对机器人的作业、参数等数据进行备份或恢复'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 1248, 'line_end': 1251, 'excerpt': '<tr>\n<td><p><img src="img/08c5f325424c83484354762f52cb7c198da8d7c84af7c4eb3e53a05e3c87d500.png"/></p></td>\n<td><p>对机器人的作业、参数等数据进行备份或恢复。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_04', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '7.4', 'section_title': '7.4 U盘卸载', 'heading_path': ['7 备份、恢复资料', '7.4 U盘卸载'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），在“U盘卸载”中，应如何理解或操作“点击主菜单【备份恢复】-【卸载USB】进入界面”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击主菜单【备份恢复】-【卸载USB】进入界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9337, 'line_end': 9337, 'excerpt': '点击主菜单【备份恢复】-【卸载USB】进入界面。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_05', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.2', 'section_title': '6.2 作业界面', 'heading_path': ['6 状态显示', '6.2 作业界面'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），“作业界面”中“执行中的程序指令信息会显示在作业界面上”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['执行中的程序指令信息会显示在作业界面上'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 8999, 'line_end': 8999, 'excerpt': '执行中的程序指令信息会显示在作业界面上。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_06', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.3.1', 'section_title': '4.3.1 新建作业', 'heading_path': ['4 程序的示教', '4.3 示教程序', '4.3.1 新建作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“新建作业”时，“参数1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 参数1 选择后台作业'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 8141, 'line_end': 8145, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>参数1</p></td>\n<td><p>选择后台作业。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_07', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '9.1.5', 'section_title': '9.1.5 控制类', 'heading_path': ['9 附录', '9.1 基础指令表', '9.1.5 控制类'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），在“控制类”中，应如何理解或操作“IF I[5] < 14 DELAY T = 1 ELSEIF I[5] < 10 DELAY”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['举例 IF I[5] < 14 DELAY T = 1 ELSEIF I[5] < 10 DELAY T = 5 ENDIF'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 12112, 'line_end': 12115, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td colspan="2"><p>IF I&#91;5&#93; &lt; 14</p><p>DELAY T = 1</p><p>ELSEIF I&#91;5&#93; &lt; 10</p><p>DELAY T = 5</p><p>ENDIF</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_08', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '7.1', 'section_title': '7.1 参数备份恢复', 'heading_path': ['7 备份、恢复资料', '7.1 参数备份恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），“参数备份恢复”中“注意： 请使用新松公司出厂提供的U盘，以免存储过程中发生数据丢失，同时避免对板卡的伤害。U盘必须”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FAT32'], 'expected_facts': ['注意： 请使用新松公司出厂提供的U盘，以免存储过程中发生数据丢失，同时避免对板卡的伤害', 'U盘必须为FAT32格式', '在备份和恢复时会提示备份是否成功，如没插U盘或搜索到U盘，会提示系统处于卸载状态'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9261, 'line_end': 9263, 'excerpt': '<tr>\n<td><p><strong>注意：</strong></p><p><strong>请使用新松公司出厂提供的U盘，以免存储过程中发生数据丢失，同时避免对板卡的伤害。U盘必须为FAT32格式。在备份和恢复时会提示备份是否成功，如没插U盘或搜索到U盘，会提示系统处于卸载状态。</strong></p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_09', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.8', 'section_title': '3.8 控制类', 'heading_path': ['3 程序的说明', '3.8 控制类'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），操作“控制类”时，涉及“DEFAULT”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['DEFAULT 说明 与SWITCH ... CASE...在一起使用，当CASE条件没有满足，则执行DEFAULT'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 7309, 'line_end': 7313, 'excerpt': '<tr>\n<td rowspan="2"><p>DEFAULT</p></td>\n<td><p>说明</p></td>\n<td colspan="2"><p>与SWITCH &#46;&#46;&#46; CASE&#46;&#46;&#46;在一起使用，当CASE条件没有满足，则执行DEFAULT。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_10', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.4.1', 'section_title': '1.4.1 示教器硬件', 'heading_path': ['1 概述', '1.4 控制器', '1.4.1 示教器硬件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），“示教器硬件”中“确认键（完成参数修改需要与确认键配合使用，作业指令参数修改过后点击确认跳至下一可修改参数，如果当”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['确认键（完成参数修改需要与确认键配合使用，作业指令参数修改过后点击确认跳至下一可修改参数，如果当前指令无可修改参数，跳至下一行指令编号处）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 1077, 'line_end': 1080, 'excerpt': '<tr>\n<td><p><img src="img/e2f8d2224d26d3c2a8f6a9832045d9921cce5670dea161176666f0eb2bc5855c.png"/></p></td>\n<td><p>确认键（完成参数修改需要与确认键配合使用，作业指令参数修改过后点击确认跳至下一可修改参数，如果当前指令无可修改参数，跳至下一行指令编号处） </p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_11', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.4.2', 'section_title': '1.4.2 示教器窗口', 'heading_path': ['1 概述', '1.4 控制器', '1.4.2 示教器窗口'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），操作“示教器窗口”时，涉及“进入用户配置界面，对各项条件进行配置，包括示教、原点、外部设备通讯、干涉区、中断操作等功能的配置”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['进入用户配置界面，对各项条件进行配置，包括示教、原点、外部设备通讯、干涉区、中断操作等功能的配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 1228, 'line_end': 1231, 'excerpt': '<tr>\n<td><p><img src="img/fe28ead0e845facc3ae88a659f237f519b580ade45425d5e0e9297a43a232b4f.png"/></p></td>\n<td><p>进入用户配置界面，对各项条件进行配置，包括示教、原点、外部设备通讯、干涉区、中断操作等功能的配置。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_12', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '7.1', 'section_title': '7.1 参数备份恢复', 'heading_path': ['7 备份、恢复资料', '7.1 参数备份恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“参数备份恢复”时，“对参数进行备份和恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 对参数进行备份和恢复，备份和恢复完成后，会有恢复成功、需要重启机器人的提示', '这时，对机器人进行重启即可'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9267, 'line_end': 9267, 'excerpt': '2. 对参数进行备份和恢复，备份和恢复完成后，会有恢复成功、需要重启机器人的提示。这时，对机器人进行重启即可。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_13', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.4.1', 'section_title': '1.4.1 示教器硬件', 'heading_path': ['1 概述', '1.4 控制器', '1.4.1 示教器硬件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），在“示教器硬件”中，应如何理解或操作“仅在作业列表页面有效，删除光标所在行的作业。”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['仅在作业列表页面有效，删除光标所在行的作业'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 1069, 'line_end': 1072, 'excerpt': '<tr>\n<td><p><img src="img/648639c9ae365a782dd1e3998e306882ed0fc391ec0f53eaa564d48c0f432caa.png"/></p></td>\n<td><p>仅在作业列表页面有效，删除光标所在行的作业。</p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_14', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.10', 'section_title': '2.10 软重启', 'heading_path': ['2 机器人各项设定', '2.10 软重启'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），在“软重启”中，应如何理解或操作“重启功能使机器人控制器重启”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['重启功能使机器人控制器重启，又名软启动，系统中的伺服、IO模块等不会重启'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 3358, 'line_end': 3358, 'excerpt': '重启功能使机器人控制器重启，又名软启动，系统中的伺服、IO模块等不会重启。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_15', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.5.2', 'section_title': '2.5.2 空间干涉属性', 'heading_path': ['2 机器人各项设定', '2.5 干涉区', '2.5.2 空间干涉属性'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“空间干涉属性”时，“输入各项参数后”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 输入各项参数后，点击保存'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 3023, 'line_end': 3023, 'excerpt': '1. 输入各项参数后，点击保存。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_16', 'category': 'general_operation', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '10.3', 'section_title': '10.3 提示信息', 'heading_path': ['10 报警信息', '10.3 提示信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），在“提示信息”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 提示信息 报警原因 解决措施'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 17877, 'line_end': 17882, 'excerpt': '<tr>\n<th><p><strong>报警码</strong></p></th>\n<th><p><strong>提示信息</strong></p></th>\n<th><p><strong>报警原因</strong></p></th>\n<th><p><strong>解决措施</strong></p></th>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_17', 'category': 'general_operation', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.3.2', 'section_title': '5.3.2 单步测试程序', 'heading_path': ['5 程序的运行', '5.3 对程序进行测试', '5.3.2 单步测试程序'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），配置“单步测试程序”时，“警告： 运行机器人前，需要确认机器人运动范围内，没有人员以及不必要的设备，以免发生意想不到的问题”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['警告： 运行机器人前，需要确认机器人运动范围内，没有人员以及不必要的设备，以免发生意想不到的问题，造成人员伤亡或设备损坏'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 8791, 'line_end': 8793, 'excerpt': '<tr>\n<td><p><strong>警告：</strong></p><p><strong>运行机器人前，需要确认机器人运动范围内，没有人员以及不必要的设备，以免发生意想不到的问题，造成人员伤亡或设备损坏。</strong></p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_18', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.7', 'section_title': '6.7 报警日志', 'heading_path': ['6 状态显示', '6.7 报警日志'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），操作“报警日志”时，涉及“注意：清空按钮点击后，会使报警复位，同时报警只记录在历史记录中。”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：清空按钮点击后，会使报警复位，同时报警只记录在历史记录中'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9221, 'line_end': 9223, 'excerpt': '<tr>\n<td><p><strong>注意：清空按钮点击后，会使报警复位，同时报警只记录在历史记录中。</strong></p></td>\n</tr>', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_19', 'category': 'general_operation', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '8.1', 'section_title': '8.1 通讯设备的设置', 'heading_path': ['8 应用与操作', '8.1 通讯设备的设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），“通讯设备的设置”中“机器人的通讯配置方式及接口有以下几种”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['机器人的通讯配置方式及接口有以下几种:'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 9353, 'line_end': 9353, 'excerpt': '机器人的通讯配置方式及接口有以下几种:', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}, {'id': 'sample_general_operation_ea92c45f1d_20', 'category': 'general_operation', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.2.2', 'section_title': '4.2.2 3种运行模式', 'heading_path': ['4 程序的示教', '4.2 开机并手动控制', '4.2.2 3种运行模式'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），操作“3种运行模式”时，涉及“在T1或者T2模式”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['在T1或者T2模式，可以在安全门开启的情况下手动操作机器人', '必须在DEADMAN被按下的情况下，才可以运动机器人，DEADMAN没有按下时，机器人不能运动'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'line_start': 7816, 'line_end': 7816, 'excerpt': '在T1或者T2模式，可以在安全门开启的情况下手动操作机器人。必须在DEADMAN被按下的情况下，才可以运动机器人，DEADMAN没有按下时，机器人不能运动。', 'sha256': 'cf5383747a666af1d7b046fd139f883e9af1b84027c33a87496172d0cb26a948'}]}]
register_sample_bank('general_operation', GENERAL_OPERATION_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md', 'seed': 3101165652757987459, 'population': 3536, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(719, 719), (7752, 7752), (8295, 8295), (9790, 9790), (10163, 10163), (9717, 9717), (13107, 13110), (4645, 4645), (9878, 9878), (5247, 5250), (3912, 3912), (11996, 12000), (2179, 2179), (8679, 8679), (1370, 1370), (3424, 3424), (8989, 8992), (12098, 12101), (9122, 9122), (9637, 9637)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md', 'seed': 11337335553799648717, 'population': 4943, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(10924, 10924), (5399, 5403), (9797, 9797), (11482, 11482), (10451, 10451), (5536, 5536), (9069, 9069), (13300, 13304), (12557, 12559), (1938, 1943), (10154, 10154), (1106, 1109), (10960, 10960), (11124, 11124), (10982, 10984), (2187, 2187), (9877, 9877), (4382, 4385), (10578, 10578), (16985, 16985)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md', 'seed': 437819207427042140, 'population': 2776, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(9123, 9123), (2023, 2025), (1248, 1251), (9337, 9337), (8999, 8999), (8141, 8145), (12112, 12115), (9261, 9263), (7309, 7313), (1077, 1080), (1228, 1231), (9267, 9267), (1069, 1072), (3358, 3358), (3023, 3023), (17877, 17882), (8791, 8793), (9221, 9223), (9353, 9353), (7816, 7816)]}])


In [ ]:
# | notest
GENERAL_OPERATION_SAMPLE_IDS = None
general_operation_sample_cases = select_cases(GENERAL_OPERATION_SAMPLE_CASES, GENERAL_OPERATION_SAMPLE_IDS)
display(pd.DataFrame(general_operation_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in general_operation_sample_cases])
general_operation_sample_metrics = await evaluation.run(general_operation_sample_cases, concurrency=EVALUATION_CONCURRENCY)
general_operation_sample_results = {case["id"]: evaluation.load(case) for case in general_operation_sample_cases}
display(general_operation_sample_metrics)
# Inspect a complete result: evaluation.inspect(general_operation_sample_cases[0]["id"])
evaluation.show_answers(general_operation_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_general_operation_e8c48bad9b_01,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,procedure,prose,False,True,8,False,0,0,6,False,False,False,None,None,5745,0,True,unsupported,,,abstained
sample_general_operation_e8c48bad9b_02,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,23890,0,True,included,,,not reviewed
sample_general_operation_e8c48bad9b_03,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,instruction,prose,False,True,8,False,0,0,6,False,False,False,False,False,20034,0,True,unsupported,,,abstained
sample_general_operation_e8c48bad9b_04,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,parameter,prose,False,True,8,False,0,0,7,False,False,False,None,None,10819,0,True,unsupported,,,not reviewed
sample_general_operation_e8c48bad9b_05,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,prerequisite,prose,False,True,8,False,0,0,2,False,False,False,None,None,17852,0,True,included,,,not reviewed
sample_general_operation_e8c48bad9b_06,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,19436,0,True,included,,,not reviewed
sample_general_operation_e8c48bad9b_07,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,23332,2,True,included,,,not reviewed
sample_general_operation_e8c48bad9b_08,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,instruction,prose,False,True,8,False,0,0,2,False,False,False,None,None,17492,0,True,unsupported,,,not reviewed
sample_general_operation_e8c48bad9b_09,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,procedure,prose,False,True,8,False,0,0,3,False,False,False,None,None,15908,0,True,unsupported,,,not reviewed
sample_general_operation_e8c48bad9b_10,general_operation,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,20306,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed


### sample_general_operation_e8c48bad9b_01

根据《SX322001《新松机器人通用操作手册》-V1.4》（V3.0 / V1.4），在“维修、保养时的安全”中，应如何理解或操作“更换机器人部件时”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 新松机器人通用操作手册（SX322001）的修订记录表头行，定义了四列关系：文件版本、修订日期、修订人及主要变化。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 18239–18244; corpus 5.0, revision V5.0 / C-1; object `f870ade72f05b28a8bfb656d3a485c340947f626f27b51b7c0a1d90c65e9a33e`.

- 新松机器人通用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 13–15; corpus 5.0, revision V5.0 / C-1; object `5ec630f8aa237d2cc4bd8658f1f8ee2e0898155ee36a5a3b61768bcd71f3ca37`.

- 新松机器人通用操作手册，编号 SX322001
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 483–486; corpus 4.0, revision V5.0 / A-4; object `0fa0fe1f9248cd8de37872f90e54ee8f642cc7bf54636b2e8e456a5202b63209`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 新松机器人通用操作手册（文档编号：SX322001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 480–483; corpus 5.0, revision V5.0 / C-0; object `7723af1da440d3260a499cef189d99aef7de63c3cbfdace5cad603c336881343`.

- 新松机器人安全手册 | 编号：SX022001
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 464–467; corpus 5.0, revision V5.0 / C-0; object `31c5015c89320d02afaae82d255823e3a93029caf42d936d3a6a803504cfad03`.

#### Answer

The retrieved evidence (E1–E8) only provides document identification information (title, document number SX322001, revision history table, and cross-references from other manuals). None of the evidence contains the actual content of the '维修、保养时的安全' (Safety during maintenance and servicing) section, nor does it include any text describing the procedure, purpose, or result of '更换机器人部件时' (when replacing robot components). Additionally, the question references revision V1.4 (V3.0 / V1.4), while the available evidence is from revision V5.0 / C-1. The specific safety-maintenance operational content requested is entirely absent from the supplied evidence.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: mixed.

The answer correctly identifies that the supplied evidence (E1-E8) consists only of document metadata, revision tables, and cross-references from other manuals. It accurately notes that the specific content regarding 'Safety during maintenance and servicing' and the procedure for 'replacing robot components' is missing from the provided evidence. It also correctly points out the revision mismatch (Question asks for V1.4, Evidence is V5.0). Therefore, the refusal to answer the specific operational question is appropriate and supported by the lack of relevant evidence.

...

#### Answer

The question asks about the '通讯设备的设置' section in SX322001 (C-1), specifically the statement '机器人的通讯配置方式及接口有以下几种'. The retrieved evidence from C-1 (E4) only covers the IN instruction for I/O variable assignment, not communication device configuration. The menu structure for '外部通讯设置' (DevNet, 串口, EtherCAT, Modbus, etc.) appears only in E3, which is from the B-5 revision, not C-1. No C-1 evidence describes the communication configuration methods, interfaces, or their syntax/parameters. Therefore a supported contextual example cannot be constructed from the supplied evidence.

### sample_general_operation_ea92c45f1d_20

根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），操作“3种运行模式”时，涉及“在T1或者T2模式”的前提条件或注意事项是什么？

#### Retrieved content summary

- 该行描述机器人状态显示图标，包含T1/T2模式、执行模式和远程模式三种状态；特别警告T2模式下机器人可达全速，存在安全风险，非必要请勿使用以保障示教工程师安全。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 1336–1343; corpus 4.0, revision V5.0 / B-5; object `cc00666eedf5828bc798b78db3b59dd856d4019495cde12226b405e86bce0e3e`.

- 该行描述机器人当前执行模式的状态显示图标，包含T1/T2模式、执行模式和远程模式；并警告T2模式下机器人可达全速，非必要时应避免使用以保障示教工程师安全。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 1151–1158; corpus 5.0, revision V5.0 / C-1; object `0702887af883f15c3a1a602187074d1256f71e11c880d893fc88aebd4ea13016`.

- 模式选择开关用于在示教模式（含T1/T2，默认T1）、自动运行模式和远程模式间切换。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 1056–1059; corpus 4.0, revision V5.0 / B-5; object `3f43b1a9e56231b09fab3c5d6f20e1d98ceead1059038e49e675f0a9f6046e86`.

- 该表格行展示了新松机器人操作手柄上**模式切换按键**的功能说明：通过操作图中所示的人机交互控制界面（含中心模式切换/启动键），可实现机器人**示教、执行、远程**三种运行模式的切换。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 1081–1084; corpus 5.0, revision V5.0 / C-1; object `350792a0c4726798159b404d2d06bf1a23d6eac35b19c9b72095e6bafd3fdaa4`.

- 该条目展示了机器人T1/T2模式切换的操作界面或图示，并附有一项重要安全警告：T2模式下机器人可运行至全速，存在较高人身安全风险，因此除非必要，否则应严禁使用T2模式，以保障示教工程师的安全。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 1113–1120; corpus 4.0, revision V5.0 / B-5; object `3c6a84a489e1db83690ebc781ac22e235b80e816f15fbf979606402cd0a38835`.

- 该按钮用于切换机器人模式与循环方式：与 SHIFT 键组合按可在 T1/T2 模式间切换（警告：T2 可达全速，非必要勿用，以防示教安全风险）；单独按下可循环切换运行循环方式：单步 → 单循环 → 自动（提示：手动或自动运行前请确认当前循环方式）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 985–996; corpus 5.0, revision V5.0 / C-1; object `941a08f41ca59186f27acb37e0062d3eb86584e251eeee43e34ab05a2b007c69`.

- **摘要：**
同时按下正向运动和DEADMAN时，机器人按程序执行，需确认联锁状态（开启时执行逻辑及运动指令，关闭时仅执行运动指令）、循环模式及运行速度。⚠️ **警告**：T2模式下机器人可达全速，非紧急情况请勿使用，以确保示教工程师人身安全。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 1165–1172; corpus 4.0, revision V5.0 / B-5; object `9654cf51975c3701dc941c75334d98bba218b4748ce82214f90c53e5c97ec60f`.

- 模式选择开关用于在示教模式（含T1和T2，默认为T1）、自动运行模式和远程模式之间进行切换。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 898–901; corpus 5.0, revision V5.0 / C-1; object `c5a3bf496ca9955a04bf61f1d52f1317855f6ab5617dc3ef05b0fec4db8985a1`.

#### Answer

根据《SX322001《新松机器人通用操作手册》(C-1)》（V5.0 / C-1），3种运行模式为：示教模式（含T1和T2）、执行模式、远程模式。示教模式默认为T1模式。涉及T1或T2模式的核心注意事项（警告）为：T2模式下，机器人的运动速度可以到达全速，如果不是不得不使用T2模式，请不要使用T2模式，以保证示教工程师的人身安全。此外，在手动或自动运行前需确认当前模式及循环方式。

### Example — general_operation

操作员需要在新松机器人示教器上切换3种运行模式（示教/执行/远程），并在示教模式下选择T1或T2子模式进行手动运动或程序正反向运动。

**Environment and prerequisites:** 前提条件：机器人已上电，DEADMAN安全开关处于中间有效位置；变量：当前模式（默认T1）、速度倍率、循环方式（单步/单循环/自动）；用户占位符：[目标模式]、[速度倍率值]。

**Expected result:** 操作员成功切换至目标运行模式；若选择T2模式，系统提示全速警告，操作员确认必要性后方可继续；运行前已确认循环方式与速度倍率，确保示教工程师人身安全。

**Manual:** SX322001《新松机器人通用操作手册》(C-1) [E2] [E4] [E6] [E8]

```text
模式选择开关：可以在示教模式，自动运行模式和远程模式间切换；示教模式包括T1和T2模式，默认为T1模式
警告：T2模式下，机器人的运动速度可以到达全速，如果不是不得不使用T2模式，请不要使用T2模式，以保证示教工程师的人身安全
请在手动或自动运行前，确认当前模式及循环方式
```

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: partial; revision fidelity: mixed.

The answer fails to include the two expected facts: (1) that in T1 or T2 mode, the robot can be manually operated with the safety door open, and (2) that the robot can only move when the DEADMAN switch is pressed (and cannot move when it is not). While the answer correctly cites the warning about T2 mode speed from the C-1 manual (E2, E4, E6, E8), it omits the critical safety prerequisites regarding the safety door and the DEADMAN switch. The provided evidence [E1] and [E3] (from B-5) and [E8] (from C-1) contain the DEADMAN requirement, but the specific fact about the safety door is only present in the reference text, not explicitly in the supplied evidence snippets for C-1. However, the answer does not mention the safety door fact at all, which is a required expected fact. Furthermore, the answer includes a code example block which is not requested and not supported by the evidence as a procedural requirement. The answer is incomplete because it misses the core safety conditions (safety door and DEADMAN) that are the primary focus of the question's expected facts.

## Vision — sampled manual revisions

60 cases from 3 CSV files. Set `VISION_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
VISION_SAMPLE_CASES = [{'id': 'sample_vision_af4fcdcff7_01', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.2', 'section_title': '4.2 视觉系统初始化', 'heading_path': ['4 通信协议', '4.2 视觉系统初始化'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“视觉系统初始化”中“0x51”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['0x51：清除当前测量值成功应答，为0x21指令的应答指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 622, 'line_end': 622, 'excerpt': '0x51：清除当前测量值成功应答，为0x21指令的应答指令；', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_02', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“作业参数”中，应如何理解或操作“拍照错误码存储位置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['8. 拍照错误码存储位置： 拍照错误码放在该存储序号的整型变量中', '错误码类型如下表'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 166, 'line_end': 166, 'excerpt': '8. 拍照错误码存储位置： 拍照错误码放在该存储序号的整型变量中。错误码类型如下表。', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_03', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“报警信息”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 子码 出现问题 解决方式'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 803, 'line_end': 808, 'excerpt': '<tr>\n<td><p>报警码</p></td>\n<td><p>子码</p></td>\n<td><p>出现问题</p></td>\n<td><p>解决方式</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_04', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.5', 'section_title': '2.5 计算偏移值', 'heading_path': ['2 指令集', '2.5 计算偏移值'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“计算偏移值”中，应如何理解或操作“计算偏移值，并将偏移量存储值对应的P变量中”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['功能 计算偏移值，并将偏移量存储值对应的P变量中'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 489, 'line_end': 492, 'excerpt': '<tr>\n<td><p>功能</p></td>\n<td colspan="2"><p>计算偏移值，并将偏移量存储值对应的P变量中</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_05', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3', 'section_title': '3 视觉标定', 'heading_path': ['3 视觉标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“视觉标定”中，应如何理解或操作“针对视觉标定”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 502, 'line_end': 502, 'excerpt': '针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定。', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_06', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.5', 'section_title': '2.5 计算偏移值', 'heading_path': ['2 指令集', '2.5 计算偏移值'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），配置“计算偏移值”时，“参数1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 参数1 运动的速度值'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 493, 'line_end': 497, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>参数1</p></td>\n<td><p>运动的速度值</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_07', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“作业拍照”中“VS\\_TG\\_P T=0 NO=0 发送拍照指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['VS\\_TG\\_P T=0 NO=0 发送拍照指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 235, 'line_end': 235, 'excerpt': 'VS\\_TG\\_P T=0 NO=0 发送拍照指令', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_08', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.3', 'section_title': '2.3 相机发送拍照指令', 'heading_path': ['2 指令集', '2.3 相机发送拍照指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“相机发送拍照指令”中“参数1”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['说明 参数1 动作标志', '0视觉准备测量，1触发视觉测量（范围：0-255）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 431, 'line_end': 435, 'excerpt': '<tr>\n<td rowspan="4"><p> 说明</p></td>\n<td><p>参数1</p></td>\n<td><p>动作标志； 0视觉准备测量，1触发视觉测量（范围：0&#45;255）   </p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_09', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.5', 'section_title': '4.5 获取测量结果', 'heading_path': ['4 通信协议', '4.5 获取测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“获取测量结果”中，应如何理解或操作“送信（机器人→视觉） 0X23”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 送信（机器人→视觉） 0X23'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 680, 'line_end': 680, 'excerpt': '1. 送信（机器人→视觉） 0X23', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_10', 'category': 'vision', 'kind': 'alarm', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“报警信息”列出的报警7565表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7565'], 'expected_facts': ['7565 0 数据长度错误 视觉与机器人传输数据长度错误'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 875, 'line_end': 880, 'excerpt': '<tr>\n<td><p>7565</p></td>\n<td><p>0</p></td>\n<td><p>数据长度错误</p></td>\n<td><p>视觉与机器人传输数据长度错误。</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_11', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），配置“作业参数”时，“0x09”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['0x09 直径测量超出范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 211, 'line_end': 214, 'excerpt': '<tr>\n<td><p>0x09</p></td>\n<td><p>直径测量超出范围</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_12', 'category': 'vision', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.7.4', 'section_title': '4.7.4 特殊', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.4 特殊'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），操作“特殊”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：消息头长度超过255字符，案例如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 792, 'line_end': 792, 'excerpt': '注意：消息头长度超过255字符，案例如下：', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_13', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.7.3', 'section_title': '4.7.3 测量触发和获取测量结果流程数据示例', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.3 测量触发和获取测量结果流程数据示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“测量触发和获取测量结果流程数据示例”中“执行获取测量结果指令(VSGPI\\_P PO#2 NO=01)”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSGPI', 'PO#2'], 'expected_facts': ['执行获取测量结果指令(VSGPI\\_P PO#2 NO=01)，触发测量检测结果返回正常，才能发送获取测量结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 764, 'line_end': 764, 'excerpt': '执行获取测量结果指令(VSGPI\\_P PO#2 NO=01)，触发测量检测结果返回正常，才能发送获取测量结果；', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_14', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“作业参数”中，应如何理解或操作“0x04”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['0x04 左右或者上下特征高度差不符合要求'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 191, 'line_end': 194, 'excerpt': '<tr>\n<td><p>0x04</p></td>\n<td><p>左右或者上下特征高度差不符合要求</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_15', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.4', 'section_title': '2.4 获取相机测量结果', 'heading_path': ['2 指令集', '2.4 获取相机测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），配置“获取相机测量结果”时，“参数4”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['参数4 区域号（范围：0-65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 475, 'line_end': 478, 'excerpt': '<tr>\n<td><p>参数4</p></td>\n<td><p>区域号（范围：0&#45;65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_16', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“作业拍照”中“VSTG\\_P T=1 NO=0 发送拍照指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSTG'], 'expected_facts': ['VSTG\\_P T=1 NO=0 发送拍照指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 289, 'line_end': 289, 'excerpt': 'VSTG\\_P T=1 NO=0 发送拍照指令', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_17', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.1', 'section_title': '2.1 相机拍照指令', 'heading_path': ['2 指令集', '2.1 相机拍照指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“相机拍照指令”中“给相机发送拍照指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['功能 给相机发送拍照指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 381, 'line_end': 384, 'excerpt': '<tr>\n<td><p>功能</p></td>\n<td colspan="2"><p>给相机发送拍照指令</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_18', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.7.1', 'section_title': '4.7.1 初始化流程数据发送和接收示例', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.1 初始化流程数据发送和接收示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），在“初始化流程数据发送和接收示例”中，应如何理解或操作“00 00 00 00 00 00 00 00 01 20 01 00 00 00 00 00”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['00 00 00 00 00 00 00 00 01 20 01 00 00 00 00 00 00 同步帧，间隔1S发送请求，若1S内未收到，再次进行初始化请求;'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 736, 'line_end': 736, 'excerpt': '00 00 00 00 00 00 00 00 01 20 01 00 00 00 00 00 00 同步帧，间隔1S发送请求，若1S内未收到，再次进行初始化请求;', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_19', 'category': 'vision', 'kind': 'alarm', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“报警信息”列出的报警7558表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7558'], 'expected_facts': ['7558 0 相机初始化失败 视觉方面配置错误'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 839, 'line_end': 844, 'excerpt': '<tr>\n<td><p>7558</p></td>\n<td><p>0</p></td>\n<td><p>相机初始化失败</p></td>\n<td><p>视觉方面配置错误。</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_af4fcdcff7_20', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），配置“作业参数”时，“0x08”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['0x08 Z方向超出范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'line_start': 207, 'line_end': 210, 'excerpt': '<tr>\n<td><p>0x08</p></td>\n<td><p>Z方向超出范围</p></td>\n</tr>', 'sha256': 'a31106456acf5244c8d8842bd1b71fc2be0079a2f9e1a3f7ea688f638f0f0a95'}]}, {'id': 'sample_vision_a28ebaa719_01', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“作业参数”中“trans偏移变量存储位置（校正方式14”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. trans偏移变量存储位置（校正方式14,15）：vs\\_movl指令将相机检测的旋转偏移转换到机器人坐标系下，存放在该存储位置对应的位置变量中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 182, 'line_end': 182, 'excerpt': '2. trans偏移变量存储位置（校正方式14,15）：vs\\_movl指令将相机检测的旋转偏移转换到机器人坐标系下，存放在该存储位置对应的位置变量中。', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_02', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.4', 'section_title': '2.4 获取相机测量结果', 'heading_path': ['2 指令集', '2.4 获取相机测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），配置“获取相机测量结果”时，“参数4”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['参数4 区域号（范围：0-65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 497, 'line_end': 500, 'excerpt': '<tr>\n<td><p>参数4</p></td>\n<td><p>区域号（范围：0&#45;65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_03', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），配置“作业参数”时，“0x07”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['0x07 XY方向超出范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 225, 'line_end': 228, 'excerpt': '<tr>\n<td><p>0x07</p></td>\n<td><p>XY方向超出范围</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_04', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '4.5', 'section_title': '4.5 获取测量结果', 'heading_path': ['4 通信协议', '4.5 获取测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“获取测量结果”中“0x53”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['0x53：测量结果应答，为0x23指令的应答指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 712, 'line_end': 712, 'excerpt': '0x53：测量结果应答，为0x23指令的应答指令；', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_05', 'category': 'vision', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“报警信息”列出的报警7565表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7565'], 'expected_facts': ['7565 0 数据长度错误 视觉与机器人传输数据长度错误', '检查相机发给机器人的数据，长度是否满足协议要求'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 994, 'line_end': 999, 'excerpt': '<tr>\n<td rowspan="4"><p>7565</p></td>\n<td><p>0</p></td>\n<td rowspan="4"><p>数据长度错误</p></td>\n<td rowspan="4"><p>视觉与机器人传输数据长度错误。</p><p>检查相机发给机器人的数据，长度是否满足协议要求</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_06', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '3', 'section_title': '3 视觉标定', 'heading_path': ['3 视觉标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“视觉标定”中，应如何理解或操作“针对视觉标定”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 522, 'line_end': 522, 'excerpt': '针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定。', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_07', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.3', 'section_title': '2.3 相机发送拍照指令', 'heading_path': ['2 指令集', '2.3 相机发送拍照指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“相机发送拍照指令”中“参数1”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['说明 参数1 动作标志', '0视觉准备测量，1触发视觉测量（范围：0-255）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 455, 'line_end': 459, 'excerpt': '<tr>\n<td rowspan="4"><p>说明</p></td>\n<td><p>参数1</p></td>\n<td><p>动作标志； 0视觉准备测量，1触发视觉测量（范围：0&#45;255）   </p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_08', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '4.7.4', 'section_title': '4.7.4 特殊', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.4 特殊'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“特殊”中，应如何理解或操作“浮点型转换成十六进制后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['浮点型转换成十六进制后，发送的高低位顺序需调换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 806, 'line_end': 806, 'excerpt': '浮点型转换成十六进制后，发送的高低位顺序需调换；', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_09', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“概要”中，应如何理解或操作“该手册说明了视觉应用软件包的使用方法”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['该手册说明了视觉应用软件包的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 150, 'line_end': 150, 'excerpt': '该手册说明了视觉应用软件包的使用方法。', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_10', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“报警信息”中，应如何理解或操作“x方向位置超限”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1 x方向位置超限 可扩大x轴位置限位'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 880, 'line_end': 884, 'excerpt': '<tr>\n<td><p>1</p></td>\n<td><p>x方向位置超限</p></td>\n<td><p>可扩大x轴位置限位</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_11', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），配置“报警信息”时，“Ry旋转角度超限”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['8 Ry旋转角度超限 可扩大y轴旋转角度范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 915, 'line_end': 919, 'excerpt': '<tr>\n<td><p>8</p></td>\n<td><p>Ry旋转角度超限</p></td>\n<td><p>可扩大y轴旋转角度范围</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_12', 'category': 'vision', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '4.7.4', 'section_title': '4.7.4 特殊', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.4 特殊'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），操作“特殊”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：消息头长度超过255字符，案例如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 812, 'line_end': 812, 'excerpt': '注意：消息头长度超过255字符，案例如下：', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_13', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“作业拍照”中，应如何理解或操作“DELAY T=0.2 等待数据返回延时”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['DELAY T=0.2 等待数据返回延时'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 265, 'line_end': 265, 'excerpt': 'DELAY T=0.2 等待数据返回延时', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_14', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“作业拍照”中“VSTG\\_P T=1 NO=0 发送拍照指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSTG'], 'expected_facts': ['VSTG\\_P T=1 NO=0 发送拍照指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 317, 'line_end': 317, 'excerpt': 'VSTG\\_P T=1 NO=0 发送拍照指令', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_15', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.4', 'section_title': '2.4 获取相机测量结果', 'heading_path': ['2 指令集', '2.4 获取相机测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），配置“获取相机测量结果”时，“参数2”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['参数2 定位还是质量检测', '0是调用定位功能1调用打磨质量检测功能', '2是调用凸起高度计算功能（范围：0-255）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 489, 'line_end': 492, 'excerpt': '<tr>\n<td><p>参数2</p></td>\n<td><p>定位还是质量检测；0是调用定位功能1调用打磨质量检测功能；2是调用凸起高度计算功能（范围：0&#45;255）</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_16', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），配置“作业参数”时，“0x09”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['0x09 直径测量超出范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 233, 'line_end': 236, 'excerpt': '<tr>\n<td><p>0x09</p></td>\n<td><p>直径测量超出范围</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_17', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '4.3', 'section_title': '4.3 同步帧', 'heading_path': ['4 通信协议', '4.3 同步帧'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“同步帧”中“0x50”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['0x50，同步帧心跳应答，为0x20指令的应答'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 666, 'line_end': 666, 'excerpt': '0x50，同步帧心跳应答，为0x20指令的应答；', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_18', 'category': 'vision', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“报警信息”列出的报警7554表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7554'], 'expected_facts': ['7554 0 目标位置丢失 视觉未检测到目标', '重新进行拍照'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 934, 'line_end': 939, 'excerpt': '<tr>\n<td><p>7554</p></td>\n<td><p>0</p></td>\n<td><p>目标位置丢失</p></td>\n<td><p>视觉未检测到目标。重新进行拍照。</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_19', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.3', 'section_title': '2.3 相机发送拍照指令', 'heading_path': ['2 指令集', '2.3 相机发送拍照指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），“相机发送拍照指令”中“VSTG_N_P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSTG_N_P'], 'expected_facts': ['VSTG_N_P N=（参数1） T=（参数2） W=（参数3）A=（参数4）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 448, 'line_end': 450, 'excerpt': '<tr>\n<td colspan="3"><p>VSTG&#95;N&#95;P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）</p></td>\n</tr>', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_a28ebaa719_20', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '4.2', 'section_title': '4.2 视觉系统初始化', 'heading_path': ['4 通信协议', '4.2 视觉系统初始化'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': '', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V4.0 / A-1），在“视觉系统初始化”中，应如何理解或操作“送信（机器人→视觉） 0X21”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 送信（机器人→视觉） 0X21'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 628, 'line_end': 628, 'excerpt': '1. 送信（机器人→视觉） 0X21', 'sha256': '2ff500ca67d192106961db48caa8e9a8f17c3b9930388546e17e6c2050c10ca4'}]}, {'id': 'sample_vision_b61f315fb1_01', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），配置“作业参数”时，“0x00”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['值 説明 0x00 测量成功/系统状态正常,检测个数为0时，检测结果OK 0x01 图像采集失败 0x02 无目标激光条 0x03 激光特征初步提取失败 0x04 左右或者上下特征高度差不符合要求 0x05 激光特征精确提取失败 0x06 激光特征精确提取超出ROI区域 0x07 XY方向超出范围 0x08 Z方向超出范围 0x09 直径测量超出范围 0xFF 其他异常'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 137, 'line_end': 137, 'excerpt': '<table><tr><td><p><strong>值</strong></p></td><td><p><strong>説明</strong></p></td></tr><tr><td><p>0x00</p></td><td><p>测量成功/系统状态正常,检测个数为0时，检测结果OK</p></td></tr><tr><td><p>0x01</p></td><td><p>图像采集失败</p></td></tr><tr><td><p>0x02</p></td><td><p>无目标激光条</p></td></tr><tr><td><p>0x03</p></td><td><p>激光特征初步提取失败</p></td></tr><tr><td><p>0x04</p></td><td><p>左右或者上下特征高度差不符合要求</p></td></tr><tr><td><p>0x05</p></td><td><p>激光特征精确提取失败</p></td></tr><tr><td><p>0x06</p></td><td><p>激光特征精确提取超出ROI区域</p></td></tr><tr><td><p>0x07</p></td><td><p>XY方向超出范围</p></td></tr><tr><td><p>0x08</p></td><td><p>Z方向超出范围</p></td></tr><tr><td><p>0x09</p></td><td><p>直径测量超出范围</p></td></tr><tr><td><p>0xFF</p></td><td><p>其他异常</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_02', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '3', 'section_title': '3 视觉标定', 'heading_path': ['3 视觉标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“视觉标定”中，应如何理解或操作“针对视觉标定”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 321, 'line_end': 321, 'excerpt': '针对视觉标定，在相机端进行标定，按照相机厂商提供的方法标定。', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_03', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.4', 'section_title': '2.4 获取相机测量结果', 'heading_path': ['2 指令集', '2.4 获取相机测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），配置“获取相机测量结果”时，“VSGPI_N_P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['VSGPI_N_P N=（参数1） T=（参数2） W=（参数3）A=（参数4） 功能 获取相机测量结果 说明 参数1 动作标志', '0视觉准备测量，1触发视觉测量（范围：0-255） 参数2 定位还是质量检测', '0是调用定位功能1调用打磨质量检测功能'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 313, 'line_end': 313, 'excerpt': '<table><tr><td colspan="3"><p>VSGPI_N_P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>获取相机测量结果</p></td></tr><tr><td rowspan="4"><p>说明</p></td><td><p>参数1</p></td><td><p>动作标志； 0视觉准备测量，1触发视觉测量（范围：0-255）   </p></td></tr><tr><td><p>参数2</p></td><td><p>定位还是质量检测；0是调用定位功能1调用打磨质量检测功能；2是调用凸起高度计算功能（范围：0-255）</p></td></tr><tr><td><p>参数3</p></td><td><p>工件号（范围：0--65535）</p></td></tr><tr><td><p>参数4</p></td><td><p>区域号（范围：0-65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_04', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 视觉系统参数配置', 'heading_path': ['1 视觉应用参数配置', '1.1 视觉系统参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“视觉系统参数配置”中，应如何理解或操作“配置完成后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 配置完成后，点击【保存】'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 108, 'line_end': 108, 'excerpt': '4. 配置完成后，点击【保存】。', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_05', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.4', 'section_title': '4.4 视觉测量触发', 'heading_path': ['4 通信协议', '4.4 视觉测量触发'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“视觉测量触发”中，应如何理解或操作“0x24”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['0x24：触发视觉测量（切换场景并确认状态）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 385, 'line_end': 385, 'excerpt': '0x24：触发视觉测量（切换场景并确认状态）；', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_06', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“作业拍照”中“VS\\_TG\\_P T=0 NO=0 发送拍照指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['VS\\_TG\\_P T=0 NO=0 发送拍照指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 159, 'line_end': 159, 'excerpt': 'VS\\_TG\\_P T=0 NO=0 发送拍照指令', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_07', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.7.3', 'section_title': '4.7.3 测量触发和获取测量结果流程数据示例', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.3 测量触发和获取测量结果流程数据示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“测量触发和获取测量结果流程数据示例”中“机器人执行拍照指令（VSTG\\_N\\_P N=1 T=0 W= 1 A=0”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSTG'], 'expected_facts': ['机器人执行拍照指令（VSTG\\_N\\_P N=1 T=0 W= 1 A=0，其中N代表动作标志，T代表的是定位还是质量检测，W为工件号，A为区域号)'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 497, 'line_end': 497, 'excerpt': '机器人执行拍照指令（VSTG\\_N\\_P N=1 T=0 W= 1 A=0，其中N代表动作标志，T代表的是定位还是质量检测，W为工件号，A为区域号)；', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_08', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“概要”中，应如何理解或操作“该手册说明了视觉应用软件包的使用方法”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['该手册说明了视觉应用软件包的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 95, 'line_end': 95, 'excerpt': '该手册说明了视觉应用软件包的使用方法。', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_09', 'category': 'vision', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.7.4', 'section_title': '4.7.4 特殊', 'heading_path': ['4 通信协议', '4.7 数据发送流程说明', '4.7.4 特殊'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），操作“特殊”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：消息头长度超过255字符，案例如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 517, 'line_end': 517, 'excerpt': '注意：消息头长度超过255字符，案例如下：', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_10', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“报警信息”中，应如何理解或操作“提示码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['提示码 子码 出现问题 解决方式 7570 0 相机连接重新建立 1 7571 0 相机初始化完成 1 7572 0 标定数据已经发送 1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 531, 'line_end': 531, 'excerpt': '<table><tr><td><p>提示码</p></td><td><p>子码</p></td><td><p>出现问题</p></td><td><p>解决方式</p></td></tr><tr><td rowspan="2"><p>7570</p></td><td><p>0</p></td><td rowspan="2"><p>相机连接重新建立</p></td><td></td></tr><tr><td><p>1</p></td><td></td></tr><tr><td rowspan="2"><p>7571</p></td><td><p>0</p></td><td rowspan="2"><p>相机初始化完成</p></td><td></td></tr><tr><td><p>1</p></td><td></td></tr><tr><td rowspan="2"><p>7572</p></td><td><p>0</p></td><td rowspan="2"><p>标定数据已经发送</p></td><td></td></tr><tr><td><p>1</p></td><td></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_11', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 报警信息', 'heading_path': ['5 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“报警信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['TG_P', 'TCP'], 'expected_facts': ['报警码 子码 出现问题 解决方式 7550 0 TG_P:TCP连接断开 1.查看网线是否完好 2.ip连接是否正确 1 TG:TCP连接断开 2 GPI_P:TCP连接断开 3 GPI:TCP连接断开 4 TG_P:TCP连接断开 5 GPI:TCP发送失败 6 VG:TCP发送失败 7 8 9 7551 0 用户坐标系号错误 用户坐标系要求1到8 7552 0 目标位置超界 视觉计算偏移大于修正范围', '需要增加修正限位', '1 x方向位置超限 可扩大x轴位置限位 2 y方向位置超限 可扩大y轴位置限位 3 Rz旋转角度超限 可扩大z轴旋转角度范围 4 x方向位置超限 可扩大x轴位置限位 5 y方向位置超限 可扩大y轴位置限位 6 z方向位置超限 可扩大z轴位置限位 7 Rz旋转角度超限 可扩大z轴旋转角度范围 8 Ry旋转角度超限 可扩大y轴旋转角度范围 9 Rx旋转角度超限 可扩大x轴旋转角度范围 7553 0 离线运动失败 重新进行离线运动作业'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 527, 'line_end': 527, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>子码</p></td><td><p>出现问题</p></td><td><p>解决方式</p></td></tr><tr><td rowspan="10"><p>7550</p></td><td><p>0</p></td><td><p>TG_P:TCP连接断开</p></td><td rowspan="10"><p>1.查看网线是否完好</p><p>2.ip连接是否正确</p></td></tr><tr><td><p>1</p></td><td><p>TG:TCP连接断开</p></td></tr><tr><td><p>2</p></td><td><p>GPI_P:TCP连接断开</p></td></tr><tr><td><p>3</p></td><td><p>GPI:TCP连接断开</p></td></tr><tr><td><p>4</p></td><td><p>TG_P:TCP连接断开</p></td></tr><tr><td><p>5</p></td><td><p>GPI:TCP发送失败</p></td></tr><tr><td><p>6</p></td><td rowspan="4"><p>VG:TCP发送失败</p></td></tr><tr><td><p>7</p></td></tr><tr><td><p>8</p></td></tr><tr><td><p>9</p></td></tr><tr><td><p>7551</p></td><td><p>0</p></td><td><p>用户坐标系号错误</p></td><td><p>用户坐标系要求1到8</p></td></tr><tr><td rowspan="10"><p>7552</p></td><td><p>0</p></td><td><p>目标位置超界</p></td><td><p>视觉计算偏移大于修正范围。</p><p>需要增加修正限位。</p></td></tr><tr><td><p>1</p></td><td><p>x方向位置超限</p></td><td><p>可扩大x轴位置限位</p></td></tr><tr><td><p>2</p></td><td><p>y方向位置超限</p></td><td><p>可扩大y轴位置限位</p></td></tr><tr><td><p>3</p></td><td><p>Rz旋转角度超限</p></td><td><p>可扩大z轴旋转角度范围</p></td></tr><tr><td><p>4</p></td><td><p>x方向位置超限</p></td><td><p>可扩大x轴位置限位</p></td></tr><tr><td><p>5</p></td><td><p>y方向位置超限</p></td><td><p>可扩大y轴位置限位</p></td></tr><tr><td><p>6</p></td><td><p>z方向位置超限</p></td><td><p>可扩大z轴位置限位</p></td></tr><tr><td><p>7</p></td><td><p>Rz旋转角度超限</p></td><td><p>可扩大z轴旋转角度范围</p></td></tr><tr><td><p>8</p></td><td><p>Ry旋转角度超限</p></td><td><p>可扩大y轴旋转角度范围</p></td></tr><tr><td><p>9</p></td><td><p>Rx旋转角度超限</p></td><td><p>可扩大x轴旋转角度范围</p></td></tr><tr><td rowspan="2"><p>7553</p></td><td><p>0</p></td><td rowspan="2"><p>离线运动失败</p></td><td rowspan="2"><p>重新进行离线运动作业。</p></td></tr><tr><td><p>1</p></td></tr><tr><td><p>7554</p></td><td><p>0</p></td><td><p>目标位置丢失</p></td><td><p>视觉未检测到目标。重新进行拍照。</p></td></tr><tr><td rowspan="2"><p>7555</p></td><td><p>0</p></td><td rowspan="2"><p>同步帧返回超时</p></td><td rowspan="2"><p>视觉与机器人未同步。检查网线连接。</p></td></tr><tr><td><p>1</p></td></tr><tr><td rowspan="2"><p>7558</p></td><td><p>0</p></td><td rowspan="2"><p>相机初始化失败</p></td><td rowspan="2"><p>视觉方面配置错误。检查网络通讯状态</p></td></tr><tr><td><p>1</p></td></tr><tr><td rowspan="2"><p>7560</p></td><td><p>0</p></td><td rowspan="2"><p>请先打开要标定的作业</p></td><td rowspan="2"><p>标定时未打开作业。打开作业即可。</p></td></tr><tr><td><p>1</p></td></tr><tr><td><p>7561</p></td><td><p>0</p></td><td><p>请根据情况选择修正方式</p></td><td><p>选取正确的视觉修正方式即可。</p></td></tr><tr><td><p>7562</p></td><td><p>0</p></td><td><p>图像采集超时</p></td><td><p>图像采集未成功。检查相机是否正常</p></td></tr><tr><td><p>7563</p></td><td><p>0</p></td><td><p>未发送图像采集指令</p></td><td><p>需查看作业，观察是否有图像采集指令。在执行获取位置指令前要先执行拍照指令。</p></td></tr><tr><td rowspan="2"><p>7564</p></td><td><p>0</p></td><td rowspan="2"><p>图像处理失败</p></td><td rowspan="2"><p>图像处理错误。重新拍照或者更新视觉模板。</p></td></tr><tr><td><p>1</p></td></tr><tr><td rowspan="4"><p>7565</p></td><td><p>0</p></td><td rowspan="4"><p>数据长度错误</p></td><td rowspan="4"><p>视觉与机器人传输数据长度错误。</p><p>检查相机发给机器人的数据，长度是否满足协议要求</p></td></tr><tr><td><p>1</p></td></tr><tr><td><p>2</p></td></tr><tr><td><p>3</p></td></tr><tr><td><p>7566</p></td><td><p>0</p></td><td><p>修正逻辑错误</p></td><td><p>是否进行多重修正方式错误。</p></td></tr><tr><td><p>7567</p></td><td><p>0</p></td><td><p>视觉通讯类型配置错误</p></td><td><p>配置参数要求1或2</p></td></tr><tr><td><p>7568</p></td><td><p>0</p></td><td><p>视觉TCP发送失败</p></td><td><p>检查网络通讯状态</p></td></tr><tr><td><p>7569</p></td><td><p>0</p></td><td><p>离线数据指针为空</p></td><td><p>请联系软件研发人员</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_12', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.5', 'section_title': '2.5 计算偏移值指令', 'heading_path': ['2 指令集', '2.5 计算偏移值指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“计算偏移值指令”中“VSGOFFSET_P CorrectType=（参数1）”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSGOFFSET_P'], 'expected_facts': ['VSGOFFSET_P CorrectType=（参数1） 功能 计算偏移值，并将偏移量存储值对应的P变量中 说明 参数1 校正方式（13、14、15） 13：视觉给予的位姿进行存储到shift偏移变量存储位置对应的位置变量中', '14：将用户坐标系的旋转和偏移转换到机器人坐标系下的旋转和偏移存到【trans偏移变量存储位置】对应的位置变量中', '15：将工具坐标系的旋转和偏移转换到机器人坐标系下的旋转和偏移存到【trans偏移变量存储位置】对应的位置变量中'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 317, 'line_end': 317, 'excerpt': '<table><tr><td colspan="3"><p>VSGOFFSET_P CorrectType=（参数1）</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>计算偏移值，并将偏移量存储值对应的P变量中</p></td></tr><tr><td><p>说明</p></td><td><p>参数1</p></td><td><p>校正方式（13、14、15）</p><p>13：视觉给予的位姿进行存储到shift偏移变量存储位置对应的位置变量中。</p><p>14：将用户坐标系的旋转和偏移转换到机器人坐标系下的旋转和偏移存到【trans偏移变量存储位置】对应的位置变量中。</p><p>15：将工具坐标系的旋转和偏移转换到机器人坐标系下的旋转和偏移存到【trans偏移变量存储位置】对应的位置变量中。</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_13', 'category': 'vision', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2', 'section_title': '2.2 获取相机测量结果', 'heading_path': ['2 指令集', '2.2 获取相机测量结果'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），配置“获取相机测量结果”时，“VSGPI  PO#=（参数1）  NO=（参数2）”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['VSGPI PO#=（参数1） NO=（参数2） 功能 获取相机测量结果 说明 参数1 视觉场景组 参数2 视觉作业号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 305, 'line_end': 305, 'excerpt': '<table><tr><td colspan="3"><p><a id="_Hlk163400126"></a>VSGPI  PO#=（参数1）  NO=（参数2）</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>获取相机测量结果</p></td></tr><tr><td rowspan="2"><p>说明</p></td><td><p>参数1</p></td><td><p>视觉场景组</p></td></tr><tr><td><p>参数2</p></td><td><p>视觉作业号</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_14', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 视觉系统参数配置', 'heading_path': ['1 视觉应用参数配置', '1.1 视觉系统参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“视觉系统参数配置”中，应如何理解或操作“X/Y/Z/AX/AY/AZ方向的修正限位”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. X/Y/Z/AX/AY/AZ方向的修正限位：为位姿方向限值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 106, 'line_end': 106, 'excerpt': '2. X/Y/Z/AX/AY/AZ方向的修正限位：为位姿方向限值。', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_15', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.3', 'section_title': '4.3 同步帧', 'heading_path': ['4 通信协议', '4.3 同步帧'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“同步帧”中，应如何理解或操作“0X20”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['0X20:同步帧心跳请求'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 361, 'line_end': 361, 'excerpt': '0X20:同步帧心跳请求；', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_16', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.3', 'section_title': '1.3 作业参数', 'heading_path': ['1 视觉应用参数配置', '1.3 作业参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“作业参数”中“VS\\_GPI标志位的存储位置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['4. VS\\_GPI标志位的存储位置： 相机VS\\_GPI指令返回值放在该存储位置对应的整型变量中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 129, 'line_end': 129, 'excerpt': '4. VS\\_GPI标志位的存储位置： 相机VS\\_GPI指令返回值放在该存储位置对应的整型变量中。', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_17', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.2', 'section_title': '4.2 视觉系统初始化', 'heading_path': ['4 通信协议', '4.2 视觉系统初始化'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“视觉系统初始化”中“0x51”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['0x51：清除当前测量值成功应答，为0x21指令的应答指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 347, 'line_end': 347, 'excerpt': '0x51：清除当前测量值成功应答，为0x21指令的应答指令；', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_18', 'category': 'vision', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 相机发送拍照指令', 'heading_path': ['2 指令集', '2.3 相机发送拍照指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），“相机发送拍照指令”中“VSTG_N_P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VSTG_N_P'], 'expected_facts': ['VSTG_N_P N=（参数1） T=（参数2） W=（参数3）A=（参数4） 功能 给相机发送拍照指令 说明 参数1 动作标志', '0视觉准备测量，1触发视觉测量（范围：0-255） 参数2 定位还是质量检测', '0是调用定位功能1是调用打磨质量检测功能'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 309, 'line_end': 309, 'excerpt': '<table><tr><td colspan="3"><p>VSTG_N_P  N=（参数1）  T=（参数2） W=（参数3）A=（参数4）</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>给相机发送拍照指令</p></td></tr><tr><td rowspan="4"><p>说明</p></td><td><p>参数1</p></td><td><p>动作标志； 0视觉准备测量，1触发视觉测量（范围：0-255）   </p></td></tr><tr><td><p>参数2</p></td><td><p>定位还是质量检测；0是调用定位功能1是调用打磨质量检测功能； 2是调用凸起高度计算功能</p><p>（范围：0-255）</p></td></tr><tr><td><p>参数3</p></td><td><p>工件号（范围：0--65535）</p></td></tr><tr><td><p>参数4</p></td><td><p>区域号（范围：0-65535）当调用凸起高度计算功能时为左右边区分标志位，0为左，1为右。</p></td></tr></table>', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_19', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4', 'section_title': '1.4 作业拍照', 'heading_path': ['1 视觉应用参数配置', '1.4 作业拍照'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“作业拍照”中，应如何理解或操作“IF I99 ==02 L12 工件类型2执行L12”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['IF I99 ==02 L12 工件类型2执行L12'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 239, 'line_end': 239, 'excerpt': 'IF I99 ==02 L12 工件类型2执行L12', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}, {'id': 'sample_vision_b61f315fb1_20', 'category': 'vision', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.6', 'section_title': '4.6 获取机器人位姿', 'heading_path': ['4 通信协议', '4.6 获取机器人位姿'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“获取机器人位姿”中，应如何理解或操作“0x40”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['0x40，机器人位姿坐标应答，消息体最少28字节，不定长'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'line_start': 441, 'line_end': 441, 'excerpt': '0x40，机器人位姿坐标应答，消息体最少28字节，不定长；', 'sha256': '477ad524077d44e5355230f6e7f305c33cff2019786d022464a7761e179efc8e'}]}]
register_sample_bank('vision', VISION_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md', 'seed': 4703487091905968718, 'population': 148, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(622, 622), (166, 166), (803, 808), (489, 492), (502, 502), (493, 497), (235, 235), (431, 435), (680, 680), (875, 880), (211, 214), (792, 792), (764, 764), (191, 194), (475, 478), (289, 289), (381, 384), (736, 736), (839, 844), (207, 210)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'seed': 2424595973294392418, 'population': 167, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(182, 182), (497, 500), (225, 228), (712, 712), (994, 999), (522, 522), (455, 459), (806, 806), (150, 150), (880, 884), (915, 919), (812, 812), (265, 265), (317, 317), (489, 492), (233, 236), (666, 666), (934, 939), (448, 450), (628, 628)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md', 'seed': 8163715398223486617, 'population': 105, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(137, 137), (321, 321), (313, 313), (108, 108), (385, 385), (159, 159), (497, 497), (95, 95), (517, 517), (531, 531), (527, 527), (317, 317), (305, 305), (106, 106), (361, 361), (129, 129), (347, 347), (309, 309), (239, 239), (441, 441)]}])


In [ ]:
# | notest
VISION_SAMPLE_IDS = None
vision_sample_cases = select_cases(VISION_SAMPLE_CASES, VISION_SAMPLE_IDS)
display(pd.DataFrame(vision_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in vision_sample_cases])
vision_sample_metrics = await evaluation.run(vision_sample_cases, concurrency=EVALUATION_CONCURRENCY)
vision_sample_results = {case["id"]: evaluation.load(case) for case in vision_sample_cases}
display(vision_sample_metrics)
# Inspect a complete result: evaluation.inspect(vision_sample_cases[0]["id"])
evaluation.show_answers(vision_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_vision_af4fcdcff7_01,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,12158,0,True,unsupported,,,not_assessable
sample_vision_af4fcdcff7_02,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,9700,0,True,unsupported,,,abstained
sample_vision_af4fcdcff7_03,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,procedure,table_row,False,True,8,False,0,0,7,False,False,False,None,None,15494,0,True,unsupported,,,not reviewed
sample_vision_af4fcdcff7_04,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,5437,0,True,included,,,not reviewed
sample_vision_af4fcdcff7_05,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,7683,0,True,unsupported,,,not reviewed
sample_vision_af4fcdcff7_06,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,parameter,table_row,False,True,8,False,0,0,1,False,False,False,None,None,6134,0,True,included,,,not reviewed
sample_vision_af4fcdcff7_07,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,4576,0,True,included,,,not reviewed
sample_vision_af4fcdcff7_08,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,7746,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed
sample_vision_af4fcdcff7_09,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,11385,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed
sample_vision_af4fcdcff7_10,vision,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,alarm,table_row,False,True,8,False,0,0,0,False,False,False,False,False,11804,0,True,unsupported,,,incorrect


### sample_vision_af4fcdcff7_01

根据《SX322005《新松机器人视觉应用操作手册》-V1.0》（V3.0 / V1.0），“视觉系统初始化”中“0x51”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 该行描述新松机器人视觉应用（SX322005手册）中代码 **7571** 对应状态“**相机初始化完成**”，参数值为 **1**（第4列无内容）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 1060–1063; corpus 5.0, revision V5.0 / A-1; object `7e8167fcc0551b469623975fca19532881701c85c77eed1b1cbe60499b8773a7`.

- 该行为表格第 5 行（索引 4），来自《新松机器人视觉应用操作手册》V5.0。行内包含四个单元格：

1.  **列 1**：值为 `7571`，该单元格垂直跨越两行（rowspan=2），表示此代码/ID 适用于本行及下一行。
2.  **列 2**：值为 `0`，仅适用于当前行。
3.  **列 3**：值为 `相机初始化完成`，该单元格同样垂直跨越两行（rowspan=2），表明此状态/描述适用于本行及下一行。
4.  **列 4**：值为空字符串，仅适用于当前行。

**摘要**：在第 5 行中，代码/ID `7571` 和状态描述 `相机初始化完成` 均垂直合并，同时涵盖当前行与下一行；而数值 `0` 和空值仅属于当前行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 1054–1059; corpus 5.0, revision V5.0 / A-1; object `83649dfd76705f282c3e8fbfc88bc10ae52fd3c524b2c070fdbb5bab4e7cc779`.

- 基于表格行数据，该行的摘要如下：

**视觉初始化命令（0x21）的成功应答为 0x51，对应操作“清除当前测量值”。**

> **解析说明：**
> 根据单元格顺序推断列关系（通常此类通信协议表结构为：`指令名称` | `指令码` | `应答/结果描述` | `应答码` | `备注/功能说明`）：
> - **指令**：视觉初始化
> - **指令码**：0x21
> - **应答描述**：初始化成功应答
> - **应答码**：0x51
> - **附加功能/备注**：清除当前测量值
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 582–588; corpus 4.0, revision V5.0 / A-1; object `fbabf7f5cb79e3d56b8b9845eea994b8510b67a25ff92749d7b1d68aa429f61d`.

- 该命令条目对应于视觉初始化操作，其命令码为0x21，对应的应答为0x51（初始化成功应答），该操作会清除当前测量值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 582–588; corpus 5.0, revision V5.0 / A-1; object `ad00f94f0ca8e7d050f45f2f3cf038fcd40872f48b151f6ac9c513e8b146c5ba`.

- 该行属于《新松机器人视觉应用操作手册》，包含三个单元格：第一列“说明”跨两行（rowspan=2），第二列为“参数1”，第三列为“视觉场景组”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 413–417; corpus 5.0, revision V5.0 / A-1; object `9b0a865fb2ad7ab4a2b23fde80953724b0e233bf2600f443e6d5e5c03dfa8d77`.

- 新松机器人视觉应用手册中代码7561（参数0）的操作指引：提示用户“请根据情况选择修正方式”，操作说明为“选取正确的视觉修正方式即可”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 967–972; corpus 5.0, revision V5.0 / A-1; object `cbca48df6d5d88b8178478dcb4ed6c21699fff5f1290f9df73f2afdef97af0d1`.

- 该行为表格子标题行（源自《新松机器人视觉应用操作手册》V5.0），第1、3列分别标注“名称”，第2、4列分别标注“指令”，用于对两组参数进行并列说明；第5列“作用”为跨两行合并单元格，表明该行是某功能作用描述部分的起始子行，与上一行共同构成同一作用字段下的双行结构。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 611–616; corpus 5.0, revision V5.0 / A-1; object `d2ee4993c55540adfb8b026053d8d799166cd0fa465addb72be51e6152dc2efe`.

- 新松机器人视觉应用操作手册（A/1 版本）于2025年2月26日由沙景超更新，主要修订内容为：1.增加计算偏移指令；2.完善报警码与提示码。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 40–45; corpus 4.0, revision V5.0 / A-1; object `26d8eeed935cceb159e79dd3cc4d1e2355f7ba2e1773a96bb55db8186191663d`.

#### Answer

根据手册证据（E3），0x51 是

#### Model-assisted source review

Assessment: **not_assessable**; completeness: partial; citation support: not_applicable; revision fidelity: mixed.

The generated answer is a truncated fragment ('根据手册证据（E3），0x51 是') and does not state the function, syntax, parameters, or behavior of 0x51. It therefore fails the completeness requirement. The supplied evidence (E3) does contain the relevant fact that 0x51 is the '初始化成功应答' (initialization success response) to the 0x21 '视觉初始化' (vision initialization) command, with the action '清除当前测量值' (clear current measurement values). However, because the answer was not fully generated, it cannot be judged as correct or incorrect on its merits. The evidence is adequate to answer the question, so this is not an appropriate abstention. The answer cites E3, which is a valid source for the fact, but the answer itself is incomplete.

...

#### Answer

The retrieved evidence documents the VSTG_P (send photography command) and VSGPI (get measurement result) instructions with their parameters (视觉场景组, 视觉作业号), the TCP/IP communication protocol between the robot controller and vision processor, and alarm codes. However, the specific line 'IF I99 ==02 L12 工件类型2执行L12' is a robot-program-level conditional branch statement. None of the provided evidence documents: (1) the IF conditional syntax used in the robot program, (2) the definition or purpose of internal register I99, (3) the L12 label or label-jump mechanism, or (4) the '工件类型' (workpiece type) selection logic within the '作业拍照' workflow. The evidence only covers the vision instruction set (VSTG_P, VSGPI) and their parameters, not the program-flow control constructs that would invoke them conditionally based on workpiece type.

### sample_vision_b61f315fb1_20

根据《SX322005《新松机器人视觉应用操作手册》（A-1)》（V5.0 / A-1），在“获取机器人位姿”中，应如何理解或操作“0x40”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- **获取机器人位姿**（指令码 `0x30`）：用于获取**机器人当前位姿**；**校准时**（指令码 `0x40`）：用于获取**实际坐标**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 617–623; corpus 4.0, revision V5.0 / A-1; object `8de5bc100ccce2faaf92f3c778a7fe7add160968796446ee473d59fba1371d5b`.

- 获取机器人位姿（代码 0x30）用于获取机器人当前位姿；0x40 用于校准时获取实际坐标。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 617–623; corpus 5.0, revision V5.0 / A-1; object `233a6bc16fb69700f0a8f076e05417d21eb3c75f79ea319ac0ff58cb955e86d1`.

- 新松机器人视觉应用操作手册（A/1 版本）于2025年2月26日由沙景超更新，主要修订内容为：1.增加计算偏移指令；2.完善报警码与提示码。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 40–45; corpus 4.0, revision V5.0 / A-1; object `26d8eeed935cceb159e79dd3cc4d1e2355f7ba2e1773a96bb55db8186191663d`.

- 第5行（row_index 4）：包含4个空白单元格（列1-4），无文本、无图片、无继承值，非表头，来源为《新松机器人视觉应用操作手册》第46-51行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 46–51; corpus 4.0, revision V5.0 / A-1; object `011fe6fe6e65ec7a4fa0cab56516172a934e1fdc330989fcec1d3b56cbad466d`.

- 新松机器人视觉应用操作说明书
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 15–17; corpus 5.0, revision V5.0 / A-1; object `af37bf6ac03191675cbfe8d4e6bcf575b866b28be3cc92ff42b91c73bba683e3`.

- 新松机器人视觉应用操作说明书
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 15–17; corpus 4.0, revision V5.0 / A-1; object `d02ae2ffdc3a7484d6f945c8a1f0337efebbee10e51fb2ce12498b58589712fa`.

- 该行属于《新松机器人视觉应用操作手册》，包含三个单元格：第一列“说明”跨两行（rowspan=2），第二列为“参数1”，第三列为“视觉场景组”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 413–417; corpus 5.0, revision V5.0 / A-1; object `9b0a865fb2ad7ab4a2b23fde80953724b0e233bf2600f443e6d5e5c03dfa8d77`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

#### Answer

In the SX322005 vision application manual (V5.0 / A-1), 0x40 is the response command code in the '视觉→机器人' (Vision→Robot) communication table. When the Vision Processor (VC) sends request 0x30 '获取机器人位姿' (Get Robot Pose) to the Robot Controller (RC) over TCP/IP (port 9999), the RC replies with 0x40 '机器人当前位姿' (Robot Current Pose). The stated purpose is '校准时，获取实际坐标' — during calibration, obtain the actual coordinates. The message consists of a header (the 0x40 byte) followed by a body containing the pose data. In practice, the RC acts as the TCP client (172.16.1.30) and the VC as the server (172.16.1.63); after the VC issues 0x30, the RC's 0x40 response carries the six-DOF pose needed for calibration offset computation.

### Example — vision

During a vision-guided calibration session, the Vision Processor requests the Robot Controller's current pose so it can compute the coordinate offset between the camera frame and the robot base frame. The operator triggers the calibration routine, the VC sends the 0x30 request over the established TCP connection, and the RC responds with 0x40 carrying the actual joint/cartesian coordinates.

**Environment and prerequisites:** Prerequisites: RC and VC are on the same 172.16.1.0/24 subnet; RC IP 172.16.1.30, VC IP 172.16.1.63; both listening on port 9999; TCP/IP protocol active. Variables: rc (socket object), resp (bytes received), pose_body (bytes after the 0x40 header). Initial values: socket not yet connected; no data received. User-supplied placeholders: none — all addresses and ports are fixed by the manual.

**Expected result:** The RC returns a message whose first byte is 0x40, confirming it is the '机器人当前位姿' response. The remaining bytes (message body) contain the robot's current pose coordinates, which the calibration routine uses to compute the offset between the vision coordinate system and the robot coordinate system.

**Manual:** SX322005《新松机器人视觉应用操作手册》（A-1) V5.0 / A-1 [E1]

```python
import socket

rc = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
rc.connect(('172.16.1.63', 9999))
rc.sendall(bytes([0x30]))
resp = rc.recv(256)
assert resp[0] == 0x40, "Expected 0x40 robot-pose response"
pose_body = resp[1:]
print(f"Calibration coordinates: {pose_body.hex()}")
```

#### Model-assisted source review

Assessment: **partial**; completeness: partial; citation support: partial; revision fidelity: faithful.

The answer correctly identifies 0x40 as the response to the 0x30 '获取机器人位姿' request and cites the purpose '校准时，获取实际坐标' from the supplied evidence [E1]. However, it fails to include the specific expected facts regarding the message body structure ('消息体最少28字节，不定长') found in the references. Instead, it makes unsupported claims about the message structure (e.g., 'header (the 0x40 byte)') and the specific content ('six-DOF pose') which are not present in the provided evidence snippets. The evidence only confirms the command code, name, and purpose, not the byte-level structure or specific DOF count.

## Teaching Free Welding — sampled manual revisions

20 cases from 1 CSV files. Set `TEACHING_FREE_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
TEACHING_FREE_WELDING_SAMPLE_CASES = [{'id': 'sample_teaching_free_welding_7f65a517d8_01', 'category': 'teaching_free_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.5.5', 'section_title': '3.5.5 桁架选择', 'heading_path': ['3 服务端功能描述', '3.5 系统建模', '3.5.5 桁架选择'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），操作“桁架选择”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：不需要改动，客户端根据需要已经实现了自动切换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 582, 'line_end': 582, 'excerpt': '**注意：不需要改动，客户端根据需要已经实现了自动切换。**', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_02', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.2', 'section_title': '4.2 从机器人', 'heading_path': ['4 作业流程', '4.2 从机器人'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“从机器人”中，应如何理解或操作“从机器人后台的作业主要包含【从站主作业】、【子站清枪作业】、【从站免示教焊接作业】以及与【工艺交”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['从机器人后台的作业主要包含【从站主作业】、【子站清枪作业】、【从站免示教焊接作业】以及与【工艺交互信号互锁作业】'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 798, 'line_end': 798, 'excerpt': '从机器人后台的作业主要包含【从站主作业】、【子站清枪作业】、【从站免示教焊接作业】以及与【工艺交互信号互锁作业】', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_03', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.2', 'section_title': '2.2 RobotToolInstall自动安装', 'heading_path': ['2 软件安装', '2.2 RobotToolInstall自动安装'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“RobotToolInstall自动安装”中，应如何理解或操作“进入RobotToolInstall文件夹”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['进入RobotToolInstall文件夹，找到RobotToolInstall.exe，右键以管理员身份运行'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 206, 'line_end': 206, 'excerpt': '进入RobotToolInstall文件夹，找到RobotToolInstall.exe，**右键以管理员身份运行**。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_04', 'category': 'teaching_free_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.2.4', 'section_title': '5.2.4 焊缝列表及操作', 'heading_path': ['5 客户端功能描述', '5.2 客户端操作', '5.2.4 焊缝列表及操作'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），配置“焊缝列表及操作”时，“传感器对工件完成扫描后生成的焊缝信息显示在焊缝列表中”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['传感器对工件完成扫描后生成的焊缝信息显示在焊缝列表中，并可对焊缝信息进行参数编辑及其他操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 890, 'line_end': 890, 'excerpt': '传感器对工件完成扫描后生成的焊缝信息显示在焊缝列表中，并可对焊缝信息进行参数编辑及其他操作。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_05', 'category': 'teaching_free_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.8.5', 'section_title': '3.8.5 焊接系统配置', 'heading_path': ['3 服务端功能描述', '3.8 焊接应用', '3.8.5 焊接系统配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），配置“焊接系统配置”时，“用于配置焊接系统使用过程中会用到的各种参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['用于配置焊接系统使用过程中会用到的各种参数', '这些参数可以根据工件情况进行调整'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 710, 'line_end': 710, 'excerpt': '用于配置焊接系统使用过程中会用到的各种参数。这些参数可以根据工件情况进行调整。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_06', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.2.4', 'section_title': '5.2.4 焊缝列表及操作', 'heading_path': ['5 客户端功能描述', '5.2 客户端操作', '5.2.4 焊缝列表及操作'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“焊缝列表及操作”中，应如何理解或操作“焊缝恢复”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 焊缝恢复：对跳过的焊缝恢复原始状态（仅对跳过的焊缝有效）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 918, 'line_end': 918, 'excerpt': '1. 焊缝恢复：对跳过的焊缝恢复原始状态（仅对跳过的焊缝有效）', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_07', 'category': 'teaching_free_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.2.6', 'section_title': '5.2.6 控制面板操作', 'heading_path': ['5 客户端功能描述', '5.2 客户端操作', '5.2.6 控制面板操作'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），操作“控制面板操作”时，涉及“删除焊缝”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. 删除焊缝：当发现扫描规划后，多出某条焊缝时，可通过删除焊缝将不必要的焊缝删除', '（在重置焊缝条件下有效）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 1078, 'line_end': 1078, 'excerpt': '1. 删除焊缝：当发现扫描规划后，多出某条焊缝时，可通过删除焊缝将不必要的焊缝删除。（在重置焊缝条件下有效）', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_08', 'category': 'teaching_free_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.2', 'section_title': '3.1.2 登陆界面', 'heading_path': ['3 服务端功能描述', '3.1 服务端启动', '3.1.2 登陆界面'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），“登陆界面”中“默认自动打开页面【作业管理】”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['默认自动打开页面【作业管理】，工作站的设备包含的作业和指令就由该页面控制，在后台执行'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 476, 'line_end': 476, 'excerpt': '默认自动打开页面【作业管理】，工作站的设备包含的作业和指令就由该页面控制，在后台执行。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_09', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.8', 'section_title': '3.8 焊接应用', 'heading_path': ['3 服务端功能描述', '3.8 焊接应用'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“焊接应用”中，应如何理解或操作“【焊接应用】模块是实施人员再配置好设备后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['【焊接应用】模块是实施人员再配置好设备后，重点配置和使用的另一个模块'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 608, 'line_end': 608, 'excerpt': '【焊接应用】模块是实施人员再配置好设备后，重点配置和使用的另一个模块。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_10', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1', 'section_title': '1 引言', 'heading_path': ['1 引言'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“引言”中，应如何理解或操作“本文档是沈阳新松机器人自动化股份有限公司的产品”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['本文档是沈阳新松机器人自动化股份有限公司的产品： 机器人免示教焊接软件针对用户所编写的操作手册'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 178, 'line_end': 178, 'excerpt': '本文档是沈阳新松机器人自动化股份有限公司的产品： 机器人免示教焊接软件针对用户所编写的操作手册。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_11', 'category': 'teaching_free_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.8.1', 'section_title': '3.8.1 焊接规则配置', 'heading_path': ['3 服务端功能描述', '3.8 焊接应用', '3.8.1 焊接规则配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），操作“焊接规则配置”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：部分规则以客户端【工件扫描配置】为主'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 664, 'line_end': 664, 'excerpt': '**注意：部分规则以客户端【工件扫描配置】为主。**', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_12', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.1', 'section_title': '4.1 主机器人', 'heading_path': ['4 作业流程', '4.1 主机器人'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“主机器人”中，应如何理解或操作“主机器人后台的作业主要包含【组立版主作业】、【清枪】、【免示教焊接作业】以及与【工艺交互信号互锁”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['主机器人后台的作业主要包含【组立版主作业】、【清枪】、【免示教焊接作业】以及与【工艺交互信号互锁作业】'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 750, 'line_end': 750, 'excerpt': '主机器人后台的作业主要包含【组立版主作业】、【清枪】、【免示教焊接作业】以及与【工艺交互信号互锁作业】', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_13', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.4.1', 'section_title': '2.4.1 客户端-IIS安装流程', 'heading_path': ['2 软件安装', '2.4 客户端自动安装', '2.4.1 客户端-IIS安装流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“客户端-IIS安装流程”中，应如何理解或操作“点击【2.IIS安装】”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击【2.IIS安装】，弹出IIS组件安装视频，根据视频操作安装IIS组件，可能需要联网操作下载必要文件，下载可能需要等待一段时间'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 366, 'line_end': 366, 'excerpt': '点击【2.IIS安装】，弹出IIS组件安装视频，根据视频操作安装IIS组件，可能需要联网操作下载必要文件，下载可能需要等待一段时间。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_14', 'category': 'teaching_free_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.3', 'section_title': '5.3 工件扫描配置', 'heading_path': ['5 客户端功能描述', '5.3 工件扫描配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），配置“工件扫描配置”时，“进行扫描工件作业前”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['进行扫描工件作业前，先要进行工件相关参数配置工作，这样才能保证工件焊缝识别的正确性', '左侧功能菜单点击【工件扫描配置】，右侧显示工件扫描配置页面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 1117, 'line_end': 1117, 'excerpt': '进行扫描工件作业前，先要进行工件相关参数配置工作，这样才能保证工件焊缝识别的正确性。左侧功能菜单点击【工件扫描配置】，右侧显示工件扫描配置页面。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_15', 'category': 'teaching_free_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.8.5', 'section_title': '3.8.5 焊接系统配置', 'heading_path': ['3 服务端功能描述', '3.8 焊接应用', '3.8.5 焊接系统配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），配置“焊接系统配置”时，“重要参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['重要参数：焊接单元、焊点类型、清枪长度、过焊孔（示教过焊孔、交叉过焊孔、开放过焊孔）等'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 712, 'line_end': 712, 'excerpt': '重要参数：焊接单元、焊点类型、清枪长度、过焊孔（示教过焊孔、交叉过焊孔、开放过焊孔）等。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_16', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.2.3', 'section_title': '5.2.3 服务日志', 'heading_path': ['5 客户端功能描述', '5.2 客户端操作', '5.2.3 服务日志'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“服务日志”中，应如何理解或操作“显示最新一条服务端的服务日志”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['显示最新一条服务端的服务日志'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 882, 'line_end': 882, 'excerpt': '显示最新一条服务端的服务日志', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_17', 'category': 'teaching_free_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.2.6', 'section_title': '5.2.6 控制面板操作', 'heading_path': ['5 客户端功能描述', '5.2 客户端操作', '5.2.6 控制面板操作'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），操作“控制面板操作”时，涉及“当对工件扫描后”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['当对工件扫描后，当焊缝数量或焊缝类型不正确时，可以通过点云显示区域控制面板中的【添加焊缝】、【删除焊缝】、【编辑焊缝】功能来手动校准焊缝，满足现场焊接条件'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 985, 'line_end': 985, 'excerpt': '当对工件扫描后，当焊缝数量或焊缝类型不正确时，可以通过点云显示区域控制面板中的【添加焊缝】、【删除焊缝】、【编辑焊缝】功能来手动校准焊缝，满足现场焊接条件。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_18', 'category': 'teaching_free_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.2.1', 'section_title': '3.2.1 作业管理', 'heading_path': ['3 服务端功能描述', '3.2 流程控制', '3.2.1 作业管理'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），“作业管理”中“针对本项目”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['针对本项目，【作业管理】内的作业和指令暂不需要修改，需要具有一定的编程知识并且了解工作站工艺才能进行优化修改'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 492, 'line_end': 492, 'excerpt': '针对本项目，【作业管理】内的作业和指令暂不需要修改，需要具有一定的编程知识并且了解工作站工艺才能进行优化修改。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_19', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.1', 'section_title': '3.1.1 授权方式', 'heading_path': ['3 服务端功能描述', '3.1 服务端启动', '3.1.1 授权方式'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“授权方式”中，应如何理解或操作“对于正式版”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['对于正式版，需要在客户终端机器上运行获取授权验证码程序，然后根据该验证码我方提供给客户对应的正式版授权文件'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 456, 'line_end': 456, 'excerpt': '对于正式版，需要在客户终端机器上运行获取授权验证码程序，然后根据该验证码我方提供给客户对应的正式版授权文件。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}, {'id': 'sample_teaching_free_welding_7f65a517d8_20', 'category': 'teaching_free_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1', 'section_title': '1 引言', 'heading_path': ['1 引言'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“引言”中，应如何理解或操作“在文档中”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['在文档中，对客户端系统的各项功能进行了详细而具体的说明和操作描述', '通过本文档，用户可以了解服务端、客户端的所有功能和使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'line_start': 180, 'line_end': 180, 'excerpt': '在文档中，对客户端系统的各项功能进行了详细而具体的说明和操作描述。通过本文档，用户可以了解服务端、客户端的所有功能和使用方法。', 'sha256': '0382bdcfb87cec5307435fc5b19b5cdc0306990412d0f60c08d852299a539bb0'}]}]
register_sample_bank('teaching_free_welding', TEACHING_FREE_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md', 'seed': 1855469976689087915, 'population': 127, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(582, 582), (798, 798), (206, 206), (890, 890), (710, 710), (918, 918), (1078, 1078), (476, 476), (608, 608), (178, 178), (664, 664), (750, 750), (366, 366), (1117, 1117), (712, 712), (882, 882), (985, 985), (492, 492), (456, 456), (180, 180)]}])


In [ ]:
# | notest
TEACHING_FREE_WELDING_SAMPLE_IDS = None
teaching_free_welding_sample_cases = select_cases(TEACHING_FREE_WELDING_SAMPLE_CASES, TEACHING_FREE_WELDING_SAMPLE_IDS)
display(pd.DataFrame(teaching_free_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in teaching_free_welding_sample_cases])
teaching_free_welding_sample_metrics = await evaluation.run(teaching_free_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
teaching_free_welding_sample_results = {case["id"]: evaluation.load(case) for case in teaching_free_welding_sample_cases}
display(teaching_free_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(teaching_free_welding_sample_cases[0]["id"])
evaluation.show_answers(teaching_free_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_teaching_free_welding_7f65a517d8_01,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,prerequisite,prose,True,True,8,True,5,3,3,True,False,False,None,None,5701,0,True,unsupported,,,abstained
sample_teaching_free_welding_7f65a517d8_02,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,5,3,3,True,False,False,None,None,5410,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_03,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,6,3,2,True,False,False,None,None,5223,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_04,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,parameter,prose,True,True,8,True,3,2,5,True,False,False,None,None,11605,0,True,unsupported,,,abstained
sample_teaching_free_welding_7f65a517d8_05,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,parameter,prose,True,True,8,True,3,2,5,True,False,False,None,None,12411,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_06,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,3,2,5,True,False,False,None,None,5978,1,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_07,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,prerequisite,prose,True,True,8,True,6,3,2,True,False,False,None,None,6174,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_08,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,instruction,prose,True,True,8,True,6,3,2,True,False,False,None,None,6181,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_09,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,5,3,3,True,False,False,None,None,6884,0,True,unsupported,,,not reviewed
sample_teaching_free_welding_7f65a517d8_10,teaching_free_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,4,2,4,True,False,False,None,None,5424,0,True,unsupported,,,not reviewed


### sample_teaching_free_welding_7f65a517d8_01

根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），操作“桁架选择”时，涉及“注意”的前提条件或注意事项是什么？

#### Retrieved content summary

- 新松机器人免示教焊接软件操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 7–9; corpus 4.0, revision V3.0 / V1.0; object `dfe3159a01c20e8b9a0100230992d237eef0804bc8c369ec1fe86cf509fd167a`.

- 新松机器人免示教焊接软件操作手册（工业机器人技术资料）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 3–13; corpus 5.0, revision V3.0 / V1.0; object `24ccad338a9ea275f1f3b5ee5b50b729d32842d27a1ecd326876fdb55bc2d00f`.

- 新松机器人免示教焊接软件操作手册（工业机器人技术资料）
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 3–13; corpus 4.0, revision V3.0 / V1.0; object `f32c34c69d02e8df3c4819bebaecc01f1d59a807687f072389c335026c26f2ea`.

- 该行位于《新松机器人免示教焊接软件操作手册》中，包含4个非标题列，所有单元格内容均为空，无跨行或跨列合并，且未继承任何值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 48–53; corpus 4.0, revision V3.0 / V1.0; object `4f9395395fef8584d54eb4c9724453c040db12d8130c2be982e9a3a1262c8427`.

- 该行来自《新松机器人免示教焊接软件操作手册》V1.0 文档第 4 号表格的第 3 行，包含 4 个空单元格（均无文本、无跨行/跨列合并），无可见列头或继承值，实质内容为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 36–41; corpus 5.0, revision V3.0 / V1.0; object `b7fb5f30236acef493a7ab3a92b5dd23090c17e105cc9ae339da634bc32220df`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 新松机器人通用操作手册（文档编号：SX322001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 480–483; corpus 5.0, revision V5.0 / C-0; object `7723af1da440d3260a499cef189d99aef7de63c3cbfdace5cad603c336881343`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

#### Answer

The retrieved evidence (E1–E5) only contains the manual's title page, revision history table, and document references. None of the provided excerpts include the section on '桁架选择' (gantry/truss selection) or any associated '注意' (caution) prerequisites. The specific operational content and its caution notes are not present in the supplied evidence, so a supported contextual example cannot be constructed.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The supplied evidence (E1-E5) consists only of the manual's title page, revision history, and document references. It does not contain the specific section on '桁架选择' (gantry selection) or the associated '注意' (caution) note. The expected fact ('注意：不需要改动，客户端根据需要已经实现了自动切换') is not present in the provided evidence snippets. Therefore, the model's refusal to answer based on the lack of relevant evidence is an appropriate abstention rather than a hallucination or an incorrect answer.

...

#### Answer

在所提供的检索证据（E1–E8）中，仅包含该手册的封面标题（

### sample_teaching_free_welding_7f65a517d8_20

根据《SX322008《新松机器人免示教焊接软件操作手册》-V1.0》（V3.0 / V1.0），在“引言”中，应如何理解或操作“在文档中”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行为文档标题行，内容为“新松机器人免示教焊接软件操作手册”，表明该行用于标识或展示该操作手册的名称。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 7–9; corpus 5.0, revision V3.0 / V1.0; object `255d2ff6f15616d0e2c2bc25933fb86d7bd3427f4a4c1506a7dad86966fd8045`.

- 该行（第 6 行）包含 4 个空白单元格，无可见文本、图片或继承值；其所属文档为《新松机器人免示教焊接软件操作手册》V1.0（SX322008），但该行本身未提供具体信息。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 54–59; corpus 5.0, revision V3.0 / V1.0; object `fb4a4acaa20d3b4eb7d239a85031575353281a7d7f1e457f71c3f05b1e9908c3`.

- 该行来自《新松机器人免示教焊接软件操作手册》V1.0 文档第 4 号表格的第 3 行，包含 4 个空单元格（均无文本、无跨行/跨列合并），无可见列头或继承值，实质内容为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 36–41; corpus 5.0, revision V3.0 / V1.0; object `b7fb5f30236acef493a7ab3a92b5dd23090c17e105cc9ae339da634bc32220df`.

- 该行位于《新松机器人免示教焊接软件操作手册》中，包含4个非标题列，所有单元格内容均为空，无跨行或跨列合并，且未继承任何值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 48–53; corpus 4.0, revision V3.0 / V1.0; object `4f9395395fef8584d54eb4c9724453c040db12d8130c2be982e9a3a1262c8427`.

- 新松机器人免示教焊接软件操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md`, lines 7–9; corpus 4.0, revision V3.0 / V1.0; object `dfe3159a01c20e8b9a0100230992d237eef0804bc8c369ec1fe86cf509fd167a`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- **新松机器人螺柱焊应用操作手册**（文档标题/章节名，来源文件：SX322013《新松机器人螺柱焊应用操作手册》(A-2)，位于产品手册/软件类/V5.0版本目录）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 13–15; corpus 5.0, revision V5.0 / A-2; object `3580d5a8c5116a37b6bf8e2524d4a1e2392c9979a45ca913b8f10e6e1e9c6ad3`.

- 新松机器人通用操作手册（文档编号：SX322001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 480–483; corpus 5.0, revision V5.0 / C-0; object `7723af1da440d3260a499cef189d99aef7de63c3cbfdace5cad603c336881343`.

#### Answer

The retrieved evidence (E1–E8) does not contain the actual text of the '引言' (Introduction) section of SX322008《新松机器人免示教焊接软件操作手册》-V1.0. E1 only shows the title-page table, E2 shows the revision-history table, and E3/E4/E5 reference line ranges (36-41, 48-53, 7-9) but provide no readable content. E6–E8 belong to entirely different manuals (grinding, stud welding, laser welding). No evidence mentions the phrase '在文档中' as an operable term, a UI element, or an instruction within the Introduction. Therefore the specific syntax, context, or operational meaning of '在文档中' in the Introduction cannot be established from the supplied evidence.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1-E8) does not contain the specific text from the 'Introduction' (引言) section of the target manual (SX322008). E1 and E2 show title and revision tables, while E3-E5 are empty or lack content. E6-E8 are from different manuals. The expected facts ('在文档中，对客户端系统的各项功能进行了详细而具体的说明和操作描述' and '通过本文档，用户可以了解服务端、客户端的所有功能和使用方法') are not present in the supplied evidence. Therefore, the model's refusal to answer based on lack of evidence is appropriate and not a hallucination.

## Offline Interface — sampled manual revisions

80 cases from 4 CSV files. Set `OFFLINE_INTERFACE_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
OFFLINE_INTERFACE_SAMPLE_CASES = [{'id': 'sample_offline_interface_b155ae970e_01', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“概要”中“该手册说明了C#及C++版离线编程接口库的接口函数的使用方法”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['该手册说明了C#及C++版离线编程接口库的接口函数的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 690, 'line_end': 690, 'excerpt': '该手册说明了C#及C++版离线编程接口库的接口函数的使用方法。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_02', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.10', 'section_title': '5.10 INST', 'heading_path': ['5 结构体描述', '5.10 INST'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“INST”中“功能描述”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 功能描述：RC指令结构体'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9589, 'line_end': 9589, 'excerpt': '1. 功能描述：RC指令结构体。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_03', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.2.2', 'section_title': '2.2.2 使用C++编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.2 使用C++编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“使用C++编写”中，应如何理解或操作“打开（新建）项目”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['打开（新建）项目，右键工程项目属性，找到C/C++,点击“附加包含目录”，选择存储头文件的路径，如图2-2所示', '找到链接器中的常规，附加库目录，选择存储RobotSDK.lib文件的路径，如图2-3所示', '点击链接器中的输入，在如图2-4标记位置输入相应的lib文件'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 720, 'line_end': 720, 'excerpt': '打开（新建）项目，右键工程项目属性，找到C/C++,点击“附加包含目录”，选择存储头文件的路径，如图2-2所示；找到链接器中的常规，附加库目录，选择存储RobotSDK.lib文件的路径，如图2-3所示；点击链接器中的输入，在如图2-4标记位置输入相应的lib文件。如果使用64位平台，需要在项目中添加#include“Windows.h”。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_04', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.7', 'section_title': '5.7 RPY', 'heading_path': ['5 结构体描述', '5.7 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），配置“RPY”时，“float ry”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['float ry; //方向变量y，单位度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9551, 'line_end': 9551, 'excerpt': 'float ry; //方向变量y，单位度', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_05', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.6.5', 'section_title': '4.6.5 ARCOFFSet', 'heading_path': ['4 C++接口函数介绍', '4.6 应用类指令（弧焊应用）', '4.6.5 ARCOFFSet'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），配置“ARCOFFSet”时，“float varU = 10”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['float varU = 10; //电压范围10~130'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 8269, 'line_end': 8269, 'excerpt': 'float varU = 10; //电压范围10~130', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_06', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.4.15', 'section_title': '3.4.15 Delay', 'heading_path': ['3 C#接口函数介绍', '3.4 执行类指令', '3.4.15 Delay'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），配置“Delay”时，“DelayTime(in) 延迟时间”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['DelayTime(in) 延迟时间，单位: s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 3282, 'line_end': 3282, 'excerpt': 'DelayTime(in) 延迟时间，单位: s；', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_07', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.6.13', 'section_title': '3.6.13 WVLIBON', 'heading_path': ['3 C#接口函数介绍', '3.6 应用类指令（弧焊应用）', '3.6.13 WVLIBON'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“WVLIBON”中“功能描述”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 功能描述：调用摆动库参数来进行摆动，在控制器端需要缓存到指令执行队列'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 4245, 'line_end': 4245, 'excerpt': '1. 功能描述：调用摆动库参数来进行摆动，**在控制器端需要缓存到指令执行队列**。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_08', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“错误码描述”中“-26”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['-26 加载指令配置文件失败'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9939, 'line_end': 9942, 'excerpt': '<tr>\n<td><p><strong>&#45;26</strong></p></td>\n<td><p>加载指令配置文件失败</p></td>\n</tr>', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_09', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.2.2', 'section_title': '2.2.2 使用C++编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.2 使用C++编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“使用C++编写”中“Demo程序用使用C++编写”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['VS2013'], 'expected_facts': ['Demo程序用使用C++编写，Microsoft Visual Studio 2013编译，其他版本可能会出现问题', '运行需要VS2013运行', '离线接口库RobotSDK.dll文件，使用C++编写，Microsoft Visual Studio 2013编译'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 718, 'line_end': 718, 'excerpt': 'Demo程序用使用C++编写，Microsoft Visual Studio 2013编译，其他版本可能会出现问题。运行需要VS2013运行。离线接口库RobotSDK.dll文件，使用C++编写，Microsoft Visual Studio 2013编译。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_10', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.3.5', 'section_title': '4.3.5 ReadAlarm', 'heading_path': ['4 C++接口函数介绍', '4.3 查询类指令', '4.3.5 ReadAlarm'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“ReadAlarm”中，应如何理解或操作“int errorCode = ReadAlarm(ErrCode”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int errorCode = ReadAlarm(ErrCode,conncetID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 6450, 'line_end': 6450, 'excerpt': 'int errorCode = ReadAlarm(ErrCode,conncetID);', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_11', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '4.4.4', 'section_title': '4.4.4 DelJob', 'heading_path': ['4 C++接口函数介绍', '4.4 执行类指令', '4.4.4 DelJob'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“DelJob”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ROBOTSDK'], 'expected_facts': ['2. 函数声明：ROBOTSDK\\_API int DelJob(char \\* jobName ,intConnectID)'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 7177, 'line_end': 7177, 'excerpt': '2. 函数声明：ROBOTSDK\\_API int DelJob(char \\* jobName ,intConnectID)；', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_12', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.8', 'section_title': '5.8 ParameterType', 'heading_path': ['5 结构体描述', '5.8 ParameterType'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“ParameterType”中，应如何理解或操作“INT\\_4”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['INT\\_4, //整形'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9566, 'line_end': 9566, 'excerpt': 'INT\\_4, //整形', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_13', 'category': 'offline_interface', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），操作“错误码描述”时，涉及“-13”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['-13 机器人处于执行状态，禁止该功能'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9887, 'line_end': 9890, 'excerpt': '<tr>\n<td><p><strong>&#45;13</strong></p></td>\n<td><p>机器人处于执行状态，禁止该功能</p></td>\n</tr>', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_14', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.5.4', 'section_title': '3.5.4 DGO\\_Out8', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.4 DGO\\_Out8'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“DGO\\_Out8”中，应如何理解或操作“Channel(in) 组号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['Channel(in) 组号，有效组号1-8'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 3602, 'line_end': 3602, 'excerpt': 'Channel(in) 组号，有效组号1-8；', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_15', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '7.3', 'section_title': '7.3 RPY', 'heading_path': ['7 概念解释', '7.3 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': "根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“RPY”中，应如何理解或操作“x'y'z'坐标系最初与世界坐标系原点位置重合”？请解释该项的操作方法、用途或结果。", 'anchors': [], 'expected_facts': ["x'y'z'坐标系最初与世界坐标系原点位置重合，各轴方向一致", "x'y'z'移动到图4-2所示位置，需要经过位置变化和方向变化", '位置变化用px、py、pz表示，方向变化用rx、ry、rz表示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9991, 'line_end': 9991, 'excerpt': "x'y'z'坐标系最初与世界坐标系原点位置重合，各轴方向一致。x'y'z'移动到图4-2所示位置，需要经过位置变化和方向变化。位置变化用px、py、pz表示，方向变化用rx、ry、rz表示：", 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_16', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“错误码描述”中，应如何理解或操作“-14”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['-14 远程控制功能未启动'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9891, 'line_end': 9894, 'excerpt': '<tr>\n<td><p><strong>&#45;14</strong></p></td>\n<td><p>远程控制功能未启动</p></td>\n</tr>', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_17', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '7.2', 'section_title': '7.2 NOAP（T6）', 'heading_path': ['7 概念解释', '7.2 NOAP（T6）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': "根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），配置“NOAP（T6）”时，“O的三个单位向量表示出y'相对世界坐标系的方向”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。", 'anchors': [], 'expected_facts': ["O的三个单位向量表示出y'相对世界坐标系的方向"], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9981, 'line_end': 9981, 'excerpt': "O的三个单位向量表示出y'相对世界坐标系的方向；", 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_18', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.9', 'section_title': '5.9 INST\\_PARA', 'heading_path': ['5 结构体描述', '5.9 INST\\_PARA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“INST\\_PARA”中“char val[100]”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['char val[100]; //RC指令参数值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9581, 'line_end': 9581, 'excerpt': 'char val[100]; //RC指令参数值', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_19', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2.4.2', 'section_title': '2.4.2 远程控制准备', 'heading_path': ['2 使用前准备', '2.4 下位机准备', '2.4.2 远程控制准备'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），在“远程控制准备”中，应如何理解或操作“将外部启动配置成1013”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['将外部启动配置成1013，外部上电配成1016 ,其余保存默认值1281'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 763, 'line_end': 763, 'excerpt': '**将外部启动配置成1013，外部上电配成1016 ,其余保存默认值1281。**', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_b155ae970e_20', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '5.16', 'section_title': '5.16 WELD\\_CRAFT\\_PARA', 'heading_path': ['5 结构体描述', '5.16 WELD\\_CRAFT\\_PARA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），配置“WELD\\_CRAFT\\_PARA”时，“功能描述”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 功能描述：弧焊工艺参数结构体'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'line_start': 9735, 'line_end': 9735, 'excerpt': '1. 功能描述：弧焊工艺参数结构体。', 'sha256': '879b9a05d69bc3470825c61682fc579436e5e9bc241827acc570265b9992a1bb'}]}, {'id': 'sample_offline_interface_bb17a82e78_01', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '4.4.43', 'section_title': '4.4.43 SendPlayStart', 'heading_path': ['4 C++接口函数介绍', '4.4 执行类指令', '4.4.43 SendPlayStart'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“SendPlayStart”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ROBOTSDK'], 'expected_facts': ['2. 函数声明：ROBOTSDK\\_API int SendPlayStart(int ConnectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 12623, 'line_end': 12623, 'excerpt': '2. 函数声明：ROBOTSDK\\_API int SendPlayStart(int ConnectID);', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_02', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '7.2', 'section_title': '7.2 NOAP（T6）', 'heading_path': ['7 概念解释', '7.2 NOAP（T6）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': "根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），配置“NOAP（T6）”时，“A的三个单位向量表示出z'相对世界坐标系的方向”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。", 'anchors': [], 'expected_facts': ["A的三个单位向量表示出z'相对世界坐标系的方向"], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15511, 'line_end': 15511, 'excerpt': "A的三个单位向量表示出z'相对世界坐标系的方向；", 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_03', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '7.3', 'section_title': '7.3 RPY', 'heading_path': ['7 概念解释', '7.3 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': "根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“RPY”中，应如何理解或操作“rx表示x'y'z'绕世界坐标系x轴旋转的角度”？请解释该项的操作方法、用途或结果。", 'anchors': [], 'expected_facts': ["rx表示x'y'z'绕世界坐标系x轴旋转的角度"], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15521, 'line_end': 15521, 'excerpt': "rx表示x'y'z'绕世界坐标系x轴旋转的角度；", 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_04', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“错误码描述”中“未初始化”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['-6 未初始化 RobotInit接口调用失败，查看可执行目录下是否存在SysConfig.xml文档'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15377, 'line_end': 15381, 'excerpt': '<tr>\n<td><p>&#45;6</p></td>\n<td><p>未初始化</p></td>\n<td><p>RobotInit接口调用失败，查看可执行目录下是否存在SysConfig&#46;xml文档</p></td>\n</tr>', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_05', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '2.2.2', 'section_title': '2.2.2 使用C++编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.2 使用C++编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“使用C++编写”中“需要把接口库中的config文件夹、icon文件夹、Position.bak、job\\_to\\_”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['需要把接口库中的config文件夹、icon文件夹、Position.bak、job\\_to\\_txt.exe、RobotSDK.dll、LicenceManager.dll、SysConfig.xml、InstConfig.xml、SiaSunRobot.lic拷贝到可执行程序输出目录下', '把Robot\\_Datatype.h，RobotSDK.h、RobotSDK.lib文件放到代码目录下'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 1014, 'line_end': 1014, 'excerpt': '需要把接口库中的config文件夹、icon文件夹、Position.bak、job\\_to\\_txt.exe、RobotSDK.dll、LicenceManager.dll、SysConfig.xml、InstConfig.xml、SiaSunRobot.lic拷贝到可执行程序输出目录下。把Robot\\_Datatype.h，RobotSDK.h、RobotSDK.lib文件放到代码目录下。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_06', 'category': 'offline_interface', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），操作“错误码描述”时，涉及“-35”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['-35 机器人禁止此操作 机器人处于运动中，机器人作业执行中'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15477, 'line_end': 15481, 'excerpt': '<tr>\n<td><p>&#45;35</p></td>\n<td><p>机器人禁止此操作</p></td>\n<td><p>机器人处于运动中，机器人作业执行中</p></td>\n</tr>', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_07', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '4.3.4', 'section_title': '4.3.4 ReadTool', 'heading_path': ['4 C++接口函数介绍', '4.3 查询类指令', '4.3.4 ReadTool'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），配置“ReadTool”时，“Rz(out) 工具坐标参数Rz分量”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['Rz(out) 工具坐标参数Rz分量'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 9365, 'line_end': 9365, 'excerpt': 'Rz(out) 工具坐标参数Rz分量；', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_08', 'category': 'offline_interface', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '3.5.13', 'section_title': '3.5.13 WriteUOMulti', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.13 WriteUOMulti'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），操作“WriteUOMulti”时，涉及“使用时注意此用户IO输出是否已被其他设备等占用”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['使用时注意此用户IO输出是否已被其他设备等占用，避免发生错误'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 6179, 'line_end': 6179, 'excerpt': '使用时注意此用户IO输出是否已被其他设备等占用，避免发生错误。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_09', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“错误码描述”中，应如何理解或操作“发送失败返回值，代表网络断开”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['-2 发送失败返回值，代表网络断开 网络通讯异常，检查网线是否虚连，通过ping命令检查'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15357, 'line_end': 15361, 'excerpt': '<tr>\n<td><p>&#45;2</p></td>\n<td><p>发送失败返回值，代表网络断开</p></td>\n<td><p>网络通讯异常，检查网线是否虚连，通过ping命令检查。</p></td>\n</tr>', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_10', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '3.5.14', 'section_title': '3.5.14 WriteGUOMulti', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.14 WriteGUOMulti'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），配置“WriteGUOMulti”时，“channel(in) IO通道号起始编号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['channel(in) IO通道号起始编号，8位：取值范围1-8', '16位：取值范围1-256'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 6221, 'line_end': 6221, 'excerpt': 'channel(in) IO通道号起始编号，8位：取值范围1-8；16位：取值范围1-256；', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_11', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '3.4.19', 'section_title': '3.4.19 SetAxisInsertData', 'heading_path': ['3 C#接口函数介绍', '3.4 执行类指令', '3.4.19 SetAxisInsertData'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“SetAxisInsertData”中，应如何理解或操作“功能描述”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 功能描述：设置轴干涉区'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 4813, 'line_end': 4813, 'excerpt': '1. 功能描述：设置轴干涉区。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_12', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '4.4.29', 'section_title': '4.4.29 SendJogMotion', 'heading_path': ['4 C++接口函数介绍', '4.4 执行类指令', '4.4.29 SendJogMotion'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“SendJogMotion”中，应如何理解或操作“errorCode = SendJogMotion(type”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['errorCode = SendJogMotion(type, dir, ConnectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 12073, 'line_end': 12073, 'excerpt': 'errorCode = SendJogMotion(type, dir, ConnectID);', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_13', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '2.2.1', 'section_title': '2.2.1 使用C#编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.1 使用C#编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“使用C#编写”中，应如何理解或操作“引用时需用将RobotSDKWrapper.dll文件拷贝到某一目录下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['引用时需用将RobotSDKWrapper.dll文件拷贝到某一目录下，打开工程，点击"引用",右键，点击"添加引用"，弹出如图2-1所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 998, 'line_end': 998, 'excerpt': '引用时需用将RobotSDKWrapper.dll文件拷贝到某一目录下，打开工程，点击"引用",右键，点击"添加引用"，弹出如图2-1所示：', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_14', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '3.2.13', 'section_title': '3.2.13 Movj\\_P2P\\_NoEndBack', 'heading_path': ['3 C#接口函数介绍', '3.2 运动类指令', '3.2.13 Movj\\_P2P\\_NoEndBack'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“Movj\\_P2P\\_NoEndBack”中“功能描述”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 功能描述：单点关节型运动，在控制器端需要缓存到指令执行队列，不带执行完成反馈'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 2054, 'line_end': 2054, 'excerpt': '1. 功能描述：单点关节型运动，在控制器端需要缓存到指令执行队列，不带执行完成反馈。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_15', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“概要”中“该手册说明了C#及C++版离线编程接口库的接口函数的使用方法”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['该手册说明了C#及C++版离线编程接口库的接口函数的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 956, 'line_end': 956, 'excerpt': '该手册说明了C#及C++版离线编程接口库的接口函数的使用方法。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_16', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '5.7', 'section_title': '5.7 RPY', 'heading_path': ['5 结构体描述', '5.7 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），配置“RPY”时，“float rx”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['float rx; //方向变量x，单位度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15052, 'line_end': 15052, 'excerpt': 'float rx; //方向变量x，单位度', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_17', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '5.9', 'section_title': '5.9 INST\\_PARA', 'heading_path': ['5 结构体描述', '5.9 INST\\_PARA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“INST\\_PARA”中“char minVal[100]”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['char minVal[100]; // RC指令参数最小值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15086, 'line_end': 15086, 'excerpt': 'char minVal[100]; // RC指令参数最小值', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_18', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），配置“错误码描述”时，“返回数据无效”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['-5 返回数据无效 返回数据不在有效范围内'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15372, 'line_end': 15376, 'excerpt': '<tr>\n<td><p>&#45;5</p></td>\n<td><p>返回数据无效</p></td>\n<td><p>返回数据不在有效范围内</p></td>\n</tr>', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_19', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '5.12', 'section_title': '5.12 STACK\\_TYPE', 'heading_path': ['5 结构体描述', '5.12 STACK\\_TYPE'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），在“STACK\\_TYPE”中，应如何理解或操作“功能描述”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 功能描述：垛型枚举'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 15148, 'line_end': 15148, 'excerpt': '1. 功能描述：垛型枚举。', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_bb17a82e78_20', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / B-4', 'section': '4.3.12', 'section_title': '4.3.12 ReadJointBaseNOAP', 'heading_path': ['4 C++接口函数介绍', '4.3 查询类指令', '4.3.12 ReadJointBaseNOAP'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'source_version': '', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-4)》（V4.0 / B-4），“ReadJointBaseNOAP”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ROBOTSDK'], 'expected_facts': ['2. 函数声明：ROBOTSDK\\_API int ReadJointBaseNOAP(float \\*N,float \\*O,float \\*A,float \\*P,int jointNum,float \\*jointVal,int ConnectID)'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'line_start': 9602, 'line_end': 9602, 'excerpt': '2. 函数声明：ROBOTSDK\\_API int ReadJointBaseNOAP(float \\*N,float \\*O,float \\*A,float \\*P,int jointNum,float \\*jointVal,int ConnectID)；', 'sha256': 'f3942c59f13d71dd5a46831afc5ee4c3f64df66cdb4d353b55df409227333257'}]}, {'id': 'sample_offline_interface_3e18c9c39a_01', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '2.1.1', 'section_title': '2.1.1 C++版本', 'heading_path': ['2 使用前准备', '2.1 系统环境要求', '2.1.1 C++版本'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“C++版本”中，应如何理解或操作“软件要求”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 软件要求：gcc 11.4.0，python 3.10.12'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 1045, 'line_end': 1045, 'excerpt': '3. 软件要求：gcc 11.4.0，python 3.10.12', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_02', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '5.9', 'section_title': '5.9 INST\\_PARA', 'heading_path': ['5 结构体描述', '5.9 INST\\_PARA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“INST\\_PARA”中“ParameterType paraType”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['ParameterType paraType; //RC指令参数类型'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 17833, 'line_end': 17833, 'excerpt': 'ParameterType paraType; //RC指令参数类型', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_03', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '3.5.5', 'section_title': '3.5.5 WriteGUO', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.5 WriteGUO'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“WriteGUO”中，应如何理解或操作“int result = RobotSdkWrapper. WriteGUO (IoStatus”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int result = RobotSdkWrapper. WriteGUO (IoStatus ,Channel, IOGroupFlag ,connectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 6054, 'line_end': 6054, 'excerpt': 'int result = RobotSdkWrapper. WriteGUO (IoStatus ,Channel, IOGroupFlag ,connectID);', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_04', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“概要”中“该手册说明了C#及C++版离线编程接口库的接口函数的使用方法”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['该手册说明了C#及C++版离线编程接口库的接口函数的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 1027, 'line_end': 1027, 'excerpt': '该手册说明了C#及C++版离线编程接口库的接口函数的使用方法。', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_05', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '4.5.11', 'section_title': '4.5.11 ReadUI', 'heading_path': ['4 C++接口函数介绍', '4.5 IO类指令', '4.5.11 ReadUI'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），配置“ReadUI”时，“IOMap(in) IO索引号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['IOMap(in) IO索引号，取值范围1-4096'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 14511, 'line_end': 14511, 'excerpt': 'IOMap(in) IO索引号，取值范围1-4096；；', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_06', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“错误码描述”中“错误码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SOCKETID'], 'expected_facts': ['错误码 错误码定义 说明 0 函数调用成功 -1 发送失败返回值，代表应答超时 网络通讯异常，检查网线是否虚连，通过ping命令检查', '-2 发送失败返回值，代表网络断开 网络通讯异常，检查网线是否虚连，通过ping命令检查', '-3 参数无效 输入参数不在取值范围内，检查参数值 -4 返回数据校验码错误 网络异常，数据丢失 -5 返回数据无效 返回数据不在有效范围内 -6 未初始化 RobotInit接口调用失败，查看可执行目录下是否存在SysConfig.xml文档 -9 控制器状态错误 机器人端有报警，需要清除报警 -11 作业空间已满 机器人存储作业空间无剩余，可删除部分作业 -13 机器人处于执行状态，禁止该功能 机器人处于执行状态，不能调用该功能接口 -14 远程控制功能未启动 -15 RC不支持该功能 离线接口库版本和机器人程序版本不匹配，机器人程序版本较低 -16 配置文件打开失败 -17 接收数据包大小为0 网络异常，数据丢失 -18 接收数据包消息体长度错误 网络异常，数据丢失 -19 接收数据包对象为空 网络异常，数据丢失 -20 子功能码错误 传输数据有问题 -21 SOCKETID错误 该网络连接ID不存在，与机器人断开 -22 授权文件无效 授权文件SiasunRobot.lic不存在，或者时间过期或者机器人主板信息不符 -25 加载系统配置文件失败 -26 加载指令配置文件失败 -30 数据长度错误 传输数据有问题 -31 作业转换失败 -32 文件路径不能包含空格 -33 作业转换软件初始化失败 -34 日志文件创建失败 -35 机器人禁止此操作 机器人处于运动中，机器人作业执行中 -36 作业名不合法 机器人作业名只允许大写字母、数字、下划线'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 18216, 'line_end': 18216, 'excerpt': '<table><tr><td><p><strong>错误码</strong></p></td><td><p><strong>错误码定义</strong></p></td><td><p><strong>说明</strong></p></td></tr><tr><td><p>0</p></td><td><p>函数调用成功</p></td><td></td></tr><tr><td><p>-1</p></td><td><p>发送失败返回值，代表应答超时</p></td><td><p>网络通讯异常，检查网线是否虚连，通过ping命令检查。</p></td></tr><tr><td><p>-2</p></td><td><p>发送失败返回值，代表网络断开</p></td><td><p>网络通讯异常，检查网线是否虚连，通过ping命令检查。</p></td></tr><tr><td><p>-3</p></td><td><p>参数无效</p></td><td><p>输入参数不在取值范围内，检查参数值</p></td></tr><tr><td><p>-4</p></td><td><p>返回数据校验码错误</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-5</p></td><td><p>返回数据无效</p></td><td><p>返回数据不在有效范围内</p></td></tr><tr><td><p>-6</p></td><td><p>未初始化</p></td><td><p>RobotInit接口调用失败，查看可执行目录下是否存在SysConfig.xml文档</p></td></tr><tr><td><p>-9</p></td><td><p>控制器状态错误</p></td><td><p>机器人端有报警，需要清除报警</p></td></tr><tr><td><p>-11</p></td><td><p>作业空间已满</p></td><td><p>机器人存储作业空间无剩余，可删除部分作业</p></td></tr><tr><td><p>-13</p></td><td><p>机器人处于执行状态，禁止该功能</p></td><td><p>机器人处于执行状态，不能调用该功能接口</p></td></tr><tr><td><p>-14</p></td><td><p>远程控制功能未启动</p></td><td></td></tr><tr><td><p>-15</p></td><td><p>RC不支持该功能</p></td><td><p>离线接口库版本和机器人程序版本不匹配，机器人程序版本较低</p></td></tr><tr><td><p>-16</p></td><td><p>配置文件打开失败</p></td><td></td></tr><tr><td><p>-17</p></td><td><p>接收数据包大小为0</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-18</p></td><td><p>接收数据包消息体长度错误</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-19</p></td><td><p>接收数据包对象为空</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-20</p></td><td><p>子功能码错误</p></td><td><p>传输数据有问题</p></td></tr><tr><td><p>-21</p></td><td><p>SOCKETID错误</p></td><td><p>该网络连接ID不存在，与机器人断开</p></td></tr><tr><td><p>-22</p></td><td><p>授权文件无效</p></td><td><p>授权文件SiasunRobot.lic不存在，或者时间过期或者机器人主板信息不符</p></td></tr><tr><td><p>-25</p></td><td><p>加载系统配置文件失败</p></td><td></td></tr><tr><td><p>-26</p></td><td><p>加载指令配置文件失败</p></td><td></td></tr><tr><td><p>-30</p></td><td><p>数据长度错误</p></td><td><p>传输数据有问题</p></td></tr><tr><td><p>-31</p></td><td><p>作业转换失败</p></td><td></td></tr><tr><td><p>-32</p></td><td><p>文件路径不能包含空格</p></td><td></td></tr><tr><td><p>-33</p></td><td><p>作业转换软件初始化失败</p></td><td></td></tr><tr><td><p>-34</p></td><td><p>日志文件创建失败</p></td><td></td></tr><tr><td><p>-35</p></td><td><p>机器人禁止此操作</p></td><td><p>机器人处于运动中，机器人作业执行中</p></td></tr><tr><td><p>-36</p></td><td><p>作业名不合法</p></td><td><p>机器人作业名只允许大写字母、数字、下划线</p></td></tr></table>', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_07', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '4.6.9', 'section_title': '4.6.9 ARCL\\_P', 'heading_path': ['4 C++接口函数介绍', '4.6 应用类指令（弧焊应用）', '4.6.9 ARCL\\_P'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“ARCL\\_P”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ROBOTSDK', 'ARCL'], 'expected_facts': ['2. 函数声明：ROBOTSDK\\_API int ARCL\\_P(short posNum,float speed,short acc,short cnt,int ConnectID)'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 15142, 'line_end': 15142, 'excerpt': '2. 函数声明：ROBOTSDK\\_API int ARCL\\_P(short posNum,float speed,short acc,short cnt,int ConnectID)；', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_08', 'category': 'offline_interface', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '3.5.13', 'section_title': '3.5.13 WriteUOMulti', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.13 WriteUOMulti'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），操作“WriteUOMulti”时，涉及“使用时注意此用户IO输出是否已被其他设备等占用”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['使用时注意此用户IO输出是否已被其他设备等占用，避免发生错误'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 6293, 'line_end': 6293, 'excerpt': '使用时注意此用户IO输出是否已被其他设备等占用，避免发生错误。', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_09', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '2.2.1', 'section_title': '2.2.1 使用C#编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.1 使用C#编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“使用C#编写”中“需要把接口库中的config文件夹、icon文件夹、Position.bak、 job\\_to\\”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['需要把接口库中的config文件夹、icon文件夹、Position.bak、 job\\_to\\_txt.exe、'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 1065, 'line_end': 1065, 'excerpt': '需要把接口库中的config文件夹、icon文件夹、Position.bak、 job\\_to\\_txt.exe、', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_10', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '4.3.17', 'section_title': '4.3.17 ReadJobNameList', 'heading_path': ['4 C++接口函数介绍', '4.3 查询类指令', '4.3.17 ReadJobNameList'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“ReadJobNameList”中，应如何理解或操作“//获得jobNameList”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['//获得jobNameList，作业名最长8位+1位空格，每9位截取一个作业名'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 11413, 'line_end': 11413, 'excerpt': '//获得jobNameList，作业名最长8位+1位空格，每9位截取一个作业名', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_11', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '5.7', 'section_title': '5.7 RPY', 'heading_path': ['5 结构体描述', '5.7 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），配置“RPY”时，“float rz”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['float rz; //方向变量z，单位度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 17809, 'line_end': 17809, 'excerpt': 'float rz; //方向变量z，单位度', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_12', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '5.25', 'section_title': '5.25 Wave\\_File\\_Param\\_TYPE2', 'heading_path': ['5 结构体描述', '5.25 Wave\\_File\\_Param\\_TYPE2'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“Wave\\_File\\_Param\\_TYPE2”中，应如何理解或操作“float rad”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['float rad; //半径（0-100mm）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 18149, 'line_end': 18149, 'excerpt': 'float rad; //半径（0-100mm）', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_13', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '3.5.2', 'section_title': '3.5.2 DO\\_Out', 'heading_path': ['3 C#接口函数介绍', '3.5 IO类指令', '3.5.2 DO\\_Out'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），配置“DO\\_Out”时，“IOMap(in) IO索引号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['IOMap(in) IO索引号，取值范围1-4096'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 5944, 'line_end': 5944, 'excerpt': 'IOMap(in) IO索引号，取值范围1-4096；', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_14', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '7.3', 'section_title': '7.3 RPY', 'heading_path': ['7 概念解释', '7.3 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“RPY”中，应如何理解或操作“如图7-2所示”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ["如图7-2所示，目标点坐标系（x'y'z'）相对世界坐标系的位姿"], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 18246, 'line_end': 18246, 'excerpt': "如图7-2所示，目标点坐标系（x'y'z'）相对世界坐标系的位姿。", 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_15', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '3.4.15', 'section_title': '3.4.15 Send\\_ShiftOnPOff', 'heading_path': ['3 C#接口函数介绍', '3.4 执行类指令', '3.4.15 Send\\_ShiftOnPOff'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“Send\\_ShiftOnPOff”中“参数说明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['3. 参数说明：ConnectID(in) 网络连接ID，ConnectUDP接口的返回值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 4731, 'line_end': 4731, 'excerpt': '3. 参数说明：ConnectID(in) 网络连接ID，ConnectUDP接口的返回值；', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_16', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '7.2', 'section_title': '7.2 NOAP（T6）', 'heading_path': ['7 概念解释', '7.2 NOAP（T6）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': "根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），配置“NOAP（T6）”时，“O的三个单位向量表示出y'相对世界坐标系的方向”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。", 'anchors': [], 'expected_facts': ["O的三个单位向量表示出y'相对世界坐标系的方向"], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 18238, 'line_end': 18238, 'excerpt': "O的三个单位向量表示出y'相对世界坐标系的方向；", 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_17', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '2.2.1', 'section_title': '2.2.1 使用C#编写', 'heading_path': ['2 使用前准备', '2.2 上位机准备', '2.2.1 使用C#编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“使用C#编写”中，应如何理解或操作“点击“浏览”按钮”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击“浏览”按钮，找到存储RobotSDKWrapper.dll文件的目录，选择该文件，点击OK按钮即可'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 1075, 'line_end': 1075, 'excerpt': '点击“浏览”按钮，找到存储RobotSDKWrapper.dll文件的目录，选择该文件，点击OK按钮即可。', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_18', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '5.9', 'section_title': '5.9 INST\\_PARA', 'heading_path': ['5 结构体描述', '5.9 INST\\_PARA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），“INST\\_PARA”中“char minVal[100]”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['char minVal[100]; // RC指令参数最小值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 17839, 'line_end': 17839, 'excerpt': 'char minVal[100]; // RC指令参数最小值', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_19', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '3.2.5', 'section_title': '3.2.5 MOVJ2', 'heading_path': ['3 C#接口函数介绍', '3.2 运动类指令', '3.2.5 MOVJ2'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），在“MOVJ2”中，应如何理解或操作“int result = RobotSdkWrapper.MOVJ2(ref joints[0]”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int result = RobotSdkWrapper.MOVJ2(ref joints[0], tool, user ,sp, acc ,cnt,connectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 1589, 'line_end': 1589, 'excerpt': 'int result = RobotSdkWrapper.MOVJ2(ref joints[0], tool, user ,sp, acc ,cnt,connectID);', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_3e18c9c39a_20', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-7', 'section': '4.3.7', 'section_title': '4.3.7 ReadUserVarBYTE', 'heading_path': ['4 C++接口函数介绍', '4.3 查询类指令', '4.3.7 ReadUserVarBYTE'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'source_version': 'rc4', 'question': '根据《SX322009《新松机器人离线接口库使用手册》(B-7)》（V5.0 / B-7），配置“ReadUserVarBYTE”时，“varID(in)变量编号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['varID(in)变量编号，取值范围1-100'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'line_start': 11024, 'line_end': 11024, 'excerpt': 'varID(in)变量编号，取值范围1-100；', 'sha256': 'aeb40bbb5df60af9ec010b92b5871286f71f1d00a89f8e90fd23cc824bd4d0ef'}]}, {'id': 'sample_offline_interface_287b7e4c03_01', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.11', 'section_title': '5.11 DHParam', 'heading_path': ['5 结构体描述', '5.11 DHParam'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），配置“DHParam”时，“功能描述”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 功能描述：DH参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 15999, 'line_end': 15999, 'excerpt': '1. 功能描述：DH参数', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_02', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.4.53', 'section_title': '4.4.53 TransJob\\_Rotate', 'heading_path': ['4 C#接口函数介绍', '4.4 执行类指令', '4.4.53 TransJob\\_Rotate'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“TransJob\\_Rotate”中，应如何理解或操作“int[] tgtNum = new int[3]”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int[] tgtNum = new int[3];'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 15733, 'line_end': 15733, 'excerpt': 'int[] tgtNum = new int[3];', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_03', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1', 'section_title': '1 概要', 'heading_path': ['1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“概要”中“该手册说明了C++及C#版离线编程接口库的接口函数的使用方法”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['该手册说明了C++及C#版离线编程接口库的接口函数的使用方法'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 1061, 'line_end': 1061, 'excerpt': '该手册说明了C++及C#版离线编程接口库的接口函数的使用方法。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_04', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.4.54', 'section_title': '3.4.54 TransJob\\_Mirror', 'heading_path': ['3 C++接口函数介绍', '3.4 执行类指令', '3.4.54 TransJob\\_Mirror'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“TransJob\\_Mirror”中，应如何理解或操作“coorNum[2] = 0”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['coorNum[2] = 0;'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 8686, 'line_end': 8686, 'excerpt': 'coorNum[2] = 0;', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_05', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.4.25', 'section_title': '4.4.25 DeleteBreakpoint', 'heading_path': ['4 C#接口函数介绍', '4.4 执行类指令', '4.4.25 DeleteBreakpoint'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“DeleteBreakpoint”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 函数声明：int DeleteBreakpoint(string jobName, int lineNum, int connectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 14967, 'line_end': 14967, 'excerpt': '2. 函数声明：int DeleteBreakpoint(string jobName, int lineNum, int connectID);', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_06', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.5.3', 'section_title': '2.5.3 执行作业', 'heading_path': ['2 使用前准备', '2.5 接口库使用建议', '2.5.3 执行作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“执行作业”中，应如何理解或操作“在程序逻辑中”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['7. 在程序逻辑中，如果需要急停：Send\\_Stop，取消急停：Reset\\_Stop、Send\\_Reset'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 1171, 'line_end': 1171, 'excerpt': '7. 在程序逻辑中，如果需要急停：Send\\_Stop，取消急停：Reset\\_Stop、Send\\_Reset。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_07', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6', 'section_title': '6 错误码描述', 'heading_path': ['6 错误码描述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“错误码描述”中“错误码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SFTP', 'SOCKETID'], 'expected_facts': ['错误码 错误码定义 说明 0 函数调用成功 -1 发送失败返回值，代表应答超时 网络通讯异常，检查网线是否虚连，通过ping命令检查', '-2 发送失败返回值，代表网络断开 网络通讯异常，检查网线是否虚连，通过ping命令检查', '-3 参数无效 输入参数不在取值范围内，检查参数值 -4 返回数据校验码错误 网络异常，数据丢失 -5 返回数据无效 返回数据不在有效范围内 -6 未初始化 在调用接口前，调用RobotInit进行初始化'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 16049, 'line_end': 16049, 'excerpt': '<table><thead><tr><th><p><strong>错误码</strong></p></th><th><p><strong>错误码定义</strong></p></th><th><p><strong>说明</strong></p></th></tr></thead><tbody><tr><td><p>0</p></td><td><p>函数调用成功</p></td><td></td></tr><tr><td><p>-1</p></td><td><p>发送失败返回值，代表应答超时</p></td><td><p>网络通讯异常，检查网线是否虚连，通过ping命令检查。</p></td></tr><tr><td><p>-2</p></td><td><p>发送失败返回值，代表网络断开</p></td><td><p>网络通讯异常，检查网线是否虚连，通过ping命令检查。</p></td></tr><tr><td><p>-3</p></td><td><p>参数无效</p></td><td><p>输入参数不在取值范围内，检查参数值</p></td></tr><tr><td><p>-4</p></td><td><p>返回数据校验码错误</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-5</p></td><td><p>返回数据无效</p></td><td><p>返回数据不在有效范围内</p></td></tr><tr><td><p>-6</p></td><td><p>未初始化</p></td><td><p>在调用接口前，调用RobotInit进行初始化；</p><p>SFTP传输文件前，调用ConnectSFtp进行初始化</p></td></tr><tr><td><p>-9</p></td><td><p>控制器状态错误</p></td><td><p>机器人端有报警，需要清除报警</p></td></tr><tr><td><p>-11</p></td><td><p>作业空间已满</p></td><td><p>机器人存储作业空间无剩余，可删除部分作业</p></td></tr><tr><td><p>-13</p></td><td><p>机器人处于执行状态，禁止该功能</p></td><td><p>机器人处于执行状态，不能调用该功能接口</p></td></tr><tr><td><p>-14</p></td><td><p>远程控制功能未启动</p></td><td><p>调用SetControlPriorty接口切换控制权限</p></td></tr><tr><td><p>-15</p></td><td><p>RC不支持该功能</p></td><td><p>离线接口库版本和机器人程序版本不匹配，机器人程序版本较低</p></td></tr><tr><td><p>-17</p></td><td><p>接收数据包大小为0</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-18</p></td><td><p>接收数据包消息体长度错误</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-19</p></td><td><p>接收数据包对象为空</p></td><td><p>网络异常，数据丢失</p></td></tr><tr><td><p>-20</p></td><td><p>子功能码错误</p></td><td><p>传输数据有问题</p></td></tr><tr><td><p>-21</p></td><td><p>SOCKETID错误</p></td><td><p>该网络连接ID不存在，与机器人断开</p></td></tr><tr><td><p>-22</p></td><td><p>授权文件无效</p></td><td><p>授权文件SiasunRobot.lic不存在，或者时间过期或者机器人主板信息不符</p></td></tr><tr><td><p>-25</p></td><td><p>加载系统配置文件失败</p></td><td><p>检查SysConfig.xml文件</p></td></tr><tr><td><p>-30</p></td><td><p>数据长度错误</p></td><td><p>传输数据有问题</p></td></tr><tr><td><p>-34</p></td><td><p>日志文件创建失败</p></td><td></td></tr><tr><td><p>-35</p></td><td><p>机器人禁止此操作</p></td><td><p>机器人处于运动中，机器人作业执行中</p></td></tr><tr><td><p>-36</p></td><td><p>作业名不合法</p></td><td><p>*.ftj 宏作业 *.txt 示教和后台作业</p><p>作业名只允许字母、数字、下划线</p></td></tr><tr><td><p>-37</p></td><td><p>ftp服务器错误</p></td><td></td></tr><tr><td><p>-38</p></td><td><p>数组长度为0</p></td><td></td></tr><tr><td><p>-39</p></td><td><p>接受消息key不一致</p></td><td></td></tr><tr><td><p>-40</p></td><td><p>SFTP服务错误</p></td><td></td></tr><tr><td><p>-41</p></td><td><p>json转换失败</p></td><td></td></tr><tr><td><p>-42</p></td><td><p>模式错误</p></td><td><p>控制器要求指定的的模式（示教/执行/远程）</p></td></tr><tr><td><p>-43</p></td><td><p>未定义异常</p></td><td></td></tr><tr><td><p>-44</p></td><td><p>数据内容过大</p></td><td><p>单次发送数据过多，减少批量发送的数据个数</p></td></tr><tr><td><p>-45</p></td><td><p>加载授权文件失败</p></td><td><p>检查授权文件是否存在</p></td></tr></tbody></table>', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_08', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.4.30', 'section_title': '3.4.30 StartDebug', 'heading_path': ['3 C++接口函数介绍', '3.4 执行类指令', '3.4.30 StartDebug'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“StartDebug”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 函数声明：int StartDebug(int connectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 7986, 'line_end': 7986, 'excerpt': '2. 函数声明：int StartDebug(int connectID);', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_09', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.2', 'section_title': '5.2 TIME\\_DATA', 'heading_path': ['5 结构体描述', '5.2 TIME\\_DATA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“TIME\\_DATA”中，应如何理解或操作“int month”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int month; //月'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 15873, 'line_end': 15873, 'excerpt': 'int month; //月', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_10', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.5.3', 'section_title': '2.5.3 执行作业', 'heading_path': ['2 使用前准备', '2.5 接口库使用建议', '2.5.3 执行作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“执行作业”中“外部上电”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['4. 外部上电：Send\\_PowerOn，接口调用成功之后，线程停止2000ms，保证硬件上电'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 1168, 'line_end': 1168, 'excerpt': '4. 外部上电：Send\\_PowerOn，接口调用成功之后，线程停止2000ms，保证硬件上电。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_11', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.4', 'section_title': '5.4 SPEED\\_DATA', 'heading_path': ['5 结构体描述', '5.4 SPEED\\_DATA'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“SPEED\\_DATA”中“备注”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['备注：如果速度类型为关节速度，则每个关节都有一个速度，接口ReadSpeed返回值中的speed[12]数组，根据机器人当前的关节数量从索引0开始获取每个关节速度即可', '轨迹线速度，有效速度值只有一个，读取speed[0]即可'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 15901, 'line_end': 15901, 'excerpt': '备注：如果速度类型为关节速度，则每个关节都有一个速度，接口ReadSpeed返回值中的speed[12]数组，根据机器人当前的关节数量从索引0开始获取每个关节速度即可。轨迹线速度，有效速度值只有一个，读取speed[0]即可。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_12', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.3.3', 'section_title': '3.3.3 SetCurToolNum', 'heading_path': ['3 C++接口函数介绍', '3.3 设置类指令', '3.3.3 SetCurToolNum'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），配置“SetCurToolNum”时，“curToolNum(in) 工具坐标系号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['curToolNum(in) 工具坐标系号，取值范围1-32，最大值取决于文件数量扩展界面工具坐标系的数量（默认为8）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 4810, 'line_end': 4810, 'excerpt': 'curToolNum(in) 工具坐标系号，取值范围1-32，最大值取决于文件数量扩展界面工具坐标系的数量（默认为8）；', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_13', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.2.4', 'section_title': '4.2.4 ReadPOSE\\_RPY2', 'heading_path': ['4 C#接口函数介绍', '4.2 查询类指令', '4.2.4 ReadPOSE\\_RPY2'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），配置“ReadPOSE\\_RPY2”时，“toolNum(in) 工具坐标系”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['toolNum(in) 工具坐标系，取值范围1-32，当coordinate为1时，该值有效'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 8943, 'line_end': 8943, 'excerpt': 'toolNum(in) 工具坐标系，取值范围1-32，当coordinate为1时，该值有效；', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_14', 'category': 'offline_interface', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.3', 'section_title': '2.3 网络连接', 'heading_path': ['2 使用前准备', '2.3 网络连接'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），操作“网络连接”时，涉及“注意： 控制周期100ms，在一个控制周期内完成数据的发送和接收。 一个PC端一个应用程序可以连”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意： 控制周期100ms，在一个控制周期内完成数据的发送和接收', '一个PC端一个应用程序可以连接一个或者多个机器人，不允许一个PC端多个应用程序连接同一个机器人，也不允许多个PC端连接同一个机器人'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 1134, 'line_end': 1134, 'excerpt': '<table><tr><td><p><img src="img/25dc0dc552eb1f5b5f2e14db79ace0b2bd7b5ad4ed8222d6293052bead37bcd4.png"/>注意：</p><ol><li>控制周期100ms，在一个控制周期内完成数据的发送和接收。</li><li>一个PC端一个应用程序可以连接一个或者多个机器人，不允许一个PC端多个应用程序连接同一个机器人，也不允许多个PC端连接同一个机器人</li></ol></td></tr></table>', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_15', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '7.1', 'section_title': '7.1 世界坐标系', 'heading_path': ['7 概念解释', '7.1 世界坐标系'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“世界坐标系”中，应如何理解或操作“新松机器人的世界坐标系原点与机器人直角坐标系相同”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['新松机器人的世界坐标系原点与机器人直角坐标系相同', '定义在1轴轴线与2轴所在水平面的交点上', '如图 7–1所示，红线代表x轴正方向，绿线代表y轴正方向，蓝线代表z轴正方向'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 16055, 'line_end': 16055, 'excerpt': '新松机器人的世界坐标系原点与机器人直角坐标系相同。定义在1轴轴线与2轴所在水平面的交点上。如图 7–1所示，红线代表x轴正方向，绿线代表y轴正方向，蓝线代表z轴正方向。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_16', 'category': 'offline_interface', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.1', 'section_title': '5.1 RPY', 'heading_path': ['5 结构体描述', '5.1 RPY'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），配置“RPY”时，“double rz”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['double rz; //方向变量z，单位度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 15862, 'line_end': 15862, 'excerpt': 'double rz; //方向变量z，单位度', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_17', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.2.35', 'section_title': '4.2.35 ReadCurMode', 'heading_path': ['4 C#接口函数介绍', '4.2 查询类指令', '4.2.35 ReadCurMode'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“ReadCurMode”中，应如何理解或操作“int errorCode = ReadCurMode(out int mode”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['int errorCode = ReadCurMode(out int mode, connectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 9801, 'line_end': 9801, 'excerpt': 'int errorCode = ReadCurMode(out int mode, connectID);', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_18', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.3.34', 'section_title': '3.3.34 WriteUOMulti', 'heading_path': ['3 C++接口函数介绍', '3.3 设置类指令', '3.3.34 WriteUOMulti'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“WriteUOMulti”中，应如何理解或操作“功能描述”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 功能描述：批量设置取用户IO输出状态'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 5659, 'line_end': 5659, 'excerpt': '1. 功能描述：批量设置取用户IO输出状态。', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_19', 'category': 'offline_interface', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.4.18', 'section_title': '4.4.18 JudgePos', 'heading_path': ['4 C#接口函数介绍', '4.4 执行类指令', '4.4.18 JudgePos'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），“JudgePos”中“函数声明”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['RPY'], 'expected_facts': ['2. 函数声明：int JudgePos(RPY rpy, RPY toolCoor, RPY userCoor, out int result, int ConnectID);'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 14724, 'line_end': 14724, 'excerpt': '2. 函数声明：int JudgePos(RPY rpy, RPY toolCoor, RPY userCoor, out int result, int ConnectID);', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}, {'id': 'sample_offline_interface_287b7e4c03_20', 'category': 'offline_interface', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.1.1', 'section_title': '2.1.1 C++版本', 'heading_path': ['2 使用前准备', '2.1 系统环境要求', '2.1.1 C++版本'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“C++版本”中，应如何理解或操作“软件要求”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 软件要求：C++17，Microsoft Visual C++ 2022'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'line_start': 1072, 'line_end': 1072, 'excerpt': '4. 软件要求：C++17，Microsoft Visual C++ 2022', 'sha256': 'e7f50daed7d43093d6994c56daa3f111feb76c706432a3d1e76c576c2e755ffe'}]}]
register_sample_bank('offline_interface', OFFLINE_INTERFACE_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md', 'seed': 2898738795637056308, 'population': 1583, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(690, 690), (9589, 9589), (720, 720), (9551, 9551), (8269, 8269), (3282, 3282), (4245, 4245), (9939, 9942), (718, 718), (6450, 6450), (7177, 7177), (9566, 9566), (9887, 9890), (3602, 3602), (9991, 9991), (9891, 9894), (9981, 9981), (9581, 9581), (763, 763), (9735, 9735)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md', 'seed': 8136979333225638502, 'population': 2167, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(12623, 12623), (15511, 15511), (15521, 15521), (15377, 15381), (1014, 1014), (15477, 15481), (9365, 9365), (6179, 6179), (15357, 15361), (6221, 6221), (4813, 4813), (12073, 12073), (998, 998), (2054, 2054), (956, 956), (15052, 15052), (15086, 15086), (15372, 15376), (15148, 15148), (9602, 9602)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md', 'seed': 12380753165650530768, 'population': 2296, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1045, 1045), (17833, 17833), (6054, 6054), (1027, 1027), (14511, 14511), (18216, 18216), (15142, 15142), (6293, 6293), (1065, 1065), (11413, 11413), (17809, 17809), (18149, 18149), (5944, 5944), (18246, 18246), (4731, 4731), (18238, 18238), (1075, 1075), (17839, 17839), (1589, 1589), (11024, 11024)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md', 'seed': 14483123359730106704, 'population': 1902, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(15999, 15999), (15733, 15733), (1061, 1061), (8686, 8686), (14967, 14967), (1171, 1171), (16049, 16049), (7986, 7986), (15873, 15873), (1168, 1168), (15901, 15901), (4810, 4810), (8943, 8943), (1134, 1134), (16055, 16055), (15862, 15862), (9801, 9801), (5659, 5659), (14724, 14724), (1072, 1072)]}])


In [ ]:
# | notest
OFFLINE_INTERFACE_SAMPLE_IDS = None
offline_interface_sample_cases = select_cases(OFFLINE_INTERFACE_SAMPLE_CASES, OFFLINE_INTERFACE_SAMPLE_IDS)
display(pd.DataFrame(offline_interface_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in offline_interface_sample_cases])
offline_interface_sample_metrics = await evaluation.run(offline_interface_sample_cases, concurrency=EVALUATION_CONCURRENCY)
offline_interface_sample_results = {case["id"]: evaluation.load(case) for case in offline_interface_sample_cases}
display(offline_interface_sample_metrics)
# Inspect a complete result: evaluation.inspect(offline_interface_sample_cases[0]["id"])
evaluation.show_answers(offline_interface_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_offline_interface_b155ae970e_01,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,9393,0,True,unsupported,,,abstained
sample_offline_interface_b155ae970e_02,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,9405,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_03,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,False,True,8,False,0,0,1,False,False,False,None,None,9124,0,True,unsupported,,,abstained
sample_offline_interface_b155ae970e_04,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,False,True,8,False,0,0,1,False,False,False,None,None,13247,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_05,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,False,True,8,False,0,0,5,False,False,False,None,None,17291,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_06,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,False,True,8,False,0,0,4,False,False,False,None,None,21829,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed
sample_offline_interface_b155ae970e_07,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,14008,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_08,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,18136,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_09,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,instruction,prose,False,True,8,False,0,0,0,False,False,False,False,False,12168,0,True,unsupported,,,not reviewed
sample_offline_interface_b155ae970e_10,offline_interface,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,False,True,8,False,0,0,2,False,False,False,None,None,21743,0,True,unsupported,,,not reviewed


### sample_offline_interface_b155ae970e_01

根据《SX322009《新松机器人离线接口库使用手册》-V1.0》（V3.0 / V1.0），“概要”中“该手册说明了C#及C++版离线编程接口库的接口函数的使用方法”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 新松机器人离线接口库使用手册中的表格行展示了三个核心字段：“值”、“状态”和“含义”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 12160–12164; corpus 4.0, revision V5.0 / B-7; object `c6ede2043ba6aadb8427ef5e191733cb3b5c6bd0601a78e00bfcb29b0df96f85`.

- 该行为新松机器人离线接口库手册中的表格数据行（索引7），包含三列：
1. **序号**：32
2. **模式**：示教模式
3. **第三列**：空值（无内容）

所有单元格均为独立单元格（无合并继承），来自V5.0版本SX322009手册。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 12190–12194; corpus 4.0, revision V5.0 / B-7; object `0d20cd13c863cdfe4b9d08d35c9adcfb22c5f73cf25975ccd7bbbe7a514d1823`.

- 该表格行显示文档标题为“新松机器人离线接口库操作手册”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md`, lines 13–15; corpus 5.0, revision V5.0 / C-1; object `ab8c53ec48a6d1077d911a868e4fcb925e58441bfda7e48cf9053bb24ff18537`.

- 该表格行（索引2）包含单个加粗文本单元格：“新松机器人离线接口库操作手册”，作为标题标识，无其他列或合并单元格关系。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 7–9; corpus 4.0, revision V5.0 / B-7; object `3ff964a18eef632e3c4d6ab8843b784a9164f9a9f6a111abdd23e1e7bf181b8a`.

- 该表格为**结构极简的文档标题页**，无数据内容，仅用于标识文档属性：

- **第1行**：空占位行，无实际内容。
- **第2行**：显示文档主标题为 **“新松机器人离线接口库操作手册”**。
- **第3行**：标注文档类别为 **“工业机器人 技术资料”**。

> 摘要：该表格仅包含三行单列结构，核心信息为文档名称《新松机器人离线接口库操作手册》及其所属类别“工业机器人技术资料”，无数据、无合并单元格、无复杂布局，实质为一份技术手册的封面式标题块。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md`, lines 9–19; corpus 5.0, revision V5.0 / C-1; object `4a50e90d08b9a83ed4820ef01a883ed952602b538682ddbb0da6a0e705e51fff`.

- 该表格行位于《新松机器人离线接口库使用手册》(B-7) V5.0 版本中，定义了状态码说明表的三个核心字段：**值**（Status Code）、**状态**（Status）和 **含义**（Meaning/Description）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 3659–3663; corpus 4.0, revision V5.0 / B-7; object `cbb0fad77a002fa4f3c632bfd683725d2223dd84a11454a8dd7b1b45fc4c3056`.

- 该表格为单列无表头结构，共三行：首行空白；第二行为加粗标题“新松机器人离线接口库操作手册”；第三行为加粗副标题或分类标识“工业机器人 技术资料”。整体内容简洁，仅用于标识文档主题与类别，无数据列或复杂单元格关系。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 3–13; corpus 4.0, revision V5.0 / B-7; object `217a63e14cd60105306bffaf0d66265199fc0665a1f0bb353c99f368093ef1f3`.

- 该表格行（索引 32）定义了新松机器人离线接口库数据结构中的一个参数：参数名为 **PARA_S_UI2**，其数据长度为 **23 字节**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 3465–3468; corpus 4.0, revision V5.0 / B-7; object `cdc953433e99c5c4c20c82446b27d29a0b3f291c2f6e59ee32114492082c0182`.

#### Answer

The question asks about the '概要' (Overview) section of SX322009《新松机器人离线接口库使用手册》V1.0, specifically the sentence '该手册说明了C#及C++版离线编程接口库的接口函数的使用方法'. However, none of the retrieved evidence (E1–E8) contains this overview/summary text. The evidence only provides: (1) robot status bit-value tables (E1, E2, E6), (2) a table of function interface names with byte sizes such as Movl_P2P (69 bytes), Movj_P2P (57 bytes), etc. (E8), and (3) title-page tables (E3, E4, E5, E7). The specific '概要' paragraph describing the manual's scope for C# and C++ offline programming interface functions is absent from all retrieved snippets. Additionally, the question references V1.0, while all evidence is from V5.0/B-7 or V5.0/C-1 revisions. Without the actual overview text or any C#/C++ function signature, parameter list, or usage example in the evidence, a supported contextual example cannot be constructed.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: mixed.

The answer correctly identifies that the supplied evidence (E1-E8) does not contain the specific '概要' (Overview) text requested in the question. The evidence consists of status bit tables, function interface byte sizes, and title pages from V5.0 revisions, whereas the question asks about the V1.0 manual's overview. Since the specific fact ('该手册说明了C#及C++版离线编程接口库的接口函数的使用方法') is not present in the provided evidence snippets, the model's refusal to generate a supported answer is appropriate. The answer accurately describes the content of the provided evidence and notes the revision mismatch.

...

#### Answer

在所提供的检索证据（E1–E8）中，没有任何一条包含

### sample_offline_interface_287b7e4c03_20

根据《SX322009《新松机器人离线接口库操作手册》(C-1)》（V5.0 / C-1），在“C++版本”中，应如何理解或操作“软件要求”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行为新松机器人离线接口库手册中的表格数据行（索引7），包含三列：
1. **序号**：32
2. **模式**：示教模式
3. **第三列**：空值（无内容）

所有单元格均为独立单元格（无合并继承），来自V5.0版本SX322009手册。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 12190–12194; corpus 4.0, revision V5.0 / B-7; object `0d20cd13c863cdfe4b9d08d35c9adcfb22c5f73cf25975ccd7bbbe7a514d1823`.

- 新松机器人离线接口库使用手册中的表格行展示了三个核心字段：“值”、“状态”和“含义”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 12160–12164; corpus 4.0, revision V5.0 / B-7; object `c6ede2043ba6aadb8427ef5e191733cb3b5c6bd0601a78e00bfcb29b0df96f85`.

- 该表格行（索引2）包含单个加粗文本单元格：“新松机器人离线接口库操作手册”，作为标题标识，无其他列或合并单元格关系。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 7–9; corpus 4.0, revision V5.0 / B-7; object `3ff964a18eef632e3c4d6ab8843b784a9164f9a9f6a111abdd23e1e7bf181b8a`.

- 该表格行显示文档标题为“新松机器人离线接口库操作手册”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库操作手册》(C-1)/SX322009《新松机器人离线接口库操作手册》(C-1).md`, lines 13–15; corpus 5.0, revision V5.0 / C-1; object `ab8c53ec48a6d1077d911a868e4fcb925e58441bfda7e48cf9053bb24ff18537`.

- 该表格行位于《新松机器人离线接口库使用手册》(B-7) V5.0 版本中，定义了状态码说明表的三个核心字段：**值**（Status Code）、**状态**（Status）和 **含义**（Meaning/Description）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人离线接口库使用手册》(B-7)/SX322009《新松机器人离线接口库使用手册》(B-7).md`, lines 3659–3663; corpus 4.0, revision V5.0 / B-7; object `cbb0fad77a002fa4f3c632bfd683725d2223dd84a11454a8dd7b1b45fc4c3056`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

#### Answer

The retrieved evidence does not contain any content describing '软件要求' (software requirements) in the 'C++版本' section of SX322009《新松机器人离线接口库操作手册》(C-1) V5.0/C-1. Evidence E4 confirms the existence of the C-1 manual but only shows its title page. The remaining evidence (E1, E2, E5) comes from a different revision (B-7, 使用手册) and only contains robot status value tables (上电/暂停/急停/运行/示教模式/执行模式/远程模式). No evidence describes the C++ version's software requirements, their operation method, purpose, or expected result.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence does not contain the specific '软件要求' (software requirements) for the C++ version of the requested manual (SX322009 C-1). Evidence E4 confirms the existence of the C-1 manual but only provides the title page. Other evidence (E1, E2, E5) is from a different revision (B-7) and contains unrelated status tables. Therefore, the refusal to answer is appropriate and supported by the lack of relevant evidence.

## Load Analysis — sampled manual revisions

20 cases from 1 CSV files. Set `LOAD_ANALYSIS_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
LOAD_ANALYSIS_SAMPLE_CASES = [{'id': 'sample_load_analysis_ae1feeb5c2_01', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），如何通过名称关键字和安装方式查找机器人，支持哪些触发查询的方法？', 'anchors': [], 'expected_facts': ['1. 机器人查询：在"名称"输入框中输入查询关键字和选择"安装方式",点击查询或回车可以查找对应的工业机器人'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 98, 'line_end': 98, 'excerpt': '1. **机器人查询：**在"名称"输入框中输入查询关键字和选择"安装方式",点击查询或回车可以查找对应的工业机器人。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_02', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），进行负载计算前必须先完成哪一步机器人选择操作？', 'anchors': [], 'expected_facts': ['2. 机器人列表：显示所有工业机器人型号列表，单击选中所需机器人名称后，会显示对应的技术参数', '注意：只有在选择机器人后才能进行负载计算'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 99, 'line_end': 99, 'excerpt': '2. **机器人列表：**显示所有工业机器人型号列表，单击选中所需机器人名称后，会显示对应的技术参数。**注意：只有在选择机器人后才能进行负载计算。**', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_03', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），在机器人列表中选中型号后，界面会显示什么信息？', 'anchors': [], 'expected_facts': ['2. 机器人列表：显示所有工业机器人型号列表，单击选中所需机器人名称后，会显示对应的技术参数', '注意：只有在选择机器人后才能进行负载计算'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 99, 'line_end': 99, 'excerpt': '2. **机器人列表：**显示所有工业机器人型号列表，单击选中所需机器人名称后，会显示对应的技术参数。**注意：只有在选择机器人后才能进行负载计算。**', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_04', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），负载数据输入区域支持输入机器人哪些部位的负载数据？', 'anchors': [], 'expected_facts': ['3. 负载数据输入：此区域可根据用户需要输入末端、手臂负载数据'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 100, 'line_end': 100, 'excerpt': '3. **负载数据输入：**此区域可根据用户需要输入末端、手臂负载数据。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_05', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），负载数据输入完成后，怎样启动负载合格性计算？', 'anchors': [], 'expected_facts': ['4. 负载分析：负载数据输入完成后，点击此按钮可以根据当前输入值进行负载是否合格的计算'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 101, 'line_end': 101, 'excerpt': '4. **负载分析：**负载数据输入完成后，点击此按钮可以根据当前输入值进行负载是否合格的计算。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_06', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），怎样清空已经计算过的负载分析结果？', 'anchors': [], 'expected_facts': ['5. 清空结果：点击此按钮清空所有已经计算后的结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 102, 'line_end': 102, 'excerpt': '5. **清空结果：**点击此按钮清空所有已经计算后的结果。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_07', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），机器人图形显示区包含哪些图形，它们有助于理解什么？', 'anchors': [], 'expected_facts': ['6. 机器人图形显示：显示机器人外形，手臂负载图形以及末端负载示意图，便于用户理解机器人负载情况'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 103, 'line_end': 103, 'excerpt': '6. **机器人图形显示：**显示机器人外形，手臂负载图形以及末端负载示意图，便于用户理解机器人负载情况。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_08', 'category': 'load_analysis', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），计算结果区能够回答哪些负载范围和使用需求方面的问题？', 'anchors': [], 'expected_facts': ['7. 计算结果显示：显示计算后的信息，包括负载数据是否超出允许范围，机器人是否能满足当前试验负载需求等结果', '如图2-2所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 104, 'line_end': 104, 'excerpt': '7. **计算结果显示：**显示计算后的信息，包括负载数据是否超出允许范围，机器人是否能满足当前试验负载需求等结果。如图2-2所示。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_09', 'category': 'load_analysis', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），负载分析结果为红色时表示什么状态？', 'anchors': [], 'expected_facts': ['注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 106, 'line_end': 106, 'excerpt': '**注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态。**', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_10', 'category': 'load_analysis', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），负载分析结果为黄色时表示什么状态，手册给出的临界阈值是多少？', 'anchors': [], 'expected_facts': ['注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 106, 'line_end': 106, 'excerpt': '**注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态。**', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_11', 'category': 'load_analysis', 'kind': 'parameter', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），负载分析结果为绿色时表示什么状态？', 'anchors': [], 'expected_facts': ['注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 106, 'line_end': 106, 'excerpt': '**注：红色表示不合格超出负载范围，黄色表示当前处于是临界状态（＞91%)，绿色表示合格状态。**', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_12', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '2', 'section_title': '2 新松机器人负载分析软件说明', 'heading_path': ['2 新松机器人负载分析软件说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），状态栏显示什么信息，把鼠标移到界面区域时能获得什么帮助？', 'anchors': [], 'expected_facts': ['1. 状态栏：显示负载计算状态等实时信息，当鼠标移动到界面的某个区域时，会显示提示信息'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 114, 'line_end': 114, 'excerpt': '1. **状态栏：**显示负载计算状态等实时信息，当鼠标移动到界面的某个区域时，会显示提示信息。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_13', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.1', 'section_title': '3.1.1 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z一致', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', '3.1.1 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z一致'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），工具数模与机器人法兰坐标轴一致时，应怎样录入参数并查看分析结果？', 'anchors': [], 'expected_facts': ['按图3-2箭头所指，将对应值添加到机器人负载分析软件对应位置（注意长度、重量、惯量等值单位），单击负载分析，查看分析结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 136, 'line_end': 136, 'excerpt': '按图3-2箭头所指，将对应值添加到机器人负载分析软件对应位置（**注意长度、重量、惯量等值单位**），单击负载分析，查看分析结果。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_14', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.2', 'section_title': '3.1.2 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z不一致', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', '3.1.2 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z不一致'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），工具数模坐标轴与法兰坐标轴不一致时，应以哪个坐标系填写质量、质心和惯量？', 'anchors': [], 'expected_facts': ['填写负载质量质心和惯量时按机器人法兰坐标系为基准，选择工具对应坐标的值（注意长度、重量、惯量等值单位），单击负载分析，查看分析结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 146, 'line_end': 146, 'excerpt': '填写负载质量质心和惯量时按机器人法兰坐标系为基准，选择工具对应坐标的值（**注意长度、重量、惯量等值单位**），单击负载分析，查看分析结果。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_15', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.1', 'section_title': '3.1.1 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z一致', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', '3.1.1 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z一致'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），工具数模与法兰坐标轴一致时，填写负载参数需要注意哪些量的单位？', 'anchors': [], 'expected_facts': ['按图3-2箭头所指，将对应值添加到机器人负载分析软件对应位置（注意长度、重量、惯量等值单位），单击负载分析，查看分析结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 136, 'line_end': 136, 'excerpt': '按图3-2箭头所指，将对应值添加到机器人负载分析软件对应位置（**注意长度、重量、惯量等值单位**），单击负载分析，查看分析结果。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_16', 'category': 'load_analysis', 'kind': 'prerequisite', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1.2', 'section_title': '3.1.2 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z不一致', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', '3.1.2 工具数模X\\Y\\Z与机器人法兰坐标X\\Y\\Z不一致'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），工具数模与法兰坐标轴不一致时，选择坐标值后应执行什么操作确认负载结果？', 'anchors': [], 'expected_facts': ['填写负载质量质心和惯量时按机器人法兰坐标系为基准，选择工具对应坐标的值（注意长度、重量、惯量等值单位），单击负载分析，查看分析结果'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 146, 'line_end': 146, 'excerpt': '填写负载质量质心和惯量时按机器人法兰坐标系为基准，选择工具对应坐标的值（**注意长度、重量、惯量等值单位**），单击负载分析，查看分析结果。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_17', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.2', 'section_title': '3.2 结果分析', 'heading_path': ['3 负载分析软件参数设置示例', '3.2 结果分析'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），分析结果显示合格时，应如何判断机器人是否满足使用要求？', 'anchors': [], 'expected_facts': ['分析结果显示合格时机器人满足使用要求，分析结果显示不合格时机器人无法满足使用要求，如图3-4'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 156, 'line_end': 156, 'excerpt': '分析结果显示合格时机器人满足使用要求，分析结果显示不合格时机器人无法满足使用要求，如图3-4。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_18', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.2', 'section_title': '3.2 结果分析', 'heading_path': ['3 负载分析软件参数设置示例', '3.2 结果分析'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），分析结果显示不合格时，应如何判断当前机器人能否满足使用要求？', 'anchors': [], 'expected_facts': ['分析结果显示合格时机器人满足使用要求，分析结果显示不合格时机器人无法满足使用要求，如图3-4'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 156, 'line_end': 156, 'excerpt': '分析结果显示合格时机器人满足使用要求，分析结果显示不合格时机器人无法满足使用要求，如图3-4。', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_19', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1', 'section_title': '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），手册在工具数模示意图旁标注的工具质心惯量以什么位置为基准？', 'anchors': [], 'expected_facts': ['工具质心惯量（机器人六轴法兰中心为基准）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 124, 'line_end': 124, 'excerpt': '工具质心惯量（机器人六轴法兰中心为基准）', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}, {'id': 'sample_load_analysis_ae1feeb5c2_20', 'category': 'load_analysis', 'kind': 'procedure', 'profile': '', 'revision': 'V3.0 / V1.0', 'section': '3.1', 'section_title': '3.1 确认工具数模坐标系与机器人法兰坐标系一致性', 'heading_path': ['3 负载分析软件参数设置示例', '3.1 确认工具数模坐标系与机器人法兰坐标系一致性'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），手册在工具数模示意图旁标注的工具质心偏移以什么位置为基准？', 'anchors': [], 'expected_facts': ['工具质心偏移值（机器人六轴法兰中心为基准）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'line_start': 126, 'line_end': 126, 'excerpt': '工具质心偏移值（机器人六轴法兰中心为基准）', 'sha256': '537554e7d123488b9870968ca483f425f1e2e70e9451bc030187bdadffdf8110'}]}]
register_sample_bank('load_analysis', LOAD_ANALYSIS_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md', 'seed': 6019444407783838675, 'population': 16, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'curated distinct short-manual tasks', 'selected_lines': [(98, 98), (99, 99), (99, 99), (100, 100), (101, 101), (102, 102), (103, 103), (104, 104), (106, 106), (106, 106), (106, 106), (114, 114), (136, 136), (146, 146), (136, 136), (146, 146), (156, 156), (156, 156), (124, 124), (126, 126)]}])


In [ ]:
# | notest
LOAD_ANALYSIS_SAMPLE_IDS = None
load_analysis_sample_cases = select_cases(LOAD_ANALYSIS_SAMPLE_CASES, LOAD_ANALYSIS_SAMPLE_IDS)
display(pd.DataFrame(load_analysis_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in load_analysis_sample_cases])
load_analysis_sample_metrics = await evaluation.run(load_analysis_sample_cases, concurrency=EVALUATION_CONCURRENCY)
load_analysis_sample_results = {case["id"]: evaluation.load(case) for case in load_analysis_sample_cases}
display(load_analysis_sample_metrics)
# Inspect a complete result: evaluation.inspect(load_analysis_sample_cases[0]["id"])
evaluation.show_answers(load_analysis_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_load_analysis_ae1feeb5c2_01,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,8,5,0,True,False,False,None,None,4498,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_02,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,prerequisite,prose,True,True,8,True,8,5,0,True,False,False,None,None,4498,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_03,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,prerequisite,prose,True,True,8,True,8,5,0,True,False,False,None,None,4498,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_04,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,8,5,0,True,False,False,None,None,4498,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_05,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,7,4,1,True,False,False,None,None,4709,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_06,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,8,4,0,True,False,False,None,None,4461,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_07,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,procedure,prose,True,True,8,True,8,4,0,True,False,False,None,None,4460,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_08,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,True,True,8,True,8,4,0,True,False,False,None,None,4460,0,True,unsupported,,,abstained
sample_load_analysis_ae1feeb5c2_09,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,True,True,8,True,8,4,0,True,False,False,None,None,4460,0,True,unsupported,,,not reviewed
sample_load_analysis_ae1feeb5c2_10,load_analysis,assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,parameter,prose,True,True,8,True,8,4,0,True,False,False,None,None,4461,0,True,unsupported,,,not reviewed


### sample_load_analysis_ae1feeb5c2_01

根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），如何通过名称关键字和安装方式查找机器人，支持哪些触发查询的方法？

#### Retrieved content summary

- 新松机器人负载分析软件使用手册封面，标注为工业机器人技术资料
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 3–13; corpus 4.0, revision V3.0 / V1.0; object `afb0e80538d8af0a37ccd34eef8a290ddea7410d042b615df79b2056bc294f09`.

- 新松机器人负载分析软件使用手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 7–9; corpus 4.0, revision V3.0 / V1.0; object `86769aa5445daa91a1b9fff8ab293f4d382dccf931d52a7778402b630217c9ea`.

- 新松机器人负载分析软件使用手册（SX322010）文档记录显示，V1.0版本由沙景超于2024年7月31日完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 30–35; corpus 5.0, revision V3.0 / V1.0; object `7cda9c4c4eeef5bf6c518b4b9408767d045040ba974b2a8ecdca121b15f466b3`.

- 该表格行位于第2行，包含一个单元格，其文本内容为“新松机器人负载分析软件使用手册”，表明该行标识了文档的具体标题或主题。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 7–9; corpus 5.0, revision V3.0 / V1.0; object `fa1885c329e5b2f278cf14e16e9923a574b0a8c67b28e7ccd299f3d3d9caee56`.

- 该行表明新松公司（SX322010手册相关）提供“工业机器人 技术资料”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 10–12; corpus 4.0, revision V3.0 / V1.0; object `456f11f401adcb51501500455491d4986a570ec86ce432d2fcb0a498b0ee86ae`.

- 该行为《新松机器人负载分析软件使用手册》中表格的第 7 行，包含 4 个完全为空的单元格，无具体数据或继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 60–65; corpus 5.0, revision V3.0 / V1.0; object `e46bdbec9c8d655b83cda8b1e20e935389ab2c2aef8a332294d2785150fd951c`.

- 该行位于《新松机器人负载分析软件使用手册》文档的表格中（行索引为6），由4个空单元格组成，无标题或合并单元格继承值，未包含任何具体信息。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 54–59; corpus 4.0, revision V3.0 / V1.0; object `e3aec35c6a751a09e0a208cd12e620c29678b1e2d2c17f8aad95879c077f5a08`.

- 该行位于《新松机器人负载分析软件使用手册》第6行，包含4个空单元格，无具体数据或继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 54–59; corpus 5.0, revision V3.0 / V1.0; object `c076bb6b70dbd74eff14980e6a2252ccb19f4c96ee60495db82cc4a9a33d8935`.

#### Answer

根据检索到的手册证据（E1-E8），仅包含《新松机器人负载分析软件使用手册》的封面标题（

### sample_load_analysis_ae1feeb5c2_02

根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），进行负载计算前必须先完成哪一步机器人选择操作？

...

#### Answer

The retrieved evidence (E1–E8) only contains the manual's cover page, title, version-history table, and several empty table cells. None of the snippets mention the tool numerical model diagram (工具数模示意图), the tool centroid inertia annotation (工具质心惯量), or any reference position (基准) such as the tool flange center, the robot TCP, or the tool's own coordinate origin. Because the specific passage that would state the inertia reference datum is absent from the supplied evidence, the question cannot be answered from the provided material.

### sample_load_analysis_ae1feeb5c2_20

根据《SX322010《新松机器人负载分析软件使用手册》-V1.0》（V3.0 / V1.0），手册在工具数模示意图旁标注的工具质心偏移以什么位置为基准？

#### Retrieved content summary

- 新松机器人负载分析软件使用手册封面，标注为工业机器人技术资料
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 3–13; corpus 4.0, revision V3.0 / V1.0; object `afb0e80538d8af0a37ccd34eef8a290ddea7410d042b615df79b2056bc294f09`.

- 新松机器人负载分析软件使用手册（SX322010）文档记录显示，V1.0版本由沙景超于2024年7月31日完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 30–35; corpus 5.0, revision V3.0 / V1.0; object `7cda9c4c4eeef5bf6c518b4b9408767d045040ba974b2a8ecdca121b15f466b3`.

- 新松机器人负载分析软件使用手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 7–9; corpus 4.0, revision V3.0 / V1.0; object `86769aa5445daa91a1b9fff8ab293f4d382dccf931d52a7778402b630217c9ea`.

- 该行为《新松机器人负载分析软件使用手册》中表格的第 7 行，包含 4 个完全为空的单元格，无具体数据或继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 60–65; corpus 5.0, revision V3.0 / V1.0; object `e46bdbec9c8d655b83cda8b1e20e935389ab2c2aef8a332294d2785150fd951c`.

- 该行位于《新松机器人负载分析软件使用手册》第6行，包含4个空单元格，无具体数据或继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 54–59; corpus 5.0, revision V3.0 / V1.0; object `c076bb6b70dbd74eff14980e6a2252ccb19f4c96ee60495db82cc4a9a33d8935`.

- 该行位于《新松机器人负载分析软件使用手册》文档的表格中（行索引为6），由4个空单元格组成，无标题或合并单元格继承值，未包含任何具体信息。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 54–59; corpus 4.0, revision V3.0 / V1.0; object `e3aec35c6a751a09e0a208cd12e620c29678b1e2d2c17f8aad95879c077f5a08`.

- 该行表明新松公司（SX322010手册相关）提供“工业机器人 技术资料”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md`, lines 10–12; corpus 4.0, revision V3.0 / V1.0; object `456f11f401adcb51501500455491d4986a570ec86ce432d2fcb0a498b0ee86ae`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

#### Answer

The retrieved evidence (E1–E8) only contains the cover page, title, version history table, and empty table cells of the manual SX322010《新松机器人负载分析软件使用手册》-V1.0. None of the evidence excerpts include the section describing the tool numerical model diagram (工具数模示意图) or any annotation specifying the reference point (基准) for the tool center of gravity offset (工具质心偏移). The specific content about the offset reference is not present in the provided evidence, so a supported contextual example cannot be constructed.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The question asks for the reference point for the tool center of gravity offset as annotated in the manual. The expected fact is '机器人六轴法兰中心为基准' (robot six-axis flange center as reference). However, the supplied evidence (E1-E8) only contains the cover page, title, version history, and empty table cells. None of the evidence excerpts contain the specific section describing the tool numerical model diagram or the annotation for the tool center of gravity offset. Therefore, the model correctly abstained from answering because the necessary information was not present in the retrieved context. The refusal is appropriate and not a hallucination.

## Palletizing — sampled manual revisions

40 cases from 2 CSV files. Set `PALLETIZING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
PALLETIZING_SAMPLE_CASES = [{'id': 'sample_palletizing_c7344ab204_01', 'category': 'palletizing', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.2', 'section_title': '1.2.2 码垛属性文件配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.2 码垛属性文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），配置“码垛属性文件配置”时，“层高度”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['6. 层高度：每层的高度，单位为mm，可以精确到0.001mm'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 141, 'line_end': 141, 'excerpt': '6. 层高度：每层的高度，单位为mm，可以精确到0.001mm。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_02', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3', 'section_title': '1.3 码垛指令', 'heading_path': ['1 码垛操作', '1.3 码垛指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“码垛指令”中“参数项1”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['说明 参数项1 码垛文件号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 304, 'line_end': 308, 'excerpt': '<tr>\n<td rowspan="2"><p>说明</p></td>\n<td colspan="3"><p>参数项1</p></td>\n<td colspan="2"><p>码垛文件号</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_03', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.2', 'section_title': '1.2.2 码垛属性文件配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.2 码垛属性文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“码垛属性文件配置”中，应如何理解或操作“每列工件数”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 每列工件数：用户坐标系Y方向的工件数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 138, 'line_end': 138, 'excerpt': '3. 每列工件数：用户坐标系Y方向的工件数。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_04', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.3', 'section_title': '1.2.3 计数器设置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.3 计数器设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），操作“计数器设置”时，涉及“这里面注意的是个数计数器是为了表示码垛了多少个工件（实际为个数计数器中的数-1）”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['这里面注意的是个数计数器是为了表示码垛了多少个工件（实际为个数计数器中的数-1）', '可以设置成用户想表示的数值', '但是不要超过要码垛的总的个数，如果超过，系统会自动进行复位操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 192, 'line_end': 192, 'excerpt': '这里面注意的是个数计数器是为了表示码垛了多少个工件（实际为个数计数器中的数-1）。可以设置成用户想表示的数值。但是不要超过要码垛的总的个数，如果超过，系统会自动进行复位操作。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_05', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3', 'section_title': '1.3 码垛指令', 'heading_path': ['1 码垛操作', '1.3 码垛指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“码垛指令”中“PAL L”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PAL'], 'expected_facts': ['PAL L 功能 码垛以直线移动到示教点，含位置点信息'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 223, 'line_end': 227, 'excerpt': '<tr>\n<td rowspan="4"><p>PAL L</p></td>\n<td><p>功能</p></td>\n<td colspan="5"><p>码垛以直线移动到示教点，含位置点信息</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_06', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.6.2', 'section_title': '1.6.2 码垛作业', 'heading_path': ['1 码垛操作', '1.6 码垛实例', '1.6.2 码垛作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“码垛作业”中，应如何理解或操作“针对上图1-11”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['针对上图1-11，进行编写作业如下'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 437, 'line_end': 437, 'excerpt': '针对上图1-11，进行编写作业如下。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_07', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.2', 'section_title': '1.2.2 码垛属性文件配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.2 码垛属性文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），操作“码垛属性文件配置”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：工件数参数需要输入整数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 143, 'line_end': 143, 'excerpt': '**注意**：工件数参数需要输入整数。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_08', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3', 'section_title': '1.3 码垛指令', 'heading_path': ['1 码垛操作', '1.3 码垛指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“码垛指令”中“参数项2,3,4”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PAL'], 'expected_facts': ['参数项2,3,4 根据PAL S中的码垛模式：如果按行码垛，则参数分别为行计数器数值，列计数器数值，层计数器数值', '如果按列码垛，则参数项分别为列计数器数值，行计数器数值，层计数器数值'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 309, 'line_end': 312, 'excerpt': '<tr>\n<td colspan="3"><p>参数项2,3,4</p></td>\n<td colspan="2"><p>根据PAL S中的码垛模式：如果按行码垛，则参数分别为行计数器数值，列计数器数值，层计数器数值；如果按列码垛，则参数项分别为列计数器数值，行计数器数值，层计数器数值</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_09', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.2', 'section_title': '1.2.2 码垛属性文件配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.2 码垛属性文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“码垛属性文件配置”中，应如何理解或操作“每行工件偏置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 每行工件偏置：X方向上相邻码垛工件之间抓取点之间的距离', '保证足够大，使得工件可以放置进去', '每行工件偏置数值是工件宽（长）与工件之间相应方向的间隔距离的和'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 139, 'line_end': 139, 'excerpt': '4. 每行工件偏置：X方向上相邻码垛工件之间抓取点之间的距离。保证足够大，使得工件可以放置进去。每行工件偏置数值是工件宽（长）与工件之间相应方向的间隔距离的和。可以精确到0.001mm。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_10', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.2', 'section_title': '1.2.2 码垛属性文件配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.2 码垛属性文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），操作“码垛属性文件配置”时，涉及“托盘属性文件中的参数值必须是整数”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['托盘属性文件中的参数值必须是整数，且要大于或等于0，输入小于零的数值，系统会自动修改为0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 169, 'line_end': 169, 'excerpt': '托盘属性文件中的参数值必须是整数，且要大于或等于0，输入小于零的数值，系统会自动修改为0。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_11', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3', 'section_title': '1.3 码垛指令', 'heading_path': ['1 码垛操作', '1.3 码垛指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“码垛指令”中“PAL R <参数项>”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PAL'], 'expected_facts': ['格式 PAL R <参数项>'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 282, 'line_end': 285, 'excerpt': '<tr>\n<td><p>格式</p></td>\n<td colspan="5"><p>PAL R &lt;参数项&gt;</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_12', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.6.2', 'section_title': '1.6.2 码垛作业', 'heading_path': ['1 码垛操作', '1.6 码垛实例', '1.6.2 码垛作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“码垛作业”中，应如何理解或操作“PAL E #2”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['010 PAL E #2 码垛结束标志'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 490, 'line_end': 494, 'excerpt': '<tr>\n<td><p>010</p></td>\n<td><p><strong>PAL E &#35;2</strong></p></td>\n<td><p>码垛结束标志</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_13', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3.2', 'section_title': '1.3.2 PAL L&PAL J指令说明', 'heading_path': ['1 码垛操作', '1.3 码垛指令', '1.3.2 PAL L&PAL J指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“PAL L&PAL J指令说明”中“OUT[5]=1 与上条指令结合假设为控制夹手放下工件的指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT'], 'expected_facts': ['OUT[5]=1 与上条指令结合假设为控制夹手放下工件的指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 365, 'line_end': 365, 'excerpt': 'OUT[5]=1 与上条指令结合假设为控制夹手放下工件的指令', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_14', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.3', 'section_title': '1.2.3 计数器设置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.3 计数器设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“计数器设置”中，应如何理解或操作“个数计数器数值”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 个数计数器数值：将要执行码垛的个数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 188, 'line_end': 188, 'excerpt': '4. 个数计数器数值：将要执行码垛的个数', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_15', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3.2', 'section_title': '1.3.2 PAL L&PAL J指令说明', 'heading_path': ['1 码垛操作', '1.3 码垛指令', '1.3.2 PAL L&PAL J指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“PAL L&PAL J指令说明”中“例”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PAL'], 'expected_facts': ['例： PAL S 1 1'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 357, 'line_end': 357, 'excerpt': '例： PAL S 1 1', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_16', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.1', 'section_title': '1.2.1 用户坐标系配置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.1 用户坐标系配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“用户坐标系配置”中，应如何理解或操作“用户坐标系通常设定在托盘上平面的一角”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['用户坐标系通常设定在托盘上平面的一角，标定的方法使用3点法，第一点确定用户坐标系的原点，第二点确定用户坐标系的X轴方向，第三点确定Y轴方向，Z轴方向由右手定则确定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 122, 'line_end': 122, 'excerpt': '用户坐标系通常设定在托盘上平面的一角，标定的方法使用3点法，第一点确定用户坐标系的原点，第二点确定用户坐标系的X轴方向，第三点确定Y轴方向，Z轴方向由右手定则确定。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_17', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3', 'section_title': '1.3 码垛指令', 'heading_path': ['1 码垛操作', '1.3 码垛指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“码垛指令”中“PAL E #15”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PAL'], 'expected_facts': ['举例 PAL E #15'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 273, 'line_end': 276, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td colspan="5"><p>PAL E &#35;15</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_18', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.2.3', 'section_title': '1.2.3 计数器设置', 'heading_path': ['1 码垛操作', '1.2 码垛配置', '1.2.3 计数器设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“计数器设置”中，应如何理解或操作“行数计数器数值”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 行数计数器数值：将要执行码垛的行数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 185, 'line_end': 185, 'excerpt': '1. 行数计数器数值：将要执行码垛的行数', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_19', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.3.1', 'section_title': '1.3.1 PAL S指令中的模式含义', 'heading_path': ['1 码垛操作', '1.3 码垛指令', '1.3.1 PAL S指令中的模式含义'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），“PAL S指令中的模式含义”中“目前支持两种模式1-2”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['目前支持两种模式1-2，如图2，3两种模式分别为按照行码垛和按照列码垛', '行码垛和列码垛是按照用户坐标系的X轴和Y轴方向定义的'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 343, 'line_end': 343, 'excerpt': '目前支持两种模式1-2，如图2，3两种模式分别为按照行码垛和按照列码垛。行码垛和列码垛是按照用户坐标系的X轴和Y轴方向定义的。', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_c7344ab204_20', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / V1.0', 'section': '1.6.2', 'section_title': '1.6.2 码垛作业', 'heading_path': ['1 码垛操作', '1.6 码垛实例', '1.6.2 码垛作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'source_version': '', 'question': '根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），在“码垛作业”中，应如何理解或操作“MOVL VL=300”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['001 MOVL VL=300 固定位置取工件的位置点'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'line_start': 445, 'line_end': 449, 'excerpt': '<tr>\n<td><p>001</p></td>\n<td><p>MOVL VL=300</p></td>\n<td><p>固定位置取工件的位置点</p></td>\n</tr>', 'sha256': 'd7dc66b1e8b183beeb4b855ed583f8aa8ab5602364f399a05ec9ac963e73e884'}]}, {'id': 'sample_palletizing_d12de074dd_01', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.7.6.1.5', 'section_title': '3.7.6.1.5 垛型预定义', 'heading_path': ['3 工作流程&功能', '3.7 按照向导提示进行码垛工艺配置', '3.7.6 垛型管理', '3.7.6.1 新建垛型', '3.7.6.1.5 垛型预定义'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“垛型预定义”中，应如何理解或操作“正反交错： 同一层不同行旋转90°，相邻层180°镜像”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3 正反交错： 同一层不同行旋转90°，相邻层180°镜像'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1325, 'line_end': 1329, 'excerpt': '<tr>\n<td><p>3</p></td>\n<td><p>正反交错：</p><p>同一层不同行旋转90°，相邻层180°镜像</p></td>\n<td><img src="img/ab78e2700f2cfd59df45b216afe0bc5c3a070c531f722759fc5ab2011ec09132.png"/></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_02', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.3.2', 'section_title': '1.3.2 路径点配置', 'heading_path': ['1 码垛应用简介', '1.3 功能概述', '1.3.2 路径点配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），操作“路径点配置”时，涉及“在抓放路径中”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['在抓放路径中，除去系统默认必须的点外，用户可自行根据工况添加过渡点，以避免环境干涉', '同时可配置机器人在运动到该点位后的IO信号操作，以便同其他设备进行交互'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 439, 'line_end': 439, 'excerpt': '在抓放路径中，除去系统默认必须的点外，用户可自行根据工况添加过渡点，以避免环境干涉。同时可配置机器人在运动到该点位后的IO信号操作，以便同其他设备进行交互。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_03', 'category': 'palletizing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.7.3.2.1', 'section_title': '3.7.3.2.1 点位信息维护', 'heading_path': ['3 工作流程&功能', '3.7 按照向导提示进行码垛工艺配置', '3.7.3 抓取路径设置', '3.7.3.2 路径点管理', '3.7.3.2.1 点位信息维护'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），配置“点位信息维护”时，“V”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['3. V:机器人运动速度', '运动类型为MOVJ时，V表示最大移动速度的百分之几，单位为%', '运动类型为MOVL时，V表示线速度，单位为 mm/s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 996, 'line_end': 996, 'excerpt': '     3. V:机器人运动速度。运动类型为MOVJ时，V表示最大移动速度的百分之几，单位为%；运动类型为MOVL时，V表示线速度，单位为 mm/s。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_04', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.7.4.2.1', 'section_title': '3.7.4.2.1 点位信息维护', 'heading_path': ['3 工作流程&功能', '3.7 按照向导提示进行码垛工艺配置', '3.7.4 码放路径设置', '3.7.4.2 路径点管理', '3.7.4.2.1 点位信息维护'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），操作“点位信息维护”时，涉及“获取当前RPY”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['10. 获取当前RPY：获取机器人当前位置信息赋值到X/Y/Z/Rx/Ry/Rz中', '获取的位置信息基于当前的工具系和用户系', '同时注意，用户提交点位信息时，会同步更新机器人当前的工具系号和用户系号为该界面中指定的“工具参考系”、“用户参考系”值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1150, 'line_end': 1150, 'excerpt': '     10. 获取当前RPY：获取机器人当前位置信息赋值到X/Y/Z/Rx/Ry/Rz中。获取的位置信息基于当前的工具系和用户系。同时注意，用户提交点位信息时，会同步更新机器人当前的工具系号和用户系号为该界面中指定的“工具参考系”、“用户参考系”值。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_05', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.3', 'section_title': '4.3 备份恢复', 'heading_path': ['4 其他功能', '4.3 备份恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“备份恢复”中，应如何理解或操作“部分备份/恢复”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['部分备份/恢复：支持对产品、垛型、IO的其中一项或者几项配置信息进行单独备份和恢复，以便多机器人工作站场景快速复用产品和垛型信息'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1584, 'line_end': 1584, 'excerpt': '部分备份/恢复：支持对产品、垛型、IO的其中一项或者几项配置信息进行单独备份和恢复，以便多机器人工作站场景快速复用产品和垛型信息。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_06', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.1', 'section_title': '1.1 概述', 'heading_path': ['1 码垛应用简介', '1.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“概述”中，应如何理解或操作“新松机器人码垛应用包是运行在第5代RC控制器的机器人应用软件”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['新松机器人码垛应用包是运行在第5代RC控制器的机器人应用软件，具备直观的图形化配置向导、灵活的垛型设计，多工艺场景适配，以及自动生成码垛作业程序等功能，旨在尽可能的减少学习和编程的时间', '使用起来便捷又高效'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 357, 'line_end': 357, 'excerpt': '新松机器人码垛应用包是运行在第5代RC控制器的机器人应用软件，具备直观的图形化配置向导、灵活的垛型设计，多工艺场景适配，以及自动生成码垛作业程序等功能，旨在尽可能的减少学习和编程的时间。使用起来便捷又高效。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_07', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.8.2', 'section_title': '3.8.2 程序基础说明', 'heading_path': ['3 工作流程&功能', '3.8 编程', '3.8.2 程序基础说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），“程序基础说明”中“- 在 特 定 条 件 下”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ELSE', 'GOTO'], 'expected_facts': ['- 在 特 定 条 件 下 ， 对 机 器 人 运 行 流 程 进 行 控 制 或 改 变 的 逻 辑 指 令 （IF\\ELSE\\GOTO\\LABEL 等），延迟机器人程序运行的等待指令（DELAY）等'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1448, 'line_end': 1448, 'excerpt': '- 在 特 定 条 件 下 ， 对 机 器 人 运 行 流 程 进 行 控 制 或 改 变 的 逻 辑 指 令 （IF\\ELSE\\GOTO\\LABEL 等），延迟机器人程序运行的等待指令（DELAY）等。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_08', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5', 'section_title': '5 故障排除', 'heading_path': ['5 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“故障排除”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 报警子码 报警 等级 报警原因 处理方法'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1607, 'line_end': 1614, 'excerpt': '<tr>\n<td><p><strong>报警码</strong></p></td>\n<td><p><strong>报警信息</strong></p></td>\n<td><p><strong>报警子码</strong></p></td>\n<td><p><strong>报警</strong></p><p><strong>等级</strong></p></td>\n<td><p><strong>报警原因</strong></p></td>\n<td><p><strong>处理方法</strong></p></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_09', 'category': 'palletizing', 'kind': 'alarm', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5', 'section_title': '5 故障排除', 'heading_path': ['5 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），“故障排除”列出的报警30001表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['30001'], 'expected_facts': ['30001 码垛工艺生成失败 错误 码垛工艺生成过程配置出现问题 修改码垛工艺相关配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1615, 'line_end': 1622, 'excerpt': '<tr>\n<td><p>30001</p></td>\n<td><p>码垛工艺生成失败</p></td>\n<td></td>\n<td><p>错误</p></td>\n<td><p>码垛工艺生成过程配置出现问题</p></td>\n<td><p>修改码垛工艺相关配置</p></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_10', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.1', 'section_title': '2.1 SR50A-50/2.15、SR50A-70/2.15管线包安装', 'heading_path': ['2 管线包安装', '2.1 SR50A-50/2.15、SR50A-70/2.15管线包安装'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“SR50A-50/2.15、SR50A-70/2.15管线包安装”中，应如何理解或操作“SR50A系列机器人管线包套机安装示意图”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['SR50A系列机器人管线包套机安装示意图，如“图 2-1机器人管线包套机图”、“图 2-2管线包组件图”、“图 2-3管线包安装示意图”所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 495, 'line_end': 495, 'excerpt': 'SR50A系列机器人管线包套机安装示意图，如“图 2-1机器人管线包套机图”、“图 2-2管线包组件图”、“图 2-3管线包安装示意图”所示：', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_11', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.1', 'section_title': '2.1 SR50A-50/2.15、SR50A-70/2.15管线包安装', 'heading_path': ['2 管线包安装', '2.1 SR50A-50/2.15、SR50A-70/2.15管线包安装'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），操作“SR50A-50/2.15、SR50A-70/2.15管线包安装”时，涉及“注意：安装时机器人六轴关节值调整至90°”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：安装时机器人六轴关节值调整至90°'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 508, 'line_end': 510, 'excerpt': '<tr>\n<td><p><img src="img/25dc0dc552eb1f5b5f2e14db79ace0b2bd7b5ad4ed8222d6293052bead37bcd4.png"/>  注意：安装时机器人六轴关节值调整至90°</p></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_12', 'category': 'palletizing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.3.8', 'section_title': '1.3.8 工艺备份', 'heading_path': ['1 码垛应用简介', '1.3 功能概述', '1.3.8 工艺备份'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），配置“工艺备份”时，“可将码垛工艺整体进行备份恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['可将码垛工艺整体进行备份恢复，同时也支持产品、垛型、IO等数据的单项备份和恢复，以便参数进行复用，减少产线其他设备参数调试时间'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 483, 'line_end': 483, 'excerpt': '可将码垛工艺整体进行备份恢复，同时也支持产品、垛型、IO等数据的单项备份和恢复，以便参数进行复用，减少产线其他设备参数调试时间。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_13', 'category': 'palletizing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.3', 'section_title': '4.3 备份恢复', 'heading_path': ['4 其他功能', '4.3 备份恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），配置“备份恢复”时，“整体备份恢复”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['整体备份恢复：备份恢复所有码垛应用的配置参数信息'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1582, 'line_end': 1582, 'excerpt': '整体备份恢复：备份恢复所有码垛应用的配置参数信息。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_14', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.7.6.1.3', 'section_title': '3.7.6.1.3 层布局操作', 'heading_path': ['3 工作流程&功能', '3.7 按照向导提示进行码垛工艺配置', '3.7.6 垛型管理', '3.7.6.1 新建垛型', '3.7.6.1.3 层布局操作'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“层布局操作”中，应如何理解或操作“层下移”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 层下移：选择某层，按下层下移图标，该层与下层位置互换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1283, 'line_end': 1283, 'excerpt': '4. **层下移：**选择某层，按下层下移图标，该层与下层位置互换。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_15', 'category': 'palletizing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.8.3', 'section_title': '3.8.3 程序示例', 'heading_path': ['3 工作流程&功能', '3.8 编程', '3.8.3 程序示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），配置“程序示例”时，“经过向导指引”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['经过向导指引，用户完成每步参数配置后，系统自动生成机器人码垛作业程序，程序包含以下三个部分：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1461, 'line_end': 1461, 'excerpt': '经过向导指引，用户完成每步参数配置后，系统自动生成机器人码垛作业程序，程序包含以下三个部分：', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_16', 'category': 'palletizing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.4.1', 'section_title': '3.4.1 5点法标定', 'heading_path': ['3 工作流程&功能', '3.4 设置工具坐标系', '3.4.1 5点法标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），操作“5点法标定”时，涉及“注意：同一坐标的TCP和姿态需要在同一个工具号下进行标定”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：同一坐标的TCP和姿态需要在同一个工具号下进行标定'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 658, 'line_end': 660, 'excerpt': '<tr>\n<td><p><img src="img/25dc0dc552eb1f5b5f2e14db79ace0b2bd7b5ad4ed8222d6293052bead37bcd4.png"/>  注意：同一坐标的TCP和姿态需要在同一个工具号下进行标定</p></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_17', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.2', 'section_title': '4.2 异常处理', 'heading_path': ['4 其他功能', '4.2 异常处理'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“异常处理”中，应如何理解或操作“一键码满”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 一键码满：强制设置码垛状态为完成，PLC识别该状态后，重新开始新的码垛流程'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1576, 'line_end': 1576, 'excerpt': '3. 一键码满：强制设置码垛状态为完成，PLC识别该状态后，重新开始新的码垛流程。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_18', 'category': 'palletizing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.3.6', 'section_title': '1.3.6 实时显示', 'heading_path': ['1 码垛应用简介', '1.3 功能概述', '1.3.6 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），在“实时显示”中，应如何理解或操作“码垛任务过程中”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['码垛任务过程中，可查看各个拆码工位的实时码垛信息，基本垛型信息、工位状态、机器人姿态等', '方便用户实时掌握码垛作业进度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 475, 'line_end': 475, 'excerpt': '码垛任务过程中，可查看各个拆码工位的实时码垛信息，基本垛型信息、工位状态、机器人姿态等。方便用户实时掌握码垛作业进度。', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_19', 'category': 'palletizing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.8.2', 'section_title': '3.8.2 程序基础说明', 'heading_path': ['3 工作流程&功能', '3.8 编程', '3.8.2 程序基础说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），“程序基础说明”中“除了运动指令之外的逻辑指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['3. 除了运动指令之外的逻辑指令，包括：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1442, 'line_end': 1442, 'excerpt': '3. 除了运动指令之外的逻辑指令，包括：', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}, {'id': 'sample_palletizing_d12de074dd_20', 'category': 'palletizing', 'kind': 'alarm', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5', 'section_title': '5 故障排除', 'heading_path': ['5 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），“故障排除”列出的报警30007表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['30007'], 'expected_facts': ['30007 抓取作业生成失败,请检查抓取路径点 错误 抓取路径不符合要求 检查并修改抓取路径,使其符合说明书要求'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'line_start': 1663, 'line_end': 1670, 'excerpt': '<tr>\n<td><p>30007</p></td>\n<td><p>抓取作业生成失败,请检查抓取路径点</p></td>\n<td></td>\n<td><p>错误</p></td>\n<td><p>抓取路径不符合要求</p></td>\n<td><p>检查并修改抓取路径,使其符合说明书要求</p></td>\n</tr>', 'sha256': '4cac42a16919302868e36f2135a24cb7f4a011f440836cd35542f5041c7845e4'}]}]
register_sample_bank('palletizing', PALLETIZING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md', 'seed': 11131778095006722805, 'population': 93, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(141, 141), (304, 308), (138, 138), (192, 192), (223, 227), (437, 437), (143, 143), (309, 312), (139, 139), (169, 169), (282, 285), (490, 494), (365, 365), (188, 188), (357, 357), (122, 122), (273, 276), (185, 185), (343, 343), (445, 449)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md', 'seed': 16712533996109494487, 'population': 291, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1325, 1329), (439, 439), (996, 996), (1150, 1150), (1584, 1584), (357, 357), (1448, 1448), (1607, 1614), (1615, 1622), (495, 495), (508, 510), (483, 483), (1582, 1582), (1283, 1283), (1461, 1461), (658, 660), (1576, 1576), (475, 475), (1442, 1442), (1663, 1670)]}])


In [ ]:
# | notest
PALLETIZING_SAMPLE_IDS = None
palletizing_sample_cases = select_cases(PALLETIZING_SAMPLE_CASES, PALLETIZING_SAMPLE_IDS)
display(pd.DataFrame(palletizing_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in palletizing_sample_cases])
palletizing_sample_metrics = await evaluation.run(palletizing_sample_cases, concurrency=EVALUATION_CONCURRENCY)
palletizing_sample_results = {case["id"]: evaluation.load(case) for case in palletizing_sample_cases}
display(palletizing_sample_metrics)
# Inspect a complete result: evaluation.inspect(palletizing_sample_cases[0]["id"])
evaluation.show_answers(palletizing_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_palletizing_c7344ab204_01,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,6675,0,True,unsupported,,,abstained
sample_palletizing_c7344ab204_02,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,instruction,table_row,False,True,8,False,0,0,2,False,False,False,None,None,13395,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_03,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,6848,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_04,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,6228,0,True,unsupported,,,abstained
sample_palletizing_c7344ab204_05,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,instruction,table_row,False,True,8,False,0,0,2,False,False,False,False,False,9655,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_06,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,6365,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_07,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,6365,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_08,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,instruction,table_row,False,True,8,False,0,0,2,False,False,False,False,False,13395,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_09,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,procedure,prose,False,True,8,False,0,0,1,False,False,False,None,None,6767,0,True,unsupported,,,not reviewed
sample_palletizing_c7344ab204_10,palletizing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,6738,0,True,unsupported,,,not reviewed


### sample_palletizing_c7344ab204_01

根据《SX322002《新松机器人码垛应用操作手册》-V1.0》（V4.0 / V1.0），配置“码垛属性文件配置”时，“层高度”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。

#### Retrieved content summary

- 新松机器人码垛应用手册表格第一行，列标题依次为：序号、垛型名称、示例。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1310–1314; corpus 4.0, revision V5.0 / C-1; object `9e982dec52d916a5e51a5c601db7cd0a81fde7b68a19ea6603b81d2f00dcf4c1`.

- 新松机器人码垛应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 11–13; corpus 5.0, revision V5.0 / C-1; object `0c1e59b7b04ae94ea26bcc5048cd8d974162112a05d0ccef8ef3b064ac2ff6ff`.

- 新松机器人码垛应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 11–13; corpus 4.0, revision V5.0 / C-1; object `ff91c63721415163f70e38155cb975c2cb58c48f8553d5b8f9d15865e1e494c4`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- 码垛方式：重叠式（产品横/竖向排列，每层码放方式相同）；示意图：两个并排的3×3网格（左橙右黑线、右蓝），无具体机器人实体或参数。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1315–1319; corpus 5.0, revision V5.0 / C-1; object `14f3f12cbfef0241c3e99a787b53c3d620e4c86a24dd935e455cc36fd71131a0`.

- 在《新松机器人码垛应用操作手册》的“各作业者可执行作业一览”表中，操作“以操作盘复位报警”仅被赋予权限的角色（对应表中的两个“○”列）可执行，其余角色（对应空白列）无权执行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 150–155; corpus 4.0, revision V5.0 / C-1; object `35a335344e8067275d36a861e9ff4fdb7ed56c2d7b55d66d77d7101af2a10151`.

- 该表为《SX322002 新松机器人码垛应用操作手册》的修订记录，包含四列（文件版本、修订日期、修订人、主要变化）。目前仅有一行数据：C/1 版本于 2025 年 9 月由罗晓芳修订，主要变化为初版完成。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1739–1752; corpus 5.0, revision V5.0 / C-1; object `8eae16665a51bb0efcf5de4c95bea9c45c0c28b476cc0a26494a451e4deae8f3`.

- 码垛模式“纵横交错”（编号4）：同层码放方向相同，相邻层旋转90°；行内含对应示意图。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1330–1334; corpus 5.0, revision V5.0 / C-1; object `06f4ed88f8c1ed474dc4c51352d03a865c51891e544afbb73efd8baeb7192460`.

#### Answer

在所提供的检索证据（E1–E8）中，没有任何一条涉及

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: mixed.

The question asks for specific facts about '层高度' (layer height) from a V4.0/V1.0 manual. The expected fact is '6. 层高度：每层的高度，单位为mm，可以精确到0.001mm'. The supplied evidence (E1-E8) consists of excerpts from a V5.0/C-1 manual and does not contain the specific text regarding '层高度' parameters, units, or precision. The answer correctly identifies that the provided evidence does not contain the requested information ('在所提供的检索证据（E1–E8）中，没有任何一条涉及'). Since the necessary facts are missing from the supplied evidence, the refusal to answer with specific details is an appropriate abstention rather than a hallucination or incorrect answer.

...

#### Retrieved content summary

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- 新松机器人码垛应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 11–13; corpus 5.0, revision V5.0 / C-1; object `0c1e59b7b04ae94ea26bcc5048cd8d974162112a05d0ccef8ef3b064ac2ff6ff`.

- 新松机器人码垛应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 11–13; corpus 4.0, revision V5.0 / C-1; object `ff91c63721415163f70e38155cb975c2cb58c48f8553d5b8f9d15865e1e494c4`.

- 新松机器人码垛应用手册表格第一行，列标题依次为：序号、垛型名称、示例。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1310–1314; corpus 4.0, revision V5.0 / C-1; object `9e982dec52d916a5e51a5c601db7cd0a81fde7b68a19ea6603b81d2f00dcf4c1`.

- 摘要：文档编号 SX322001，名称为《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 5.0, revision V5.0 / C-1; object `39cf2401024003d288a583d3b5b383bfdaeb0e366ecdaf30f92d0e17dfefead4`.

- SX022001《新松机器人安全手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 242–244; corpus 4.0, revision V5.0 / C-1; object `ddc4af8489b5e88375ed41009cfe14823bb6f1eb7a4041b7fea8ac88e243ef64`.

- 在《新松机器人码垛应用操作手册》的“各作业者可执行作业一览”表中，操作“以操作盘复位报警”仅被赋予权限的角色（对应表中的两个“○”列）可执行，其余角色（对应空白列）无权执行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 150–155; corpus 4.0, revision V5.0 / C-1; object `35a335344e8067275d36a861e9ff4fdb7ed56c2d7b55d66d77d7101af2a10151`.

- **摘要**：
该表格行（索引3）展示了新松机器人码垛应用中的循环指令：**`WHILE 1 == 1`**，其功能为**开启一个无限循环**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1480–1483; corpus 4.0, revision V5.0 / C-1; object `7cc8bf57cdb5cb4f0e830bed82937fb44c374f22b1007aee34bb86eb8e0530ef`.

#### Answer

根据E8证据，SX322002手册

### Example — palletizing

在码垛程序中，操作员需要编写一个持续运行的主循环，使机器人不断检测来料信号并执行取放动作，而非仅执行一次运动指令。

**Environment and prerequisites:** 新松码垛机器人控制器已上电，程序编辑模式已打开；DI[8] 接来料光电传感器，DO[5] 接夹爪电磁阀；子作业文件 pickJob.txt 与 placeJob.txt 已存在于程序目录中。

**Expected result:** NOP 标记程序起点；WHILE 1 == 1 开启一个无限循环，使循环体内的逻辑指令（如 IF 判断、OUT 输出、DELAY 延时、CALL 子作业调用）被反复执行；ENDWHILE 终止该循环区间。整个结构构成除运动指令外的逻辑控制框架。

**Manual:** SX322002《新松机器人码垛应用操作手册》(C-1) V5.0 / C-1 [E8]

```text
NOP
WHILE 1 == 1
ENDWHILE
```

### sample_palletizing_d12de074dd_20

根据《SX322002《新松机器人码垛应用操作手册》(C-1)》（V5.0 / C-1），“故障排除”列出的报警30007表示什么，原文给出了哪些原因或处理办法？

#### Retrieved content summary

- 在《新松机器人码垛应用操作手册》的“各作业者可执行作业一览”表中，操作“以操作盘复位报警”仅被赋予权限的角色（对应表中的两个“○”列）可执行，其余角色（对应空白列）无权执行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 150–155; corpus 4.0, revision V5.0 / C-1; object `35a335344e8067275d36a861e9ff4fdb7ed56c2d7b55d66d77d7101af2a10151`.

- 错误代码 30001（码垛工艺生成失败）：原因为码垛工艺生成过程配置出现问题，解决方法是修改码垛工艺相关配置。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1615–1622; corpus 5.0, revision V5.0 / C-1; object `54eacafec0fdbff1bcda85289adf72aa20315a9e5b802eda02b13dde164e6f4f`.

- 错误 30001（码垛工艺生成失败）：原因为码垛工艺生成过程配置异常，建议修改码垛工艺相关配置。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 1615–1622; corpus 4.0, revision V5.0 / C-1; object `1ada7aac0ddd6afd2ce1c21b562f4838004b4ceb29e9bafd6d5f980ab4736bbe`.

- 报警码3007：伺服模块获取总线类型失败（原因：伺服驱动器信息加载错误），解决措施为复位或重启系统，若反复出现请联系新松售后人员。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 16375–16380; corpus 5.0, revision V5.0 / C-1; object `8676375ee4d45f08447acf635961e30f809837a55199c4ef5ed02c04be9aa736`.

- 报警码7020（新松机器人点焊应用），其余字段（报警信息、子码、子码说明、处理方法）为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 2048–2054; corpus 5.0, revision V5.0 / A-2; object `ee7974777b60cf935e7b523241ea75d4ff63a94fd43143304fa2d26da080abbc`.

- 新松机器人点焊应用中，报警码7000（焊接异常）下子码20~26表示“焊接故障”，处理方法为联系技术人员。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1685–1689; corpus 4.0, revision V5.0 / A-2; object `a9dccb0d4e507f6875375fa737413635ed7a2b4075b2479b94fda8d93e1eb3ae`.

- 新松机器人跟随应用操作手册（SX322017）报警表格定义行：列依次为报警码、报警等级、报警信息、报警原因、处理措施；无跨行/跨列合并，各单元格独立且未继承父级值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 604–610; corpus 4.0, revision V5.0 / A-0; object `73e63b2225c640d2b608ce544fc7c26d548d74458386b8b58a25e597fcfb4f66`.

- 报警码 **1302**（指令异常保护:SV）；报警原因未列出；解决措施：出现驱动器报警代码 27.0，若反复出现请联系新松售后人员。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 18299–18304; corpus 4.0, revision V5.0 / B-5; object `c92c0e7a9c0f9393b922497adb202877da34598798906695b4604a08613bedf3`.

#### Answer

该问题是对《SX322002《新松机器人码垛应用操作手册》(C-1)》(V5.0 / C-1) 故障排除章节中报警码 30007 的文档查阅。根据 E2 证据，报警 30007 的信息为：报警信息——

## Grinding — sampled manual revisions

40 cases from 2 CSV files. Set `GRINDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
GRINDING_SAMPLE_CASES = [{'id': 'sample_grinding_c126e40fe2_01', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.1.3', 'section_title': '1.1.3 力控执行器标定', 'heading_path': ['2 功能', '1.1 力控执行器配置', '1.1.3 力控执行器标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），操作“力控执行器标定”时，涉及“每次开始作业之前进行位移复位操作”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['每次开始作业之前进行位移复位操作', '首次使用打磨头或更换了部件或系统气源状态变化或控制效果不理想时，必须执行一次标定，顺序为：先位移复位、进而重量标定、最后自动标定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1072, 'line_end': 1072, 'excerpt': '每次开始作业之前进行位移复位操作。首次使用打磨头或更换了部件或系统气源状态变化或控制效果不理想时，必须执行一次标定，顺序为：先位移复位、进而重量标定、最后自动标定。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_02', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.3', 'section_title': '1.3 功能', 'heading_path': ['1 打磨应用简介', '1.3 功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“功能”中，应如何理解或操作“可通过获取机器人打磨焊缝的曲面高度”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['可通过获取机器人打磨焊缝的曲面高度，来判断此次打磨质量是否符合要求', '操作人员可结合质量检测结果来决定是否需要多次反复打磨'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 692, 'line_end': 692, 'excerpt': '可通过获取机器人打磨焊缝的曲面高度，来判断此次打磨质量是否符合要求。操作人员可结合质量检测结果来决定是否需要多次反复打磨。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_03', 'category': 'grinding', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1', 'section_title': '1 故障排除', 'heading_path': ['1 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），“故障排除”列出的报警7135表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7135'], 'expected_facts': ['7135 位移下限预警 控制器编号 警告 力控执行器位移超出范围'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1764, 'line_end': 1771, 'excerpt': '<tr>\n<td><p>7135</p></td>\n<td><p>位移下限预警</p></td>\n<td><p>控制器编号</p></td>\n<td><p>警告</p></td>\n<td><p>力控执行器位移超出范围</p></td>\n<td></td>\n</tr>', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_04', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '3.3', 'section_title': '3.3 编程规划及指导', 'heading_path': ['3 编程', '3.3 编程规划及指导'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），“编程规划及指导”中“在编写程序之前”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['在编写程序之前，您应该根据具体使用场景对程序进行规划', '规划包括在为机器人编程完成特定任务之前，考虑执行该任务的最佳方法', '这有助于您在编写程序时选择合适的指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1681, 'line_end': 1681, 'excerpt': '在编写程序之前，您应该根据具体使用场景对程序进行规划。规划包括在为机器人编程完成特定任务之前，考虑执行该任务的最佳方法。这有助于您在编写程序时选择合适的指令。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_05', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1', 'section_title': '1 故障排除', 'heading_path': ['1 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“故障排除”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 报警子码 报警 等级 报警原因 处理方法'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1724, 'line_end': 1731, 'excerpt': '<tr>\n<td><p><strong>报警码</strong></p></td>\n<td><p><strong>报警信息</strong></p></td>\n<td><p><strong>报警子码</strong></p></td>\n<td><p><strong>报警</strong></p><p><strong>等级</strong></p></td>\n<td><p><strong>报警原因</strong></p></td>\n<td><p><strong>处理方法</strong></p></td>\n</tr>', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_06', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.5.2', 'section_title': '1.5.2 打磨质量检测指令', 'heading_path': ['2 功能', '1.5 打磨质量检测', '1.5.2 打磨质量检测指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），“打磨质量检测指令”中“QualityCheck NumC=I1#开启打磨质量检测功能”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['I1#'], 'expected_facts': ['QualityCheck NumC=I1#开启打磨质量检测功能，I1中保存用于检测的力控执行器编号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1475, 'line_end': 1475, 'excerpt': 'QualityCheck NumC=I1#开启打磨质量检测功能，I1中保存用于检测的力控执行器编号', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_07', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '3.2', 'section_title': '3.2 程序说明', 'heading_path': ['3 编程', '3.2 程序说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“程序说明”中，应如何理解或操作“打开选中的程序后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['打开选中的程序后，进入作业编辑界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1647, 'line_end': 1647, 'excerpt': '打开选中的程序后，进入作业编辑界面。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_08', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.3', 'section_title': '1.3 功能', 'heading_path': ['1 打磨应用简介', '1.3 功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），配置“功能”时，“在示教模式下”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在示教模式下，以超级用户权限进行力控执行器的常规配置，方便机器人与力控执行器建立通讯、监控力控执行器的打磨力、位移、下刀力、出刀力等控制参数', '也可将力控执行器的输出IO映射到机器人的用户输出IO上，从而进行更灵活的安全控制', '当首次使用力控执行器或更改气源后需进行力控执行器的标定，从而进行力控执行器的初始化操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 684, 'line_end': 684, 'excerpt': '在示教模式下，以超级用户权限进行力控执行器的常规配置，方便机器人与力控执行器建立通讯、监控力控执行器的打磨力、位移、下刀力、出刀力等控制参数。也可将力控执行器的输出IO映射到机器人的用户输出IO上，从而进行更灵活的安全控制。当首次使用力控执行器或更改气源后需进行力控执行器的标定，从而进行力控执行器的初始化操作。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_09', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.7', 'section_title': '1.7 柜内力控控制器硬接线', 'heading_path': ['1 套机安装', '1.7 柜内力控控制器硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），配置“柜内力控控制器硬接线”时，“OUT2”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['OUT2 实际打磨力监控预警,超过设定范围则接通'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 869, 'line_end': 873, 'excerpt': '<tr>\n<td><p>OUT2</p></td>\n<td><p>实际打磨力监控预警,超过设定范围则接通</p></td>\n<td></td>\n</tr>', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_10', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.7', 'section_title': '2.7 力控执行器输出IO映射', 'heading_path': ['2 工作流程', '2.7 力控执行器输出IO映射'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“力控执行器输出IO映射”中，应如何理解或操作“[打磨应用-IO配置]进入力控执行器输出IO映射界面”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['[打磨应用-IO配置]进入力控执行器输出IO映射界面，输入对应力控执行器输出IO的机器人IO号', '机器人IO号可按现场实际情况配，机器人用户输出IO一般是从33号开始', '配置后按对应测试方式验证IO是否输出（在实时显示中查看）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 996, 'line_end': 996, 'excerpt': '[打磨应用-IO配置]进入力控执行器输出IO映射界面，输入对应力控执行器输出IO的机器人IO号。机器人IO号可按现场实际情况配，机器人用户输出IO一般是从33号开始。配置后按对应测试方式验证IO是否输出（在实时显示中查看）。其中电机回0的输出IO较特殊，置位较快，可以编写后台作业，监控该信号是否输出。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_11', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.8.2', 'section_title': '1.8.2 快捷键设置', 'heading_path': ['2 功能', '1.8 快捷键设置', '1.8.2 快捷键设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“快捷键设置”中，应如何理解或操作“[力加载]”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. [力加载]:加载打磨力按钮，与之对应的硬按键为op2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1587, 'line_end': 1587, 'excerpt': '5. **[力加载]:**加载打磨力按钮，与之对应的硬按键为op2', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_12', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '3.3.3', 'section_title': '3.3.3 为摆动打磨任务编写作业', 'heading_path': ['3 编程', '3.3 编程规划及指导', '3.3.3 为摆动打磨任务编写作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），配置“为摆动打磨任务编写作业”时，“要使用摆动打磨时”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['要使用摆动打磨时，首先进行摆动库参数设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1701, 'line_end': 1701, 'excerpt': '要使用摆动打磨时，首先进行摆动库参数设置。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_13', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.1.3', 'section_title': '1.1.3 力控执行器标定', 'heading_path': ['2 功能', '1.1 力控执行器配置', '1.1.3 力控执行器标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），配置“力控执行器标定”时，“自动标定”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['3. 自动标定：输入[打磨力范围]和[打磨角度范围]，点击[标定]按钮力控执行器自动完成一系列动作之后完成自动标定', '标定时设置的力和角度范围实际会影响气源装置，所以标定完成后，在打磨过程中设置的力和磨头角度就要在这个范围中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1078, 'line_end': 1078, 'excerpt': '3. 自动标定：输入**[打磨力范围]**和**[打磨角度范围]**，点击**[标定]**按钮力控执行器自动完成一系列动作之后完成自动标定；标定时设置的力和角度范围实际会影响气源装置，所以标定完成后，在打磨过程中设置的力和磨头角度就要在这个范围中。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_14', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.7', 'section_title': '1.7 柜内力控控制器硬接线', 'heading_path': ['1 套机安装', '1.7 柜内力控控制器硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“柜内力控控制器硬接线”中，应如何理解或操作“注释：其中IN3默认通过硬接线置1，来启用出刀力。其余IO功能可根据客户需求选配硬接线到机器人上”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['注释：其中IN3默认通过硬接线置1，来启用出刀力', '其余IO功能可根据客户需求选配硬接线到机器人上'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 907, 'line_end': 909, 'excerpt': '<tr>\n<td><p><a id="_Toc530061087"></a><a id="_Toc88571481"></a><a id="_Toc531013711"></a><a id="_Toc530141847"></a><a id="_Toc515305857"></a><a id="_Toc89069306"></a><a id="_Toc160525934"></a><a id="_Toc532819118"></a><a id="_Toc515282538"></a><strong>注释：其中IN3默认通过硬接线置1，来启用出刀力。其余IO功能可根据客户需求选配硬接线到机器人上</strong></p></td>\n</tr>', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_15', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.4', 'section_title': '2.4 跟力控执行器通讯', 'heading_path': ['2 工作流程', '2.4 跟力控执行器通讯'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），配置“跟力控执行器通讯”时，“打开[打磨应用-通讯参数]”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['打开[打磨应用-通讯参数]，先勾选使能复选框，再点击[保存]，当连接状态由OFF变为ON时，表示力控执行器连接成功'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 938, 'line_end': 938, 'excerpt': '打开[打磨应用-通讯参数]，先勾选使能复选框，再点击[保存]，当连接状态由OFF变为ON时，表示力控执行器连接成功。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_16', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.6', 'section_title': '2.6 打磨力、位移、出入刀监控设置', 'heading_path': ['2 工作流程', '2.6 打磨力、位移、出入刀监控设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），“打磨力、位移、出入刀监控设置”中“界面中的“打磨力设置”可按需进行设置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['界面中的“打磨力设置”可按需进行设置，如果此处成功设置了打磨力，则作业中可以省却用打磨力设置指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 970, 'line_end': 970, 'excerpt': '界面中的“打磨力设置”可按需进行设置，如果此处成功设置了打磨力，则作业中可以省却用打磨力设置指令。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_17', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '2.6', 'section_title': '2.6 打磨力、位移、出入刀监控设置', 'heading_path': ['2 工作流程', '2.6 打磨力、位移、出入刀监控设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），操作“打磨力、位移、出入刀监控设置”时，涉及“时间”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['时间：>=0，单位毫秒，与工件接触的时刻，第一个值必须为 0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 978, 'line_end': 978, 'excerpt': '时间：>=0，单位毫秒，与工件接触的时刻，第一个值必须为 0。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_18', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.5', 'section_title': '1.5 打磨工具安装方式', 'heading_path': ['1 套机安装', '1.5 打磨工具安装方式'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），操作“打磨工具安装方式”时，涉及“注意： 安装时机器人六轴关节值调整至90°”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意： 安装时机器人六轴关节值调整至90°'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 733, 'line_end': 735, 'excerpt': '<tr>\n<td><img src="img/25dc0dc552eb1f5b5f2e14db79ace0b2bd7b5ad4ed8222d6293052bead37bcd4.png"/><p><strong>注意：</strong></p><p><strong>安装时机器人六轴关节值调整至90°</strong></p></td>\n</tr>', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_19', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.1.4', 'section_title': '1.1.4 力控执行器控制参数设置', 'heading_path': ['2 功能', '1.1 力控执行器配置', '1.1.4 力控执行器控制参数设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），操作“力控执行器控制参数设置”时，涉及“--时间”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. --时间：以出刀力控制信号被触发的时刻为 0 时刻，第一个值必须为 0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 1117, 'line_end': 1117, 'excerpt': '   1. --时间：以出刀力控制信号被触发的时刻为 0 时刻，第一个值必须为 0', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_c126e40fe2_20', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-1', 'section': '1.4.1', 'section_title': '1.4.1 硬件限制', 'heading_path': ['1 打磨应用简介', '1.4 限制', '1.4.1 硬件限制'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'source_version': '', 'question': '根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），在“硬件限制”中，应如何理解或操作“力控打磨工具仅适配本体型号为SR25A-35/1.80、SR50A-50/2.10、SR50A-”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['力控打磨工具仅适配本体型号为SR25A-35/1.80、SR50A-50/2.10、SR50A-70/2.10的机器人，力控打磨工具控制器（以下简称力控执行器）的型号为ICG04'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'line_start': 718, 'line_end': 718, 'excerpt': '力控打磨工具仅适配本体型号为SR25A-35/1.80、SR50A-50/2.10、SR50A-70/2.10的机器人，力控打磨工具控制器（以下简称力控执行器）的型号为ICG04。', 'sha256': '68eee8e3267494ad125ec5b7208b12487375eedbcf2e29c2d7961d5aaca54249'}]}, {'id': 'sample_grinding_ee21a9c44b_01', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.2', 'section_title': '3.2 程序说明', 'heading_path': ['3 编程', '3.2 程序说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），“程序说明”中“指示机器人从当前位置以预期方式移动到目标位置的运动指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 指示机器人从当前位置以预期方式移动到目标位置的运动指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1438, 'line_end': 1438, 'excerpt': '2. 指示机器人从当前位置以预期方式移动到目标位置的运动指令。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_02', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.2.1', 'section_title': '1.2.1 SR50套机', 'heading_path': ['1 打磨应用简介', '1.2 物理组件', '1.2.1 SR50套机'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“SR50套机”中，应如何理解或操作“SR50套机主要应用于黑色金属打磨应用场景”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['SR50套机主要应用于黑色金属打磨应用场景，其组成见下图：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 449, 'line_end': 449, 'excerpt': 'SR50套机主要应用于黑色金属打磨应用场景，其组成见下图：', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_03', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.7.1', 'section_title': '2.7.1 双头模型一', 'heading_path': ['2 工作流程', '2.7 工具坐标系标定', '2.7.1 双头模型一'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“双头模型一”中，应如何理解或操作“工具系1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['工具系1：Px:154.45+红框处距离、Py:0、Pz:285.5、Rx:0、Ry:0、Rz:0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 803, 'line_end': 803, 'excerpt': '工具系1：Px:154.45+红框处距离、Py:0、Pz:285.5、Rx:0、Ry:0、Rz:0', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_04', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.3.3', 'section_title': '3.3.3 为摆动打磨任务编写作业', 'heading_path': ['3 编程', '3.3 编程规划及指导', '3.3.3 为摆动打磨任务编写作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“为摆动打磨任务编写作业”时，“要使用摆动打磨时”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['要使用摆动打磨时，首先进行摆动库参数设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1471, 'line_end': 1471, 'excerpt': '要使用摆动打磨时，首先进行摆动库参数设置。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_05', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.3', 'section_title': '1.3 功能', 'heading_path': ['1 打磨应用简介', '1.3 功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“功能”时，“在打磨方向以特定的角度和幅度周期性的摇摆进行打磨”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在打磨方向以特定的角度和幅度周期性的摇摆进行打磨，由此增大打磨路径宽度并提高打磨强度和效率', '打磨应用摆动库可以存储110个摆动库参数，支持多种折线型摆动方式'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 491, 'line_end': 491, 'excerpt': '在打磨方向以特定的角度和幅度周期性的摇摆进行打磨，由此增大打磨路径宽度并提高打磨强度和效率。打磨应用摆动库可以存储110个摆动库参数，支持多种折线型摆动方式。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_06', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.4.3', 'section_title': '1.4.3 摆动打磨指令', 'heading_path': ['2 功能', '1.4 摆动打磨', '1.4.3 摆动打磨指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），“摆动打磨指令”中“摆动打磨指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['摆动打磨指令，包含摆动开始、摆动结束指令以及摆动运动指令', '这些指令是使机器人执行摆动打磨动作的必要指令', '在摆动开始指令和摆动结束指令之间所示教的动作指令，都会执行摆动动作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1179, 'line_end': 1179, 'excerpt': '摆动打磨指令，包含摆动开始、摆动结束指令以及摆动运动指令。这些指令是使机器人执行摆动打磨动作的必要指令。在摆动开始指令和摆动结束指令之间所示教的动作指令，都会执行摆动动作。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_07', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.1.3', 'section_title': '1.1.3 力控制器标定', 'heading_path': ['2 功能', '1.1 力控制器配置', '1.1.3 力控制器标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“力控制器标定”时，“[工具编号]”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['6. [工具编号]：力控制器的工具编号，设置范围1~8'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 980, 'line_end': 980, 'excerpt': '6. **[工具编号]：**力控制器的工具编号，设置范围1~8；', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_08', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.1.4', 'section_title': '1.1.4 力控制器控制参数设置', 'heading_path': ['2 功能', '1.1 力控制器配置', '1.1.4 力控制器控制参数设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），操作“力控制器控制参数设置”时，涉及“--时间”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. --时间：以出刀力控制信号被触发的时刻为 0 时刻，第一个值必须为 0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1006, 'line_end': 1006, 'excerpt': '   1. --时间：以出刀力控制信号被触发的时刻为 0 时刻，第一个值必须为 0', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_09', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.3.2', 'section_title': '1.3.2 路径规划设置', 'heading_path': ['2 功能', '1.3 打磨轨迹快速生成', '1.3.2 路径规划设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“路径规划设置”中，应如何理解或操作“[起始轨迹点]”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['7. [起始轨迹点]：示教起始轨迹点，即打磨入刀轨迹'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1093, 'line_end': 1093, 'excerpt': '7. **[起始轨迹点]：**示教起始轨迹点，即打磨入刀轨迹；', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_10', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.5', 'section_title': '2.5 打磨力、位移、出入刀监控设置', 'heading_path': ['2 工作流程', '2.5 打磨力、位移、出入刀监控设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），“打磨力、位移、出入刀监控设置”中“界面中的“打磨力设置”可按需进行设置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['界面中的“打磨力设置”可按需进行设置，如果此处成功设置了打磨力，则作业中可以省却用打磨力设置指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 761, 'line_end': 761, 'excerpt': '界面中的“打磨力设置”可按需进行设置，如果此处成功设置了打磨力，则作业中可以省却用打磨力设置指令。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_11', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.5', 'section_title': '2.5 打磨力、位移、出入刀监控设置', 'heading_path': ['2 工作流程', '2.5 打磨力、位移、出入刀监控设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），操作“打磨力、位移、出入刀监控设置”时，涉及“时间”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['时间：>=0，单位毫秒，与工件接触的时刻，第一个值必须为 0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 769, 'line_end': 769, 'excerpt': '时间：>=0，单位毫秒，与工件接触的时刻，第一个值必须为 0。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_12', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.4', 'section_title': '2.4 首次使用时进行标定', 'heading_path': ['2 工作流程', '2.4 首次使用时进行标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“首次使用时进行标定”时，“自动标定”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['3. 自动标定：输入[打磨力范围]-150.300（默认）和[打磨角度范围]-90.90，点击[标定]按钮力控制器自动完成一系列动作之后完成自动标定', '完成后会进行标定成功的提示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 747, 'line_end': 747, 'excerpt': '3. 自动标定：输入[打磨力范围]-150.300（默认）和[打磨角度范围]-90.90，点击[标定]按钮力控制器自动完成一系列动作之后完成自动标定；完成后会进行标定成功的提示。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_13', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1', 'section_title': '1 故障排除', 'heading_path': ['1 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），“故障排除”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['TCP', 'WVOFF'], 'expected_facts': ['报警码 报警信息 报警子码 报警 等级 报警原因 处理方法 7131 水气单元报警 气压报警 警告 外部设备气压异常 7132 没有需要恢复的打磨参数文件 警告 执行备份恢复时，U盘中没有可恢复的打磨参数文件 将需要会的打磨参数文件复制到U盘中，再进行恢复 7133 电机转速不正常报警 控制器编号 警告 打磨电机没有达到转速 7134 控制器未连接 控制器编号 警告 力控制器未使能', '力控制器IP和端口号配置错误', '力控制器网线连接不正常 检查机器人与力控制器之间的网线连接是否正常'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1493, 'line_end': 1493, 'excerpt': '<table><tr><td><p><strong>报警码</strong></p></td><td><p><strong>报警信息</strong></p></td><td><p><strong>报警子码</strong></p></td><td><p><strong>报警</strong></p><p><strong>等级</strong></p></td><td><p><strong>报警原因</strong></p></td><td><p><strong>处理方法</strong></p></td></tr><tr><td><p>7131</p></td><td><p>水气单元报警</p></td><td><p>气压报警</p></td><td><p>警告</p></td><td><p>外部设备气压异常</p></td><td></td></tr><tr><td><p>7132</p></td><td><p>没有需要恢复的打磨参数文件</p></td><td></td><td><p>警告</p></td><td><p>执行备份恢复时，U盘中没有可恢复的打磨参数文件</p></td><td><p>将需要会的打磨参数文件复制到U盘中，再进行恢复</p></td></tr><tr><td><p>7133</p></td><td><p>电机转速不正常报警</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>打磨电机没有达到转速</p></td><td></td></tr><tr><td><p>7134</p></td><td><p>控制器未连接</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>力控制器未使能；</p><p>力控制器IP和端口号配置错误；</p><p>力控制器网线连接不正常</p></td><td><p>检查机器人与力控制器之间的网线连接是否正常；</p><p>检查机器人IP地址是否与力控制器的IP地址在同一网段</p><p>检查[打磨应用-通讯参数]的IP地址和端口与力控制器的IP地址和端口是否一致，且使能。</p></td></tr><tr><td><p>7135</p></td><td><p>位移下限预警</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>力控制器位移超出范围</p></td><td></td></tr><tr><td><p>7136</p></td><td><p>断点恢复超出范围,请移动机器人位置</p></td><td></td><td><p>警告</p></td><td><p>在开启断点恢复的情况下，机器人当前位置与要恢复的位置距离超出设定距离</p></td><td><p>将机器人位置手动朝断点恢复位置移动一段距离或者按清堆栈取消断点恢复</p></td></tr><tr><td><p>7137</p></td><td><p>位移上限预警</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>力控制器位移超出范围</p></td><td></td></tr><tr><td><p>7138</p></td><td><p>实际打磨力预警</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>力控制器实际打磨力超出范围</p></td><td></td></tr><tr><td><p>7139</p></td><td><p>发送数据失败</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>力控制器没有成功连接</p></td><td><p>处理方式同上</p></td></tr><tr><td><p>7140</p></td><td><p>数据回复超时</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>向力控制器发送控制指令，没有收到力控制器的回复消息。</p></td><td><p>检查力控制器是否成功连接，如果未能成功连接处理方法同上。</p></td></tr><tr><td><p>7141</p></td><td><p>指令参数错误</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>1、指令参数中通过I变量或者R变量设置的输入参数，输入值超出范围的。</p></td><td><p>检查指令参数，是否超出范围。</p></td></tr><tr><td><p>7142</p></td><td><p>控制器硬件断开连接</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>网线断开或网线虚连或者力控制器未开机</p></td><td><p>检查网线连接情况，重新插拔一下网线。</p></td></tr><tr><td><p>7143</p></td><td><p>控制器TCP断开连接</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>网线断开或网线虚连或者力控制器未开机</p></td><td><p>检查网线连接情况，重新插拔一下网线。</p></td></tr><tr><td><p>7144</p></td><td><p>电机回原位失败</p></td><td><p>控制器编号</p></td><td><p>警告</p></td><td><p>电机回原点失败</p></td><td><p>联系技术员</p></td></tr><tr><td><p>7145</p></td><td><p>打磨质量检测失败</p></td><td><p>1：参数错误</p><p>2：未使能打磨质量检测功能</p><p>3：计算失败</p></td><td><p>警告</p></td><td><p>NumC =I#参数1指令中，I变量值超出范围</p><p>GetResult Re指令中Re参数对应的I变量编码超出范围；</p><p>未执行QualityCheck指令使能质量检测开关</p><p>打磨质量算法计算失败</p></td><td><p>联系技术员</p></td></tr><tr><td><p>7146</p></td><td><p>打磨应用断点恢复位置失败</p></td><td><p>无</p></td><td><p>警告</p></td><td></td><td><p>一般不会出现这种报警，如果出现，联系开发人员</p></td></tr><tr><td><p>7147</p></td><td><p>打磨应用断点恢复力控设备失败</p></td><td><p>无</p></td><td><p>警告</p></td><td rowspan="2"><p>力控制器通讯出现异常。暂停、急停或机器人故障时，会同步停止力控制器力输出和电机转动，如果机器人跟力控制器通讯失败会出现此警告</p></td><td rowspan="2"><p>可按照7134报警的处理方式进行处理</p></td></tr><tr><td><p>7148</p></td><td><p>打磨应用停止力控设备失败</p></td><td><p>无</p></td><td><p>警告</p></td></tr><tr><td><p>7149</p></td><td><p>无可用摆动点位,请先单步执行WVOFF指令,在进行后续过程</p></td><td><p>无</p></td><td><p>警告</p></td><td></td><td><p>单步执行WVOFF指令可解决</p></td></tr><tr><td><p>7100</p></td><td><p>刀柄1松刀</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>刀柄1松刀</p></td><td></td></tr><tr><td><p>7101</p></td><td><p>刀柄1夹刀</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>刀柄1夹刀</p></td><td></td></tr><tr><td><p>7102</p></td><td><p>刀柄2松刀</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>刀柄2松刀</p></td><td></td></tr><tr><td><p>7103</p></td><td><p>刀柄2夹刀</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>刀柄2夹刀</p></td><td></td></tr><tr><td><p>7104</p></td><td><p>电机位置归零成功</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>电机位置归零成功</p></td><td></td></tr><tr><td><p>7106</p></td><td><p>控制器TCP连接成功</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>力控制器连接成功</p></td><td></td></tr><tr><td><p>7107</p></td><td><p>位移下限预警</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>实际位移&lt;位移下限</p></td><td></td></tr><tr><td><p>7108</p></td><td><p>位移上限预警</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>实际位移&gt;位移上限</p></td><td></td></tr><tr><td><p>7109</p></td><td><p>实际打磨力预警</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>实际打磨力和设定打磨力误差超过打磨力预警</p></td><td></td></tr><tr><td><p>7110</p></td><td><p>电机转速不正常报警</p></td><td><p>控制器编号</p></td><td><p>提示</p></td><td><p>电机未达到设定的转速值</p></td><td></td></tr><tr><td><p>7111</p></td><td><p>打磨应用断点恢复成功</p></td><td><p>无</p></td><td><p>提示</p></td><td></td><td></td></tr><tr><td><p>7112</p></td><td><p>打磨应用断点恢复位置超出范围</p></td><td><p>无</p></td><td><p>提示</p></td><td><p>在开启断点恢复的情况下，机器人当前位置与要恢复的位置距离超出设定距离</p></td><td><p>将机器人位置手动朝断点恢复位置移动一段距离或者按清堆栈取消断点恢复</p></td></tr><tr><td><p>7113</p></td><td><p>水气单元报警</p></td><td><p>气压报警</p></td><td><p>提示</p></td><td><p>外部设备气压异常</p></td><td></td></tr><tr><td><p>7114</p></td><td><p>标定完成</p></td><td><p>1、重量标定完成完<br/>2、自动标定完成</p></td><td><p>提示</p></td><td></td><td></td></tr><tr><td><p>7115</p></td><td><p>快捷键功能</p></td><td><p>1、力加载完成<br/>2、电机启停执行完成<br/>3、夹刀执行完成<br/>4、松刀执行完成<br/>5、电机回零执行完成<br/>6、位移复位执行完成<br/>7、监控开关打开执行完成<br/>8、监控开关关闭执行完成<br/>9、全速开关打开执行完成<br/>10、全速开关关闭执行完成<br/>11、工具启停执行完成</p></td><td><p>提示</p></td><td><p>正在操作快捷按键</p></td><td></td></tr><tr><td><p>7116</p></td><td><p>标定进行中，请勿移动机器人</p></td><td></td><td><p>提示</p></td><td><p>正在标定力控</p></td><td></td></tr></table>', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_14', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.5.2', 'section_title': '1.5.2 SR25套机', 'heading_path': ['1 套机安装', '1.5 .打磨工具安装方式', '1.5.2 SR25套机'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“SR25套机”中，应如何理解或操作“SR25套机3C打磨应用场景(气动打磨头)—机器人末端工具安装方式”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['SR25套机3C打磨应用场景(气动打磨头)—机器人末端工具安装方式，如图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 541, 'line_end': 541, 'excerpt': 'SR25套机3C打磨应用场景(气动打磨头)—机器人末端工具安装方式，如图所示。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_15', 'category': 'grinding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.7', 'section_title': '1.7 柜内力控控制器硬接线', 'heading_path': ['1 套机安装', '1.7 柜内力控控制器硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），操作“柜内力控控制器硬接线”时，涉及“IO 功能 备注 IN1 设定打磨压力 打磨力I/0真值表在M8412XXB触摸屏界面上设置”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['IO 功能 备注 IN1 设定打磨压力 打磨力I/0真值表在M8412XXB触摸屏界面上设置', '00:打磨力1 01:打磨力2 10:打磨力3 11:打磨力4 IN2 设定打磨压力 IN3 触发出刀力 出刀力 IN4 电机位置归零 IN5 位移传感器复位 IN6 电机转速控制 转速I/0真值表在M8412XXB触摸屏界面上', '00:电机停止(必须设为固定值0) 01:转速1 10:转速2 11:转速3 IN7 电机转速控制 IN8 电机位置传感器信号 OUT1 换刀1 控制换刀1气管的通断 OUT2 实际打磨力监控预警,超过设定范围则接通 OUT3 位移传感小于设定值则接通 OUT4 位移传感大于设定值则接通 OUT5 换刀2 控制换刀2气管的通断 OUT6 电机位置归零状态指示 断开:未完成;接通:归零成功 OUT7 电机转速不正常报警 断开:电机转速正常;接通:电机转速不正常 OUT8 电机位置传感器控制'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 653, 'line_end': 653, 'excerpt': '<table><tr><td><p><strong>IO</strong></p></td><td><p><strong>功能</strong></p></td><td><p><strong>备注</strong></p></td></tr><tr><td><p>IN1</p></td><td><p>设定打磨压力</p></td><td rowspan="2"><p>打磨力I/0真值表在M8412XXB触摸屏界面上设置。</p><p>00:打磨力1 </p><p>01:打磨力2</p><p>10:打磨力3</p><p>11:打磨力4</p></td></tr><tr><td><p>IN2</p></td><td><p>设定打磨压力</p></td></tr><tr><td><p>IN3</p></td><td><p>触发出刀力</p></td><td><p>出刀力</p></td></tr><tr><td><p>IN4</p></td><td><p>电机位置归零</p></td><td></td></tr><tr><td><p>IN5</p></td><td><p>位移传感器复位</p></td><td></td></tr><tr><td><p>IN6</p></td><td><p>电机转速控制</p></td><td rowspan="2"><p>转速I/0真值表在M8412XXB触摸屏界面上。</p><p>00:电机停止(必须设为固定值0)</p><p>01:转速1</p><p>10:转速2</p><p>11:转速3</p></td></tr><tr><td><p>IN7</p></td><td><p>电机转速控制</p></td></tr><tr><td><p>IN8</p></td><td><p>电机位置传感器信号</p></td><td></td></tr><tr><td><p>OUT1</p></td><td><p>换刀1</p></td><td><p>控制换刀1气管的通断</p></td></tr><tr><td><p>OUT2</p></td><td><p>实际打磨力监控预警,超过设定范围则接通</p></td><td></td></tr><tr><td><p>OUT3</p></td><td><p>位移传感小于设定值则接通</p></td><td></td></tr><tr><td><p>OUT4</p></td><td><p>位移传感大于设定值则接通</p></td><td></td></tr><tr><td><p>OUT5</p></td><td><p>换刀2</p></td><td><p>控制换刀2气管的通断</p></td></tr><tr><td><p>OUT6</p></td><td><p>电机位置归零状态指示</p></td><td><p>断开:未完成;接通:归零成功</p></td></tr><tr><td><p>OUT7</p></td><td><p>电机转速不正常报警</p></td><td><p>断开:电机转速正常;接通:电机转速不正常</p></td></tr><tr><td><p>OUT8</p></td><td><p>电机位置传感器控制</p></td><td></td></tr></table>', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_16', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.1', 'section_title': '3.1 概述', 'heading_path': ['3 编程', '3.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“概述”中，应如何理解或操作“进行点焊、喷漆、弧焊或打磨等操作”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 进行点焊、喷漆、弧焊或打磨等操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1407, 'line_end': 1407, 'excerpt': '2. **进行点焊、喷漆、弧焊或打磨等操作；**', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_17', 'category': 'grinding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.2', 'section_title': '3.2 程序说明', 'heading_path': ['3 编程', '3.2 程序说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），“程序说明”中“对机器人内部变量（包括位置变量）赋值”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['5. 对机器人内部变量（包括位置变量）赋值，计算，运算的指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 1441, 'line_end': 1441, 'excerpt': '5. 对机器人内部变量（包括位置变量）赋值，计算，运算的指令。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_18', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.4.2', 'section_title': '1.4.2 软件限制', 'heading_path': ['1 打磨应用简介', '1.4 限制', '1.4.2 软件限制'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“软件限制”中，应如何理解或操作“打磨应用仅限在RC软件版本4.8.121.8之上配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['打磨应用仅限在RC软件版本4.8.121.8之上配置', '力控制器的固件版本应在20240902之上', '打磨应用的版本应在V2.3.2.2之上，打磨应用与其他应用暂不兼容'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 517, 'line_end': 517, 'excerpt': '打磨应用仅限在RC软件版本4.8.121.8之上配置。力控制器的固件版本应在20240902之上。打磨应用的版本应在V2.3.2.2之上，打磨应用与其他应用暂不兼容。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_19', 'category': 'grinding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.7.1', 'section_title': '2.7.1 双头模型一', 'heading_path': ['2 工作流程', '2.7 工具坐标系标定', '2.7.1 双头模型一'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），在“双头模型一”中，应如何理解或操作“双头模型则需要设定2个坐标系”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['双头模型则需要设定2个坐标系，打磨应用工具坐标系无需设置方向，可根据打磨头的安装位置和磨头尺寸直接进行设定'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 797, 'line_end': 797, 'excerpt': '双头模型则需要设定2个坐标系，打磨应用工具坐标系无需设置方向，可根据打磨头的安装位置和磨头尺寸直接进行设定。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}, {'id': 'sample_grinding_ee21a9c44b_20', 'category': 'grinding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.3', 'section_title': '1.3 功能', 'heading_path': ['1 打磨应用简介', '1.3 功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“功能”时，“在示教模式下”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在示教模式下，以超级用户权限进行力控制器的常规配置，方便机器人与力控制器建立通讯、监控力控制器的打磨力、位移、下刀力、出刀力等控制参数', '也可将力控制器的输出IO映射到机器人的用户输出IO上，从而进行更灵活的安全控制', '当首次使用力控制器或更改气源后需进行力控制器的标定，从而进行力控制器的初始化操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'line_start': 479, 'line_end': 479, 'excerpt': '在示教模式下，以超级用户权限进行力控制器的常规配置，方便机器人与力控制器建立通讯、监控力控制器的打磨力、位移、下刀力、出刀力等控制参数。也可将力控制器的输出IO映射到机器人的用户输出IO上，从而进行更灵活的安全控制。当首次使用力控制器或更改气源后需进行力控制器的标定，从而进行力控制器的初始化操作。', 'sha256': '835f6c1e4cb46f98e64e452d3316e78ae7e81bffac6f31c9e541a02e9bac0f62'}]}]
register_sample_bank('grinding', GRINDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md', 'seed': 14868358824356594603, 'population': 299, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1072, 1072), (692, 692), (1764, 1771), (1681, 1681), (1724, 1731), (1475, 1475), (1647, 1647), (684, 684), (869, 873), (996, 996), (1587, 1587), (1701, 1701), (1078, 1078), (907, 909), (938, 938), (970, 970), (978, 978), (733, 735), (1117, 1117), (718, 718)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md', 'seed': 6896981894458189149, 'population': 262, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1438, 1438), (449, 449), (803, 803), (1471, 1471), (491, 491), (1179, 1179), (980, 980), (1006, 1006), (1093, 1093), (761, 761), (769, 769), (747, 747), (1493, 1493), (541, 541), (653, 653), (1407, 1407), (1441, 1441), (517, 517), (797, 797), (479, 479)]}])


In [ ]:
# | notest
GRINDING_SAMPLE_IDS = None
grinding_sample_cases = select_cases(GRINDING_SAMPLE_CASES, GRINDING_SAMPLE_IDS)
display(pd.DataFrame(grinding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in grinding_sample_cases])
grinding_sample_metrics = await evaluation.run(grinding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
grinding_sample_results = {case["id"]: evaluation.load(case) for case in grinding_sample_cases}
display(grinding_sample_metrics)
# Inspect a complete result: evaluation.inspect(grinding_sample_cases[0]["id"])
evaluation.show_answers(grinding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_grinding_c126e40fe2_01,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,8367,0,True,unsupported,,,abstained
sample_grinding_c126e40fe2_02,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,4829,0,True,unsupported,,,not reviewed
sample_grinding_c126e40fe2_03,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,alarm,table_row,False,True,8,False,0,0,1,False,False,False,False,False,17595,0,True,unsupported,,,partial
sample_grinding_c126e40fe2_04,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,6300,0,True,unsupported,,,abstained
sample_grinding_c126e40fe2_05,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,procedure,table_row,False,True,8,False,0,0,6,False,False,False,None,None,22885,3,True,unsupported,,,not reviewed
sample_grinding_c126e40fe2_06,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,instruction,prose,False,True,8,False,0,0,0,False,False,False,False,False,6518,0,True,included,,,not reviewed
sample_grinding_c126e40fe2_07,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,6302,0,True,unsupported,,,not reviewed
sample_grinding_c126e40fe2_08,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,6436,0,True,unsupported,,,not reviewed
sample_grinding_c126e40fe2_09,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,8424,0,True,unsupported,,,not reviewed
sample_grinding_c126e40fe2_10,grinding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md,V4.0 / A-1,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,6386,0,True,unsupported,,,not reviewed


### sample_grinding_c126e40fe2_01

根据《SX322006新松机器人打磨应用操作手册（A-1)》（V4.0 / A-1），操作“力控执行器标定”时，涉及“每次开始作业之前进行位移复位操作”的前提条件或注意事项是什么？

#### Retrieved content summary

- 行摘要：新松机器人打磨应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 15–17; corpus 4.0, revision V5.0 / A-2; object `9a74bea2928cbb282591199f3ac9cb542a9c1d0e1be095a6749ba1e794a16dae`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 文档《新松机器人打磨应用操作手册》（A-2）的修订记录行：版本 A/2，日期 2025 年 4 月 16 日，修改人 罗晓芳，内容 修改通讯配置与快捷键功能描述，并增加气动型力控制器说明。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 46–51; corpus 4.0, revision V5.0 / A-2; object `4465e396ac7d4a20aac4f15aebe2da2f3d2f06afb17ac4ed5e9f7ff348f05afc`.

- 新松机器人打磨应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 15–17; corpus 5.0, revision V5.0 / A-2; object `3a870ca31f3478e8879735898218bdd317048cafed3d83bef258c00f3d854793`.

- 新松机器人打磨应用手册中的操作项“以示教器的数据设定”，其第2列内容为空，第3列和第4列均标记为“〇”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 282–287; corpus 5.0, revision V5.0 / A-2; object `9647ba66666726896e7c2752eca95795e738cd754fc91f3f9ffde4f049ee52fb`.

- 该表格汇总了新松机器人打磨应用的指令列表，采用三列结构（指令名称、参数、功能），并对每条指令提供了“绝对指令”（带数据类型标记如 `I#`/`R#`）和“相对指令”（无标记）两种参数形式。主要指令摘要如下：

*   **SetForce**：**设置打磨力**。参数包括控制器编号 (NumC, 1~8)、力大小 (Force, 依标定范围定) 和通讯延时阈值 (T, 超时则报警)。
*   **SetMotorV**：**设置电机转速**。参数包括控制器编号 (NumC, 1~8)、目标转速 (Speed) 和延时设定 (T, 未在时限内达到目标转速则报警)。
*   **SetGDRE**（仅限电动力控）：**控制换刀气阀**。参数包括控制器编号 (NumC, 1~8)、换刀阀编号 (NumG, 1~2)、开关状态 (Switch, 1开/0关) 和延时设定 (T, 切换失败则报警)。
*   **ToggleTool**：**切换工具**。参数包括控制器编号 (NumC, 1~8)、工具编号 (NumT, 1~8，气动力控模式下为1~3) 和延时设定 (T, 切换失败则报警)。
*   **ResetMotor**（仅限电动力控）：**电机回原点**。参数包括控制器编号 (NumC, 1~8) 和延时设定 (T, 回原点失败则报警)。
*   **PStart**：**开启力控报警监控**。无参数。开启后，力控制器报警将导致机器人暂停；关闭时，报警仅提示不暂停。
*   **PEnd**：**关闭力控报警监控**。无参数。用于在报警状态下通过关闭监控（将警告降级为提示）来允许指令继续执行或解除暂停。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 1470–1526; corpus 4.0, revision V5.0 / A-2; object `4cd1273fc2af98f36fddb3597c0101c24e8d23cf6279e073ba99f1a999aa5aed`.

- 该行为文档条目，包含手册名称《新松机器人电气使用及维护手册》及其对应编号 SX222001，源自《新松机器人打磨应用操作手册》V5.0 版本（SX322006）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 469–472; corpus 4.0, revision V5.0 / A-2; object `11fe2ed38668d73449508fcb8c5e75100b389c760b2548d839acf6fec4ef957f`.

- 文档《SX322006《新松机器人打磨应用操作手册》（A-2)》变更记录（第3行）：版本 **A/1**，日期 **2025年2月28日**，修订人 **罗晓芳**，修订内容 **增加水汽单元功能说明**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 40–45; corpus 5.0, revision V5.0 / A-2; object `478d9064a3da8f0b730c261221ecae39ea1a8874be552ec11e4e40ad96e9ba5f`.

#### Answer

The retrieved evidence does not contain any explicit statement about '每次开始作业之前进行位移复位操作' as a prerequisite or note for '力控执行器标定'. Evidence E6 documents the ResetMotor instruction (电机回原点, only valid under electric-type force control) with parameters NumC and T, but it is presented as a standalone instruction in a command table, not as a prerequisite tied to a calibration procedure. No section in the provided excerpts describes a '力控执行器标定' workflow with associated pre-conditions. Additionally, the question references revision V4.0/A-1, while all retrieved evidence is from V5.0/A-2, so the specific A-1 text is absent. The required prerequisite/note linking displacement reset to calibration is not present in any evidence item.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: mixed.

The answer correctly identifies that the supplied evidence does not contain the specific text regarding '每次开始作业之前进行位移复位操作' as a prerequisite for '力控执行器标定'. It accurately notes that the retrieved evidence is from revision V5.0/A-2, whereas the question asks for V4.0/A-1. Since the specific facts required to answer the question are not present in the provided evidence, the refusal to answer is appropriate and not a hallucination.

...

#### Answer

The retrieved manual evidence (E1–E8) from SX322006《新松机器人打磨应用操作手册》(A-2) V5.0/A-2 contains only document-reference tables, revision-history tables, role/permission assignment tables, and the manual title. None of the provided excerpts include the section describing '双头模型一' or the instruction '双头模型则需要设定2个坐标系'. There is no operational procedure, coordinate-system setup syntax, or dual-head model configuration content in the supplied evidence. Therefore a rich contextual example with a concrete code/instruction statement cannot be constructed from the available data.

### sample_grinding_ee21a9c44b_20

根据《SX322006《新松机器人打磨应用操作手册》（A-2)》（V5.0 / A-2），配置“功能”时，“在示教模式下”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。

#### Retrieved content summary

- 新松机器人打磨应用手册中的操作项“以示教器的数据设定”，其第2列内容为空，第3列和第4列均标记为“〇”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 282–287; corpus 5.0, revision V5.0 / A-2; object `9647ba66666726896e7c2752eca95795e738cd754fc91f3f9ffde4f049ee52fb`.

- 该行来自新松机器人打磨应用手册（SX322006, V5.0），描述功能项“以示教器选择程序”：第2列为空，第3、4列均标记为“〇”（表示未适用或无）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 246–251; corpus 4.0, revision V5.0 / A-2; object `f468f94517d146f3fec5af894c440a26b74332da678ee592f3fdfb8055f98dba`.

- 该行为《新松机器人打磨应用操作手册》中的一项功能条目，描述为“以示教器开始程序”，第二列为空，第三列与第四列均标记为“〇”（通常表示未启用、无对应状态或不适用），整体体现该特定操作模式在相应维度下的状态配置。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 264–269; corpus 4.0, revision V5.0 / A-2; object `4c5cd60c6d529afa751716d2170d027775182d71632b7b126e6b6c1f9c288470`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 文档《新松机器人打磨应用操作手册》（A-2）的修订记录行：版本 A/2，日期 2025 年 4 月 16 日，修改人 罗晓芳，内容 修改通讯配置与快捷键功能描述，并增加气动型力控制器说明。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 46–51; corpus 4.0, revision V5.0 / A-2; object `4465e396ac7d4a20aac4f15aebe2da2f3d2f06afb17ac4ed5e9f7ff348f05afc`.

- 文档《SX322006《新松机器人打磨应用操作手册》（A-2)》变更记录（第3行）：版本 **A/1**，日期 **2025年2月28日**，修订人 **罗晓芳**，修订内容 **增加水汽单元功能说明**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 40–45; corpus 5.0, revision V5.0 / A-2; object `478d9064a3da8f0b730c261221ecae39ea1a8874be552ec11e4e40ad96e9ba5f`.

- 该行来自《新松机器人打磨应用操作手册》，包含4个单元格（第1-4列），所有单元格内容均为空，无跨行/跨列合并，无继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 58–63; corpus 4.0, revision V5.0 / A-2; object `f41ea04de89bf6cbdecf7611336c89e474884531271506bdc0b2dabe91edb92c`.

- 该行为文档条目，包含手册名称《新松机器人电气使用及维护手册》及其对应编号 SX222001，源自《新松机器人打磨应用操作手册》V5.0 版本（SX322006）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 469–472; corpus 4.0, revision V5.0 / A-2; object `11fe2ed38668d73449508fcb8c5e75100b389c760b2548d839acf6fec4ef957f`.

#### Answer

根据检索到的手册证据（E1–E8），SX322006《新松机器人打磨应用操作手册》（A-2）V5.0/A-2 中仅包含：(1) 作业内容责任矩阵（操作者/程序员/维修人员对各操作的权限标记）；(2) 相关文档编号列表；(3) 修订历史。所有证据中均未出现

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: faithful.

The question asks for specific parameter meanings, values, or units for the 'Function' configuration in 'Teaching Mode' as listed in the manual. The supplied evidence (E1-E8) contains only a responsibility matrix (who can perform which operations), a list of related document numbers, and revision history. None of the supplied evidence snippets contain the specific text regarding force controller configuration, IO mapping, or calibration mentioned in the expected facts. The answer correctly identifies that the retrieved evidence does not contain the requested specific parameter details and refuses to hallucinate them. Therefore, it is an appropriate abstention due to inadequate supplied evidence.

## Spot Welding — sampled manual revisions

40 cases from 2 CSV files. Set `SPOT_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
SPOT_WELDING_SAMPLE_CASES = [{'id': 'sample_spot_welding_d91e7cd656_01', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.6', 'section_title': '3.6 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.6 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“通用协议焊接流程”中，应如何理解或操作“焊点编号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['6 焊点编号 输出IO 32'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1317, 'line_end': 1322, 'excerpt': '<tr>\n<td><p>6</p></td>\n<td><p>焊点编号</p></td>\n<td><p>输出IO</p></td>\n<td><p>32</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_02', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.6', 'section_title': '3.6 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.6 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“通用协议焊接流程”时，“焊接流程”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['焊接流程：（只通过焊机焊接，不通过焊机读取压力，板厚等参数）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1325, 'line_end': 1325, 'excerpt': '焊接流程：（只通过焊机焊接，不通过焊机读取压力，板厚等参数）', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_03', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '6', 'section_title': '6 伺服点焊实时显示', 'heading_path': ['6 伺服点焊实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“伺服点焊实时显示”中，应如何理解或操作“在远程模式下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 在远程模式下，也可以进入实时显示界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1563, 'line_end': 1563, 'excerpt': '1. 在远程模式下，也可以进入实时显示界面。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_04', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.5', 'section_title': '4.5 Init', 'heading_path': ['4 点焊指令集', '4.5 Init'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“Init”时，“<参数1>：焊点个数0-100个”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 <参数1>：焊点个数0-100个'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1452, 'line_end': 1455, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>&lt;参数1&gt;：焊点个数0&#45;100个</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_05', 'category': 'spot_welding', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '7', 'section_title': '7 伺服点焊报警信息', 'heading_path': ['7 伺服点焊报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），“伺服点焊报警信息”列出的报警7037表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7037'], 'expected_facts': ['7037 焊点多焊,个数为:'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1768, 'line_end': 1772, 'excerpt': '<tr>\n<td><p>7037</p></td>\n<td><p>焊点多焊,个数为: </p></td>\n<td></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_06', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.7', 'section_title': '4.7 ChangerDresser', 'heading_path': ['4 点焊指令集', '4.7 ChangerDresser'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“ChangerDresser”中，应如何理解或操作“A04_SW ChangerDresser Init ；注释 A04_SW ChangerDre”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['举例 A04_SW ChangerDresser Init', '注释 A04_SW ChangerDresser TipDress Rotation', '注释 A04_SW ChangerDresser CapChange FWD Duration=500'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1516, 'line_end': 1519, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td><p>A04&#95;SW ChangerDresser Init ；注释</p><p>A04&#95;SW ChangerDresser TipDress Rotation； 注释</p><p>A04&#95;SW ChangerDresser CapChange FWD Duration=500 ；注释</p><p>A04&#95;SW ChangerDresser CapCheck OnPos ；注释</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_07', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 IO配置', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.3 IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），操作“IO配置”时，涉及“空运行IO配置-输入信号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['4. 空运行IO配置-输入信号：只有焊接开关中焊接使能为远程模式时，该信号才有效，1：焊接过程中空运行，0：焊接过程中正常焊接'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 239, 'line_end': 239, 'excerpt': '4. 空运行IO配置-输入信号：只有焊接开关中焊接使能为远程模式时，该信号才有效，1：焊接过程中空运行，0：焊接过程中正常焊接。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_08', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '6', 'section_title': '6 伺服点焊实时显示', 'heading_path': ['6 伺服点焊实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“伺服点焊实时显示”时，“实时显示中”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 实时显示中，主要是显示当前跟点焊有关的参数项的状态', '如当前的模式状态，焊钳编号，修磨数据', '焊点相关的程序号，板厚参数，压力参数等'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1567, 'line_end': 1567, 'excerpt': '1. 实时显示中，主要是显示当前跟点焊有关的参数项的状态。如当前的模式状态，焊钳编号，修磨数据。焊点相关的程序号，板厚参数，压力参数等。便于在生产过程中对设备的使用情况有个清晰的了解。具体如图所示。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_09', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1', 'section_title': '1 点焊基本配置流程', 'heading_path': ['1 点焊基本配置流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“点焊基本配置流程”时，“根据上面介绍的流程配置点焊相关的参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['根据上面介绍的流程配置点焊相关的参数，涉及到的具体内容介绍，见后文', '请根据实际配置过程，查找对应的说明'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 164, 'line_end': 164, 'excerpt': '根据上面介绍的流程配置点焊相关的参数，涉及到的具体内容介绍，见后文。请根据实际配置过程，查找对应的说明。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_10', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.7', 'section_title': '4.7 ChangerDresser', 'heading_path': ['4 点焊指令集', '4.7 ChangerDresser'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），“ChangerDresser”中“一体式修磨换帽功能指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['一体式修磨换帽功能指令 功能 初始化一体式修磨换帽机'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1503, 'line_end': 1507, 'excerpt': '<tr>\n<td rowspan="4"><p>一体式修磨换帽功能指令</p></td>\n<td><p>功能</p></td>\n<td><p>初始化一体式修磨换帽机</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_11', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 点焊快捷键', 'heading_path': ['5 点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），操作“点焊快捷键”时，涉及“点击相应按钮后”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['点击相应按钮后，可以实现对应功能', '注意：快捷键必须先调出界面才好使', '同时OP2到OP6的功能，如图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1526, 'line_end': 1526, 'excerpt': '点击相应按钮后，可以实现对应功能。注意：快捷键必须先调出界面才好使。同时OP2到OP6的功能，如图所示。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_12', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.4', 'section_title': '2.4 焊钳特性配置', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.4 焊钳特性配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“焊钳特性配置”中，应如何理解或操作“最大压力(N)”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 最大压力(N): 加压时压力超过该值，则使用最大压力'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 290, 'line_end': 290, 'excerpt': '2. 最大压力(N): 加压时压力超过该值，则使用最大压力。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_13', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.22', 'section_title': '2.22 后台修磨操作说明', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.22 后台修磨操作说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），“后台修磨操作说明”中“DRESS\\_PNo指令为调用焊机的参数（只针对博士焊机）”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['DRESS'], 'expected_facts': ['DRESS\\_PNo指令为调用焊机的参数（只针对博士焊机）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 847, 'line_end': 847, 'excerpt': 'DRESS\\_PNo指令为调用焊机的参数（只针对博士焊机）。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_14', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 伺服点焊指令', 'heading_path': ['3 伺服点焊指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），“伺服点焊指令”中“SPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 PRS=”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SPOTL', 'ACC'], 'expected_facts': ['7. SPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 PRS=1 TWC=0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 868, 'line_end': 868, 'excerpt': '7. SPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 PRS=1 TWC=0；', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_15', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 点焊快捷键', 'heading_path': ['5 点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“点焊快捷键”中，应如何理解或操作“开电流”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['9. 开电流：打开电流开关'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1544, 'line_end': 1544, 'excerpt': '9. 开电流：打开电流开关。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_16', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.10', 'section_title': '2.10 焊钳诊断配置', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.10 焊钳诊断配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“焊钳诊断配置”时，“移动侧补偿值”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['5. 移动侧补偿值：工件表面据上电极距离的补偿值（mm），夹板修磨时更新该参数，即使修磨后人为修改该参数值后，下一次修磨后该参数值仍然和当前固定侧磨损量相等'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 462, 'line_end': 462, 'excerpt': '5. 移动侧补偿值：工件表面据上电极距离的补偿值（mm），夹板修磨时更新该参数，即使修磨后人为修改该参数值后，下一次修磨后该参数值仍然和当前固定侧磨损量相等。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_17', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.4', 'section_title': '3.4 多次加压', 'heading_path': ['3 伺服点焊指令', '3.4 多次加压'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），操作“多次加压”时，涉及“开始执行接触压力”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['开始执行接触压力，之后执行一至四阶段压力，每次可设置加压时间，结束条件设为等待结束时是最后一次加压', '如表所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1156, 'line_end': 1156, 'excerpt': '开始执行接触压力，之后执行一至四阶段压力，每次可设置加压时间，结束条件设为等待结束时是最后一次加压。如表所示：', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_18', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '7', 'section_title': '7 伺服点焊报警信息', 'heading_path': ['7 伺服点焊报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“伺服点焊报警信息”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 解决办法'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1578, 'line_end': 1582, 'excerpt': '<tr>\n<td><p>报警码</p></td>\n<td><p>报警信息</p></td>\n<td><p>解决办法</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_19', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.6', 'section_title': '3.6 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.6 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“通用协议焊接流程”中，应如何理解或操作“焊接/试验”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1 焊接/试验 输出IO 1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1287, 'line_end': 1292, 'excerpt': '<tr>\n<td><p>1</p></td>\n<td><p>焊接/试验</p></td>\n<td><p>输出IO</p></td>\n<td><p>1</p></td>\n</tr>', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_d91e7cd656_20', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '6', 'section_title': '6 伺服点焊实时显示', 'heading_path': ['6 伺服点焊实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“伺服点焊实时显示”中，应如何理解或操作“点击实时显示后进入界面”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击实时显示后进入界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'line_start': 1557, 'line_end': 1557, 'excerpt': '点击实时显示后进入界面。', 'sha256': 'dbde9adb99f164b1021dda681499195a43cb0e0ab4e16edaafff09f46c482d9d'}]}, {'id': 'sample_spot_welding_f6c81e4db2_01', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.5', 'section_title': '3.5 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.5 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），配置“通用协议焊接流程”时，“焊接流程”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['焊接流程：（只通过焊机焊接，不通过焊机读取压力，板厚等参数）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1046, 'line_end': 1046, 'excerpt': '焊接流程：（只通过焊机焊接，不通过焊机读取压力，板厚等参数）', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_02', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4.3', 'section_title': '4.3 ProcessCheck', 'heading_path': ['4 点焊指令集', '4.3 ProcessCheck'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“ProcessCheck”中“检查指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A04_SW'], 'expected_facts': ['检查指令 功能 工艺循环结束检查 格式 A04_SW ProcessCheck', '注释 说明 无参数', '焊点数一致时，下面提示'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1083, 'line_end': 1083, 'excerpt': '<table><tr><td rowspan="4"><p>检查指令</p></td><td><p>功能</p></td><td><p>工艺循环结束检查 </p></td></tr><tr><td><p>格式</p></td><td><p>A04_SW ProcessCheck；注释</p></td></tr><tr><td><p>说明</p></td><td><p>无参数。焊点数一致时，下面提示。焊点数不一致时，报警缺少焊点个数；</p></td></tr><tr><td><p>举例</p></td><td><p>A04_SW ProcessCheck；注释</p></td></tr></table>', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_03', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.5', 'section_title': '3.5 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.5 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“通用协议焊接流程”中，应如何理解或操作“检查焊接后是否有报警”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 检查焊接后是否有报警：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1063, 'line_end': 1063, 'excerpt': '1. 检查焊接后是否有报警：', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_04', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '6', 'section_title': '6 伺服点焊实时显示', 'heading_path': ['6 伺服点焊实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），配置“伺服点焊实时显示”时，“实时显示中”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 实时显示中，主要是显示当前跟点焊有关的参数项的状态', '如当前的模式状态，焊钳编号，修磨数据', '焊点相关的程序号，板厚参数，压力参数等'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1144, 'line_end': 1144, 'excerpt': '1. 实时显示中，主要是显示当前跟点焊有关的参数项的状态。如当前的模式状态，焊钳编号，修磨数据。焊点相关的程序号，板厚参数，压力参数等。便于在生产过程中对设备的使用情况有个清晰的了解。具体如图所示。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_05', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '5', 'section_title': '5 点焊快捷键', 'heading_path': ['5 点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），操作“点焊快捷键”时，涉及“点击相应按钮后”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['点击相应按钮后，可以实现对应功能', '注意：快捷键必须先调出界面才好使', '同时同步OP2到OP6的功能，如图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1105, 'line_end': 1105, 'excerpt': '点击相应按钮后，可以实现对应功能。注意：快捷键必须先调出界面才好使。同时同步OP2到OP6的功能，如图所示。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_06', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.20', 'section_title': '2.20 焊钳压力标定', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.20 焊钳压力标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“焊钳压力标定”中，应如何理解或操作“在扭矩压力标定完成后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['在扭矩压力标定完成后，需要继续标定码盘值和加压目标修正值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 733, 'line_end': 733, 'excerpt': '在扭矩压力标定完成后，需要继续标定码盘值和加压目标修正值。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_07', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.11', 'section_title': '2.11 间隙参数配置', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.11 间隙参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“间隙参数配置”中“默认动作方式为矩形运动”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SPOTL', 'SPOTJ'], 'expected_facts': ['默认动作方式为矩形运动，当动作方式为矩形时，SPOTL/SPOTJ指令运动前后会张开到设置的距上下电极打点距离', '打点前距上电极距离：间隙运动中加压前工件表面据上电极的距离'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 477, 'line_end': 477, 'excerpt': '默认动作方式为矩形运动，当动作方式为矩形时，SPOTL/SPOTJ指令运动前后会张开到设置的距上下电极打点距离；打点前距上电极距离：间隙运动中加压前工件表面据上电极的距离', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_08', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.1', 'section_title': '3.1 焊接指令流程', 'heading_path': ['3 伺服点焊指令', '3.1 焊接指令流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“焊接指令流程”中“移动焊钳到下一个焊点”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['4. 移动焊钳到下一个焊点'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 962, 'line_end': 962, 'excerpt': '4. 移动焊钳到下一个焊点。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_09', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.13', 'section_title': '2.13 手动加压配置', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.13 手动加压配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），配置“手动加压配置”时，“压力测定模式开关默认为OFF”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 压力测定模式开关默认为OFF，置成ON后会显示压力计厚度参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 510, 'line_end': 510, 'excerpt': '1. 压力测定模式开关默认为OFF，置成ON后会显示压力计厚度参数。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_10', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1', 'section_title': '1 点焊基本配置流程', 'heading_path': ['1 点焊基本配置流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），操作“点焊基本配置流程”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：焊钳轴配置必须要先配置正确，后再配置其他参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 107, 'line_end': 107, 'excerpt': '**注意：焊钳轴配置必须要先配置正确，后再配置其他参数。**', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_11', 'category': 'spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.23', 'section_title': '2.23 后台修磨操作说明', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.23 后台修磨操作说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），操作“后台修磨操作说明”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：TWC参数支持比例修磨03和打板修磨012'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 866, 'line_end': 866, 'excerpt': '**注意：TWC参数支持比例修磨03和打板修磨012。**', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_12', 'category': 'spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1', 'section_title': '1 点焊基本配置流程', 'heading_path': ['1 点焊基本配置流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），配置“点焊基本配置流程”时，“根据上面介绍的流程配置点焊相关的参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['根据上面介绍的流程配置点焊相关的参数，涉及到的具体内容介绍，见后文', '请根据实际配置过程，查找对应的说明'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 105, 'line_end': 105, 'excerpt': '根据上面介绍的流程配置点焊相关的参数，涉及到的具体内容介绍，见后文。请根据实际配置过程，查找对应的说明。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_13', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '5', 'section_title': '5 点焊快捷键', 'heading_path': ['5 点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“点焊快捷键”中，应如何理解或操作“焊接使能关”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['10. 焊接使能关：关闭电流开关', '关闭后，在“工艺开”模式下也不焊接，且不同焊机进行焊接交互信号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1122, 'line_end': 1122, 'excerpt': '10. 焊接使能关：关闭电流开关。关闭后，在“工艺开”模式下也不焊接，且不同焊机进行焊接交互信号。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_14', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '6', 'section_title': '6 伺服点焊实时显示', 'heading_path': ['6 伺服点焊实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“伺服点焊实时显示”中，应如何理解或操作“在执行模式下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 在执行模式下，也可以进入实时显示界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1136, 'line_end': 1136, 'excerpt': '1. 在执行模式下，也可以进入实时显示界面。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_15', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '7', 'section_title': '7 伺服点焊报警信息', 'heading_path': ['7 伺服点焊报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“伺服点焊报警信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OFF', 'PLC'], 'expected_facts': ['报警码 报警信息 子码 子码说明 处理方法 7000 焊接异常 0~4 点焊超时 请检查焊机特性中焊接结束检测时间是否有误 5~19 伺服点焊焊机报警 检查焊机状态，消除焊机控制器报警 20~26 焊接故障 请联系技术人员 27~30 焊机未配置 检查焊机设置中是否配置焊机 31 焊接故障 请联系技术人员 32 点焊超时或焊接过程中出现暂停 请检查焊接过程中是否有暂停 33 获取参数失败 请检查焊机 34 电阻检测异常 请检查焊机 7001 点焊指令干涉区暂停 0~17 请检查干涉区配置或当前干涉区状态 7002 焊钳轴运动错误或当前压力错误 请检查焊钳加压是否正确以及扭矩输出不为0 7003 点焊焊接异常 0~5 请检查焊机IO信号 6 输入IO，异常，警告，或者异常代码IO在焊接时状态异常', '请检查焊机端状态，确保焊机无报警 7 给焊机发送焊点编号IO错误', '请检查对应IO配置 8 给焊机发送焊点编号IO错误'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1154, 'line_end': 1154, 'excerpt': '<table><thead><tr><th><p><strong>报警码</strong></p></th><th><p><strong>报警信息</strong></p></th><th><p><strong>子码</strong></p></th><th><p><strong>子码说明</strong></p></th><th><p><strong>处理方法</strong></p></th></tr></thead><tbody><tr><td rowspan="8"><p>7000</p></td><td rowspan="8"><p>焊接异常</p></td><td><p>0~4</p></td><td><p>点焊超时</p></td><td><p>请检查焊机特性中焊接结束检测时间是否有误</p></td></tr><tr><td><p>5~19</p></td><td><p>伺服点焊焊机报警</p></td><td><p>检查焊机状态，消除焊机控制器报警</p></td></tr><tr><td><p>20~26</p></td><td><p>焊接故障</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>27~30</p></td><td><p>焊机未配置</p></td><td><p>检查焊机设置中是否配置焊机</p></td></tr><tr><td><p>31</p></td><td><p>焊接故障</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>32</p></td><td><p>点焊超时或焊接过程中出现暂停</p></td><td><p>请检查焊接过程中是否有暂停</p></td></tr><tr><td><p>33</p></td><td><p>获取参数失败</p></td><td><p>请检查焊机</p></td></tr><tr><td><p>34</p></td><td><p>电阻检测异常</p></td><td><p>请检查焊机</p></td></tr><tr><td><p>7001</p></td><td><p>点焊指令干涉区暂停</p></td><td><p>0~17</p></td><td><p>\u3000</p></td><td><p>请检查干涉区配置或当前干涉区状态</p></td></tr><tr><td><p>7002</p></td><td><p>焊钳轴运动错误或当前压力错误</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查焊钳加压是否正确以及扭矩输出不为0</p></td></tr><tr><td rowspan="9"><p>7003</p></td><td rowspan="9"><p>点焊焊接异常</p></td><td><p>0~5</p></td><td><p>\u3000</p></td><td><p>请检查焊机IO信号</p></td></tr><tr><td><p>6</p></td><td><p>输入IO，异常，警告，或者异常代码IO在焊接时状态异常。</p></td><td><p>请检查焊机端状态，确保焊机无报警</p></td></tr><tr><td><p>7</p></td><td><p>给焊机发送焊点编号IO错误。</p></td><td><p>请检查对应IO配置</p></td></tr><tr><td><p>8</p></td><td><p>给焊机发送焊点编号IO错误。</p></td><td><p>请检查对应IO配置</p></td></tr><tr><td><p>9</p></td><td><p>输入IO，准备完成IO在焊接时为OFF。</p></td><td><p>请检查焊机端状态，排查焊机问题</p></td></tr><tr><td><p>10</p></td><td><p>焊接完成后收到焊机异常，警告，或者异常代码IO为ON。</p></td><td><p>请检查焊机端状态，检查焊机报警原因</p></td></tr><tr><td><p>11</p></td><td><p>\u3000</p></td><td><p>请检查焊机IO信号</p></td></tr><tr><td><p>12</p></td><td><p>\u3000</p></td><td><p>请检查焊机IO信号</p></td></tr><tr><td><p>13</p></td><td><p>异常或者报警IO，同时配置了通常和取反信号。</p></td><td><p>请检查输入IO配置</p></td></tr><tr><td rowspan="3"><p>7004</p></td><td rowspan="3"><p>焊机未就绪</p></td><td><p>0~13</p></td><td><p>\u3000</p></td><td><p>请检查焊机端状态，确保焊机各状态输出信号准备就绪</p></td></tr><tr><td><p>14</p></td><td><p>\u3000</p></td><td><p>请检查焊机端压力配置</p></td></tr><tr><td><p>15</p></td><td><p>温控报警</p></td><td><p>请检查焊机温控</p></td></tr><tr><td><p>7005</p></td><td><p>指针为空</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7006</p></td><td><p>通讯状态异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>7007</p></td><td><p>数据异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td rowspan="6"><p>7008</p></td><td rowspan="6"><p>焊钳快换取/放指令异常</p></td><td><p>0~5</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>6</p></td><td><p>不支持当前校零方式</p></td><td><p>请选择正确的校零方式</p></td></tr><tr><td><p>7~14</p></td><td><p>清除驱动器报警出现错误</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>15</p></td><td><p>放枪位置错误</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>16</p></td><td><p>清除驱动器圈数出现错误</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>17</p></td><td><p>放枪位置错误</p></td><td><p>\u3000</p></td></tr><tr><td><p>7009</p></td><td><p>焊钳自动校零异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td rowspan="19"><p>7010</p></td><td rowspan="19"><p>焊钳轴运动错误</p></td><td><p>0</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>1</p></td><td><p>快捷键执行中错误</p></td><td><p>请检查参数设置中焊接压力配置是否有误</p></td></tr><tr><td><p>2</p></td><td><p>加压中，规划到位但是扭矩未到限制值</p></td><td><p>请检查参数设置中焊接压力配置是否有误</p></td></tr><tr><td><p>3</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>4</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>5</p></td><td><p>\u3000</p></td><td><p>请检查参数设置中焊接压力配置是否有误</p></td></tr><tr><td><p>6</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7</p></td><td><p>快捷键执行中错误，如未到位时就松开使能</p></td><td><p>检查上电使能状态</p></td></tr><tr><td><p>8</p></td><td><p>\u3000</p></td><td><p>检查轴数配置是否有误</p></td></tr><tr><td><p>9</p></td><td><p>快捷键执行中错误，规划到位但是扭矩未到限制值</p></td><td><p>请检查参数设置中焊接压力</p></td></tr><tr><td><p>10</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>11</p></td><td><p>手动点焊报警，加压过程压力未到设定值</p></td><td><p>请检查参数设置中焊接压力</p></td></tr><tr><td><p>12</p></td><td><p>点焊指令张开状态和内部标志不匹配</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>13</p></td><td><p>设定压力过小，焊钳未移动</p></td><td><p>增大焊接压力设定</p></td></tr><tr><td><p>14</p></td><td><p>设定压力过小，焊钳未移动</p></td><td><p>增大焊接压力设定</p></td></tr><tr><td><p>15</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>16</p></td><td><p>设定压力过小，焊钳未移动</p></td><td><p>增大焊接压力设定</p></td></tr><tr><td><p>17</p></td><td><p>执行点焊闭合中，提前压到工件，无法走到闭合目标点</p></td><td><p>检查板厚设定</p></td></tr><tr><td><p>18</p></td><td><p>执行点焊闭合中，提前压到工件，无法走到闭合目标点</p></td><td><p>检查板厚设定</p></td></tr><tr><td><p>7011</p></td><td><p>请选择正确的校零类型</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>检查校零类型</p></td></tr><tr><td><p>7012</p></td><td><p>文件操作失败</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7013</p></td><td><p>修磨量计算异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>单次修磨量超过界面范围，请检查修磨中是否存在异常</p></td></tr><tr><td rowspan="8"><p>7014</p></td><td rowspan="8"><p>后台修磨相关错误</p></td><td><p>0</p></td><td><p>未完成修磨动作时，又出现同一类型修磨指令</p></td><td><p>当前修磨流程无法继续执行，修改作业后，手动修磨</p></td></tr><tr><td><p>1</p></td><td><p>指令错误</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>2</p></td><td><p>未配置修磨完成信号IO号</p></td><td><p>配置IO信号后，重新执行该指令以便开始后台修磨</p></td></tr><tr><td><p>3</p></td><td><p>压力文件号错误</p></td><td><p>修改正确的压力文件号</p></td></tr><tr><td><p>4</p></td><td><p>修磨类型参数错误，只支持0和3</p></td><td><p>修改正确的修磨类型编号</p></td></tr><tr><td><p>5</p></td><td><p>未配置修磨完成信号IO号</p></td><td><p>配置IO信号并且清除后台修磨数据</p></td></tr><tr><td><p>6</p></td><td><p>执行修磨0时，压力错误。</p></td><td><p>检查压力参数等是否正确</p></td></tr><tr><td><p>7</p></td><td><p>执行修磨时，修磨复位IO为ON。</p></td><td><p>检查IO信号状态是否正确</p></td></tr><tr><td><p>7015</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7016</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7017</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7018</p></td><td><p>断电数据大小超界</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7019</p></td><td><p>断电数据校验失败</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7020</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7021</p></td><td><p>点焊指令集错误</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查指令集参数是否正确</p></td></tr><tr><td><p>7022</p></td><td><p>修磨器异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查修磨器状态</p></td></tr><tr><td><p>7023</p></td><td><p>水流量异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查实际流量与设定流量值</p></td></tr><tr><td><p>7024</p></td><td><p>压缩空气异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查气压表状态</p></td></tr><tr><td><p>7025</p></td><td><p>当前作业轴组信息不正确</p></td><td><p>0~5</p></td><td><p>\u3000</p></td><td><p>请检查焊钳轴配置和作业轴数配置</p></td></tr><tr><td><p>7026</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td rowspan="4"><p>7027</p></td><td rowspan="4"><p>点焊指令巡检暂停</p></td><td><p>0</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>1</p></td><td><p>巡检IO异常</p></td><td><p>请检查与PLC通讯状态及巡检IO交互是否异常</p></td></tr><tr><td><p>2</p></td><td><p>巡检IO异常</p></td><td><p>请检查与PLC通讯状态及巡检IO交互是否异常</p></td></tr><tr><td><p>3</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7028</p></td><td><p>小原焊机报警</p></td><td><p>0~129</p></td><td><p>\u3000</p></td><td><p>小原焊机反馈错误，请咨询焊机厂家</p></td></tr><tr><td><p>7029</p></td><td><p>点焊参数</p></td><td><p>0~11</p></td><td><p>\u3000</p></td><td><p>请检查参数是否错误</p></td></tr><tr><td><p>7030</p></td><td><p>修磨数据异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请重新修磨</p></td></tr><tr><td><p>7031</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7032</p></td><td><p>指令焊钳号与当前结合焊钳号不一致</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查快捷键界面中焊钳号输入是否有误</p></td></tr><tr><td><p>7033</p></td><td><p>点焊响应安全门暂停</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7034</p></td><td><p>焊接开始条件号错误，请检查</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查指令入参</p></td></tr><tr><td><p>7035</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7036</p></td><td><p>焊点少焊,个数为: </p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查生产过程中是否焊点少焊</p></td></tr><tr><td><p>7037</p></td><td><p>焊点多焊,个数为:</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查生产过程中是否焊点多焊</p></td></tr><tr><td rowspan="3"><p>7038</p></td><td rowspan="3"><p>板厚检测时：</p></td><td><p>0</p></td><td><p>修磨补偿未开启</p></td><td><p>请检查修磨补偿开关是否是关闭状态</p></td></tr><tr><td><p>1</p></td><td><p>板厚超下限值</p></td><td><p>请检查板厚设定值与工件实际板厚</p></td></tr><tr><td><p>2</p></td><td><p>板厚超上限值</p></td><td><p>请检查板厚设定值与工件实际板厚</p></td></tr><tr><td rowspan="2"><p>7039</p></td><td rowspan="2"><p>逻辑IO分配错误:</p></td><td><p>0</p></td><td><p>逻辑IO未分配点焊应用</p></td><td><p>请在界面IO-&gt;逻辑IO配置中配置输入输出IO</p></td></tr><tr><td><p>1</p></td><td><p>逻辑IO分配与点焊IO配置号不匹配</p></td><td><p>请检确认逻辑IO分配范围包含点焊IO</p></td></tr><tr><td><p>7040</p></td><td><p>鸿柏焊机不支持清报警功能</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7041</p></td><td><p>焊机类型非鸿柏焊机</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7042</p></td><td><p>伺服焊钳轴未使能</p></td><td><p>0~9</p></td><td><p>\u3000</p></td><td><p>请检查点焊开关配置中焊钳轴是否使能</p></td></tr><tr><td><p>7043</p></td><td><p>焊机焊接未使能</p></td><td><p>0~9</p></td><td><p>\u3000</p></td><td><p>请检查点焊开关配置中焊接使能是否打开</p></td></tr><tr><td><p>7044</p></td><td><p>屏蔽轴清报警超时</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7045</p></td><td><p>小原焊机重焊提示</p></td><td><p>0~129</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7046</p></td><td><p>当前规划库不是2.0</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请确认用户配置中规划库版本是否是2.0</p></td></tr><tr><td><p>7047</p></td><td><p>未找到上一条运动指令信息</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7048</p></td><td><p>伺服点焊清空堆栈</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7049</p></td><td><p>请求更换电极</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请更换电极</p></td></tr><tr><td><p>7050</p></td><td><p>点焊指令执行异常，请检查</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请检查修磨指令参数</p></td></tr><tr><td><p>7051</p></td><td><p>修磨指令校准码盘基准值</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7052</p></td><td><p>点焊应用dnet开关已关闭</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7053</p></td><td><p>虚拟控制器打开，点焊调试过程</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7054</p></td><td><p>数据采集异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7055</p></td><td><p>快捷键：</p></td><td><p>\u3000</p></td><td><p>快捷键对应提示</p></td><td><p>\u3000</p></td></tr><tr><td><p>7056</p></td><td><p>快捷键：</p></td><td><p>\u3000</p></td><td><p>快捷键对应提示</p></td><td><p>\u3000</p></td></tr><tr><td><p>7057</p></td><td><p>焊点个数检测无效</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td rowspan="4"><p>7058</p></td><td rowspan="4"><p>板厚检测时：</p></td><td><p>0</p></td><td><p>挠曲补偿未开启</p></td><td><p>请检查修磨补偿开关是否是关闭状态</p></td></tr><tr><td><p>1</p></td><td><p>板厚超下限值</p></td><td><p>请检查板厚设定值与工件实际板厚</p></td></tr><tr><td><p>2</p></td><td><p>板厚超上限值</p></td><td><p>请检查板厚设定值与工件实际板厚</p></td></tr><tr><td><p>3</p></td><td><p>码盘值转关节值出错</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7059</p></td><td><p>换帽预警</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>电极帽使用寿命接近设定值，请准备新电极帽</p></td></tr><tr><td><p>7060</p></td><td><p>上电情况下执行此操作</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7061</p></td><td><p>文件打开异常</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>7062</p></td><td><p>焊点个数检查</p></td><td><p>\u3000</p></td><td><p>漏焊提示</p></td><td><p>\u3000</p></td></tr><tr><td><p>7063</p></td><td><p>换帽中不检测单次修磨量</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7064</p></td><td><p>故障提示</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7065</p></td><td><p>巡检配置为0功能未启用</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7066</p></td><td><p>应用配置已更改，已初始化</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7067</p></td><td><p>通讯为屏蔽状态</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>提示用户,无需处理</p></td></tr><tr><td><p>7068</p></td><td><p>后台修磨</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr><tr><td><p>7069</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td><td><p>\u3000</p></td></tr></tbody></table>', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_16', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4.5', 'section_title': '4.5 Init', 'heading_path': ['4 点焊指令集', '4.5 Init'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“Init”中“初始化指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A04_SW'], 'expected_facts': ['初始化指令 功能 初始化点焊工艺状态', '格式 A04_SW SWInit Count=1(参数1)', '注释 说明 <参数1>：焊点个数0-100个 举例 A04_SW SWInit Count=10'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1091, 'line_end': 1091, 'excerpt': '<table><tr><td rowspan="4"><p>初始化指令</p></td><td><p>功能</p></td><td><p>初始化点焊工艺状态。</p></td></tr><tr><td><p>格式</p></td><td><p>A04_SW SWInit Count=1(参数1)； 注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：焊点个数0-100个</p></td></tr><tr><td><p>举例</p></td><td><p>A04_SW SWInit Count=10；注释</p></td></tr></table>', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_17', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.5', 'section_title': '3.5 通用协议焊接流程', 'heading_path': ['3 伺服点焊指令', '3.5 通用协议焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“通用协议焊接流程”中，应如何理解或操作“在上面焊接异常的情况下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['在上面焊接异常的情况下，或者正常焊接结束后', '会将输出IO，焊接开始，焊点号确认，以及焊点标号IO都输出OFF'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 1069, 'line_end': 1069, 'excerpt': '在上面焊接异常的情况下，或者正常焊接结束后。会将输出IO，焊接开始，焊点号确认，以及焊点标号IO都输出OFF。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_18', 'category': 'spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.17', 'section_title': '2.17 板厚检测', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.17 板厚检测'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），在“板厚检测”中，应如何理解或操作“TCHK\\_TOL”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['TCHK\\_TOL:设置上下限值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 667, 'line_end': 667, 'excerpt': 'TCHK\\_TOL:设置上下限值。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_19', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.6.2', 'section_title': '2.6.2 多次加压', 'heading_path': ['2 伺服点焊机器人应用配置项', '2.6 焊接压力配置', '2.6.2 多次加压'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“多次加压”中“作业中的点焊指令不需要变化”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 作业中的点焊指令不需要变化，根据指令中的压力文件号，找到对应的压力文件', '如果设定的是压力开关为ON，则按照多次加压来实现', '其中，从焊机调用压力的指令不支持多次加压'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 317, 'line_end': 317, 'excerpt': '1. 作业中的点焊指令不需要变化，根据指令中的压力文件号，找到对应的压力文件。如果设定的是压力开关为ON，则按照多次加压来实现。其中，从焊机调用压力的指令不支持多次加压。', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}, {'id': 'sample_spot_welding_f6c81e4db2_20', 'category': 'spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3', 'section_title': '3 伺服点焊指令', 'heading_path': ['3 伺服点焊指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“伺服点焊指令”中“GUNPICK GUN #1 TF=1”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GUNPICK', 'GUN'], 'expected_facts': ['GUNPICK GUN #1 TF=1 功能 焊钳取枪指令 说明 参数项 1 焊钳编号 参数项 2 工具坐标系号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'line_start': 937, 'line_end': 937, 'excerpt': '<table><tr><td colspan="3"><p>GUNPICK GUN #1 TF=1</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>焊钳取枪指令</p></td></tr><tr><td rowspan="2"><p>说明</p></td><td><p>参数项 1</p></td><td><p>焊钳编号</p></td></tr><tr><td><p>参数项 2</p></td><td><p>工具坐标系号</p></td></tr></table>', 'sha256': '4473d7414c9116e2b835deb4ac2e9e939c93aa7f3eac8512644a1e5f83dbb4b1'}]}]
register_sample_bank('spot_welding', SPOT_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md', 'seed': 5984111028755981447, 'population': 404, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1317, 1322), (1325, 1325), (1563, 1563), (1452, 1455), (1768, 1772), (1516, 1519), (239, 239), (1567, 1567), (164, 164), (1503, 1507), (1526, 1526), (290, 290), (847, 847), (868, 868), (1544, 1544), (462, 462), (1156, 1156), (1578, 1582), (1287, 1292), (1557, 1557)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md', 'seed': 12265761256332521805, 'population': 387, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1046, 1046), (1083, 1083), (1063, 1063), (1144, 1144), (1105, 1105), (733, 733), (477, 477), (962, 962), (510, 510), (107, 107), (866, 866), (105, 105), (1122, 1122), (1136, 1136), (1154, 1154), (1091, 1091), (1069, 1069), (667, 667), (317, 317), (937, 937)]}])


In [ ]:
# | notest
SPOT_WELDING_SAMPLE_IDS = None
spot_welding_sample_cases = select_cases(SPOT_WELDING_SAMPLE_CASES, SPOT_WELDING_SAMPLE_IDS)
display(pd.DataFrame(spot_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in spot_welding_sample_cases])
spot_welding_sample_metrics = await evaluation.run(spot_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
spot_welding_sample_results = {case["id"]: evaluation.load(case) for case in spot_welding_sample_cases}
display(spot_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(spot_welding_sample_cases[0]["id"])
evaluation.show_answers(spot_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_spot_welding_d91e7cd656_01,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,23431,0,True,unsupported,,,not reviewed
sample_spot_welding_d91e7cd656_02,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,7107,0,True,unsupported,,,abstained
sample_spot_welding_d91e7cd656_03,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,7117,0,True,unsupported,,,abstained
sample_spot_welding_d91e7cd656_04,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,7519,0,True,included,,,not reviewed
sample_spot_welding_d91e7cd656_05,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,alarm,table_row,False,True,8,False,0,0,0,False,False,False,False,False,22955,2,True,unsupported,,,incorrect
sample_spot_welding_d91e7cd656_06,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,3,False,False,False,None,None,9958,0,True,included,,,not reviewed
sample_spot_welding_d91e7cd656_07,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,7288,0,True,unsupported,,,not reviewed
sample_spot_welding_d91e7cd656_08,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,7107,0,True,unsupported,,,not reviewed
sample_spot_welding_d91e7cd656_09,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,7107,0,True,unsupported,,,not reviewed
sample_spot_welding_d91e7cd656_10,spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,7541,0,True,included,,,not reviewed


### sample_spot_welding_d91e7cd656_01

根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），在“通用协议焊接流程”中，应如何理解或操作“焊点编号”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该表格行来自《新松机器人点焊应用操作手册》(SX322007)，描述了一条指令配置：步骤序号 **2** 对应事件 **焊接开始**，触发动作为 **输出IO**，参数值为 **1**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1380–1385; corpus 4.0, revision V5.0 / A-2; object `0b7f2934df81fae53b8121def6d49422f1846ac73e21e3c24f0f9b9b596ff997`.

- 新松机器人点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 11–13; corpus 5.0, revision V5.0 / A-2; object `84fbddc59ed577db3eb4eb99bccfcaa80459690ab956da1d77fffa769d9853bd`.

- 该行位于《新松机器人点焊应用操作手册》的表格中（非表头行）。第一列为跨7行的合并单元格，内容为“说明”；第二列为“参数项 1”；第三列为“位置变量号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1113–1117; corpus 4.0, revision V5.0 / A-2; object `b5c444d2c0bafd3856f6b07414124f83e3f22fc7c2f7e000948a46607ba42bbf`.

- 该行位于《新松机器人点焊应用操作手册》中，属于“说明”分组（该单元格纵向合并，覆盖后续8行）。该行具体内容为：参数项为“参数项 1”，对应的位置变量号为“位置变量号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1068–1072; corpus 4.0, revision V5.0 / A-2; object `94213c70d21ef33a938a8e11c805dd9dcff89aafe92d0b2dc6cc8ecf0a984441`.

- 该表格行（索引 2，来自《新松机器人点焊应用操作手册》）记录了第 1 项配置：功能类型为“焊接/试验”，对应的信号为“输出 IO”，其参数值设为“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1374–1379; corpus 5.0, revision V5.0 / A-2; object `dc01c72c705d306b3f413f83037cc36cead70c548d40d43507d037f2deb1334e`.

- **摘要**：该表行（第3行，非表头）位于《新松机器人点焊应用操作手册》中，定义了一个参数配置项：在“说明”类别（跨2行）下，“参数项 1”对应的值为“焊钳编号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1195–1199; corpus 5.0, revision V5.0 / A-2; object `d1f8bebd1e937a0ec54f7567fe66906a8b493e8d10dae185db7c00e9050232f7`.

- 该表格行（索引2，源自《新松机器人点焊应用操作手册》）包含四项数据：序号1、状态“焊接完成”、信号类型“输入IO”以及数值“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1329–1334; corpus 5.0, revision V5.0 / A-2; object `cbd8148ae8724c55290bd10c9f598cc6306871aa955689467fe5e4e33fc7e616`.

- 报警码7020（新松机器人点焊应用），其余字段（报警信息、子码、子码说明、处理方法）为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 2048–2054; corpus 5.0, revision V5.0 / A-2; object `ee7974777b60cf935e7b523241ea75d4ff63a94fd43143304fa2d26da080abbc`.

#### Answer

根据手册证据，'焊点编号'在通用协议焊接流程中是一个**输出IO信号组**，共32位（E1）。其用途是机器人向焊机发送当前正在执行的焊点编号，使焊机能够识别焊点并调用对应的焊接参数（如压力、电流等）。在焊接流程中，机器人通过这32位输出IO以二进制编码方式将焊点号传递给焊机。若发送过程中IO配置错误，将触发报警码7003子码7或8（'给焊机发送焊点编号IO错误'），处理方法为'请检查对应IO配置'（E8）。此外，输出IO表中还有'焊点号确认'（1位输出IO），用于焊机确认已收到焊点编号（E1）。整体而言，焊点编号是机器人-焊机通讯协议中的关键标识信号，确保焊机对每个焊点执行正确的焊接工艺参数。

### sample_spot_welding_d91e7cd656_02

根据《SX322007《新松机器人点焊应用操作手册》-A-0》（V4.0 / A-0），配置“通用协议焊接流程”时，“焊接流程”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。

...

#### Answer

The question asks about a specific feature called '多次加压' (multiple pressurization) and the statement '作业中的点焊指令不需要变化' (the spot welding instruction during operation does not need to change). After reviewing all retrieved evidence (E1–E8), none of the records contain any mention of '多次加压' as a named feature, nor the phrase '作业中的点焊指令不需要变化'. The evidence only documents: (1) SPOTJ … WELD … (welding with gap motion, E2), (2) SPOTJ … DRESS … (electrode dress/pressurization with wear calculation, E4/E7), (3) IO signal tables (E3/E8), and (4) the manual title page (E1). No syntax, parameter, or behavioral description for a 'multiple pressurization' mode or the stated invariance of the spot-weld instruction is present in the supplied excerpts. Therefore the evidence cannot support a contextual example for this specific claim.

### sample_spot_welding_f6c81e4db2_20

根据《SX322007《新松机器人点焊应用操作手册》（A-2）》（V5.0 / A-2），“伺服点焊指令”中“GUNPICK GUN #1 TF=1”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 新松机器人点焊指令 `SPOTJ P[1]`，配置参数：VJ=10, ACC=100, CNT=0, WELD, GUN=1, PROGNo=123456789, DISUP=5, DISDOWN=5
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1061–1063; corpus 4.0, revision V5.0 / A-2; object `29377e3877f2cba35e6908c98f93b1bce5ebdfbd75dd23b44381a5da0f0d758a`.

- 该行位于《新松机器人点焊应用操作手册》中，属于“说明”分组（该单元格纵向合并，覆盖后续8行）。该行具体内容为：参数项为“参数项 1”，对应的位置变量号为“位置变量号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1068–1072; corpus 4.0, revision V5.0 / A-2; object `94213c70d21ef33a938a8e11c805dd9dcff89aafe92d0b2dc6cc8ecf0a984441`.

- 该表格行来自《新松机器人点焊应用操作手册》(SX322007)，描述了一条指令配置：步骤序号 **2** 对应事件 **焊接开始**，触发动作为 **输出IO**，参数值为 **1**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1380–1385; corpus 4.0, revision V5.0 / A-2; object `0b7f2934df81fae53b8121def6d49422f1846ac73e21e3c24f0f9b9b596ff997`.

- 该行位于《新松机器人点焊应用操作手册》的表格中（非表头行）。第一列为跨7行的合并单元格，内容为“说明”；第二列为“参数项 1”；第三列为“位置变量号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1113–1117; corpus 4.0, revision V5.0 / A-2; object `b5c444d2c0bafd3856f6b07414124f83e3f22fc7c2f7e000948a46607ba42bbf`.

- 新松机器人点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 11–13; corpus 5.0, revision V5.0 / A-2; object `84fbddc59ed577db3eb4eb99bccfcaa80459690ab956da1d77fffa769d9853bd`.

- 该行合并了3列，内容为一条新松机器人点焊程序状态指令：`SPOTJ P[1] VJ=10 ACC=100 CNT=0 DRESS GUN=1 PROGNo=123456789 TWC=0`，表明执行点焊程序P[1]时，速度10、加速度100、循环0、工装状态、焊枪1、程序号123456789且未启用TWC。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1147–1149; corpus 5.0, revision V5.0 / A-2; object `e6b5a4d4bab937ae07d3d6b7a556bb4936abd124037e9ed42b42740c57253e17`.

- 该表格行（索引 2，来自《新松机器人点焊应用操作手册》）记录了第 1 项配置：功能类型为“焊接/试验”，对应的信号为“输出 IO”，其参数值设为“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1374–1379; corpus 5.0, revision V5.0 / A-2; object `dc01c72c705d306b3f413f83037cc36cead70c548d40d43507d037f2deb1334e`.

- 该表格定义了“GUNPICK GUN #1 TF=1”的功能为“焊钳取枪指令”，其说明部分包含两项关键参数：参数项1对应“焊钳编号”，参数项2对应“工具坐标系号”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1187–1204; corpus 4.0, revision V5.0 / A-2; object `9839eaf61ff54e6e396265b5670d5362a094482727a1e5ec197909114366350e`.

#### Answer

A contextual code example could not be produced in the required format after a retry. The retrieved content summary remains available. Validation issue: Expecting value: line 1 column 1 (char 0)

#### Execution diagnostic

Assessment: **not_assessable**; completeness: unknown; citation support: not_applicable; revision fidelity: unclear.

The retrieval/answer stage failed: Response validation failed: Expecting value: line 1 column 1 (char 0)

## Material Handling — sampled manual revisions

40 cases from 2 CSV files. Set `MATERIAL_HANDLING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
MATERIAL_HANDLING_SAMPLE_CASES = [{'id': 'sample_material_handling_aee5375b4a_01', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3.6', 'section_title': '1.3.6 电磁阀配置', 'heading_path': ['1 界面配置', '1.3 搬运参数配置', '1.3.6 电磁阀配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），操作“电磁阀配置”时，涉及“输出”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出：气缸打开和气缸关闭，用户输出IO配置，状态位对应IO的状态', '输出的状态可手动强制输出，条件是上电+deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 303, 'line_end': 303, 'excerpt': '输出：气缸打开和气缸关闭，用户输出IO配置，状态位对应IO的状态。输出的状态可手动强制输出，条件是上电+deadman使能。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_02', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），操作“报警说明”时，涉及“提示：7258”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['提示：7258 当前输出IO存在模拟状态,请注意!'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 898, 'line_end': 904, 'excerpt': '<tr>\n<td><p>提示：7258</p></td>\n<td><p>当前输出IO存在模拟状态,请注意&#33;</p></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_03', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 电磁阀指令集', 'heading_path': ['2 搬运指令及快捷键', '2.3 电磁阀指令集'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），“电磁阀指令集”中“电磁阀指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['电磁阀指令 功能 电磁阀控制，打开/关闭，检测/不检测，实时检测打开/关闭'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 402, 'line_end': 406, 'excerpt': '<tr>\n<td rowspan="4"><p>电磁阀指令</p></td>\n<td><p>功能</p></td>\n<td><p>电磁阀控制，打开/关闭，检测/不检测，实时检测打开/关闭。 </p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_04', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 搬运运动点', 'heading_path': ['2 搬运指令及快捷键', '2.1 搬运运动点'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“搬运运动点”中，应如何理解或操作“MOVJ_FRAME P3 VJ = 20 ACC=100 CNT= 0 INTERLOCK=1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['举例 MOVJ_FRAME P3 VJ = 20 ACC=100 CNT= 0 INTERLOCK=1 //以关节速度运动到P3点，进行安全门请求判断', 'MOVL_FRAME P4 VL = 200 ACC=100 CNT=0 ET=1 INTERLOCK=1 //以直角速度运动到P4点，进行安全门请求判断'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 353, 'line_end': 356, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td><p>MOVJ&#95;FRAME P3 VJ = 20 ACC=100 CNT= 0 INTERLOCK=1  //以关节速度运动到P3点，进行安全门请求判断。</p><p>MOVL&#95;FRAME P4 VL = 200 ACC=100 CNT=0 ET=1 INTERLOCK=1  //以直角速度运动到P4点，进行安全门请求判断。</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_05', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“基础配置”中，应如何理解或操作“机器人型号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 机器人型号：根据现场机器人实际型号配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 120, 'line_end': 120, 'excerpt': '1. 机器人型号：根据现场机器人实际型号配置。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_06', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3.3', 'section_title': '1.3.3 抓手参数配置', 'heading_path': ['1 界面配置', '1.3 搬运参数配置', '1.3.3 抓手参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），配置“抓手参数配置”时，“长度”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['长度：从起始IO算起，需要监控的长度范围'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 201, 'line_end': 201, 'excerpt': '长度：从起始IO算起，需要监控的长度范围。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_07', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），配置“报警说明”时，“报警：7233”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['报警：7233 超时时间为0, 抓手号： 0 子码为抓手号 请检查配置界面中超时时间参数是否配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 835, 'line_end': 841, 'excerpt': '<tr>\n<td><p>报警：7233</p></td>\n<td><p>超时时间为0, 抓手号：</p></td>\n<td><p>0</p></td>\n<td><p>子码为抓手号</p></td>\n<td><p>请检查配置界面中超时时间参数是否配置</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_08', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“报警说明”中，应如何理解或操作“报警：7222”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警：7222 电磁阀监控异常：抓手号 0 子码为抓手号 电磁阀实时监控异常，请检查相关抓手号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 758, 'line_end': 764, 'excerpt': '<tr>\n<td><p>报警：7222</p></td>\n<td><p>电磁阀监控异常：抓手号</p></td>\n<td><p>0</p></td>\n<td><p>子码为抓手号</p></td>\n<td><p>电磁阀实时监控异常，请检查相关抓手号</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_09', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），“报警说明”中“报警：7221”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警：7221 当前指令不支持1.0规划，请更改！ 0 搬运应用只支持2.0规划，请重新配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 751, 'line_end': 757, 'excerpt': '<tr>\n<td><p>报警：7221</p></td>\n<td><p>当前指令不支持1&#46;0规划，请更改！</p></td>\n<td><p>0</p></td>\n<td></td>\n<td><p>搬运应用只支持2&#46;0规划，请重新配置。</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_10', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），配置“快捷键”时，“吸盘电磁阀组号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['吸盘电磁阀组号：参数项为1-8', '通过快捷键来控制该吸盘电磁阀组打开、关闭'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 493, 'line_end': 493, 'excerpt': '吸盘电磁阀组号：参数项为1-8。通过快捷键来控制该吸盘电磁阀组打开、关闭。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_11', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“作业模板”中，应如何理解或操作“A01_CMN UserRequest Request UserNo=3”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['A01_CMN UserRequest Request UserNo=3 //请求进入抓件区域'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 526, 'line_end': 529, 'excerpt': '<tr>\n<td><p>A01&#95;CMN UserRequest Request UserNo=3 </p></td>\n<td><p>//请求进入抓件区域</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_12', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3.7', 'section_title': '1.3.7 吸盘电磁阀配置', 'heading_path': ['1 界面配置', '1.3 搬运参数配置', '1.3.7 吸盘电磁阀配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），操作“吸盘电磁阀配置”时，涉及“输出”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出：真空打开和真空关闭，用户输出IO配置，状态位对应IO的状态', '输出的状态可手动强制输出，条件是上电+deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 325, 'line_end': 325, 'excerpt': '输出：真空打开和真空关闭，用户输出IO配置，状态位对应IO的状态。输出的状态可手动强制输出，条件是上电+deadman使能。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_13', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），操作“报警说明”时，涉及“报警：7218”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['报警：7218 吸盘信号屏蔽次数达到设定值 0 吸盘信号的屏蔽次数已达到设定值并清零,请注意！'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 730, 'line_end': 736, 'excerpt': '<tr>\n<td><p>报警：7218</p></td>\n<td><p>吸盘信号屏蔽次数达到设定值</p></td>\n<td><p>0</p></td>\n<td></td>\n<td><p>吸盘信号的屏蔽次数已达到设定值并清零,请注意！</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_14', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 工件传感器指令集', 'heading_path': ['2 搬运指令及快捷键', '2.2 工件传感器指令集'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），“工件传感器指令集”中“A03_MH PartPresent WithPart (参数1) GripNo=(参数2) A”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A03_MH'], 'expected_facts': ['格式 A03_MH PartPresent WithPart (参数1) GripNo=(参数2) AllType(参数3) Motion (参数4) ;注释内容 A03_MH PartPresent NoPart (参数1) GripNo=(参数2) TypeNo=(参数3) NoMotion (参数4) ;注释内容'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 377, 'line_end': 380, 'excerpt': '<tr>\n<td><p>格式</p></td>\n<td><p>A03&#95;MH PartPresent WithPart &#40;参数1&#41; GripNo=&#40;参数2&#41; AllType&#40;参数3&#41; Motion &#40;参数4&#41; ;注释内容</p><p>A03&#95;MH PartPresent NoPart &#40;参数1&#41; GripNo=&#40;参数2&#41; TypeNo=&#40;参数3&#41; NoMotion &#40;参数4&#41; ;注释内容</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_15', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“应用切换”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['应用切换:根据所配置的应用，可以切换不同的OP1应用的快捷键界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 487, 'line_end': 487, 'excerpt': '应用切换:根据所配置的应用，可以切换不同的OP1应用的快捷键界面。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_16', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3.2', 'section_title': '1.3.2 水气单元', 'heading_path': ['1 界面配置', '1.3 搬运参数配置', '1.3.2 水气单元'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“水气单元”中，应如何理解或操作“水流量检测开关”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 水流量检测开关，打开后，后台开始监控水流量对应的IO信号', '根据对应情况进行报警'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 170, 'line_end': 170, 'excerpt': '1. 水流量检测开关，打开后，后台开始监控水流量对应的IO信号。根据对应情况进行报警。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_17', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3.3', 'section_title': '1.3.3 抓手参数配置', 'heading_path': ['1 界面配置', '1.3 搬运参数配置', '1.3.3 抓手参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），配置“抓手参数配置”时，“将应用类型配置为吉利搬运”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['将应用类型配置为吉利搬运，输入IO分配：IO起始位配置为1，长度为40', '输出IO分配：IO起始位配置为1，长度为40', '通讯状态IO起始位为300，长度为2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 187, 'line_end': 187, 'excerpt': '将应用类型配置为吉利搬运，输入IO分配：IO起始位配置为1，长度为40。输出IO分配：IO起始位配置为1，长度为40。通讯状态IO起始位为300，长度为2。上述IO范围仅为举例说明，具体使用请改变IO范围。', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_18', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），配置“报警说明”时，“报警：7204”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['报警：7204 抓手号超界错误 0 - 抓手号赋值问题, 请赋值在设置抓手号1-8范围内'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 639, 'line_end': 645, 'excerpt': '<tr>\n<td><p>报警：7204</p></td>\n<td><p>抓手号超界错误</p></td>\n<td><p>0</p></td>\n<td><p>&#45;</p></td>\n<td><p>抓手号赋值问题, 请赋值在设置抓手号1&#45;8范围内</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_19', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），在“报警说明”中，应如何理解或操作“报警：7213”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警：7213 吸盘真空关闭信号检测异常 0 - 请检查吸盘真空打开信号为OFF状态'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 702, 'line_end': 708, 'excerpt': '<tr>\n<td><p>报警：7213</p></td>\n<td><p>吸盘真空关闭信号检测异常</p></td>\n<td><p>0</p></td>\n<td><p>&#45;</p></td>\n<td><p>请检查吸盘真空打开信号为OFF状态</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_aee5375b4a_20', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），“报警说明”中“报警：7228”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警：7228 指令输入抓手号通讯断开：抓手号 0 子码为抓手号 请检查当前执行指令抓手号的通讯状态'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'line_start': 800, 'line_end': 806, 'excerpt': '<tr>\n<td><p>报警：7228</p></td>\n<td><p>指令输入抓手号通讯断开：抓手号</p></td>\n<td><p>0</p></td>\n<td><p>子码为抓手号</p></td>\n<td><p>请检查当前执行指令抓手号的通讯状态</p></td>\n</tr>', 'sha256': '6119838e2ff087a4c8f9003b32e3bd6c3e0ab0b1b7232b02bc882f3796d4c089'}]}, {'id': 'sample_material_handling_40fac4d4f7_01', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.6', 'section_title': '1.3.6 电磁阀配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.6 电磁阀配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），操作“电磁阀配置”时，涉及“输出”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出：气缸打开和气缸关闭，用户输出IO配置，状态位对应IO的状态', '输出的状态可手动强制输出，条件是上电+deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 246, 'line_end': 246, 'excerpt': '输出：气缸打开和气缸关闭，用户输出IO配置，状态位对应IO的状态。输出的状态可手动强制输出，条件是上电+deadman使能。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_02', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.7', 'section_title': '1.3.7 吸盘电磁阀配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.7 吸盘电磁阀配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“吸盘电磁阀配置”时，“到位检测时间”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['到位检测时间：真空打开或关闭时的超时时间判断'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 264, 'line_end': 264, 'excerpt': '到位检测时间：真空打开或关闭时的超时时间判断。单位为秒。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_03', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“巡检配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['巡检配置：在执行运动+工艺点时判断机器人是否停止', '机器人巡检号与停止焊点号一致，机器人停止', '其中起始IO号的含义是：巡检功能起始IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 376, 'line_end': 376, 'excerpt': '巡检配置：在执行运动+工艺点时判断机器人是否停止。机器人巡检号与停止焊点号一致，机器人停止。其中起始IO号的含义是：巡检功能起始IO号。输入巡检信号：巡检功能输入巡检信号。输出确认信号：巡检功能输出确认信号。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_04', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“快捷键”时，“吸盘电磁阀组号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['吸盘电磁阀组号：参数项为1-8', '通过快捷键来控制该吸盘电磁阀组打开、关闭'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 362, 'line_end': 362, 'excerpt': '吸盘电磁阀组号：参数项为1-8。通过快捷键来控制该吸盘电磁阀组打开、关闭。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_05', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322011《新松机器人搬运应用操作手册》(A-0).md”（V5.0 / A-0），搬运模板在抓件前如何确认工具和区域，并检查工件、夹头和吸盘的状态？', 'anchors': ['CheckTool', 'UserRequest', 'NoPart', 'AllOpen', 'AllVacOff'], 'expected_facts': ['A02_TC CheckTool ToolNo=1 //确认工具', 'A01_CMN UserRequest Request UserNo=1 //请求进入大区', 'A01_CMN UserRequest Request UserNo=3 //请求进入抓件区域', 'A01_CMN UserRequest AllocateZone UserNo=21 //进入干涉区', 'A03_MH PartPresent NoPart GripNo=1 AllType NoMotion //无工件检测', 'A03_MH Gripper AllOpen GripNo=1 NoCheck NoMotion //打开所有夹头并检测', 'A03_MH Vacuum AllVacOff GripNo=1 Check NoMotion //吸盘吹气到位检测', 'A01_CMN UserRequest Request UserNo=5 //抓件完成请求离开', 'A01_CMN UserRequest ReleaseZone UserNo=22 //释放干涉区', 'A01_CMN UserRequest Finish UserNo =4 //请求离开抓件区域'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 382, 'line_end': 382, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>A02_TC CheckTool ToolNo=1    </p></td><td><p>//确认工具</p></td></tr><tr><td><p>A01_CMN UserRequest Request UserNo=1 </p></td><td><p>//请求进入大区</p></td></tr><tr><td><p>A01_CMN UserRequest Request UserNo=3 </p></td><td><p>//请求进入抓件区域</p></td></tr><tr><td><p>A01_CMN UserRequest AllocateZone UserNo=21 </p></td><td><p>//进入干涉区 </p></td></tr><tr><td><p>A03_MH PartPresent NoPart GripNo=1 AllType NoMotion</p></td><td><p>//无工件检测</p></td></tr><tr><td><p>A03_MH Gripper AllOpen GripNo=1 NoCheck NoMotion </p></td><td><p>//打开所有夹头并检测</p></td></tr><tr><td><p>A03_MH Vacuum AllVacOff GripNo=1 Check NoMotion  </p></td><td><p> //吸盘吹气到位检测</p></td></tr><tr><td><p>MOVJ P1 VJ=100 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>MOVL P2 VL=200 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>MOVL P3 VL=200 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>A03_MH PartPresent WithPart GripNo=1 AllType Motion</p></td><td><p>//传感器检测并后台监控</p></td></tr><tr><td><p>A03_MH Gripper AllClose GripNo=1 Check Motion  </p></td><td><p>//夹紧所有夹头并检测</p></td></tr><tr><td><p>A03_MH Vacuum AllVacOn GripNo=1 Check Motion  </p></td><td><p>//吸盘吸气到位检测并空</p></td></tr><tr><td><p>A01_CMN UserRequest Request UserNo=5    </p></td><td><p>//抓件完成请求离开</p></td></tr><tr><td><p>MOVL P4 VL=300 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>MOVJ P5 VJ=100 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>MOVJ P6 VJ=100 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>MOVJ P7 VJ=100 ACC 100 CNT 0 </p></td><td></td></tr><tr><td><p>A01_CMN UserRequest ReleaseZone UserNo=22  </p></td><td><p>//释放干涉区</p></td></tr><tr><td><p>A01_CMN UserRequest Finish UserNo =4 </p></td><td><p>//请求离开抓件区域</p></td></tr><tr><td><p>RET</p></td><td></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_06', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.5', 'section_title': '1.3.5 工件组配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.5 工件组配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），在“工件组配置”中，应如何理解或操作“传感器1-传感器8”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['传感器1-传感器8：针对当前车型，可以最多配置8个传感器'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 228, 'line_end': 228, 'excerpt': '传感器1-传感器8：针对当前车型，可以最多配置8个传感器。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_07', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.1', 'section_title': '2.1 搬运运动点', 'heading_path': ['2 搬运指令及快捷键', '2.1 搬运运动点'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），“搬运运动点”中“搬运指令集路径”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A03'], 'expected_facts': ['搬运指令集路径：指令集->应用指令集->搬运->A03\\_MH'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 292, 'line_end': 292, 'excerpt': '搬运指令集路径：指令集->应用指令集->搬运->A03\\_MH；', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_08', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），“报警说明”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OFF', 'SIM'], 'expected_facts': ['报警码 报警信息 子码 子码信息 解决方法 报警：7201 工件传感器监控异常：抓手号 0 子码为抓手号 工件传感器实时监控异常，请检查相关抓手号 报警：7202 吸盘监控异常: 抓手号 0 子码为抓手号 吸盘实时监控异常，请检查相关抓手号 报警：7203 文件操作失败 0 - 联系技术人员 报警：7204 抓手号超界错误 0 - 抓手号赋值问题, 请赋值在设置抓手号1-8范围内 报警：7205 电磁阀组号超界错误 0 - 电磁阀组号赋值问题,请赋值在该电磁阀组号1-8范围内 报警：7206 超时时间为0，请更改！ - - 修改超时时间非0 报警：7207 吸盘组号超界错误 0 - 吸盘组号赋值问题,请赋值在该吸盘组号1-8范围内 报警：7208 指针为空 0 - 联系技术人员 报警：7209 电磁阀打开信号检测异常 0 - 请检查电磁阀打开信号状态 报警：7210 工件传感器信号检测异常 0 - 请检查工件传感器信号状态 报警：7211 电磁阀关闭信号检测异常 0 - 请检查电磁阀关闭信号状态 报警：7212 吸盘真空打开信号检测异常 0 - 请检查吸盘真空打开信号为ON状态 报警：7213 吸盘真空关闭信号检测异常 0 - 请检查吸盘真空打开信号为OFF状态 报警：7215 指令名称不在配置界面中 0 指令中的工件传感器名称与配置界面中的名称不一致，请检查！ 报警：7216 工件传感器屏蔽次数达到设定值 0 工件传感器的屏蔽次数已达到设定值并清零,请注意！ 报警：7217 电磁阀信号屏蔽次数达到设定值 0 电磁阀信号的屏蔽次数已达到设定值并清零,请注意！ 报警：7218 吸盘信号屏蔽次数达到设定值 0 吸盘信号的屏蔽次数已达到设定值并清零,请注意！ 报警：7219 配置IO不在逻辑IO分配范围内 0 请核对IO配置和逻辑IO分配,修改配置范围 报警：7220 机器人干涉 0 检查干涉区 报警：7221 当前指令不支持1.0规划，请更改！ 0 搬运应用只支持2.0规划，请重新配置', '报警：7222 电磁阀监控异常：抓手号 0 子码为抓手号 电磁阀实时监控异常，请检查相关抓手号 报警：7223 工件传感器监控异常：序号 0 子码为工件传感器号 检查工件传感器 报警：7224 电磁阀监控异常：电磁阀组号 0 子码为电磁阀组号 检查电磁阀 报警：7225 电磁阀监控异常：序号 0 子码为电磁阀号 检查电磁阀 报警：7226 吸盘监控异常：吸盘组号 0 子码为吸盘组号 检查吸盘 报警：7227 吸盘监控异常：序号 0 子码为吸盘号 检查吸盘 报警：7228 指令输入抓手号通讯断开：抓手号 0 子码为抓手号 请检查当前执行指令抓手号的通讯状态 报警：7229 断电数据大小超界 0 断电恢复数据超界，请联系专业人员 报警：7230 断电数据校验失败 0 数据检验失败，请关闭断电恢复开关重启，再重新打开 报警：7231 水流量异常 0 请检查水流量外设设备是否正常 报警：7232 气流量异常 0 请检查气流量外设设备是否正常 报警：7233 超时时间为0, 抓手号： 0 子码为抓手号 请检查配置界面中超时时间参数是否配置 报警：7234 通讯状态异常 0 请检查相应的外设设备的通讯线或模块是否正常 提示：7251 执行电磁阀打开快捷功能 0 提示：7252 执行电磁阀关闭快捷功能 0 提示：7253 执行吸盘电磁阀打开快捷功能 0 提示：7254 执行吸盘电磁阀关闭快捷功能 0 提示：7255 响应安全门请求暂停中 0 提示：7256 当前模式为SIM模式! 0 提示：7257 当前输入IO存在模拟状态,请注意! 提示：7258 当前输出IO存在模拟状态,请注意! 提示：7259 应用配置已更改，已初始化! 提示：7260 通讯为屏蔽状态 提示：7261 当前抓手已手动屏蔽，抓手号： 子码为抓手号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 386, 'line_end': 386, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码信息</p></td><td><p>解决方法</p></td></tr><tr><td><p>报警：7201</p></td><td><p>工件传感器监控异常：抓手号</p></td><td><p>0</p></td><td><p>子码为抓手号</p></td><td><p>工件传感器实时监控异常，请检查相关抓手号</p></td></tr><tr><td><p>报警：7202</p></td><td><p>吸盘监控异常: 抓手号</p></td><td><p>0</p></td><td><p>子码为抓手号</p></td><td><p>吸盘实时监控异常，请检查相关抓手号</p></td></tr><tr><td><p>报警：7203</p></td><td><p>文件操作失败</p></td><td><p>0</p></td><td><p>-</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>报警：7204</p></td><td><p>抓手号超界错误</p></td><td><p>0</p></td><td><p>-</p></td><td><p>抓手号赋值问题, 请赋值在设置抓手号1-8范围内</p></td></tr><tr><td><p>报警：7205</p></td><td><p>电磁阀组号超界错误</p></td><td><p>0</p></td><td><p>-</p></td><td><p>电磁阀组号赋值问题,请赋值在该电磁阀组号1-8范围内</p></td></tr><tr><td><p>报警：7206</p></td><td><p>超时时间为0，请更改！</p></td><td><p>-</p></td><td><p>-</p></td><td><p>修改超时时间非0</p></td></tr><tr><td><p>报警：7207</p></td><td><p>吸盘组号超界错误</p></td><td><p>0</p></td><td><p>-</p></td><td><p>吸盘组号赋值问题,请赋值在该吸盘组号1-8范围内</p></td></tr><tr><td><p>报警：7208</p></td><td><p>指针为空</p></td><td><p>0</p></td><td><p>-</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>报警：7209</p></td><td><p>电磁阀打开信号检测异常</p></td><td><p>0</p></td><td><p>-</p></td><td><p>请检查电磁阀打开信号状态</p></td></tr><tr><td><p>报警：7210</p></td><td><p>工件传感器信号检测异常</p></td><td><p>0</p></td><td><p>-</p></td><td><p>请检查工件传感器信号状态</p></td></tr><tr><td><p>报警：7211</p></td><td><p>电磁阀关闭信号检测异常</p></td><td><p>0</p></td><td><p>-</p></td><td><p>请检查电磁阀关闭信号状态</p></td></tr><tr><td><p>报警：7212</p></td><td><p>吸盘真空打开信号检测异常</p></td><td><p>0</p></td><td><p>-</p></td><td><p>请检查吸盘真空打开信号为ON状态</p></td></tr><tr><td><p>报警：7213</p></td><td><p>吸盘真空关闭信号检测异常</p></td><td><p>0</p></td><td><p>-</p></td><td><p>请检查吸盘真空打开信号为OFF状态</p></td></tr><tr><td><p>报警：7215</p></td><td><p>指令名称不在配置界面中</p></td><td><p>0</p></td><td></td><td><p>指令中的工件传感器名称与配置界面中的名称不一致，请检查！</p></td></tr><tr><td><p>报警：7216</p></td><td><p>工件传感器屏蔽次数达到设定值</p></td><td><p>0</p></td><td></td><td><p>工件传感器的屏蔽次数已达到设定值并清零,请注意！</p></td></tr><tr><td><p>报警：7217</p></td><td><p>电磁阀信号屏蔽次数达到设定值</p></td><td><p>0</p></td><td></td><td><p>电磁阀信号的屏蔽次数已达到设定值并清零,请注意！</p></td></tr><tr><td><p>报警：7218</p></td><td><p>吸盘信号屏蔽次数达到设定值</p></td><td><p>0</p></td><td></td><td><p>吸盘信号的屏蔽次数已达到设定值并清零,请注意！</p></td></tr><tr><td><p>报警：7219</p></td><td><p>配置IO不在逻辑IO分配范围内</p></td><td><p>0</p></td><td></td><td><p>请核对IO配置和逻辑IO分配,修改配置范围</p></td></tr><tr><td><p>报警：7220</p></td><td><p>机器人干涉</p></td><td><p>0</p></td><td></td><td><p>检查干涉区</p></td></tr><tr><td><p>报警：7221</p></td><td><p>当前指令不支持1.0规划，请更改！</p></td><td><p>0</p></td><td></td><td><p>搬运应用只支持2.0规划，请重新配置。</p></td></tr><tr><td><p>报警：7222</p></td><td><p>电磁阀监控异常：抓手号</p></td><td><p>0</p></td><td><p>子码为抓手号</p></td><td><p>电磁阀实时监控异常，请检查相关抓手号</p></td></tr><tr><td><p>报警：7223</p></td><td><p>工件传感器监控异常：序号</p></td><td><p>0</p></td><td><p>子码为工件传感器号</p></td><td><p>检查工件传感器</p></td></tr><tr><td><p>报警：7224</p></td><td><p>电磁阀监控异常：电磁阀组号</p></td><td><p>0</p></td><td><p>子码为电磁阀组号</p></td><td><p>检查电磁阀</p></td></tr><tr><td><p>报警：7225</p></td><td><p>电磁阀监控异常：序号</p></td><td><p>0</p></td><td><p>子码为电磁阀号</p></td><td><p>检查电磁阀</p></td></tr><tr><td><p>报警：7226</p></td><td><p>吸盘监控异常：吸盘组号</p></td><td><p>0</p></td><td><p>子码为吸盘组号</p></td><td><p>检查吸盘</p></td></tr><tr><td><p>报警：7227</p></td><td><p>吸盘监控异常：序号</p></td><td><p>0</p></td><td><p>子码为吸盘号</p></td><td><p>检查吸盘</p></td></tr><tr><td><p>报警：7228</p></td><td><p>指令输入抓手号通讯断开：抓手号</p></td><td><p>0</p></td><td><p>子码为抓手号</p></td><td><p>请检查当前执行指令抓手号的通讯状态</p></td></tr><tr><td><p>报警：7229</p></td><td><p>断电数据大小超界</p></td><td><p>0</p></td><td></td><td><p>断电恢复数据超界，请联系专业人员</p></td></tr><tr><td><p>报警：7230</p></td><td><p>断电数据校验失败</p></td><td><p>0</p></td><td></td><td><p>数据检验失败，请关闭断电恢复开关重启，再重新打开</p></td></tr><tr><td><p>报警：7231</p></td><td><p>水流量异常</p></td><td><p>0</p></td><td></td><td><p>请检查水流量外设设备是否正常</p></td></tr><tr><td><p>报警：7232</p></td><td><p>气流量异常</p></td><td><p>0</p></td><td></td><td><p>请检查气流量外设设备是否正常</p></td></tr><tr><td><p>报警：7233</p></td><td><p>超时时间为0, 抓手号：</p></td><td><p>0</p></td><td><p>子码为抓手号</p></td><td><p>请检查配置界面中超时时间参数是否配置</p></td></tr><tr><td><p>报警：7234</p></td><td><p>通讯状态异常</p></td><td><p>0</p></td><td></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>提示：7251</p></td><td><p>执行电磁阀打开快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7252</p></td><td><p>执行电磁阀关闭快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7253</p></td><td><p>执行吸盘电磁阀打开快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7254</p></td><td><p>执行吸盘电磁阀关闭快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7255</p></td><td><p>响应安全门请求暂停中</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7256</p></td><td><p>当前模式为SIM模式!</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7257</p></td><td><p>当前输入IO存在模拟状态,请注意!</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7258</p></td><td><p>当前输出IO存在模拟状态,请注意!</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7259</p></td><td><p>应用配置已更改，已初始化!</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7260</p></td><td><p>通讯为屏蔽状态</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7261</p></td><td><p>当前抓手已手动屏蔽，抓手号：</p></td><td></td><td><p>子码为抓手号</p></td><td></td></tr></table>', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_09', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.7', 'section_title': '1.3.7 吸盘电磁阀配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.7 吸盘电磁阀配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），操作“吸盘电磁阀配置”时，涉及“输出”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出：真空打开和真空关闭，用户输出IO配置，状态位对应IO的状态', '输出的状态可手动强制输出，条件是上电+deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 268, 'line_end': 268, 'excerpt': '输出：真空打开和真空关闭，用户输出IO配置，状态位对应IO的状态。输出的状态可手动强制输出，条件是上电+deadman使能。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_10', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.2', 'section_title': '1.3.2 水气单元', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.2 水气单元'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“水气单元”时，“延时检查水流量”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['3. 延时检查水流量，复位，换枪等，需要重新打开水阀的时候，延时一定时间检测水流量，单位ms'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 115, 'line_end': 115, 'excerpt': '3. 延时检查水流量，复位，换枪等，需要重新打开水阀的时候，延时一定时间检测水流量，单位ms。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_11', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“同步快捷键OP2-OP5”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['同步快捷键OP2-OP5：可以配置气缸打开、气缸关闭、真空打开、真空关闭、用户IO输出'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 366, 'line_end': 366, 'excerpt': '同步快捷键OP2-OP5：可以配置气缸打开、气缸关闭、真空打开、真空关闭、用户IO输出。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_12', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“快捷键”时，“抓手号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['抓手号：参数项为1-8'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 358, 'line_end': 358, 'excerpt': '抓手号：参数项为1-8。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_13', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 搬运应用配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322011《新松机器人搬运应用操作手册》(A-0).md”（V5.0 / A-0），要在“1.1 基础配置”中配置“总线IO映射”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：IO->总线IO映射'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 77, 'line_end': 77, 'excerpt': '配置路径：IO->总线IO映射；', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_14', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 工件传感器指令集', 'heading_path': ['2 搬运指令及快捷键', '2.2 工件传感器指令集'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），“工件传感器指令集”中“工件传感器指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A03_MH'], 'expected_facts': ['工件传感器指令 功能 检查抓手是否存在工件，实时监控/不实时监控', '格式 A03_MH PartPresent WithPart (参数1) GripNo=(参数2) AllType(参数3) Motion (参数4) ;注释内容 A03_MH PartPresent NoPart (参数1) GripNo=(参数2) TypeNo=(参数3) NoMotion (参数4) ;注释内容 说明 <参数1>：有工件/无工件，参数项分别为WithPart/NoPart', '<参数2>：抓手编号，参数项为1-8'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 304, 'line_end': 304, 'excerpt': '<table><tr><td rowspan="4"><p>工件传感器指令</p></td><td><p>功能</p></td><td><p>检查抓手是否存在工件，实时监控/不实时监控。</p></td></tr><tr><td><p>格式</p></td><td><p>A03_MH PartPresent WithPart (参数1) GripNo=(参数2) AllType(参数3) Motion (参数4) ;注释内容</p><p>A03_MH PartPresent NoPart (参数1) GripNo=(参数2) TypeNo=(参数3) NoMotion (参数4) ;注释内容</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：有工件/无工件，参数项分别为WithPart/NoPart；</p><p>&lt;参数2&gt;：抓手编号，参数项为1-8；</p><p>&lt;参数3&gt;：工件组名称，参数项为AllType/TypeNo=工件组名称；AllType表示不区分信号检测所有工件传感器，TypeNo=工件组名称，检测对应工件组所配置的工件传感器；为增加易用性，在工件传感器号后面加入注释列表，点击注释列表会出现工件组的名称，返回到指令中输出工件组对应的名称；</p><p>&lt;参数4&gt;：关闭实时监控状态/打开实时监控状态，参数项为NoMotion/Motion；</p></td></tr><tr><td><p>举例</p></td><td><p>A03_MH PartPresent WithPart GripNo=1 AllType Motion ;注释内容 //抓手一检测工件传感器全部信号有工件，并打开实时监控。</p><p>A03_MH PartPresent WithPart GripNo=1 AllType NoMotion ;注释内容 //抓手一检测工件传感器全部信号有工件，并关闭实时监控。</p><p>A03_MH PartPresent NoPart GripNo=2 TypeNo= 123 Motion ;注释内容 //抓手二检测工件组名称为123的工件传感器信号无工件，打开实时监控。</p><p>A03_MH PartPresent NoPart GripNo=2 AllType NoMotion ;注释内容 //抓手二检测工件传感器全部信号无工件，关闭实时监控。</p></td></tr></table>', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_15', 'category': 'material_handling', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.3', 'section_title': '1.3.3 抓手参数配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.3 抓手参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），操作“抓手参数配置”时，涉及“搬运IO配置和起始位和长度要在逻辑IO分配的搬运起始位和长度范围内”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['搬运IO配置和起始位和长度要在逻辑IO分配的搬运起始位和长度范围内', '若不满足条件则报警，若满足条件，将抓手配置的IO根据设置的输入和输出起始位顺序填入'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 172, 'line_end': 172, 'excerpt': '搬运IO配置和起始位和长度要在逻辑IO分配的搬运起始位和长度范围内。若不满足条件则报警，若满足条件，将抓手配置的IO根据设置的输入和输出起始位顺序填入。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_16', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3.3', 'section_title': '1.3.3 抓手参数配置', 'heading_path': ['1 搬运应用配置', '1.3 搬运参数配置', '1.3.3 抓手参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“抓手参数配置”时，“将应用类型配置为吉利搬运”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['将应用类型配置为吉利搬运，输入IO分配：IO起始位配置为1，长度为40', '输出IO分配：IO起始位配置为1，长度为40', '通讯状态IO起始位为300，长度为2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 130, 'line_end': 130, 'excerpt': '将应用类型配置为吉利搬运，输入IO分配：IO起始位配置为1，长度为40。输出IO分配：IO起始位配置为1，长度为40。通讯状态IO起始位为300，长度为2。上述IO范围仅为举例说明，具体使用请改变IO范围。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_17', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“PLC-DRY”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['PLC-DRY: 代表的是PLC发送工艺关-干运行模式的输入IO号、输出IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 372, 'line_end': 372, 'excerpt': 'PLC-DRY: 代表的是PLC发送工艺关-干运行模式的输入IO号、输出IO号。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_18', 'category': 'material_handling', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.6', 'section_title': '2.6 快捷键', 'heading_path': ['2 搬运指令及快捷键', '2.6 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），配置“快捷键”时，“电磁阀组号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['电磁阀组号：参数项为1-8', '通过快捷键来控制该电磁阀组打开、关闭'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 360, 'line_end': 360, 'excerpt': '电磁阀组号：参数项为1-8。通过快捷键来控制该电磁阀组打开、关闭。', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_19', 'category': 'material_handling', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 搬运应用配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322011《新松机器人搬运应用操作手册》(A-0).md”（V5.0 / A-0），要在“1.1 基础配置”中配置“型号配置”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：系统设置->型号配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 65, 'line_end': 65, 'excerpt': '配置路径：系统设置->型号配置；', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}, {'id': 'sample_material_handling_40fac4d4f7_20', 'category': 'material_handling', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.4', 'section_title': '2.4 吸盘电磁阀指令集', 'heading_path': ['2 搬运指令及快捷键', '2.4 吸盘电磁阀指令集'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），“吸盘电磁阀指令集”中“吸盘指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A03_MH'], 'expected_facts': ['吸盘指令 功能 吸盘控制，打开/关闭，检测/不检测，实时检测打开/关闭', '格式 A03_MH Vacuum VacOn(参数1) GripNo=1(参数2) VacNo =1 (参数3)Check(参数4) Motion(参数5)', '注释内容 A03_MH Vacuum VacOff(参数1) GripNo=1(参数2) VacNo =1 (参数3)NoCheck(参数4) Motion(参数5)'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'line_start': 328, 'line_end': 328, 'excerpt': '<table><tr><td rowspan="4"><p>吸盘指令</p></td><td><p>功能</p></td><td colspan="2"><p>吸盘控制，打开/关闭，检测/不检测，实时检测打开/关闭。 </p></td></tr><tr><td><p>格式</p></td><td><p>A03_MH Vacuum VacOn(参数1) GripNo=1(参数2) VacNo =1 (参数3)Check(参数4)  Motion(参数5) ；注释内容</p><p>A03_MH Vacuum VacOff(参数1) GripNo=1(参数2) VacNo =1 (参数3)NoCheck(参数4)  Motion(参数5) ；注释内容</p><p>A03_MH Vacuum AllVacOn(参数1) GripNo=1(参数2) Check(参数4)  Motion(参数5) ；注释内容</p><p>A03_MH Vacuum AllVacOff(参数1) GripNo=1(参数2) Check(参数4)  Motion(参数5) ；注释内容</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：打开/关闭，参数项分别为VacOn / VacOff / AllVacOn / AllVacOff； </p><p>&lt;参数2&gt;：抓手编号，参数项为1-8；</p><p>&lt;参数3&gt;：吸盘阀片组号，参数项为1-8；为增加易用性，在吸盘电磁阀阀片号后面加入注释列表，点击注释列表会出现电磁阀阀片的名称，返回对应的吸盘电磁阀阀片号；</p><p>&lt;参数4&gt;：检查状态/不检查状态，参数项为Check/NoCheck；</p><p>&lt;参数5&gt;：关闭实时监控状态/打开实时监控状态，参数项为NoMotion/Motion； </p></td></tr><tr><td><p>举例</p></td><td><p>A03_MH Vacuum VacOn GripNo=1 VacNo =1 Check Motion ;注释内容 //抓手一吸盘1组打开，检测，并打开实时监控。</p><p>A03_MH Vacuum VacOn GripNo=1 VacNo =2 Check NoMotion ;注释内容 //抓手一吸盘2组打开，检测，并关闭实时监控。</p><p>A03_MH Vacuum VacOff GripNo=2 VacNo =3 NoCheck Motion ;注释内容 //抓手二吸盘3组关闭，不检测，打开实时监控。</p><p>A03_MH Vacuum AllVacOn GripNo=2 NoCheck Motion ;注释内容 //抓手二吸盘全部打开，不检测，打开实时监控。</p><p>A03_MH Vacuum AllVacOff GripNo=2 Check Motion ;注释内容 //抓手二吸盘全部关闭，检测，打开实时监控。</p></td></tr></table>', 'sha256': '2880645657d70f881b0dc9530aa52f1191eaaf92a4cbfa82a20e5aa8287e4c6b'}]}]
register_sample_bank('material_handling', MATERIAL_HANDLING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md', 'seed': 14943740547044382649, 'population': 158, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(303, 303), (898, 904), (402, 406), (353, 356), (120, 120), (201, 201), (835, 841), (758, 764), (751, 757), (493, 493), (526, 529), (325, 325), (730, 736), (377, 380), (487, 487), (170, 170), (187, 187), (639, 645), (702, 708), (800, 806)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md', 'seed': 12691973408965427153, 'population': 81, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(246, 246), (264, 264), (376, 376), (362, 362), (382, 382), (228, 228), (292, 292), (386, 386), (268, 268), (115, 115), (366, 366), (358, 358), (77, 77), (304, 304), (172, 172), (130, 130), (372, 372), (360, 360), (65, 65), (328, 328)]}])


In [ ]:
# | notest
MATERIAL_HANDLING_SAMPLE_IDS = None
material_handling_sample_cases = select_cases(MATERIAL_HANDLING_SAMPLE_CASES, MATERIAL_HANDLING_SAMPLE_IDS)
display(pd.DataFrame(material_handling_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in material_handling_sample_cases])
material_handling_sample_metrics = await evaluation.run(material_handling_sample_cases, concurrency=EVALUATION_CONCURRENCY)
material_handling_sample_results = {case["id"]: evaluation.load(case) for case in material_handling_sample_cases}
display(material_handling_sample_metrics)
# Inspect a complete result: evaluation.inspect(material_handling_sample_cases[0]["id"])
evaluation.show_answers(material_handling_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_material_handling_aee5375b4a_01,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,prerequisite,prose,False,True,8,False,0,0,4,False,False,False,None,None,11441,0,True,unsupported,,,incorrect
sample_material_handling_aee5375b4a_02,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,prerequisite,table_row,False,True,8,False,0,0,0,False,False,False,None,None,15034,0,True,included,,,not reviewed
sample_material_handling_aee5375b4a_03,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,14835,0,True,included,,,incorrect
sample_material_handling_aee5375b4a_04,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,9086,0,True,included,,,not reviewed
sample_material_handling_aee5375b4a_05,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,7,False,False,False,None,None,5178,0,True,unsupported,,,not reviewed
sample_material_handling_aee5375b4a_06,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,5,False,False,False,None,None,7009,0,True,unsupported,,,not reviewed
sample_material_handling_aee5375b4a_07,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,14925,0,True,unsupported,,,not reviewed
sample_material_handling_aee5375b4a_08,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,15179,0,True,unsupported,,,not reviewed
sample_material_handling_aee5375b4a_09,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,1,False,False,False,None,None,16376,0,True,unsupported,,,not reviewed
sample_material_handling_aee5375b4a_10,material_handling,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,1,False,False,False,None,None,14952,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed


### sample_material_handling_aee5375b4a_01

根据《SX322011《新松机器人搬运应用操作手册》-A-0》（V4.0 / A-0），操作“电磁阀配置”时，涉及“输出”的前提条件或注意事项是什么？

#### Retrieved content summary

- 该表行为第1行（非表头），包含1个单元格（第1列），其内容为空，无合并单元格或继承值，属于文档“SX322011《新松机器人搬运应用操作手册》(A-0)”中的空白占位符行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 2–4; corpus 5.0, revision V5.0 / A-0; object `366ba655b5ab7028c11b2a194674067dba02791d2ade53683f4ecb4714b01910`.

- 第3行为空行，包含4个无内容单元格，源自《新松机器人搬运应用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 40–45; corpus 5.0, revision V5.0 / A-0; object `4da9286c6b4e8a8ac055f595f2d3657da67a9516e83910de9abf6272da4e4502`.

- 新松机器人搬运应用（SX322011）中，提示代码 **7252** 对应执行 **电磁阀关闭快捷功能**，参数值为 **0**；其余两列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 872–878; corpus 5.0, revision V5.0 / A-0; object `bd294a0232e7adb5e7922e626393bc0a465766c3d22aeb0f19fc62f05a1c8c46`.

- 该表格行（索引35）来自《新松机器人搬运应用操作手册》(A-0) V5.0 版本，描述了一条提示项：

*   **提示编号/内容**：提示 7251
*   **功能描述**：执行电磁阀打开快捷功能
*   **数值参数**：0
*   **其他列**：第4、5列为空

**摘要**：手册中提示 7251 用于执行“电磁阀打开”快捷功能，对应参数值为 0。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 865–871; corpus 4.0, revision V5.0 / A-0; object `36562bf66d8df3c47b1a4175fb8c6d26a6608f6555f8ef424f008b9198f72185`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

- 该表格行来自《新松机器人点焊应用操作手册》(SX322007)，描述了一条指令配置：步骤序号 **2** 对应事件 **焊接开始**，触发动作为 **输出IO**，参数值为 **1**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1380–1385; corpus 4.0, revision V5.0 / A-2; object `0b7f2934df81fae53b8121def6d49422f1846ac73e21e3c24f0f9b9b596ff997`.

#### Answer

根据检索到的手册证据，E3（V5.0/A-0）中列出了与电磁阀相关的报警码和提示，例如：报警7205（电磁阀组号超界，需赋值在1-8范围内）、报警7219（配置IO不在逻辑IO分配范围内）、提示7258（当前输出IO存在模拟状态，请注意）。然而，这些均为报警/提示信息，并非

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: unsupported; revision fidelity: mixed.

The question asks for the specific prerequisites or notes regarding 'Output' (输出) in the 'Solenoid Valve Configuration' (电磁阀配置) section of the V4.0/A-0 manual. The expected facts are: 1) Output corresponds to cylinder open/close and user IO configuration; 2) Manual forced output requires power-on + deadman enable. The supplied evidence (E1-E8) does not contain these specific facts. E3 contains alarm codes related to solenoid valves (e.g., 7205, 7219) and a prompt about output IO simulation (7258), but it does not state the definition of 'Output' in the configuration context nor the condition for manual forced output (power-on + deadman). The answer provided is a fragment that lists these alarm codes but fails to answer the specific question about the 'Output' configuration prerequisites. It also incorrectly attributes the evidence to V5.0/A-0 (E3 is V5.0) while the question asks for V4.0/A-0, although the content mismatch is the primary issue. Since the evidence lacks the required facts, a correct response would have been an abstention or a statement that the specific configuration details for 'Output' were not found in the provided snippets, rather than listing unrelated alarm codes as if they were the answer.

...

#### Answer

The retrieved evidence (E1–E4) only contains the cover page, title, and revision-history table of SX322011《新松机器人搬运应用操作手册》(A-0) V5.0/A-0. None of the provided excerpts include the content of section '1.1 基础配置' or any menu path for '型号配置'. The specific navigation path (e.g., 系统设置 → 基础配置 → 型号配置) is not present in the supplied evidence, so a supported contextual example cannot be constructed.

### sample_material_handling_40fac4d4f7_20

根据《SX322011《新松机器人搬运应用操作手册》(A-0)》（V5.0 / A-0），“吸盘电磁阀指令集”中“吸盘指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 该表行为第1行（非表头），包含1个单元格（第1列），其内容为空，无合并单元格或继承值，属于文档“SX322011《新松机器人搬运应用操作手册》(A-0)”中的空白占位符行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 2–4; corpus 5.0, revision V5.0 / A-0; object `366ba655b5ab7028c11b2a194674067dba02791d2ade53683f4ecb4714b01910`.

- 该条目属于**吸盘指令**（继承自上方合并单元格），其**说明**列详细定义了5个参数：参数1为开关状态（VacOn/VacOff/AllVacOn/AllVacOff）；参数2为抓手编号（1-8）；参数3为吸盘阀片组号（1-8，支持通过注释列表查看名称）；参数4为检查状态（Check/NoCheck）；参数5为实时监控状态（NoMotion/Motion）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 449–452; corpus 4.0, revision V5.0 / A-0; object `993b4bee261b049b133e5cc67555ef55b12ad57844c9cd182a55d15c4de6619e`.

- 该表行（第14行）描述了一条机器人控制指令：

*   **指令代码**：`A03_MH Vacuum AllVacOn GripNo=1 Check Motion`
*   **功能说明**：检测1号夹爪（GripNo=1）的吸盘吸气是否到位，并执行相关运动检查。

此条目来源于《新松机器人搬运应用操作手册》V5.0版本。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 582–585; corpus 5.0, revision V5.0 / A-0; object `5f69f71900029325bd617ca0644cacc3298df023368a301fbe84d6bf3c7f080c`.

- 该行描述**吸盘指令**的**参数说明**：参数1控制开/关（VacOn/VacOff/AllVacOn/AllVacOff）；参数2指定抓手编号（1–8）；参数3指定吸盘阀片组号（1–8，附注释列表可查名称）；参数4选择是否检查状态（Check/NoCheck）；参数5控制实时监控开/关（NoMotion/Motion）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 449–452; corpus 5.0, revision V5.0 / A-0; object `579f45ac43434ffc0004214b3c4e59023f0ba6b8059ffa2b1c0cff032f9313d0`.

- **吸盘指令**的功能描述为：实现吸盘控制（打开/关闭）、检测/不检测切换及实时检测状态。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 440–444; corpus 4.0, revision V5.0 / A-0; object `80668275dd61b6cbfbecd292f2a54502b47fe9a80c2eebbba7042c50f07c50fc`.

- 该行属于**吸盘指令**类别，展示了 `VacOn`、`VacOff`、`AllVacOn` 和 `AllVacOff` 四种真空控制指令的**格式**。指令结构均包含动作类型、参数1、抓手编号（GripNo）、真空编号（VacNo，仅单路指令包含）、检查模式（Check/NoCheck）及运动模式（Motion），并支持附加注释。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 445–448; corpus 4.0, revision V5.0 / A-0; object `32d4b8077c03be7553d7844d6827079c77bfc134d0b0a088fba9540344704bf8`.

- 新松机器人搬运应用操作手册（V5.0）中记录的一条运动指令：`MOVJ P1 VJ=100 ACC 100 CNT 0`（第9行，首列）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 562–565; corpus 5.0, revision V5.0 / A-0; object `b8fa6be8992556ebac9779471c91ac0e151bcd113c4e3c41bdc47a9d2832f780`.

- 该条目属于**吸盘指令**部分（继承自合并单元格），提供了**举例**说明。内容展示了5条`A03_MH Vacuum`指令的语法及功能，分别控制抓手1或2的特定吸盘或全部吸盘执行**开启（VacOn/AllVacOn）**或**关闭（VacOff/AllVacOff）**操作，并包含**检测状态**（Check/NoCheck）与**实时监控状态**（Motion/NoMotion）的参数配置。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人搬运应用操作手册》(A-0)/SX322011《新松机器人搬运应用操作手册》(A-0).md`, lines 453–456; corpus 5.0, revision V5.0 / A-0; object `69a52918f43a7d5f2f4e24f48a3e8c9057f8fe8fb8e398c7e561d39c5bb3421b`.

#### Answer

根据《SX322011《新松机器人搬运应用操作手册》(A-0)》V5.0/A-0，

### Example — material_handling

在搬运程序中，机器人抓手1的吸盘1组需要对工件执行吸气吸附操作，同时启用到位检测和实时监控。

**Environment and prerequisites:** 机器人已配置A03_MH物料搬运模块；抓手1（GripNo=1）已安装吸盘；吸盘电磁阀阀片组1（VacNo=1）已接线；工件位于抓取位上方；程序处于执行状态。

**Expected result:** 执行VacOn指令后，抓手1的吸盘1组电磁阀打开，产生负压吸附工件；Check参数使系统检测吸盘是否到位（负压是否达标）；Motion参数开启实时监控，若吸附过程中负压异常则触发报警。

**Manual:** SX322011《新松机器人搬运应用操作手册》(A-0) V5.0/A-0 [E2] [E5] [E6] [E8] [E3]

```text
A03_MH PartPresent NoPart GripNo=1 AllType NoMotion
A03_MH Vacuum VacOn GripNo=1 VacNo =1 Check Motion ;注释内容
A03_MH PartPresent WithPart GripNo=1 AllType Motion
```

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: partial; revision fidelity: faithful.

The answer fails to directly answer the question about the function, syntax, and parameters of the '吸盘指令' (Vacuum Command) as defined in the manual. Instead, it provides a fabricated scenario ('Example — material_handling') with specific prerequisites and physical behaviors (e.g., 'negative pressure', 'alarm triggering') that are not present in the provided evidence. The evidence [E2], [E5], [E6], and [E8] clearly defines the function as '吸盘控制，打开/关闭，检测/不检测，实时检测打开/关闭' and lists the syntax and parameter meanings (e.g., Check/NoCheck, Motion/NoMotion). The answer omits the explicit definition of the command's function and syntax requested by the user, replacing it with inferred operational details that are unsupported by the source text. While it cites the correct manual, the content is largely hallucinated context rather than a faithful extraction of the manual's definitions.

## Adhesive Dispensing — sampled manual revisions

40 cases from 2 CSV files. Set `ADHESIVE_DISPENSING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
ADHESIVE_DISPENSING_SAMPLE_CASES = [{'id': 'sample_adhesive_dispensing_663588ace7_01', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“指令类型异常”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2 指令类型异常 联系技术人员'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1452, 'line_end': 1456, 'excerpt': '<tr>\n<td><p>2</p></td>\n<td><p>指令类型异常</p></td>\n<td><p>联系技术人员</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_02', 'category': 'adhesive_dispensing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），操作“快捷键”时，涉及“开关枪”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['开关枪：实现手动快捷开胶枪、关胶枪', '出胶前提条件要满足，不满足时会有提示', '操作过程中出现异常在状态信息条可显示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 557, 'line_end': 557, 'excerpt': '开关枪：实现手动快捷开胶枪、关胶枪。出胶前提条件要满足，不满足时会有提示。操作过程中出现异常在状态信息条可显示。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_03', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“涂胶加热”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['涂胶加热：强制加热', '可在机器人状态信息条中显示提示，包括过程中的异常'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 551, 'line_end': 551, 'excerpt': '涂胶加热：强制加热。可在机器人状态信息条中显示提示，包括过程中的异常。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_04', 'category': 'adhesive_dispensing', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”列出的报警7103表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7103'], 'expected_facts': ['7103 指令异常 0 胶机系统号参数错误 请检查指令中胶机系统号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1390, 'line_end': 1396, 'excerpt': '<tr>\n<td><p>7103</p></td>\n<td><p>指令异常</p></td>\n<td><p>0</p></td>\n<td><p>胶机系统号参数错误</p></td>\n<td><p>请检查指令中胶机系统号</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_05', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用类型配置', 'heading_path': ['1 基础配置', '1.2 应用类型配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“应用类型配置”中，应如何理解或操作“选择涂胶应用”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['选择涂胶应用，保存并重启'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 178, 'line_end': 178, 'excerpt': '选择涂胶应用，保存并重启。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_06', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 涂胶参数', 'heading_path': ['1 基础配置', '1.4 胶机系统', '1.4.3 涂胶参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据手册“SX322012《新松机器人涂胶功能使用手册》-A-0.md”（V4.0 / A-0），要在“1.4.3 涂胶参数”中配置“涂胶参数”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：涂胶->胶机系统->涂胶参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 262, 'line_end': 262, 'excerpt': '配置路径：涂胶->胶机系统->涂胶参数；', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_07', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“胶机系统2模式已切换至手动模式！请检查”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1 胶机系统2模式已切换至手动模式！请检查 请检查胶机系统模式'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1348, 'line_end': 1352, 'excerpt': '<tr>\n<td><p>1</p></td>\n<td><p>胶机系统2模式已切换至手动模式！请检查</p></td>\n<td><p>请检查胶机系统模式</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_08', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1.4', 'section_title': '1.1.4 涂胶轨迹GLUE作业模板', 'heading_path': ['1 模板作业', '1.1 涂胶流程作业模板', '1.1.4 涂胶轨迹GLUE作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“涂胶轨迹GLUE作业模板”中，应如何理解或操作“A08_GLUE GlueEnd Gun=1 Fill=No”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['A08_GLUE GlueEnd Gun=1 Fill=No 循环结束'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 819, 'line_end': 822, 'excerpt': '<tr>\n<td><p>A08&#95;GLUE GlueEnd Gun=1 Fill=No</p></td>\n<td><p>循环结束</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_09', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2.5', 'section_title': '1.2.5 定量机冲洗指令集GLUEPURGE', 'heading_path': ['1 模板作业', '1.2 涂胶指令集作业模板', '1.2.5 定量机冲洗指令集GLUEPURGE'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“定量机冲洗指令集GLUEPURGE”中“IF   IN#42=ON GOTO L112”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['IN#42', 'GOTO'], 'expected_facts': ['IF IN#42=ON GOTO L112 判断是否有冲洗请求'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1309, 'line_end': 1312, 'excerpt': '<tr>\n<td><p>IF   IN&#35;42=ON GOTO L112</p></td>\n<td><p>判断是否有冲洗请求</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_10', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.2', 'section_title': '2.2.2 指令集格式及参数说明', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.2 指令集说明', '2.2.2 指令集格式及参数说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“指令集格式及参数说明”中“定量机填充指令集”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['定量机填充指令集 功能 定量机填充'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 504, 'line_end': 508, 'excerpt': '<tr>\n<td rowspan="4"><p>定量机填充指令集</p></td>\n<td><p>功能</p></td>\n<td><p>定量机填充</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_11', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 涂胶参数', 'heading_path': ['1 基础配置', '1.4 胶机系统', '1.4.3 涂胶参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“涂胶参数”中“功能”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GLUEON'], 'expected_facts': ['功能：涂胶参数是后续涂胶效果呈现的一个重要参数，以文件号的形式，作为涂胶指令GLUEON的一个传入参数', '该界面可配置涂胶方式，胶条宽度，机器人最大速度', '最大涂胶量根据胶条宽度与机器人最大速度可自动生成'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 268, 'line_end': 268, 'excerpt': '功能：涂胶参数是后续涂胶效果呈现的一个重要参数，以文件号的形式，作为涂胶指令GLUEON的一个传入参数。该界面可配置涂胶方式，胶条宽度，机器人最大速度。最大涂胶量根据胶条宽度与机器人最大速度可自动生成。系数K、B值。每个胶机系统最大支持8个文件号。具体配置项含义如下：', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_12', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“切换指令参数范围异常”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1 切换指令参数范围异常 检查指令中参数范围是否有误'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1447, 'line_end': 1451, 'excerpt': '<tr>\n<td><p>1</p></td>\n<td><p>切换指令参数范围异常</p></td>\n<td><p>检查指令中参数范围是否有误</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_13', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“硬按键OP2-OP5”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 硬按键OP2-OP5：可配置其中，硬按键OP2-OP5快捷键为可配置，配置项可通过下拉框选择（如下图所示）包括涂胶加热、泵机启动、涂胶清洗、开关胶、涂胶测量、涂胶填充6个功能项', '软按键1与软按键2为被动配置，由于各快捷键功能项（包括软按键）不会重复，当软按键1与软按键2的功能项被硬按键OP2-OP5选择时，软按键自动替换原硬按键的功能项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 549, 'line_end': 549, 'excerpt': '2. 硬按键OP2-OP5：可配置其中，硬按键OP2-OP5快捷键为可配置，配置项可通过下拉框选择（如下图所示）包括涂胶加热、泵机启动、涂胶清洗、开关胶、涂胶测量、涂胶填充6个功能项。软按键1与软按键2为被动配置，由于各快捷键功能项（包括软按键）不会重复，当软按键1与软按键2的功能项被硬按键OP2-OP5选择时，软按键自动替换原硬按键的功能项。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_14', 'category': 'adhesive_dispensing', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”列出的报警7109表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7109'], 'expected_facts': ['7109 涂胶结束指令异常 0 指令类型异常 联系技术人员'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1462, 'line_end': 1468, 'excerpt': '<tr>\n<td rowspan="2"><p>7109</p></td>\n<td rowspan="2"><p>涂胶结束指令异常</p></td>\n<td><p>0</p></td>\n<td><p>指令类型异常</p></td>\n<td><p>联系技术人员</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_15', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用类型配置', 'heading_path': ['1 基础配置', '1.2 应用类型配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“应用类型配置”中，应如何理解或操作“输入IO分配”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['输入IO分配：IO起始位配置为33，长度为224'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 184, 'line_end': 184, 'excerpt': '输入IO分配：IO起始位配置为33，长度为224。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_16', 'category': 'adhesive_dispensing', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.6', 'section_title': '1.4.6 断点恢复', 'heading_path': ['1 基础配置', '1.4 胶机系统', '1.4.6 断点恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），配置“断点恢复”时，“功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['功能：涂胶过程中打断时，机器人在关胶后继续行驶了一段距离，为保证断点恢复时，不会出现断点处缺胶情况，需回退一段距离，保证涂胶轨迹均有胶量涂敷', '默认回退距离范围为0-20mm'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 311, 'line_end': 311, 'excerpt': '功能：涂胶过程中打断时，机器人在关胶后继续行驶了一段距离，为保证断点恢复时，不会出现断点处缺胶情况，需回退一段距离，保证涂胶轨迹均有胶量涂敷。默认回退距离范围为0-20mm。', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_17', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“胶机系统3模式已切换至手动模式！请检查”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2 胶机系统3模式已切换至手动模式！请检查 请检查胶机系统模式'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1353, 'line_end': 1357, 'excerpt': '<tr>\n<td><p>2</p></td>\n<td><p>胶机系统3模式已切换至手动模式！请检查</p></td>\n<td><p>请检查胶机系统模式</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_18', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1.3', 'section_title': '1.1.3 胶机准备阶段DO\\_TOOLNEED作业模板', 'heading_path': ['1 模板作业', '1.1 涂胶流程作业模板', '1.1.3 胶机准备阶段DO\\_TOOLNEED作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），在“胶机准备阶段DO\\_TOOLNEED作业模板”中，应如何理解或操作“PULSE   OT#37 =ON T=0.1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['PULSE OT#37 =ON T=0.1 填充脉冲信号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 742, 'line_end': 745, 'excerpt': '<tr>\n<td><p>PULSE   OT&#35;37 =ON T=0&#46;1</p></td>\n<td><p>填充脉冲信号</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_19', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2.4', 'section_title': '1.2.4 定量机填充指令集GLUEFILL', 'heading_path': ['1 模板作业', '1.2 涂胶指令集作业模板', '1.2.4 定量机填充指令集GLUEFILL'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“定量机填充指令集GLUEFILL”中“== 2.0 GOTO L116”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GOTO', 'L116'], 'expected_facts': ['== 2.0 GOTO L116'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 1104, 'line_end': 1106, 'excerpt': '<tr>\n<td><p>   == 2&#46;0 GOTO L116</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_663588ace7_20', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.2', 'section_title': '2.2.2 指令集格式及参数说明', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.2 指令集说明', '2.2.2 指令集格式及参数说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“指令集格式及参数说明”中“<参数1>：功能定义； <参数2>：定量机号； 存外部实型变量491 ； <参数3>：程序号；”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['说明 <参数1>：功能定义', '<参数2>：定量机号', '存外部实型变量491'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'line_start': 445, 'line_end': 448, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；<strong>存外部实型变量491</strong>；</p><p>&lt;参数3&gt;：程序号；<strong>存外部实型变量492</strong>；</p><p>&lt;参数4&gt;：参数号；<strong>存外部实型变量493</strong>；</p><p>&lt;参数5&gt;：冲洗（1不冲洗/2强制冲洗/3根据是否有请求冲洗）；<strong>存外部实型变量494；</strong></p><p>&lt;参数6&gt;：填充（1不填充/2强制填充/3根据是否有请求填充）；<strong>存外部实型变量495；</strong></p><p>&lt;参数7&gt;：注释；</p></td>\n</tr>', 'sha256': '4e21d7a71c5c3ba7fabecc7275bbe90e7643eae9b4fbc1afa29d599f63f645d4'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_01', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4.3', 'section_title': '1.4.3 涂胶参数', 'heading_path': ['1 涂胶应用配置', '1.4 胶机系统', '1.4.3 涂胶参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“涂胶参数”中，应如何理解或操作“胶条宽度”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 胶条宽度：设置需要涂敷的胶条宽度（胶条直径），该配置需要与涂胶机上所配置出胶胶条宽度一致'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 190, 'line_end': 190, 'excerpt': '2. 胶条宽度：设置需要涂敷的胶条宽度（胶条直径），该配置需要与涂胶机上所配置出胶胶条宽度一致。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_02', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.1.1', 'section_title': '2.1.1 指令选择', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.1 指令说明', '2.1.1 指令选择'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“指令选择”中“可选择对应单选项”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['可选择对应单选项，也可点击标准动作，修改上图4条标准动作', '具体如下所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 272, 'line_end': 272, 'excerpt': '可选择对应单选项，也可点击标准动作，修改上图4条标准动作。具体如下所示：', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_03', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4.3', 'section_title': '1.4.3 涂胶参数', 'heading_path': ['1 涂胶应用配置', '1.4 胶机系统', '1.4.3 涂胶参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“涂胶参数”中“功能”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GLUEON'], 'expected_facts': ['功能：涂胶参数是后续涂胶效果呈现的一个重要参数，以文件号的形式，作为涂胶指令GLUEON的一个传入参数', '该界面可配置涂胶方式，胶条宽度，机器人最大速度', '最大涂胶量根据胶条宽度与机器人最大速度可自动生成'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 187, 'line_end': 187, 'excerpt': '功能：涂胶参数是后续涂胶效果呈现的一个重要参数，以文件号的形式，作为涂胶指令GLUEON的一个传入参数。该界面可配置涂胶方式，胶条宽度，机器人最大速度。最大涂胶量根据胶条宽度与机器人最大速度可自动生成。系数K、B值。每个胶机系统最大支持8个文件号。具体配置项含义如下：', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_04', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 涂胶指令集作业模板', 'heading_path': ['1 模板作业', '1.2 涂胶指令集作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“涂胶指令集作业模板”中“定量机填充指令集GLUEFILL”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GLUEFILL'], 'expected_facts': ['1. 定量机填充指令集GLUEFILL'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 369, 'line_end': 369, 'excerpt': '1. **定量机填充指令集GLUEFILL**', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_05', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2', 'section_title': '2 常见报警及解决方法', 'heading_path': ['2 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“常见报警及解决方法”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['rpy_to_trans'], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法 7101 涂胶机反馈异常 0 胶机系统1模式已切换至手动模式！请检查 请检查胶机系统模式 1 胶机系统2模式已切换至手动模式！请检查 请检查胶机系统模式 2 胶机系统3模式已切换至手动模式！请检查 请检查胶机系统模式 3 胶机系统4模式已切换至手动模式！请检查 请检查胶机系统模式 4 胶机系统5模式已切换至手动模式！请检查 请检查胶机系统模式 5 胶机系统6模式已切换至手动模式！请检查 请检查胶机系统模式 6 胶机系统7模式已切换至手动模式！请检查 请检查胶机系统模式 7 胶机系统8模式已切换至手动模式！请检查 请检查胶机系统模式 7102 获取指令文件号异常 0 胶机参数文件号异常 请检查指令中参数文件号 7103 指令异常 0 胶机系统号参数错误 请检查指令中胶机系统号 7104 数据采集异常 请将整型变量21的值置1，并将作业中设置整型变量20置1的指令删除！ 7105 当前作业名出错 联系技术人员 7106 指令类型出错 联系技术人员 7107 涂胶点位写入出错 0 rpy_to_trans转换出错 联系技术人员 1 点位坐标系类型不支持 联系技术人员 2 点位存储类型不存在 联系技术人员 3 运动类型不支持 联系技术人员 7108 涂胶开始指令异常 0 指令参数范围异常 检查指令中参数范围是否有误 1 切换指令参数范围异常 检查指令中参数范围是否有误 2 指令类型异常 联系技术人员 3 状态机状态异常 联系技术人员 7109 涂胶结束指令异常 0 指令类型异常 联系技术人员 1 状态异常 联系技术人员 7110 初始化指令异常 0 胶机系统号参数错误 检查胶机系统参数号是否有误 7111 前瞻过程中机器人暂停或出错 联系技术人员 7112 涂胶线程注册redis句柄出错 联系技术人员 7113 涂胶应用相关逻辑IO未分配 请在IO->逻辑IO分配菜单下配置相关输入输出IO 7114 涂胶应用逻辑IO分配地址与涂胶应用配置中不匹配 检查应用中IO设置的范围与逻辑IO分配范围包含关系 7115 机器人干涉 请检查机器人是否存在干涉 7116 压缩空气异常 请检查气单元是否异常 7117 通讯状态异常 请检查胶机通讯状态或相关线缆 7118 文件写失败 联系技术人员 7119 文件读失败 联系技术人员 7120 离线运动失败 联系技术人员 7121 涂胶响应安全门暂停 请检查安全门状态'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 379, 'line_end': 379, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码报警信息</p></td><td><p>解决方法</p></td></tr><tr><td rowspan="8"><p>7101</p></td><td rowspan="8"><p>涂胶机反馈异常</p></td><td><p>0</p></td><td><p>胶机系统1模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>1</p></td><td><p>胶机系统2模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>2</p></td><td><p>胶机系统3模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>3</p></td><td><p>胶机系统4模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>4</p></td><td><p>胶机系统5模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>5</p></td><td><p>胶机系统6模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>6</p></td><td><p>胶机系统7模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>7</p></td><td><p>胶机系统8模式已切换至手动模式！请检查</p></td><td><p>请检查胶机系统模式</p></td></tr><tr><td><p>7102</p></td><td><p>获取指令文件号异常</p></td><td><p>0</p></td><td><p>胶机参数文件号异常</p></td><td><p>请检查指令中参数文件号</p></td></tr><tr><td><p>7103</p></td><td><p>指令异常</p></td><td><p>0</p></td><td><p>胶机系统号参数错误</p></td><td><p>请检查指令中胶机系统号</p></td></tr><tr><td><p>7104</p></td><td><p>数据采集异常</p></td><td></td><td></td><td><p>请将整型变量21的值置1，并将作业中设置整型变量20置1的指令删除！</p></td></tr><tr><td><p>7105</p></td><td><p>当前作业名出错</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7106</p></td><td><p>指令类型出错</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td rowspan="4"><p>7107</p></td><td rowspan="4"><p>涂胶点位写入出错</p></td><td><p>0</p></td><td><p>rpy_to_trans转换出错</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>1</p></td><td><p>点位坐标系类型不支持</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>2</p></td><td><p>点位存储类型不存在</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>3</p></td><td><p>运动类型不支持</p></td><td><p>联系技术人员</p></td></tr><tr><td rowspan="4"><p>7108</p></td><td rowspan="4"><p>涂胶开始指令异常</p></td><td><p>0</p></td><td><p>指令参数范围异常</p></td><td><p>检查指令中参数范围是否有误</p></td></tr><tr><td><p>1</p></td><td><p>切换指令参数范围异常</p></td><td><p>检查指令中参数范围是否有误</p></td></tr><tr><td><p>2</p></td><td><p>指令类型异常</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>3</p></td><td><p>状态机状态异常</p></td><td><p>联系技术人员</p></td></tr><tr><td rowspan="2"><p>7109</p></td><td rowspan="2"><p>涂胶结束指令异常</p></td><td><p>0</p></td><td><p>指令类型异常</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>1</p></td><td><p>状态异常</p></td><td><p>联系技术人员</p></td></tr><tr><td><p>7110</p></td><td><p>初始化指令异常</p></td><td><p>0</p></td><td><p>胶机系统号参数错误</p></td><td><p>检查胶机系统参数号是否有误</p></td></tr><tr><td><p>7111</p></td><td><p>前瞻过程中机器人暂停或出错</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7112</p></td><td><p>涂胶线程注册redis句柄出错</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7113</p></td><td><p>涂胶应用相关逻辑IO未分配</p></td><td></td><td></td><td><p>请在IO-&gt;逻辑IO分配菜单下配置相关输入输出IO</p></td></tr><tr><td><p>7114</p></td><td><p>涂胶应用逻辑IO分配地址与涂胶应用配置中不匹配</p></td><td></td><td></td><td><p>检查应用中IO设置的范围与逻辑IO分配范围包含关系</p></td></tr><tr><td><p>7115</p></td><td><p>机器人干涉</p></td><td></td><td></td><td><p>请检查机器人是否存在干涉</p></td></tr><tr><td><p>7116</p></td><td><p>压缩空气异常</p></td><td></td><td></td><td><p>请检查气单元是否异常</p></td></tr><tr><td><p>7117</p></td><td><p>通讯状态异常</p></td><td></td><td></td><td><p>请检查胶机通讯状态或相关线缆</p></td></tr><tr><td><p>7118</p></td><td><p>文件写失败</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7119</p></td><td><p>文件读失败</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7120</p></td><td><p>离线运动失败</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>7121</p></td><td><p>涂胶响应安全门暂停</p></td><td></td><td></td><td><p>请检查安全门状态</p></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_06', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“PLC-SIM”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 329, 'line_end': 329, 'excerpt': '5. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_07', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4.3', 'section_title': '1.4.3 涂胶参数', 'heading_path': ['1 涂胶应用配置', '1.4 胶机系统', '1.4.3 涂胶参数'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322012《新松机器人涂胶应用操作手册》(A-1).md”（V5.0 / A-1），要在“1.4.3 涂胶参数”中配置“涂胶参数”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：涂胶->胶机系统->涂胶参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 181, 'line_end': 181, 'excerpt': '配置路径：涂胶->胶机系统->涂胶参数；', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_08', 'category': 'adhesive_dispensing', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），操作“快捷键”时，涉及“开关枪”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['开关枪：实现手动快捷开胶枪、关胶枪', '出胶前提条件要满足，不满足时会有提示', '操作过程中出现异常在状态信息条可显示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 323, 'line_end': 323, 'excerpt': '开关枪：实现手动快捷开胶枪、关胶枪。出胶前提条件要满足，不满足时会有提示。操作过程中出现异常在状态信息条可显示。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_09', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 涂胶流程作业模板', 'heading_path': ['1 模板作业', '1.1 涂胶流程作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322012《新松机器人涂胶应用操作手册》(A-1).md”（V5.0 / A-1），涂胶流程模板怎样选择胶机系统、初始化胶枪，并完成沿轨迹开胶、关胶和循环结束？', 'anchors': ['GlueSetSysNum', 'GlueInit', 'GLUEON', 'GLUEOFF', 'GlueEnd'], 'expected_facts': ['A08_GLUE GlueSetSysNum SysNum=1 设置涂胶系统', 'A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes 涂胶初始化', 'GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo =1 TIME =20 开胶', 'GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEOFF TIME=20 关胶', 'A08_GLUE GlueEnd Gun=1 Fill=No 循环结束'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 353, 'line_end': 353, 'excerpt': '<table><tr><td><p>NOP   </p></td><td></td></tr><tr><td><p>SET UF   #0</p></td><td><p>用户坐标系选择</p></td></tr><tr><td><p>SET TF   #1</p></td><td><p>工具坐标系选择</p></td></tr><tr><td><p>PAYLOAD   #1</p></td><td><p>负载-抓手加车架</p></td></tr><tr><td><p>MOVJ\tPR102 VJ=30.0 ACC=30 CNT=0</p></td><td><p>冲锋位</p></td></tr><tr><td><p>MOVJ\tP1 VJ=30.0 ACC=100 CNT=1</p></td><td></td></tr><tr><td><p>;</p></td><td></td></tr><tr><td><p>A08_GLUE GlueSetSysNum SysNum=1</p></td><td><p>设置涂胶系统</p></td></tr><tr><td><p>A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes</p></td><td><p>涂胶初始化</p></td></tr><tr><td><p>MOVL\tP4 VL=1000.0 ACC=100 CNT=30 ET=1</p></td><td></td></tr><tr><td><p>GLUEL  P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo =1 TIME =20 </p></td><td><p>开胶</p></td></tr><tr><td><p>GLUEL  P1 VL=300 ACC=100 CNT=10 ET=1 GLUEOFF TIME=20</p></td><td><p>关胶</p></td></tr><tr><td><p>MOVL\tP4 VL=1000.0 ACC=100 CNT=30 ET=1</p></td><td></td></tr><tr><td><p>A08_GLUE GlueEnd Gun=1 Fill=No</p></td><td><p>循环结束</p></td></tr><tr><td><p>A08_GLUE GlueFillCheck Gun=1 Check</p></td><td><p>填充检查（SCA胶机循环结束后回自动填充）</p></td></tr><tr><td><p>MOVJ\tP2 VJ=30.0 ACC=100 CNT=1</p></td><td></td></tr><tr><td><p>MOVJ\tPR102 VJ=30.0 ACC=30 CNT=0</p></td><td><p>回冲锋位</p></td></tr><tr><td><p>RET</p></td><td></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_10', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4.1', 'section_title': '1.4.1 胶机品牌', 'heading_path': ['1 涂胶应用配置', '1.4 胶机系统', '1.4.1 胶机品牌'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“胶机品牌”中，应如何理解或操作“胶机品牌”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 胶机品牌：根据现场实际涂胶机品牌设置，当前程序适配的胶机包括INTEC、SCA、曦创以及通用协议'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 148, 'line_end': 148, 'excerpt': '1. 胶机品牌：根据现场实际涂胶机品牌设置，当前程序适配的胶机包括INTEC、SCA、曦创以及通用协议。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_11', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.1.2', 'section_title': '2.1.2 指令格式及参数说明', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.1 指令说明', '2.1.2 指令格式及参数说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“指令格式及参数说明”中“开胶枪指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GLUEL', 'ACC'], 'expected_facts': ['开胶枪指令 功能 开胶枪 格式 GLUEL P VL= ACC= CNT= ET= INTERLOCK= GLUEON GUN= CALIB= PARANo = TIME/DIS= 说明 <参数1>：点位', '<参数2>：速度', '<参数3>：加速度'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 284, 'line_end': 284, 'excerpt': '<table><tr><td rowspan="4"><p>开胶枪指令</p></td><td><p>功能</p></td><td><p>开胶枪</p></td></tr><tr><td><p>格式</p></td><td><p>GLUEL  P  VL=  ACC=  CNT=  ET=  INTERLOCK=  GLUEON  GUN=  CALIB=  PARANo =  TIME/DIS= </p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：点位；</p><p>&lt;参数2&gt;：速度；</p><p>&lt;参数3&gt;：加速度；</p><p>&lt;参数4&gt;：逼近值；</p><p>&lt;参数5&gt;：外部工具；</p><p>&lt;参数6&gt;：内部锁定；</p><p>&lt;参数7&gt;：功能定义；</p><p>&lt;参数8&gt;：定量机；</p><p>&lt;参数9&gt;：涂胶参数文件号；</p><p>&lt;参数10&gt;：参数号（发给胶机的）；</p><p>&lt;参数11&gt;：提前开胶时间/距离（负值提前，正值滞后）；</p></td></tr><tr><td><p>举例</p></td><td><p>GLUEL  P1 VL=300 ACC=100 CNT=10 ET=1 INTERLOCK=1 GLUEON GUN=1 CALIB=1 PARANo =1 TIME =20 </p></td></tr><tr><td rowspan="4"><p>关胶枪指令</p></td><td><p>功能</p></td><td><p>关胶枪</p></td></tr><tr><td><p>格式</p></td><td><p>GLUEL  P  VL=  ACC=  CNT=  ET=  INTERLOCK=  GLUEOFF  TIME/DIS=</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：点位；</p><p>&lt;参数2&gt;：速度；</p><p>&lt;参数3&gt;：加速度；</p><p>&lt;参数4&gt;：逼近值；</p><p>&lt;参数5&gt;：外部工具；</p><p>&lt;参数6&gt;：内部锁定；</p><p>&lt;参数7&gt;：功能定义；</p><p>&lt;参数8&gt;：提前关胶时间/距离（负值提前，正值滞后）；</p></td></tr><tr><td><p>举例</p></td><td><p>GLUEL  P1 VL=300 ACC=100 CNT=10 ET=1 INTERLOCK=1 GLUEOFF TIME=20</p></td></tr><tr><td rowspan="4"><p>切换参数指令</p></td><td><p>功能</p></td><td><p>切换参数</p></td></tr><tr><td><p>格式</p></td><td><p>GLUEL  P  VL=  ACC=  CNT=  ET=  GLUESWITCH  CALIB= </p><p>PARANo =  TIME/DIS=</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：点位；</p><p>&lt;参数2&gt;：速度；</p><p>&lt;参数3&gt;：加速度；</p><p>&lt;参数4&gt;：逼近值；</p><p>&lt;参数5&gt;：外部工具；</p><p>&lt;参数6&gt;：内部锁定；</p><p>&lt;参数7&gt;：功能定义；</p><p>&lt;参数8&gt;：涂胶参数文件号；</p><p>&lt;参数9&gt;：参数号（发给胶机的）；</p><p>&lt;参数10&gt;：提前切换参数时间/距离（负值提前，正值滞后）； </p></td></tr><tr><td><p>举例</p></td><td><p>GLUEL  P1 VL=300 ACC=100 CNT=10 ET=1 INTERLOCK=1 GLUESWITCH CALIB =1 PARANo =2  TIME =10</p></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_12', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 涂胶指令集作业模板', 'heading_path': ['1 模板作业', '1.2 涂胶指令集作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322012《新松机器人涂胶应用操作手册》(A-1).md”（V5.0 / A-1），胶机启动模板如何等待自动模式及加热器、泵就绪，并调用冲洗和填充作业？', 'anchors': ['IN#65', 'IN#69', 'IN#66', 'GLUEPURGE', 'GLUEFILL'], 'expected_facts': ['WAIT IN#65=ON T=-1.0 等待自动模式', 'WAIT IN#69=ON T=-1.0 等待加热器准备就绪', 'WAIT IN#66=ON T=-1.0 等待泵接通', 'CALL GLUEPURGE 调用冲洗指令集', 'CALL GLUEFILL 调用填充指令集'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 359, 'line_end': 359, 'excerpt': '<table><tr><td><p>NOP   </p></td><td></td></tr><tr><td><p>IF   IN#38=ON GOTO L101</p></td><td><p>判断胶机模式是否为模拟</p></td></tr><tr><td><p>IF   I498 &lt;&gt;0 GOTO L101</p></td><td><p>判断机器人模式（NORMAL，SIM，DRY）</p></td></tr><tr><td><p>WAIT   IN#65=ON T=-1.0</p></td><td><p>等待自动模式</p></td></tr><tr><td><p>OUT   OT#46=ON</p></td><td><p>系统接通信号置ON</p></td></tr><tr><td><p>OUT   OT#40=ON</p></td><td><p>运行许可信号置ON</p></td></tr><tr><td><p>PULSE   OT#47 =ON T=0.1</p></td><td><p>再触发加热器</p></td></tr><tr><td><p>WAIT   IN#72=ON T=-1.0</p></td><td><p>等待加热器/泵无故障</p></td></tr><tr><td><p>WAIT   IN#68=ON T=-1.0</p></td><td><p>等待加热器接通</p></td></tr><tr><td><p>WAIT   IN#69=ON T=-1.0</p></td><td><p>等待加热器准备就绪</p></td></tr><tr><td><p>PULSE   OT#48 =ON T=0.1</p></td><td><p>再触发泵</p></td></tr><tr><td><p>WAIT   IN#66=ON T=-1.0</p></td><td><p>等待泵接通</p></td></tr><tr><td><p>CALL   GLUEPURGE</p></td><td><p>调用冲洗指令集</p></td></tr><tr><td><p>CALL   GLUEFILL</p></td><td><p>调用填充指令集</p></td></tr><tr><td><p>OUT   OG#1 =I492</p></td><td><p>输出程序号至胶机</p></td></tr><tr><td><p>WAIT   IG#1 ==I492 T=-1.0</p></td><td><p>等待胶机反馈程序号结果</p></td></tr><tr><td><p>OUT   OG#2 =I493</p></td><td><p>输出参数号至胶机</p></td></tr><tr><td><p>WAIT   IG#2 ==I493 T=-1.0</p></td><td><p>等待胶机反馈参数号结果</p></td></tr><tr><td><p>OUT   OT#33=ON</p></td><td><p>启动</p></td></tr><tr><td><p>WAIT   IN#48=ON T=-1.0</p></td><td><p>等待涂胶准备就绪信号</p></td></tr><tr><td><p>   L101</p></td><td></td></tr><tr><td><p>RET   </p></td><td></td></tr><tr><td><p>END   </p></td><td></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_13', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“软按键1（涂胶测量）”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 软按键1（涂胶测量）：被动配置，脉冲信号发送，手动开关胶后或异常涂胶流程循环未进行测量时，需要手动进行测量', '操作过程中出现异常在状态信息条可显示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 326, 'line_end': 326, 'excerpt': '2. 软按键1（涂胶测量）：被动配置，脉冲信号发送，手动开关胶后或异常涂胶流程循环未进行测量时，需要手动进行测量。操作过程中出现异常在状态信息条可显示。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_14', 'category': 'adhesive_dispensing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 应用类型配置', 'heading_path': ['1 涂胶应用配置', '1.2 应用类型配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），配置“应用类型配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场涂胶应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 99, 'line_end': 99, 'excerpt': '逻辑IO分配：根据现场涂胶应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_15', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 涂胶流程作业模板', 'heading_path': ['1 模板作业', '1.1 涂胶流程作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322012《新松机器人涂胶应用操作手册》(A-1).md”（V5.0 / A-1），涂胶流程模板在冲洗或填充之前如何检查运行模式和胶桶状态，胶量不足时怎样处理？', 'anchors': ['IN#38', 'I498', 'IN#71', 'USERALM'], 'expected_facts': ['IF IN#38=ON GOTO L101 判断胶机模式是否为模拟', 'IF I498 <>0 GOTO L101 判断机器人模式（NORMAL，SIM，DRY）', 'IF IN#71=OFF GOTO L101 判断胶桶状态', 'USERALM CODE=2600 报警胶桶胶量不足10%'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 349, 'line_end': 349, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>IF   IN#38=ON GOTO L101</p></td><td><p>判断胶机模式是否为模拟</p></td></tr><tr><td><p>IF   I498 &lt;&gt;0 GOTO L101</p></td><td><p>判断机器人模式（NORMAL，SIM，DRY）</p></td></tr><tr><td><p>WAIT   IN#65=ON T=-1.0</p></td><td><p>等待自动模式</p></td></tr><tr><td><p>IF   IN#71=OFF GOTO L101</p></td><td><p>判断胶桶状态</p></td></tr><tr><td><p>USERALM   CODE=2600</p></td><td><p>报警胶桶胶量不足10%</p></td></tr><tr><td><p>OUT   OT#46=ON</p></td><td><p>系统接通信号置ON</p></td></tr><tr><td><p>OUT   OT#40=ON</p></td><td><p>运行许可信号置ON</p></td></tr><tr><td><p>PULSE   OT#47 =ON T=0.1</p></td><td><p>再触发加热器</p></td></tr><tr><td><p>WAIT   IN#72=ON T=-1.0</p></td><td><p>等待加热器/泵无故障</p></td></tr><tr><td><p>WAIT   IN#68=ON T=-1.0</p></td><td><p>等待加热器接通</p></td></tr><tr><td><p>WAIT   IN#69=ON T=-1.0</p></td><td><p>等待加热器准备就绪</p></td></tr><tr><td><p>PULSE   OT#48 =ON T=0.1</p></td><td><p>再触发泵</p></td></tr><tr><td><p>WAIT   IN#66=ON T=-1.0</p></td><td><p>等待泵接通</p></td></tr><tr><td><p>IF   IN#42=OFF GOTO L102</p></td><td><p>判断是否有请求冲洗</p></td></tr><tr><td><p>OUT   OT#33=OFF</p></td><td><p>启动信号置OFF</p></td></tr><tr><td><p>WAIT   IN#48=OFF T=-1.0</p></td><td><p>等待涂胶准备就绪信号为OFF</p></td></tr><tr><td><p>OUT   OT#43=ON</p></td><td><p>允许冲洗信号置ON</p></td></tr><tr><td><p>OUT   OT#41=ON</p></td><td><p>冲洗信号置ON</p></td></tr><tr><td><p>WAIT   IN#46=ON T=-1.0</p></td><td><p>等待冲洗结束</p></td></tr><tr><td><p>OUT   OT#43=OFF</p></td><td><p>允许冲洗信号置OFF</p></td></tr><tr><td><p>OUT   OT#41=OFF</p></td><td><p>冲洗信号置OFF</p></td></tr><tr><td><p>WAIT   IN#45=ON T=-1.0</p></td><td><p>等待定量机满</p></td></tr><tr><td><p>L102</p></td><td></td></tr><tr><td><p>IF   IN#45=OFF GOTO L103</p></td><td><p>判断定量机是否已满</p></td></tr><tr><td><p>OUT   OT#33=OFF</p></td><td><p>启动信号置OFF</p></td></tr><tr><td><p>WAIT   IN#48=OFF T=-1.0</p></td><td><p>等待涂胶准备就绪信号为OFF</p></td></tr><tr><td><p>PULSE   OT#37 =ON T=0.1</p></td><td><p>填充脉冲信号</p></td></tr><tr><td><p>WAIT   IN#45=ON T=-1.0</p></td><td><p>等待定量机已满</p></td></tr><tr><td><p>L103</p></td><td></td></tr><tr><td><p>RET   </p></td><td></td></tr><tr><td><p>END   </p></td><td></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_16', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4.1', 'section_title': '1.4.1 胶机品牌', 'heading_path': ['1 涂胶应用配置', '1.4 胶机系统', '1.4.1 胶机品牌'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“胶机品牌”中，应如何理解或操作“通讯时延”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 通讯时延：胶机与机器人控制系统存在通讯时延，该通讯时延会影响胶机实时响应机器人发送的涂胶模拟量对应速度', '通过设置该时延，机器人可前瞻或滞后发送规划速度，已达到出胶量与机器人速度匹配', '由于系统通讯周期为4ms，该值推荐设置为4的倍数，未满足4的倍数，遵循除4取整原则，即设置20ms与设置23ms，效果一致'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 149, 'line_end': 149, 'excerpt': '2. 通讯时延：胶机与机器人控制系统存在通讯时延，该通讯时延会影响胶机实时响应机器人发送的涂胶模拟量对应速度。通过设置该时延，机器人可前瞻或滞后发送规划速度，已达到出胶量与机器人速度匹配。由于系统通讯周期为4ms，该值推荐设置为4的倍数，未满足4的倍数，遵循除4取整原则，即设置20ms与设置23ms，效果一致。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_17', 'category': 'adhesive_dispensing', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2.2', 'section_title': '2.2.2 指令集格式及参数说明', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.2 指令集说明', '2.2.2 指令集格式及参数说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），“指令集格式及参数说明”中“涂胶初始化指令集”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A08_GLUE'], 'expected_facts': ['涂胶初始化指令集 功能 涂胶初始化 格式 A08_GLUE GlueInit Gun= ProgNo= ParaNo= Purge=No/Yes/Auto Fill=No/Yes/ Auto注释 说明 <参数1>：功能定义', '<参数2>：定量机号', '存外部实型变量491'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 304, 'line_end': 304, 'excerpt': '<table><tr><td rowspan="4"><p>涂胶初始化指令集</p></td><td><p>功能</p></td><td><p>涂胶初始化</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE GlueInit Gun= ProgNo= ParaNo= Purge=No/Yes/Auto  Fill=No/Yes/ Auto注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；<strong>存外部实型变量491</strong>；</p><p>&lt;参数3&gt;：程序号；<strong>存外部实型变量492</strong>；</p><p>&lt;参数4&gt;：参数号；<strong>存外部实型变量493</strong>；</p><p>&lt;参数5&gt;：冲洗（1不冲洗/2强制冲洗/3根据是否有请求冲洗）；<strong>存外部实型变量494；</strong></p><p>&lt;参数6&gt;：填充（1不填充/2强制填充/3根据是否有请求填充）；<strong>存外部实型变量495；</strong></p><p>&lt;参数7&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes 强制填充，冲洗</p></td></tr><tr><td rowspan="4"><p>涂胶结束指令集</p></td><td><p>功能</p></td><td><p>涂胶结束，进行测量并根据参数是否进行填充</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE  GlueEnd  Gun=  Fill=No/Yes/ Auto注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；<strong>存外部实型变量491；</strong></p><p>&lt;参数2&gt;：填充（1不填充/2强制填充/3根据是否有请求填充）；<strong>存外部实型变量495；</strong></p><p>&lt;参数3&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GlueEnd Gun=1 Fill=Yes 强制填充</p></td></tr><tr><td rowspan="4"><p>定量机是否填满检查指令集</p></td><td><p>功能</p></td><td><p>定量机是否填满检查</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE  GlueFillCheck  Gun=  NoCheck/Check 注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；<strong>存外部实型变量491；</strong></p><p>&lt;参数2&gt;：是否检查（1不检查/2检查）；<strong>存外部实型变量496；</strong></p><p>&lt;参数3&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GlueFillCheck Gun=1 Check 检查</p></td></tr><tr><td rowspan="4"><p>冲洗指令集</p></td><td><p>功能</p></td><td><p>冲洗指令集</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE  GluePurge  Gun=  Purge=No/Yes/ Auto注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；存外部实型变量491；</p><p>&lt;参数2&gt;：冲洗（1不冲洗/2强制冲洗/3根据是否有请求冲洗）<strong>存外部实型变量494；</strong></p><p>&lt;参数3&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GluePurge Gun=1  Purge=Yes 强制冲洗</p></td></tr><tr><td rowspan="4"><p>定量机填充指令集</p></td><td><p>功能</p></td><td><p>定量机填充</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE  GlueFill  Gun=  Fill=No/Yes/ Auto 注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：定量机号；存外部实型变量491；</p><p>&lt;参数3&gt;：填充（1不填充/2强制填充/3根据是否有请求填充）；<strong>存外部实型变量495；</strong></p><p>&lt;参数4&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GlueFill Gun=1 Fill=Yes 强制填充</p></td></tr><tr><td rowspan="4"><p>设置涂胶系统指令集</p></td><td><p>功能</p></td><td><p>设置涂胶系统</p></td></tr><tr><td><p>格式</p></td><td><p>A08_GLUE GlueSetSysNum  SysNum=  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：涂胶系统；<strong>存外部实型变量497；</strong></p><p>&lt;参数3&gt;：注释；</p></td></tr><tr><td><p>举例</p></td><td><p>A08_GLUE GlueSetSysNum SysNum=1涂胶系统</p></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_18', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 涂胶指令集作业模板', 'heading_path': ['1 模板作业', '1.2 涂胶指令集作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322012《新松机器人涂胶应用操作手册》(A-1).md”（V5.0 / A-1），冲洗作业模板怎样依据R494选择不排胶、强制排胶或根据请求排胶的分支？', 'anchors': ['SELECT', '== 2.0', '== 3.0', 'IN#42'], 'expected_facts': ['SELECT R494 == 1.0 GOTO L111 根据入参值判断是否冲洗（1不排胶，2强制排胶，3根据请求）', '== 2.0 GOTO L112', '== 3.0 GOTO L113', 'IF IN#42=ON GOTO L112 判断是否有冲洗请求'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 375, 'line_end': 375, 'excerpt': '<table><tr><td><p>NOP   </p></td><td></td></tr><tr><td><p>IF   IN#38=ON GOTO L101</p></td><td><p>判断胶机模式是否为模拟</p></td></tr><tr><td><p>IF   I498 &lt;&gt;0 GOTO L101</p></td><td><p>判断机器人模式（NORMAL，SIM，DRY）</p></td></tr><tr><td><p>   L114</p></td><td></td></tr><tr><td><p>SELECT   R494 == 1.0 GOTO L111</p></td><td rowspan="3"><p>根据入参值判断是否冲洗（1不排胶，2强制排胶，3根据请求）</p></td></tr><tr><td><p>   == 2.0 GOTO L112</p></td></tr><tr><td><p>   == 3.0 GOTO L113</p></td></tr><tr><td><p>PAUSE   </p></td><td></td></tr><tr><td><p>DELAY   T=0.1</p></td><td></td></tr><tr><td><p>GOTO   L114</p></td><td></td></tr><tr><td><p>   L112</p></td><td></td></tr><tr><td><p>OUT   OT#46=ON</p></td><td><p>系统接通信号置ON</p></td></tr><tr><td><p>OUT   OT#40=ON</p></td><td><p>运行许可信号置ON</p></td></tr><tr><td><p>WAIT   IN#69=ON T=-1.0</p></td><td><p>等待加热准备就绪</p></td></tr><tr><td><p>WAIT   IN#66=ON T=-1.0</p></td><td><p>等待泵准备就绪</p></td></tr><tr><td><p>WAIT   IN#72=ON T=-1.0</p></td><td><p>等待加热器\\泵无故障</p></td></tr><tr><td><p>WAIT   IN#33=ON T=-1.0</p></td><td><p>等待无常见故障</p></td></tr><tr><td><p>OUT   OT#33=OFF</p></td><td><p>启动信号置OFF</p></td></tr><tr><td><p>WAIT   IN#48=OFF T=-1.0</p></td><td><p>等待涂胶准备就绪信号</p></td></tr><tr><td><p>OUT   OT#43=ON</p></td><td><p>允许冲洗信号置ON</p></td></tr><tr><td><p>OUT   OT#41=ON</p></td><td><p>冲洗信号置ON</p></td></tr><tr><td><p>WAIT   IN#46=ON T=-1.0</p></td><td><p>等待冲洗结束信号</p></td></tr><tr><td><p>OUT   OT#43=OFF</p></td><td><p>允许冲洗信号置OFF</p></td></tr><tr><td><p>OUT   OT#41=OFF</p></td><td><p>冲洗信号置OFF</p></td></tr><tr><td><p>WAIT   IN#33=ON T=-1.0</p></td><td><p>等待无常见故障</p></td></tr><tr><td><p>GOTO   L111</p></td><td></td></tr><tr><td><p>   L113</p></td><td></td></tr><tr><td><p>IF   IN#42=ON GOTO L112</p></td><td><p>判断是否有冲洗请求</p></td></tr><tr><td><p>   L111</p></td><td></td></tr><tr><td><p>   L101</p></td><td></td></tr><tr><td><p>RET</p></td><td></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_19', 'category': 'adhesive_dispensing', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明以及作业流程', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“硬按键OP6、软按键”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 硬按键OP6、软按键： 工艺开\\工艺关-仿真\\工艺关-干运行三种模式切换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 325, 'line_end': 325, 'excerpt': '1. 硬按键OP6、软按键： 工艺开\\工艺关-仿真\\工艺关-干运行三种模式切换。', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}, {'id': 'sample_adhesive_dispensing_4a3c30c770_20', 'category': 'adhesive_dispensing', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.5', 'section_title': '1.5 实时显示', 'heading_path': ['1 涂胶应用配置', '1.5 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），配置“实时显示”时，“涂胶实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['涂胶实时显示功能，可查看当前涂胶应用相关参数，如模式状态、胶枪系统、程序号、子程序号等参数', '可在本地和远程等模式下进行查看，界面如下图1-15所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'line_start': 250, 'line_end': 250, 'excerpt': '涂胶实时显示功能，可查看当前涂胶应用相关参数，如模式状态、胶枪系统、程序号、子程序号等参数；可在本地和远程等模式下进行查看，界面如下图1-15所示：', 'sha256': 'eade5df7f78216de47e8d3d2be0f8e5b9dfa9161b23aab5600a1c5b7882309b0'}]}]
register_sample_bank('adhesive_dispensing', ADHESIVE_DISPENSING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md', 'seed': 3176737300409451659, 'population': 219, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(1452, 1456), (557, 557), (551, 551), (1390, 1396), (178, 178), (262, 262), (1348, 1352), (819, 822), (1309, 1312), (504, 508), (268, 268), (1447, 1451), (549, 549), (1462, 1468), (184, 184), (311, 311), (1353, 1357), (742, 745), (1104, 1106), (445, 448)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md', 'seed': 15901414279259908538, 'population': 92, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(190, 190), (272, 272), (187, 187), (369, 369), (379, 379), (329, 329), (181, 181), (323, 323), (353, 353), (148, 148), (284, 284), (359, 359), (326, 326), (99, 99), (349, 349), (149, 149), (304, 304), (375, 375), (325, 325), (250, 250)]}])


In [ ]:
# | notest
ADHESIVE_DISPENSING_SAMPLE_IDS = None
adhesive_dispensing_sample_cases = select_cases(ADHESIVE_DISPENSING_SAMPLE_CASES, ADHESIVE_DISPENSING_SAMPLE_IDS)
display(pd.DataFrame(adhesive_dispensing_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in adhesive_dispensing_sample_cases])
adhesive_dispensing_sample_metrics = await evaluation.run(adhesive_dispensing_sample_cases, concurrency=EVALUATION_CONCURRENCY)
adhesive_dispensing_sample_results = {case["id"]: evaluation.load(case) for case in adhesive_dispensing_sample_cases}
display(adhesive_dispensing_sample_metrics)
# Inspect a complete result: evaluation.inspect(adhesive_dispensing_sample_cases[0]["id"])
evaluation.show_answers(adhesive_dispensing_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_adhesive_dispensing_663588ace7_01,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,2,False,False,False,None,None,13063,0,True,unsupported,,,not_assessable
sample_adhesive_dispensing_663588ace7_02,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,8142,0,True,unsupported,,,abstained
sample_adhesive_dispensing_663588ace7_03,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,9183,0,True,unsupported,,,not reviewed
sample_adhesive_dispensing_663588ace7_04,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,alarm,table_row,False,True,8,False,0,0,0,False,False,False,False,False,12732,0,True,unsupported,,,correct
sample_adhesive_dispensing_663588ace7_05,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,9686,0,True,unsupported,,,not reviewed
sample_adhesive_dispensing_663588ace7_06,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,11102,0,True,unsupported,,,not reviewed
sample_adhesive_dispensing_663588ace7_07,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,1,False,False,False,None,None,11671,0,True,unsupported,,,not reviewed
sample_adhesive_dispensing_663588ace7_08,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,15018,0,True,included,,,not reviewed
sample_adhesive_dispensing_663588ace7_09,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,11488,0,True,included,,,not reviewed
sample_adhesive_dispensing_663588ace7_10,adhesive_dispensing,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,None,None,10536,0,True,included,,,not reviewed


### sample_adhesive_dispensing_663588ace7_01

根据《SX322012《新松机器人涂胶功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“指令类型异常”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 这是一张来自新松机器人涂胶应用操作手册的报警信息表，包含以下五列：报警码、报警信息、子码、子码报警信息、解决方法。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1269–1275; corpus 5.0, revision V5.0 / A-1; object `37dbdac856b735b52e89b341e021bc783f1da133b0107b18d9abbe9586476745`.

- 文档《SX322012新松机器人涂胶应用操作手册》版本A/0于2025年3月18日由沙景超完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1504–1509; corpus 5.0, revision V5.0 / A-1; object `2e7e06ed4c4c2d51462e2cb8cd98d196a251764b5b676ddec8e0412a200a7a70`.

- 新松机器人涂胶应用操作手册（V5.0）第5行：指令“MOVJ PR101 VJ=30.0 ACC=100 CNT=0”，功能描述“回原位”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 527–530; corpus 5.0, revision V5.0 / A-1; object `abc2f825192923a511be3091cd62c6df9257e4fb1c3c4127e12161100ee7e43b`.

- 该表格行来自《新松机器人涂胶应用操作手册》，描述了一条机器人指令：执行指令 `WAIT IN#33=ON T=-1.0`，其功能含义为“等待无常见故障”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1232–1235; corpus 4.0, revision V5.0 / A-1; object `916eb69817fbb87fe02f50122c696ff7e9e552cb2f7f35ceb3b4ef82210a9930`.

- 该行数据展示了新松机器人涂胶应用中的一组具体指令及其对应的操作状态：指令参数为“GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo =1 TIME =20”，其对应的操作状态为“开胶”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 742–745; corpus 4.0, revision V5.0 / A-1; object `83ccf96afed15aba0f13659052fc9f4bedc58b114984c22cf1d31c49e3c02eb5`.

- 该表格行位于文档 `SX322012《新松机器人涂胶应用操作手册》(A-1).md`（行号 14-16），包含一个加粗的标题文本：“**工业机器人 技术资料**”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 14–16; corpus 4.0, revision V5.0 / A-1; object `3b07ca9e890a36f44a19345f09c675c5837a06a7b6244b86719a489bab1fc415`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

#### Answer

根据检索到的手册证据（E1），

#### Model-assisted source review

Assessment: **not_assessable**; completeness: none; citation support: not_applicable; revision fidelity: mixed.

The generated answer is a fragment ('根据检索到的手册证据（E1），') and does not provide the requested information. The expected fact is that the resolution for '指令类型异常' is '联系技术人员'. The supplied evidence E1 contains the text '指令类型异常' and '联系技术人员' in the context of alarm codes 7108 and 7109. However, the evidence is from a V5.0 manual, while the question specifically asks about a V4.0 manual. More importantly, the answer itself is incomplete and fails to state the fact. Since the answer is a fragment and omits the requested facts, it is not complete. It is not a valid abstention because it does not explicitly state that the information is unavailable or that the evidence is insufficient for the specific revision requested; it simply trails off. Therefore, it is assessed as not_assessable/fragment.

...

#### Answer

The retrieved manual evidence (E1–E8) from SX322012《新松机器人涂胶应用操作手册》(A-1) V5.0/A-1 contains only revision-history tables, program-structure snippets (NOP, CALL, MOVJ, IF, WAIT, OUT, PULSE, RET, END), and cross-references to other manuals. None of the provided excerpts mention a 

### sample_adhesive_dispensing_4a3c30c770_20

根据《SX322012《新松机器人涂胶应用操作手册》(A-1)》（V5.0 / A-1），配置“实时显示”时，“涂胶实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。

#### Retrieved content summary

- 文档《SX322012新松机器人涂胶应用操作手册》版本A/0于2025年3月18日由沙景超完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1504–1509; corpus 5.0, revision V5.0 / A-1; object `2e7e06ed4c4c2d51462e2cb8cd98d196a251764b5b676ddec8e0412a200a7a70`.

- 该行数据展示了新松机器人涂胶应用中的一组具体指令及其对应的操作状态：指令参数为“GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo =1 TIME =20”，其对应的操作状态为“开胶”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 742–745; corpus 4.0, revision V5.0 / A-1; object `83ccf96afed15aba0f13659052fc9f4bedc58b114984c22cf1d31c49e3c02eb5`.

- 新松机器人涂胶应用操作手册（V5.0）第5行：指令“MOVJ PR101 VJ=30.0 ACC=100 CNT=0”，功能描述“回原位”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 527–530; corpus 5.0, revision V5.0 / A-1; object `abc2f825192923a511be3091cd62c6df9257e4fb1c3c4127e12161100ee7e43b`.

- 该表格行位于文档 `SX322012《新松机器人涂胶应用操作手册》(A-1).md`（行号 14-16），包含一个加粗的标题文本：“**工业机器人 技术资料**”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 14–16; corpus 4.0, revision V5.0 / A-1; object `3b07ca9e890a36f44a19345f09c675c5837a06a7b6244b86719a489bab1fc415`.

- 摘要：表格行（索引24，非标题行）包含两列内容。第一列值为“L102”（来源文件：SX322012《新松机器人涂胶应用操作手册》(A-1)，行661-664）。第二列为空。无合并单元格继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 661–664; corpus 4.0, revision V5.0 / A-1; object `a7b0feca41546abc10d97ddb0648275d33d2513111bbba3e90291a26998eb5ea`.

- 文档版本 **A/1**（《新松机器人涂胶应用操作手册》V5.0），发布日期 **2025年9月2日**，修订人 **沙景超**，主要修订内容为 **新增曦创协议界面配置**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1510–1515; corpus 5.0, revision V5.0 / A-1; object `f6796f08fe4b665e8166116c73032ac29b7148edf356f5af1d656533d190a26d`.

- 该表行（索引7）包含两列数据，无表头或合并单元格继承值。第一列内容为“PAUSE”，第二列为空。此行位于新松机器人涂胶应用操作手册（V5.0版本，文件SX322012）的表格中，源文件路径指向产品手册软件类目录。行内无图像、子元素或额外摘要信息，所有单元格均为普通数据单元格（非表头）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 905–908; corpus 5.0, revision V5.0 / A-1; object `79d1a8ad86b40b7944a2424c75768956c05ef457cbcb0aa671efa514bd438864`.

- 这是一张来自新松机器人涂胶应用操作手册的报警信息表，包含以下五列：报警码、报警信息、子码、子码报警信息、解决方法。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 1269–1275; corpus 5.0, revision V5.0 / A-1; object `37dbdac856b735b52e89b341e021bc783f1da133b0107b18d9abbe9586476745`.

#### Answer

在所提供的检索证据（E1–E8）中，没有任何一条记录涉及

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1-E8) does not contain the specific facts requested regarding the 'Real-time Display' parameters (mode status, glue gun system, program numbers, etc.). The provided evidence consists of revision history, code snippets, and alarm tables, none of which match the expected facts. Therefore, the refusal to answer based on the lack of evidence is appropriate and not a hallucination.

## Stud Welding — sampled manual revisions

40 cases from 2 CSV files. Set `STUD_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
STUD_WELDING_SAMPLE_CASES = [{'id': 'sample_stud_welding_94ff3b27ba_01', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“巡检配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['9. 巡检配置：在执行运动+工艺点时判断机器人是否停止', '机器人巡检号与停止焊点号一致，机器人停止', '其中起始IO号的含义是：巡检功能起始IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 488, 'line_end': 488, 'excerpt': '9. 巡检配置：在执行运动+工艺点时判断机器人是否停止。机器人巡检号与停止焊点号一致，机器人停止。其中起始IO号的含义是：巡检功能起始IO号。输入巡检信号：巡检功能输入巡检信号。输出确认信号：巡检功能输出确认信号。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_02', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“工艺作业模版”中“USERALM CODE=2810；”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['USERALM', 'CODE'], 'expected_facts': ['5 USERALM CODE=2810', '报警，请重新执行运动+工艺指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 643, 'line_end': 647, 'excerpt': '<tr>\n<td><p>5</p></td>\n<td><p>USERALM CODE=2810；</p></td>\n<td><p>报警，请重新执行运动&#43;工艺指令。</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_03', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4', 'section_title': '1.4 工作参数配置', 'heading_path': ['1 界面配置', '1.4 工作参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“工作参数配置”中“焊点计数”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 焊点计数：主要用于防漏焊', '该值为当前记录的焊点数量，用于与初始化指令设置的焊点总数比较，判断是否多焊或漏焊', '事例详见作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 187, 'line_end': 187, 'excerpt': '2. 焊点计数：主要用于防漏焊。该值为当前记录的焊点数量，用于与初始化指令设置的焊点总数比较，判断是否多焊或漏焊。事例详见作业。尼尔森焊机分多枪，最多5个焊枪。埃姆哈特焊机、永创、鸿柏焊机和尼尔森参数项功能一致。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_04', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“焊接超时”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['9 焊接超时 检查焊机报警'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 892, 'line_end': 896, 'excerpt': '<tr>\n<td><p>9</p></td>\n<td><p>焊接超时</p></td>\n<td><p>检查焊机报警</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_05', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.5', 'section_title': '1.5 IO配置', 'heading_path': ['1 界面配置', '1.5 IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“IO配置”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['路径：螺柱焊->IO设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 192, 'line_end': 192, 'excerpt': '路径：螺柱焊->IO设置', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_06', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 界面配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），配置“应用配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场螺柱焊应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项', '外设IO，需要进行监控通讯状态，需要配置起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 146, 'line_end': 146, 'excerpt': '逻辑IO分配：根据现场螺柱焊应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。外设IO，需要进行监控通讯状态，需要配置起始位和长度项。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_07', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“程序号超界”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['3 程序号超界 输入程序号超界，检查作业中焊接指令参数'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 862, 'line_end': 866, 'excerpt': '<tr>\n<td><p>3</p></td>\n<td><p>程序号超界</p></td>\n<td><p>输入程序号超界，检查作业中焊接指令参数</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_08', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 螺柱焊运动+焊接指令', 'heading_path': ['2 指令、快捷键说明', '2.1 螺柱焊运动+焊接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“螺柱焊运动+焊接指令”中“STUDL P(参数1) VL =(参数2) ACC =(参数3) CNT =(参数4) ET=”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['STUDL', 'ACC'], 'expected_facts': ['格式 STUDL P(参数1) VL =(参数2) ACC =(参数3) CNT =(参数4) ET=(参数5) INTERLOCK=(参数6) GUN=(参数7) PROGNo=(参数8) ;注释内容 STUDJ P(参数1) VJ =(参数2) ACC =(参数3) CNT =(参数4) INTERLOCK=(参数6) GUN=(参数7) PROGNo=(参数8) ;注释内容'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 268, 'line_end': 271, 'excerpt': '<tr>\n<td><p>格式</p></td>\n<td><p>STUDL P&#40;参数1&#41; VL =&#40;参数2&#41; ACC =&#40;参数3&#41; CNT =&#40;参数4&#41; ET=&#40;参数5&#41; INTERLOCK=&#40;参数6&#41; GUN=&#40;参数7&#41; PROGNo=&#40;参数8&#41; ;注释内容</p><p>STUDJ P&#40;参数1&#41; VJ =&#40;参数2&#41; ACC =&#40;参数3&#41; CNT =&#40;参数4&#41; INTERLOCK=&#40;参数6&#41; GUN=&#40;参数7&#41; PROGNo=&#40;参数8&#41; ;注释内容</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_09', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“工艺作业模版”中，应如何理解或操作“USERALM CODE=2802；”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['USERALM CODE=2802', '警告:系统没准备好'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 783, 'line_end': 787, 'excerpt': '<tr>\n<td></td>\n<td><p>USERALM CODE=2802；</p></td>\n<td><p>警告:系统没准备好。</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_10', 'category': 'stud_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 界面配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），操作“通讯配置”时，涉及“输出信号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出信号：IO状态可手动输出，输出条件时示教模式下上电并Deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 242, 'line_end': 242, 'excerpt': '输出信号：IO状态可手动输出，输出条件时示教模式下上电并Deadman使能。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_11', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），配置“快捷键”时，“铆枪号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 铆枪号:螺柱焊应用触发相应系统对应的焊枪号的快捷功能', '根据配置的范围，可以配置从1-5'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 481, 'line_end': 481, 'excerpt': '2. 铆枪号:螺柱焊应用触发相应系统对应的焊枪号的快捷功能。根据配置的范围，可以配置从1-5。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_12', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.2', 'section_title': '3.2 防漏焊的基本作业流程（利用防漏焊指令实现）', 'heading_path': ['3 模板作业', '3.2 防漏焊的基本作业流程（利用防漏焊指令实现）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“防漏焊的基本作业流程（利用防漏焊指令实现）”中“STUDJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['STUDJ', 'ACC'], 'expected_facts': ['STUDJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P5，焊枪2进行焊接'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 588, 'line_end': 591, 'excerpt': '<tr>\n<td><p>STUDJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td>\n<td><p>运动到螺柱焊焊接位置P5，焊枪2进行焊接。</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_13', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.1', 'section_title': '3.1 基本作业流程', 'heading_path': ['3 模板作业', '3.1 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“基本作业流程”中，应如何理解或操作“STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P2,进行焊接'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 513, 'line_end': 516, 'excerpt': '<tr>\n<td><p>STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td>\n<td><p>运动到螺柱焊焊接位置P2,进行焊接。</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_14', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），配置“常见报警及解决方法”时，“报警：7911”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['报警：7911 螺柱焊配置IO不再逻辑IO分配范围 0 检查配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 985, 'line_end': 991, 'excerpt': '<tr>\n<td><p>报警：7911</p></td>\n<td><p>螺柱焊配置IO不再逻辑IO分配范围</p></td>\n<td><p>0</p></td>\n<td></td>\n<td><p>检查配置</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_15', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“同步快捷键OP2-OP5”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 同步快捷键OP2-OP5：焊枪前进、焊枪后退、送钉功能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 482, 'line_end': 482, 'excerpt': '3. 同步快捷键OP2-OP5：焊枪前进、焊枪后退、送钉功能。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_16', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“提示：7957”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['提示：7957 执行螺柱焊前进和后退快捷功能 0'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 1090, 'line_end': 1096, 'excerpt': '<tr>\n<td><p>提示：7957</p></td>\n<td><p>执行螺柱焊前进和后退快捷功能</p></td>\n<td><p>0</p></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_17', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 界面配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“通讯配置”中，应如何理解或操作“输入信号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['输入信号：IO状态只显示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 244, 'line_end': 244, 'excerpt': '输入信号：IO状态只显示。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_18', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4', 'section_title': '1.4 工作参数配置', 'heading_path': ['1 界面配置', '1.4 工作参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），配置“工作参数配置”时，“路径”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['路径：螺柱焊->工作参数，以尼尔森焊机为例'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 180, 'line_end': 180, 'excerpt': '路径：螺柱焊->工作参数，以尼尔森焊机为例。', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_19', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“提示：7955”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['提示：7955 当前指令不支持1.0规划，请更改 0'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 1076, 'line_end': 1082, 'excerpt': '<tr>\n<td><p>提示：7955</p></td>\n<td><p>当前指令不支持1&#46;0规划，请更改</p></td>\n<td><p>0</p></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_94ff3b27ba_20', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 螺柱焊运动+焊接指令', 'heading_path': ['2 指令、快捷键说明', '2.1 螺柱焊运动+焊接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），“螺柱焊运动+焊接指令”中“STUDJ P1 VJ = 20 ACC=100 CNT=0 INTERLOCK=1 GUN =”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['STUDJ', 'ACC'], 'expected_facts': ['举例 STUDJ P1 VJ = 20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 //以关节速度运动到P1点，进行焊接', 'STUDL P2 VL= 200 ACC=100 CNT=0 ET=1 INTERLOCK=2 GUN = 2 PROGNo = 3 //以直角速度运动到P2点，进行焊接', 'STUDJ P3 VJ = 20 ACC=100 CNT=0 GUN = 2 PROGNo = 3 //以关节速度运动到P3点，进行焊接'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'line_start': 276, 'line_end': 279, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td><p>STUDJ P1 VJ = 20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3  //以关节速度运动到P1点，进行焊接。</p><p>STUDL P2 VL= 200 ACC=100 CNT=0 ET=1 INTERLOCK=2 GUN = 2 PROGNo = 3 //以直角速度运动到P2点，进行焊接。</p><p>STUDJ P3 VJ = 20 ACC=100 CNT=0 GUN = 2 PROGNo = 3 //以关节速度运动到P3点，进行焊接。</p><p>STUDL P4 VL= 200 ACC=100 CNT=0 ET=1 GUN = 2 PROGNo = 3 //以直角速度运动到P4点，进行焊接。</p></td>\n</tr>', 'sha256': '4e73012e50bdeef2ce046c0e17db6d04c806a90c846cb558fbaee819f109c5e9'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_01', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“常见报警及解决方法”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PLC', 'NT151'], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法 报警：7901 螺柱焊焊接异常 0 焊机故障 请检查焊机状态 1 机器人发出程序号与接收程序号不一致 请联系技术人员 2 焊点数量设置错误 实际焊点个数比设置数量大，检查漏焊设置数值', '3 程序号超界 请修改程序号设置为1-255之间 4 车型号超界 请修改车型号设置为0-15之间 5 焊接超时 请人工检查焊机状态 6 不良焊接 请人工检查焊接状态 7 机器人误发预选无工序信号 修改机器人输出无工序状态为0 8 无效焊点检测 请检查焊接状态，并执行检测指令 9 焊接超时 请人工检查焊接状态 10 漏钉 需手动添钉后，在执行工艺指令 11 机器人循环就绪信号异常 请检查焊机状态 报警：7902 螺柱焊焊前状态异常 0 焊机不在线 检查在线信号 1 焊机未回到原位 使用快捷键焊机回原位后，在执行工艺指令 2 系统未就绪 请检查铆机状态 3 系统有故障 请检查铆机状态 4 焊接完成信号未清除 请修改程序号设置为1-255之间 报警：7903 螺柱焊焊机报警 0 检查焊机 报警：7904 文件操作失败 0 请联系技术人员 报警：7905 螺柱焊程序号异常 0 检查作业 报警：7906 螺柱焊焊点漏焊 0 子码为焊枪号 报警：7907 螺柱焊焊点多焊 0 子码为焊枪号 报警：7908 焊接判定结果 0 只在永创焊机使用 报警：7909 送料器卡钉异常报警 0 只在永创焊机使用 报警：7910 卡钉或不通或没钉，请检查送钉机构 0 只在永创焊机使用 报警：7911 螺柱焊配置IO不再逻辑IO分配范围 0 检查配置 报警：7912 机器人干涉 0 检查干涉区 报警：7913 巡检功能异常 0 信号异常 请检查PLC设备状态 1 巡检号为0 请检查PLC设备状态 报警：7914 螺柱焊指令集错误 0 请检查作业名称 报警：7915 螺柱焊-NT151监控异常 请检查NT151模块功能 报警：7916 响应安全门请求暂停中 请检查安全门状态 报警：7917 水流量异常 请检查实际流量与设定流量值 报警：7918 气流量异常 请检查气压表状态 报警：7919 通讯状态异常 请检查相应的外设设备的通讯线或模块是否正常 报警：7920 请判断鸿柏螺柱焊协议，使用协议异常 请核对鸿柏设备协议IO,联系技术人员 提示：7951 螺柱焊漏焊个数为： 0 子码为漏焊个数 提示：7952 螺柱焊多焊个数为： 0 提示：7953 当前模式为非NORMAL模式 0 提示：7954 螺柱焊清空堆栈 0 提示：7955 当前指令不支持1.0规划，请更改 0 提示：7956 实际焊点与设置一致', '0 提示：7957 执行螺柱焊前进快捷功能 0 提示：7958 执行螺柱焊出钉快捷功能 0 提示：7960 螺柱焊巡检暂停中 0 提示：7961 巡检配置为0功能未启用 提示：7962 执行螺柱焊后退快捷功能 提示：7963 通讯为屏蔽状态 提示：7964 快捷键 0 焊枪前进执行完成 1 焊枪回退执行完成 2 送钉执行完成 3 焊枪前进执行异常 4 焊枪回退执行异常 5 送钉执行异常'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 327, 'line_end': 327, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码报警信息</p></td><td><p>解决方法</p></td></tr><tr><td rowspan="12"><p>报警：7901</p></td><td rowspan="12"><p>螺柱焊焊接异常</p></td><td><p>0</p></td><td><p> 焊机故障</p></td><td><p>请检查焊机状态</p></td></tr><tr><td><p>1</p></td><td><p>机器人发出程序号与接收程序号不一致</p></td><td><p>请联系技术人员</p></td></tr><tr><td><p>2</p></td><td><p>焊点数量设置错误</p></td><td><p>实际焊点个数比设置数量大，检查漏焊设置数值。</p></td></tr><tr><td><p>3</p></td><td><p>程序号超界</p></td><td><p>请修改程序号设置为1-255之间</p></td></tr><tr><td><p>4</p></td><td><p>车型号超界</p></td><td><p>请修改车型号设置为0-15之间</p></td></tr><tr><td><p>5</p></td><td><p>焊接超时</p></td><td><p>请人工检查焊机状态</p></td></tr><tr><td><p>6</p></td><td><p>不良焊接</p></td><td><p>请人工检查焊接状态</p></td></tr><tr><td><p>7</p></td><td><p>机器人误发预选无工序信号</p></td><td><p>修改机器人输出无工序状态为0</p></td></tr><tr><td><p>8</p></td><td><p>无效焊点检测</p></td><td><p>请检查焊接状态，并执行检测指令</p></td></tr><tr><td><p>9</p></td><td><p>焊接超时</p></td><td><p>请人工检查焊接状态</p></td></tr><tr><td><p>10</p></td><td><p>漏钉</p></td><td><p>需手动添钉后，在执行工艺指令</p></td></tr><tr><td><p>11</p></td><td><p>机器人循环就绪信号异常</p></td><td><p>请检查焊机状态</p></td></tr><tr><td rowspan="5"><p>报警：7902</p></td><td rowspan="5"><p>螺柱焊焊前状态异常</p></td><td><p>0</p></td><td><p>焊机不在线</p></td><td><p>检查在线信号</p></td></tr><tr><td><p>1</p></td><td><p>焊机未回到原位</p></td><td><p>使用快捷键焊机回原位后，在执行工艺指令</p></td></tr><tr><td><p>2</p></td><td><p>系统未就绪</p></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>3</p></td><td><p>系统有故障</p></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>4</p></td><td><p>焊接完成信号未清除</p></td><td><p>请修改程序号设置为1-255之间</p></td></tr><tr><td><p>报警：7903</p></td><td><p>螺柱焊焊机报警</p></td><td><p>0</p></td><td></td><td><p>检查焊机</p></td></tr><tr><td><p>报警：7904</p></td><td><p>文件操作失败</p></td><td><p>0</p></td><td></td><td><p>请联系技术人员</p></td></tr><tr><td><p>报警：7905</p></td><td><p>螺柱焊程序号异常</p></td><td><p>0</p></td><td></td><td><p>检查作业</p></td></tr><tr><td><p>报警：7906</p></td><td><p>螺柱焊焊点漏焊</p></td><td><p>0</p></td><td></td><td><p>子码为焊枪号</p></td></tr><tr><td><p>报警：7907</p></td><td><p>螺柱焊焊点多焊</p></td><td><p>0</p></td><td></td><td><p>子码为焊枪号</p></td></tr><tr><td><p>报警：7908</p></td><td><p>焊接判定结果</p></td><td><p>0</p></td><td></td><td><p>只在永创焊机使用</p></td></tr><tr><td><p>报警：7909</p></td><td><p>送料器卡钉异常报警</p></td><td><p>0</p></td><td></td><td><p>只在永创焊机使用</p></td></tr><tr><td><p>报警：7910</p></td><td><p>卡钉或不通或没钉，请检查送钉机构</p></td><td><p>0</p></td><td></td><td><p>只在永创焊机使用</p></td></tr><tr><td><p>报警：7911</p></td><td><p>螺柱焊配置IO不再逻辑IO分配范围</p></td><td><p>0</p></td><td></td><td><p>检查配置</p></td></tr><tr><td><p>报警：7912</p></td><td><p>机器人干涉</p></td><td><p>0</p></td><td></td><td><p>检查干涉区</p></td></tr><tr><td rowspan="2"><p>报警：7913</p></td><td rowspan="2"><p>巡检功能异常</p></td><td><p>0</p></td><td><p>信号异常</p></td><td><p>请检查PLC设备状态</p></td></tr><tr><td><p>1</p></td><td><p>巡检号为0</p></td><td><p>请检查PLC设备状态</p></td></tr><tr><td><p>报警：7914</p></td><td><p>螺柱焊指令集错误</p></td><td><p>0</p></td><td></td><td><p>请检查作业名称</p></td></tr><tr><td><p>报警：7915</p></td><td><p>螺柱焊-NT151监控异常</p></td><td></td><td></td><td><p>请检查NT151模块功能</p></td></tr><tr><td><p>报警：7916</p></td><td><p>响应安全门请求暂停中</p></td><td></td><td></td><td><p>请检查安全门状态</p></td></tr><tr><td><p>报警：7917</p></td><td><p>水流量异常</p></td><td></td><td></td><td><p>请检查实际流量与设定流量值</p></td></tr><tr><td><p>报警：7918</p></td><td><p>气流量异常</p></td><td></td><td></td><td><p>请检查气压表状态</p></td></tr><tr><td><p>报警：7919</p></td><td><p>通讯状态异常</p></td><td></td><td></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>报警：7920</p></td><td><p>请判断鸿柏螺柱焊协议，使用协议异常</p></td><td></td><td></td><td><p>请核对鸿柏设备协议IO,联系技术人员</p></td></tr><tr><td><p>提示：7951</p></td><td><p>螺柱焊漏焊个数为：</p></td><td><p>0</p></td><td></td><td><p>子码为漏焊个数</p></td></tr><tr><td><p>提示：7952</p></td><td><p>螺柱焊多焊个数为：</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7953</p></td><td><p>当前模式为非NORMAL模式</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7954</p></td><td><p>螺柱焊清空堆栈</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7955</p></td><td><p>当前指令不支持1.0规划，请更改</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7956</p></td><td><p>实际焊点与设置一致。</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7957</p></td><td><p>执行螺柱焊前进快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7958</p></td><td><p>执行螺柱焊出钉快捷功能</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7960</p></td><td><p>螺柱焊巡检暂停中</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>提示：7961</p></td><td><p>巡检配置为0功能未启用</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7962</p></td><td><p>执行螺柱焊后退快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：7963</p></td><td><p>通讯为屏蔽状态</p></td><td></td><td></td><td></td></tr><tr><td rowspan="6"><p>提示：7964</p></td><td rowspan="6"><p>快捷键</p></td><td><p>0</p></td><td><p>焊枪前进执行完成</p></td><td></td></tr><tr><td><p>1</p></td><td><p>焊枪回退执行完成</p></td><td></td></tr><tr><td><p>2</p></td><td><p>送钉执行完成</p></td><td></td></tr><tr><td><p>3</p></td><td><p>焊枪前进执行异常</p></td><td></td></tr><tr><td><p>4</p></td><td><p>焊枪回退执行异常</p></td><td></td></tr><tr><td><p>5</p></td><td><p>送钉执行异常</p></td><td></td></tr></table>', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_02', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“工艺作业模版”中“IF IN#4108 =OFF;GOTO L10;”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['IN#4108', 'OFF'], 'expected_facts': ['1 IF IN#4108 =OFF;GOTO L10; 读急停的IO位，OFF为没有读到暂停、急停信号', '为OFF的时候，将会跳转至L10,程序结束', '为ON的时候，将会继续向下执行，执行复位的操作并报警'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 319, 'line_end': 319, 'excerpt': '<table><tr><td><p>1</p></td><td><p>IF IN#4108 =OFF;GOTO L10;</p></td><td><p>读急停的IO位，OFF为没有读到暂停、急停信号。为OFF的时候，将会跳转至L10,程序结束。为ON的时候，将会继续向下执行，执行复位的操作并报警。</p></td></tr><tr><td><p>2</p></td><td><p>OUT OT#17 = OFF;</p></td><td><p>1号枪焊接开始复位</p></td></tr><tr><td><p>3</p></td><td><p>OUT OT#2 = OFF;</p></td><td><p>焊接有效复位</p></td></tr><tr><td><p>4</p></td><td><p>OUT OGFA#4=0</p></td><td><p>13.0-15.3 stud id 复位</p></td></tr><tr><td><p>5</p></td><td><p>USERALM CODE=2810；</p></td><td><p>报警，请重新执行运动+工艺指令。</p></td></tr><tr><td><p>6</p></td><td><p>L10</p></td><td><p>L10</p></td></tr><tr><td><p>7</p></td><td><p>END</p></td><td></td></tr></table>', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_03', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.1', 'section_title': '2.1 螺柱焊运动+焊接指令', 'heading_path': ['2 指令、快捷键说明', '2.1 螺柱焊运动+焊接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“螺柱焊运动+焊接指令”中“指令含义”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['指令含义：螺柱焊运动+焊接指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 227, 'line_end': 227, 'excerpt': '指令含义：螺柱焊运动+焊接指令。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_04', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.2', 'section_title': '3.2 防漏焊的基本作业流程（利用防漏焊指令实现）', 'heading_path': ['3 模板作业', '3.2 防漏焊的基本作业流程（利用防漏焊指令实现）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322013《新松机器人螺柱焊应用操作手册》(A-2).md”（V5.0 / A-2），作业模板中A20_STUD、MOVL、STUDJ的执行顺序和作用是什么？', 'anchors': ['A20_STUD', 'MOVL', 'STUDJ'], 'expected_facts': ['A20_STUD Init Gun=1 Count=2 螺柱焊初始化', 'MOVL VL=20 安全点', 'STUDJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到螺柱焊焊接位置P1，焊枪1进行焊接。', 'MOVL VL=20 运动到安全位置', 'STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到螺柱焊焊接位置P2，焊枪1进行焊接。', 'MOVL VL=20 运动到安全位置', 'A20_STUD Init Gun=2 Count=3 ; 螺柱焊初始化', 'MOVL VL=20 运动到螺柱焊焊接位置', 'STUDJ P3 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P3，焊枪2进行焊接。', 'MOVL VL=20 运动到安全位置', 'STUDJ P4 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P4，焊枪2进行焊接。', 'MOVL VL=20 运动到安全位置', 'MOVL VL=20 运动到螺柱焊焊接位置', 'STUDJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P5，焊枪2进行焊接。', 'MOVL VL=20 运动到安全位置', 'MOVL VL=20 运动到下一个螺柱焊焊接位置', 'STUDJ P6 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P6，焊枪2进行焊接。', 'MOVL VL=20 运动到安全位置', 'A20_STUD ProcessCheck 防漏焊检测'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 307, 'line_end': 307, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>A20_STUD Init Gun=1 Count=2 </p></td><td><p>螺柱焊初始化</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>安全点</p></td></tr><tr><td><p>STUDJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P1，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P2，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>A20_STUD Init Gun=2 Count=3 ;</p></td><td><p>螺柱焊初始化</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到螺柱焊焊接位置</p></td></tr><tr><td><p>STUDJ P3 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P3，焊枪2进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>STUDJ P4 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P4，焊枪2进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到螺柱焊焊接位置</p></td></tr><tr><td><p>STUDJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P5，焊枪2进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到下一个螺柱焊焊接位置</p></td></tr><tr><td><p>STUDJ P6 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P6，焊枪2进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>A20_STUD ProcessCheck</p></td><td><p>防漏焊检测</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_05', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322013《新松机器人螺柱焊应用操作手册》(A-2).md”（V5.0 / A-2），涂胶流程模板在冲洗或填充之前如何检查运行模式和胶桶状态，胶量不足时怎样处理？', 'anchors': ['USERALM'], 'expected_facts': ['USERALM CODE=2800； 警告:枪置位有误；', 'USERALM CODE=2801； 警告:焊接完成；', 'USERALM CODE=2802； 警告:系统没准备好。', 'USERALM CODE=2803； 系统有错误'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 323, 'line_end': 323, 'excerpt': '<table><tr><td></td><td></td><td></td></tr><tr><td></td><td><p>NOP</p></td><td></td></tr><tr><td></td><td><p>IF IN#22 == OFF GOTO L1;</p></td><td><p>1号枪在后置位为ON。如果1号枪未在后置位，跳转到L1,报警：枪置位有误；</p></td></tr><tr><td></td><td><p>OUT OT#17 = OFF;</p></td><td><p>1号枪焊接开始</p></td></tr><tr><td></td><td><p>IF IN#17 == ON GOTO L2;</p></td><td><p>1号枪焊接未完成，继续向下执行作业。1号枪焊接完成，跳转至L2:焊接完成。</p></td></tr><tr><td></td><td><p>IF IN#1 == OFF GOTO L3;</p></td><td><p>系统准备好，继续向下执行作业。如果系统没准备好，跳转至L3:系统没准备好。</p></td></tr><tr><td></td><td><p>IF IN#1 == OFF GOTO L4;</p></td><td><p>该位含义是没有错误。系统没有错误，继续向下执行作业。如果系统没准备好，跳转至L4:系统有错误。</p></td></tr><tr><td></td><td><p>OUT OT#2 =ON;</p></td><td><p>焊接有效</p></td></tr><tr><td></td><td><p>OUT OGFA#4=R201;</p></td><td><p>13.0-15.3 stud id 为目标值。将实型变量R201赋值给铆钉ID。</p></td></tr><tr><td></td><td><p>OUT OT#17 =ON;</p></td><td><p>1号枪开始铆接</p></td></tr><tr><td></td><td><p>WAIT IN#17 == ON  T=10;</p></td><td><p>10s内是否接受到铆接完成信号,等待铆接进程结束；</p></td></tr><tr><td></td><td><p>DELAY T=0.2</p></td><td><p>等待0.2S。</p></td></tr><tr><td></td><td><p>OUT OGFA#4=0</p></td><td><p>13.0-15.3 stud id 为0</p></td></tr><tr><td></td><td><p>OUT OT#17 =OFF;</p></td><td><p>1号枪焊接结束</p></td></tr><tr><td></td><td><p>IF IN#22 == OFF GOTO L1;</p></td><td><p>1号枪在后置位</p></td></tr><tr><td></td><td><p>INC R100</p></td><td><p>焊接完成，焊点计数自动+1；R100的初始值为0</p></td></tr><tr><td></td><td><p>GOTO L6</p></td><td><p>至L6结束</p></td></tr><tr><td></td><td><p>L1</p></td><td><p>L1</p></td></tr><tr><td></td><td><p>USERALM CODE=2800；</p></td><td><p>警告:枪置位有误；</p></td></tr><tr><td></td><td><p>GOTO L5</p></td><td><p>至L5复位结束</p></td></tr><tr><td></td><td><p>L2</p></td><td><p>L2</p></td></tr><tr><td></td><td><p>USERALM CODE=2801；</p></td><td><p>警告:焊接完成；</p></td></tr><tr><td></td><td><p>GOTO L5</p></td><td><p>至L5复位结束</p></td></tr><tr><td></td><td><p>L3</p></td><td><p>L3</p></td></tr><tr><td></td><td><p>USERALM CODE=2802；</p></td><td><p>警告:系统没准备好。</p></td></tr><tr><td></td><td><p>GOTO L5</p></td><td><p>至L5复位结束</p></td></tr><tr><td></td><td><p>L4</p></td><td><p>L4</p></td></tr><tr><td></td><td><p>USERALM CODE=2803；</p></td><td><p>系统有错误</p></td></tr><tr><td></td><td><p>L5</p></td><td><p>L5</p></td></tr><tr><td></td><td><p>OUT OT#17 = OFF;</p></td><td><p>警告:1号枪焊接开始复位</p></td></tr><tr><td></td><td><p>OUT OT#2 = OFF;</p></td><td><p>焊接有效复位</p></td></tr><tr><td></td><td><p>OUT OGFA#4=0</p></td><td><p>13.0-15.3 stud id 复位</p></td></tr><tr><td></td><td><p>L6</p></td><td><p>L6</p></td></tr><tr><td></td><td><p>END</p></td><td><p>结束</p></td></tr></table>', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_06', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.4', 'section_title': '1.4 工作参数配置', 'heading_path': ['1 螺柱焊应用配置', '1.4 工作参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“工作参数配置”中“焊点计数”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 焊点计数：主要用于防漏焊', '该值为当前记录的焊点数量，用于与初始化指令设置的焊点总数比较，判断是否多焊或漏焊', '事例详见作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 134, 'line_end': 134, 'excerpt': '2. 焊点计数：主要用于防漏焊。该值为当前记录的焊点数量，用于与初始化指令设置的焊点总数比较，判断是否多焊或漏焊。事例详见作业。尼尔森焊机分多枪，最多5个焊枪。埃姆哈特焊机、永创、鸿柏焊机和尼尔森参数项功能一致。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_07', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.5', 'section_title': '1.5 IO配置', 'heading_path': ['1 螺柱焊应用配置', '1.5 IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“IO配置”中，应如何理解或操作“鸿柏”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['鸿柏：输入：66位 输出：58位'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 153, 'line_end': 153, 'excerpt': '鸿柏：输入：66位 输出：58位；', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_08', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“快捷键”中，应如何理解或操作“模式存储变量号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['8. 模式存储变量号：工作模式存储变量用于作业判断', '其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 292, 'line_end': 292, 'excerpt': '8. 模式存储变量号：工作模式存储变量用于作业判断。其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_09', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），配置“快捷键”时，“铆枪号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 铆枪号:螺柱焊应用触发相应系统对应的焊枪号的快捷功能', '根据配置的范围，可以配置从1-5'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 286, 'line_end': 286, 'excerpt': '2. 铆枪号:螺柱焊应用触发相应系统对应的焊枪号的快捷功能。根据配置的范围，可以配置从1-5。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_10', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 螺柱焊应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），配置“应用配置”时，“通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 107, 'line_end': 107, 'excerpt': '通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_11', 'category': 'stud_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 螺柱焊应用配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），操作“通讯配置”时，涉及“输出信号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['输出信号：IO状态可手动输出，输出条件时示教模式下上电并Deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 189, 'line_end': 189, 'excerpt': '输出信号：IO状态可手动输出，输出条件时示教模式下上电并Deadman使能。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_12', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '3.1', 'section_title': '3.1 基本作业流程', 'heading_path': ['3 模板作业', '3.1 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322013《新松机器人螺柱焊应用操作手册》(A-2).md”（V5.0 / A-2），作业模板中MOVL、STUDJ、ACC的执行顺序和作用是什么？', 'anchors': ['MOVL', 'STUDJ', 'ACC'], 'expected_facts': ['MOVL VL=20 安全点', 'STUDJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P1,进行焊接。', 'MOVL VL=20 运动到安全位置', 'STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到螺柱焊焊接位置P2,进行焊接。', 'MOVL VL=20 运动到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 301, 'line_end': 301, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>安全点</p></td></tr><tr><td><p>STUDJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P1,进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>STUDJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到螺柱焊焊接位置P2,进行焊接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_13', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.1', 'section_title': '2.1 螺柱焊运动+焊接指令', 'heading_path': ['2 指令、快捷键说明', '2.1 螺柱焊运动+焊接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“螺柱焊运动+焊接指令”中“根据吉利目前的现场要求”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['STUDCONNECT'], 'expected_facts': ['根据吉利目前的现场要求，工艺通过作业的形式呈现，减少内部实现', '因此针对埃姆哈特的焊机，该方案通过调用子作业“STUDCONNECT”的方式，呈现出来，具体作业形式详见第4章工艺作业模版'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 235, 'line_end': 235, 'excerpt': '**根据吉利目前的现场要求，工艺通过作业的形式呈现，减少内部实现。因此针对埃姆哈特的焊机，该方案通过调用子作业“STUDCONNECT”的方式，呈现出来，具体作业形式详见第4章工艺作业模版。**', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_14', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“工艺作业模版”中，应如何理解或操作“埃姆哈特-后台作业”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 埃姆哈特-后台作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 317, 'line_end': 317, 'excerpt': '1. **埃姆哈特-后台作业**', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_15', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 螺柱焊应用配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“通讯配置”中，应如何理解或操作“各品牌焊机显示的协议位内容不同”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['各品牌焊机显示的协议位内容不同，参考各焊机协议位进行设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 193, 'line_end': 193, 'excerpt': '各品牌焊机显示的协议位内容不同，参考各焊机协议位进行设置。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_16', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“快捷键”中，应如何理解或操作“PLC-SIM”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['6. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 290, 'line_end': 290, 'excerpt': '6. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_17', 'category': 'stud_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.7', 'section_title': '1.7 实时显示', 'heading_path': ['1 螺柱焊应用配置', '1.7 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），配置“实时显示”时，“螺柱焊实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['螺柱焊实时显示功能，可查看螺柱焊的相关参数，如模式状态、枪状态显示、程序号、程序点计数等参数', '可在本地和远程等模式下进行查看'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 201, 'line_end': 201, 'excerpt': '螺柱焊实时显示功能，可查看螺柱焊的相关参数，如模式状态、枪状态显示、程序号、程序点计数等参数；可在本地和远程等模式下进行查看。', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_18', 'category': 'stud_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '2.2', 'section_title': '2.2 螺柱焊指令集', 'heading_path': ['2 指令、快捷键说明', '2.2 螺柱焊指令集'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），“螺柱焊指令集”中“指令路径”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A20'], 'expected_facts': ['指令路径：指令集->应用指令集->点焊->A20\\_STUD'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 239, 'line_end': 239, 'excerpt': '指令路径：指令集->应用指令集->点焊->A20\\_STUD；', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_19', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“工艺作业模版”中，应如何理解或操作“埃姆哈特-前台作业”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 埃姆哈特-前台作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 321, 'line_end': 321, 'excerpt': '1. **埃姆哈特-前台作业**', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}, {'id': 'sample_stud_welding_7f5ad54e6a_20', 'category': 'stud_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-2', 'section': '1.5', 'section_title': '1.5 IO配置', 'heading_path': ['1 螺柱焊应用配置', '1.5 IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“IO配置”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['路径：螺柱焊->IO设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'line_start': 139, 'line_end': 139, 'excerpt': '路径：螺柱焊->IO设置', 'sha256': '302ef2834a534a5c85a73921c97bfc1f49d8bc2a73fc7bfe25d084ae022c746b'}]}]
register_sample_bank('stud_welding', STUD_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md', 'seed': 856869179209436081, 'population': 167, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(488, 488), (643, 647), (187, 187), (892, 896), (192, 192), (146, 146), (862, 866), (268, 271), (783, 787), (242, 242), (481, 481), (588, 591), (513, 516), (985, 991), (482, 482), (1090, 1096), (244, 244), (180, 180), (1076, 1082), (276, 279)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md', 'seed': 7819831001545578843, 'population': 73, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(327, 327), (319, 319), (227, 227), (307, 307), (323, 323), (134, 134), (153, 153), (292, 292), (286, 286), (107, 107), (189, 189), (301, 301), (235, 235), (317, 317), (193, 193), (290, 290), (201, 201), (239, 239), (321, 321), (139, 139)]}])


In [ ]:
# | notest
STUD_WELDING_SAMPLE_IDS = None
stud_welding_sample_cases = select_cases(STUD_WELDING_SAMPLE_CASES, STUD_WELDING_SAMPLE_IDS)
display(pd.DataFrame(stud_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in stud_welding_sample_cases])
stud_welding_sample_metrics = await evaluation.run(stud_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
stud_welding_sample_results = {case["id"]: evaluation.load(case) for case in stud_welding_sample_cases}
display(stud_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(stud_welding_sample_cases[0]["id"])
evaluation.show_answers(stud_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_stud_welding_94ff3b27ba_01,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,4,False,False,False,None,None,12215,0,True,unsupported,,,not_assessable
sample_stud_welding_94ff3b27ba_02,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,2,False,False,False,False,False,16104,0,True,included,,,not_assessable
sample_stud_welding_94ff3b27ba_03,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,instruction,prose,False,True,8,False,0,0,3,False,False,False,None,None,15063,0,True,included,,,not reviewed
sample_stud_welding_94ff3b27ba_04,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,1,False,False,False,None,None,17862,0,True,unsupported,,,not reviewed
sample_stud_welding_94ff3b27ba_05,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,5,False,False,False,None,None,6985,0,True,unsupported,,,not reviewed
sample_stud_welding_94ff3b27ba_06,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,5,False,False,False,None,None,13111,0,True,unsupported,,,not reviewed
sample_stud_welding_94ff3b27ba_07,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,4,False,False,False,None,None,12525,1,True,unsupported,,,not reviewed
sample_stud_welding_94ff3b27ba_08,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,8901,0,False,unavailable,,"Response validation failed: Include distinct setup, requested instruction, and follow-up statements",not reviewed
sample_stud_welding_94ff3b27ba_09,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,5,False,False,False,None,None,12067,1,True,unsupported,,,not reviewed
sample_stud_welding_94ff3b27ba_10,stud_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md,V4.0 / A-0,prerequisite,prose,False,True,8,False,0,0,5,False,False,False,None,None,16165,0,True,unsupported,,,not reviewed


### sample_stud_welding_94ff3b27ba_01

根据《SX322013《新松机器人螺柱焊功能使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“巡检配置”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- **新松机器人螺柱焊应用操作手册**（文档标题/章节名，来源文件：SX322013《新松机器人螺柱焊应用操作手册》(A-2)，位于产品手册/软件类/V5.0版本目录）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 13–15; corpus 5.0, revision V5.0 / A-2; object `3580d5a8c5116a37b6bf8e2524d4a1e2392c9979a45ca913b8f10e6e1e9c6ad3`.

- 新松机器人螺柱焊应用操作手册（SX322013）中，针对“提示：7964”（即焊枪回退执行异常），对应的快捷键为“4”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 1162–1166; corpus 4.0, revision V5.0 / A-2; object `dae382a9ec7f3bd7bad9ee410293feaf5721b808434616277e0d16f679510b31`.

- 新松机器人螺柱焊应用操作手册（V5.0）报警表格的表头行，包含以下五列字段：报警码、报警信息、子码、子码报警信息、解决方法。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 829–835; corpus 5.0, revision V5.0 / A-2; object `5d7ab4c4a67abded91dcee9773119e838c5128ce5d728de088c2ff1a17b97b9e`.

- 该表行（第14行）包含三列数据：第一列为空，第二列和第三列均为“L3”。此记录来自《新松机器人螺柱焊应用操作手册》(SX322013) 的表格结构，用于描述特定配置或等级参数。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 428–432; corpus 5.0, revision V5.0 / A-2; object `d9fd32fe736d9e26cd53f838632fab473251c095bc4991da5014144c6b86cc4e`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

- 该行记录关联了文档《新松机器人通用操作手册》及其文档编号 SX322001。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 5.0, revision V5.0 / A-2; object `4e081d154738d8df9c7f9b20b12131610b5c3808e06bbec37f00ee89b35e9fa7`.

#### Answer

根据检索到的手册证据（E2），

#### Model-assisted source review

Assessment: **not_assessable**; completeness: none; citation support: not_applicable; revision fidelity: mixed.

The generated answer is a fragment ('根据检索到的手册证据（E2），') and does not provide the requested explanation of the '巡检配置' (Inspection Configuration) shortcut. It fails to state the operation method, purpose, or result (e.g., judging if the robot stops, matching inspection numbers with welding points, or IO definitions) required by the question and expected facts. Although the supplied evidence (E2) contains relevant alarm codes (7913, 7961) and mentions '快捷键' (shortcuts), it does not contain the specific descriptive text for the '巡检配置' shortcut found in the expected reference (V4.0/A-0). The answer does not constitute a valid refusal or abstention because it does not explicitly state that the information is unavailable; it simply cuts off. Therefore, it is not a complete answer, nor is it a proper abstention.

...

#### Answer

The retrieved manual evidence (E1–E8) does not contain any reference to '埃姆哈特-前台作业' (Emhart – Front-end Operation) or a '工艺作业模版' (Process Operation Template) section that defines this term. E1/E2 only show the document title page; E3 lists alarm codes (7901–7964) for stud welding; E4 shows a generic I/O program template (NOP, IF IN#xx, OUT OT#xx, WAIT, DELAY, GOTO, END) without naming any Emhart-specific front-end operation; E5–E8 belong to other application manuals (grinding, palletizing, arc welding, laser welding). No evidence describes the operation method, purpose, or expected result of '埃姆哈特-前台作业'.

### sample_stud_welding_7f5ad54e6a_20

根据《SX322013《新松机器人螺柱焊应用操作手册》(A-2)》（V5.0 / A-2），在“IO配置”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- **新松机器人螺柱焊应用操作手册**（文档标题/章节名，来源文件：SX322013《新松机器人螺柱焊应用操作手册》(A-2)，位于产品手册/软件类/V5.0版本目录）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 13–15; corpus 5.0, revision V5.0 / A-2; object `3580d5a8c5116a37b6bf8e2524d4a1e2392c9979a45ca913b8f10e6e1e9c6ad3`.

- 该表格行（索引 2）包含一个单元格，内容为“新松机器人螺柱焊应用操作手册”。该行非表头，无跨行或跨列继承值，仅作为该文档标题的独立条目出现。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 13–15; corpus 4.0, revision V5.0 / A-2; object `e95fa8a4686c28e6afc1c47f932b1b784a37f553e6ca0add76b9590b1221076f`.

- 该表行（第14行）包含三列数据：第一列为空，第二列和第三列均为“L3”。此记录来自《新松机器人螺柱焊应用操作手册》(SX322013) 的表格结构，用于描述特定配置或等级参数。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322013《新松机器人螺柱焊应用操作手册》(A-2)/SX322013《新松机器人螺柱焊应用操作手册》(A-2).md`, lines 428–432; corpus 5.0, revision V5.0 / A-2; object `d9fd32fe736d9e26cd53f838632fab473251c095bc4991da5014144c6b86cc4e`.

- 该表格行来自《新松机器人点焊应用操作手册》(SX322007)，描述了一条指令配置：步骤序号 **2** 对应事件 **焊接开始**，触发动作为 **输出IO**，参数值为 **1**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1380–1385; corpus 4.0, revision V5.0 / A-2; object `0b7f2934df81fae53b8121def6d49422f1846ac73e21e3c24f0f9b9b596ff997`.

- **摘要：**
该行定义了新松机器人弧焊应用手册中的一个参数：**“焊接IO输出”**。其配置要求为：**需根据实际IO模块的接线情况以及所使用到的具体功能，填写对应的端口号。**

**关键信息点：**
*   **参数名称**：焊接IO输出
*   **填写依据**：IO模块接线方式 + 实际使用的功能
*   **填写内容**：具体的端口号
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 1908–1911; corpus 4.0, revision V5.0 / B-5; object `63633ffc8bb767a7133cc357e812644f667069150b96e33d8a3be313adcafcb0`.

- 该表格行（索引 2，来自《新松机器人点焊应用操作手册》）记录了第 1 项配置：功能类型为“焊接/试验”，对应的信号为“输出 IO”，其参数值设为“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1374–1379; corpus 5.0, revision V5.0 / A-2; object `dc01c72c705d306b3f413f83037cc36cead70c548d40d43507d037f2deb1334e`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 表格第3行内容为：SX322001《新松机器人通用操作手册》。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人码垛应用操作手册》(C-1)/SX322002《新松机器人码垛应用操作手册》(C-1).md`, lines 248–250; corpus 4.0, revision V5.0 / C-1; object `fc6d3386c97bf91282081b2c968d0c0f3d3be65486fba39d2c5d568b80bf28a9`.

#### Answer

The retrieved evidence from SX322013《新松机器人螺柱焊应用操作手册》(A-2) (V5.0 / A-2) only contains the document title page (E1, E2) and a program-logic table for riveting-count checking (E3). None of the provided excerpts include the 

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1, E2, E3) does not contain the specific 'IO配置' or '路径' details requested. It accurately reflects the limitation of the provided context without hallucinating information. The expected fact '路径：螺柱焊->IO设置' is not present in the supplied evidence snippets, making the abstention appropriate.

## Spr Riveting — sampled manual revisions

40 cases from 2 CSV files. Set `SPR_RIVETING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
SPR_RIVETING_SAMPLE_CASES = [{'id': 'sample_spr_riveting_a598e00732_01', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“基础配置”中，应如何理解或操作“柜体型号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 柜体型号：根据现场柜体实际型号配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 136, 'line_end': 136, 'excerpt': '1. 柜体型号：根据现场柜体实际型号配置。', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_02', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.4', 'section_title': '2.2.4 SPR实现铆枪连接和断开', 'heading_path': ['2 指令、快捷键说明', '2.2 SPR指令集', '2.2.4 SPR实现铆枪连接和断开'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），配置“SPR实现铆枪连接和断开”时，“<参数1>：铆枪号； <参数2>：控制器号； <参数3>：连接或断开；Couple/ DeCou”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 <参数1>：铆枪号', '<参数2>：控制器号', '<参数3>：连接或断开'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 920, 'line_end': 923, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>&lt;参数1&gt;：铆枪号；</p><p>&lt;参数2&gt;：控制器号；</p><p>&lt;参数3&gt;：连接或断开；Couple/ DeCouple</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_03', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.2', 'section_title': '4.2 固极', 'heading_path': ['4 工艺作业模版', '4.2 固极'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“固极”中，应如何理解或操作“OUT OT#1= OFF;”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['OUT OT#1= OFF;'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2458, 'line_end': 2462, 'excerpt': '<tr>\n<td></td>\n<td><p>OUT OT&#35;1= OFF;</p></td>\n<td></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_04', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），配置“常见报警及解决方法”时，“报警：10012”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['报警：10012 配置IO不在逻辑IO分配范围内 检查IO配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2617, 'line_end': 2623, 'excerpt': '<tr>\n<td><p>报警：10012</p></td>\n<td><p>配置IO不在逻辑IO分配范围内</p></td>\n<td></td>\n<td></td>\n<td><p>检查IO配置</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_05', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.2', 'section_title': '3.2 防漏铆的基本作业流程', 'heading_path': ['3 模板作业', '3.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“防漏铆的基本作业流程”中，应如何理解或操作“MOVL VL=20”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['MOVL VL=20 运动到SPR铆接位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 1512, 'line_end': 1515, 'excerpt': '<tr>\n<td><p>MOVL VL=20</p></td>\n<td><p>运动到SPR铆接位置</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_06', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警：10005”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警：10005 主机有报警 检查铆机'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2568, 'line_end': 2574, 'excerpt': '<tr>\n<td><p>报警：10005</p></td>\n<td><p>主机有报警</p></td>\n<td></td>\n<td></td>\n<td><p>检查铆机</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_07', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.2', 'section_title': '4.2 固极', 'heading_path': ['4 工艺作业模版', '4.2 固极'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），“固极”中“OUT OT#840= ON;”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'OT#840'], 'expected_facts': ['OUT OT#840= ON; 输出铆接指令为ON'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2288, 'line_end': 2292, 'excerpt': '<tr>\n<td></td>\n<td><p>OUT OT&#35;840= ON;</p></td>\n<td><p>输出铆接指令为ON</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_08', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.7', 'section_title': '1.7 实现显示', 'heading_path': ['1 界面配置', '1.7 实现显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），配置“实现显示”时，“SPR实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['SPR实时显示功能，可查看SPR的相关参数，如通讯状态、枪号、程序号等参数', '可在本地和远程等模式下进行查看，界面如下所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 260, 'line_end': 260, 'excerpt': 'SPR实时显示功能，可查看SPR的相关参数，如通讯状态、枪号、程序号等参数；可在本地和远程等模式下进行查看，界面如下所示：', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_09', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.6', 'section_title': '2.2.6 填充铆钉控制指令', 'heading_path': ['2 指令、快捷键说明', '2.2 SPR指令集', '2.2.6 填充铆钉控制指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），“填充铆钉控制指令”中“SPR填充铆钉控制指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SPR'], 'expected_facts': ['SPR填充铆钉控制指令 功能 指令实现开始装填铆钉'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 1206, 'line_end': 1210, 'excerpt': '<tr>\n<td rowspan="4"><p><a id="_Hlk157940343"></a>SPR填充铆钉控制指令</p></td>\n<td><p>功能</p></td>\n<td><p>指令实现开始装填铆钉；</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_10', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.3', 'section_title': '2.2.3 SPR实现铆枪开口位置', 'heading_path': ['2 指令、快捷键说明', '2.2 SPR指令集', '2.2.3 SPR实现铆枪开口位置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“SPR实现铆枪开口位置”中，应如何理解或操作“USERALM CODE=2601”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['USERALM CODE=2601 报警：没设置开口'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 874, 'line_end': 878, 'excerpt': '<tr>\n<td></td>\n<td><p>USERALM CODE=2601</p></td>\n<td><p>报警：没设置开口</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_11', 'category': 'spr_riveting', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.5', 'section_title': '1.5 IO设置', 'heading_path': ['1 界面配置', '1.5 IO设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），操作“IO设置”时，涉及“根据Dnet IO映射起始位和长度以及铆机协议内容”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['根据Dnet IO映射起始位和长度以及铆机协议内容，配置IO号', 'ATLAS铆机的输入和输出IO，是根据“4321”的顺序进行排布，不是常规的“1234”，请注意！'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 226, 'line_end': 226, 'excerpt': '根据Dnet IO映射起始位和长度以及铆机协议内容，配置IO号。**ATLAS铆机的输入和输出IO，是根据“4321”的顺序进行排布，不是常规的“1234”，请注意！**', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_12', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4', 'section_title': '1.4 工作模式参数配置', 'heading_path': ['1 界面配置', '1.4 工作模式参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），“工作模式参数配置”中“铆点计数”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 铆点计数：主要用于防漏铆', '该值为当前记录的铆点数量，用于与初始化指令设置的铆点总数比较，判断是否多铆或漏铆', '事例详见作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 202, 'line_end': 202, 'excerpt': '1. 铆点计数：主要用于防漏铆。该值为当前记录的铆点数量，用于与初始化指令设置的铆点总数比较，判断是否多铆或漏铆。事例详见作业。史丹利铆机分多枪，最多四个铆枪。', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_13', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“报警：10028”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警：10028 提前吹钉ID号与当前ID号不一致 修改下一个顶号与下一条指令的顶号一致'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2729, 'line_end': 2735, 'excerpt': '<tr>\n<td><p>报警：10028</p></td>\n<td><p>提前吹钉ID号与当前ID号不一致</p></td>\n<td></td>\n<td></td>\n<td><p>修改下一个顶号与下一条指令的顶号一致</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_14', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据手册“SX322014《新松机器人SPR铆接功能使用手册》-A-0.md”（V4.0 / A-0），要在“1.1 基础配置”中配置“型号配置”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：系统设置->型号配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 134, 'line_end': 134, 'excerpt': '配置路径：系统设置->型号配置', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_15', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2.5', 'section_title': '2.2.5 SPR实现装钉站装钉和填充站检', 'heading_path': ['2 指令、快捷键说明', '2.2 SPR指令集', '2.2.5 SPR实现装钉站装钉和填充站检'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），配置“SPR实现装钉站装钉和填充站检”时，“<参数1>：有效/无效；ActivateOn/ActivateOff，固极和ATLAS写入用户整”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 <参数1>：有效/无效', 'ActivateOn/ActivateOff，固极和ATLAS写入用户整型变量442', '<参数2>：检测/不检测; Check/ NoCheck，固极和ATLAS写入用户整型变量443'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 949, 'line_end': 952, 'excerpt': '<tr>\n<td><p>说明</p></td>\n<td><p>&lt;参数1&gt;：有效/无效；ActivateOn/ActivateOff，固极和ATLAS写入用户整型变量442；  </p><p>&lt;参数2&gt;：检测/不检测; Check/ NoCheck，固极和ATLAS写入用户整型变量443；  </p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_16', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.1', 'section_title': '4.1 Atlas', 'heading_path': ['4 工艺作业模版', '4.1 Atlas'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“Atlas”中，应如何理解或操作“IF I435> 80 GOTOL24;”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['IF I435> 80 GOTOL24; 用户整型变量435大于80，跳转L24,报开口设置过大'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 1692, 'line_end': 1696, 'excerpt': '<tr>\n<td></td>\n<td><p>IF I435&gt; 80 GOTOL24;</p></td>\n<td><p>用户整型变量435大于80，跳转L24,报开口设置过大。</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_17', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3.2', 'section_title': '3.2 防漏铆的基本作业流程', 'heading_path': ['3 模板作业', '3.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“防漏铆的基本作业流程”中，应如何理解或操作“SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=102 NEXTPROGNo =103 GUNOPEN=40; 运动到SPR铆接位置P1，铆枪2进行铆接'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 1516, 'line_end': 1519, 'excerpt': '<tr>\n<td><p>SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=102 NEXTPROGNo =103 GUNOPEN=40;</p></td>\n<td><p>运动到SPR铆接位置P1，铆枪2进行铆接。</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_18', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“提示：10065”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['提示：10065 快捷键 0 开枪完成'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2856, 'line_end': 2862, 'excerpt': '<tr>\n<td rowspan="9"><p>提示：10065</p></td>\n<td rowspan="9"><p>快捷键</p></td>\n<td><p>0</p></td>\n<td><p>开枪完成</p></td>\n<td></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_19', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4.2', 'section_title': '4.2 固极', 'heading_path': ['4 工艺作业模版', '4.2 固极'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），“固极”中“OUT OT#840 = OFF;”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'OT#840'], 'expected_facts': ['OUT OT#840 = OFF; 输出铆接指令为OFF'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 2303, 'line_end': 2307, 'excerpt': '<tr>\n<td></td>\n<td><p>OUT OT&#35;840 = OFF;</p></td>\n<td><p>输出铆接指令为OFF</p></td>\n</tr>', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_a598e00732_20', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4', 'section_title': '1.4 工作模式参数配置', 'heading_path': ['1 界面配置', '1.4 工作模式参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），配置“工作模式参数配置”时，“路径”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['路径：SPR->工作参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'line_start': 196, 'line_end': 196, 'excerpt': '路径：SPR->工作参数；', 'sha256': 'b7b8ac11dc1bcd9a4e66523f8884dc346847f27847d2bb2ef31f3c2f06c8f286'}]}, {'id': 'sample_spr_riveting_9950195c71_01', 'category': 'spr_riveting', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.5', 'section_title': '1.5 IO设置', 'heading_path': ['1 SPR应用配置', '1.5 IO设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），操作“IO设置”时，涉及“根据Dnet IO映射起始位和长度以及铆机协议内容”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['根据Dnet IO映射起始位和长度以及铆机协议内容，配置IO号', 'ATLAS铆机的输入和输出IO，是根据“4321”的顺序进行排布，不是常规的“1234”，请注意！'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 169, 'line_end': 169, 'excerpt': '根据Dnet IO映射起始位和长度以及铆机协议内容，配置IO号。**ATLAS铆机的输入和输出IO，是根据“4321”的顺序进行排布，不是常规的“1234”，请注意！**', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_02', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.2', 'section_title': '4.2 固极', 'heading_path': ['4 工艺作业模版', '4.2 固极'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“固极”中，应如何理解或操作“其作业需要调用“SPRCONNECT””？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['其作业需要调用“SPRCONNECT”，该子作业进行使用铆接流程'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 436, 'line_end': 436, 'excerpt': '其作业需要调用“SPRCONNECT”，该子作业进行使用铆接流程。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_03', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“常见报警及解决方法”中，应如何理解或操作“用户自定义报警-ATLAS”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 用户自定义报警-ATLAS'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 452, 'line_end': 452, 'excerpt': '1. 用户自定义报警-ATLAS', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_04', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），配置“快捷键”时，“铆枪号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能', '根据配置的范围，可以配置从1-16'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 389, 'line_end': 389, 'excerpt': '2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能。根据配置的范围，可以配置从1-16。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_05', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '3.2', 'section_title': '3.2 防漏铆的基本作业流程', 'heading_path': ['3 模板作业', '3.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322014《新松机器人SPR铆接应用操作手册》(A-1).md”（V5.0 / A-1），作业模板中A13_SPR、MOVL、SPRL的执行顺序和作用是什么？', 'anchors': ['A13_SPR', 'MOVL', 'SPRL'], 'expected_facts': ['A13_SPR Init Gun=1 RCNo =1 Count=10 SPR初始化', 'MOVL VL=20 安全点', 'SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=100 NEXTPROGNo =101 GUNOPEN=40; 运动到SPR铆接位置P1，铆枪1进行铆接。', 'MOVL VL=20 运动到安全位置', 'SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=101 NEXTPROGNo =102 GUNOPEN=40; 运动到SPR铆接位置P2，铆枪1进行铆接。', 'MOVL VL=20 运动到安全位置', 'A13_SPR Init Gun=2 RCNo =1 Count=4 SPR初始化', 'MOVL VL=20 运动到SPR铆接位置', 'SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=102 NEXTPROGNo =103 GUNOPEN=40; 运动到SPR铆接位置P1，铆枪2进行铆接。', 'MOVL VL=20 运动到安全位置', 'SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=103 NEXTPROGNo =104 GUNOPEN=40; 运动到SPR铆接位置P2，铆枪2进行铆接。', 'MOVL VL=20 运动到安全位置', 'MOVL VL=20 运动到SPR铆接位置', 'SPRL P3 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=104 NEXTPROGNo =105 GUNOPEN=40; 运动到SPR铆接位置P3，铆枪2进行铆接。', 'MOVL VL=20 运动到安全位置', 'MOVL VL=20 运动到下一个SPR铆接位置', 'SPRL P4 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=105 NEXTPROGNo =106 GUNOPEN=40; 运动到SPR铆接位置P4，铆枪2进行铆接。', 'MOVL VL=20 运动到安全位置', 'A13_SPR ProcessCheck 防漏铆检测'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 410, 'line_end': 410, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>A13_SPR Init Gun=1 RCNo =1 Count=10</p></td><td><p>SPR初始化</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>安全点</p></td></tr><tr><td><p>SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=100 NEXTPROGNo =101 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P1，铆枪1进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=101 NEXTPROGNo =102 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P2，铆枪1进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>A13_SPR Init Gun=2 RCNo =1 Count=4</p></td><td><p>SPR初始化</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到SPR铆接位置</p></td></tr><tr><td><p>SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=102 NEXTPROGNo =103 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P1，铆枪2进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=103 NEXTPROGNo =104 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P2，铆枪2进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到SPR铆接位置</p></td></tr><tr><td><p>SPRL P3 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=104 NEXTPROGNo =105 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P3，铆枪2进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到下一个SPR铆接位置</p></td></tr><tr><td><p>SPRL P4 VL =200 ACC =100 CNT =0 ET=1 GUN=2 RCNo=1 PROGNo=105 NEXTPROGNo =106 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P4，铆枪2进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>A13_SPR ProcessCheck</p></td><td><p>防漏铆检测</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_06', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.4', 'section_title': '1.4 工作模式参数配置', 'heading_path': ['1 SPR应用配置', '1.4 工作模式参数配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），“工作模式参数配置”中“铆点计数”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['1. 铆点计数：主要用于防漏铆', '该值为当前记录的铆点数量，用于与初始化指令设置的铆点总数比较，判断是否多铆或漏铆', '事例详见作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 145, 'line_end': 145, 'excerpt': '1. 铆点计数：主要用于防漏铆。该值为当前记录的铆点数量，用于与初始化指令设置的铆点总数比较，判断是否多铆或漏铆。事例详见作业。史丹利铆机分多枪，最多四个铆枪。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_07', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.2', 'section_title': '4.2 固极', 'heading_path': ['4 工艺作业模版', '4.2 固极'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322014《新松机器人SPR铆接应用操作手册》(A-1).md”（V5.0 / A-1），涂胶流程模板在冲洗或填充之前如何检查运行模式和胶桶状态，胶量不足时怎样处理？', 'anchors': ['USERALM'], 'expected_facts': ['USERALM CODE =2602; 用户自定义报警：铆机系统不正常', 'USERALM CODE =2603; 报警：铆机未自动', 'USERALM CODE =2607; 报警：铆机系统未准备', 'USERALM CODE=2600 报警：开口设置过大', 'USERALM CODE=2601 报警：没设置开口', 'USERALM CODE=2608 报警：铆接失败'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 446, 'line_end': 446, 'excerpt': '<table><tr><td></td><td><p>NOP</p></td><td><p>开始</p></td></tr><tr><td></td><td><p>IF IN#818 == OFF GOTO L1;</p></td><td><p>等待输入系统正常信号为OFF，跳转至L1</p></td></tr><tr><td></td><td><p>IF IN#819 == OFF GOTO L2;</p></td><td><p>等待自动模式为OFF，跳转至L2</p></td></tr><tr><td></td><td><p>IF IN#827 == OFF GOTO L4;</p></td><td><p>等待系统准备为OFF，跳转至L4</p></td></tr><tr><td></td><td><p>GOTO L3;</p></td><td><p>没收到无故障信号，跳转至L3;</p></td></tr><tr><td></td><td><p>L1</p></td><td><p>L1</p></td></tr><tr><td></td><td><p>USERALM CODE =2602;</p></td><td><p>用户自定义报警：铆机系统不正常</p></td></tr><tr><td></td><td><p>L2</p></td><td><p>L2</p></td></tr><tr><td></td><td><p>USERALM CODE =2603;</p></td><td><p>报警：铆机未自动</p></td></tr><tr><td></td><td><p>L4</p></td><td><p>L4</p></td></tr><tr><td></td><td><p>USERALM CODE =2607;</p></td><td><p>报警：铆机系统未准备</p></td></tr><tr><td></td><td><p>L3</p></td><td><p>L3</p></td></tr><tr><td></td><td><p>OUT OT#1= ON;</p></td><td></td></tr><tr><td></td><td><p>IF I435&gt; 80 GOTO L24;</p></td><td><p>用户整型变量435大于80，跳转L24,报开口设置过大。</p></td></tr><tr><td></td><td><p>IF I435&gt; 40 GOTO L21;</p></td><td><p>用户整型变量435大于40，跳转L21。</p></td></tr><tr><td></td><td><p>IF I435&gt; 20 GOTO L22;</p></td><td><p>用户整型变量435大于20，跳转L22。</p></td></tr><tr><td></td><td><p>IF I435&gt; 0 GOTO L23;</p></td><td><p>用户整型变量435大于0，跳转L23。</p></td></tr><tr><td></td><td><p>GOTO L25;</p></td><td><p>否则，跳转至L25</p></td></tr><tr><td></td><td><p>L21</p></td><td><p>L21</p></td></tr><tr><td></td><td><p>OUT OT#848 = ON;</p></td><td><p>输出半开口80,置为ON</p></td></tr><tr><td></td><td><p>OUT OT#845 = OFF;</p></td><td><p>输出至开口位置,置为ON</p></td></tr><tr><td></td><td><p>GOTO L101;</p></td><td><p>跳转至L101</p></td></tr><tr><td></td><td><p>L22</p></td><td><p>L22</p></td></tr><tr><td></td><td><p>OUT OT#847 = ON;</p></td><td><p>输出半开口40,置为ON</p></td></tr><tr><td></td><td><p>OUT OT#845 = OFF;</p></td><td><p>输出至开口位置,置为ON</p></td></tr><tr><td></td><td><p>GOTO L101;</p></td><td><p>跳转至L101</p></td></tr><tr><td></td><td><p>L23</p></td><td><p>L23</p></td></tr><tr><td></td><td><p>OUT OT#846 = ON;</p></td><td><p>输出半开口20,置为ON</p></td></tr><tr><td></td><td><p>OUT OT#845 = OFF;</p></td><td><p>输出至开口位置,置为ON</p></td></tr><tr><td></td><td><p>GOTO L101;</p></td><td><p>跳转至L101</p></td></tr><tr><td></td><td><p>L24</p></td><td><p>L24</p></td></tr><tr><td></td><td><p>USERALM CODE=2600</p></td><td><p>报警：开口设置过大</p></td></tr><tr><td></td><td><p>GOTO L101;</p></td><td><p>跳转至L101</p></td></tr><tr><td></td><td><p>L25</p></td><td><p>L25</p></td></tr><tr><td></td><td><p>USERALM CODE=2601</p></td><td><p>报警：没设置开口</p></td></tr><tr><td></td><td><p>GOTO L101;</p></td><td><p>跳转至L101</p></td></tr><tr><td></td><td><p>L101</p></td><td><p>L101</p></td></tr><tr><td></td><td><p>OUT OT#2= ON;</p></td><td></td></tr><tr><td></td><td><p>OUT OGH#55= I433;</p></td><td><p>将铆钉ID赋值到865位至880.</p></td></tr><tr><td></td><td><p>OUT OT#833= ON;</p></td><td><p>输出供钉请求置为ON</p></td></tr><tr><td></td><td><p>OUT OT#3= ON;</p></td><td></td></tr><tr><td></td><td><p>WAIT IN#834=ON T=1;</p></td><td><p>等待铆钉A输入为ON</p></td></tr><tr><td></td><td><p>OUT OT#833= OFF;</p></td><td><p>输出供钉请求置为OFF</p></td></tr><tr><td></td><td><p>OUT OT#850= ON;</p></td><td><p>输出铆钉确认请求为ON</p></td></tr><tr><td></td><td><p>OUT OT#840= ON;</p></td><td><p>输出铆接指令为ON</p></td></tr><tr><td></td><td><p>WAIT IN#840=ON T=1;</p></td><td><p>等待铆接完成信号为ON</p></td></tr><tr><td></td><td><p>OUT OT#850 = OFF;</p></td><td><p>输出铆钉确认请求为OFF</p></td></tr><tr><td></td><td><p>OUT OT#840 = OFF;</p></td><td><p>输出铆接指令为OFF</p></td></tr><tr><td></td><td><p>OUT OGH#55 = 0;</p></td><td><p>将铆钉ID赋值为0</p></td></tr><tr><td></td><td><p>L50</p></td><td><p>L50</p></td></tr><tr><td></td><td><p>IF OT#822 == ON GOTO L51;</p></td><td></td></tr><tr><td></td><td><p>IF IN#859 == ON GOTO L51;</p></td><td><p>等待输入锚点OK，跳转至L51</p></td></tr><tr><td></td><td><p>IF IN#858 == ON GOTO L52;</p></td><td><p>等待输入锚点NOK，跳转至L52</p></td></tr><tr><td></td><td><p>GOTO L50</p></td><td><p>跳转至L50</p></td></tr><tr><td></td><td><p>L51</p></td><td><p>L51</p></td></tr><tr><td></td><td><p>INC I408</p></td><td><p>铆钉总数减一</p></td></tr><tr><td></td><td><p>OUT OT#4= ON;</p></td><td></td></tr><tr><td></td><td><p>GOTO L99</p></td><td><p>跳转至L99</p></td></tr><tr><td></td><td><p>L52</p></td><td><p>L52</p></td></tr><tr><td></td><td><p>OUT OT#840= OFF;</p></td><td></td></tr><tr><td></td><td><p>USERALM CODE=2608</p></td><td><p>报警：铆接失败</p></td></tr><tr><td></td><td><p>PULSE OT#824=ON T=1;</p></td><td></td></tr><tr><td></td><td><p>DELAY T=4;</p></td><td></td></tr><tr><td></td><td><p>L99</p></td><td><p>L99</p></td></tr><tr><td></td><td><p>WAIT IN#846=ON T=1;</p></td><td><p>等待输入半开口为ON</p></td></tr><tr><td></td><td><p>L678</p></td><td></td></tr><tr><td></td><td><p>OUT OT#848 = OFF;</p></td><td><p>输出半开口80为OFF</p></td></tr><tr><td></td><td><p>OUT OT#847 = OFF;</p></td><td><p>输出半开口40为OFF</p></td></tr><tr><td></td><td><p>OUT OT#846 = OFF;</p></td><td><p>输出半开口20为OFF</p></td></tr><tr><td></td><td><p>IF I434==0 GOTO L76</p></td><td><p>如果下一个铆钉ID为0，跳转至L76</p></td></tr><tr><td></td><td><p>OUT OGH#55 = I434;</p></td><td><p>将下一个铆钉ID赋值给865位至880.</p></td></tr><tr><td></td><td><p>OUT OT#833 = ON;</p></td><td><p>输出供钉请求信号</p></td></tr><tr><td></td><td><p>GOTO L999</p></td><td></td></tr><tr><td></td><td><p>L76</p></td><td><p>L76</p></td></tr><tr><td></td><td><p>OUT OT#845 = ON;</p></td><td><p>输出至开口位置为ON</p></td></tr><tr><td></td><td><p>WAIT IN#845=ON T=1;</p></td><td><p>等待输入全开口为ON</p></td></tr><tr><td></td><td><p>OUT OT#845 = OFF;</p></td><td><p>输出至开口位置为OFF</p></td></tr><tr><td></td><td><p>L999</p></td><td></td></tr><tr><td></td><td><p>OUT OT#1= OFF;</p></td><td></td></tr><tr><td></td><td><p>OUT OT#2= OFF;</p></td><td></td></tr><tr><td></td><td><p>OUT OT#3= OFF;</p></td><td></td></tr><tr><td></td><td><p>OUT OT#4= OFF;</p></td><td></td></tr><tr><td></td><td><p>RET</p></td><td><p>返回</p></td></tr><tr><td></td><td><p>END</p></td><td><p>结束</p></td></tr></table>', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_08', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“点击示教器OP1键”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击示教器OP1键，可弹出快捷键应用配置界面：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 380, 'line_end': 380, 'excerpt': '点击示教器OP1键，可弹出快捷键应用配置界面：', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_09', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），“常见报警及解决方法”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['HOME', 'SPR'], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法 报警：10001 铆接前准备异常 0 等待铆接信号异常 1、观察外部接线是否有损坏或者未正确接线 2、是否在铆接过程找中，开启暂停导致', '1 报错 请检查铆接设备状态 2 铆接完成信号已置位 检查铆接完成信号 3 铆机系统不正常 请检查铆机状态 4 设备未自动 铆机未设置自动状态 5 设备未准备 铆机没有准备好，请等待！ 报警：10002 参数超界 0 开口位置超界 检查铆接指令参数 1 铆钉ID号超界 检查铆接指令参数 2 下一个铆钉ID号超界 检查铆接指令参数 3 工具号超界 检查铆接指令参数 报警：10003 铆接超时 检查铆机报警 报警：10004 铆接完成未复位 检查铆枪退回到位信号 报警：10005 主机有报警 检查铆机 报警：10006 未回到HOME位 手动机器人恢复到home位 报警：10007 实际开口位置小于设定值 报警：10008 铆点存在漏铆 子码为铆枪号 报警：10009 铆点存在多铆 子码为铆枪号 报警：10010 无效铆点检测 报警：10011 文件操作失败 报警：10012 配置IO不在逻辑IO分配范围内 检查IO配置 报警：10013 机器人干涉 检查干涉区 报警：10014 SPR指令巡检暂停 请检查PLC设备状态 报警：10015 响应安全门请求暂停中 请检查安全门状态 报警：10016 设备急停 检查铆机状态 报警：10017 水流量异常 请检查实际流量与设定流量值 报警：10018 气流量异常 请检查气压表状态 报警：10019 SPR设备急停 请检查铆机状态 报警：10020 SPR指令集错误 请检查作业名称 报警：10021 SPR-NT151监控异常 请检查NT151模块功能 报警：10022 开口位置过大 请修改开口值小于80 报警：10023 供钉异常 请检查铆机状态 报警：10024 铆接失败 请检查铆机状态 报警：10025 开口异常 请检查铆机状态 报警：10026 充钉未完成 请检查铆机状态 报警：10027 允许贴合信号异常 请检查铆机状态 报警：10028 提前吹钉ID号与当前ID号不一致 修改下一个顶号与下一条指令的顶号一致 报警：10029 通讯状态异常 请检查相应的外设设备的通讯线或模块是否正常 提示：10051 当前为非NORMAL模式 提示：10052 SPR铆接清空堆栈完成 提示：10053 不支持1.0算法 提示：10054 铆点数量正常 无漏铆 提示：10055 漏铆个数为: 提示：10056 多铆个数为: 提示：10057 执行SPR铆枪开枪快捷功能 提示：10058 执行SPR铆枪关枪快捷功能 提示：10059 执行SPR铆枪出钉快捷功能 提示：10060 SPR铆枪打开 0 执行SPR铆枪开口大小为20 1 执行SPR铆枪开口大小为40 2 执行SPR铆枪开口大小为60 3 执行SPR铆枪开口大小为80 提示：10061 巡检配置为0功能未启用 提示：10062 执行SPR铆枪激活电机 提示：10063 执行SPR铆枪关闭电机 提示：10064 通讯为屏蔽状态 提示：10065 快捷键 0 开枪完成 1 关枪完成 2 开口20执行完成 3 开口40执行完成 4 开口60执行完成 5 开口80执行完成 6 激活电机执行完成 7 激活电机执行异常 8 铆枪出钉执行完成'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 450, 'line_end': 450, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码报警信息</p></td><td><p>解决方法</p></td></tr><tr><td rowspan="6"><p>报警：10001</p></td><td rowspan="6"><p>铆接前准备异常</p></td><td><p>0</p></td><td><p>等待铆接信号异常</p></td><td><p>1、观察外部接线是否有损坏或者未正确接线</p><p>2、是否在铆接过程找中，开启暂停导致。</p></td></tr><tr><td><p>1</p></td><td><p>报错</p></td><td><p>请检查铆接设备状态</p></td></tr><tr><td><p>2</p></td><td><p>铆接完成信号已置位</p></td><td><p>检查铆接完成信号</p></td></tr><tr><td><p>3</p></td><td><p>铆机系统不正常</p></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>4</p></td><td><p>设备未自动</p></td><td><p>铆机未设置自动状态</p></td></tr><tr><td><p>5</p></td><td><p>设备未准备</p></td><td><p>铆机没有准备好，请等待！</p></td></tr><tr><td rowspan="4"><p>报警：10002</p></td><td rowspan="4"><p>参数超界</p></td><td><p>0</p></td><td><p>开口位置超界</p></td><td><p>检查铆接指令参数</p></td></tr><tr><td><p>1</p></td><td><p>铆钉ID号超界</p></td><td><p>检查铆接指令参数</p></td></tr><tr><td><p>2</p></td><td><p>下一个铆钉ID号超界</p></td><td><p>检查铆接指令参数</p></td></tr><tr><td><p>3</p></td><td><p>工具号超界</p></td><td><p>检查铆接指令参数</p></td></tr><tr><td><p>报警：10003</p></td><td><p>铆接超时</p></td><td></td><td></td><td><p>检查铆机报警</p></td></tr><tr><td><p>报警：10004</p></td><td><p>铆接完成未复位</p></td><td></td><td></td><td><p>检查铆枪退回到位信号</p></td></tr><tr><td><p>报警：10005</p></td><td><p>主机有报警</p></td><td></td><td></td><td><p>检查铆机</p></td></tr><tr><td><p>报警：10006</p></td><td><p>未回到HOME位</p></td><td></td><td></td><td><p>手动机器人恢复到home位</p></td></tr><tr><td><p>报警：10007</p></td><td><p>实际开口位置小于设定值</p></td><td></td><td></td><td></td></tr><tr><td><p>报警：10008</p></td><td><p>铆点存在漏铆</p></td><td></td><td><p>子码为铆枪号</p></td><td></td></tr><tr><td><p>报警：10009</p></td><td><p>铆点存在多铆</p></td><td></td><td><p>子码为铆枪号</p></td><td></td></tr><tr><td><p>报警：10010</p></td><td><p>无效铆点检测</p></td><td></td><td></td><td></td></tr><tr><td><p>报警：10011</p></td><td><p>文件操作失败</p></td><td></td><td></td><td></td></tr><tr><td><p>报警：10012</p></td><td><p>配置IO不在逻辑IO分配范围内</p></td><td></td><td></td><td><p>检查IO配置</p></td></tr><tr><td><p>报警：10013</p></td><td><p>机器人干涉</p></td><td></td><td></td><td><p>检查干涉区</p></td></tr><tr><td><p>报警：10014</p></td><td><p>SPR指令巡检暂停</p></td><td></td><td></td><td><p>请检查PLC设备状态</p></td></tr><tr><td><p>报警：10015</p></td><td><p>响应安全门请求暂停中</p></td><td></td><td></td><td><p>请检查安全门状态</p></td></tr><tr><td><p>报警：10016</p></td><td><p>设备急停</p></td><td></td><td></td><td><p>检查铆机状态</p></td></tr><tr><td><p>报警：10017</p></td><td><p>水流量异常</p></td><td></td><td></td><td><p>请检查实际流量与设定流量值</p></td></tr><tr><td><p>报警：10018</p></td><td><p>气流量异常</p></td><td></td><td></td><td><p>请检查气压表状态</p></td></tr><tr><td><p>报警：10019</p></td><td><p>SPR设备急停</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10020</p></td><td><p>SPR指令集错误</p></td><td></td><td></td><td><p>请检查作业名称</p></td></tr><tr><td><p>报警：10021</p></td><td><p>SPR-NT151监控异常</p></td><td></td><td></td><td><p>请检查NT151模块功能</p></td></tr><tr><td><p>报警：10022</p></td><td><p>开口位置过大</p></td><td></td><td></td><td><p>请修改开口值小于80</p></td></tr><tr><td><p>报警：10023</p></td><td><p>供钉异常</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10024</p></td><td><p>铆接失败</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10025</p></td><td><p>开口异常</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10026</p></td><td><p>充钉未完成</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10027</p></td><td><p>允许贴合信号异常</p></td><td></td><td></td><td><p>请检查铆机状态</p></td></tr><tr><td><p>报警：10028</p></td><td><p>提前吹钉ID号与当前ID号不一致</p></td><td></td><td></td><td><p>修改下一个顶号与下一条指令的顶号一致</p></td></tr><tr><td><p>报警：10029</p></td><td><p>通讯状态异常</p></td><td></td><td></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>提示：10051</p></td><td><p>当前为非NORMAL模式</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10052</p></td><td><p>SPR铆接清空堆栈完成</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10053</p></td><td><p>不支持1.0算法</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10054</p></td><td><p>铆点数量正常</p></td><td></td><td></td><td><p>无漏铆</p></td></tr><tr><td><p>提示：10055</p></td><td><p>漏铆个数为:</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10056</p></td><td><p>多铆个数为:</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10057</p></td><td><p>执行SPR铆枪开枪快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10058</p></td><td><p>执行SPR铆枪关枪快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10059</p></td><td><p>执行SPR铆枪出钉快捷功能</p></td><td></td><td></td><td></td></tr><tr><td rowspan="4"><p>提示：10060</p></td><td rowspan="4"><p>SPR铆枪打开</p></td><td><p>0</p></td><td><p>执行SPR铆枪开口大小为20</p></td><td></td></tr><tr><td><p>1</p></td><td><p>执行SPR铆枪开口大小为40</p></td><td></td></tr><tr><td><p>2</p></td><td><p>执行SPR铆枪开口大小为60</p></td><td></td></tr><tr><td><p>3</p></td><td><p>执行SPR铆枪开口大小为80</p></td><td></td></tr><tr><td><p>提示：10061</p></td><td><p>巡检配置为0功能未启用</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10062</p></td><td><p>执行SPR铆枪激活电机</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10063</p></td><td><p>执行SPR铆枪关闭电机</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10064</p></td><td><p>通讯为屏蔽状态</p></td><td></td><td></td><td></td></tr><tr><td rowspan="9"><p>提示：10065</p></td><td rowspan="9"><p>快捷键</p></td><td><p>0</p></td><td><p>开枪完成</p></td><td></td></tr><tr><td><p>1</p></td><td><p>关枪完成</p></td><td></td></tr><tr><td><p>2</p></td><td><p>开口20执行完成</p></td><td></td></tr><tr><td><p>3</p></td><td><p>开口40执行完成</p></td><td></td></tr><tr><td><p>4</p></td><td><p>开口60执行完成</p></td><td></td></tr><tr><td><p>5</p></td><td><p>开口80执行完成</p></td><td></td></tr><tr><td><p>6</p></td><td><p>激活电机执行完成</p></td><td></td></tr><tr><td><p>7</p></td><td><p>激活电机执行异常</p></td><td></td></tr><tr><td><p>8</p></td><td><p>铆枪出钉执行完成</p></td><td></td></tr></table>', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_10', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.3', 'section_title': '1.3 铆机配置', 'heading_path': ['1 SPR应用配置', '1.3 铆机配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），配置“铆机配置”时，“选择SPR工艺的主机品牌”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['选择SPR工艺的主机品牌，默认为无铆机，工作模式参数和IO配置中的铆机IO输入和输出界面是根据铆机品牌显示不同的参数', '目前已经适配史丹利、ATLAS、固极、通用品牌'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 135, 'line_end': 135, 'excerpt': '选择SPR工艺的主机品牌，默认为无铆机，工作模式参数和IO配置中的铆机IO输入和输出界面是根据铆机品牌显示不同的参数。目前已经适配史丹利、ATLAS、固极、通用品牌。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_11', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.1', 'section_title': '2.1 SPR运动+铆接指令', 'heading_path': ['2 指令、快捷键说明', '2.1 SPR运动+铆接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），“SPR运动+铆接指令”中“其中史丹利铆机”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ATLAS', 'SPRCONNECT'], 'expected_facts': ['其中史丹利铆机，不需要调用子作业', 'ATLAS、固极品牌的铆机需要调用子作业 “SPRCONNECT”', '具体作业形式详见第4章工艺作业模版'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 239, 'line_end': 239, 'excerpt': '其中史丹利铆机，不需要调用子作业。ATLAS、固极品牌的铆机需要调用子作业 “SPRCONNECT”。具体作业形式详见第4章工艺作业模版。。选择“通用”方式不需要调用子作业，具体IO配置方式参考“IO设置”。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_12', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 SPR应用配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“基础配置”中，应如何理解或操作“总线设备”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 总线设备：根据现场实际总线设备配置节点及IO标志位'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 83, 'line_end': 83, 'excerpt': '1. 总线设备：根据现场实际总线设备配置节点及IO标志位。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_13', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.1', 'section_title': '4.1 Atlas', 'heading_path': ['4 工艺作业模版', '4.1 Atlas'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“Atlas”中，应如何理解或操作“涉及到机器人急停”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['涉及到机器人急停，需要从控制柜体中急停与设备相连，无需软件进行，具体问题联系电气工程师'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 428, 'line_end': 428, 'excerpt': '涉及到机器人急停，需要从控制柜体中急停与设备相连，无需软件进行，具体问题联系电气工程师。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_14', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“常见报警及解决方法”中，应如何理解或操作“用户自定义报警-固极”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 用户自定义报警-固极'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 456, 'line_end': 456, 'excerpt': '1. 用户自定义报警-固极', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_15', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '3.1', 'section_title': '3.1 基本作业流程', 'heading_path': ['3 模板作业', '3.1 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322014《新松机器人SPR铆接应用操作手册》(A-1).md”（V5.0 / A-1），作业模板中MOVL、SPRL、ACC的执行顺序和作用是什么？', 'anchors': ['MOVL', 'SPRL', 'ACC'], 'expected_facts': ['MOVL VL=20 安全点', 'SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=100 NEXTPROGNo =101 GUNOPEN=40; 运动到SPR铆接位置P1,进行铆接。', 'MOVL VL=20 运动到安全位置', 'SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo =1 PROGNo=100 NEXTPROGNo =101 GUNOPEN=40; 运动到SPR铆接位置P2,进行铆接。', 'MOVL VL=20 运动到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 404, 'line_end': 404, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>安全点</p></td></tr><tr><td><p>SPRL P1 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo=1 PROGNo=100\xa0NEXTPROGNo =101 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P1,进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>SPRL P2 VL =200 ACC =100 CNT =0 ET=1 GUN=1 RCNo =1 PROGNo=100 NEXTPROGNo =101 GUNOPEN=40;</p></td><td><p>运动到SPR铆接位置P2,进行铆接。</p></td></tr><tr><td><p>MOVL VL=20</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_16', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“巡检配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['9. 巡检配置：在执行运动+工艺点时判断机器人是否停止', '机器人巡检号与停止焊点号一致，机器人停止', '其中起始IO号的含义是：巡检功能起始IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 396, 'line_end': 396, 'excerpt': '9. 巡检配置：在执行运动+工艺点时判断机器人是否停止。机器人巡检号与停止焊点号一致，机器人停止。其中起始IO号的含义是：巡检功能起始IO号。输入巡检信号：巡检功能输入巡检信号。输出确认信号：巡检功能输出确认信号。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_17', 'category': 'spr_riveting', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.5', 'section_title': '1.5 IO设置', 'heading_path': ['1 SPR应用配置', '1.5 IO设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），配置“IO设置”时，“SPR工艺IO设置的起始位和长度”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['SPR工艺IO设置的起始位和长度，在逻辑IO分配中FDS应用配置的起始位和长度范围内,否则会报警', '针对SPR中的通用品牌，输入和输出的IO位配置0即可'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 151, 'line_end': 151, 'excerpt': 'SPR工艺IO设置的起始位和长度，在逻辑IO分配中FDS应用配置的起始位和长度范围内,否则会报警。针对SPR中的通用品牌，输入和输出的IO位配置0即可。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_18', 'category': 'spr_riveting', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2.5', 'section_title': '2.2.5 SPR实现装钉站装钉和填充站检', 'heading_path': ['2 指令、快捷键说明', '2.2 SPR指令集', '2.2.5 SPR实现装钉站装钉和填充站检'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），“SPR实现装钉站装钉和填充站检”中“SPR供钉系统控制指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SPR', 'A13_SPR'], 'expected_facts': ['SPR供钉系统控制指令 功能 指令实现装钉站预装铆钉，填充站状态检测', '格式 A13_SPR SPRPrepare ActivateOn <参数1> Check<参数2> ;注释内容 A13_SPR SPRPrepare ActivateOff <参数1> Check<参数2> ;注释内容 A13_SPR SPRPrepare ActivateOn <参数1> NoCheck<参数2> ;注释内容. 说明 <参数1>：有效/无效', 'ActivateOn/ActivateOff，固极和ATLAS写入用户整型变量442'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 324, 'line_end': 324, 'excerpt': '<table><tr><td rowspan="4"><p>SPR供钉系统控制指令</p></td><td><p>功能</p></td><td><p>指令实现装钉站预装铆钉，填充站状态检测；</p></td></tr><tr><td><p>格式</p></td><td><p>A13_SPR SPRPrepare ActivateOn &lt;参数1&gt; Check&lt;参数2&gt; ;注释内容</p><p>A13_SPR SPRPrepare ActivateOff &lt;参数1&gt; Check&lt;参数2&gt; ;注释内容</p><p>A13_SPR SPRPrepare ActivateOn &lt;参数1&gt; NoCheck&lt;参数2&gt; ;注释内容.</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：有效/无效；ActivateOn/ActivateOff，固极和ATLAS写入用户整型变量442；  </p><p>&lt;参数2&gt;：检测/不检测; Check/ NoCheck，固极和ATLAS写入用户整型变量443；  </p></td></tr><tr><td><p>举例</p></td><td><p>A13_SPR SPRPrepare ActivateOn Check ;注释内容 //发送预装钉信号，检测反馈信号；ActivateOff</p></td></tr></table>', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_19', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 SPR应用配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“基础配置”中，应如何理解或操作“总线IO映射”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 总线IO映射：根据设备实际情况配置输入和输出IO映射的通讯类型，总线ID，总线起始，IO起始字，长度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 87, 'line_end': 87, 'excerpt': '1. 总线IO映射：根据设备实际情况配置输入和输出IO映射的通讯类型，总线ID，总线起始，IO起始字，长度。', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}, {'id': 'sample_spr_riveting_9950195c71_20', 'category': 'spr_riveting', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4.1', 'section_title': '4.1 Atlas', 'heading_path': ['4 工艺作业模版', '4.1 Atlas'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“Atlas”中，应如何理解或操作“其作业需要调用“SPRCONNECT””？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['其作业需要调用“SPRCONNECT”，该子作业进行使用铆接流程;'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'line_start': 422, 'line_end': 422, 'excerpt': '其作业需要调用“SPRCONNECT”，该子作业进行使用铆接流程;', 'sha256': '6221185a25429db8666cb6df2ed6dfda8b742356211c58ee5da7b41738206d2c'}]}]
register_sample_bank('spr_riveting', SPR_RIVETING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md', 'seed': 8105448192162342470, 'population': 351, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(136, 136), (920, 923), (2458, 2462), (2617, 2623), (1512, 1515), (2568, 2574), (2288, 2292), (260, 260), (1206, 1210), (874, 878), (226, 226), (202, 202), (2729, 2735), (134, 134), (949, 952), (1692, 1696), (1516, 1519), (2856, 2862), (2303, 2307), (196, 196)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md', 'seed': 2731613309903269168, 'population': 103, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(169, 169), (436, 436), (452, 452), (389, 389), (410, 410), (145, 145), (446, 446), (380, 380), (450, 450), (135, 135), (239, 239), (83, 83), (428, 428), (456, 456), (404, 404), (396, 396), (151, 151), (324, 324), (87, 87), (422, 422)]}])


In [ ]:
# | notest
SPR_RIVETING_SAMPLE_IDS = None
spr_riveting_sample_cases = select_cases(SPR_RIVETING_SAMPLE_CASES, SPR_RIVETING_SAMPLE_IDS)
display(pd.DataFrame(spr_riveting_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in spr_riveting_sample_cases])
spr_riveting_sample_metrics = await evaluation.run(spr_riveting_sample_cases, concurrency=EVALUATION_CONCURRENCY)
spr_riveting_sample_results = {case["id"]: evaluation.load(case) for case in spr_riveting_sample_cases}
display(spr_riveting_sample_metrics)
# Inspect a complete result: evaluation.inspect(spr_riveting_sample_cases[0]["id"])
evaluation.show_answers(spr_riveting_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_spr_riveting_a598e00732_01,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,19476,0,True,unsupported,,,abstained
sample_spr_riveting_a598e00732_02,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,12519,0,True,included,,,incorrect
sample_spr_riveting_a598e00732_03,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,16019,0,True,included,,,not reviewed
sample_spr_riveting_a598e00732_04,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,21284,0,True,unsupported,,,not reviewed
sample_spr_riveting_a598e00732_05,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,10935,0,True,included,,,not reviewed
sample_spr_riveting_a598e00732_06,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,15972,0,True,unsupported,,,not reviewed
sample_spr_riveting_a598e00732_07,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,10503,0,True,included,,,not reviewed
sample_spr_riveting_a598e00732_08,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,15030,0,True,unsupported,,,not reviewed
sample_spr_riveting_a598e00732_09,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,8082,0,True,included,,,not reviewed
sample_spr_riveting_a598e00732_10,spr_riveting,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,20519,1,True,included,,,not reviewed


### sample_spr_riveting_a598e00732_01

根据《SX322014《新松机器人SPR铆接功能使用手册》-A-0》（V4.0 / A-0），在“基础配置”中，应如何理解或操作“柜体型号”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行位于《新松机器人SPR铆接应用操作手册》中，包含三列内容：第一列和第三列为空，第二列内容为“NOP”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1631–1635; corpus 4.0, revision V5.0 / A-1; object `f5c32aaf548d65ec38057adf1ed13d1f474cf666db2481c7694a03033d1f821b`.

- 该行位于新松机器人SPR铆接应用操作手册的表格中，第一列为空，第二列和第三列均值为“L3”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1344–1348; corpus 4.0, revision V5.0 / A-1; object `e7ad9efabe430f4d615e4ce81365cf5021bef602aa9890a688d7c1fbdcae31cc`.

- **摘要：**
该表格行包含文本内容“新松机器人SPR铆接应用操作手册”，位于第2行第1列，无合并单元格，非表头行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 13–15; corpus 4.0, revision V5.0 / A-1; object `2049a032cb9fda7572ce5704a9c14ae85efd99c6bc07e9b40ad39545e59c2d06`.

- 该行数据位于新松机器人SPR铆接应用操作手册中，包含三个单元格：第1列和第3列为空，第2列内容为“NOP”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1631–1635; corpus 5.0, revision V5.0 / A-1; object `dee1edee8332b49cc9d00a9d4944215d826eeda4a88a79e474a207e4987751c0`.

- 该行位于《新松机器人SPR铆接应用操作手册》，第1列内容为空，第2列和第3列均为“L101”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1946–1950; corpus 4.0, revision V5.0 / A-1; object `74f3372c0da3386bb9d5e8185fbec961b3b29271ba20092bb887a43e66d0825b`.

- 该行位于《新松机器人SPR铆接应用操作手册》中，第一列为空，第二列和第三列均填写为“L25”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1911–1915; corpus 5.0, revision V5.0 / A-1; object `af8ab1c4171ab6601ee0fa7495dbea6dcddd56b45bd98bdb2e8bcbcc9341ae07`.

- 该行数据源自《新松机器人SPR铆接应用操作手册》，位于第8行。第一列为空，第二列和第三列均为“L2”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 2082–2086; corpus 4.0, revision V5.0 / A-1; object `e736eed3b4586cb8d7f72ca786aa113e6195e05a991b00d8ad2bbc5ec3c8c25c`.

- 该表格行（索引 1）包含三列信息：第一列为空；第二列为指令/功能标识 **NOP**；第三列为操作描述 **开始**。

**摘要：**
在《新松机器人SPR铆接应用操作手册》的表格中，第 2 列标记为 **NOP**，第 3 列对应说明为 **开始**（第 1 列无内容）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 682–686; corpus 5.0, revision V5.0 / A-1; object `6c83264a93afb868c5ef268d9e5f0010fa162641fbdce651a170bc79b8ffebf6`.

#### Answer

在所提供的检索证据（E1–E8）中，所有内容均来自《SX322014 新松机器人SPR铆接应用操作手册》(A-1) 的 V5.0 版本，且仅涉及机器人程序指令（如 NOP、IF、GOTO、OUT、WAIT、USERALM、RET、END 等）以及开口位置设置、铆钉 ID 赋值、报警代码等运行时逻辑。没有任何一条证据提及

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: mixed.

The question asks for information from a specific V4.0/A-0 manual regarding '柜体型号' (Cabinet Model). The supplied evidence (E1-E8) exclusively contains code snippets and metadata from a V5.0/A-1 manual titled '新松机器人SPR铆接应用操作手册'. None of the provided evidence contains the term '柜体型号' or any configuration instructions related to it. The answer correctly identifies that the evidence does not contain the requested information and refuses to answer based on the lack of support in the provided context. This is a valid abstention because the necessary facts are missing from the supplied evidence.

...

#### Answer

在所提供的检索证据（E1–E8）中，所有片段均为程序逻辑表（NOP、IF IN#xx、OUT OT#xx、WAIT IN#xx、GOTO Lxx、USERALM CODE 等指令序列），展示的是铆接流程中各 IO 信号（如 IN#56、OT#57、OGH#55、OGFA#3 等）在程序中的读写操作。然而，没有任何一条证据包含

### sample_spr_riveting_9950195c71_20

根据《SX322014《新松机器人SPR铆接应用操作手册》(A-1)》（V5.0 / A-1），在“Atlas”中，应如何理解或操作“其作业需要调用“SPRCONNECT””？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行位于《新松机器人SPR铆接应用操作手册》中，包含三列内容：第一列和第三列为空，第二列内容为“NOP”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1631–1635; corpus 4.0, revision V5.0 / A-1; object `f5c32aaf548d65ec38057adf1ed13d1f474cf666db2481c7694a03033d1f821b`.

- 该行数据位于新松机器人SPR铆接应用操作手册中，包含三个单元格：第1列和第3列为空，第2列内容为“NOP”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1631–1635; corpus 5.0, revision V5.0 / A-1; object `dee1edee8332b49cc9d00a9d4944215d826eeda4a88a79e474a207e4987751c0`.

- 新松机器人SPR铆接应用操作手册的A/0版于2025年4月28日由沙景超完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 2993–2998; corpus 4.0, revision V5.0 / A-1; object `7f035fe246e6928abf7e54a39c5ad62a7d0f5c3e4be865ab0c803b8b862a437c`.

- 新松机器人SPR铆接应用操作手册的A/0版本由沙景超于2025年4月28日完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 2993–2998; corpus 5.0, revision V5.0 / A-1; object `e3bd0911ab54bddd0fcafc0b75a6a6b85bc0d893e6d5c8132fb756079534070b`.

- **摘要：**
该表格行包含文本内容“新松机器人SPR铆接应用操作手册”，位于第2行第1列，无合并单元格，非表头行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 13–15; corpus 4.0, revision V5.0 / A-1; object `2049a032cb9fda7572ce5704a9c14ae85efd99c6bc07e9b40ad39545e59c2d06`.

- 该行位于新松机器人SPR铆接应用操作手册的表格中，第一列为空，第二列和第三列均值为“L3”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1344–1348; corpus 4.0, revision V5.0 / A-1; object `e7ad9efabe430f4d615e4ce81365cf5021bef602aa9890a688d7c1fbdcae31cc`.

- 该行位于《新松机器人SPR铆接应用操作手册》，第1列内容为空，第2列和第3列均为“L101”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 1946–1950; corpus 4.0, revision V5.0 / A-1; object `74f3372c0da3386bb9d5e8185fbec961b3b29271ba20092bb887a43e66d0825b`.

- 该行是单列单元格，内容为“新松机器人SPR铆接应用操作手册”，无表头、无合并继承值；可视为该文档或表格的标题/名称行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322014《新松机器人SPR铆接应用操作手册》(A-1)/SX322014《新松机器人SPR铆接应用操作手册》(A-1).md`, lines 13–15; corpus 5.0, revision V5.0 / A-1; object `c18d995e6aec981f2e3791b92fbf21df574007179d57cb26f3ca5fb7cbaae1a6`.

#### Answer

The question asks how to understand or operate the statement '其作业需要调用SPRCONNECT' within 'Atlas' as described in the SPR riveting manual. However, none of the retrieved evidence (E1–E8) contains any mention of 'SPRCONNECT', 'Atlas', or any related function/command/module. The evidence only covers: (1) robot program instruction tables for SPR riveting I/O logic (E1, E6), (2) the document revision history (E3, E4), and (3) the title page (E5, E8). There is no documented syntax, operation method, purpose, or result for 'SPRCONNECT' in 'Atlas' in the supplied excerpts. Therefore a supported contextual example cannot be constructed.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1-E8) does not contain the terms 'SPRCONNECT' or 'Atlas'. The evidence provided consists of robot program code snippets, revision history, and title pages, none of which address the specific question about the 'SPRCONNECT' function in 'Atlas'. Therefore, the refusal to answer due to lack of evidence is appropriate and not a hallucination.

## Tool Quick Change — sampled manual revisions

40 cases from 2 CSV files. Set `TOOL_QUICK_CHANGE_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
TOOL_QUICK_CHANGE_SAMPLE_CASES = [{'id': 'sample_tool_quick_change_13a3d305c6_01', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 414, 'line_end': 420, 'excerpt': '<tr>\n<td><p>报警码</p></td>\n<td><p>报警信息</p></td>\n<td><p>子码</p></td>\n<td><p>子码报警信息</p></td>\n<td><p>解决方法</p></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_02', 'category': 'tool_quick_change', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”列出的报警7301表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7301'], 'expected_facts': ['7301 工具解锁/锁紧时， play_stat为空'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 421, 'line_end': 427, 'excerpt': '<tr>\n<td><p>7301</p></td>\n<td><p>工具解锁/锁紧时，</p><p>play&#95;stat为空</p></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_03', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.5', 'section_title': '1.5 实型变量说明', 'heading_path': ['1 界面配置', '1.5 实型变量说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），配置“实型变量说明”时，“待检查的停靠站不占用/占用状态”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['469 待检查的停靠站不占用/占用状态 1：检查占用状态，对应参数Occupied 2：检查不占用状态，对应参数Free'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 263, 'line_end': 267, 'excerpt': '<tr>\n<td><p>469</p></td>\n<td><p>待检查的停靠站不占用/占用状态</p></td>\n<td><p>1：检查占用状态，对应参数Occupied</p><p>2：检查不占用状态，对应参数Free</p></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_04', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 快换指令说明', 'heading_path': ['2 快换指令及快捷键说明', '2.1 快换指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“快换指令说明”中“A02_TC  CheckTool  ToolNo=  注释”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A02_TC'], 'expected_facts': ['格式 A02_TC CheckTool ToolNo= 注释'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 328, 'line_end': 331, 'excerpt': '<tr>\n<td><p>格式</p></td>\n<td><p>A02&#95;TC  CheckTool  ToolNo=  注释</p></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_05', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“基本作业流程”中“例”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['DOCKING'], 'expected_facts': ['例：如图3-1所示，在主作业中添加参数类型第一项为Docking指令（具体添加步骤见本文档中指令说明部分）', '新建子作业，作业名为“DOCKING”，该作业中实现工具解锁/锁紧具体逻辑流程，子作业中可使用对应实型变量'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 409, 'line_end': 409, 'excerpt': '例：如图3-1所示，在主作业中添加参数类型第一项为Docking指令（具体添加步骤见本文档中指令说明部分）。新建子作业，作业名为“DOCKING”，该作业中实现工具解锁/锁紧具体逻辑流程，子作业中可使用对应实型变量。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_06', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据手册“SX322015《新松机器人快换应用操作手册》-A-0.md”（V4.0 / A-0），要在“1.1 基础配置”中配置“总线IO映射”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：IO->总线IO映射'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 120, 'line_end': 120, 'excerpt': '配置路径：IO->总线IO映射；', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_07', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.5', 'section_title': '1.5 实型变量说明', 'heading_path': ['1 界面配置', '1.5 实型变量说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“实型变量说明”中“执行快换各功能指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['执行快换各功能指令，指令参数会存储在对应的实型变量中，以供编写子作业时使用，当前各功能参数存储位置及说明如下表所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 208, 'line_end': 208, 'excerpt': '执行快换各功能指令，指令参数会存储在对应的实型变量中，以供编写子作业时使用，当前各功能参数存储位置及说明如下表所示：', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_08', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“基本作业流程”中，应如何理解或操作“工具切换功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. 工具切换功能："CHANGETOOL"'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 403, 'line_end': 403, 'excerpt': '  5. 工具切换功能："CHANGETOOL"', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_09', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“按键OP2-OP3”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 按键OP2-OP3：可配置，硬按键OP2、OP3快捷键为可配置，配置项可通过下拉框选择（如下图所示）包括盖板打开、盖板关闭、用户IO输出3个功能项', '各快捷键功能项不会重复，当OP2按键功能项被OP3按键选择时，OP2自动替换OP3按键的功能项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 385, 'line_end': 385, 'excerpt': '2. 按键OP2-OP3：可配置，硬按键OP2、OP3快捷键为可配置，配置项可通过下拉框选择（如下图所示）包括盖板打开、盖板关闭、用户IO输出3个功能项。各快捷键功能项不会重复，当OP2按键功能项被OP3按键选择时，OP2自动替换OP3按键的功能项。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_10', 'category': 'tool_quick_change', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”列出的报警7302表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7302'], 'expected_facts': ['7302 停靠站盖板操作时，play_stat为空'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 428, 'line_end': 434, 'excerpt': '<tr>\n<td><p>7302</p></td>\n<td><p>停靠站盖板操作时，play&#95;stat为空</p></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_11', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.5', 'section_title': '1.5 实型变量说明', 'heading_path': ['1 界面配置', '1.5 实型变量说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），配置“实型变量说明”时，“检查盖板状态/不检查盖板状态”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['465 检查盖板状态/不检查盖板状态 1：不检查，对应参数NoCheck 2：检查，对应参数Check'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 241, 'line_end': 245, 'excerpt': '<tr>\n<td><p>465</p></td>\n<td><p>检查盖板状态/不检查盖板状态</p></td>\n<td><p>1：不检查，对应参数NoCheck</p><p>2：检查，对应参数Check</p></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_12', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 快换指令说明', 'heading_path': ['2 快换指令及快捷键说明', '2.1 快换指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“快换指令说明”中“指令选择界面如图所示”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A02'], 'expected_facts': ['指令选择界面如图所示，路径：指令集->应用指令集->快换->A02-TC,跳转至插入指令界面，该界面参数类型第一个下拉框可选择对应的快换应用功能，同时当前界面随选择的快换应用功能动态改变'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 280, 'line_end': 280, 'excerpt': '指令选择界面如图所示，路径：指令集->应用指令集->快换->A02-TC,跳转至插入指令界面，该界面参数类型第一个下拉框可选择对应的快换应用功能，同时当前界面随选择的快换应用功能动态改变。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_13', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据手册“SX322015《新松机器人快换应用操作手册》-A-0.md”（V4.0 / A-0），要在“1.1 基础配置”中配置“型号配置”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：系统设置->型号配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 108, 'line_end': 108, 'excerpt': '配置路径：系统设置->型号配置；', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_14', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“基本作业流程”中，应如何理解或操作“停靠站盖板操作功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 停靠站盖板操作功能："COVER"'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 400, 'line_end': 400, 'excerpt': '  2. 停靠站盖板操作功能："COVER"', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_15', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“OP1打开快捷键配置界面”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['OP1打开快捷键配置界面，界面打开时，再按OP1关闭该界面', '应用切换项选择快换应用，选择对应停靠站号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 378, 'line_end': 378, 'excerpt': 'OP1打开快捷键配置界面，界面打开时，再按OP1关闭该界面。应用切换项选择快换应用，选择对应停靠站号。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_16', 'category': 'tool_quick_change', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”列出的报警7305表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7305'], 'expected_facts': ['7305 工具切换时，play_stat为空'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 449, 'line_end': 455, 'excerpt': '<tr>\n<td><p>7305</p></td>\n<td><p>工具切换时，play&#95;stat为空</p></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_17', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 界面配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），配置“应用配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场快换应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 136, 'line_end': 136, 'excerpt': '逻辑IO分配：根据现场快换应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_18', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.1', 'section_title': '2.1 快换指令说明', 'heading_path': ['2 快换指令及快捷键说明', '2.1 快换指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），“快换指令说明”中“检查停靠站占用状态”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['检查停靠站占用状态 功能 检查停靠站占用状态指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 340, 'line_end': 344, 'excerpt': '<tr>\n<td rowspan="4"><p>检查停靠站占用状态</p></td>\n<td><p>功能</p></td>\n<td><p>检查停靠站占用状态指令</p></td>\n</tr>', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_19', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 界面配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“基础配置”中，应如何理解或操作“总线设备”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 总线设备：根据现场实际总线设备配置节点及IO标志位'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 114, 'line_end': 114, 'excerpt': '1. 总线设备：根据现场实际总线设备配置节点及IO标志位。', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_13a3d305c6_20', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“基本作业流程”中，应如何理解或操作“- 1. 工具解锁/锁紧功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['- 1. 工具解锁/锁紧功能：“DOCKING”'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'line_start': 399, 'line_end': 399, 'excerpt': '- 1. 工具解锁/锁紧功能：“DOCKING”', 'sha256': '858dfdd2e9823d3ee8d420222f4de371106708d1903b20096692def12294b4de'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_01', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 快换应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），配置“应用配置”时，“通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 95, 'line_end': 95, 'excerpt': '通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_02', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“盖板打开”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 盖板打开：打开对应停靠站号的盖板'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 186, 'line_end': 186, 'excerpt': '1. **盖板打开：**打开对应停靠站号的盖板。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_03', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），“基本作业流程”中“例”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['DOCKING'], 'expected_facts': ['例：如图3-1所示，在主作业中添加参数类型第一项为Docking指令（具体添加步骤见本文档中指令说明部分）', '新建子作业，作业名为“DOCKING”，该作业中实现工具解锁/锁紧具体逻辑流程，子作业中可使用对应实型变量'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 204, 'line_end': 204, 'excerpt': '例：如图3-1所示，在主作业中添加参数类型第一项为Docking指令（具体添加步骤见本文档中指令说明部分）。新建子作业，作业名为“DOCKING”，该作业中实现工具解锁/锁紧具体逻辑流程，子作业中可使用对应实型变量。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_04', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.5', 'section_title': '1.5 实型变量说明', 'heading_path': ['1 快换应用配置', '1.5 实型变量说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），“实型变量说明”中“执行快换各功能指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['执行快换各功能指令，指令参数会存储在对应的实型变量中，以供编写子作业时使用，当前各功能参数存储位置及说明如下表所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 151, 'line_end': 151, 'excerpt': '执行快换各功能指令，指令参数会存储在对应的实型变量中，以供编写子作业时使用，当前各功能参数存储位置及说明如下表所示：', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_05', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“基本作业流程”中，应如何理解或操作“检查停靠站占用状态功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 检查停靠站占用状态功能："CHECKOCCUPIED"'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 197, 'line_end': 197, 'excerpt': '  4. 检查停靠站占用状态功能："CHECKOCCUPIED"', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_06', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法 7301 工具解锁/锁紧时， play_stat为空 7302 停靠站盖板操作时，play_stat为空 7303 检查当前连接工具时，play_stat为空 7304 检查停靠站占用状态时，play_stat为空 7305 工具切换时，play_stat为空'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 208, 'line_end': 208, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码报警信息</p></td><td><p>解决方法</p></td></tr><tr><td><p>7301</p></td><td><p>工具解锁/锁紧时，</p><p>play_stat为空</p></td><td></td><td></td><td></td></tr><tr><td><p>7302</p></td><td><p>停靠站盖板操作时，play_stat为空</p></td><td></td><td></td><td></td></tr><tr><td><p>7303</p></td><td><p>检查当前连接工具时，play_stat为空</p></td><td></td><td></td><td></td></tr><tr><td><p>7304</p></td><td><p>检查停靠站占用状态时，play_stat为空</p></td><td></td><td></td><td></td></tr><tr><td><p>7305</p></td><td><p>工具切换时，play_stat为空</p></td><td></td><td></td><td></td></tr></table>', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_07', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.1', 'section_title': '1.4.1 快换盘品牌', 'heading_path': ['1 快换应用配置', '1.4 快换盘配置', '1.4.1 快换盘品牌'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“快换盘品牌”中，应如何理解或操作“当前可配置品牌史陶比尔与桥田”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['当前可配置品牌史陶比尔与桥田'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 128, 'line_end': 128, 'excerpt': '当前可配置品牌史陶比尔与桥田。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_08', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.1', 'section_title': '2.1 快换指令说明', 'heading_path': ['2 快换指令及快捷键说明', '2.1 快换指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），“快换指令说明”中“快换工具锁紧、解锁指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A02_TC'], 'expected_facts': ['快换工具锁紧、解锁指令 功能 工具解锁指令 格式 A02_TC Docking Unlock/Lock ToolNo= 注释 说明 <参数1>：功能定义', '<参数2>：锁紧/解锁', '<参数3>：工具编号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 169, 'line_end': 169, 'excerpt': '<table><tr><td rowspan="4"><p>快换工具锁紧、解锁指令</p></td><td><p>功能</p></td><td><p>工具解锁指令</p></td></tr><tr><td><p>格式</p></td><td><p>A02_TC  Docking  Unlock/Lock  ToolNo=  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：锁紧/解锁；</p><p>&lt;参数3&gt;：工具编号；范围1-16；</p></td></tr><tr><td><p>举例</p></td><td><p>A02_TC Docking Unlock ToolNo=1 解锁</p></td></tr><tr><td rowspan="4"><p>停靠站盖板打开/关闭指令</p></td><td><p>功能</p></td><td><p>停靠站盖板操作指令</p></td></tr><tr><td><p>格式</p></td><td><p>A02_TC  Cover  StationNo=  Close/Open  Check/NoCheck </p><p>Occupied/Free  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：停靠站编号；</p><p>&lt;参数3&gt;：打开/关闭；</p><p>&lt;参数4&gt;：检查盖板状态/不检查盖板状态；</p><p>&lt;参数5&gt;：占用/空闲；</p></td></tr><tr><td><p>举例</p></td><td><p>A02_TC Cover StationNo=1 Close Check Occupied 检查</p></td></tr><tr><td rowspan="4"><p>检查当前连接工具编号</p></td><td><p>功能</p></td><td><p>检查当前连接工具指令</p></td></tr><tr><td><p>格式</p></td><td><p>A02_TC  CheckTool  ToolNo=  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；&lt;参数2&gt;：工具编号；</p></td></tr><tr><td><p>举例</p></td><td><p>A02_TC CheckTool ToolNo=1检测抓手工具ID；</p></td></tr><tr><td rowspan="4"><p>检查停靠站占用状态</p></td><td><p>功能</p></td><td><p>检查停靠站占用状态指令</p></td></tr><tr><td><p>格式</p></td><td><p>A02_TC  CheckOccupied  StationNo=  Occupied/Free  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：停靠站编号；</p><p>&lt;参数3&gt;：占用/空闲；</p></td></tr><tr><td><p>举例</p></td><td><p>A02_TC CheckOccupied StationNo=1 Occupied检测支架工具在位; </p><p>A02_TC CheckOccupied StationNo=1 Free检测支架无在位; </p></td></tr><tr><td rowspan="4"><p>工具切换指令</p></td><td><p>功能</p></td><td><p>工具切换指令</p></td></tr><tr><td><p>格式</p></td><td><p>A02_TC  ChangeTool  ToolNo=  注释</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：功能定义；</p><p>&lt;参数2&gt;：工具编号；</p></td></tr><tr><td><p>举例</p></td><td><p>A02_TC ChangeTool ToolNo=1切换至对应工具1</p></td></tr></table>', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_09', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3', 'section_title': '1.3 停靠站配置', 'heading_path': ['1 快换应用配置', '1.3 停靠站配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），配置“停靠站配置”时，“信号显示”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 信号显示：点击信号显示按钮，界面会根据配置的起始IO与长度，自动生成对应IO号', '具体IO根据现场实际情况可进行修改', '修改值需在上述对应配置的起始位与长度范围内'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 116, 'line_end': 116, 'excerpt': '2. **信号显示**：点击信号显示按钮，界面会根据配置的起始IO与长度，自动生成对应IO号。具体IO根据现场实际情况可进行修改。修改值需在上述对应配置的起始位与长度范围内。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_10', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“用户IO输出”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 用户IO输出：在输出IO号栏中输入需要输出的IO号，按键后会输出对应IO号的值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 188, 'line_end': 188, 'excerpt': '3. **用户IO输出：**在输出IO号栏中输入需要输出的IO号，按键后会输出对应IO号的值。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_11', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“基本作业流程”中，应如何理解或操作“- 1. 工具解锁/锁紧功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['- 1. 工具解锁/锁紧功能：“DOCKING”'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 194, 'line_end': 194, 'excerpt': '- 1. 工具解锁/锁紧功能：“DOCKING”', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_12', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3', 'section_title': '1.3 停靠站配置', 'heading_path': ['1 快换应用配置', '1.3 停靠站配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“停靠站配置”中，应如何理解或操作“完成停靠站个数配置后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['完成停靠站个数配置后，此时可进行停靠站IO设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 109, 'line_end': 109, 'excerpt': '完成停靠站个数配置后，此时可进行停靠站IO设置。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_13', 'category': 'tool_quick_change', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.1', 'section_title': '2.1 快换指令说明', 'heading_path': ['2 快换指令及快捷键说明', '2.1 快换指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），“快换指令说明”中“指令选择界面如图所示”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A02'], 'expected_facts': ['指令选择界面如图所示，路径：指令集->应用指令集->快换->A02-TC,跳转至插入指令界面，该界面参数类型第一个下拉框可选择对应的快换应用功能，同时当前界面随选择的快换应用功能动态改变'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 161, 'line_end': 161, 'excerpt': '指令选择界面如图所示，路径：指令集->应用指令集->快换->A02-TC,跳转至插入指令界面，该界面参数类型第一个下拉框可选择对应的快换应用功能，同时当前界面随选择的快换应用功能动态改变。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_14', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.2', 'section_title': '1.4.2 快换盘IO配置', 'heading_path': ['1 快换应用配置', '1.4 快换盘配置', '1.4.2 快换盘IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），配置“快换盘IO配置”时，“快换盘输入输出IO的起始地址及长度配置”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 快换盘输入输出IO的起始地址及长度配置：起始地址与长度需包含在逻辑IO分配（配置路径：IO->逻辑IO分配）中快换应用对应IO范围'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 142, 'line_end': 142, 'excerpt': '1. 快换盘输入输出IO的起始地址及长度配置：起始地址与长度需包含在逻辑IO分配（配置路径：IO->逻辑IO分配）中快换应用对应IO范围。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_15', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“按键OP1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 按键OP1：打开快捷键配置界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 179, 'line_end': 179, 'excerpt': '1. 按键OP1：打开快捷键配置界面。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_16', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“基本作业流程”中，应如何理解或操作“快换应用功能与子作业名对应关系”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['快换应用功能与子作业名对应关系：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 192, 'line_end': 192, 'excerpt': '快换应用功能与子作业名对应关系：', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_17', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 基础配置', 'heading_path': ['1 快换应用配置', '1.1 基础配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322015《新松机器人快换应用操作手册》(A-0).md”（V5.0 / A-0），要在“1.1 基础配置”中配置“总线IO映射”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：IO->总线IO映射'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 63, 'line_end': 63, 'excerpt': '配置路径：IO->总线IO映射；', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_18', 'category': 'tool_quick_change', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 快换应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），配置“应用配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场快换应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 79, 'line_end': 79, 'excerpt': '逻辑IO分配：根据现场快换应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_19', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 快捷键', 'heading_path': ['2 快换指令及快捷键说明', '2.2 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“快捷键”中，应如何理解或操作“盖板关闭”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 盖板关闭：关闭对应停靠站号的盖板'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 187, 'line_end': 187, 'excerpt': '2. **盖板关闭：**关闭对应停靠站号的盖板。', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}, {'id': 'sample_tool_quick_change_70f1c99a9b_20', 'category': 'tool_quick_change', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 基本作业流程', 'heading_path': ['3 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“基本作业流程”中，应如何理解或操作“工具切换功能”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. 工具切换功能："CHANGETOOL"'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'line_start': 198, 'line_end': 198, 'excerpt': '  5. 工具切换功能："CHANGETOOL"', 'sha256': 'b731e8995b3ec8611f8edfe6a11b42b364d23259ecf6f85e99df9a38a3f05c3d'}]}]
register_sample_bank('tool_quick_change', TOOL_QUICK_CHANGE_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md', 'seed': 5852011900220294045, 'population': 78, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(414, 420), (421, 427), (263, 267), (328, 331), (409, 409), (120, 120), (208, 208), (403, 403), (385, 385), (428, 434), (241, 245), (280, 280), (108, 108), (400, 400), (378, 378), (449, 455), (136, 136), (340, 344), (114, 114), (399, 399)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md', 'seed': 2234389628025635169, 'population': 44, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(95, 95), (186, 186), (204, 204), (151, 151), (197, 197), (208, 208), (128, 128), (169, 169), (116, 116), (188, 188), (194, 194), (109, 109), (161, 161), (142, 142), (179, 179), (192, 192), (63, 63), (79, 79), (187, 187), (198, 198)]}])


In [ ]:
# | notest
TOOL_QUICK_CHANGE_SAMPLE_IDS = None
tool_quick_change_sample_cases = select_cases(TOOL_QUICK_CHANGE_SAMPLE_CASES, TOOL_QUICK_CHANGE_SAMPLE_IDS)
display(pd.DataFrame(tool_quick_change_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in tool_quick_change_sample_cases])
tool_quick_change_sample_metrics = await evaluation.run(tool_quick_change_sample_cases, concurrency=EVALUATION_CONCURRENCY)
tool_quick_change_sample_results = {case["id"]: evaluation.load(case) for case in tool_quick_change_sample_cases}
display(tool_quick_change_sample_metrics)
# Inspect a complete result: evaluation.inspect(tool_quick_change_sample_cases[0]["id"])
evaluation.show_answers(tool_quick_change_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_tool_quick_change_13a3d305c6_01,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,7,False,False,False,None,None,17763,0,True,unsupported,,,abstained
sample_tool_quick_change_13a3d305c6_02,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,alarm,table_row,False,True,8,False,0,0,7,False,False,False,False,False,22994,2,True,unsupported,,,incorrect
sample_tool_quick_change_13a3d305c6_03,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,1,False,False,False,None,None,15370,0,True,included,,,not_assessable
sample_tool_quick_change_13a3d305c6_04,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,8908,0,True,included,,,not reviewed
sample_tool_quick_change_13a3d305c6_05,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,instruction,prose,False,True,8,False,0,0,3,False,False,False,False,False,5335,0,True,unsupported,,,not reviewed
sample_tool_quick_change_13a3d305c6_06,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,5,False,False,False,None,None,15732,0,True,included,,,not reviewed
sample_tool_quick_change_13a3d305c6_07,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,instruction,prose,False,True,8,False,0,0,3,False,False,False,None,None,5047,0,True,unsupported,,,not reviewed
sample_tool_quick_change_13a3d305c6_08,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,3,False,False,False,None,None,5335,0,True,unsupported,,,not reviewed
sample_tool_quick_change_13a3d305c6_09,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,3,False,False,False,None,None,5047,0,True,unsupported,,,not reviewed
sample_tool_quick_change_13a3d305c6_10,tool_quick_change,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md,V4.0 / A-0,alarm,table_row,False,True,8,False,0,0,7,False,False,False,False,False,23345,1,True,unsupported,,,not reviewed


### sample_tool_quick_change_13a3d305c6_01

根据《SX322015《新松机器人快换应用操作手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行来自《新松机器人快换应用操作手册》(SX322015) V5.0 版本文档，位于第 5 行（非表头行）。该行包含 4 个单元格（列 1-4），所有单元格均为空值，无合并单元格继承，也无关联图片或子内容。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 52–57; corpus 5.0, revision V5.0 / A-0; object `a6b54128502429e077847160cc4b1f1d2e160b655b98c758e3adcbc58291a0e0`.

- 这是新松机器人通用操作手册中的报警处理表头，包含四列：报警码、报警信息、报警原因及解决措施。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 16989–16994; corpus 4.0, revision V5.0 / B-5; object `4894cfd0e51c38df05ee4223b7574a92ae6eaf1dce749751be53eb3c42396247`.

- 这是《新松机器人通用操作手册》中报警信息表格的**表头行**，定义了四列结构：**报警码**、**提示信息**、**报警原因**、**解决措施**，用于后续行列出具体故障代码及其对应的诊断与处理方法。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 26395–26400; corpus 4.0, revision V5.0 / B-5; object `5fe9ea44c7ab279f95c05a1e173689ccad518a4bd88fa0c620488e0723bb83da`.

- 新松机器人跟随应用操作手册（SX322017）报警表格定义行：列依次为报警码、报警等级、报警信息、报警原因、处理措施；无跨行/跨列合并，各单元格独立且未继承父级值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 604–610; corpus 4.0, revision V5.0 / A-0; object `73e63b2225c640d2b608ce544fc7c26d548d74458386b8b58a25e597fcfb4f66`.

- 新松机器人激光焊应用操作手册（V5.0版）表15第9行记录了一项报警处理操作：通过“以操作盘复位报警”进行复位；后续两列均为“〇”（表示该操作在对应状态下不适用或未生效），中间一列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 272–277; corpus 5.0, revision V5.0 / A-4; object `32a36756814437a2d3d7fa9fa775c29b176c3b4e6e13c70983c2af3018e3e8bd`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 该行是《新松机器人点焊应用操作手册》（A-2）中报警信息表的**表头行**，定义了五个关键列及其对应关系：

1.  **报警码**：用于标识主要报警代码。
2.  **报警信息**：对应报警码的文字描述。
3.  **子码**：报警的详细子分类代码。
4.  **子码说明**：对子码具体含义的解释。
5.  **处理方法**：针对该报警及子码的故障排查与解决建议。

此结构构成了点焊机器人报警故障诊断的逻辑框架：通过“报警码”和“子码”定位具体故障，查阅“说明”理解原因，并执行“处理方法”进行修复。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1664–1670; corpus 4.0, revision V5.0 / A-2; object `22941e4fac6314bf33acea0557a9da7237206ab140d42ed46e4cc9dae1305232`.

- 新松机器人 FDS 应用操作手册（V5.0）中的报警信息表结构：包含报警码、报警信息、子码、子码报警信息及解决方法五个字段。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 789–795; corpus 4.0, revision V5.0 / A-1; object `1d939c9236e70d6edc7557f7f11088b8aa1654d89126b8a8a7a7abaed3581747`.

#### Answer

The question asks specifically about the '报警码' (alarm code) entry in the '常见报警及解决方法' (Common Alarms and Solutions) section of the SX322015《新松机器人快换应用操作手册》(A-0) manual. However, the only retrieved evidence from this manual (E1) contains solely the revision history table (lines 52-57), showing the document version A/0, revision date 2025-03-18, and author 沙景超. It does not include any content from the '常见报警及解决方法' section, no alarm code definitions, no alarm information, no alarm causes, and no resolution measures. The other evidence items (E2-E8) come from different manuals (通用操作手册, 跟随应用操作手册, 激光焊应用操作手册, 点焊应用操作手册, FDS应用操作手册) and, while they illustrate the general table structure used across Siasun robot manuals (报警码 | 报警信息 | 报警原因 | 解决措施), they do not provide the specific alarm codes or their descriptions as documented in the SX322015 quick-change application manual. Without the actual alarm code entries from the SX322015 manual, a supported contextual example cannot be constructed.

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: mixed.

The answer correctly identifies that the supplied evidence (E1) from the requested manual (SX322015) only contains revision history and lacks the specific '常见报警及解决方法' section content. It accurately notes that other evidence items (E2-E8) are from different manuals and do not provide the specific alarm codes for the requested document. Therefore, the refusal to provide a specific answer is appropriate given the inadequate evidence.

...

#### Answer

The question asks specifically about the '快捷键' (shortcut keys) section and how to operate '盖板关闭' (cover close) within that section. The retrieved evidence (E5) documents the '停靠站盖板操作功能 Cover' variable interface, where variable 464 with value 1 corresponds to parameter CLOSE (关闭盖板). However, none of the retrieved evidence contains a section explicitly labeled '快捷键' (shortcut keys). The evidence only shows a variable-based programming interface table, not a shortcut-key or quick-button operation section. Without the actual '快捷键' section content, I cannot confirm the specific shortcut key mapping, button label, or UI interaction described in that section.

### sample_tool_quick_change_70f1c99a9b_20

根据《SX322015《新松机器人快换应用操作手册》(A-0)》（V5.0 / A-0），在“基本作业流程”中，应如何理解或操作“工具切换功能”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 新松机器人快换应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 11–13; corpus 4.0, revision V5.0 / A-0; object `2c89a717b53c81863bfc8dd5a5587050a99aa85b982aadc3bd2fc33c1274d516`.

- 新松机器人快换应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 11–13; corpus 5.0, revision V5.0 / A-0; object `69bd67235fd583764b7b504b4c64e007c357d383237df957b21a538e2fbec5fd`.

- 该行来自《新松机器人快换应用操作手册》(SX322015) V5.0 版本文档，位于第 5 行（非表头行）。该行包含 4 个单元格（列 1-4），所有单元格均为空值，无合并单元格继承，也无关联图片或子内容。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 52–57; corpus 5.0, revision V5.0 / A-0; object `a6b54128502429e077847160cc4b1f1d2e160b655b98c758e3adcbc58291a0e0`.

- 该行为《新松机器人快换应用操作手册》(V5.0, 文档SX322015) 中表格的第5行数据（索引4），包含4个空单元格，无具体文本内容。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 46–51; corpus 5.0, revision V5.0 / A-0; object `d7f0433c08c1e2e8be5cd8083970b05cf8813c135150d11e2933d65e9f6a6abc`.

- 该表格为“新松机器人快换应用操作手册”的工业机器人技术资料封面或标题页信息，包含手册名称及资料类别。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322015《新松机器人快换应用操作手册》(A-0)/SX322015《新松机器人快换应用操作手册》(A-0).md`, lines 7–17; corpus 4.0, revision V5.0 / A-0; object `9b51f34db3856c3fe68583ea35172ee53156edfac32e3e1e4cbd48953602931d`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

- 新松机器人通用操作手册，编号 SX322001
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 483–486; corpus 4.0, revision V5.0 / A-4; object `0fa0fe1f9248cd8de37872f90e54ee8f642cc7bf54636b2e8e456a5202b63209`.

#### Answer

The retrieved evidence (E1–E5) for SX322015《新松机器人快换应用操作手册》(A-0) V5.0/A-0 only contains the document title page (lines 7–17) and the revision-history table (lines 46–57). None of the cited lines include the actual body text of the 

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1-E5) only contains the title page and revision history, lacking the specific body text for the 'Tool Switching Function' (CHANGETOOL). It appropriately refuses to generate unsupported operational details, which is a valid abstention given the inadequate evidence.

## Fds — sampled manual revisions

40 cases from 2 CSV files. Set `FDS_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
FDS_SAMPLE_CASES = [{'id': 'sample_fds_7c3c2ef958_01', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警：10201”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警：10201 文件操作失败 联系技术人员'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 817, 'line_end': 823, 'excerpt': '<tr>\n<td><p>报警：10201</p></td>\n<td><p>文件操作失败</p></td>\n<td></td>\n<td></td>\n<td><p>联系技术人员</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_02', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 界面配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），配置“应用配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场FDS应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 154, 'line_end': 154, 'excerpt': '逻辑IO分配：根据现场FDS应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_03', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3', 'section_title': '1.3 快捷键', 'heading_path': ['1 指令、快捷键说明', '1.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），配置“快捷键”时，“铆枪号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能', '根据配置的范围，可以配置从1-16'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 517, 'line_end': 517, 'excerpt': '2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能。根据配置的范围，可以配置从1-16。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_04', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3', 'section_title': '1.3 快捷键', 'heading_path': ['1 指令、快捷键说明', '1.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“模式存储变量号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['8. 模式存储变量号：工作模式存储变量用于作业判断', '其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 523, 'line_end': 523, 'excerpt': '8. 模式存储变量号：工作模式存储变量用于作业判断。其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_05', 'category': 'fds', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 界面配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），操作“通讯配置”时，涉及“输出信号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. 输出信号：IO状态可手动输出', '若执行开关中用户IO输出保护开关为ON，则输出条件为示教模式下上电并Deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 241, 'line_end': 241, 'excerpt': '1. 输出信号：IO状态可手动输出。若执行开关中用户IO输出保护开关为ON，则输出条件为示教模式下上电并Deadman使能。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_06', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 工艺作业模版', 'heading_path': ['3 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“工艺作业模版”中，应如何理解或操作“PULSE OT#36=ON T=0.5;”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['PULSE OT#36=ON T=0.5; 输出不合格继续'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 780, 'line_end': 784, 'excerpt': '<tr>\n<td></td>\n<td><p>PULSE OT&#35;36=ON T=0&#46;5;</p></td>\n<td><p>输出不合格继续</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_07', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 界面配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据手册“SX322016《新松机器人FDS应用使用手册》-A-0.md”（V4.0 / A-0），要在“1.2 应用配置”中配置“通讯状态IO分配”，应按什么菜单路径进入？', 'anchors': [], 'expected_facts': ['配置路径：超级用户->IO->逻辑IO分配->通讯状态IO分配'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 168, 'line_end': 168, 'excerpt': '配置路径：超级用户->IO->逻辑IO分配->通讯状态IO分配；', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_08', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 防漏铆的基本作业流程', 'heading_path': ['2 模板作业', '2.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“防漏铆的基本作业流程”中，应如何理解或操作“MOVL P9 VL=20 ACC=100 CNT=100”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['MOVL P9 VL=20 ACC=100 CNT=100 运动到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 592, 'line_end': 595, 'excerpt': '<tr>\n<td><p>MOVL P9 VL=20 ACC=100 CNT=100</p></td>\n<td><p>运动到安全位置</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_09', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“报警：10214”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FDS'], 'expected_facts': ['报警：10214 FDS指令集错误 请检查作业名称'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 908, 'line_end': 914, 'excerpt': '<tr>\n<td><p>报警：10214</p></td>\n<td><p>FDS指令集错误</p></td>\n<td></td>\n<td></td>\n<td><p>请检查作业名称</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_10', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 FDS运动+铆接指令', 'heading_path': ['1 指令、快捷键说明', '1.1 FDS运动+铆接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），“FDS运动+铆接指令”中“指令含义”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FDS'], 'expected_facts': ['指令含义：FDS运动+焊接指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 261, 'line_end': 261, 'excerpt': '指令含义：FDS运动+焊接指令。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_11', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），配置“常见报警及解决方法”时，“报警：10203”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['报警：10203 配置IO不在逻辑IO分配范围内 检查IO配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 831, 'line_end': 837, 'excerpt': '<tr>\n<td><p>报警：10203</p></td>\n<td><p>配置IO不在逻辑IO分配范围内</p></td>\n<td></td>\n<td></td>\n<td><p>检查IO配置</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_12', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警：10204”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警：10204 机器人干涉 检查干涉区'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 838, 'line_end': 844, 'excerpt': '<tr>\n<td><p>报警：10204</p></td>\n<td><p>机器人干涉</p></td>\n<td></td>\n<td></td>\n<td><p>检查干涉区</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_13', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.7', 'section_title': '1.7 实时显示', 'heading_path': ['1 界面配置', '1.7 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），配置“实时显示”时，“FDS实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['FDS实时显示功能，可查看FDS铆接的相关参数，如通讯状态、枪号、程序号等参数', '可在本地和远程等模式下进行查看，界面如下所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 247, 'line_end': 247, 'excerpt': 'FDS实时显示功能，可查看FDS铆接的相关参数，如通讯状态、枪号、程序号等参数；可在本地和远程等模式下进行查看，界面如下所示：', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_14', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.3', 'section_title': '1.3 快捷键', 'heading_path': ['1 指令、快捷键说明', '1.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“快捷键”中，应如何理解或操作“PLC-SIM”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['6. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 521, 'line_end': 521, 'excerpt': '6. PLC-SIM:代表的是PLC发送工艺关-仿真模式的输入IO号、输出IO号。', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_15', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 工艺作业模版', 'heading_path': ['3 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“工艺作业模版”中，应如何理解或操作“WAIT IN#33=ON T=1;”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['WAIT IN#33=ON T=1; 等待系统在自动模式信号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 670, 'line_end': 674, 'excerpt': '<tr>\n<td></td>\n<td><p>WAIT IN&#35;33=ON T=1;</p></td>\n<td><p>等待系统在自动模式信号</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_16', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 界面配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“通讯配置”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['路径：FDS->通讯配置->输入/输出'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 213, 'line_end': 213, 'excerpt': '路径：FDS->通讯配置->输入/输出', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_17', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 防漏铆的基本作业流程', 'heading_path': ['2 模板作业', '2.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“防漏铆的基本作业流程”中，应如何理解或操作“A14_FDS ProcessCheck”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['A14_FDS ProcessCheck FDS防漏焊检测'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 628, 'line_end': 631, 'excerpt': '<tr>\n<td><p>A14&#95;FDS ProcessCheck</p></td>\n<td><p>FDS防漏焊检测</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_18', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），“常见报警及解决方法”中“报警：10212”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警：10212 无效铆点检测 重新执行检测指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 894, 'line_end': 900, 'excerpt': '<tr>\n<td><p>报警：10212</p></td>\n<td><p>无效铆点检测</p></td>\n<td></td>\n<td></td>\n<td><p>重新执行检测指令</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_19', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.2.2', 'section_title': '1.2.2 FDS防漏铆检测指令', 'heading_path': ['1 指令、快捷键说明', '1.2 FDS指令集', '1.2.2 FDS防漏铆检测指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），“FDS防漏铆检测指令”中“IF I427 == 00 GOTO L1;”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['I427', 'GOTO'], 'expected_facts': ['IF I427 == 00 GOTO L1; 漏铆计数为0，跳转至L1;'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 398, 'line_end': 402, 'excerpt': '<tr>\n<td></td>\n<td><p>IF I427 == 00 GOTO L1;</p></td>\n<td><p>漏铆计数为0，跳转至L1;</p></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_7c3c2ef958_20', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 常见报警及解决方法', 'heading_path': ['4 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'source_version': '', 'question': '根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“提示：10252”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['提示：10252 fds铆接清空堆栈完成'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'line_start': 929, 'line_end': 935, 'excerpt': '<tr>\n<td><p>提示：10252</p></td>\n<td><p>fds铆接清空堆栈完成</p></td>\n<td></td>\n<td></td>\n<td></td>\n</tr>', 'sha256': 'fcb91d100bf1438480c98cfe478e866c6027542fcca8aa0d69386f7852f24a26'}]}, {'id': 'sample_fds_c9f17d0e88_01', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '5', 'section_title': '5 常见报警及解决方法', 'heading_path': ['5 常见报警及解决方法'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），“常见报警及解决方法”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['PLC', 'FDS'], 'expected_facts': ['报警码 报警信息 子码 子码报警信息 解决方法 报警：10201 文件操作失败 联系技术人员 报警：10202 巡检异常 请检查PLC设备状态 报警：10203 配置IO不在逻辑IO分配范围内 检查IO配置 报警：10204 机器人干涉 检查干涉区 报警：10205 水流量异常 请检查实际流量与设定流量值 报警：10206 气流量异常 请检查气压表状态 报警：10207 FDS-NT151监控异常 请检查NT151模块功能 报警：10208 FDS焊接前状态异常 报警：10209 FDS铆接超时 请人工检查焊接状态 报警：10210 铆点存在漏铆 请人工补焊 报警：10211 铆点存在多铆 请人工除钉 报警：10212 无效铆点检测 重新执行检测指令 报警：10213 响应安全门请求暂停中 请检查安全门状态 报警：10214 FDS指令集错误 请检查作业名称 报警：10215 通讯状态异常 请检查相应的外设设备的通讯线或模块是否正常 提示：10251 当前为非NORMAL模式 提示：10252 fds铆接清空堆栈完成 提示：10253 不支持1.0算法 提示：10254 响应安全门请求暂停中 提示：10257 执行FDS顶出元件快捷功能 提示：10258 执行FDS供给元件快捷功能 提示：10259 执行FDS伸出压板快捷功能 提示：10260 巡检配置为0功能未启用 提示：10261 执行FDS标定运行功能 提示：10262 标定运行功能完成 提示：10263 铆点数量正常 提示：10264 漏铆个数为: 提示：10265 多铆个数为: 提示：10266 通讯为屏蔽状态 提示：10267 快捷键 0 顶出元件完成 1 供给元件完成 2 顶出元件异常 3 供给元件异常 4 伸出压板完成 5 标定运行功能完成 6 标定运行功能异常'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 335, 'line_end': 335, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码报警信息</p></td><td><p>解决方法</p></td></tr><tr><td><p>报警：10201</p></td><td><p>文件操作失败</p></td><td></td><td></td><td><p>联系技术人员</p></td></tr><tr><td><p>报警：10202</p></td><td><p>巡检异常</p></td><td></td><td></td><td><p>请检查PLC设备状态</p></td></tr><tr><td><p>报警：10203</p></td><td><p>配置IO不在逻辑IO分配范围内</p></td><td></td><td></td><td><p>检查IO配置</p></td></tr><tr><td><p>报警：10204</p></td><td><p>机器人干涉</p></td><td></td><td></td><td><p>检查干涉区</p></td></tr><tr><td><p>报警：10205</p></td><td><p>水流量异常</p></td><td></td><td></td><td><p>请检查实际流量与设定流量值</p></td></tr><tr><td><p>报警：10206</p></td><td><p>气流量异常</p></td><td></td><td></td><td><p>请检查气压表状态</p></td></tr><tr><td><p>报警：10207</p></td><td><p>FDS-NT151监控异常</p></td><td></td><td></td><td><p>请检查NT151模块功能</p></td></tr><tr><td><p>报警：10208</p></td><td><p>FDS焊接前状态异常</p></td><td></td><td></td><td></td></tr><tr><td><p>报警：10209</p></td><td><p>FDS铆接超时</p></td><td></td><td></td><td><p>请人工检查焊接状态</p></td></tr><tr><td><p>报警：10210</p></td><td><p>铆点存在漏铆</p></td><td></td><td></td><td><p>请人工补焊</p></td></tr><tr><td><p>报警：10211</p></td><td><p>铆点存在多铆</p></td><td></td><td></td><td><p>请人工除钉</p></td></tr><tr><td><p>报警：10212</p></td><td><p>无效铆点检测</p></td><td></td><td></td><td><p>重新执行检测指令</p></td></tr><tr><td><p>报警：10213</p></td><td><p>响应安全门请求暂停中</p></td><td></td><td></td><td><p>请检查安全门状态</p></td></tr><tr><td><p>报警：10214</p></td><td><p>FDS指令集错误</p></td><td></td><td></td><td><p>请检查作业名称</p></td></tr><tr><td><p>报警：10215</p></td><td><p>通讯状态异常</p></td><td></td><td></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>提示：10251</p></td><td><p>当前为非NORMAL模式</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10252</p></td><td><p>fds铆接清空堆栈完成</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10253</p></td><td><p>不支持1.0算法</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10254</p></td><td><p>响应安全门请求暂停中</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10257</p></td><td><p>执行FDS顶出元件快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10258</p></td><td><p>执行FDS供给元件快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10259</p></td><td><p>执行FDS伸出压板快捷功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10260</p></td><td><p>巡检配置为0功能未启用</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10261</p></td><td><p>执行FDS标定运行功能</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10262</p></td><td><p>标定运行功能完成</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10263</p></td><td><p>铆点数量正常</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10264</p></td><td><p>漏铆个数为:</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10265</p></td><td><p>多铆个数为:</p></td><td></td><td></td><td></td></tr><tr><td><p>提示：10266</p></td><td><p>通讯为屏蔽状态</p></td><td></td><td></td><td></td></tr><tr><td rowspan="7"><p>提示：10267</p></td><td rowspan="7"><p>快捷键</p></td><td><p>0</p></td><td><p>顶出元件完成</p></td><td></td></tr><tr><td><p>1</p></td><td><p>供给元件完成</p></td><td></td></tr><tr><td><p>2</p></td><td><p>顶出元件异常</p></td><td></td></tr><tr><td><p>3</p></td><td><p>供给元件异常</p></td><td></td></tr><tr><td><p>4</p></td><td><p>伸出压板完成</p></td><td></td></tr><tr><td><p>5</p></td><td><p>标定运行功能完成</p></td><td></td></tr><tr><td><p>6</p></td><td><p>标定运行功能异常</p></td><td></td></tr></table>', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_02', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '3.2', 'section_title': '3.2 防漏铆的基本作业流程', 'heading_path': ['3 模板作业', '3.2 防漏铆的基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322016《新松机器人FDS应用操作手册》(A-1).md”（V5.0 / A-1），作业模板中A14_FDS、MOVL、ACC的执行顺序和作用是什么？', 'anchors': ['A14_FDS', 'MOVL', 'ACC'], 'expected_facts': ['A14_FDS Init Gun=1 Count=6 FDS初始化', 'MOVL P7 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P1，焊枪1进行焊接。', 'MOVL P8 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P2，焊枪1进行焊接。', 'MOVL P9 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P3 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P3，焊枪1进行焊接。', 'MOVL P10 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P4 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P4，焊枪1进行焊接。', 'MOVL P11 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P5，焊枪1进行焊接。', 'MOVL P12 VL=20 ACC=100 CNT=100 运动到安全位置', 'FDSJ P6 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3 运动到焊接位置P6，焊枪1进行焊接。', 'MOVL P13 VL=20 ACC=100 CNT=100 运动到安全位置', 'A14_FDS ProcessCheck FDS防漏焊检测'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 313, 'line_end': 313, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>A14_FDS Init Gun=1 Count=6 </p></td><td><p>FDS初始化</p></td></tr><tr><td><p>MOVL P7 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P1，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P8 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P2，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P9 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P3 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P3，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P10 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P4 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P4，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P11 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P5 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P5，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P12 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P6 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 1 PROGNo = 3</p></td><td><p>运动到焊接位置P6，焊枪1进行焊接。</p></td></tr><tr><td><p>MOVL P13 VL=20 ACC=100 CNT=100</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>A14_FDS ProcessCheck</p></td><td><p>FDS防漏焊检测</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_03', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），配置“快捷键”时，“铆枪号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能', '根据配置的范围，可以配置从1-16'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 292, 'line_end': 292, 'excerpt': '2. 铆枪号:FDS应用触发相应系统对应的焊枪号的快捷功能。根据配置的范围，可以配置从1-16。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_04', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“同步快捷键OP6”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 同步快捷键OP6：工艺开、工艺关-干运行、工艺关-仿真'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 294, 'line_end': 294, 'excerpt': '4. 同步快捷键OP6：工艺开、工艺关-干运行、工艺关-仿真。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_05', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 FDS应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“应用配置”中，应如何理解或操作“建议输入IO分配”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['建议输入IO分配：IO起始位配置为33，长度为160'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 99, 'line_end': 99, 'excerpt': '建议输入IO分配：IO起始位配置为33，长度为160。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_06', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2.2', 'section_title': '2.2.2 FDS防漏铆检测指令', 'heading_path': ['2 指令、快捷键说明', '2.2 FDS指令集', '2.2.2 FDS防漏铆检测指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），“FDS防漏铆检测指令”中“ATLAS、巍博的铆机需要调用子作业”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ATLAS', 'FDSRIVETCHECK'], 'expected_facts': ['ATLAS、巍博的铆机需要调用子作业，需要进行调用“FDSRIVETCHECK”'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 262, 'line_end': 262, 'excerpt': 'ATLAS、巍博的铆机需要调用子作业，需要进行调用“FDSRIVETCHECK”。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_07', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 FDS应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），配置“应用配置”时，“逻辑IO分配”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['逻辑IO分配：根据现场FDS应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 95, 'line_end': 95, 'excerpt': '逻辑IO分配：根据现场FDS应用实际分配的逻辑IO范围配置输入和输出IO的应用类型，IO起始位和长度项。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_08', 'category': 'fds', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 FDS应用配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），操作“通讯配置”时，涉及“输出信号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['1. 输出信号：IO状态可手动输出', '若执行开关中用户IO输出保护开关为ON，则输出条件为示教模式下上电并Deadman使能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 184, 'line_end': 184, 'excerpt': '1. 输出信号：IO状态可手动输出。若执行开关中用户IO输出保护开关为ON，则输出条件为示教模式下上电并Deadman使能。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_09', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“工艺作业模版”中，应如何理解或操作“其作业需要调用“FDSRIVETPROCESS””？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['其作业需要调用“FDSRIVETPROCESS”，该子作业进行使用铆接流程'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 323, 'line_end': 323, 'excerpt': '其作业需要调用“FDSRIVETPROCESS”，该子作业进行使用铆接流程', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_10', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '3.1', 'section_title': '3.1 基本作业流程', 'heading_path': ['3 模板作业', '3.1 基本作业流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322016《新松机器人FDS应用操作手册》(A-1).md”（V5.0 / A-1），作业模板中MOVL、ACC、CNT的执行顺序和作用是什么？', 'anchors': ['MOVL', 'ACC', 'CNT'], 'expected_facts': ['MOVL P3 VL=20 ACC=100 CNT=0 安全点', 'FDSJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到焊接位置P1,进行焊接。', 'MOVL P4 VL=20 ACC=100 CNT=0 运动到安全位置', 'FDSJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3 运动到焊接位置P2,进行焊接。', 'MOVL P5 VL=20 ACC=100 CNT=0 运动到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 307, 'line_end': 307, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVL P3 VL=20 ACC=100 CNT=0</p></td><td><p>安全点</p></td></tr><tr><td><p>FDSJ P1 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到焊接位置P1,进行焊接。</p></td></tr><tr><td><p>MOVL P4 VL=20 ACC=100 CNT=0</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>FDSJ P2 VJ=20 ACC=100 CNT=0 INTERLOCK=1 GUN = 2 PROGNo = 3</p></td><td><p>运动到焊接位置P2,进行焊接。</p></td></tr><tr><td><p>MOVL P5 VL=20 ACC=100 CNT=0</p></td><td><p>运动到安全位置</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_11', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“模式存储变量号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['8. 模式存储变量号：工作模式存储变量用于作业判断', '其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 298, 'line_end': 298, 'excerpt': '8. 模式存储变量号：工作模式存储变量用于作业判断。其中工艺开模式的值为0，工艺关-仿真模式的值为1，工艺关-干运行模式的值为2。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_12', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.6', 'section_title': '1.6 通讯配置', 'heading_path': ['1 FDS应用配置', '1.6 通讯配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“通讯配置”中，应如何理解或操作“各品牌FDS显示的协议位内容不同”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 各品牌FDS显示的协议位内容不同，参考通讯协议位进行设置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 186, 'line_end': 186, 'excerpt': '3. 各品牌FDS显示的协议位内容不同，参考通讯协议位进行设置。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_13', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2.2', 'section_title': '2.2.2 FDS防漏铆检测指令', 'heading_path': ['2 指令、快捷键说明', '2.2 FDS指令集', '2.2.2 FDS防漏铆检测指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），“FDS防漏铆检测指令”中“功能”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['2. 功能：检测所有铆枪是否出现漏焊'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 256, 'line_end': 256, 'excerpt': '2. 功能：检测所有铆枪是否出现漏焊。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_14', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.7', 'section_title': '1.7 实时显示', 'heading_path': ['1 FDS应用配置', '1.7 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），配置“实时显示”时，“FDS实时显示功能”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['FDS实时显示功能，可查看FDS铆接的相关参数，如通讯状态、枪号、程序号等参数', '可在本地和远程等模式下进行查看，界面如下所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 190, 'line_end': 190, 'excerpt': 'FDS实时显示功能，可查看FDS铆接的相关参数，如通讯状态、枪号、程序号等参数；可在本地和远程等模式下进行查看，界面如下所示：', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_15', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322016《新松机器人FDS应用操作手册》(A-1).md”（V5.0 / A-1），涂胶流程模板在冲洗或填充之前如何检查运行模式和胶桶状态，胶量不足时怎样处理？', 'anchors': ['IN#38', 'USERALM'], 'expected_facts': ['IF IN#38 == ON GOTO L1; 如果输入信号结果合格为ON，跳转至L1', 'USERALM CODE =2600; 自定义报警，铆接失败'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 331, 'line_end': 331, 'excerpt': '<table><tr><td></td><td><p>NOP</p></td><td><p>开始</p></td></tr><tr><td></td><td><p>WAIT IN#33=ON T=1;</p></td><td><p>等待系统在自动模式信号</p></td></tr><tr><td></td><td><p>WAIT IN#35=ON T=1;</p></td><td><p>等待系统无故障信号</p></td></tr><tr><td></td><td><p>WAIT IN#34=ON T=1;</p></td><td><p>等待系统启动就绪信号</p></td></tr><tr><td></td><td><p>OUT OT#45 = ON;</p></td><td><p>输出工具编号1</p></td></tr><tr><td></td><td><p>OUT OG#1 = I422;</p></td><td><p>TypeID号I422赋给1-8为组IO</p></td></tr><tr><td></td><td><p>OUT OGH#5 = I423;</p></td><td><p>铆钉ID号I423赋给65-80</p></td></tr><tr><td></td><td><p>WAIT IN#57=ON T=1;</p></td><td><p>等待输入结合点正常信号</p></td></tr><tr><td></td><td><p>WAIT IN#54=ON T=1;</p></td><td><p>等待元件在头部信号</p></td></tr><tr><td></td><td><p>OUT OT#34 = ON;</p></td><td><p>输出启动主轴信号</p></td></tr><tr><td></td><td><p>WAIT IN#36=ON T=3;</p></td><td><p>等待完成信息的输入信号</p></td></tr><tr><td></td><td><p>IF IN#36 == OFF GOTO L2;</p></td><td><p>如果启动就绪的输入信号为OFF,跳转至L2</p></td></tr><tr><td></td><td><p>IF IN#39 == ON GOTO L2;</p></td><td><p>如果结果不合格的输入信号为ON,跳转至L2</p></td></tr><tr><td></td><td><p>OUT OT#34 = OFF;</p></td><td><p>输出启动就绪信号为OFF</p></td></tr><tr><td></td><td><p>L3</p></td><td><p>L3</p></td></tr><tr><td></td><td><p>IF IN#38 == ON GOTO L1;</p></td><td><p>如果输入信号结果合格为ON，跳转至L1</p></td></tr><tr><td></td><td><p>IF IN#39 == ON GOTO L2;</p></td><td><p>如果输入信号结果不合格为ON，跳转至L2</p></td></tr><tr><td></td><td><p>GOTO L3</p></td><td><p>跳转至L3</p></td></tr><tr><td></td><td><p>L1</p></td><td><p>L1</p></td></tr><tr><td></td><td><p>INC I427</p></td><td><p>计数</p></td></tr><tr><td></td><td><p>GOTO L4</p></td><td><p>跳转至L4</p></td></tr><tr><td></td><td><p>L2</p></td><td><p>L2</p></td></tr><tr><td></td><td><p>USERALM CODE =2600;</p></td><td><p>自定义报警，铆接失败</p></td></tr><tr><td></td><td><p>PULSE OT#36=ON T=0.5;</p></td><td><p>输出不合格继续</p></td></tr><tr><td></td><td><p>OUT OT#34 = OFF;</p></td><td><p>将启动主轴置为OFF</p></td></tr><tr><td></td><td><p>L4</p></td><td><p>L4</p></td></tr><tr><td></td><td><p>RET</p></td><td><p>返回</p></td></tr><tr><td></td><td><p>END</p></td><td><p>结束</p></td></tr></table>', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_16', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.3', 'section_title': '2.3 快捷键', 'heading_path': ['2 指令、快捷键说明', '2.3 快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“快捷键”中，应如何理解或操作“点击示教器OP1键”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['点击示教器OP1键，可弹出快捷键应用配置界面：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 283, 'line_end': 283, 'excerpt': '点击示教器OP1键，可弹出快捷键应用配置界面：', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_17', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.8', 'section_title': '1.8 水气单元', 'heading_path': ['1 FDS应用配置', '1.8 水气单元'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“水气单元”中，应如何理解或操作“气压检测开关”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2) 气压检测开关，打开后，后台开始监控气压对应的IO信号'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 206, 'line_end': 206, 'excerpt': '2) 气压检测开关，打开后，后台开始监控气压对应的IO信号。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_18', 'category': 'fds', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '2.2.2', 'section_title': '2.2.2 FDS防漏铆检测指令', 'heading_path': ['2 指令、快捷键说明', '2.2 FDS指令集', '2.2.2 FDS防漏铆检测指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），“FDS防漏铆检测指令”中“指令含义”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FDS'], 'expected_facts': ['1. 指令含义：FDS防漏铆检测指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 255, 'line_end': 255, 'excerpt': '1. 指令含义：FDS防漏铆检测指令', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_19', 'category': 'fds', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '1.2', 'section_title': '1.2 应用配置', 'heading_path': ['1 FDS应用配置', '1.2 应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），配置“应用配置”时，“通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 111, 'line_end': 111, 'excerpt': '通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内。', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}, {'id': 'sample_fds_c9f17d0e88_20', 'category': 'fds', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-1', 'section': '4', 'section_title': '4 工艺作业模版', 'heading_path': ['4 工艺作业模版'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“工艺作业模版”中，应如何理解或操作“Atlas—前台作业”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. Atlas—前台作业'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'line_start': 329, 'line_end': 329, 'excerpt': '1. **Atlas—前台作业**', 'sha256': 'fd5086f4440882cc4ebf5e96a25f818bd529d9e9bd8d3dc28d399d71acc6d4f6'}]}]
register_sample_bank('fds', FDS_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md', 'seed': 13498448389395540393, 'population': 156, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(817, 823), (154, 154), (517, 517), (523, 523), (241, 241), (780, 784), (168, 168), (592, 595), (908, 914), (261, 261), (831, 837), (838, 844), (247, 247), (521, 521), (670, 674), (213, 213), (628, 631), (894, 900), (398, 402), (929, 935)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md', 'seed': 16192987573911774730, 'population': 77, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(335, 335), (313, 313), (292, 292), (294, 294), (99, 99), (262, 262), (95, 95), (184, 184), (323, 323), (307, 307), (298, 298), (186, 186), (256, 256), (190, 190), (331, 331), (283, 283), (206, 206), (255, 255), (111, 111), (329, 329)]}])


In [ ]:
# | notest
FDS_SAMPLE_IDS = None
fds_sample_cases = select_cases(FDS_SAMPLE_CASES, FDS_SAMPLE_IDS)
display(pd.DataFrame(fds_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in fds_sample_cases])
fds_sample_metrics = await evaluation.run(fds_sample_cases, concurrency=EVALUATION_CONCURRENCY)
fds_sample_results = {case["id"]: evaluation.load(case) for case in fds_sample_cases}
display(fds_sample_metrics)
# Inspect a complete result: evaluation.inspect(fds_sample_cases[0]["id"])
evaluation.show_answers(fds_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_fds_7c3c2ef958_01,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,1,False,False,False,None,None,12986,0,True,unsupported,,,incorrect
sample_fds_7c3c2ef958_02,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,10388,0,True,unsupported,,,not_assessable
sample_fds_7c3c2ef958_03,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,16869,0,True,unsupported,,,not reviewed
sample_fds_7c3c2ef958_04,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,12512,0,True,unsupported,,,not reviewed
sample_fds_7c3c2ef958_05,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,prerequisite,prose,False,True,8,False,0,0,0,False,False,False,None,None,10288,0,True,unsupported,,,not reviewed
sample_fds_7c3c2ef958_06,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,9075,0,True,included,,,not reviewed
sample_fds_7c3c2ef958_07,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,10288,0,True,unsupported,,,not reviewed
sample_fds_7c3c2ef958_08,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,4,False,False,False,None,None,11535,0,True,included,,,not reviewed
sample_fds_7c3c2ef958_09,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,1,False,False,False,False,False,11551,0,True,unsupported,,,not reviewed
sample_fds_7c3c2ef958_10,fds,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md,V4.0 / A-0,instruction,prose,False,True,8,False,0,0,0,False,False,False,False,False,7750,0,True,included,,,not reviewed


### sample_fds_7c3c2ef958_01

根据《SX322016《新松机器人FDS应用使用手册》-A-0》（V4.0 / A-0），在“常见报警及解决方法”中，应如何理解或操作“报警：10201”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 新松机器人 FDS 应用操作手册（V5.0）中的报警信息表结构：包含报警码、报警信息、子码、子码报警信息及解决方法五个字段。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 789–795; corpus 4.0, revision V5.0 / A-1; object `1d939c9236e70d6edc7557f7f11088b8aa1654d89126b8a8a7a7abaed3581747`.

- 该表为新松机器人 FDS 系统的报警与提示代码对照表，共含 **5 列**（报警码、报警信息、子码、子码报警信息、解决方法）及 **33 行数据**，其中最后一组为**合并单元格**结构。

### 1. 表头结构
| 列1 | 列2 | 列3 | 列4 | 列5 |
| :--- | :--- | :--- | :--- | :--- |
| 报警码 | 报警信息 | 子码 | 子码报警信息 | 解决方法 |

### 2. 主要分类摘要

#### A. 报警类（需处理故障，含解决方法建议）
此部分主要涉及硬件异常、通信故障及操作错误，**列3-4（子码）均为空**。

| 报警码 | 报警信息 | 解决方法/建议 |
| :--- | :--- | :--- |
| **10201** | 文件操作失败 | 联系技术人员 |
| **10202** | 巡检异常 | 检查 PLC 设备状态 |
| **10203** | 配置 IO 不在逻辑 IO 分配范围内 | 检查 IO 配置 |
| **10204** | 机器人干涉 | 检查干涉区 |
| **10205** | 水流量异常 | 检查实际流量与设定流量值 |
| **10206** | 气流量异常 | 检查气压表状态 |
| **10207** | FDS-NT151 监控异常 | 检查 NT151 模块功能 |
| **10208** | FDS 焊接前状态异常 | *（无解决方法建议）* |
| **10209** | FDS 铆接超时 | 人工检查焊接状态 |
| **10210** | 铆点存在漏铆 | 人工补焊 |
| **10211** | 铆点存在多铆 | 人工除钉 |
| **10212** | 无效铆点检测 | 重新执行检测指令 |
| **10213** | 响应安全门请求暂停中 | 检查安全门状态 |
| **10214** | FDS 指令集错误 | 检查作业名称 |
| **10215** | 通讯状态异常 | 检查外设通讯线或模块 |

#### B. 提示类（状态通知或功能执行反馈，通常无解决方法）
此部分多为系统状态通知、功能执行结果或参数反馈，**列3-5 均为空**。

*   **模式/状态提示**：
    *   `10251`: 当前为非 NORMAL 模式
    *   `10254`: 响应安全门请求暂停中
    *   `10260`: 巡检配置为 0，功能未启用
    *   `10266`: 通讯为屏蔽状态
*   **功能执行/结果反馈**：
    *   `10252`: FDS 铆接清空堆栈完成
    *   `10257`: 执行 FDS 顶出元件快捷功能
    *   `10258`: 执行 FDS 供给元件快捷功能
    *   `10259`: 执行 FDS 伸出压板快捷功能
    *   `10261`: 执行 FDS 标定运行功能
    *   `10262`: 标定运行功能完成
*   **检测数据/限制**：
    *   `10253`: 不支持 1.0 算法
    *   `10263`: 铆点数量正常
    *   `10264`: 漏铆个数为: [数值]
    *   `10265`: 多铆个数为: [数值]

#### C. 特殊合并行（快捷键定义）
**提示码 10267** 对应“快捷键”，该行使用了 `rowspan=7` 合并了前两列，定义了 7 个子状态（子码 0-6）：

| 子码 | 含义 |
| :--- | :--- |
| **0** | 顶出元件完成 |
| **1** | 供给元件完成 |
| **2** | 顶出元件异常 |
| **3** | 供给元件异常 |
| **4** | 伸出压板完成 |
| **5** | 标定运行功能完成 |
| **6** | 标定运行功能异常 |

### 3. 关键观察
1.  **子码使用情况**：除最后一组（10267）外，所有报警和提示的“子码”和“子码报警信息”列均为空。
2.  **解决方法缺失**：部分报警（如 10208）和所有提示类代码均未提供具体的“解决方法”，提示类代码主要用于状态显示而非故障排除。
3.  **业务领域**：内容高度聚焦于 **FDS（自动供钉系统）** 与 **机器人铆接/焊接** 过程，涵盖气/水流、干涉检测、安全门联锁及通讯监控。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 788–1036; corpus 5.0, revision V5.0 / A-1; object `c24369423742a93cf67dd17b1b36cc0f259080bebe76fdd5f7b5900dbe6fc4ef`.

- 该表格来自新松机器人FDS应用操作手册，详细列出了报警（Alarm）与提示（Prompt/Status）代码、对应信息及处理方法。主要内容分类如下：

**1. 报警代码 (10201-10215)：需采取解决措施**
*   **系统/IO/通信类**：
    *   **10201** 文件操作失败：联系技术人员。
    *   **10202** 巡检异常：检查PLC设备状态。
    *   **10203** IO配置超出逻辑分配范围：检查IO配置。
    *   **10215** 通讯状态异常：检查外设通讯线或模块。
    *   **10207** FDS-NT151监控异常：检查NT151模块功能。
    *   **10214** FDS指令集错误：检查作业名称。
*   **流体/气压类**：
    *   **10205** 水流量异常：核对实际与设定流量值。
    *   **10206** 气流量异常：检查气压表状态。
*   **机械/安全类**：
    *   **10204** 机器人干涉：检查干涉区。
    *   **10213** 响应安全门请求暂停：检查安全门状态。
*   **焊接/铆接工艺类**：
    *   **10208** FDS焊接前状态异常（无特定解决方法）。
    *   **10209** FDS铆接超时：人工检查焊接状态。
    *   **10210** 铆点漏铆：人工补焊。
    *   **10211** 铆点多铆：人工除钉。
    *   **10212** 无效铆点检测：重新执行检测指令。

**2. 提示代码 (10251-10266)：状态反馈或功能执行通知**
*   **状态/模式**：
    *   **10251** 非NORMAL模式。
    *   **10254** 响应安全门请求暂停中。
    *   **10260** 巡检配置为0（功能未启用）。
    *   **10266** 通讯处于屏蔽状态。
    *   **10253** 不支持1.0算法。
*   **功能执行反馈**：
    *   **10252** 铆接清空堆栈完成。
    *   **10257/10258/10259** 分别执行顶出、供给、伸出压板快捷功能。
    *   **10261/10262** 执行并完成FDS标定运行功能。
    *   **10263/10264/10265** 铆点检测结果反馈（数量正常、漏铆个数、多铆个数）。

**3. 特殊结构 (10267)**
*   **10267** 对应“快捷键”状态定义（合并单元格），包含子码0-6：
    *   0-1: 顶出/供给元件完成
    *   2-3: 顶出/供给元件异常
    *   4: 伸出压板完成
    *   5-6: 标定运行功能完成/异常

**总结**：该表是FDS设备故障诊断与状态监控的核心参考，其中102xx系列代码前段（10201-10215）侧重于故障排查与处理，后段（10251+）侧重于系统状态确认与操作反馈。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 788–1036; corpus 4.0, revision V5.0 / A-1; object `8e8b8cdb0f62e88d28a4c5b8b9cc49868958171c92d1cfc6ce7c6787b17ad17d`.

- 文档《新松机器人FDS应用操作手册》(SX322016) 版本 A/0 于 2025年4月28日由沙景超完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 1047–1052; corpus 4.0, revision V5.0 / A-1; object `9b35575531b186fb1af2f9ac1c95758165feab936451908aee1736f2355c8c3a`.

- 新松机器人FDS应用操作手册中，报警代码 USERALM CODE=2603 对应的故障描述为“少铆”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 426–430; corpus 4.0, revision V5.0 / A-1; object `00d6367193a3a96da432de5e8507bacba3eef5411518248a8ec6e3b091a4777e`.

- 该行（索引29）来自《新松机器人FDS应用操作手册》，包含两列有效信息：第1列为“提示：10265”，第2列为“多铆个数为:”，其余三列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 985–991; corpus 4.0, revision V5.0 / A-1; object `74adb91a38c89e04d4fc41727f9de7c2343f6654fe3e146e510bef3e9b6f12a6`.

- 新松机器人FDS应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 7–9; corpus 4.0, revision V5.0 / A-1; object `085d53ff3f614abae97e48f92dc9ee9cb50f7f7d99212bb941f67f5e9e26c010`.

- 文档编号为 SX322001 的《新松机器人通用操作手册》条目。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人打磨应用操作手册》（A-2)/SX322006《新松机器人打磨应用操作手册》（A-2).md`, lines 477–480; corpus 4.0, revision V5.0 / A-2; object `eeb85144aed415dcf80f7a5e2ade6f7654973cee88d2d18e0a5fd01271a65bc9`.

#### Answer

根据手册证据（E1、E2），报警码10201的含义为

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: supported; revision fidelity: mixed.

The answer is a fragment and does not provide the requested explanation of the operation, purpose, or result for alarm 10201. It only states that the meaning is, without completing the sentence. The supplied evidence (E1, E2) clearly indicates that alarm 10201 means '文件操作失败' (File operation failed) and the resolution is '联系技术人员' (Contact technical personnel). The answer fails to convey these facts. While the citation to E1 and E2 is valid, the content is incomplete and does not answer the question.

...

#### Answer

The question asks for the parameter meanings, allowed values, or unit requirements explicitly listed for the configuration rule '通讯状态IO起始位需在逻辑IO输入IO分配的通讯状态范围内'. The retrieved evidence (E3, E8) only contains alarm code 10203 '配置IO不在逻辑IO分配范围内' with resolution '检查IO配置', which indicates the consequence of violating the rule but does not provide a parameter specification table listing the parameter name, data type, allowed range, default value, or unit for the communication-status IO starting bit. No other evidence entry contains a configuration parameter table with values or units for this specific requirement. Therefore the evidence cannot support a contextual example that states parameter values or units.

### sample_fds_c9f17d0e88_20

根据《SX322016《新松机器人FDS应用操作手册》(A-1)》（V5.0 / A-1），在“工艺作业模版”中，应如何理解或操作“Atlas—前台作业”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 文档《新松机器人FDS应用操作手册》(SX322016) 版本 A/0 于 2025年4月28日由沙景超完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 1047–1052; corpus 4.0, revision V5.0 / A-1; object `9b35575531b186fb1af2f9ac1c95758165feab936451908aee1736f2355c8c3a`.

- 新松机器人FDS应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 7–9; corpus 4.0, revision V5.0 / A-1; object `085d53ff3f614abae97e48f92dc9ee9cb50f7f7d99212bb941f67f5e9e26c010`.

- 新松机器人FDS应用操作手册（V5.0版本）中表格第16行（索引15）的数据：第2列和第3列的值均为“L3”，第1列为空，未显示表头或合并单元格继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 714–718; corpus 5.0, revision V5.0 / A-1; object `944bcc247c945d2f95e34c54b9176b3bfd5c3f92009aa85d9684b8efc172c99b`.

- 该表格行来自新松机器人FDS应用操作手册，显示：第二列为“NOP”，第三列为“开始”，第一列为空（可能继承上方合并单元格的值）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 300–304; corpus 4.0, revision V5.0 / A-1; object `a64f0965d753dd3873a9d8bb2e55ff8204d79098a95fd786172fb7bae4eef014`.

- 新松机器人 FDS 应用操作手册（V5.0）中的报警信息表结构：包含报警码、报警信息、子码、子码报警信息及解决方法五个字段。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 789–795; corpus 4.0, revision V5.0 / A-1; object `1d939c9236e70d6edc7557f7f11088b8aa1654d89126b8a8a7a7abaed3581747`.

- 文档《新松机器人FDS应用操作手册》(V5.0)的修订记录：2025年4月28日，沙景超完成初版 (A/0)。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 1047–1052; corpus 5.0, revision V5.0 / A-1; object `5a5445f9a4ca2663887da25fda038d802680653e979347f36f6d4542139174d1`.

- 该表格为**新松机器人 FDS 应用操作手册**的标题信息块，属于工业机器人技术资料，第一行为空，后续两行分别显示文档主标题及技术类别。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 3–13; corpus 4.0, revision V5.0 / A-1; object `b18fc5d2b2479a608a3330992d8e6dd8e35cf5558335ab2e4890be7e492ba50c`.

- 该行（索引29）来自《新松机器人FDS应用操作手册》，包含两列有效信息：第1列为“提示：10265”，第2列为“多铆个数为:”，其余三列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人FDS应用操作手册》(A-1)/SX322016《新松机器人FDS应用操作手册》(A-1).md`, lines 985–991; corpus 4.0, revision V5.0 / A-1; object `74adb91a38c89e04d4fc41727f9de7c2343f6654fe3e146e510bef3e9b6f12a6`.

#### Answer

在所提供的检索证据（E1–E8）中，未找到任何关于

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: faithful.

The question asks for an explanation of 'Atlas—前台作业' (Atlas—Front-end Operation) within the '工艺作业模版' (Process Operation Template) section of the manual. The supplied evidence (E1-E8) contains revision history, title pages, code snippets for specific tasks (riveting/welding logic), and alarm codes. None of the provided evidence snippets contain the text 'Atlas—前台作业' or any description of its operation, purpose, or results. The reference list mentions '1. **Atlas—前台作业**', but the actual content explaining it is not present in the supplied evidence blocks. Therefore, the model correctly identified that the necessary information was missing from the provided context and abstained from answering. This is an appropriate abstention because the evidence is inadequate to answer the specific question about the 'Atlas' template.

## Tracking — sampled manual revisions

60 cases from 3 CSV files. Set `TRACKING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
TRACKING_SAMPLE_CASES = [{'id': 'sample_tracking_68d3839791_01', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“报警说明”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警等级 报警信息 报警原因 处理措施'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 604, 'line_end': 610, 'excerpt': '<tr>\n<td><p>报警码</p></td>\n<td><p>报警等级</p></td>\n<td><p>报警信息</p></td>\n<td><p>报警原因</p></td>\n<td><p>处理措施</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_02', 'category': 'tracking', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“报警说明”列出的报警7615表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7615'], 'expected_facts': ['7615 警告（暂停） 编码器数据存储异常 软件内部存储编码器滤波值的变量异常 重启机器人即可，出现该报警联系售后人员'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 646, 'line_end': 652, 'excerpt': '<tr>\n<td><p>7615</p></td>\n<td><p>警告（暂停）</p></td>\n<td><p>编码器数据存储异常</p></td>\n<td><p>软件内部存储编码器滤波值的变量异常</p></td>\n<td><p>重启机器人即可，出现该报警联系售后人员。</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_03', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.5', 'section_title': '2.5 GR\\_MOVLP', 'heading_path': ['2 跟随指令及快捷键', '2.5 GR\\_MOVLP'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），配置“GR\\_MOVLP”时，“参数1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 参数1 全局位置变量'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 361, 'line_end': 365, 'excerpt': '<tr>\n<td rowspan="3"><p>说明</p></td>\n<td><p>参数1</p></td>\n<td><p>全局位置变量</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_04', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），配置“速度计算数据（非速度传输型）”时，“速度滤波值”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['3. 速度滤波值：范围为1-1000，若传送带速度波动过大，可适当设置该值对速度进行均值滤波，修改后重启机器人即可使速度波动减小，默认值为20'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 221, 'line_end': 221, 'excerpt': '3. 速度滤波值：范围为1-1000，若传送带速度波动过大，可适当设置该值对速度进行均值滤波，修改后重启机器人即可使速度波动减小，默认值为20。', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_05', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.2', 'section_title': '1.4.2 编码器配置', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.2 编码器配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“编码器配置”中，应如何理解或操作“通路确认”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3. 通路确认：该界面共有两页，可配置两路编码器或实现两路plc传输速度', '例如，在第一路传送带界面中将通路确认设置为1，即可实现第一路编码器值的获取或第一路plc对速度的获取'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 192, 'line_end': 192, 'excerpt': '3. 通路确认：该界面共有两页，可配置两路编码器或实现两路plc传输速度。例如，在第一路传送带界面中将通路确认设置为1，即可实现第一路编码器值的获取或第一路plc对速度的获取。', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_06', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.5', 'section_title': '1.4.5 跟随文件', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.5 跟随文件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“跟随文件”中“其中”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['其中，Px为沿用户系跟随距离的一部分（具体参照指令说明），Py和Pz分别为在随动过程中用户系下的Y、Z方向偏移量'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 235, 'line_end': 235, 'excerpt': '其中，Px为沿用户系跟随距离的一部分（具体参照指令说明），Py和Pz分别为在随动过程中用户系下的Y、Z方向偏移量。', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_07', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.6', 'section_title': '2.6 FL\\_MOVL指令', 'heading_path': ['2 跟随指令及快捷键', '2.6 FL\\_MOVL指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“FL\\_MOVL指令”中“FL_MOVL”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_MOVL'], 'expected_facts': ['FL_MOVL 功能 跟随运动指令，该指令不需要示教点位，用户X方向机器人运动速度与参数1设定速度一致，机器人沿用户X方向运动距离为参数2+随动位置设置界面px', '沿用户系YZ方向运动距离为随动位置设置界面py/pz值'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 385, 'line_end': 389, 'excerpt': '<tr>\n<td rowspan="6"><p>FL&#95;MOVL</p></td>\n<td><p>功能</p></td>\n<td colspan="2"><p>跟随运动指令，该指令不需要示教点位，用户X方向机器人运动速度与参数1设定速度一致，机器人沿用户X方向运动距离为参数2&#43;随动位置设置界面px；沿用户系YZ方向运动距离为随动位置设置界面py/pz值。</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_08', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.4', 'section_title': '2.4 GR\\_MOVL', 'heading_path': ['2 跟随指令及快捷键', '2.4 GR\\_MOVL'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“GR\\_MOVL”中，应如何理解或操作“GL_MOVL VL=R1 T=R2 #8”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['举例 GL_MOVL VL=R1 T=R2 #8'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 341, 'line_end': 344, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td colspan="2"><p>GL&#95;MOVL VL=R1 T=R2 &#35;8</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_09', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“作业模板”中，应如何理解或操作“IRQOFF  LEV=1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['IRQOFF LEV=1 //关闭相同等级中断'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 546, 'line_end': 549, 'excerpt': '<tr>\n<td><p>IRQOFF  LEV=1  </p></td>\n<td><p>//关闭相同等级中断</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_10', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“作业模板”中“FL_MOVL VL=R10 T=R2 #2”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_MOVL', 'R10'], 'expected_facts': ['FL_MOVL VL=R10 T=R2 #2 //可继续添加FL_MOVL指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 542, 'line_end': 545, 'excerpt': '<tr>\n<td><p>FL&#95;MOVL VL=R10 T=R2 &#35;2</p></td>\n<td><p>//可继续添加FL&#95;MOVL指令</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_11', 'category': 'tracking', 'kind': 'prerequisite', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），操作“作业模板”时，涉及“SET VEL=100”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['SET VEL=100 //速度百分比（必须为100）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 486, 'line_end': 489, 'excerpt': '<tr>\n<td><p>SET VEL=100        </p></td>\n<td><p>//速度百分比（必须为100）</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_12', 'category': 'tracking', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“报警说明”列出的报警7602表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7602'], 'expected_facts': ['7602 警告（暂停） 传送带速度或板宽未设定 FL_MOVL/GR_MOVL指令前两个参数为0 查看FL_MOVL/GR_MOVL指令运动速度和跟随距离参数对应的实型变量是否为0，若为0赋值即可'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 618, 'line_end': 624, 'excerpt': '<tr>\n<td><p>7602</p></td>\n<td><p>警告（暂停）</p></td>\n<td><p>传送带速度或板宽未设定</p></td>\n<td><p>FL&#95;MOVL/GR&#95;MOVL指令前两个参数为0</p></td>\n<td><p>查看FL&#95;MOVL/GR&#95;MOVL指令运动速度和跟随距离参数对应的实型变量是否为0，若为0赋值即可。</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_13', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.4', 'section_title': '2.4 GR\\_MOVL', 'heading_path': ['2 跟随指令及快捷键', '2.4 GR\\_MOVL'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），配置“GR\\_MOVL”时，“参数1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['说明 参数1 编码器读取的传送带的实时速度，单位mm/s'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 328, 'line_end': 332, 'excerpt': '<tr>\n<td rowspan="3"><p>说明</p></td>\n<td><p>参数1</p></td>\n<td><p>编码器读取的传送带的实时速度，单位mm/s</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_14', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），配置“速度计算数据（非速度传输型）”时，“在机器人的“其他参数“界面中”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在机器人的“其他参数“界面中，编码器MAC\\_ID要和DevNet的ID相同对应上方DevNet的值设置一致，编码器转动方向1为逆时针、2为顺时针（顺时针为编码器值递增，逆时针为编码器值递减', '编码器值过0时，需要知道编码器转向），如下图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 211, 'line_end': 211, 'excerpt': '在机器人的“其他参数“界面中，编码器MAC\\_ID要和DevNet的ID相同对应上方DevNet的值设置一致，编码器转动方向1为逆时针、2为顺时针（顺时针为编码器值递增，逆时针为编码器值递减。编码器值过0时，需要知道编码器转向），如下图所示。', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_15', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.1', 'section_title': '1.1 环境配置（编码器DevNet通讯）', 'heading_path': ['1 跟随应用配置', '1.1 环境配置（编码器DevNet通讯）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“环境配置（编码器DevNet通讯）”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['路径：用户配置—>外部通讯配置—>开关配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 132, 'line_end': 132, 'excerpt': '路径：用户配置—>外部通讯配置—>开关配置；', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_16', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“速度计算数据（非速度传输型）”中“速度判定最小/最大值”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'OFF'], 'expected_facts': ['4. 速度判定最小/最大值：传送带速度监控指令（FL\\_VERROR）执行后，速度若连续5拍（5\\*16 = 80ms）在该范围区间内，则速度状态监控OUT信号输出一次OFF（0）', '若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）', '若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 222, 'line_end': 222, 'excerpt': '4. 速度判定最小/最大值：传送带速度监控指令（FL\\_VERROR）执行后，速度若连续5拍（5\\*16 = 80ms）在该范围区间内，则速度状态监控OUT信号输出一次OFF（0）；若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）；若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值。执行FL\\_CLOSE指令后可关闭该功能，关闭功能后速度状态监控OUT信号不会随速度变化发生变化。', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_17', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.2', 'section_title': '2.2 FL\\_VERROR 指令', 'heading_path': ['2 跟随指令及快捷键', '2.2 FL\\_VERROR 指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“FL\\_VERROR 指令”中“FL_VERROR”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_VERROR'], 'expected_facts': ['举例 FL_VERROR'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 284, 'line_end': 287, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td><p>FL&#95;VERROR</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_18', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '2.5', 'section_title': '2.5 GR\\_MOVLP', 'heading_path': ['2 跟随指令及快捷键', '2.5 GR\\_MOVLP'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“GR\\_MOVLP”中，应如何理解或操作“GL_MOVLP  P1 VL=R1 T=R2”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['举例 GL_MOVLP P1 VL=R1 T=R2'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 374, 'line_end': 377, 'excerpt': '<tr>\n<td><p>举例</p></td>\n<td colspan="2"><p>GL&#95;MOVLP  P1 VL=R1 T=R2</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_19', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“作业模板”中，应如何理解或操作“PAUSE”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['PAUSE //机器人暂停'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 579, 'line_end': 582, 'excerpt': '<tr>\n<td><p>PAUSE</p></td>\n<td><p>//机器人暂停</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_68d3839791_20', 'category': 'tracking', 'kind': 'alarm', 'profile': '', 'revision': 'V4.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'source_version': '', 'question': '根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），“报警说明”列出的报警7613表示什么，原文给出了哪些原因或处理办法？', 'anchors': ['7613'], 'expected_facts': ['7613 警告（暂停） 未配置编码器旋转方向 【跟随—普通随动—其他参数】界面中未配置编码器旋转方向 配置编码器旋转方向并保存后复位即可'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'line_start': 639, 'line_end': 645, 'excerpt': '<tr>\n<td><p>7613</p></td>\n<td><p>警告（暂停）</p></td>\n<td><p>未配置编码器旋转方向</p></td>\n<td><p>【跟随—普通随动—其他参数】界面中未配置编码器旋转方向</p></td>\n<td><p>配置编码器旋转方向并保存后复位即可</p></td>\n</tr>', 'sha256': '5ad74dc1726042828c56b51ae170b8298873d7d9c13c0ae25c1e42967b9c3f53'}]}, {'id': 'sample_tracking_6e5b1c4749_01', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 报警说明', 'heading_path': ['4 报警说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“报警说明”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_MOVL', 'GR_MOVL'], 'expected_facts': ['报警码 报警等级 报警信息 报警原因 处理措施 7600 警告（暂停） 追随速度过大 跟随运动指令运动速度过大 查看FL_MOVL/GR_MOVL指令参数，适当降低跟随速度（指令第一个参数）或适当增大跟随/追赶距离（指令第二个参数） 7602 警告（暂停） 传送带速度或板宽未设定 FL_MOVL/GR_MOVL指令前两个参数为0 查看FL_MOVL/GR_MOVL指令运动速度和跟随距离参数对应的实型变量是否为0，若为0赋值即可', '7603 警告（暂停） 文件操作失败 系统参数问题 请联系软件人员查看内部存储是否正常 7604 警告（暂停） DeviceNet连接断开 编码器连接失败 查看编码器连线，硬件无问题则依照说明书对软件配置进行查看', '7613 警告（暂停） 未配置编码器旋转方向 【跟随—普通随动—其他参数】界面中未配置编码器旋转方向 配置编码器旋转方向并保存后复位即可 7615 警告（暂停） 编码器数据存储异常 软件内部存储编码器滤波值的变量异常 重启机器人即可，出现该报警联系售后人员'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 238, 'line_end': 238, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警等级</p></td><td><p>报警信息</p></td><td><p>报警原因</p></td><td><p>处理措施</p></td></tr><tr><td><p>7600</p></td><td><p>警告（暂停）</p></td><td><p>追随速度过大</p></td><td><p>跟随运动指令运动速度过大</p></td><td><p>查看FL_MOVL/GR_MOVL指令参数，适当降低跟随速度（指令第一个参数）或适当增大跟随/追赶距离（指令第二个参数）</p></td></tr><tr><td><p>7602</p></td><td><p>警告（暂停）</p></td><td><p>传送带速度或板宽未设定</p></td><td><p>FL_MOVL/GR_MOVL指令前两个参数为0</p></td><td><p>查看FL_MOVL/GR_MOVL指令运动速度和跟随距离参数对应的实型变量是否为0，若为0赋值即可。</p></td></tr><tr><td><p>7603</p></td><td><p>警告（暂停）</p></td><td><p>文件操作失败</p></td><td><p>系统参数问题</p></td><td><p>请联系软件人员查看内部存储是否正常</p></td></tr><tr><td><p>7604</p></td><td><p>警告（暂停）</p></td><td><p>DeviceNet连接断开</p></td><td><p>编码器连接失败</p></td><td><p>查看编码器连线，硬件无问题则依照说明书对软件配置进行查看。</p></td></tr><tr><td><p>7613</p></td><td><p>警告（暂停）</p></td><td><p>未配置编码器旋转方向</p></td><td><p>【跟随—普通随动—其他参数】界面中未配置编码器旋转方向</p></td><td><p>配置编码器旋转方向并保存后复位即可</p></td></tr><tr><td><p>7615</p></td><td><p>警告（暂停）</p></td><td><p>编码器数据存储异常</p></td><td><p>软件内部存储编码器滤波值的变量异常</p></td><td><p>重启机器人即可，出现该报警联系售后人员。</p></td></tr><tr><td><p>7616</p></td><td><p>警告（暂停）</p></td><td><p>未进行profinet通讯,不能传输速度!</p></td><td><p>实时传输速度值只能在profinet通讯下进行，当前通讯方式为devicenet通讯。</p></td><td><p>修改通讯方式为profinet通讯即可。</p></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_02', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322017《新松机器人跟随应用操作手册》(A-0).md”（V5.0 / A-0），作业模板中MOVJ、PAUSE、DELAY的执行顺序和作用是什么？', 'anchors': ['MOVJ', 'PAUSE', 'DELAY'], 'expected_facts': ['MOVJ VJ=10 //传送带速度异常，机器人移动到安全位置', 'MOVJ VJ=10 //机器人返回原位', 'PAUSE //机器人暂停', 'DELAY T=1s'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 234, 'line_end': 234, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVJ  VJ=10</p></td><td><p>//传送带速度异常，机器人移动到安全位置</p></td></tr><tr><td><p>MOVJ  VJ=10</p></td><td><p>//机器人返回原位</p></td></tr><tr><td><p>PAUSE</p></td><td><p>//机器人暂停</p></td></tr><tr><td><p>DELAY   T=1s</p></td><td></td></tr><tr><td><p>RESUME</p></td><td></td></tr><tr><td><p>RET  </p></td><td><p>//返回执行作业</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_03', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.3', 'section_title': '1.3 用户坐标系标定', 'heading_path': ['1 跟随应用配置', '1.3 用户坐标系标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“用户坐标系标定”中，应如何理解或操作“路径”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['路径：坐标->用户坐标->标定;'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 103, 'line_end': 103, 'excerpt': '路径：坐标->用户坐标->标定;', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_04', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322017《新松机器人跟随应用操作手册》(A-0).md”（V5.0 / A-0），作业模板中SET、VEL、MOVJ的执行顺序和作用是什么？', 'anchors': ['SET', 'VEL', 'MOVJ'], 'expected_facts': ['SET VEL=100 //速度百分比（必须为100）', 'SET UF #1 //用户坐标系', 'SET TF #1 //工具坐标系', 'MOVJ VJ=10 //机器人运动原位', 'SET VEL=100 //再次赋值防止变化', 'MOVJ VJ=10 //机器人跟随运动初始位置', 'MOVJ VJ=10 //回到机器人跟随运动初始位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 230, 'line_end': 230, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>SET VEL=100        </p></td><td><p>//速度百分比（必须为100）</p></td></tr><tr><td><p>SET UF  #1</p></td><td><p>//用户坐标系</p></td></tr><tr><td><p>SET TF  #1  </p></td><td><p>//工具坐标系</p></td></tr><tr><td><p>MOVJ  VJ=10</p></td><td><p>//机器人运动原位</p></td></tr><tr><td><p>FL_VERROR</p></td><td><p>//开启传送带速度监控</p></td></tr><tr><td><p>DELAY   T=1s</p></td><td></td></tr><tr><td><p>L1</p></td><td></td></tr><tr><td><p>IRQON  LEV=1  OT#1=1  SUB1</p></td><td><p>//中断检测速度监控信号，若传送带速度正常（监控信号输出0），继续执行后续作业，检测到速度异常（输出1），跳出当前作业运行中断子作业SUB1</p></td></tr><tr><td><p>SET VEL=100</p></td><td><p>//再次赋值防止变化</p></td></tr><tr><td><p>MOVJ  VJ=10  </p></td><td><p>//机器人跟随运动初始位置</p></td></tr><tr><td><p>WAIT IN#02=1 T=-1</p></td><td><p>//等待光电开关信号</p></td></tr><tr><td><p>FL_GETV  R10  F1 </p></td><td><p>//获取第一路传送带实时速度</p></td></tr><tr><td><p>GR_MOVP  PR1 VL=R10 T=R9 </p></td><td><p>//追捕</p></td></tr><tr><td><p>FL_MOVL VL=R10 T=R1 #1  </p></td><td><p>//跟随目标运动，可进行偏移</p></td></tr><tr><td><p>FL_MOVL VL=R10 T=R2 #2</p></td><td><p>//可继续添加FL_MOVL指令</p></td></tr><tr><td><p>IRQOFF  LEV=1  </p></td><td><p>//关闭相同等级中断</p></td></tr><tr><td><p>MOVJ  VJ=10</p></td><td><p>//回到机器人跟随运动初始位置</p></td></tr><tr><td><p>GOTO   L1</p></td><td></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_05', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.5', 'section_title': '1.4.5 跟随文件', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.5 跟随文件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“跟随文件”中“其中”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['其中，Px为沿用户系跟随距离的一部分（具体参照指令说明），Py和Pz分别为在随动过程中用户系下的Y、Z方向偏移量'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 178, 'line_end': 178, 'excerpt': '其中，Px为沿用户系跟随距离的一部分（具体参照指令说明），Py和Pz分别为在随动过程中用户系下的Y、Z方向偏移量。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_06', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），配置“速度计算数据（非速度传输型）”时，“在机器人的“其他参数“界面中”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在机器人的“其他参数“界面中，编码器MAC\\_ID要和DevNet的ID相同对应上方DevNet的值设置一致，编码器转动方向1为逆时针、2为顺时针（顺时针为编码器值递增，逆时针为编码器值递减', '编码器值过0时，需要知道编码器转向），如下图所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 154, 'line_end': 154, 'excerpt': '在机器人的“其他参数“界面中，编码器MAC\\_ID要和DevNet的ID相同对应上方DevNet的值设置一致，编码器转动方向1为逆时针、2为顺时针（顺时针为编码器值递增，逆时针为编码器值递减。编码器值过0时，需要知道编码器转向），如下图所示。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_07', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.3', 'section_title': '2.3 FL\\_CLOSE 指令', 'heading_path': ['2 跟随指令及快捷键', '2.3 FL\\_CLOSE 指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“FL\\_CLOSE 指令”中“FL_CLOSE”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_CLOSE', 'F_CLOSE'], 'expected_facts': ['FL_CLOSE 功能 速度监控关闭 格式 F_CLOSE 说明 该指令不区分通路，使用后两路速度检测均关闭，再次使用前需再开启FL_VERROR 举例 F_CLOSE'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 196, 'line_end': 196, 'excerpt': '<table><tr><td rowspan="4"><p>FL_CLOSE</p></td><td><p>功能</p></td><td><p>速度监控关闭</p></td></tr><tr><td><p>格式</p></td><td><p>F_CLOSE</p></td></tr><tr><td><p>说明</p></td><td><p>该指令不区分通路，使用后两路速度检测均关闭，再次使用前需再开启FL_VERROR</p></td></tr><tr><td><p>举例</p></td><td><p>F_CLOSE</p></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_08', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3', 'section_title': '3 作业模板', 'heading_path': ['3 作业模板'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“作业模板”中，应如何理解或操作“执行作业”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 执行作业：T=R部分，R值设置合理'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 228, 'line_end': 228, 'excerpt': '1. 执行作业：T=R部分，R值设置合理。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_09', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 环境配置（编码器DevNet通讯）', 'heading_path': ['1 跟随应用配置', '1.1 环境配置（编码器DevNet通讯）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“环境配置（编码器DevNet通讯）”中，应如何理解或操作“速度获取方式用DevNet方式”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['速度获取方式用DevNet方式，DevNet通讯线连接方式如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 63, 'line_end': 63, 'excerpt': '速度获取方式用DevNet方式，DevNet通讯线连接方式如下：', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_10', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“速度计算数据（非速度传输型）”中“速度判定最小/最大值”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'OFF'], 'expected_facts': ['4. 速度判定最小/最大值：传送带速度监控指令（FL\\_VERROR）执行后，速度若连续5拍（5\\*16 = 80ms）在该范围区间内，则速度状态监控OUT信号输出一次OFF（0）', '若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）', '若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 165, 'line_end': 165, 'excerpt': '4. 速度判定最小/最大值：传送带速度监控指令（FL\\_VERROR）执行后，速度若连续5拍（5\\*16 = 80ms）在该范围区间内，则速度状态监控OUT信号输出一次OFF（0）；若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）；若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值。执行FL\\_CLOSE指令后可关闭该功能，关闭功能后速度状态监控OUT信号不会随速度变化发生变化。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_11', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.4', 'section_title': '1.4.4 速度计算数据（非速度传输型）', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.4 速度计算数据（非速度传输型）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），配置“速度计算数据（非速度传输型）”时，“速度状态监控OUT信号”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['5. 速度状态监控OUT信号：范围1-1000，例如，若填入1，则传送带速度符合速度判定区间时，将【IO—用户输出—用户IO 1】状态置为OFF', '传送带速度超出速度判定区间时，将【IO—用户输出—用户IO 1】状态置为ON'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 166, 'line_end': 166, 'excerpt': '5. 速度状态监控OUT信号：范围1-1000，例如，若填入1，则传送带速度符合速度判定区间时，将【IO—用户输出—用户IO 1】状态置为OFF；传送带速度超出速度判定区间时，将【IO—用户输出—用户IO 1】状态置为ON。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_12', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.2', 'section_title': '2.2 FL\\_VERROR 指令', 'heading_path': ['2 跟随指令及快捷键', '2.2 FL\\_VERROR 指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“FL\\_VERROR 指令”中“执行该指令且未执行FL\\_CLOSE指令后”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OUT', 'OFF'], 'expected_facts': ['执行该指令且未执行FL\\_CLOSE指令后，速度若连续5拍（5\\*16 = 80ms）在设定的传送带速度范围区间内，则速度状态监控OUT信号输出一次OFF（0）', '若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）', '若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 192, 'line_end': 192, 'excerpt': '执行该指令且未执行FL\\_CLOSE指令后，速度若连续5拍（5\\*16 = 80ms）在设定的传送带速度范围区间内，则速度状态监控OUT信号输出一次OFF（0）；若连续5拍不在该范围区间内，则速度状态监控OUT信号输出一次ON（1）；若5拍内速度位于区间边缘波动，则保持波动前的OUT值，直到连续5拍速度状态稳定后将 OUT信号重新赋值。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_13', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 速度获取', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.3 速度获取'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“速度获取”中，应如何理解或操作“Profinet编码器值初始IO”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. Profinet编码器值初始IO，需要占用连续32位IO，起始IO为1，编码器值或速度值，从低到高依次从用户输入IO33—IO64读取转换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 146, 'line_end': 146, 'excerpt': '2. Profinet编码器值初始IO，需要占用连续32位IO，起始IO为1，编码器值或速度值，从低到高依次从用户输入IO33—IO64读取转换。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_14', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 速度获取', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.3 速度获取'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），配置“速度获取”时，“在“编码器配置“界面中第一路传送带的”通路确认“设置为1”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在“编码器配置“界面中第一路传送带的”通路确认“设置为1，第二路设置为0，则该界面第一路传送带参数生效，第二路未配置不生效'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 150, 'line_end': 150, 'excerpt': '在“编码器配置“界面中第一路传送带的”通路确认“设置为1，第二路设置为0，则该界面第一路传送带参数生效，第二路未配置不生效。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_15', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.4', 'section_title': '2.4 GR\\_MOVL', 'heading_path': ['2 跟随指令及快捷键', '2.4 GR\\_MOVL'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“GR\\_MOVL”中“GR_MOVL”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['GR_MOVL', 'GL_MOVL'], 'expected_facts': ['GR_MOVL 功能 追赶运动指令，该指令需要示教点位，使机器人在设定用户系X方向从静止运动至参数1设定速度，同时用户X方向到达示教点+参数2设定距离的位置，YZ方向为示教点用户YZ方向位置', '格式 GR_MOVL VL=R<参数1> T=R<参数2> #<参数项 3> 说明 参数1 编码器读取的传送带的实时速度，单位mm/s 参数2 目标跟踪距离的R变量的编号，宽度单位：mm 参数3 参数文件号 举例 GL_MOVL VL=R1 T=R2 #8'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 202, 'line_end': 202, 'excerpt': '<table><tr><td rowspan="6"><p>GR_MOVL </p></td><td><p>功能</p></td><td colspan="2"><p>追赶运动指令，该指令需要示教点位，使机器人在设定用户系X方向从静止运动至参数1设定速度，同时用户X方向到达示教点+参数2设定距离的位置，YZ方向为示教点用户YZ方向位置。</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>GR_MOVL VL=R&lt;参数1&gt; T=R&lt;参数2&gt; #&lt;参数项 3&gt;</p></td></tr><tr><td rowspan="3"><p>说明</p></td><td><p>参数1</p></td><td><p>编码器读取的传送带的实时速度，单位mm/s</p></td></tr><tr><td><p>参数2</p></td><td><p>目标跟踪距离的R变量的编号，宽度单位：mm</p></td></tr><tr><td><p>参数3</p></td><td><p>参数文件号</p></td></tr><tr><td><p>举例</p></td><td colspan="2"><p>GL_MOVL VL=R1 T=R2 #8</p></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_16', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 环境配置（编码器DevNet通讯）', 'heading_path': ['1 跟随应用配置', '1.1 环境配置（编码器DevNet通讯）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“环境配置（编码器DevNet通讯）”中，应如何理解或操作“外设通讯配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['外设通讯配置，DevNet开关打开'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 73, 'line_end': 73, 'excerpt': '外设通讯配置，DevNet开关打开。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_17', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 速度获取', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.3 速度获取'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），配置“速度获取”时，“通讯方式选择”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 通讯方式选择：设置为1，数据获取方式为Devicenet', '设置为2，数据获取方式为profinet，需要继续配置下面三个参数', '以Profinet举例'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 145, 'line_end': 145, 'excerpt': '1. 通讯方式选择：设置为1，数据获取方式为Devicenet；设置为2，数据获取方式为profinet，需要继续配置下面三个参数。以Profinet举例。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_18', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.1', 'section_title': '2.1 FL\\_GETV 指令', 'heading_path': ['2 跟随指令及快捷键', '2.1 FL\\_GETV 指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），“FL\\_GETV 指令”中“FL_GETV”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['FL_GETV'], 'expected_facts': ['FL_GETV 功能 读取实时速度 格式 FL_GETV R<参数1> F<参数2> 说明 参数1 读取编码器的传送带的实时速度并赋值给用户实型R1，单位mm/s', '参数2 通路选择，1对应第1路编码器，2对应第2路编码器 举例 FL_GETV R1 F1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 184, 'line_end': 184, 'excerpt': '<table><tr><td rowspan="5"><p>FL_GETV</p></td><td><p>功能</p></td><td colspan="2"><p>读取实时速度</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>FL_GETV   R&lt;参数1&gt; F&lt;参数2&gt;</p></td></tr><tr><td rowspan="2"><p>说明</p></td><td><p>参数1</p></td><td><p>读取编码器的传送带的实时速度并赋值给用户实型R1，单位mm/s。</p></td></tr><tr><td><p>参数2</p></td><td><p>通路选择，1对应第1路编码器，2对应第2路编码器</p></td></tr><tr><td><p>举例</p></td><td colspan="2"><p>FL_GETV R1 F1</p></td></tr></table>', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_19', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.1', 'section_title': '1.1 环境配置（编码器DevNet通讯）', 'heading_path': ['1 跟随应用配置', '1.1 环境配置（编码器DevNet通讯）'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），在“环境配置（编码器DevNet通讯）”中，应如何理解或操作“进入DevNet界面进行配置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['进入DevNet界面进行配置，将使能状态打至ON，其MAC\\_ID要与编码器ID一致，波特率为默认125，触发方式默认为轮询，连接方式默认为Group2，输入长度为4字节（根据编码器具体填写），输出长度为0字节，点击保存并重启'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 81, 'line_end': 81, 'excerpt': '进入DevNet界面进行配置，将使能状态打至ON，其MAC\\_ID要与编码器ID一致，波特率为默认125，触发方式默认为轮询，连接方式默认为Group2，输入长度为4字节（根据编码器具体填写），输出长度为0字节，点击保存并重启。', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_6e5b1c4749_20', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4.3', 'section_title': '1.4.3 速度获取', 'heading_path': ['1 跟随应用配置', '1.4 跟随参数配置', '1.4.3 速度获取'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(A-0)》（V5.0 / A-0），配置“速度获取”时，“路径”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['路径：跟随->普通随动->其他参数;'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'line_start': 139, 'line_end': 139, 'excerpt': '路径：跟随->普通随动->其他参数;', 'sha256': '4dabe576a1adb057271570ff82a189d0681d7f3b74cbacb8e5669c98a67bf200'}]}, {'id': 'sample_tracking_1fd2f74e84_01', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.1.2', 'section_title': '4.1.2 实时显示', 'heading_path': ['4 其他功能', '4.1 跟随状态监控', '4.1.2 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“实时显示”中，应如何理解或操作“传送速度”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 传送速度：传送带实时速度，在仿真模式下，传送带速度为仿真速度'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 539, 'line_end': 539, 'excerpt': '1. **传送速度：**传送带实时速度，在仿真模式下，传送带速度为仿真速度；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_02', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.1', 'section_title': '3.1 概要', 'heading_path': ['3 跟随控制指令', '3.1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概要”中，应如何理解或操作“清空传送带队列中的工件信息”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['清空传送带队列中的工件信息，可用在初次执行作业时的初始化过程中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 501, 'line_end': 501, 'excerpt': '清空传送带队列中的工件信息，可用在初次执行作业时的初始化过程中。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_03', 'category': 'tracking', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.6.3', 'section_title': '2.6.3 工件高度校准', 'heading_path': ['2 工作流程', '2.6 传送带参数设置', '2.6.3 工件高度校准'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），操作“工件高度校准”时，涉及“移动机器人”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['2. 移动机器人，将机器人TCP点放置到工件上方，点击【示教高度】按钮，得到工件上表面在用户坐标系下的坐标Z值', '示教时注意切换机器人工具系和对应的传送带标定的用户系'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 416, 'line_end': 416, 'excerpt': '2. 移动机器人，将机器人TCP点放置到工件上方，点击【示教高度】按钮，得到工件上表面在用户坐标系下的坐标Z值；示教时注意切换机器人工具系和对应的传送带标定的用户系；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_04', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.3.1', 'section_title': '5.3.1 光电检测单抓作业', 'heading_path': ['5 编程', '5.3 程序示例', '5.3.1 光电检测单抓作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“光电检测单抓作业”中，应如何理解或操作“机器人在原位点等待工件→检测工件是否到达可拾取区域→开始跟随”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['机器人在原位点等待工件→检测工件是否到达可拾取区域→开始跟随，运动到工件上方→跟随运动到抓取工件位置→控制IO夹紧工件，并保持跟随运动→延时等待抓取成功，并保持跟随运动→运动到工件上方→结束跟随→运动到放置工件位置，放下工件', '表5‑1为上述过程的作业实现'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 602, 'line_end': 602, 'excerpt': '机器人在原位点等待工件→检测工件是否到达可拾取区域→开始跟随，运动到工件上方→跟随运动到抓取工件位置→控制IO夹紧工件，并保持跟随运动→延时等待抓取成功，并保持跟随运动→运动到工件上方→结束跟随→运动到放置工件位置，放下工件。表5‑1为上述过程的作业实现。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_05', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.3.2', 'section_title': '2.3.2 配置总线属性', 'heading_path': ['2 工作流程', '2.3 DeviceNet通讯功能配置', '2.3.2 配置总线属性'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“配置总线属性”中“在端口1中进行配置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['在端口1中进行配置，【can类型】选择RspCan，【波特率】设置与编码器一致，【数据帧格式】选择1，【开关】点击为ON，完成总线属性配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 291, 'line_end': 291, 'excerpt': '在端口1中进行配置，【can类型】选择RspCan，【波特率】设置与编码器一致，【数据帧格式】选择1，【开关】点击为ON，完成总线属性配置。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_06', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.2', 'section_title': '1.2 功能概述', 'heading_path': ['1 跟随应用简介', '1.2 功能概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），配置“功能概述”时，“通过示教方式设置在跟随过程中的机器人的工作上界、拾取下界、工作下界参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['通过示教方式设置在跟随过程中的机器人的工作上界、拾取下界、工作下界参数，确定跟随工作范围'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 231, 'line_end': 231, 'excerpt': '通过示教方式设置在跟随过程中的机器人的工作上界、拾取下界、工作下界参数，确定跟随工作范围。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_07', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6', 'section_title': '6 故障排除', 'heading_path': ['6 故障排除'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“故障排除”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['CNVREF'], 'expected_facts': ['报警码 报警信息 报警子码 报警 等级 报警原因 处理方法 31001 跟随应用初始化成功 无 提示 开机初始化成功 无需处理 31002 跟随应用生成配置文件成功 无 提示 首次开始跟随应用，初始化未默认设置 无需处理 31050 传送带编号无效 处理方法:检查传送带编号是否在1-16之间 无 警告 指令执行过程中，指令参数的传送带编号不在范围 检查传送带编号是否在1-16之间 31051 工件编号无效 处理方法:检查工件编号是否在1-64之间 无 警告 指令执行过程中，指令参数的工件编号不在范围 检查工件编号是否在1-64之间 31052 传送带不在使能状态 处理方法:检查传送带是否已启用 无 警告 指令执行过程中，检测到传送带未使能 检查传送带是否已启用 31053 设置整型变量失败 处理方法:检查整型变量编号是否超出范围 无 警告 执行获取编码器编码值指令时，设置整型变量失败 检查整型变量编号是否超出范围 31054 无可跟随工件 处理方法:检查是否有工件在可拾取区域 无 警告 执行CNVREF指令时，检测到当前传送带队列中没有可以跟随的工件，可能由于其他报警导致 查看与之相关报警信息，一般由31055、31056、31057报警导致，解决相关报警可解决此问题 31055 机器人位置超出工作范围 处理方法:移动机器人到工作区域 无 警告 机器人末端位置超出在传送带参数设置的工作下限 移动机器人到工作区域，调整工作下限范围', '31056 传送带速度超速预警 处理方法:调整传送带速度或者调整报警阈值 无 警告 传送带速度大于传送带参数设置预警基准速度的120％ 调整传送带速度或者调传送带参数设置预警基准速度 31057 传送带速度过慢预警 处理方法:调整传送带速度或者调整报警阈值 无 警告 传送带速度小于传送带参数设置预警基准速度的80％ 调整传送带速度或者调传送带参数设置预警基准速度 31058 未检测到工件经过传感器 处理方法:确认是否按照要求放置工件 无 警告 在进行光电传感器标定时，没有检测到工件经过传感器 确认是否按照要求放置工件 光电传感器是否故障 31059 跟随指令CNVREF未成对出现 处理方法:检查指令是否成对出现 无 警告 在执行作业时，检测到CNVREF指令没有成对出现 修改作业，保证CNVREF 指令成对出现'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 651, 'line_end': 651, 'excerpt': '<table><tr><td><p><strong>报警码</strong></p></td><td><p><strong>报警信息</strong></p></td><td><p><strong>报警子码</strong></p></td><td><p><strong>报警</strong></p><p><strong>等级</strong></p></td><td><p><strong>报警原因</strong></p></td><td><p><strong>处理方法</strong></p></td></tr><tr><td><p>31001</p></td><td><p>跟随应用初始化成功</p></td><td><p>无</p></td><td><p>提示</p></td><td><p>开机初始化成功</p></td><td><p>无需处理</p></td></tr><tr><td><p>31002</p></td><td><p>跟随应用生成配置文件成功</p></td><td><p>无</p></td><td><p>提示</p></td><td><p>首次开始跟随应用，初始化未默认设置</p></td><td><p>无需处理</p></td></tr><tr><td><p>31050</p></td><td><p>传送带编号无效 处理方法:检查传送带编号是否在1-16之间</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>指令执行过程中，指令参数的传送带编号不在范围</p></td><td><p>检查传送带编号是否在1-16之间</p></td></tr><tr><td><p>31051</p></td><td><p>工件编号无效 处理方法:检查工件编号是否在1-64之间</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>指令执行过程中，指令参数的工件编号不在范围</p></td><td><p>检查工件编号是否在1-64之间</p></td></tr><tr><td><p>31052</p></td><td><p>传送带不在使能状态 处理方法:检查传送带是否已启用</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>指令执行过程中，检测到传送带未使能</p></td><td><p>检查传送带是否已启用</p></td></tr><tr><td><p>31053</p></td><td><p>设置整型变量失败 处理方法:检查整型变量编号是否超出范围</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>执行获取编码器编码值指令时，设置整型变量失败</p></td><td><p>检查整型变量编号是否超出范围</p></td></tr><tr><td><p>31054</p></td><td><p>无可跟随工件 处理方法:检查是否有工件在可拾取区域</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>执行CNVREF指令时，检测到当前传送带队列中没有可以跟随的工件，可能由于其他报警导致</p></td><td><p>查看与之相关报警信息，一般由31055、31056、31057报警导致，解决相关报警可解决此问题</p></td></tr><tr><td><p>31055</p></td><td><p>机器人位置超出工作范围 处理方法:移动机器人到工作区域</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>机器人末端位置超出在传送带参数设置的工作下限</p></td><td><p>移动机器人到工作区域，调整工作下限范围；</p></td></tr><tr><td><p>31056</p></td><td><p>传送带速度超速预警 处理方法:调整传送带速度或者调整报警阈值</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>传送带速度大于传送带参数设置预警基准速度的120％</p></td><td><p>调整传送带速度或者调传送带参数设置预警基准速度</p></td></tr><tr><td><p>31057</p></td><td><p>传送带速度过慢预警 处理方法:调整传送带速度或者调整报警阈值</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>传送带速度小于传送带参数设置预警基准速度的80％</p></td><td><p>调整传送带速度或者调传送带参数设置预警基准速度</p></td></tr><tr><td><p>31058</p></td><td><p>未检测到工件经过传感器 处理方法:确认是否按照要求放置工件</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>在进行光电传感器标定时，没有检测到工件经过传感器</p></td><td><p>确认是否按照要求放置工件</p><p>光电传感器是否故障</p></td></tr><tr><td><p>31059</p></td><td><p>跟随指令CNVREF未成对出现 处理方法:检查指令是否成对出现</p></td><td><p>无</p></td><td><p>警告</p></td><td><p>在执行作业时，检测到CNVREF指令没有成对出现</p></td><td><p>修改作业，保证CNVREF 指令成对出现</p></td></tr></table>', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_08', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.1', 'section_title': '1.1 概述', 'heading_path': ['1 跟随应用简介', '1.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概述”中，应如何理解或操作“跟随应用包软件（FLP V1.0.0）是基于RC5.0平台开发的工业机器人工艺包”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['跟随应用包软件（FLP V1.0.0）是基于RC5.0平台开发的工业机器人工艺包，用于实现机器人在运动传送带上对工件的动态跟踪与精准拾取', '解决传统固定坐标系下机器人无法处理连续运动工件的难题，消除传送带启停导致的产能损失，提升生产线节拍'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 183, 'line_end': 183, 'excerpt': '跟随应用包软件（FLP V1.0.0）是基于RC5.0平台开发的工业机器人工艺包，用于实现机器人在运动传送带上对工件的动态跟踪与精准拾取。解决传统固定坐标系下机器人无法处理连续运动工件的难题，消除传送带启停导致的产能损失，提升生产线节拍。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_09', 'category': 'tracking', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.3.4', 'section_title': '5.3.4 3D视觉检测作业编写', 'heading_path': ['5 编程', '5.3 程序示例', '5.3.4 3D视觉检测作业编写'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），操作“3D视觉检测作业编写”时，涉及“3D视觉编写逻辑与2D视觉编写逻辑类似”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['3D视觉编写逻辑与2D视觉编写逻辑类似，仅需注意一下几点：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 644, 'line_end': 644, 'excerpt': '3D视觉编写逻辑与2D视觉编写逻辑类似，仅需注意一下几点：', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_10', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.6.1', 'section_title': '2.6.1 传送带基本参数设置', 'heading_path': ['2 工作流程', '2.6 传送带参数设置', '2.6.1 传送带基本参数设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“传送带基本参数设置”中，应如何理解或操作“预警基准速度”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 预警基准速度：在跟随过程中，传送带实际速度与此处设置的预警基准速度差值大于20%时，机器人停止跟随，并报警，保证人身和设备的安全'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 377, 'line_end': 377, 'excerpt': '4. **预警基准速度：**在跟随过程中，传送带实际速度与此处设置的预警基准速度差值大于20%时，机器人停止跟随，并报警，保证人身和设备的安全；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_11', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.1', 'section_title': '3.1 概要', 'heading_path': ['3 跟随控制指令', '3.1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“概要”中“入队指令CNVADD、CNVADDOBJ”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['CNVADD', 'CNVADDOBJ'], 'expected_facts': ['1. 入队指令CNVADD、CNVADDOBJ'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 487, 'line_end': 487, 'excerpt': '1. 入队指令CNVADD、CNVADDOBJ', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_12', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.2', 'section_title': '1.2 功能概述', 'heading_path': ['1 跟随应用简介', '1.2 功能概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“功能概述”中“包含跟随的工件入队指令、工件出队指令、队列清空指令、获取跟随工件指令、跟随坐标系切换指令、编码器”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['包含跟随的工件入队指令、工件出队指令、队列清空指令、获取跟随工件指令、跟随坐标系切换指令、编码器值获取指令', '跟随指令在作业中合理运用，可实现复杂跟随场景的跟随功能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 239, 'line_end': 239, 'excerpt': '包含跟随的工件入队指令、工件出队指令、队列清空指令、获取跟随工件指令、跟随坐标系切换指令、编码器值获取指令。跟随指令在作业中合理运用，可实现复杂跟随场景的跟随功能。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_13', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.2', 'section_title': '5.2 程序说明', 'heading_path': ['5 编程', '5.2 程序说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“程序说明”中“对机器人内部变量（包括位置变量）赋值”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['5. 对机器人内部变量（包括位置变量）赋值，计算，运算的指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 585, 'line_end': 585, 'excerpt': '5. 对机器人内部变量（包括位置变量）赋值，计算，运算的指令。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_14', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.6.4', 'section_title': '2.6.4 边界参数校准', 'heading_path': ['2 工作流程', '2.6 传送带参数设置', '2.6.4 边界参数校准'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），配置“边界参数校准”时，“设置停止距离”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['4. 设置停止距离，在【停止距离】输入框输入停止距离数值，机器人在位置超出（工作下界-停止距离）位置时，机器人开始减速停止，并报警机器人超出范围'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 446, 'line_end': 446, 'excerpt': '4. 设置停止距离，在【停止距离】输入框输入停止距离数值，机器人在位置超出（工作下界-停止距离）位置时，机器人开始减速停止，并报警机器人超出范围；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_15', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.1.2', 'section_title': '4.1.2 实时显示', 'heading_path': ['4 其他功能', '4.1 跟随状态监控', '4.1.2 实时显示'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“实时显示”中，应如何理解或操作“前端工件位置”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 前端工件位置：传送带队列中靠近工作下限的最前端工件位置坐标(用户坐标系下X、Y、Rz分量)'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 542, 'line_end': 542, 'excerpt': '4. **前端工件位置：**传送带队列中靠近工作下限的最前端工件位置坐标(用户坐标系下X、Y、Rz分量)；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_16', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.1', 'section_title': '3.1 概要', 'heading_path': ['3 跟随控制指令', '3.1 概要'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概要”中，应如何理解或操作“获取当前时刻的编码器的数值”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['获取当前时刻的编码器的数值', '主要用在视觉检测跟随场景中'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 505, 'line_end': 505, 'excerpt': '获取当前时刻的编码器的数值。主要用在视觉检测跟随场景中。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_17', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.1', 'section_title': '5.1 概述', 'heading_path': ['5 编程', '5.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概述”中，应如何理解或操作“进行点焊、喷漆、弧焊或打磨等操作”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 进行点焊、喷漆、弧焊或打磨等操作'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 551, 'line_end': 551, 'excerpt': '2. 进行点焊、喷漆、弧焊或打磨等操作；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_18', 'category': 'tracking', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.6.1', 'section_title': '2.6.1 传送带基本参数设置', 'heading_path': ['2 工作流程', '2.6 传送带参数设置', '2.6.1 传送带基本参数设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），“传送带基本参数设置”中“传送带使能”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OFF'], 'expected_facts': ['3. 传送带使能：点击【使能】按钮，使其变为ON', 'OFF状态下传送带被禁用，使用了传送带编号的相关指令将不会生效'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 376, 'line_end': 376, 'excerpt': '3. **传送带使能：**点击【使能】按钮，使其变为ON。OFF状态下传送带被禁用，使用了传送带编号的相关指令将不会生效；', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_19', 'category': 'tracking', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.2', 'section_title': '1.2 功能概述', 'heading_path': ['1 跟随应用简介', '1.2 功能概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），配置“功能概述”时，“检测参数校准仅在传送带的检测方式为光电检测时有效”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['检测参数校准仅在传送带的检测方式为光电检测时有效，用于设置光电传感器连接的DI端口编号、检测到工件的信号类型（上升沿或下降沿）、工件类型编号、重复剔除距离，校准工件在经过检测位置时的工件坐标系位置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 235, 'line_end': 235, 'excerpt': '检测参数校准仅在传送带的检测方式为光电检测时有效，用于设置光电传感器连接的DI端口编号、检测到工件的信号类型（上升沿或下降沿）、工件类型编号、重复剔除距离，校准工件在经过检测位置时的工件坐标系位置。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}, {'id': 'sample_tracking_1fd2f74e84_20', 'category': 'tracking', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1.1', 'section_title': '1.1 概述', 'heading_path': ['1 跟随应用简介', '1.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概述”中，应如何理解或操作“跟随应用支持直线传送带和圆形传送带”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['跟随应用支持直线传送带和圆形传送带，不同类型的传送带具有不同的校准和编程方法', '系统一共可以定义16条实体传送带', '每条实体传送带有一个编码器，其信号由编码器板接收'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'line_start': 185, 'line_end': 185, 'excerpt': '跟随应用支持直线传送带和圆形传送带，不同类型的传送带具有不同的校准和编程方法。系统一共可以定义16条实体传送带。每条实体传送带有一个编码器，其信号由编码器板接收。支持多条传送带和多机器人传送带，支持视觉和光电传感器检测工件位置。', 'sha256': '4e24952564aa582702dddf556bec55c89cf6f516523b394e6dfdae5fd91eb278'}]}]
register_sample_bank('tracking', TRACKING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md', 'seed': 4483030779643802378, 'population': 102, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(604, 610), (646, 652), (361, 365), (221, 221), (192, 192), (235, 235), (385, 389), (341, 344), (546, 549), (542, 545), (486, 489), (618, 624), (328, 332), (211, 211), (132, 132), (222, 222), (284, 287), (374, 377), (579, 582), (639, 645)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md', 'seed': 15371735730882957866, 'population': 47, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(238, 238), (234, 234), (103, 103), (230, 230), (178, 178), (154, 154), (196, 196), (228, 228), (63, 63), (165, 165), (166, 166), (192, 192), (146, 146), (150, 150), (202, 202), (73, 73), (145, 145), (184, 184), (81, 81), (139, 139)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md', 'seed': 9516353047134255705, 'population': 149, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(539, 539), (501, 501), (416, 416), (602, 602), (291, 291), (231, 231), (651, 651), (183, 183), (644, 644), (377, 377), (487, 487), (239, 239), (585, 585), (446, 446), (542, 542), (505, 505), (551, 551), (376, 376), (235, 235), (185, 185)]}])


In [ ]:
# | notest
TRACKING_SAMPLE_IDS = None
tracking_sample_cases = select_cases(TRACKING_SAMPLE_CASES, TRACKING_SAMPLE_IDS)
display(pd.DataFrame(tracking_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in tracking_sample_cases])
tracking_sample_metrics = await evaluation.run(tracking_sample_cases, concurrency=EVALUATION_CONCURRENCY)
tracking_sample_results = {case["id"]: evaluation.load(case) for case in tracking_sample_cases}
display(tracking_sample_metrics)
# Inspect a complete result: evaluation.inspect(tracking_sample_cases[0]["id"])
evaluation.show_answers(tracking_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_tracking_68d3839791_01,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,4,False,False,False,None,None,22403,2,True,unsupported,,,not_assessable
sample_tracking_68d3839791_02,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,alarm,table_row,False,True,8,False,0,0,3,False,False,False,False,False,8024,1,True,unsupported,,,correct
sample_tracking_68d3839791_03,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,parameter,table_row,False,True,8,False,0,0,0,False,False,False,None,None,10366,0,True,included,,,incorrect
sample_tracking_68d3839791_04,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,parameter,prose,False,True,8,False,0,0,0,False,False,False,None,None,11638,0,True,unsupported,,,not reviewed
sample_tracking_68d3839791_05,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,procedure,prose,False,True,8,False,0,0,0,False,False,False,None,None,8512,0,True,unsupported,,,not reviewed
sample_tracking_68d3839791_06,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,instruction,prose,False,True,8,False,0,0,0,False,False,False,None,None,8208,0,True,unsupported,,,not reviewed
sample_tracking_68d3839791_07,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,6564,0,True,included,,,not reviewed
sample_tracking_68d3839791_08,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,0,False,False,False,None,None,8537,0,True,included,,,not reviewed
sample_tracking_68d3839791_09,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,procedure,table_row,False,True,8,False,0,0,2,False,False,False,None,None,22428,0,True,included,,,not reviewed
sample_tracking_68d3839791_10,tracking,assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md,V4.0 / A-0,instruction,table_row,False,True,8,False,0,0,0,False,False,False,False,False,8695,0,True,included,,,not reviewed


### sample_tracking_68d3839791_01

根据《SX322017《新松机器人跟随应用操作手册》-A-0》（V4.0 / A-0），在“报警说明”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 新松机器人跟随应用操作手册（SX322017）报警表格定义行：列依次为报警码、报警等级、报警信息、报警原因、处理措施；无跨行/跨列合并，各单元格独立且未继承父级值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 604–610; corpus 4.0, revision V5.0 / A-0; object `73e63b2225c640d2b608ce544fc7c26d548d74458386b8b58a25e597fcfb4f66`.

- SX322001《新松机器人通用操作手册》
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 242–244; corpus 5.0, revision V5.0 / C-1; object `01657a2d25e2d96b2002524cd48a57af91652580e6851057699c74e1912af5e7`.

- 该行包含四个空白单元格（第1至4列），无文本内容、无合并继承值，为结构占位行，源自《新松机器人跟随应用操作手册》(SX322017, V5.0)。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 46–51; corpus 5.0, revision V5.0 / A-0; object `3c6d6a9bf0aacfdaf01d8a9de6fa15c054d7a6520960f65fff6fed26bf887d18`.

- 该行为《新松机器人跟随应用操作手册》表 1-1（各作业者可执行作业一览）的第 9 行数据。

**摘要：**
作业项目“以操作盘复位报警”在表中的第 3 列和第 4 列对应角色/权限均标记为“○”（可执行/适用），第 2 列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 144–149; corpus 5.0, revision V5.0 / C-1; object `d4be805cceb09ffd9ae96764d2bde131f41266daf0c2f5798dc221f728e9b9f6`.

- 报警码7020（新松机器人点焊应用），其余字段（报警信息、子码、子码说明、处理方法）为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 2048–2054; corpus 5.0, revision V5.0 / A-2; object `ee7974777b60cf935e7b523241ea75d4ff63a94fd43143304fa2d26da080abbc`.

- 新松机器人点焊系统（SX322007手册）中报警码 **7026** 的条目，其报警信息、子码、子码说明及处理方法均为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 2090–2096; corpus 5.0, revision V5.0 / A-2; object `dd1d5fa3c365e1856579f602d552257291b7a62297bf99e3cdd05b33900214d2`.

- 新松机器人报警码1302为“指令异常保护:SV”，系松下驱动器报警（[参数]指对应轴号）；若驱动器报警代码27.0反复出现，需联系新松售后人员。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 13255–13260; corpus 5.0, revision V5.0 / C-1; object `6febb0669bbd6186de5ad0d49da2099b4d125437c0c57a8957a3e4bacac5eb56`.

- 新松机器人报警码 **1357** 表示**可编程器件固件匹配错误 (SV)**，通常由**驱动器损坏**引起；解决措施为**维修或更换驱动器**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1)/SX322001《新松机器人通用操作手册》(C-1).md`, lines 13573–13578; corpus 5.0, revision V5.0 / C-1; object `6c55d16fad3e47dd657b86a175dda049d68d5fd099cf5d02581a7f88b1894a1e`.

#### Answer

根据E1证据，SX322017《新松机器人跟随应用操作手册》(A-0)中

#### Model-assisted source review

Assessment: **not_assessable**; completeness: none; citation support: not_applicable; revision fidelity: mixed.

The generated answer is a fragment that does not answer the question. It merely cites evidence E1 and does not explain the operation, purpose, or result of the alarm code as requested. The expected facts (alarm code, level, info, cause, measure) are present in the supplied evidence E1, so the answer should have summarized them. The answer does not provide a complete or correct response.

...

#### Answer

在所提供的全部检索证据（E1–E8）中，均未出现

### sample_tracking_1fd2f74e84_20

根据《SX322017《新松机器人跟随应用操作手册》(C-1)》（V5.0 / C-1），在“概述”中，应如何理解或操作“跟随应用支持直线传送带和圆形传送带”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行内容为新松机器人跟随应用操作手册。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 11–13; corpus 5.0, revision V5.0 / A-0; object `a5adddd291db3fe81ed5d0bb18bed8f24bb3215ea0a1d1dd62cc68a8501c1651`.

- 新松机器人跟随应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(C-1)/SX322017《新松机器人跟随应用操作手册》(C-1).md`, lines 5–7; corpus 5.0, revision V5.0 / C-1; object `884b4ee09e1af5af4239b29fe745b617fd77b4006f859e412573c4fd734e28c2`.

- 新松机器人跟随应用操作手册（SX322017）报警表格定义行：列依次为报警码、报警等级、报警信息、报警原因、处理措施；无跨行/跨列合并，各单元格独立且未继承父级值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 604–610; corpus 4.0, revision V5.0 / A-0; object `73e63b2225c640d2b608ce544fc7c26d548d74458386b8b58a25e597fcfb4f66`.

- 该行包含四个空白单元格（第1至4列），无文本内容、无合并继承值，为结构占位行，源自《新松机器人跟随应用操作手册》(SX322017, V5.0)。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 46–51; corpus 5.0, revision V5.0 / A-0; object `3c6d6a9bf0aacfdaf01d8a9de6fa15c054d7a6520960f65fff6fed26bf887d18`.

- 该表格行（索引5，非表头）包含4个单元格，所有单元格内容均为空，无跨行/跨列合并，无继承值。此行为《新松机器人跟随应用操作手册》文档中的一个空白数据行。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 52–57; corpus 5.0, revision V5.0 / A-0; object `cf856d244fe13b31b1c8631b938836c78ee44b49f1eecd3b6174104b7a221c9a`.

- 该单元格包含文本“新松机器人跟随应用操作手册”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 11–13; corpus 4.0, revision V5.0 / A-0; object `d7257dc54b3f416d58a56c7c37b561cdd7c99b16922341b68b36c513a15d1efb`.

- 该行是《新松机器人跟随应用操作手册》(A-0) 中表格的第 6 行，包含 4 个空单元格，无可见数据内容。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 58–63; corpus 5.0, revision V5.0 / A-0; object `b30415460f1a017446fbe4979c710ccb0900136cae84802d32d963f3aad3304c`.

- 该行位于《新松机器人跟随应用操作手册》表格第6行，第一列内容为“RESUME”，第二列为空；无表头或合并单元格继承值。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322017《新松机器人跟随应用操作手册》(A-0)/SX322017《新松机器人跟随应用操作手册》(A-0).md`, lines 587–590; corpus 4.0, revision V5.0 / A-0; object `e33d7f276c384fb9cbb6d34fadedf718d8297193b55f2c72f6f8f541e546121d`.

#### Answer

The question asks about the '概述' (Overview) section of the C-1 revision manual, specifically the statement '跟随应用支持直线传送带和圆形传送带' (tracking application supports linear conveyor and circular conveyor). However, the retrieved evidence does not contain any text from the '概述' section. E2 only shows the title page of the C-1 manual. E3 shows alarm codes (7600, 7602, 7603, 7604, 7613, 7615, 7616) related to FL_MOVL/GR_MOVL instructions and conveyor/encoder configuration, but these are from the A-0 revision and from an alarm/troubleshooting section, not the '概述'. E8 shows a code snippet with MOVJ, PAUSE, DELAY, RESUME, RET, END but does not reference linear vs. circular conveyor modes. No evidence snippet contains the phrase '直线传送带和圆形传送带' or any operational instructions for selecting between linear and circular conveyor tracking modes. The specific '概述' section content describing how to configure or operate the two conveyor types is absent from all retrieved evidence.

#### Model-assisted source review

Assessment: **abstained**; completeness: partial; citation support: supported; revision fidelity: mixed.

The answer correctly identifies that the supplied evidence does not contain the specific '概述' (Overview) section text required to explain the operation of linear vs. circular conveyers. The evidence provided (E1-E8) consists of title pages, revision history, alarm codes, and code snippets, none of which contain the phrase '直线传送带和圆形传送带' or the operational details requested. Therefore, the refusal to answer the specific operational question is appropriate given the lack of relevant evidence in the retrieval context.

## Arc Welding — sampled manual revisions

40 cases from 2 CSV files. Set `ARC_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
ARC_WELDING_SAMPLE_CASES = [{'id': 'sample_arc_welding_d81b2686c9_01', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.3', 'section_title': '1.3 起始点检测', 'heading_path': ['1 智能焊接—电弧篇', '1.3 起始点检测'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），“起始点检测”中“同时”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['P102', 'F#02'], 'expected_facts': ['同时，该偏移量被存储到位置变量P102（F#02 直角坐标系下）、P112（F#06 变位机坐标系下）中，注意该过程只是保存偏移量，使用时不需要使用shift指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 2584, 'line_end': 2584, 'excerpt': '同时，该偏移量被存储到位置变量P102（F#02 直角坐标系下）、P112（F#06 变位机坐标系下）中，注意该过程只是保存偏移量，使用时不需要使用shift指令。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_02', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.4.3.1', 'section_title': '1.4.3.1 起弧文件', 'heading_path': ['1 弧焊标准机器人配置', '1.4 焊机参数配置', '1.4.3 焊接参数设置', '1.4.3.1 起弧文件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“起弧文件”时，涉及“焊接条件（漏弧检测）”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['5. 焊接条件（漏弧检测）：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 1237, 'line_end': 1237, 'excerpt': '5. 焊接条件（漏弧检测）：', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_03', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '3.2.3', 'section_title': '3.2.3 点激光传感界面', 'heading_path': ['3 智能焊接-点激光篇', '3.2 点激光传感', '3.2.3 点激光传感界面'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“点激光传感界面”时，“寻位方式”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['8. 寻位方式：0未设置，1是模拟量扫描，2是IO量传感', '若设为0则该动作跳过不执行，非0正常执行，直到完成所有动作', '初始值为0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 4022, 'line_end': 4022, 'excerpt': '8. 寻位方式：0未设置，1是模拟量扫描，2是IO量传感。若设为0则该动作跳过不执行，非0正常执行，直到完成所有动作。初始值为0。范围：0/1/2；', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_04', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.3.1.3', 'section_title': '4.3.1.3 覆盖道参数设置', 'heading_path': ['4 多道焊', '4.3 多层多道焊接数据库', '4.3.1 文件配置', '4.3.1.3 覆盖道参数设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“覆盖道参数设置”时，“覆盖道”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['覆盖道：设置覆盖道号可以直接切换到相应的覆盖道参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 4509, 'line_end': 4509, 'excerpt': '覆盖道：设置覆盖道号可以直接切换到相应的覆盖道参数。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_05', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.22.2', 'section_title': '1.22.2 注意事项', 'heading_path': ['1 辅助焊接功能', '1.22 数据映射', '1.22.2 注意事项'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），在“注意事项”中，应如何理解或操作“映射到整型或实型变量号不要重复”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 映射到整型或实型变量号不要重复，会导致丢数据'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 2410, 'line_end': 2410, 'excerpt': '2. 映射到整型或实型变量号不要重复，会导致丢数据。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_06', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.3', 'section_title': '1.3 起始点检测', 'heading_path': ['1 智能焊接—电弧篇', '1.3 起始点检测'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“起始点检测”时，“当焊缝在一定范围内发生偏移时可以通过起始点检测功能自动重新锁定正确焊缝起始点”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['当焊缝在一定范围内发生偏移时可以通过起始点检测功能自动重新锁定正确焊缝起始点', '二维起始点检测允许工件在OZ，OX方向平移后准确找出焊缝起点'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 2580, 'line_end': 2580, 'excerpt': '当焊缝在一定范围内发生偏移时可以通过起始点检测功能自动重新锁定正确焊缝起始点。二维起始点检测允许工件在OZ，OX方向平移后准确找出焊缝起点。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_07', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '3.2.5.2', 'section_title': '3.2.5.2 点激光模拟量扫描', 'heading_path': ['3 智能焊接-点激光篇', '3.2 点激光传感', '3.2.5 应用实例', '3.2.5.2 点激光模拟量扫描'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），“点激光模拟量扫描”中“行号 指令 含义 1 NOP 作业开始 2 WREFP TYPE =1 基准面指令（1-4） 3”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['NOP', 'WREFP'], 'expected_facts': ['行号 指令 含义 1 NOP 作业开始 2 WREFP TYPE =1 基准面指令（1-4） 3 ARCL VL = 20（ARCLP P150 = 20） 安全点（可将姿态变量设置为安全点，便于对照） 4 SET P501 P1 将起点和终点赋值到其他变量，两个作用：（1）传感后不改变示教点位', '（2）PLSS文件无需重复设置，相同焊缝类型可重复使用', '5 SET P502 P2 6 PLSS SEARCH #10 点激光传感指令 7 ARCLP P501=20 检测后起点位置 8 ARCON #1 #1 起弧指令 9 ARCLP P502=6 检测后的终点位置 10 ARCOFF #1 熄弧指令 10 ARCL VL = 20 安全点 11 END 作业结束'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 4131, 'line_end': 4131, 'excerpt': '<table><tr><td><p>行号</p></td><td><p>指令</p></td><td><p>含义</p></td></tr><tr><td><p>1</p></td><td><p>NOP</p></td><td><p>作业开始</p></td></tr><tr><td><p>2</p></td><td><p>WREFP TYPE =1</p></td><td><p>基准面指令（1-4）</p></td></tr><tr><td><p>3</p></td><td><p>ARCL VL = 20（ARCLP P150 = 20）</p></td><td><p>安全点（可将姿态变量设置为安全点，便于对照）</p></td></tr><tr><td><p>4</p></td><td><p>SET P501 P1</p></td><td rowspan="2"><p>将起点和终点赋值到其他变量，两个作用：（1）传感后不改变示教点位；（2）PLSS文件无需重复设置，相同焊缝类型可重复使用。</p></td></tr><tr><td><p>5</p></td><td><p>SET P502 P2</p></td></tr><tr><td><p>6</p></td><td><p>PLSS SEARCH #10</p></td><td><p>点激光传感指令</p></td></tr><tr><td><p>7</p></td><td><p>ARCLP P501=20</p></td><td><p>检测后起点位置</p></td></tr><tr><td><p>8</p></td><td><p>ARCON #1 #1</p></td><td><p>起弧指令</p></td></tr><tr><td><p>9</p></td><td><p>ARCLP P502=6</p></td><td><p>检测后的终点位置</p></td></tr><tr><td><p>10</p></td><td><p>ARCOFF #1</p></td><td><p>熄弧指令</p></td></tr><tr><td><p>10</p></td><td><p>ARCL VL = 20</p></td><td><p>安全点</p></td></tr><tr><td><p>11</p></td><td><p>END</p></td><td><p>作业结束</p></td></tr></table>', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_08', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.17', 'section_title': '4.17 报警信息', 'heading_path': ['6 报警信息', '4.17 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“报警信息”时，涉及“报警码”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['报警码 报警信息 子码 子码信息 解决方法 6490 数据存储CF卡：校验失败，数据已恢复默认值，请注意！ - - 系统错误，请联系技术人员', '6491 机器人焊枪发生碰撞，4、5、6松抱闸', '- - 请移动机器人到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 5898, 'line_end': 5898, 'excerpt': '<table><tr><td><p>报警码</p></td><td><p>报警信息</p></td><td><p>子码</p></td><td><p>子码信息</p></td><td><p>解决方法</p></td></tr><tr><td><p>6490</p></td><td><p>数据存储CF卡：校验失败，数据已恢复默认值，请注意！</p></td><td><p>-</p></td><td><p>-</p></td><td><p>系统错误，请联系技术人员。</p></td></tr><tr><td><p>6491</p></td><td><p>机器人焊枪发生碰撞，4、5、6松抱闸。</p></td><td><p>-</p></td><td><p>-</p></td><td><p>请移动机器人到安全位置。</p></td></tr></table>', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_09', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.6', 'section_title': '1.6 接触探测传感', 'heading_path': ['1 智能焊接—电弧篇', '1.6 接触探测传感'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“接触探测传感”时，涉及“接触探测传感功能是检测坡口位置”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['接触探测传感功能是检测坡口位置，适用于坡口位置偏移或同类工件坡口位置不一致的情况，根据需求可以结合其他检测方式进行起点位置的准确定位', '因TDS检测的只有Y方向偏移，所以不能单独检测焊接，需配合其他检测焊接'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 2695, 'line_end': 2695, 'excerpt': '接触探测传感功能是检测坡口位置，适用于坡口位置偏移或同类工件坡口位置不一致的情况，根据需求可以结合其他检测方式进行起点位置的准确定位。因TDS检测的只有Y方向偏移，所以不能单独检测焊接，需配合其他检测焊接。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_10', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.10.2', 'section_title': '4.10.2 多道焊自适应配置', 'heading_path': ['2 自适应功能', '4.10 多道焊坡口自适应', '4.10.2 多道焊自适应配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“多道焊自适应配置”时，“线性调节”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['线性调节：实际焊缝尺寸为1.5→参数：15'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 5008, 'line_end': 5008, 'excerpt': '线性调节：实际焊缝尺寸为1.5→参数：15', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_11', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '3.1.3.2', 'section_title': '3.1.3.2 点激光姿态标定', 'heading_path': ['3 智能焊接-点激光篇', '3.1 点激光配置', '3.1.3 点激光标定', '3.1.3.2 点激光姿态标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“点激光姿态标定”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：在姿态标定之前必须先完成位置标定，且确保当前坐标设定界面中设定值的RX、RY、RZ值为0，姿态标定结果可在坐标设定界面查看和修改'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 3872, 'line_end': 3872, 'excerpt': '注意：在姿态标定之前必须先完成位置标定，且确保当前坐标设定界面中设定值的RX、RY、RZ值为0，姿态标定结果可在坐标设定界面查看和修改。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_12', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.6.3', 'section_title': '1.6.3 使用说明', 'heading_path': ['1 辅助焊接功能', '1.6 定点摆弧', '1.6.3 使用说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“使用说明”时，“根据实际情况设置摆动参数”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 根据实际情况设置摆动参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 1522, 'line_end': 1522, 'excerpt': '1. 根据实际情况设置摆动参数。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_13', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.10.4.3', 'section_title': '4.10.4.3 扫描数据查看', 'heading_path': ['2 自适应功能', '4.10 多道焊坡口自适应', '4.10.4 焊缝尺寸获取', '4.10.4.3 扫描数据查看'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），在“扫描数据查看”中，应如何理解或操作“在弧焊→自适应→坡口扫描数据中可以对各焊接线各运动段的坡口尺寸数据进行查看”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['在弧焊→自适应→坡口扫描数据中可以对各焊接线各运动段的坡口尺寸数据进行查看，确认已扫描数据是否满足自适应功能的需求'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 5245, 'line_end': 5245, 'excerpt': '在弧焊→自适应→坡口扫描数据中可以对各焊接线各运动段的坡口尺寸数据进行查看，确认已扫描数据是否满足自适应功能的需求。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_14', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.14.10', 'section_title': '4.14.10 外部轴手自动功能', 'heading_path': ['4 外部轴', '4.14 外部轴配置', '4.14.10 外部轴手自动功能'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），在“外部轴手自动功能”中，应如何理解或操作“外部轴1”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 外部轴1，通常为第7轴，以此类推是对不同外部轴的设计'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 5835, 'line_end': 5835, 'excerpt': '1. 外部轴1，通常为第7轴，以此类推是对不同外部轴的设计。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_15', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.6.2', 'section_title': '4.6.2 指定焊道跳转', 'heading_path': ['4 多道焊', '4.6 辅助功能', '4.6.2 指定焊道跳转'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“指定焊道跳转”时，涉及“若执行覆盖道跳转为ON时”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['6. 若执行覆盖道跳转为ON时，此时若当前的执行光标P仍在循环区间内，即使示教光标T移动到循环区间外，启动时，也会提示应用禁止启动弹窗，同时有相应的报警信息，如图', '此时需手动运行机器人到循环区间外，使执行光标P在循环区间外，此时方可继续执行覆盖道指定焊道跳转'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 4735, 'line_end': 4735, 'excerpt': '6. 若执行覆盖道跳转为ON时，此时若当前的执行光标P仍在循环区间内，即使示教光标T移动到循环区间外，启动时，也会提示应用禁止启动弹窗，同时有相应的报警信息，如图。此时需手动运行机器人到循环区间外，使执行光标P在循环区间外，此时方可继续执行覆盖道指定焊道跳转。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_16', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.6.3', 'section_title': '4.6.3 多道焊作业示例', 'heading_path': ['4 多道焊', '4.6 辅助功能', '4.6.3 多道焊作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），“多道焊作业示例”中“NOP”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['NOP', 'WREFP'], 'expected_facts': ['作业 说明 NOP WREFP 1 建立坐标系 ARCL P1 VL = 100 ACC = 100 CNT = 0 安全点 MLCC ON 循环区间开始 MLON N=111 数据库调用1 ARCL P2 VL = 100 ACC = 100 CNT = 0 接近点 TRACKON ARC #1 跟踪开始 ARC 代表电弧跟踪,#1代表文件号，该指令必须置于ARCON指令之前 ARCL P3 VL = 50 ACC = 100 CNT = 0 焊缝起点 ARCON #1 #2 起弧/文件#1为起弧文件', '文件#2为焊接文件 AMENDON F#02 M=02 实时补偿指令(F#02基于直角坐标系补偿，M=02 相对纠偏)， 该指令必须置于ARCON指令之后 WVLIB ON #=107 F=1 A=4 TYPE=1 摆动库开开（调取文件107.F=1摆动频率1,A=4摆动幅值=4） ARCL P4 VL = 8 ACC = 100 CNT = 0 焊缝点 TRACKOFF ARC 电弧跟踪关闭 WVLIB OFF 摆动关闭 TRACKOFF TYPE=1 电弧跟踪关闭 AMAND OFF 纠偏关 ARCL P5 VL = 100 ACC = 100 CNT = 0 远离点 MLCC OFF 循环结束 ARCL P6 VL = 100 ACC = 100 CNT = 0 安全点 END'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 4755, 'line_end': 4755, 'excerpt': '<table><tr><td><p>作业</p></td><td><p>说明</p></td></tr><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>WREFP 1</p></td><td><p>建立坐标系</p></td></tr><tr><td><p>ARCL P1 VL = 100 ACC = 100 CNT = 0</p></td><td><p>安全点</p></td></tr><tr><td><p>MLCC ON</p></td><td><p>循环区间开始</p></td></tr><tr><td><p>MLON N=111</p></td><td><p>数据库调用1 </p></td></tr><tr><td><p>ARCL P2 VL = 100 ACC = 100 CNT = 0</p></td><td><p>接近点</p></td></tr><tr><td><p>TRACKON ARC #1</p></td><td><p>跟踪开始 ARC 代表电弧跟踪,#1代表文件号，该指令必须置于ARCON指令之前</p></td></tr><tr><td><p>ARCL P3 VL = 50 ACC = 100 CNT = 0</p></td><td><p>焊缝起点</p></td></tr><tr><td><p>ARCON #1 #2</p></td><td><p>起弧/文件#1为起弧文件；文件#2为焊接文件</p></td></tr><tr><td><p>AMENDON F#02 M=02</p></td><td><p>实时补偿指令(F#02基于直角坐标系补偿，M=02</p><p>相对纠偏)，<strong>该指令必须置于ARCON指令之后</strong></p></td></tr><tr><td><p>WVLIB ON #=107 F=1 A=4 TYPE=1</p></td><td><p>摆动库开开（调取文件107.F=1摆动频率1,A=4摆动幅值=4）</p></td></tr><tr><td><p>ARCL P4 VL = 8 ACC = 100 CNT = 0</p></td><td><p>焊缝点</p></td></tr><tr><td><p>TRACKOFF ARC</p></td><td><p>电弧跟踪关闭</p></td></tr><tr><td><p>WVLIB OFF</p></td><td><p>摆动关闭</p></td></tr><tr><td><p>TRACKOFF TYPE=1</p></td><td><p>电弧跟踪关闭</p></td></tr><tr><td><p>AMAND OFF</p></td><td><p>纠偏关</p></td></tr><tr><td><p>ARCL P5 VL = 100 ACC = 100 CNT = 0</p></td><td><p>远离点</p></td></tr><tr><td><p>MLCC OFF</p></td><td><p>循环结束</p></td></tr><tr><td><p>ARCL P6 VL = 100 ACC = 100 CNT = 0</p></td><td><p>安全点</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_17', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '4.14.8.6', 'section_title': '4.14.8.6 跟踪差设定', 'heading_path': ['4 外部轴', '4.14 外部轴配置', '4.14.8 外部轴参数的设定', '4.14.8.6 跟踪差设定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），配置“跟踪差设定”时，“跟踪差设定”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['跟踪差设定，进入这个界面，把外部轴的跟踪差值设大一些（为了让机器人先动起来）', '需要机器人满负载最大速度自动运行，轴的活动范围尽可能大些，查看实时显示里的外部轴跟踪差值，机器人运行几个循环的过程中，把外部轴的跟踪差最大值记录下来，填写到跟踪差设定界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 5693, 'line_end': 5693, 'excerpt': '跟踪差设定，进入这个界面，把外部轴的跟踪差值设大一些（为了让机器人先动起来）。需要机器人满负载最大速度自动运行，轴的活动范围尽可能大些，查看实时显示里的外部轴跟踪差值，机器人运行几个循环的过程中，把外部轴的跟踪差最大值记录下来，填写到跟踪差设定界面。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_18', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.12.3', 'section_title': '1.12.3 作业示例', 'heading_path': ['1 辅助焊接功能', '1.12 断续焊', '1.12.3 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），“作业示例”中“NOP”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['NOP', 'ARCJ'], 'expected_facts': ['作业 说明 NOP ARCJ P1 VJ = 20 ACC = 100 CNT = 0 某个初始点位 ARCL PR201 VL = 200 ACC =100 CNT = 0 焊接线起点：A点，该点指令位置为IMWON前第一个运动指令（ARCL）', '即：断续焊焊接线起点在IMWON指令执行时机器人TCP所在位置 IMWON TYPE=1 #1 断续焊开始，TYPE=1为类型，断续焊功能设定为1，其他数字无效', '#1 为文件号，共8个，对应上图菜单界面 ARCL PR202 VL = 100 ACC =100 CNT = 0 断续焊功能区间第一条焊接线终点：B点，为IMWON指令后第一个运动指令（功能不支持ARCC）'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 1863, 'line_end': 1863, 'excerpt': '<table><tr><td><p>作业</p></td><td><p>说明</p></td></tr><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>ARCJ P1 VJ = 20 ACC = 100 CNT = 0</p></td><td><p>某个初始点位</p></td></tr><tr><td><p>ARCL PR201 VL = 200 ACC =100 CNT = 0</p></td><td><p>焊接线起点：A点，该点指令位置为IMWON前第一个运动指令（ARCL）。即：断续焊焊接线起点在IMWON指令执行时机器人TCP所在位置</p></td></tr><tr><td><p>IMWON TYPE=1 #1</p></td><td><p>断续焊开始，TYPE=1为类型，断续焊功能设定为1，其他数字无效。#1 为文件号，共8个，对应上图菜单界面 </p></td></tr><tr><td><p>ARCL PR202 VL = 100 ACC =100 CNT = 0</p></td><td><p>断续焊功能区间第一条焊接线终点：B点，为IMWON指令后第一个运动指令（功能不支持ARCC）。断续焊工艺这条指令不执行动作，执行动作要执行下一条CALL指令，利用子作业完成</p></td></tr><tr><td><p>CALL 1</p></td><td><p>断续焊区间，ARCL后必须跟着CALL指令，后面子作业名称为参数界面设定子作业名称，否则只是简单的调用子作业逻辑，与断续焊功能无关</p></td></tr><tr><td><p>IMWOFF</p></td><td><p>断续焊功能结束，使能被关闭</p></td></tr><tr><td><p>END</p></td><td></td></tr><tr><td><p>作业：1</p></td><td><p>说明</p></td></tr><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>SET I2 = 0</p></td><td><p>初始计数器，I2对应为界面“当前段int号”，记录循环次数用</p></td></tr><tr><td><p>L10</p></td><td><p>L10后面指令是循环区间</p></td></tr><tr><td><p>INC I2</p></td><td><p>I2++</p></td></tr><tr><td><p>ARCL P3 VL = 100 ACC =100 CNT = 0</p></td><td><p>P3焊枪起点位置，对应上图，为P1点焊枪上提后的点位</p></td></tr><tr><td><p>ARCL P1 VL = 100 ACC =100 CNT = 0</p></td><td><p>P1为起弧点位置，焊接段的起点</p></td></tr><tr><td><p>ARCON #1 #1</p></td><td><p>起弧</p></td></tr><tr><td><p>ARCL P2 VL = 5 ACC =100 CNT = 0</p></td><td><p>P2为熄弧点位置，速度参数根据工艺要求设定</p></td></tr><tr><td><p>ARCOFF #1</p></td><td><p>熄弧</p></td></tr><tr><td><p>ARCL P4 VL = 100 ACC =100 CNT = 0</p></td><td><p>P4为焊接段终点位置，为P2的焊枪上提后点位</p></td></tr><tr><td><p>IF I2&lt;I1 L10</p></td><td><p>I1对应界面参数“总段数整型int号”，确定要循环的次数，由RC给出</p></td></tr><tr><td><p>ARCL P5 VL = 100 ACC =100 CNT = 0</p></td><td><p>P5为焊接线终点位置，对应界面参数设置。该点实质为B点位置。主作业的ARCL B 没有执行运动，拆解为模板作业执行，最终达到了B点</p></td></tr><tr><td><p>RET</p></td><td></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_19', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '1.5.1.3', 'section_title': '1.5.1.3 [状态IO开关]界面配置', 'heading_path': ['1 辅助焊接功能', '1.5 状态IO监控功能说明与配置', '1.5.1 参数配置说明', '1.5.1.3 [状态IO开关]界面配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“[状态IO开关]界面配置”时，涉及“该界面为动态显示界面”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['该界面为动态显示界面，显示开关参数项由【状态IO配置】界面参数项使能状态决定，只有使能状态为开时，【状态IO开关】界面对应参数开关才可设置', '界面对应关系示例如下图所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 1420, 'line_end': 1420, 'excerpt': '该界面为动态显示界面，显示开关参数项由【状态IO配置】界面参数项使能状态决定，只有使能状态为开时，【状态IO开关】界面对应参数开关才可设置。界面对应关系示例如下图所示：', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_d81b2686c9_20', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / B-5', 'section': '2.3.1.3', 'section_title': '2.3.1.3 作业示例', 'heading_path': ['2 智能焊接—激光篇', '2.3 激光跟踪', '2.3.1 激光跟踪', '2.3.1.3 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'source_version': 'rc4', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），操作“作业示例”时，涉及“免示教跟踪与激光跟踪—有寻位功能作业一样”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['免示教跟踪与激光跟踪—有寻位功能作业一样，免示教跟踪必须开启寻位开关和免示教开关', '标定精度、TCP精度和起点检测枪姿对免示教跟踪过程和结果影响很大'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'line_start': 3526, 'line_end': 3526, 'excerpt': '免示教跟踪与激光跟踪—有寻位功能作业一样，免示教跟踪**必须开启**寻位开关和免示教开关。标定精度、TCP精度和起点检测枪姿对免示教跟踪过程和结果影响很大。', 'sha256': 'de5e1206da2050dc6a2ec92a6b5d45ed1a0ee939320483d10fed822919527a90'}]}, {'id': 'sample_arc_welding_e39a0c6604_01', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.1.4', 'section_title': '6.1.4 作业示例', 'heading_path': ['6 多道焊', '6.1 循环焊接', '6.1.4 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），操作“作业示例”时，涉及“说明”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['说明：上述作业可以根据循环条件（WHILE I[1] < 2）执行多次覆盖道焊接'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2480, 'line_end': 2480, 'excerpt': '**说明：上述作业可以根据循环条件（WHILE I[1] < 2）执行多次覆盖道焊接。**', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_02', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '1', 'section_title': '1 弧焊系统构成', 'heading_path': ['1 弧焊系统构成'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），在“弧焊系统构成”中，应如何理解或操作“机器人弧焊系统基本构成如下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['机器人弧焊系统基本构成如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 373, 'line_end': 373, 'excerpt': '机器人弧焊系统基本构成如下：', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_03', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.9.1', 'section_title': '4.9.1 文件配置', 'heading_path': ['4 智能焊接-电弧篇', '4.9 电弧跟踪', '4.9.1 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），配置“文件配置”时，“用户可以通过【弧焊】-【电弧跟踪】”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['用户可以通过【弧焊】-【电弧跟踪】，进行常用参数的配置，并根据需求设置合理的参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2112, 'line_end': 2112, 'excerpt': '用户可以通过【弧焊】-【电弧跟踪】，进行常用参数的配置，并根据需求设置合理的参数。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_04', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.2.5', 'section_title': '2.2.5 滑台标定', 'heading_path': ['2 弧焊标准机器人配置', '2.2 外部轴配置', '2.2.5 滑台标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），操作“滑台标定”时，涉及“注意： 标定过程中改变滑台位置时必须正方向运动滑台到第二个点。 滑台标定尽量选择较远的距离，否则”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意： 标定过程中改变滑台位置时必须正方向运动滑台到第二个点', '滑台标定尽量选择较远的距离，否则影响精度'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 900, 'line_end': 900, 'excerpt': '<table><tr><td><p><img src="img/25dc0dc552eb1f5b5f2e14db79ace0b2bd7b5ad4ed8222d6293052bead37bcd4.png"/>注意：</p><p>标定过程中改变滑台位置时必须正方向运动滑台到第二个点。</p><p>滑台标定尽量选择较远的距离，否则影响精度。</p></td></tr></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_05', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.1.1', 'section_title': '5.1.1 机器人端配置', 'heading_path': ['5 智能焊接-激光篇', '5.1 线激光传感器', '5.1.1 机器人端配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），配置“机器人端配置”时，“搜索选取点”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['8. 搜索选取点：主要是静态搜索和动态搜索时，在选取结果时通过该参数设置进行选取最终位置', '范围（1-10）mm，数值越小越靠近开始搜索位置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2187, 'line_end': 2187, 'excerpt': '  8. 搜索选取点：主要是静态搜索和动态搜索时，在选取结果时通过该参数设置进行选取最终位置；范围（1-10）mm，数值越小越靠近开始搜索位置。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_06', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.1.2', 'section_title': '3.1.2 基准面设置', 'heading_path': ['3 辅助焊接功能', '3.1 弧焊坐标系统', '3.1.2 基准面设置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“基准面设置”中“WREF TYPE”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['WREF', 'TYPE'], 'expected_facts': ['WREF TYPE 功能 设置焊接基础面', '格式 WREF TYPE = N 说明 参数项 TYPE 基础面类型', '1：(0,0,1)世界下Z+方向 2：(0,1,0)世界下Z+方向 3：(1,0,0)世界Z+方向 4：(x,y,z)当前世界下焊枪（TCP）Z-方向 举例 WREF TYPE = 1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1269, 'line_end': 1269, 'excerpt': '<table><tr><td rowspan="4"><p>WREF TYPE</p></td><td><p>功能</p></td><td colspan="2"><p>设置焊接基础面。</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>WREF TYPE = N</p></td></tr><tr><td><p>说明</p></td><td><p>参数项</p></td><td><p>TYPE 基础面类型。</p><p>1：(0,0,1)世界下Z+方向</p><p>2：(0,1,0)世界下Z+方向</p><p>3：(1,0,0)世界Z+方向</p><p>4：(x,y,z)当前世界下焊枪（TCP）Z-方向</p></td></tr><tr><td><p>举例</p></td><td colspan="2"><p>WREF TYPE = 1</p></td></tr></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_07', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.5.3', 'section_title': '5.5.3 作业示例', 'heading_path': ['5 智能焊接-激光篇', '5.5 激光跟踪', '5.5.3 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），操作“作业示例”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意： LSENSORTRACKON 要求在起弧点之上，设置寻位起点尽量与起弧点平行'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2401, 'line_end': 2401, 'excerpt': '注意： LSENSORTRACKON 要求在起弧点之上，设置寻位起点尽量与起弧点平行。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_08', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.1.3', 'section_title': '6.1.3 指令说明', 'heading_path': ['6 多道焊', '6.1 循环焊接', '6.1.3 指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“指令说明”中“MLRPMON RPM”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['MLRPMON', 'RPM'], 'expected_facts': ['MLRPMON RPM 功能 根道记录开始 格式 MLRPMON RPM [参数1] 说明 参数1 文件号 MLRPMOFF 功能 根道记录结束 格式 MLRPMOFF 说明 MLOFFSETON PR RMM 功能 覆盖道偏移开始 格式 MLOFFSETON PR [参数1] RMM [参数2] 说明 参数1 全局变量（此全局变量为焊缝偏移量） 参数2 文件号 MLOFFSETOFF 功能 覆盖道偏移结束 格式 MLOFFSETOFF 说明'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2470, 'line_end': 2470, 'excerpt': '<table><tr><td rowspan="3"><p>MLRPMON</p><p>RPM</p></td><td><p>功能</p></td><td colspan="2"><p>根道记录开始</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>MLRPMON RPM [参数1]</p></td></tr><tr><td><p>说明</p></td><td><p>参数1</p></td><td><p>文件号</p></td></tr><tr><td rowspan="3"><p>MLRPMOFF</p></td><td><p>功能</p></td><td colspan="2"><p>根道记录结束</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>MLRPMOFF</p></td></tr><tr><td><p>说明</p></td><td></td><td></td></tr><tr><td rowspan="4"><p>MLOFFSETON PR RMM</p></td><td><p>功能</p></td><td colspan="2"><p>覆盖道偏移开始</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>MLOFFSETON PR [参数1]  RMM [参数2]</p></td></tr><tr><td rowspan="2"><p>说明</p></td><td><p>参数1</p></td><td><p>全局变量（此全局变量为焊缝偏移量）</p></td></tr><tr><td><p>参数2</p></td><td><p>文件号</p></td></tr><tr><td rowspan="3"><p>MLOFFSETOFF</p></td><td><p>功能</p></td><td colspan="2"><p>覆盖道偏移结束</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>MLOFFSETOFF</p></td></tr><tr><td><p>说明</p></td><td></td><td></td></tr></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_09', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.6.3', 'section_title': '4.6.3 作业示例', 'heading_path': ['4 智能焊接-电弧篇', '4.6 接触探测传感(TDS传感)', '4.6.3 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“作业示例”中“在使用TDS指令时”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['TDS'], 'expected_facts': ['在使用TDS指令时，TDS指令之前必须建立焊缝坐标系，指令之后必须有焊缝起点、起弧文件号和焊缝终点，用于指导建立焊缝坐标系'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1984, 'line_end': 1984, 'excerpt': '在使用TDS指令时，TDS指令之前必须建立焊缝坐标系，指令之后必须有焊缝起点、起弧文件号和焊缝终点，用于指导建立焊缝坐标系。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_10', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '5.1.2', 'section_title': '5.1.2 传感器端配置', 'heading_path': ['5 智能焊接-激光篇', '5.1 线激光传感器', '5.1.2 传感器端配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），在“传感器端配置”中，应如何理解或操作“前提是要在电脑端设置同网段IP地址”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['2. 前提是要在电脑端设置同网段IP地址', '操作如下：在电脑端右下角右键网络，选择网络和internet，网络和共享中心，更改适配器设置，选择机器人与传感器连接的网络，右键选择属性，双击打开Internet协议版本4，点击属性，选择使用下面的IP地址，修改IP地址为192.168.3.138即可'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2200, 'line_end': 2200, 'excerpt': '2. 前提是要在电脑端设置同网段IP地址。操作如下：在电脑端右下角右键网络，选择网络和internet，网络和共享中心，更改适配器设置，选择机器人与传感器连接的网络，右键选择属性，双击打开Internet协议版本4，点击属性，选择使用下面的IP地址，修改IP地址为192.168.3.138即可。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_11', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '8.1', 'section_title': '8.1 报警信息', 'heading_path': ['8 报警信息', '8.1 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），在“报警信息”中，应如何理解或操作“报警码”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['报警码 报警信息 解决方法 20081 焊机通讯断开！ 请检查通讯配置界面，也可能存在干扰等问题 20082 焊枪发生碰撞! 解除碰撞，检查焊机对应的焊枪碰撞信号状态 20083 焊机异常! 焊机发生故障，请复位焊机故障 20084 焊枪碰撞! 解除碰撞，检查焊机对应的焊枪碰撞信号状态 20085 压缩空气异常! 无压缩空气，检查焊接状态监控中压缩空气信号状态 20086 水流量异常! 水流量低，检查焊接状态监控中水流量信号状态 20087 保护气故障! 无保护气，检查焊接状态监控中保护气信号状态 20088 缺丝! 送丝异常，检查焊接状态监控中送丝信号状态 20089 机器人焊枪发生碰撞，4、5、6松抱闸 请移动机器人到安全位置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2534, 'line_end': 2534, 'excerpt': '<table><thead><tr><th><p><strong>报警码</strong></p></th><th><p><strong>报警信息</strong></p></th><th><p><strong>解决方法</strong></p></th></tr><tr><th><p>20081</p></th><th><p>焊机通讯断开！</p></th><th><p>请检查通讯配置界面，也可能存在干扰等问题</p></th></tr><tr><th><p>20082</p></th><th><p>焊枪发生碰撞!</p></th><th><p>解除碰撞，检查焊机对应的焊枪碰撞信号状态</p></th></tr><tr><th><p>20083</p></th><th><p>焊机异常!</p></th><th><p>焊机发生故障，请复位焊机故障</p></th></tr><tr><th><p>20084</p></th><th><p>焊枪碰撞!</p></th><th><p>解除碰撞，检查焊机对应的焊枪碰撞信号状态</p></th></tr><tr><th><p>20085</p></th><th><p>压缩空气异常!</p></th><th><p>无压缩空气，检查焊接状态监控中压缩空气信号状态</p></th></tr><tr><th><p>20086</p></th><th><p>水流量异常!</p></th><th><p>水流量低，检查焊接状态监控中水流量信号状态</p></th></tr><tr><th><p>20087</p></th><th><p>保护气故障!</p></th><th><p>无保护气，检查焊接状态监控中保护气信号状态</p></th></tr><tr><th><p>20088</p></th><th><p>缺丝!</p></th><th><p>送丝异常，检查焊接状态监控中送丝信号状态</p></th></tr><tr><th><p>20089</p></th><th><p>机器人焊枪发生碰撞，4、5、6松抱闸</p></th><th><p>请移动机器人到安全位置</p></th></tr></thead></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_12', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.3', 'section_title': '3.3 断点工艺恢复', 'heading_path': ['3 辅助焊接功能', '3.3 断点工艺恢复'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），操作“断点工艺恢复”时，涉及“多道焊跳转功能置ON时”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['6. 多道焊跳转功能置ON时，若当前的P光标不在循环区间外时，选择直接启动或暂停处启动后，会提示应用禁止启动弹窗', '需要移动机器人到循环区间外启动'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1428, 'line_end': 1428, 'excerpt': '   6. 多道焊跳转功能置ON时，若当前的P光标不在循环区间外时，选择直接启动或暂停处启动后，会提示应用禁止启动弹窗。需要移动机器人到循环区间外启动。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_13', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '8.3', 'section_title': '8.3 提示信息', 'heading_path': ['8 报警信息', '8.3 提示信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“提示信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['ARCON', 'ARCOFF'], 'expected_facts': ['报警码 报警信息 解决方法 20041 焊接允许关闭 提示信息无需解决 20042 焊接允许开启 提示信息无需解决 20043 本次工艺恢复无效开关关闭 提示信息无需解决 20044 本次工艺恢复无效开关开启 提示信息无需解决 20045 工艺恢复开关关闭 提示信息无需解决 20046 工艺恢复开关开启 提示信息无需解决 20047 预留 无 20048 焊机通讯已连接！ 提示信息无需解决 20049 焊机选择错误/缺少配置文件 检查焊机型号并重新配置焊机型号/增加配置文件 20050 焊机品牌/型号更改后重新启动生效 提示信息无需解决 20051 单步模式开启 不执行焊接流程 提示信息无需解决 20701 间隔距离为0,将上抬高度自动置为0 提示信息无需解决 20052 ARCON指令不支持偏移起弧和飞行焊 提示信息无需解决 20053 ARCOFF指令不支持摆动收弧和飞行焊 提示信息无需解决 20703 断续焊分段数量超出最大限制无法断电恢复 减少连续电焊分段数量 20704 传感动作x 寻位基准保存变量成功 提示信息无需解决 20153 全视线激光任务号与焊缝类型不符! x 处理方法：检查激光器任务号 检查激光器任务号 20154 连续异常点，检查后可继续 可继续,但是也需要人工确认一下是否有异物阻挡 20155 连续异常点，不可继续 不可继续，人工确认了也不可以继续，因为焊道信息错误了 20156 直线跟踪的过渡点不在直线上 请重新示教 20157 设置的跟踪文件号太大，超过文件数量 配置范围 1-8! 20158 跟踪过渡点中有两个点是相同的 x 检查过渡点并重新示教 20159 搜索结束，未找到起点,因为搜索范围内的焊缝未达到要求长度! 重新示教搜索终点 20160 当前位置无跟踪点! 换个位置或者检查激光器任务号是否正确 20161 示教起点距离焊缝过远! 请重新示教一个距离近的起点 20162 一秒内未寻到点 请检查线激光配置! 20164 设定全局变量失败 请检查全局变量号是否存在 20901 多道焊MLOFFSETON指令变量格式错误 请参照使用说明或咨询工作人员使用正确的指令变量格式 20902 多道焊偏移运动指令数量与根轨迹运动指令数量不一致 请保证多道焊偏移运动指令数量与根轨迹运动指令数量一致 20903 多道焊偏移运动指令变量名称与根轨迹运动指令变量名称不一致 请保证多道焊偏移运动指令变量名称与根轨迹运动指令变量名称一致 20904 多道焊无法计算偏移值 请检查作业中多道焊的偏移量参数 20165 不在一个平面上，无法相交 调大误差距离或重新示教 20166 两个圆之间没有交点 重新示教 20167 两个直线夹角过小，无法相交 重新示教 20168 直线和圆之间没有交点 重新示教'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2542, 'line_end': 2542, 'excerpt': '<table><thead><tr><th><p><strong>报警码</strong></p></th><th><p><strong>报警信息</strong></p></th><th><p><strong>解决方法</strong></p></th></tr><tr><th><p>20041</p></th><th><p>焊接允许关闭</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20042</p></th><th><p>焊接允许开启</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20043</p></th><th><p>本次工艺恢复无效开关关闭</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20044</p></th><th><p>本次工艺恢复无效开关开启</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20045</p></th><th><p>工艺恢复开关关闭</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20046</p></th><th><p>工艺恢复开关开启</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20047</p></th><th><p>预留</p></th><th><p>无</p></th></tr><tr><th><p>20048</p></th><th><p>焊机通讯已连接！</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20049</p></th><th><p>焊机选择错误/缺少配置文件</p></th><th><p>检查焊机型号并重新配置焊机型号/增加配置文件</p></th></tr><tr><th><p>20050</p></th><th><p>焊机品牌/型号更改后重新启动生效</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20051</p></th><th><p>单步模式开启 不执行焊接流程</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20701</p></th><th><p>间隔距离为0,将上抬高度自动置为0</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20052</p></th><th><p>ARCON指令不支持偏移起弧和飞行焊</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20053</p></th><th><p>ARCOFF指令不支持摆动收弧和飞行焊</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20703</p></th><th><p>断续焊分段数量超出最大限制无法断电恢复</p></th><th><p>减少连续电焊分段数量</p></th></tr><tr><th><p>20704</p></th><th><p>传感动作x 寻位基准保存变量成功</p></th><th><p>提示信息无需解决</p></th></tr><tr><th><p>20153</p></th><th><p>全视线激光任务号与焊缝类型不符! x 处理方法：检查激光器任务号</p></th><th><p>检查激光器任务号</p></th></tr><tr><th><p>20154</p></th><th><p>连续异常点，检查后可继续</p></th><th><p>可继续,但是也需要人工确认一下是否有异物阻挡</p></th></tr><tr><th><p>20155</p></th><th><p>连续异常点，不可继续</p></th><th><p>不可继续，人工确认了也不可以继续，因为焊道信息错误了</p></th></tr><tr><th><p>20156</p></th><th><p>直线跟踪的过渡点不在直线上</p></th><th><p>请重新示教</p></th></tr><tr><th><p>20157</p></th><th><p>设置的跟踪文件号太大，超过文件数量</p></th><th><p>配置范围 1-8!</p></th></tr><tr><th><p>20158</p></th><th><p>跟踪过渡点中有两个点是相同的 x </p></th><th><p>检查过渡点并重新示教</p></th></tr><tr><th><p>20159</p></th><th><p>搜索结束，未找到起点,因为搜索范围内的焊缝未达到要求长度! </p></th><th><p>重新示教搜索终点</p></th></tr><tr><th><p>20160</p></th><th><p>当前位置无跟踪点!</p></th><th><p>换个位置或者检查激光器任务号是否正确</p></th></tr><tr><th><p>20161</p></th><th><p>示教起点距离焊缝过远!</p></th><th><p>请重新示教一个距离近的起点</p></th></tr><tr><th><p>20162</p></th><th><p>一秒内未寻到点</p></th><th><p>请检查线激光配置!</p></th></tr><tr><th><p>20164</p></th><th><p>设定全局变量失败</p></th><th><p>请检查全局变量号是否存在</p></th></tr><tr><th><p>20901</p></th><th><p>多道焊MLOFFSETON指令变量格式错误</p></th><th><p>请参照使用说明或咨询工作人员使用正确的指令变量格式</p></th></tr><tr><th><p>20902</p></th><th><p>多道焊偏移运动指令数量与根轨迹运动指令数量不一致</p></th><th><p>请保证多道焊偏移运动指令数量与根轨迹运动指令数量一致</p></th></tr><tr><th><p>20903</p></th><th><p>多道焊偏移运动指令变量名称与根轨迹运动指令变量名称不一致</p></th><th><p>请保证多道焊偏移运动指令变量名称与根轨迹运动指令变量名称一致</p></th></tr><tr><th><p>20904</p></th><th><p>多道焊无法计算偏移值</p></th><th><p>请检查作业中多道焊的偏移量参数</p></th></tr><tr><th><p>20165</p></th><th><p>不在一个平面上，无法相交</p></th><th><p>调大误差距离或重新示教</p></th></tr><tr><th><p>20166</p></th><th><p>两个圆之间没有交点</p></th><th><p>重新示教</p></th></tr><tr><th><p>20167</p></th><th><p>两个直线夹角过小，无法相交</p></th><th><p>重新示教</p></th></tr><tr><th><p>20168</p></th><th><p>直线和圆之间没有交点</p></th><th><p>重新示教</p></th></tr></thead></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_14', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.1.6', 'section_title': '2.1.6 机器人应用配置', 'heading_path': ['2 弧焊标准机器人配置', '2.1 弧焊机器人配置', '2.1.6 机器人应用配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），在“机器人应用配置”中，应如何理解或操作“如果主菜单中没有弧焊”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['如果主菜单中没有弧焊，需要先切换到超级用户再选择弧焊应用'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 460, 'line_end': 460, 'excerpt': '如果主菜单中没有弧焊，需要先切换到超级用户再选择弧焊应用。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_15', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '3.2.3', 'section_title': '3.2.3 摆动文件配置', 'heading_path': ['3 辅助焊接功能', '3.2 摆焊功能', '3.2.3 摆动文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），配置“摆动文件配置”时，“点击【主菜单】-【弧焊】-【摆动参数】-【摆动文件】进入摆动参数配置界面”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['点击【主菜单】-【弧焊】-【摆动参数】-【摆动文件】进入摆动参数配置界面，如图3-8所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1331, 'line_end': 1331, 'excerpt': '点击【主菜单】-【弧焊】-【摆动参数】-【摆动文件】进入摆动参数配置界面，如图3-8所示：', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_16', 'category': 'arc_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '4.6.3', 'section_title': '4.6.3 作业示例', 'heading_path': ['4 智能焊接-电弧篇', '4.6 接触探测传感(TDS传感)', '4.6.3 作业示例'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），操作“作业示例”时，涉及“首先确保焊丝干伸长（禁止用出丝补偿时间确定干伸长）”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['首先确保焊丝干伸长（禁止用出丝补偿时间确定干伸长）'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1986, 'line_end': 1986, 'excerpt': '首先确保焊丝干伸长（禁止用出丝补偿时间确定干伸长）。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_17', 'category': 'arc_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '6.1.2.3', 'section_title': '6.1.2.3 拐角变焦', 'heading_path': ['6 多道焊', '6.1 循环焊接', '6.1.2 焊道偏移', '6.1.2.3 拐角变焦'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），在“拐角变焦”中，应如何理解或操作“当焊接线中途拐弯的情况下”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['当焊接线中途拐弯的情况下，为了保证焊接轨迹的平顺，在计算偏移后的轨迹时在拐角处会自动进行变焦，如图6-6所示，变焦功能将会自动调节焊接线的长度', '在轨迹拐角大于90°呈锐角情况下，要保证轨迹点位之间有足够的间隔距离（偏移量越大，需要的点位间隔距离需要相应提高），否则会导致焊接轨迹异常以及报警：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2454, 'line_end': 2454, 'excerpt': '当焊接线中途拐弯的情况下，为了保证焊接轨迹的平顺，在计算偏移后的轨迹时在拐角处会自动进行变焦，如图6-6所示，变焦功能将会自动调节焊接线的长度。在轨迹拐角大于90°呈锐角情况下，要保证轨迹点位之间有足够的间隔距离（偏移量越大，需要的点位间隔距离需要相应提高），否则会导致焊接轨迹异常以及报警：', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_18', 'category': 'arc_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.5.5.3', 'section_title': '2.5.5.3 焊接文件', 'heading_path': ['2 弧焊标准机器人配置', '2.5 弧焊应用参数配置', '2.5.5 焊接参数设置', '2.5.5.3 焊接文件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），配置“焊接文件”时，“焊接电压”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['5. 焊接电压：参数范围：[-50.0，50.0]', '单位V，默认值30'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1200, 'line_end': 1200, 'excerpt': '5. 焊接电压：参数范围：[-50.0，50.0]；单位V，默认值30。', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_19', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '7.1.3', 'section_title': '7.1.3 寻位指令', 'heading_path': ['7 附录', '7.1 弧焊逻辑指令', '7.1.3 寻位指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“寻位指令”中“SEARCHON TOUCH”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SEARCHON', 'TOUCH'], 'expected_facts': ['SEARCHON TOUCH 功能 起始点检测指令', '格式 SEARCH TOUCH[参数1] 参数1 起始点检测文件号', '举例 SEARCH TOUCH[1] SEARCHON C 功能 圆弧寻位指令'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 2508, 'line_end': 2508, 'excerpt': '<table><tr><td rowspan="4"><p>SEARCHON TOUCH</p></td><td><p>功能</p></td><td colspan="2"><p>起始点检测指令。</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>SEARCH TOUCH[参数1]</p></td></tr><tr><td></td><td><p>参数1</p></td><td><p>起始点检测文件号。</p></td></tr><tr><td><p>举例</p></td><td colspan="2"><p>SEARCH TOUCH[1]</p></td></tr><tr><td rowspan="4"><p>SEARCHON C</p></td><td><p>功能</p></td><td colspan="2"><p>圆弧寻位指令。</p></td></tr><tr><td><p>格式</p></td><td colspan="2"><p>SEARCHON C[参数1]</p></td></tr><tr><td></td><td><p>参数1</p></td><td><p>圆弧寻位文件号。</p></td></tr><tr><td><p>举例</p></td><td colspan="2"><p>SEARCHON C[1]</p></td></tr></table>', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}, {'id': 'sample_arc_welding_e39a0c6604_20', 'category': 'arc_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-1', 'section': '2.5.2', 'section_title': '2.5.2 焊接指令', 'heading_path': ['2 弧焊标准机器人配置', '2.5 弧焊应用参数配置', '2.5.2 焊接指令'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'source_version': 'rc5', 'question': '根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“焊接指令”中“焊接指令主要包括起弧指令、收弧指令以及焊接参数修改指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['焊接指令主要包括起弧指令、收弧指令以及焊接参数修改指令，单步不支持起弧处理', '说明如下：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'line_start': 1071, 'line_end': 1071, 'excerpt': '焊接指令主要包括起弧指令、收弧指令以及焊接参数修改指令，单步不支持起弧处理。说明如下：', 'sha256': 'cb17aba55d2a6186662accc530685811239778d3c703497252d4ee2e0c947580'}]}]
register_sample_bank('arc_welding', ARC_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md', 'seed': 45856436115095825, 'population': 1600, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(2584, 2584), (1237, 1237), (4022, 4022), (4509, 4509), (2410, 2410), (2580, 2580), (4131, 4131), (5898, 5898), (2695, 2695), (5008, 5008), (3872, 3872), (1522, 1522), (5245, 5245), (5835, 5835), (4735, 4735), (4755, 4755), (5693, 5693), (1863, 1863), (1420, 1420), (3526, 3526)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md', 'seed': 14989727907770866743, 'population': 667, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(2480, 2480), (373, 373), (2112, 2112), (900, 900), (2187, 2187), (1269, 1269), (2401, 2401), (2470, 2470), (1984, 1984), (2200, 2200), (2534, 2534), (1428, 1428), (2542, 2542), (460, 460), (1331, 1331), (1986, 1986), (2454, 2454), (1200, 1200), (2508, 2508), (1071, 1071)]}])


In [ ]:
# | notest
ARC_WELDING_SAMPLE_IDS = None
arc_welding_sample_cases = select_cases(ARC_WELDING_SAMPLE_CASES, ARC_WELDING_SAMPLE_IDS)
display(pd.DataFrame(arc_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in arc_welding_sample_cases])
arc_welding_sample_metrics = await evaluation.run(arc_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
arc_welding_sample_results = {case["id"]: evaluation.load(case) for case in arc_welding_sample_cases}
display(arc_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(arc_welding_sample_cases[0]["id"])
evaluation.show_answers(arc_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_arc_welding_d81b2686c9_01,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,instruction,prose,True,True,8,True,7,7,0,True,False,False,False,False,19210,0,True,unsupported,,,not_assessable
sample_arc_welding_d81b2686c9_02,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,prerequisite,prose,True,True,8,True,8,8,0,True,False,False,None,None,12199,0,True,unsupported,,,abstained
sample_arc_welding_d81b2686c9_03,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,parameter,prose,True,True,8,True,8,8,0,True,False,False,None,None,13750,0,True,included,,,not reviewed
sample_arc_welding_d81b2686c9_04,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,parameter,prose,True,True,8,True,8,8,0,True,False,False,None,None,12935,0,True,unsupported,,,not reviewed
sample_arc_welding_d81b2686c9_05,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,procedure,prose,True,True,8,True,6,6,0,True,False,False,None,None,16927,1,True,included,,,not reviewed
sample_arc_welding_d81b2686c9_06,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,parameter,prose,True,True,8,True,8,8,0,True,False,False,None,None,14200,0,True,included,,,not reviewed
sample_arc_welding_d81b2686c9_07,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,instruction,table_row,True,True,8,True,6,5,0,True,False,False,False,True,15957,0,True,unsupported,,,not reviewed
sample_arc_welding_d81b2686c9_08,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,prerequisite,table_row,True,True,8,False,0,0,0,False,False,False,None,None,0,8,True,not generated,,,not reviewed
sample_arc_welding_d81b2686c9_09,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,prerequisite,prose,True,True,8,True,7,7,0,True,False,False,None,None,20524,0,True,included,,,not reviewed
sample_arc_welding_d81b2686c9_10,arc_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md,V5.0 / B-5,parameter,prose,True,True,8,True,8,8,0,True,False,False,None,None,12951,0,True,unsupported,,,not reviewed


### sample_arc_welding_d81b2686c9_01

根据《SX322003《新松机器人弧焊应用操作手册》(B-5)》（V5.0 / B-5），“起始点检测”中“同时”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 新松机器人弧焊应用手册（V5.0）中的一项功能组合：一维传感 + 圆弧检测。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 4393–4395; corpus 5.0, revision V5.0 / C-1; object `385e37527ed61ae3c11c05ee47bd2512fcb410abddab90ce37610232a254a7e8`.

- 该表格行（索引24）来自《新松机器人弧焊应用操作手册》，定义了传感器配置：当参数设置为 ARCL P14、VL=10、ACC=100、CNT=0 时，对应的功能状态为“传感起点3”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 6798–6801; corpus 4.0, revision V5.0 / B-5; object `3ae48f004793b5346fe5587b48cc509a8766ecb9834e293fafe27e26d66e1469`.

- 新松机器人弧焊应用操作手册（第5行）参数记录：[4, 2, 0, 3, 50, 3, -, 150, -]，对应子项“e）”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 10503–10514; corpus 4.0, revision V5.0 / B-5; object `9be5747fbd75d6505c3ffb3a4380fe5d8e79016ef5bcb0820d3bd281ffb1f236`.

- 新松机器人弧焊应用手册中，参数配置 `ARCL P1 VL = 100 ACC = 100 CNT = 0` 被定义或标记为“起点安全点”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 4587–4590; corpus 4.0, revision V5.0 / B-5; object `5ab1e23fc8ff43178dac3139ff24cbc928b98b111052808829fa2d1e6b652d59`.

- 该行来自《新松机器人弧焊应用操作手册》（SX322003, B-5）第24行，包含两个单元格：第一列为文本“END”，第二列为空。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 3792–3795; corpus 4.0, revision V5.0 / B-5; object `8ce0e2d40a9f01455410cdc27ca26b3483f7622ffe0f1e785d70761633f6e667`.

- 该行记录为“ARCOFF #1”，第二列为空值，无继承合并单元格内容，源文档为新松机器人弧焊应用操作手册（V5.0，SX322003）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 11260–11263; corpus 4.0, revision V5.0 / B-5; object `08877cfad145909f84df7f3a3f3486f87189e4577069a39cafd370adf8e9b7c2`.

- 新松机器人弧焊应用操作手册中，某数据行（索引3）的数值依次为：2、50、0、0、0、0。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 3059–3066; corpus 4.0, revision V5.0 / B-5; object `00be736513441b4d8934ba402cf7ff1059961cd6e1af04f0cfa5005ded84fc04`.

- 新松机器人弧焊指令 `ARCL P5`（电压100，电流100，计数0）对应“传感离开点”功能。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 6464–6467; corpus 4.0, revision V5.0 / B-5; object `d5a5e3ee37166013845e24a1e63cf06bf5c422c059c55dcbf67da9e78b650f79`.

#### Answer

在所提供的全部检索证据（E1–E8）中，

#### Model-assisted source review

Assessment: **not_assessable**; completeness: none; citation support: not_applicable; revision fidelity: mixed.

The generated answer is a fragment ('在所提供的全部检索证据（E1–E8）中，') and does not answer the question. It does not state the function of '同时' in '起始点检测'. The expected fact (storing offset to P102/P112) is not present in the supplied evidence (E1-E8), so the model could not have cited it. However, the answer is not a valid refusal or abstention; it is an incomplete sentence. Therefore, it is not a correct answer, nor a valid abstention.

...

#### Answer

在所提供的 V5.0 / C-1 手册证据中，未找到以 "SEARCHON TOUCH" 为完整指令形式的记录。手册中实际记载的是两类不同指令：(1) SEARCHON 后接模式参数，如 SEARCHON 2D_R [1]（二维寻位初始化，E5/E1）、SEARCHON 3D_3R [1]（三维寻位，E4）、SEARCHON 3D[1]（一维寻位，E7）；(2) SEARCH TOUCH[1]（起始点寻位指令，E6），其格式为 "SEARCH TOUCH[参数]"，而非 "SEARCHON TOUCH"。两者语法结构不同，不能将 "SEARCHON" 前缀与 "TOUCH" 模式直接组合。由于检索证据中不存在 "SEARCHON TOUCH" 这一确切指令的语法、参数或行为描述，无法据此构建符合要求的上下文示例。

### sample_arc_welding_e39a0c6604_20

根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》（V5.0 / C-1），“焊接指令”中“焊接指令主要包括起弧指令、收弧指令以及焊接参数修改指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 新松机器人弧焊指令 `ARCL P2 VL = 20 ACC =100 CNT = 0` 用于指定焊缝起点，其中 VL=20 表示线速度，ACC=100 表示加速度，CNT=0 表示计数；在联动状态下，该指令等效于 `SARCL VL=20`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 5991–5994; corpus 4.0, revision V5.0 / B-5; object `6646b98439eee29f6f806cfde81d8dcbad741e70993de18ed4bab77de7b7544a`.

- 新松机器人弧焊指令 `MOVL P[3] VL=100 ACC=100 CNT=0` 对应功能为**焊接起点**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 5503–5506; corpus 5.0, revision V5.0 / C-1; object `965110ebb39f25d233ac8a7a8cb3469d0ffaa8162189854320e94cd7e5a4e040`.

- 该行描述了新松机器人弧焊指令 **ARCL** 的调用格式，具体为：`ARCL P/PR(参数1) VL=(参数2) ACC=(参数3) CNT=(参数4) ET/COORD=(参数5) INTERLOCK=(参数6)`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 15135–15138; corpus 4.0, revision V5.0 / B-5; object `5d77dd8f63c6e1a3f751e988d57314892084dc212a26ee63d70514a7cc438932`.

- 该行定义了一个弧焊指令：使用 `MOVL P[3] V=100 ACC=20 CNT=0` 命令，并指定焊接模式为 `ARCON[1][1]`；该指令对应的功能或说明为“焊接起点”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 4038–4041; corpus 5.0, revision V5.0 / C-1; object `a2a88ccf00bb4619bee0ae34272e7574cec37bc88ae6090e8683afe28cadd981`.

- 该行描述了机器人弧焊指令 `MOVL P[2] V=100 ACC=20 CNT=0：ARCON[1][1]` 的功能，即执行**起点变量设置与起弧**操作。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 5966–5969; corpus 5.0, revision V5.0 / C-1; object `4b5be4343a5441c4b60a20daf1ec998c87fb06d5e850fbd877a0a9524b848b56`.

- 新松机器人弧焊指令示例：ARCC P10（圆弧过程点），参数设定为 VC=30、ACC=100、CNT=0。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 12877–12880; corpus 4.0, revision V5.0 / B-5; object `ac17b5fc345b3fe521d53ff9bb8933dbd8d041a24026b5d38dc8a1ad91322b49`.

- 新松机器人弧焊指令 `ARCC P20 VC = 5 ACC = 100 CNT = 0` 表示“焊缝终点”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 6680–6683; corpus 4.0, revision V5.0 / B-5; object `840fea6db17f4a119004a8b3c390c6289e26b0b62628bcc2d54c399f733c74ed`.

- 新松机器人弧焊指令 `ARCL P5`（电压100，电流100，计数0）对应“传感离开点”功能。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 6464–6467; corpus 4.0, revision V5.0 / B-5; object `d5a5e3ee37166013845e24a1e63cf06bf5c422c059c55dcbf67da9e78b650f79`.

#### Answer

根据《SX322003《新松机器人弧焊应用操作手册》(C-1)》V5.0/C-1，焊接指令中的起弧指令为 ARCON，收弧指令为 ARCOFF。ARCON 接受两个文件号参数：第一个为起弧文件，第二个为焊接文件（如 ARCON[1][2] 表示起弧文件1、焊接文件2）；ARCOFF 接受一个文件号参数，表示收弧文件（如 ARCOFF[1] 表示收弧文件1）。在 C-1 手册的示例中，ARCON 可与 MOVL 运动指令以冒号连接（如 MOVL P[3] V=100 ACC=20 CNT=0：ARCON[1][1]），表示运动到焊接起点的同时执行起弧；ARCOFF 同理与终点运动指令连接（如 MOVL P[4] V=100 ACC=20 CNT=0：ARCOFF[1]），表示运动到焊接终点的同时执行收弧。关于'焊接参数修改指令'，所提供的 C-1 证据中未出现该指令的具体语法或参数说明，无法给出其确切格式。

### Example — arc_welding

在 C-1 手册的激光传感焊接作业中，机器人先完成起点/终点激光搜索，然后运动到焊接起点并起弧，沿焊缝运动到终点后收弧，最后回到安全点。

**Environment and prerequisites:** 前提：已选取激光传感器设备号（LSENSOR NO=1）并设置任务号（LSENSOR JOB=31）；位置变量 P[1]~P[8] 已示教；起弧文件[1]、焊接文件[2]、收弧文件[1] 已在焊接参数中配置。

**Expected result:** 机器人依次完成：运动到起点检测点→激光搜索确定焊缝起点→运动到终点检测点→激光搜索确定焊缝终点→运动到焊接起点并执行起弧（ARCON[1][2]）→运动到焊接终点并执行收弧（ARCOFF[1]）→运动到安全点，完成一次直线焊缝焊接。

**Manual:** SX322003《新松机器人弧焊应用操作手册》(C-1) V5.0 / C-1 [E2] [E4] [E5]

```text
LSENSOR NO = 1
LSENSOR JOB = 31
MOVL P[1] VL = 100 ACC = 100 CNT = 0
MOVL P[2] V=10 ACC=100 CNT=100: SEARCH LSENSOR OFF-ON ABS P[3] :OFFSET PR[98] MODE = 1
MOVL P[4] VL = 100 ACC = 100 CNT = 0
MOVL P[5] V=100 ACC=100 CNT=100: SEARCH LSENSOR OFF-ON ABS P[6] :OFFSET PR[99] MODE = 1
MOVL P[7] VL = 100 ACC = 100 CNT = 0
MOVL P[3] VL = 100 ACC = 100 CNT = 0
ARCON [1] [2]
MOVL P[6] VL = 10 ACC = 100 CNT = 0
ARCOFF [1]
MOVL P[8] VL = 100 ACC = 100 CNT = 0
END
```

#### Model-assisted source review

Assessment: **not_assessable**; completeness: partial; citation support: supported; revision fidelity: mixed.

The answer correctly identifies ARCON and ARCOFF as the arc start/stop commands and provides syntax examples (ARCON[1][2], ARCOFF[1]) supported by the supplied evidence [E2], [E4], and [E5] from the C-1 manual. It correctly notes that the specific syntax for 'welding parameter modification commands' (焊接参数修改指令) is not present in the provided evidence snippets, which is an appropriate limitation given the evidence. However, the expected facts explicitly include the statement '单步不支持起弧处理' (single-step does not support arc start processing), which is present in the reference text but omitted from the answer. Additionally, the answer includes a code example block that, while based on [E2], is not strictly required by the prompt's instruction to 'not supplement undocumented code' if the user only asked for the function/usage description, though the prompt also asks to explain the syntax/parameters actually given. The main gap is the omission of the specific behavioral constraint '单步不支持起弧处理' which is part of the expected facts and the source reference.

## Laser Welding — sampled manual revisions

40 cases from 2 CSV files. Set `LASER_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
LASER_WELDING_SAMPLE_CASES = [{'id': 'sample_laser_welding_183ed5349f_01', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），配置“文件配置”时，“激光焊应用中点击【文件配置】进入工艺参数配置界面”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 激光焊应用中点击【文件配置】进入工艺参数配置界面'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 593, 'line_end': 593, 'excerpt': '1. 激光焊应用中点击【文件配置】进入工艺参数配置界面；', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_02', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '1.1', 'section_title': '1.1 概述', 'heading_path': ['1 激光焊应用介绍', '1.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），配置“概述”时，“激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件，为客户提供了完备的激光焊接产品', '通过机器人全面控制关键部件，实现在示教器端设置激光控制参数，极大简化了操作流程，使用更加高效便捷'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 331, 'line_end': 331, 'excerpt': '激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件，为客户提供了完备的激光焊接产品。通过机器人全面控制关键部件，实现在示教器端设置激光控制参数，极大简化了操作流程，使用更加高效便捷。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_03', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“柜内与外部设备硬接线”中，应如何理解或操作“将水冷机电源线接入到电源柜”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['将水冷机电源线接入到电源柜，将地线PE接入到汇流排，将电源线接入到机器人控制柜负荷开关处，将接地线接入汇流排，如图2-9所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 418, 'line_end': 418, 'excerpt': '将水冷机电源线接入到电源柜，将地线PE接入到汇流排，将电源线接入到机器人控制柜负荷开关处，将接地线接入汇流排，如图2-9所示；', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_04', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），配置“柜内与外部设备硬接线”时，“表格 2-1套机默认信号参数表”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['表格 2-1套机默认信号参数表'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 424, 'line_end': 424, 'excerpt': '表格 2-1套机默认信号参数表', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_05', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），“文件配置”中“缓降时间”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['18. 缓降时间：从指令开始关闭激光位置到结束功率时间，范围0.000-10.000s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 619, 'line_end': 619, 'excerpt': '18. 缓降时间：从指令开始关闭激光位置到结束功率时间，范围0.000-10.000s。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_06', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '4', 'section_title': '4 激光焊指令说明', 'heading_path': ['4 激光焊指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），“激光焊指令说明”中“LASER ON”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['LASER', 'LASERON'], 'expected_facts': ['LASER ON 功能 达到目标点焊接开始', '格式 LASERON FILE=<参数1> T=<参数2> 说明 <参数1>：工艺参数文件号', '<参数2>：时间参数，单位s'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 660, 'line_end': 660, 'excerpt': '<table><tr><td rowspan="4"><p>LASER ON</p></td><td><p>功能</p></td><td><p>达到目标点焊接开始； </p></td></tr><tr><td><p>格式</p></td><td><p>LASERON FILE=&lt;参数1&gt; T=&lt;参数2&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p><p>&lt;参数2&gt;：时间参数，单位s；负值表示提前起弧，0与正值表示滞后起弧；</p></td></tr><tr><td><p>举例</p></td><td><p>LASERJ P1 VJ=10 ACC=100 CNT=0 LASERON FILE=1 T=1;</p><p>LASERL P1 VL=100 ACC=100 CNT=0 LASERON FILE=1 T=1;</p></td></tr><tr><td rowspan="4"><p>LASER OFF</p></td><td><p>功能</p></td><td><p>达到目标点焊接结束；</p></td></tr><tr><td><p>格式</p></td><td><p>LASEROFF FILE=&lt;参数1&gt; T=&lt;参数2&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p><p>&lt;参数2&gt;：时间参数，单位s；负值表示提前收弧，0与正值表示滞后收弧；；</p></td></tr><tr><td><p>举例</p></td><td><p>LASERJ P1 VJ=10 ACC=100 CNT=0 LASEROFF FILE=1 T=1;</p><p>LASERL P1 VL=100 ACC=100 CNT=0 LASEROFF FILE=1 T=1;</p></td></tr><tr><td rowspan="4"><p>LASERON</p></td><td><p>功能</p></td><td><p>焊接开始； </p></td></tr><tr><td><p>格式</p></td><td><p>LASERON FILE=&lt;参数1&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p></td></tr><tr><td><p>举例</p></td><td><p>LASERON FILE=1;</p></td></tr><tr><td rowspan="4"><p>LASEROFF</p></td><td><p>功能</p></td><td><p>焊接结束；</p></td></tr><tr><td><p>格式</p></td><td><p>LASEROFF FILE=&lt;参数1&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p></td></tr><tr><td><p>举例</p></td><td><p>LASEROFF FILE=1；</p></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_07', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '5', 'section_title': '5 模板作业', 'heading_path': ['5 模板作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322004《新松机器人激光焊应用操作手册》(A-4).md”（V5.0 / A-4），激光焊模板如何使用独立的LASERON和LASEROFF指令安排起弧、焊接运动和收弧？', 'anchors': ['LASERON', 'LASEROFF', 'MOVL P4'], 'expected_facts': ['LASERON FILE=1 起弧', 'MOVL P4 VL=10.0 ACC=100 CNT=0 焊接终点', 'LASEROFF FILE=1 收弧'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 666, 'line_end': 666, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVJ P1 VJ=10.0 ACC=100 CNT=1</p></td><td><p>安全点</p></td></tr><tr><td><p>MOVL P2 VL=300.0 ACC=100 CNT=100</p></td><td><p>接近点</p></td></tr><tr><td><p>MOVL P3 VL=100.0 ACC=100 CNT=0</p></td><td><p>焊接起点</p></td></tr><tr><td><p>LASERON FILE=1</p></td><td><p>起弧</p></td></tr><tr><td><p>MOVL P4 VL=10.0 ACC=100 CNT=0</p></td><td><p>焊接终点</p></td></tr><tr><td><p>LASEROFF FILE=1</p></td><td><p>收弧</p></td></tr><tr><td><p>MOVL P5 VL=300.0 ACC=100 CNT=100</p></td><td><p>离开点</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_08', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），操作“文件配置”时，涉及“工艺号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['9. 工艺号：不同工艺号调用激光头内不同摆动形式，工艺号0可不配置工艺号发送给激光器IO都为0，其余工艺号必须配置至少发送一个IO=1，否则报警工艺号为空，范围0-15'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 610, 'line_end': 610, 'excerpt': '9. 工艺号：不同工艺号调用激光头内不同摆动形式，工艺号0可不配置工艺号发送给激光器IO都为0，其余工艺号必须配置至少发送一个IO=1，否则报警工艺号为空，范围0-15。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_09', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.6', 'section_title': '3.6 快捷键配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.6 快捷键配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“快捷键配置”中，应如何理解或操作“操作方式”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['操作方式：首先按下示教器上的OP1按键将会打开快捷键功能，在应用切换处选择激光焊应用，之后可以根据配置通过OP2-OP6按键或快捷键界面中的软按键控制相应功能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 634, 'line_end': 634, 'excerpt': '操作方式：首先按下示教器上的OP1按键将会打开快捷键功能，在应用切换处选择激光焊应用，之后可以根据配置通过OP2-OP6按键或快捷键界面中的软按键控制相应功能。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_10', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '6', 'section_title': '6 报警信息', 'heading_path': ['6 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），“报警信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['LASERJ'], 'expected_facts': ['报警码 报警信息 处理方法 10101 激光器状态异常 检查激光器状态输入IO 10102 冷却水异常 检查冷却水状态输入IO 10103 保护气异常 检查保护气状态输入IO 10104 工艺号未配置 在输出IO工艺号配置中配置相应的工艺号 10105 文件号超界 指令调用工艺文件号超界 10106 不能在起弧状态下调用起弧指令 检查作业中的起弧指令 10107 不能在收弧状态下调用收弧指令 检查作业中的收弧指令 10108 关节运动不支持提前收弧 LASERJ指令中T只能是0 10109 激光异常超时 焊接时机器人不能停止，请检查 10110 送丝速度发送失败 检查参数是否超界或模拟量模块配置 10111 功率发送失败 检查参数是否超界或模拟量模块配置 10112 功率反馈异常 检查模拟量模块配置 10113 运动指令错误 检查圆弧指令是否使用错误 10114 圆弧指令错误 检查圆弧指令是否使用错误 10115 摆动频率发送失败 检查参数是否超界或模拟量模块配置 10116 摆动幅度发送失败 检查参数是否超界或模拟量模块配置 10117 起弧失败 检查设备异常 10118 粘丝 解除粘丝 10119 焊枪碰撞 解除焊枪碰撞 10120 摆动高度发送失败 检查参数是否超界或模拟量模块配置 10121 旋转角度发送失败 检查参数是否超界或模拟量模块配置 10122 组IO发送错误 请检查自定义组IO配置 10123 回抽长度发送失败 检查参数是否超界或模拟量模块配置 10124 补丝长度发送失败 检查参数是否超界或模拟量模块配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 670, 'line_end': 670, 'excerpt': '<table><tr><td><p><strong>报警码</strong></p></td><td><p><strong>报警信息 </strong></p></td><td><p><strong>处理方法</strong></p></td></tr><tr><td><p>10101</p></td><td><p>激光器状态异常</p></td><td><p>检查激光器状态输入IO</p></td></tr><tr><td><p>10102</p></td><td><p>冷却水异常</p></td><td><p>检查冷却水状态输入IO</p></td></tr><tr><td><p>10103</p></td><td><p>保护气异常</p></td><td><p>检查保护气状态输入IO</p></td></tr><tr><td><p>10104</p></td><td><p>工艺号未配置</p></td><td><p>在输出IO工艺号配置中配置相应的工艺号</p></td></tr><tr><td><p>10105</p></td><td><p>文件号超界</p></td><td><p>指令调用工艺文件号超界</p></td></tr><tr><td><p>10106</p></td><td><p>不能在起弧状态下调用起弧指令</p></td><td><p>检查作业中的起弧指令</p></td></tr><tr><td><p>10107</p></td><td><p>不能在收弧状态下调用收弧指令</p></td><td><p>检查作业中的收弧指令</p></td></tr><tr><td><p>10108</p></td><td><p>关节运动不支持提前收弧</p></td><td><p>LASERJ指令中T只能是0</p></td></tr><tr><td><p>10109</p></td><td><p>激光异常超时</p></td><td><p>焊接时机器人不能停止，请检查</p></td></tr><tr><td><p>10110</p></td><td><p>送丝速度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10111</p></td><td><p>功率发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10112</p></td><td><p>功率反馈异常</p></td><td><p>检查模拟量模块配置</p></td></tr><tr><td><p>10113</p></td><td><p>运动指令错误</p></td><td><p>检查圆弧指令是否使用错误</p></td></tr><tr><td><p>10114</p></td><td><p>圆弧指令错误</p></td><td><p>检查圆弧指令是否使用错误</p></td></tr><tr><td><p>10115</p></td><td><p>摆动频率发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10116</p></td><td><p>摆动幅度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10117</p></td><td><p>起弧失败</p></td><td><p>检查设备异常</p></td></tr><tr><td><p>10118</p></td><td><p>粘丝</p></td><td><p>解除粘丝</p></td></tr><tr><td><p>10119</p></td><td><p>焊枪碰撞</p></td><td><p>解除焊枪碰撞</p></td></tr><tr><td><p>10120</p></td><td><p>摆动高度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10121</p></td><td><p>旋转角度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10122</p></td><td><p>组IO发送错误</p></td><td><p>请检查自定义组IO配置</p></td></tr><tr><td><p>10123</p></td><td><p>回抽长度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>10124</p></td><td><p>补丝长度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_11', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“激光安全”中，应如何理解或操作“激光安全标识包括”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['激光安全标识包括：安全警示、激光输出头警示、激光辐射标识、产品铭牌等', '安全及信息标识详细说明如下表所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 345, 'line_end': 345, 'excerpt': '激光安全标识包括：安全警示、激光输出头警示、激光辐射标识、产品铭牌等。安全及信息标识详细说明如下表所示：', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_12', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），操作“激光安全”时，涉及“本产品属于Class 4 类激光产品”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['本产品属于Class 4 类激光产品，输出高功率的不可见激光，波长1080±10nm，会对直接或间接暴露在这种强激光下的眼睛或皮肤造成伤害，也可能引起现场火灾，因此请严格按照欧盟EN60825-1：2014 标准，所有操作或者靠近激光器的人员必须注意这些特殊危害，佩戴波长匹配且经过认证的激光防护眼镜，并做好充分的安全防护'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 343, 'line_end': 343, 'excerpt': '本产品属于Class 4 类激光产品，输出高功率的不可见激光，波长1080±10nm，会对直接或间接暴露在这种强激光下的眼睛或皮肤造成伤害，也可能引起现场火灾，因此请严格按照欧盟EN60825-1：2014 标准，所有操作或者靠近激光器的人员必须注意这些特殊危害，佩戴波长匹配且经过认证的激光防护眼镜，并做好充分的安全防护。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_13', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.1', 'section_title': '3.1 配置激光焊应用', 'heading_path': ['3 激光焊机器人应用配置项', '3.1 配置激光焊应用'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），配置“配置激光焊应用”时，“在超级用户权限下”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['在超级用户权限下，配置激光焊各项参数', '操作步骤：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 434, 'line_end': 434, 'excerpt': '在超级用户权限下，配置激光焊各项参数。操作步骤：', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_14', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“柜内与外部设备硬接线”中，应如何理解或操作“IO号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['功能 IO号 工艺号1 DO：201 工艺号2 DO：202 工艺号3 DO：203 工艺号4 DO：204 激光使能 DO：41 调制使能 DO：44 保护气 DO：49 摆动使能 DO：46 故障复位 DO：52 粘丝检测 DO：47 送丝使能 DO：42 送气使能 DO：43 气检 DO：49 手动送丝 DO：50 手动退丝 DO：51 就绪 DO：49 激光功率 GO:1 送丝速度 GO:2 摆动频率 GO:3 摆动宽度 GO:4 摆动高度 GO:5 旋转角度 GO:6 回抽长度 GO:7 补丝长度 GO:8 激光器就绪 DI:33 出光状态 DI:41 粘丝检测 DI:46 激光功率 GI:1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 426, 'line_end': 426, 'excerpt': '<table><tr><td><p><strong>功能</strong></p></td><td><p><strong>IO号</strong></p></td></tr><tr><td><p>工艺号1</p></td><td><p>DO：201</p></td></tr><tr><td><p>工艺号2</p></td><td><p>DO：202</p></td></tr><tr><td><p>工艺号3</p></td><td><p>DO：203</p></td></tr><tr><td><p>工艺号4</p></td><td><p>DO：204</p></td></tr><tr><td><p>激光使能</p></td><td><p>DO：41</p></td></tr><tr><td><p>调制使能</p></td><td><p>DO：44</p></td></tr><tr><td><p>保护气</p></td><td><p>DO：49</p></td></tr><tr><td><p>摆动使能</p></td><td><p>DO：46</p></td></tr><tr><td><p>故障复位</p></td><td><p>DO：52</p></td></tr><tr><td><p>粘丝检测</p></td><td><p>DO：47</p></td></tr><tr><td><p>送丝使能</p></td><td><p>DO：42</p></td></tr><tr><td><p>送气使能</p></td><td><p>DO：43</p></td></tr><tr><td><p>气检</p></td><td><p>DO：49</p></td></tr><tr><td><p>手动送丝</p></td><td><p>DO：50</p></td></tr><tr><td><p>手动退丝</p></td><td><p>DO：51</p></td></tr><tr><td><p>就绪</p></td><td><p>DO：49</p></td></tr><tr><td><p>激光功率</p></td><td><p>GO:1</p></td></tr><tr><td><p>送丝速度</p></td><td><p>GO:2</p></td></tr><tr><td><p>摆动频率</p></td><td><p>GO:3</p></td></tr><tr><td><p>摆动宽度</p></td><td><p>GO:4</p></td></tr><tr><td><p>摆动高度</p></td><td><p>GO:5</p></td></tr><tr><td><p>旋转角度</p></td><td><p>GO:6</p></td></tr><tr><td><p>回抽长度</p></td><td><p>GO:7</p></td></tr><tr><td><p>补丝长度</p></td><td><p>GO:8</p></td></tr><tr><td><p>激光器就绪</p></td><td><p>DI:33</p></td></tr><tr><td><p>出光状态</p></td><td><p>DI:41</p></td></tr><tr><td><p>粘丝检测</p></td><td><p>DI:46</p></td></tr><tr><td><p>激光功率</p></td><td><p>GI:1</p></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_15', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.6', 'section_title': '3.6 快捷键配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.6 快捷键配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），“快捷键配置”中“焊接速度100%”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['4. 焊接速度100%：开启焊接速度100%开关后，激光焊接过程中运动速度自动切换成100%，不再受示教器中的执行速率百分比影响', '该功能的有效区间在激光焊起弧指令执行开始到激光焊收弧指令执行结束', '暂停时该功能自动关闭，断点恢复时再开启'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 656, 'line_end': 656, 'excerpt': '4. 焊接速度100%：开启焊接速度100%开关后，激光焊接过程中运动速度自动切换成100%，不再受示教器中的执行速率百分比影响。该功能的有效区间在激光焊起弧指令执行开始到激光焊收弧指令执行结束。暂停时该功能自动关闭，断点恢复时再开启。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_16', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '5', 'section_title': '5 模板作业', 'heading_path': ['5 模板作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据手册“SX322004《新松机器人激光焊应用操作手册》(A-4).md”（V5.0 / A-4），激光焊模板怎样把起弧和收弧附加到LASERL运动指令上，起点和终点如何安排？', 'anchors': ['LASERL'], 'expected_facts': ['LASERL P3 VL=100.0 ACC=100 CNT=0 LASERON FILE=1 T=0.000 焊接起点', 'LASERL P4 VL=10.0 ACC=100 CNT=0 LASEROFFFILE=1 T=0.000 焊接终点'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 664, 'line_end': 664, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVJ P1 VJ=10.0 ACC=100 CNT=1</p></td><td><p>安全点</p></td></tr><tr><td><p>MOVL P2 VL=300.0 ACC=100 CNT=100</p></td><td><p>接近点</p></td></tr><tr><td><p>LASERL P3 VL=100.0 ACC=100 CNT=0 LASERON FILE=1 T=0.000</p></td><td><p>焊接起点</p></td></tr><tr><td><p>LASERL P4 VL=10.0 ACC=100 CNT=0 LASEROFFFILE=1 T=0.000</p></td><td><p>焊接终点</p></td></tr><tr><td><p>MOVL P5 VL=300.0 ACC=100 CNT=100</p></td><td><p>离开点</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_17', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.4', 'section_title': '3.4 工艺配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.4 工艺配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），操作“工艺配置”时，涉及“本次恢复无效”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['5. 本次恢复无效：ON/OFF，ON本次不进行恢复，OFF本次进行恢复', '默认为OFF，注意：执行完一次暂停再启动流程后，自动恢复为OFF'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 587, 'line_end': 587, 'excerpt': '5. 本次恢复无效：ON/OFF，ON本次不进行恢复，OFF本次进行恢复；默认为OFF，注意：执行完一次暂停再启动流程后，自动恢复为OFF；', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_18', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '3.2', 'section_title': '3.2 激光器配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.2 激光器配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“激光器配置”中，应如何理解或操作“进入激光焊应用中类型配置界面”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 进入激光焊应用中类型配置界面,激光器类型及通讯方式默认为“未设置”'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 448, 'line_end': 448, 'excerpt': '1. 进入激光焊应用中类型配置界面,激光器类型及通讯方式默认为“未设置”；', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_19', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '1.4', 'section_title': '1.4 硬件限制', 'heading_path': ['1 激光焊应用介绍', '1.4 硬件限制'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），在“硬件限制”中，应如何理解或操作“激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 351, 'line_end': 351, 'excerpt': '激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人。', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_183ed5349f_20', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-4', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'source_version': 'rc4, rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），操作“激光安全”时，涉及“标识 说明 激光辐射危险标识”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['标识 说明 激光辐射危险标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下', '激光输出警示标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下', '4类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'line_start': 347, 'line_end': 347, 'excerpt': '<table><tr><td><p><strong>标识</strong></p></td><td><p><strong>说明</strong></p></td></tr><tr><td><p><img alt="u=1822707928,265314136&amp;fm=26&amp;gp=0" src="img/1e87490d6d18be502bb3c04e54057a37bbbc92e5810651ba9687819def2a932c.png"/> </p></td><td><p>激光辐射危险标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/8006beca76c53998a9d8a4024f85f28dc22b78ea6ddcc74d83f099db1fb36150.png"/></p></td><td><p>激光输出警示标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/1606e5db934df3e58543902e0fa4aab2f3dd4253edffeb63956d6b57286c77c1.png"/></p></td><td><p>4类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/32e4c1942955f7a6443f7f7e86a135d4340dc809fa45bee37b2daad71110201a.png"/></p></td><td><p>2M类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img alt="IMG_256" src="img/def232dd9f64151bfa9b9756fe2366143c0bf03fe7414f751ee7832b31e1b095.jpg"/></p></td><td><p>当心触电标识，使用前确认用电需求，注意用电安全。</p></td></tr><tr><td><p> <img src="img/9eba2e52055b971ee021552181e7118f7c56df5c9b0bda3c2828cad11a00d63b.png"/></p></td><td><p>注意安全，具有安全风险，可能造成严重的人身伤害甚至危及生命安全。</p></td></tr></table>', 'sha256': '43958ddaafee74184c72d86c64b8a64557750ad5eab529e205da07b8a7937cd2'}]}, {'id': 'sample_laser_welding_bf1234db07_01', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“激光安全”中，应如何理解或操作“激光安全标识包括”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['激光安全标识包括：安全警示、激光输出头警示、激光辐射标识、产品铭牌等', '安全及信息标识详细说明如下表所示：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 385, 'line_end': 385, 'excerpt': '激光安全标识包括：安全警示、激光输出头警示、激光辐射标识、产品铭牌等。安全及信息标识详细说明如下表所示：', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_02', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '4', 'section_title': '4 激光焊指令说明', 'heading_path': ['4 激光焊指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），“激光焊指令说明”中“LASERWELD ON[起弧]”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['LASERWELD', 'LASERWELDON'], 'expected_facts': ['LASERWELD ON[起弧] 功能 达到目标点焊接开始', '格式 LASERWELDON [<参数1>] T = <参数2> 说明 <参数1>：工艺参数文件号', '<参数2>：时间参数，单位s'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 674, 'line_end': 674, 'excerpt': '<table><tr><td rowspan="4"><p>LASERWELD ON[起弧]</p></td><td><p>功能</p></td><td><p>达到目标点焊接开始； </p></td></tr><tr><td><p>格式</p></td><td><p>LASERWELDON [&lt;参数1&gt;] T = &lt;参数2&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p><p>&lt;参数2&gt;：时间参数，单位s；负值表示提前起弧，0与正值表示滞后起弧；</p></td></tr><tr><td><p>举例</p></td><td><p>MOVJ P[1] V = 20 ACC = 20 CNT = 0 : LASERWELDON [1] T = 1.00</p></td></tr><tr><td rowspan="4"><p>LASERWELD OFF[收弧]</p></td><td><p>功能</p></td><td><p>达到目标点焊接结束； </p></td></tr><tr><td><p>格式</p></td><td><p>LASERWELDOFF [&lt;参数1&gt;] T = &lt;参数2&gt;</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p><p>&lt;参数2&gt;：时间参数，单位s；负值表示提前起弧，0与正值表示滞后起弧；</p></td></tr><tr><td><p>举例</p></td><td><p>MOVL P[1] V = 20 ACC = 20 CNT = 0 : LASERWELDOFF [1] T = 1.00</p></td></tr><tr><td rowspan="4"><p>LASERWELD ON[起弧]</p></td><td><p>功能</p></td><td><p>焊接开始； </p></td></tr><tr><td><p>格式</p></td><td><p>LASERWELDON [&lt;参数1&gt;]</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p></td></tr><tr><td><p>举例</p></td><td><p>LASERWELDON [I[1]];</p></td></tr><tr><td rowspan="4"><p>LASERWELD OFF[收弧]</p></td><td><p>功能</p></td><td><p>焊接结束；</p></td></tr><tr><td><p>格式</p></td><td><p>LASERWELDOFF [&lt;参数1&gt;]</p></td></tr><tr><td><p>说明</p></td><td><p>&lt;参数1&gt;：工艺参数文件号；</p></td></tr><tr><td><p>举例</p></td><td><p>LASERWELDOFF [1]；</p></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_03', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.4', 'section_title': '3.4 工艺配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.4 工艺配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），操作“工艺配置”时，涉及“本次恢复无效”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['5. 本次恢复无效：ON/OFF，ON本次不进行恢复，OFF本次进行恢复', '默认为OFF，注意：执行完一次暂停再启动流程后，自动恢复为OFF'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 600, 'line_end': 600, 'excerpt': '5. 本次恢复无效：ON/OFF，ON本次不进行恢复，OFF本次进行恢复；默认为OFF，注意：执行完一次暂停再启动流程后，自动恢复为OFF；', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_04', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“柜内与外部设备硬接线”中，应如何理解或操作“选择套机后”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['选择套机后，相应IO配置自动同步，不需要手动配置'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 468, 'line_end': 468, 'excerpt': '选择套机后，相应IO配置自动同步，不需要手动配置；', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_05', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.3.2', 'section_title': '3.3.2 输入IO配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.3 通讯配置', '3.3.2 输入IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“输入IO配置”中，应如何理解或操作“当前数字量IO的状态”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['4. 当前数字量IO的状态，点击“刷新”获取实时状态，如图3-12所示'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 573, 'line_end': 573, 'excerpt': '4. 当前数字量IO的状态，点击“刷新”获取实时状态，如图3-12所示；', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_06', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.1', 'section_title': '1.1 概述', 'heading_path': ['1 激光焊应用介绍', '1.1 概述'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），配置“概述”时，“激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件，为客户提供了完备的激光焊接产品', '通过机器人全面控制关键部件，实现在示教器端设置激光控制参数，极大简化了操作流程，使用更加高效便捷'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 371, 'line_end': 371, 'excerpt': '激光焊应用产品集成了激光器、激光焊接头、送丝机、水冷机等关键部件，为客户提供了完备的激光焊接产品。通过机器人全面控制关键部件，实现在示教器端设置激光控制参数，极大简化了操作流程，使用更加高效便捷。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_07', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），操作“激光安全”时，涉及“本产品属于Class 4 类激光产品”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['本产品属于Class 4 类激光产品，输出高功率的不可见激光，波长1080±10nm，会对直接或间接暴露在这种强激光下的眼睛或皮肤造成伤害，也可能引起现场火灾，因此请严格按照欧盟EN60825-1：2014 标准，所有操作或者靠近激光器的人员必须注意这些特殊危害，佩戴波长匹配且经过认证的激光防护眼镜，并做好充分的安全防护'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 383, 'line_end': 383, 'excerpt': '本产品属于Class 4 类激光产品，输出高功率的不可见激光，波长1080±10nm，会对直接或间接暴露在这种强激光下的眼睛或皮肤造成伤害，也可能引起现场火灾，因此请严格按照欧盟EN60825-1：2014 标准，所有操作或者靠近激光器的人员必须注意这些特殊危害，佩戴波长匹配且经过认证的激光防护眼镜，并做好充分的安全防护。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_08', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '5', 'section_title': '5 模板作业', 'heading_path': ['5 模板作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据手册“SX322004《新松机器人激光焊应用操作手册》(C-0).md”（V5.0 / C-0），激光焊模板怎样在运动语句上附加起弧和收弧动作，所示时间参数分别是什么？', 'anchors': [':LASERWELDON', ':LASERWELDOFF'], 'expected_facts': ['MOVJP[3]V=20ACC=20CNT=0:LASERWELDON[1]T =1.00 焊接起点', 'MOVLP[4]V=20ACC=20CNT=0:LASERWELDOFF[1]T =-1.00 焊接终点'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 678, 'line_end': 678, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVJ P[1] V=10.0 ACC=100 CNT=1</p></td><td><p>安全点</p></td></tr><tr><td><p>MOVL P[2] V=300.0 ACC=100 CNT=100</p></td><td><p>接近点</p></td></tr><tr><td><p>MOVJP[3]V=20ACC=20CNT=0:LASERWELDON[1]T =1.00</p></td><td><p>焊接起点</p></td></tr><tr><td><p>MOVLP[4]V=20ACC=20CNT=0:LASERWELDOFF[1]T =-1.00</p></td><td><p>焊接终点</p></td></tr><tr><td><p>MOVL P[5] VL=300.0 ACC=100 CNT=100</p></td><td><p>离开点</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_09', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），配置“柜内与外部设备硬接线”时，“表格 2-1套机默认信号参数表”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['表格 2-1套机默认信号参数表'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 464, 'line_end': 464, 'excerpt': '表格 2-1套机默认信号参数表', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_10', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），配置“文件配置”时，“延迟停丝时间”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['20. 延迟停丝时间：提前停丝，从开始停丝到光开始关闭时间，提前停丝时间大于关光点到上一运动点时间，停丝从关光上一运动点动作，范围-10.000-10.000s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 641, 'line_end': 641, 'excerpt': '20. 延迟停丝时间：提前停丝，从开始停丝到光开始关闭时间，提前停丝时间大于关光点到上一运动点时间，停丝从关光上一运动点动作，范围-10.000-10.000s。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_11', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），“文件配置”中“缓降时间”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['18. 缓降时间：从指令开始关闭激光位置到结束功率时间，范围0.000-10.000s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 639, 'line_end': 639, 'excerpt': '18. 缓降时间：从指令开始关闭激光位置到结束功率时间，范围0.000-10.000s。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_12', 'category': 'laser_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '6', 'section_title': '6 报警信息', 'heading_path': ['6 报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），“报警信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警码 报警信息 处理方法 40001 焊接允许关闭 提示类无需处理 40002 焊接允许开启 40003 本次工艺恢复无效开关关闭 40004 本次工艺恢复无效开关开启 40005 工艺恢复开关关闭 40006 工艺恢复开关开启 40007 清空激光焊堆栈 40008 焊接允许已关闭 40009 焊枪碰撞已关闭 40010 单步运行不执行起收弧工艺 40011 不恢复工艺 40012 本次工艺恢复无效，开关已复位 40013 焊机品牌/型号更改后重新启动生效 40051 激光器状态异常 检查激光器状态输入IO 40052 冷却水异常 检查冷却水状态输入IO 40053 保护气异常 检查保护气状态输入IO 40054 工艺号未配置 在输出IO工艺号配置中配置相应的工艺号 40055 文件号超界 指令调用工艺文件号超界 40056 激光异常超时 焊接时机器人不能停止，请检查 40057 送丝速度发送失败 检查参数是否超界或模拟量模块配置 40058 功率发送失败 检查参数是否超界或模拟量模块配置 40059 功率反馈异常 检查模拟量模块配置 40060 摆动频率发送失败 检查参数是否超界或模拟量模块配置 40061 摆动宽度发送失败 检查参数是否超界或模拟量模块配置 40062 起弧失败 检查设备异常 40063 粘丝 解除粘丝 40064 焊枪碰撞 解除焊枪碰撞 40065 摆动高度发送失败 检查参数是否超界或模拟量模块配置 40066 旋转角度发送失败 检查参数是否超界或模拟量模块配置 40067 回抽长度发送失败 检查参数是否超界或模拟量模块配置 40068 补丝长度发送失败 检查参数是否超界或模拟量模块配置 40069 组IO发送错误 请检查自定义组IO配置 40070 组IO接收错误 请检查自定义组IO配置'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 684, 'line_end': 684, 'excerpt': '<table><tr><td><p><strong>报警码</strong></p></td><td><p><strong>报警信息 </strong></p></td><td><p><strong>处理方法</strong></p></td></tr><tr><td><p>40001</p></td><td><p>焊接允许关闭 </p></td><td><p>提示类无需处理</p></td></tr><tr><td><p>40002</p></td><td><p>焊接允许开启 </p></td><td></td></tr><tr><td><p>40003</p></td><td><p>本次工艺恢复无效开关关闭</p></td><td></td></tr><tr><td><p>40004</p></td><td><p>本次工艺恢复无效开关开启 </p></td><td></td></tr><tr><td><p>40005</p></td><td><p>工艺恢复开关关闭</p></td><td></td></tr><tr><td><p>40006</p></td><td><p>工艺恢复开关开启</p></td><td></td></tr><tr><td><p>40007</p></td><td><p>清空激光焊堆栈</p></td><td></td></tr><tr><td><p>40008</p></td><td><p>焊接允许已关闭</p></td><td></td></tr><tr><td><p>40009</p></td><td><p>焊枪碰撞已关闭</p></td><td></td></tr><tr><td><p>40010</p></td><td><p>单步运行不执行起收弧工艺</p></td><td></td></tr><tr><td><p>40011</p></td><td><p>不恢复工艺 </p></td><td></td></tr><tr><td><p>40012</p></td><td><p>本次工艺恢复无效，开关已复位</p></td><td></td></tr><tr><td><p>40013</p></td><td><p>焊机品牌/型号更改后重新启动生效</p></td><td></td></tr><tr><td><p>40051</p></td><td><p>激光器状态异常</p></td><td><p>检查激光器状态输入IO</p></td></tr><tr><td><p>40052</p></td><td><p>冷却水异常</p></td><td><p>检查冷却水状态输入IO</p></td></tr><tr><td><p>40053</p></td><td><p>保护气异常</p></td><td><p>检查保护气状态输入IO </p></td></tr><tr><td><p>40054</p></td><td><p>工艺号未配置</p></td><td><p>在输出IO工艺号配置中配置相应的工艺号</p></td></tr><tr><td><p>40055</p></td><td><p>文件号超界</p></td><td><p>指令调用工艺文件号超界</p></td></tr><tr><td><p>40056</p></td><td><p>激光异常超时</p></td><td><p>焊接时机器人不能停止，请检查 </p></td></tr><tr><td><p>40057</p></td><td><p>送丝速度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置 </p></td></tr><tr><td><p>40058</p></td><td><p>功率发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40059</p></td><td><p>功率反馈异常</p></td><td><p>检查模拟量模块配置</p></td></tr><tr><td><p>40060</p></td><td><p>摆动频率发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40061</p></td><td><p>摆动宽度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40062</p></td><td><p>起弧失败</p></td><td><p>检查设备异常</p></td></tr><tr><td><p>40063</p></td><td><p>粘丝</p></td><td><p>解除粘丝</p></td></tr><tr><td><p>40064</p></td><td><p>焊枪碰撞</p></td><td><p>解除焊枪碰撞</p></td></tr><tr><td><p>40065</p></td><td><p>摆动高度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置 </p></td></tr><tr><td><p>40066</p></td><td><p>旋转角度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40067</p></td><td><p>回抽长度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40068</p></td><td><p>补丝长度发送失败</p></td><td><p>检查参数是否超界或模拟量模块配置</p></td></tr><tr><td><p>40069</p></td><td><p>组IO发送错误</p></td><td><p>请检查自定义组IO配置 </p></td></tr><tr><td><p>40070</p></td><td><p>组IO接收错误</p></td><td><p>请检查自定义组IO配置 </p></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_13', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.4', 'section_title': '1.4 硬件限制', 'heading_path': ['1 激光焊应用介绍', '1.4 硬件限制'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“硬件限制”中，应如何理解或操作“激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 391, 'line_end': 391, 'excerpt': '激光焊应用适配本体型号为SR25A-25/1.80、SR12A-12/1.46的机器人。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_14', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），操作“文件配置”时，涉及“工艺号”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['9. 工艺号：不同工艺号调用激光头内不同摆动形式，工艺号0可不配置工艺号发送给激光器IO都为0，其余工艺号必须配置至少发送一个IO=1，否则报警工艺号为空，范围0-15'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 630, 'line_end': 630, 'excerpt': '9. 工艺号：不同工艺号调用激光头内不同摆动形式，工艺号0可不配置工艺号发送给激光器IO都为0，其余工艺号必须配置至少发送一个IO=1，否则报警工艺号为空，范围0-15。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_15', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '2.2', 'section_title': '2.2 柜内与外部设备硬接线', 'heading_path': ['2 套机安装', '2.2 柜内与外部设备硬接线'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“柜内与外部设备硬接线”中，应如何理解或操作“IO号”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['功能 IO号 工艺号1 DO：201 工艺号2 DO：202 工艺号3 DO：203 工艺号4 DO：204 激光使能 DO：41 调制使能 DO：44 保护气 DO：49 摆动使能 DO：46 故障复位 DO：52 粘丝检测 DO：47 送丝使能 DO：42 送气使能 DO：43 气检 DO：49 手动送丝 DO：50 手动退丝 DO：51 就绪 DO：49 激光功率 GO:1 送丝速度 GO:2 摆动频率 GO:3 摆动宽度 GO:4 摆动高度 GO:5 旋转角度 GO:6 回抽长度 GO:7 补丝长度 GO:8 激光器就绪 DI:33 出光状态 DI:41 粘丝检测 DI:46 激光功率 GI:1'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 466, 'line_end': 466, 'excerpt': '<table><tr><td><p><strong>功能</strong></p></td><td><p><strong>IO号</strong></p></td></tr><tr><td><p>工艺号1</p></td><td><p>DO：201</p></td></tr><tr><td><p>工艺号2</p></td><td><p>DO：202</p></td></tr><tr><td><p>工艺号3</p></td><td><p>DO：203</p></td></tr><tr><td><p>工艺号4</p></td><td><p>DO：204</p></td></tr><tr><td><p>激光使能</p></td><td><p>DO：41</p></td></tr><tr><td><p>调制使能</p></td><td><p>DO：44</p></td></tr><tr><td><p>保护气</p></td><td><p>DO：49</p></td></tr><tr><td><p>摆动使能</p></td><td><p>DO：46</p></td></tr><tr><td><p>故障复位</p></td><td><p>DO：52</p></td></tr><tr><td><p>粘丝检测</p></td><td><p>DO：47</p></td></tr><tr><td><p>送丝使能</p></td><td><p>DO：42</p></td></tr><tr><td><p>送气使能</p></td><td><p>DO：43</p></td></tr><tr><td><p>气检</p></td><td><p>DO：49</p></td></tr><tr><td><p>手动送丝</p></td><td><p>DO：50</p></td></tr><tr><td><p>手动退丝</p></td><td><p>DO：51</p></td></tr><tr><td><p>就绪</p></td><td><p>DO：49</p></td></tr><tr><td><p>激光功率</p></td><td><p>GO:1</p></td></tr><tr><td><p>送丝速度</p></td><td><p>GO:2</p></td></tr><tr><td><p>摆动频率</p></td><td><p>GO:3</p></td></tr><tr><td><p>摆动宽度</p></td><td><p>GO:4</p></td></tr><tr><td><p>摆动高度</p></td><td><p>GO:5</p></td></tr><tr><td><p>旋转角度</p></td><td><p>GO:6</p></td></tr><tr><td><p>回抽长度</p></td><td><p>GO:7</p></td></tr><tr><td><p>补丝长度</p></td><td><p>GO:8</p></td></tr><tr><td><p>激光器就绪</p></td><td><p>DI:33</p></td></tr><tr><td><p>出光状态</p></td><td><p>DI:41</p></td></tr><tr><td><p>粘丝检测</p></td><td><p>DI:46</p></td></tr><tr><td><p>激光功率</p></td><td><p>GI:1</p></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_16', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.3.1', 'section_title': '3.3.1 输出IO配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.3 通讯配置', '3.3.1 输出IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“输出IO配置”中，应如何理解或操作“3”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['3：D01 √ DO2 √'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 544, 'line_end': 544, 'excerpt': '3：D01 √ DO2 √', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_17', 'category': 'laser_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.3', 'section_title': '1.3 激光安全', 'heading_path': ['1 激光焊应用介绍', '1.3 激光安全'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），操作“激光安全”时，涉及“标识 说明 激光辐射危险标识”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['标识 说明 激光辐射危险标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下', '激光输出警示标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下', '4类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 387, 'line_end': 387, 'excerpt': '<table><tr><td><p><strong>标识</strong></p></td><td><p><strong>说明</strong></p></td></tr><tr><td><p><img alt="u=1822707928,265314136&amp;fm=26&amp;gp=0" src="img/1e87490d6d18be502bb3c04e54057a37bbbc92e5810651ba9687819def2a932c.png"/> </p></td><td><p>激光辐射危险标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/8006beca76c53998a9d8a4024f85f28dc22b78ea6ddcc74d83f099db1fb36150.png"/></p></td><td><p>激光输出警示标识，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/1606e5db934df3e58543902e0fa4aab2f3dd4253edffeb63956d6b57286c77c1.png"/></p></td><td><p>4类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img src="img/32e4c1942955f7a6443f7f7e86a135d4340dc809fa45bee37b2daad71110201a.png"/></p></td><td><p>2M类激光产品，避免人眼直射，请佩戴激光防护眼镜，严禁将身体任何部分暴露激光之下。</p></td></tr><tr><td><p><img alt="IMG_256" src="img/def232dd9f64151bfa9b9756fe2366143c0bf03fe7414f751ee7832b31e1b095.jpg"/></p></td><td><p>当心触电标识，使用前确认用电需求，注意用电安全。</p></td></tr><tr><td><p> <img src="img/9eba2e52055b971ee021552181e7118f7c56df5c9b0bda3c2828cad11a00d63b.png"/></p></td><td><p>注意安全，具有安全风险，可能造成严重的人身伤害甚至危及生命安全。</p></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_18', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '5', 'section_title': '5 模板作业', 'heading_path': ['5 模板作业'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据手册“SX322004《新松机器人激光焊应用操作手册》(C-0).md”（V5.0 / C-0），激光焊模板中怎样用变量选择焊接文件并分别执行起弧和收弧，二者之间有哪些运动？', 'anchors': ['LASERWELDON', 'LASERWELDOFF', 'MOVL P4'], 'expected_facts': ['LASERWELDON [I[1]] 起弧', 'MOVL P4 VL=10.0 ACC=100 CNT=0 焊接终点', 'LASERWELDOFF [I[1]] 收弧'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 680, 'line_end': 680, 'excerpt': '<table><tr><td><p>NOP</p></td><td></td></tr><tr><td><p>MOVJ P[1] VJ=10.0 ACC=100 CNT=1</p></td><td><p>安全点</p></td></tr><tr><td><p>MOVL P[2] VL=300.0 ACC=100 CNT=100</p></td><td><p>接近点</p></td></tr><tr><td><p>MOVL P[3] VL=100.0 ACC=100 CNT=0</p></td><td><p>焊接起点</p></td></tr><tr><td><p>LASERWELDON [I[1]]</p></td><td><p>起弧</p></td></tr><tr><td><p>MOVL P4 VL=10.0 ACC=100 CNT=0</p></td><td><p>焊接终点</p></td></tr><tr><td><p>LASERWELDOFF [I[1]]</p></td><td><p>收弧</p></td></tr><tr><td><p>MOVL P5 VL=300.0 ACC=100 CNT=100</p></td><td><p>离开点</p></td></tr><tr><td><p>END</p></td><td></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_19', 'category': 'laser_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '3.5', 'section_title': '3.5 文件配置', 'heading_path': ['3 激光焊机器人应用配置项', '3.5 文件配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），配置“文件配置”时，“补丝时间”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['23. 补丝时间：停丝结束后开始补丝，范围0.000-10.000s'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 644, 'line_end': 644, 'excerpt': '23. 补丝时间：停丝结束后开始补丝，范围0.000-10.000s。', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}, {'id': 'sample_laser_welding_bf1234db07_20', 'category': 'laser_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / C-0', 'section': '1.2', 'section_title': '1.2 物理组件', 'heading_path': ['1 激光焊应用介绍', '1.2 物理组件'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'source_version': 'rc5', 'question': '根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“物理组件”中，应如何理解或操作“序号 名称 A 机器人本体(SR25A-25/1.80、SR12A-12/1.46) B 激光焊”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['序号 名称 A 机器人本体(SR25A-25/1.80、SR12A-12/1.46) B 激光焊接头 C 激光器及水冷一体机 D 送丝机'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'line_start': 379, 'line_end': 379, 'excerpt': '<table><tr><td><p><strong>序号</strong></p></td><td><p><strong>名称</strong></p></td></tr><tr><td><p>A</p></td><td><p>机器人本体(SR25A-25/1.80、SR12A-12/1.46)</p></td></tr><tr><td><p>B</p></td><td><p>激光焊接头</p></td></tr><tr><td><p>C</p></td><td><p>激光器及水冷一体机</p></td></tr><tr><td><p>D</p></td><td><p>送丝机</p></td></tr></table>', 'sha256': '2e2035cc367fa70817f46b1a4f39315af4f584e0342561e753e67c61f13353e7'}]}]
register_sample_bank('laser_welding', LASER_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md', 'seed': 13866031721301861584, 'population': 99, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(593, 593), (331, 331), (418, 418), (424, 424), (619, 619), (660, 660), (666, 666), (610, 610), (634, 634), (670, 670), (345, 345), (343, 343), (434, 434), (426, 426), (656, 656), (664, 664), (587, 587), (448, 448), (351, 351), (347, 347)]}, {'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md', 'seed': 2594229048591998329, 'population': 96, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(385, 385), (674, 674), (600, 600), (468, 468), (573, 573), (371, 371), (383, 383), (678, 678), (464, 464), (641, 641), (639, 639), (684, 684), (391, 391), (630, 630), (466, 466), (544, 544), (387, 387), (680, 680), (644, 644), (379, 379)]}])


In [ ]:
# | notest
LASER_WELDING_SAMPLE_IDS = None
laser_welding_sample_cases = select_cases(LASER_WELDING_SAMPLE_CASES, LASER_WELDING_SAMPLE_IDS)
display(pd.DataFrame(laser_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in laser_welding_sample_cases])
laser_welding_sample_metrics = await evaluation.run(laser_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
laser_welding_sample_results = {case["id"]: evaluation.load(case) for case in laser_welding_sample_cases}
display(laser_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(laser_welding_sample_cases[0]["id"])
evaluation.show_answers(laser_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_laser_welding_183ed5349f_01,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,parameter,prose,True,True,8,True,6,5,2,True,False,False,None,None,8983,0,True,unsupported,,,incorrect
sample_laser_welding_183ed5349f_02,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,parameter,prose,True,True,8,True,6,6,1,True,False,False,None,None,8647,0,True,unsupported,,,not reviewed
sample_laser_welding_183ed5349f_03,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,procedure,prose,True,True,8,True,7,7,0,True,False,False,None,None,7466,0,True,unsupported,,,abstained
sample_laser_welding_183ed5349f_04,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,parameter,prose,True,True,8,True,3,3,4,True,False,False,None,None,9470,0,True,unsupported,,,not reviewed
sample_laser_welding_183ed5349f_05,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,instruction,prose,True,True,8,True,7,7,0,True,False,False,None,None,7503,0,True,unsupported,,,not reviewed
sample_laser_welding_183ed5349f_06,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,instruction,table_row,True,True,8,True,3,3,0,True,True,True,False,True,10928,0,True,included,,,not reviewed
sample_laser_welding_183ed5349f_07,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,procedure,table_row,True,True,8,True,4,4,0,True,False,False,False,False,11175,0,False,unavailable,,Response validation failed: Expecting value: line 1 column 1 (char 0),not reviewed
sample_laser_welding_183ed5349f_08,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,prerequisite,prose,True,True,8,True,7,6,0,True,False,False,None,None,7289,0,True,unsupported,,,not reviewed
sample_laser_welding_183ed5349f_09,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,procedure,prose,True,True,8,True,7,7,0,True,False,False,None,None,7294,0,True,unsupported,,,not reviewed
sample_laser_welding_183ed5349f_10,laser_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md,V5.0 / A-4,instruction,table_row,True,True,8,True,2,2,5,True,False,False,False,False,22649,3,True,unsupported,,,not reviewed


### sample_laser_welding_183ed5349f_01

根据《SX322004《新松机器人激光焊应用操作手册》(A-4)》（V5.0 / A-4），配置“文件配置”时，“激光焊应用中点击【文件配置】进入工艺参数配置界面”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。

#### Retrieved content summary

- V1.1（2024-07-17，沙景超）：完善输入/输出IO配置；新增工艺配置、文件配置及激光焊指令说明。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 42–47; corpus 4.0, revision V5.0 / A-4; object `d905d2588a97ce16b3b5b2b191552fa2243f8ed6306232045b6c4e8333c3864c`.

- 新松机器人激光焊应用操作手册（SX322004，V5.0）第8行数据：功能“以示教器开始程序”，第2列无内容，第3列和第4列均标记为“〇”（通常表示不适用或不支持该功能）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 266–271; corpus 4.0, revision V5.0 / A-4; object `e75e17da7c077755c1766321c1296f5f541b4fe845c18db1ffa801919ac92bd4`.

- 版本 V1.1（2024-07-17，沙景超）：完善 I/O 配置；新增工艺、文件配置及激光焊指令说明。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 42–47; corpus 5.0, revision V5.0 / A-4; object `273cf480c8b60e4751e74b4576c4cfbf13139a6f971723ad8e4e8e59c94dbb69`.

- 文档《新松机器人激光焊应用操作手册》(V5.0) 在版本 A/3 (2025-03-27, 沙景超) 中，新增了套机安装接线、设备安全、参数配置及报警信息介绍。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 54–59; corpus 5.0, revision V5.0 / A-4; object `23d0c48dc993afd1bb27d4d923385a41e3913a98404db556ae1250b48423b0e5`.

- 该表格行位于“新松机器人激光焊应用操作手册”文档中，具体内容为第2行（非表头），包含单个单元格，其文本为“新松机器人激光焊应用操作手册”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 11–13; corpus 5.0, revision V5.0 / A-4; object `4da76ad5000db4076aaf95dc78f44e21542900094d42c0337caba047c412f3fa`.

- 文档《新松机器人激光焊应用操作手册》版本 A/4（2025年7月14日，修订人：李同）：主要更新为“增加独立的起收弧指令”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 60–65; corpus 4.0, revision V5.0 / A-4; object `4699d146fda77b94e8138075b957b869571f68e0aa7ba215a455ac64526ebbc5`.

- 新松机器人弧焊操作手册中，第2项参数配置为：38, 0, 50, 50, 0
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 2926–2933; corpus 4.0, revision V5.0 / B-5; object `279024ee64c21ad86a5e0654508dbc1a94977512bd57dc1e28b7b0ab6355798d`.

- 新松机器人弧焊应用操作手册（第5行）参数记录：[4, 2, 0, 3, 50, 3, -, 150, -]，对应子项“e）”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 10503–10514; corpus 4.0, revision V5.0 / B-5; object `9be5747fbd75d6505c3ffb3a4380fe5d8e79016ef5bcb0820d3bd281ffb1f236`.

#### Answer

根据检索到的手册证据，E1（V5.0/A-4，lines 42-47）仅在修订记录中提及：V1.1 版本

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: unsupported; revision fidelity: mixed.

The answer fails to state the requested fact. The supplied evidence and references explicitly contain the sentence '激光焊应用中点击【文件配置】进入工艺参数配置界面', which matches the expected fact. Instead, the answer discusses revision history (V1.1) found in the parent table context of E1, which is not the requested information about the 'File Configuration' interface parameters. The answer does not address the specific question about what parameters are configured or their meanings/units, nor does it confirm the action described in the expected fact. It is not a valid abstention because the specific fact was present in the provided references/evidence context.

...

#### Answer

在所提供的检索证据中，没有任何一条明确列出

### sample_laser_welding_bf1234db07_20

根据《SX322004《新松机器人激光焊应用操作手册》(C-0)》（V5.0 / C-0），在“物理组件”中，应如何理解或操作“序号 名称 A 机器人本体(SR25A-25/1.80、SR12A-12/1.46) B 激光焊”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 新松机器人激光焊应用操作手册（SX322004，V5.0）第8行数据：功能“以示教器开始程序”，第2列无内容，第3列和第4列均标记为“〇”（通常表示不适用或不支持该功能）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 266–271; corpus 4.0, revision V5.0 / A-4; object `e75e17da7c077755c1766321c1296f5f541b4fe845c18db1ffa801919ac92bd4`.

- 新松机器人通用操作手册（文档编号：SX322001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 480–483; corpus 5.0, revision V5.0 / C-0; object `7723af1da440d3260a499cef189d99aef7de63c3cbfdace5cad603c336881343`.

- 新松机器人通用操作手册，编号 SX322001
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 483–486; corpus 4.0, revision V5.0 / A-4; object `0fa0fe1f9248cd8de37872f90e54ee8f642cc7bf54636b2e8e456a5202b63209`.

- 文档《新松机器人激光焊应用操作手册》(V5.0) 在版本 A/3 (2025-03-27, 沙景超) 中，新增了套机安装接线、设备安全、参数配置及报警信息介绍。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 54–59; corpus 5.0, revision V5.0 / A-4; object `23d0c48dc993afd1bb27d4d923385a41e3913a98404db556ae1250b48423b0e5`.

- 新松机器人电气使用及维护手册（编号：SX222001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 471–474; corpus 5.0, revision V5.0 / A-4; object `fdcdec21783790842921cad21c2fa4c96cf6bfc9601f88bda515c86a385aca2a`.

- 新松机器人电气使用及维护手册（文档编号：SX222001）
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 471–474; corpus 4.0, revision V5.0 / A-4; object `eee66b6ae1e6ecdc737fbaa1187c604da9b09ca1258f9b9be79c97d601990c80`.

- 该表包含“序号”和“名称”两列，列出以下组件映射关系：
- **A**：机器人本体（型号 SR25A-25/1.80、SR12A-12/1.46）
- **B**：激光焊接头
- **C**：激光器及水冷一体机
- **D**：送丝机
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(A-4)/SX322004《新松机器人激光焊应用操作手册》(A-4).md`, lines 551–572; corpus 5.0, revision V5.0 / A-4; object `0f0a002ceaa6aa7ecdcbf438b677f1f91b6a62ed642ddf05e18c6206a8da55fc`.

- 文档名称：新松 SR25 系列工业机器人安装及维护手册；对应文档编号：SR122002。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人激光焊应用操作手册》(C-0)/SX322004《新松机器人激光焊应用操作手册》(C-0).md`, lines 476–479; corpus 5.0, revision V5.0 / C-0; object `9e7849bb14419a47f477cc3cdb48092117809aa242beeed8a6ef53df9caafb0c`.

#### Answer

根据检索到的手册证据，E7（V5.0 / A-4 修订版）中

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: unsupported; revision fidelity: mixed.

The answer is a fragment that does not answer the question. It fails to explain the operation, purpose, or result of the physical components listed. It incorrectly cites evidence E7 (which is from revision A-4) instead of the requested revision C-0, although the table content is identical in the supplied evidence. The answer does not list the expected facts (A, B, C, D components) nor provide any explanation.

## Pneumatic Spot Welding — sampled manual revisions

20 cases from 1 CSV files. Set `PNEUMATIC_SPOT_WELDING_SAMPLE_IDS` to select case IDs; `None` selects all revisions. Run the batch cell explicitly to retrieve and answer; compatible completed results resume.

In [ ]:
# | notest
PNEUMATIC_SPOT_WELDING_SAMPLE_CASES = [{'id': 'sample_pneumatic_spot_welding_c2b8069b1b_01', 'category': 'pneumatic_spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4', 'section_title': '1.4 焊接流程', 'heading_path': ['1 气动点焊单元', '1.4 焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“焊接流程”中，应如何理解或操作“判断当前焊机是否有报警”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['判断当前焊机是否有报警：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 123, 'line_end': 123, 'excerpt': '判断当前焊机是否有报警：', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_02', 'category': 'pneumatic_spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.6', 'section_title': '1.6 焊钳零位标定', 'heading_path': ['1 气动点焊单元', '1.6 焊钳零位标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），操作“焊钳零位标定”时，涉及“在使用气动点焊功能前必须要标定相关工具坐标以及焊钳零位”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['在使用气动点焊功能前必须要标定相关工具坐标以及焊钳零位，如图所示，把移动侧电极的方向设置为工具坐标系Z+方向'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 170, 'line_end': 170, 'excerpt': '在使用气动点焊功能前必须要标定相关工具坐标以及焊钳零位，如图所示，把移动侧电极的方向设置为工具坐标系Z+方向。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_03', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3.1', 'section_title': '3.1 气动点焊运动指令说明', 'heading_path': ['3 气动点焊指令', '3.1 气动点焊运动指令说明'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“气动点焊运动指令说明”中“AIRSPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 P”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['AIRSPOTL', 'ACC'], 'expected_facts': ['4. AIRSPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 PRS=1 TWC=0'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 644, 'line_end': 644, 'excerpt': '4. AIRSPOTL P[1] VL=100 ACC=100 CNT=0 DRESS GUN=1 PRS=1 TWC=0；', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_04', 'category': 'pneumatic_spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.11', 'section_title': '2.11 手动点焊配置', 'heading_path': ['2 气动点焊机器人应用配置项', '2.11 手动点焊配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“手动点焊配置”中，应如何理解或操作“手动焊接”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['手动焊接：用于同步快捷键OP5中的焊接功能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 547, 'line_end': 547, 'excerpt': '**手动焊接：用于同步快捷键OP5中的焊接功能。**', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_05', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.4', 'section_title': '2.4 焊钳特性配置', 'heading_path': ['2 气动点焊机器人应用配置项', '2.4 焊钳特性配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“焊钳特性配置”中“大开加压延时时间（s）”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['OFF'], 'expected_facts': ['4. 大开加压延时时间（s）：“阀压力检测”开关OFF时， 该参数有效', '执行焊接和修磨指令，焊钳从大开位置开始，延时该时间后，启动焊接流程'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 397, 'line_end': 397, 'excerpt': '4. 大开加压延时时间（s）：“阀压力检测”开关OFF时， 该参数有效。执行焊接和修磨指令，焊钳从大开位置开始，延时该时间后，启动焊接流程。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_06', 'category': 'pneumatic_spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3.1.1', 'section_title': '3.1.1 AIRSPOTJ(关节运动点焊)', 'heading_path': ['3 气动点焊指令', '3.1 气动点焊运动指令说明', '3.1.1 AIRSPOTJ(关节运动点焊)'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），操作“AIRSPOTJ(关节运动点焊)”时，涉及“AIRSPOTJ P[1] VJ=10 ACC=100 CNT=0 WELD GUN=1 PRS”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['AIRSPOTJ P[1] VJ=10 ACC=100 CNT=0 WELD GUN=1 PRS=1 PROGNo=123456789 GLF=1', '功能 机器人携带焊钳走关节运动，以指定的焊钳行程、压力、焊接条件执行焊接任务', '说明 参数项 1 位置变量号 参数项 2 运行速度 参数项 3 加速度 参数项 4 逼近等级（0或1） 参数项 6 焊钳特性编号'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 652, 'line_end': 652, 'excerpt': '<table><tr><td colspan="3"><p><a id="_Hlk198110084"></a>AIRSPOTJ P[1] VJ=10 ACC=100 CNT=0 WELD GUN=1 PRS=1 PROGNo=123456789 GLF=1；</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>机器人携带焊钳走关节运动，以指定的焊钳行程、压力、焊接条件执行焊接任务。</p></td></tr><tr><td rowspan="8"><p>说明</p></td><td><p>参数项 1</p></td><td><p>位置变量号</p></td></tr><tr><td><p>参数项 2</p></td><td><p>运行速度</p></td></tr><tr><td><p>参数项 3</p></td><td><p>加速度</p></td></tr><tr><td><p>参数项 4</p></td><td><p>逼近等级（0或1）</p></td></tr><tr><td><p>参数项 6</p></td><td><p>焊钳特性编号。需要跟当前焊钳号一致</p></td></tr><tr><td><p>参数项 7</p></td><td><p>焊接压力文件号（1到32）</p></td></tr><tr><td><p>参数项 8</p></td><td><p>焊机通讯的焊接文件号</p></td></tr><tr><td><p>参数项 9</p></td><td><p>间隙参数（1到4），即焊钳在焊接前后的行程位置。</p><p>1：小开-小开（焊接前小开，焊接后回到小开）</p><p>2：大开-大开（焊接前大开，焊接后回到大开）</p><p>3：大开-小开（焊接前大开，焊接后回到小开）</p><p>4：小开-大开（焊接前小开，焊接后回到小开）</p></td></tr></table>', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_07', 'category': 'pneumatic_spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.3.2', 'section_title': '2.3.2 外设IO配置', 'heading_path': ['2 气动点焊机器人应用配置项', '2.3 设备交互IO配置', '2.3.2 外设IO配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），操作“外设IO配置”时，涉及“注意”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['注意：气压报警只能手动复位报警'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 280, 'line_end': 280, 'excerpt': '**注意：气压报警只能手动复位报警。**', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_08', 'category': 'pneumatic_spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 气动点焊快捷键', 'heading_path': ['4 气动点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），操作“气动点焊快捷键”时，涉及“按OP1”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['按OP1，调出快捷键界面，选择气动点焊应用，系统已默认配置好OP2~OP6的快捷键功能，如图4-1', '点击相应按钮后，可以实现对应功能', '注意：快捷键必须先调出快捷键应用界面才有效'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 682, 'line_end': 682, 'excerpt': '按OP1，调出快捷键界面，选择气动点焊应用，系统已默认配置好OP2~OP6的快捷键功能，如图4-1。点击相应按钮后，可以实现对应功能。注意：快捷键必须先调出快捷键应用界面才有效。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_09', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 气动点焊快捷键', 'heading_path': ['4 气动点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“气动点焊快捷键”中“SIM停留时间”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['SIM'], 'expected_facts': ['6. SIM停留时间：“工艺关-仿真”模式下，执行焊接指令，不执行加压和焊接，根据仿真停留时间设置，进行延时'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 719, 'line_end': 719, 'excerpt': '6. **SIM停留时间：**“工艺关-仿真”模式下，执行焊接指令，不执行加压和焊接，根据仿真停留时间设置，进行延时。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_10', 'category': 'pneumatic_spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '4', 'section_title': '4 气动点焊快捷键', 'heading_path': ['4 气动点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“气动点焊快捷键”中，应如何理解或操作“工艺开”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['5. 工艺开：固定配置到OP6键，按键时在“工艺开”(NORMAL)、“工艺关-仿真”(SIM)、“工艺关-干运行”(DRYRUN)三种工作模式间切换'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 706, 'line_end': 706, 'excerpt': '5. **工艺开**：固定配置到OP6键，按键时在“工艺开”(NORMAL)、“工艺关-仿真”(SIM)、“工艺关-干运行”(DRYRUN)三种工作模式间切换。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_11', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.2', 'section_title': '1.2 基本配置流程', 'heading_path': ['1 气动点焊单元', '1.2 基本配置流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“基本配置流程”中“在使用气动点焊指令进行点焊作业或手动试焊之前需进行相关参数配置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['在使用气动点焊指令进行点焊作业或手动试焊之前需进行相关参数配置', '参数配置有一定的先后顺序，其基本配置流程见下图：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 101, 'line_end': 101, 'excerpt': '在使用气动点焊指令进行点焊作业或手动试焊之前需进行相关参数配置。参数配置有一定的先后顺序，其基本配置流程见下图：', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_12', 'category': 'pneumatic_spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4', 'section_title': '1.4 焊接流程', 'heading_path': ['1 气动点焊单元', '1.4 焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），配置“焊接流程”时，“焊接流程”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['焊接流程：只通过焊机焊接，不通过焊机读取压力，板厚等参数'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 121, 'line_end': 121, 'excerpt': '焊接流程：只通过焊机焊接，不通过焊机读取压力，板厚等参数。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_13', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '5', 'section_title': '5 气动点焊报警信息', 'heading_path': ['5 气动点焊报警信息'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“气动点焊报警信息”中“报警码”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['报警码 报警信息 子码 子码说明 处理方法 10301 水流量异常 0-2 检查水流量情况 10302 压缩空气异常 0-1 检查压缩空气情况 10303 修磨器异常 0 检查修磨器旋转IO反馈情况 10304 气动点焊指令集异常 0 Init指令异常 联系开发人员 1 Init指令异常，焊钳号和焊点个数超界 检查指令参数 2 Init指令异常 联系开发人员 3-5 ProcessCheck指令异常 联系开发人员 10305 焊点少焊,个数为: 0 请检查生产过程中是否焊点少焊 10306 焊点多焊,个数为: 0 请检查生产过程中是否焊点多焊 10307 大小开切换指令异常 0-2 联系开发人员 10308 气动点焊指令异常 0-10 联系开发人员 11-12 修磨指令报警 检查当前行程IO信号是否正常 10309 机器人干涉 0 请检查interlock功能 10310 响应安全门请求暂停中 0 请检查安全门状态 10311 类型或参数设置错误 0 联系开发人员 10312 焊钳号错误 0 联系开发人员 10313 超时报警 0 切换行程超时或者机器人有暂停 检查行程切换IO信号 1 加压超时或者机器人有暂停 检查加压IO信号 2 加压超时或者机器人有暂停 检查加压IO信号 3 机器人有暂停 检查暂停来源 10314 焊钳压力错误 0 该压力未标定 标定该压力 1 模拟量压力标定只标定1组 重新标定压力 2 该压力不在标定范围内 重新标定压力 3 标定的模拟量文件号错误 重新设定模拟量文件号 4 计算出的物理量错误 联系开发人员 5 压力输出错误 请检查压力标定和模拟量标定 6 压力错误 联系开发人员 10315 焊接异常 0 焊接过程中机器人暂停 检查暂停原因 1 点焊控制器有报警 检查报警相关的IO 2 焊机焊接状态未准备好 检查准备完成IO 3-4 焊点编号发送错误 检查焊点编号IO配置及信号 5 焊机报警 焊机反馈报警，检查报警原因 6 焊接超时 未收到焊接完成信号，检查IO信号 7 焊机报警 焊机反馈报警，检查报警原因 10316 文件操作失败 0 联系开发人员 10317 大开，小开或加压输出IO为0', '0 请检查对应IO设置 10318 通讯状态异常 0 请检查相应的外设设备的通讯线或模块是否正常 10319 焊点程序号超界 0 请修改焊点程序号小于焊接条件最大值 10320 应用配置已更改，应用断电恢复数据已初始化 无 10321 断电数据 0 大小超界 联系开发人员 1 校验失败 当前应用数据恢复失败，反复出现请联系开发人员 10322 逻辑IO分配错误: 0 逻辑IO未分配点焊应用 请在界面IO->逻辑IO配置中配置输入输出IO 1 逻辑IO分配与点焊IO配置号不匹配 请检确认逻辑IO分配范围包含点焊IO 10341 焊点个数检查 0-7 一致,焊枪1-8 8-15 漏焊,焊枪1-8 16-23 多焊,焊枪1-8 10342 请求更换电极帽 0 请更换电极 10343 换帽预警 0 电极帽使用寿命接近设定值，请准备新电极帽 10344 上电情况下执行此操作 0 10345 通讯为屏蔽状态 提示用户,无需处理 10346 快捷键： 1 小开开始，请等待 2 大开开始，请等待 3 空打开始，请等待 4 焊接开始，请等待 5 正在开电流 6 正在关电流 7 正在开水 8 正在关水 10347 快捷键： 10 小开结束，已完成 11 大开开始，已完成 12 空打开始，已完成 13 焊接开始，已完成 14 开电流已完成 15 关电流已完成 16 开水已完成 17 关水已完成 19 小开异常 20 大开异常 21 空打异常 22 焊接异常 23 开电流异常 24 关电流异常 25 开水异常 26 关水异常'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 726, 'line_end': 726, 'excerpt': '<table><thead><tr><th><p><strong>报警码</strong></p></th><th><p><strong>报警信息</strong></p></th><th><p><strong>子码</strong></p></th><th><p><strong>子码说明</strong></p></th><th><p><strong>处理方法</strong></p></th></tr></thead><tbody><tr><td><p>10301</p></td><td><p>水流量异常</p></td><td><p>0-2</p></td><td></td><td><p>检查水流量情况</p></td></tr><tr><td><p>10302</p></td><td><p>压缩空气异常</p></td><td><p>0-1</p></td><td></td><td><p>检查压缩空气情况</p></td></tr><tr><td><p>10303</p></td><td><p>修磨器异常</p></td><td><p>0</p></td><td></td><td><p>检查修磨器旋转IO反馈情况</p></td></tr><tr><td rowspan="4"><p>10304</p></td><td rowspan="4"><p>气动点焊指令集异常</p></td><td><p>0</p></td><td><p>Init指令异常</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>1</p></td><td><p>Init指令异常，焊钳号和焊点个数超界</p></td><td><p>检查指令参数</p></td></tr><tr><td><p>2</p></td><td><p>Init指令异常</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>3-5</p></td><td><p>ProcessCheck指令异常</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>10305</p></td><td><p>焊点少焊,个数为:</p></td><td><p>0</p></td><td></td><td><p>请检查生产过程中是否焊点少焊</p></td></tr><tr><td><p>10306</p></td><td><p>焊点多焊,个数为:</p></td><td><p>0</p></td><td></td><td><p>请检查生产过程中是否焊点多焊</p></td></tr><tr><td><p>10307</p></td><td><p>大小开切换指令异常</p></td><td><p>0-2</p></td><td></td><td><p>联系开发人员</p></td></tr><tr><td rowspan="2"><p>10308</p></td><td rowspan="2"><p>气动点焊指令异常</p></td><td><p>0-10</p></td><td></td><td><p>联系开发人员</p></td></tr><tr><td><p>11-12</p></td><td><p>修磨指令报警</p></td><td><p>检查当前行程IO信号是否正常</p></td></tr><tr><td><p>10309</p></td><td><p>机器人干涉</p></td><td><p>0</p></td><td></td><td><p>请检查interlock功能</p></td></tr><tr><td><p>10310</p></td><td><p>响应安全门请求暂停中</p></td><td><p>0</p></td><td></td><td><p>请检查安全门状态</p></td></tr><tr><td><p>10311</p></td><td><p>类型或参数设置错误</p></td><td><p>0</p></td><td></td><td><p>联系开发人员</p></td></tr><tr><td><p>10312</p></td><td><p>焊钳号错误</p></td><td><p>0</p></td><td></td><td><p>联系开发人员</p></td></tr><tr><td rowspan="4"><p>10313</p></td><td rowspan="4"><p>超时报警</p></td><td><p>0</p></td><td><p>切换行程超时或者机器人有暂停</p></td><td><p>检查行程切换IO信号</p></td></tr><tr><td><p>1</p></td><td><p>加压超时或者机器人有暂停</p></td><td><p>检查加压IO信号</p></td></tr><tr><td><p>2</p></td><td><p>加压超时或者机器人有暂停</p></td><td><p>检查加压IO信号</p></td></tr><tr><td><p>3</p></td><td><p>机器人有暂停</p></td><td><p>检查暂停来源</p></td></tr><tr><td rowspan="7"><p>10314</p></td><td rowspan="7"><p>焊钳压力错误</p></td><td><p>0</p></td><td><p>该压力未标定</p></td><td><p>标定该压力</p></td></tr><tr><td><p>1</p></td><td><p>模拟量压力标定只标定1组</p></td><td><p>重新标定压力</p></td></tr><tr><td><p>2</p></td><td><p>该压力不在标定范围内</p></td><td><p>重新标定压力</p></td></tr><tr><td><p>3</p></td><td><p>标定的模拟量文件号错误</p></td><td><p>重新设定模拟量文件号</p></td></tr><tr><td><p>4</p></td><td><p>计算出的物理量错误</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>5</p></td><td><p>压力输出错误</p></td><td><p>请检查压力标定和模拟量标定</p></td></tr><tr><td><p>6</p></td><td><p>压力错误</p></td><td><p>联系开发人员</p></td></tr><tr><td rowspan="7"><p>10315</p></td><td rowspan="7"><p>焊接异常</p></td><td><p>0</p></td><td><p>焊接过程中机器人暂停</p></td><td><p>检查暂停原因</p></td></tr><tr><td><p>1</p></td><td><p>点焊控制器有报警</p></td><td><p>检查报警相关的IO</p></td></tr><tr><td><p>2</p></td><td><p>焊机焊接状态未准备好</p></td><td><p>检查准备完成IO</p></td></tr><tr><td><p>3-4</p></td><td><p>焊点编号发送错误</p></td><td><p>检查焊点编号IO配置及信号</p></td></tr><tr><td><p>5</p></td><td><p>焊机报警</p></td><td><p>焊机反馈报警，检查报警原因</p></td></tr><tr><td><p>6</p></td><td><p>焊接超时</p></td><td><p>未收到焊接完成信号，检查IO信号</p></td></tr><tr><td><p>7</p></td><td><p>焊机报警</p></td><td><p>焊机反馈报警，检查报警原因</p></td></tr><tr><td><p>10316</p></td><td><p>文件操作失败</p></td><td><p>0</p></td><td></td><td><p>联系开发人员</p></td></tr><tr><td><p>10317</p></td><td><p>大开，小开或加压输出IO为0。</p></td><td><p>0</p></td><td></td><td><p>请检查对应IO设置</p></td></tr><tr><td><p>10318</p></td><td><p>通讯状态异常</p></td><td><p>0</p></td><td></td><td><p>请检查相应的外设设备的通讯线或模块是否正常</p></td></tr><tr><td><p>10319</p></td><td><p>焊点程序号超界</p></td><td><p>0</p></td><td></td><td><p>请修改焊点程序号小于焊接条件最大值</p></td></tr><tr><td><p>10320</p></td><td><p>应用配置已更改，应用断电恢复数据已初始化</p></td><td></td><td></td><td><p>无</p></td></tr><tr><td rowspan="2"><p>10321</p></td><td rowspan="2"><p>断电数据</p></td><td><p>0</p></td><td><p>大小超界</p></td><td><p>联系开发人员</p></td></tr><tr><td><p>1</p></td><td><p>校验失败</p></td><td><p>当前应用数据恢复失败，反复出现请联系开发人员</p></td></tr><tr><td rowspan="2"><p>10322</p></td><td rowspan="2"><p>逻辑IO分配错误:</p></td><td><p>0</p></td><td><p>逻辑IO未分配点焊应用</p></td><td><p>请在界面IO-&gt;逻辑IO配置中配置输入输出IO</p></td></tr><tr><td><p>1</p></td><td><p>逻辑IO分配与点焊IO配置号不匹配</p></td><td><p>请检确认逻辑IO分配范围包含点焊IO</p></td></tr><tr><td rowspan="3"><p>10341</p></td><td rowspan="3"><p>焊点个数检查</p></td><td><p>0-7</p></td><td><p>一致,焊枪1-8</p></td><td></td></tr><tr><td><p>8-15</p></td><td><p>漏焊,焊枪1-8</p></td><td></td></tr><tr><td><p>16-23</p></td><td><p>多焊,焊枪1-8</p></td><td></td></tr><tr><td><p>10342</p></td><td><p>请求更换电极帽</p></td><td><p>0</p></td><td><p>请更换电极</p></td><td></td></tr><tr><td><p>10343</p></td><td><p>换帽预警</p></td><td><p>0</p></td><td><p>电极帽使用寿命接近设定值，请准备新电极帽</p></td><td></td></tr><tr><td><p>10344</p></td><td><p>上电情况下执行此操作</p></td><td><p>0</p></td><td></td><td></td></tr><tr><td><p>10345</p></td><td><p>通讯为屏蔽状态</p></td><td></td><td><p>提示用户,无需处理</p></td><td></td></tr><tr><td rowspan="8"><p>10346</p></td><td rowspan="8"><p>快捷键：</p></td><td><p>1</p></td><td><p>小开开始，请等待</p></td><td></td></tr><tr><td><p>2</p></td><td><p>大开开始，请等待</p></td><td></td></tr><tr><td><p>3</p></td><td><p>空打开始，请等待</p></td><td></td></tr><tr><td><p>4</p></td><td><p>焊接开始，请等待</p></td><td></td></tr><tr><td><p>5</p></td><td><p>正在开电流</p></td><td></td></tr><tr><td><p>6</p></td><td><p>正在关电流</p></td><td></td></tr><tr><td><p>7</p></td><td><p>正在开水</p></td><td></td></tr><tr><td><p>8</p></td><td><p>正在关水</p></td><td></td></tr><tr><td rowspan="16"><p>10347</p></td><td rowspan="16"><p>快捷键：</p></td><td><p>10</p></td><td><p>小开结束，已完成</p></td><td></td></tr><tr><td><p>11</p></td><td><p>大开开始，已完成</p></td><td></td></tr><tr><td><p>12</p></td><td><p>空打开始，已完成</p></td><td></td></tr><tr><td><p>13</p></td><td><p>焊接开始，已完成</p></td><td></td></tr><tr><td><p>14</p></td><td><p>开电流已完成</p></td><td></td></tr><tr><td><p>15</p></td><td><p>关电流已完成</p></td><td></td></tr><tr><td><p>16</p></td><td><p>开水已完成</p></td><td></td></tr><tr><td><p>17</p></td><td><p>关水已完成</p></td><td></td></tr><tr><td><p>19</p></td><td><p>小开异常</p></td><td></td></tr><tr><td><p>20</p></td><td><p>大开异常</p></td><td></td></tr><tr><td><p>21</p></td><td><p>空打异常</p></td><td></td></tr><tr><td><p>22</p></td><td><p>焊接异常</p></td><td></td></tr><tr><td><p>23</p></td><td><p>开电流异常</p></td><td></td></tr><tr><td><p>24</p></td><td><p>关电流异常</p></td><td></td></tr><tr><td><p>25</p></td><td><p>开水异常</p></td><td></td></tr><tr><td><p>26</p></td><td><p>关水异常</p></td><td></td></tr></tbody></table>', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_14', 'category': 'pneumatic_spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.4', 'section_title': '2.4 焊钳特性配置', 'heading_path': ['2 气动点焊机器人应用配置项', '2.4 焊钳特性配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），配置“焊钳特性配置”时，“路径”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['1. 路径：【主菜单】-【应用】-【气动点焊】-【参数设置】-【焊钳特性】'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 376, 'line_end': 376, 'excerpt': '1. 路径：【主菜单】-【应用】-【气动点焊】-【参数设置】-【焊钳特性】。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_15', 'category': 'pneumatic_spot_welding', 'kind': 'parameter', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3.1.3', 'section_title': '3.1.3 STROKE\\_FULL(大开)', 'heading_path': ['3 气动点焊指令', '3.1 气动点焊运动指令说明', '3.1.3 STROKE\\_FULL(大开)'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），配置“STROKE\\_FULL(大开)”时，“STROKE_FULL GUN=1;”对应哪些参数含义、取值或单位要求？只说明本条记录明确列出的内容。', 'anchors': [], 'expected_facts': ['STROKE_FULL GUN=1; 功能 切换焊钳到大开位置', '说明 参数项 1 焊钳特性编号', '需要跟当前焊钳号一致'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 660, 'line_end': 660, 'excerpt': '<table><tr><td colspan="3"><p>STROKE_FULL GUN=1;</p></td></tr><tr><td><p>功能</p></td><td colspan="2"><p>切换焊钳到大开位置。</p></td></tr><tr><td><p>说明</p></td><td><p>参数项 1</p></td><td><p>焊钳特性编号。需要跟当前焊钳号一致</p></td></tr></table>', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_16', 'category': 'pneumatic_spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.4', 'section_title': '1.4 焊接流程', 'heading_path': ['1 气动点焊单元', '1.4 焊接流程'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“焊接流程”中，应如何理解或操作“焊接结束以及焊接异常”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['1. 焊接结束以及焊接异常：'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 142, 'line_end': 142, 'excerpt': '1. 焊接结束以及焊接异常：', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_17', 'category': 'pneumatic_spot_welding', 'kind': 'prerequisite', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '1.5', 'section_title': '1.5 气动点焊快捷键', 'heading_path': ['1 气动点焊单元', '1.5 气动点焊快捷键'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），操作“气动点焊快捷键”时，涉及“功能说明”的前提条件或注意事项是什么？', 'anchors': [], 'expected_facts': ['按键 功能说明 OP1 按OP1，调出快捷键界面，界面如图1-6所示 注意：要使OP2~OP6有效，必须先按OP1调出图1-6的快捷键界面 OP2 移动焊钳到小开位置 OP3 移动焊钳到大开位置 OP4 空打(焊钳只加压，不通电流，不进行实际焊接) 注：空打时焊钳行程状态以及压力大小可进行更改，详见2.11章节：手动点焊配置 OP5 手动焊接 注：手动试焊时焊钳行程状态、焊接压力、焊接条件可进行更改，详见2.11章节：手动点焊配置 OP6 切换气动点焊工艺模式，以便进行试运行、或仿真测试 模式包括：工艺开(正常)、工艺关-仿真、工艺关-干运行，详见第5章：气动点焊快捷键'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 160, 'line_end': 160, 'excerpt': '<table><tr><td><p><strong>按键</strong></p></td><td><p><strong>功能说明</strong></p></td></tr><tr><td><p>OP1</p></td><td><p>按OP1，调出快捷键界面，界面如图1-6所示</p><p>注意：要使OP2~OP6有效，必须先按OP1调出图1-6的快捷键界面</p></td></tr><tr><td><p>OP2</p></td><td><p>移动焊钳到小开位置</p></td></tr><tr><td><p>OP3</p></td><td><p>移动焊钳到大开位置</p></td></tr><tr><td><p>OP4</p></td><td><p>空打(焊钳只加压，不通电流，不进行实际焊接)</p><p>注：空打时焊钳行程状态以及压力大小可进行更改，详见2.11章节：手动点焊配置</p></td></tr><tr><td><p>OP5</p></td><td><p>手动焊接</p><p>注：手动试焊时焊钳行程状态、焊接压力、焊接条件可进行更改，详见2.11章节：手动点焊配置</p></td></tr><tr><td><p>OP6</p></td><td><p>切换气动点焊工艺模式，以便进行试运行、或仿真测试</p><p>模式包括：工艺开(正常)、工艺关-仿真、工艺关-干运行，详见第5章：气动点焊快捷键</p></td></tr></table>', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_18', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '3.3', 'section_title': '3.3 ProcessCheck', 'heading_path': ['3 气动点焊指令', '3.3 ProcessCheck'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“ProcessCheck”中“防漏焊检测指令”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': ['A07_AIRSW', 'SIM'], 'expected_facts': ['防漏焊检测指令 功能 防漏焊检测指令，检测内部记录的焊点数是否与初始化指令设置的焊枪防漏焊检测数量一致', '若不一致，机器人暂停并给出报警', '格式 A07_AIRSW ProcessCheck'], 'representation': 'table_row', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 672, 'line_end': 672, 'excerpt': '<table><tr><td rowspan="4"><p>防漏焊检测指令</p></td><td><p>功能</p></td><td><p>防漏焊检测指令，检测内部记录的焊点数是否与初始化指令设置的焊枪防漏焊检测数量一致。若不一致，机器人暂停并给出报警。</p></td></tr><tr><td><p>格式</p></td><td><p>A07_AIRSW ProcessCheck；注释</p></td></tr><tr><td><p>说明</p></td><td><p>防漏焊检测，是在焊接指令内部进行焊点计数，计数判断原则：是否有焊接完成信号。若是SIM或DRY模式，则不进行计数。</p></td></tr><tr><td><p>举例</p></td><td><p>A07_AIRSW ProcessCheck；注释</p></td></tr></table>', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_19', 'category': 'pneumatic_spot_welding', 'kind': 'procedure', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.11', 'section_title': '2.11 手动点焊配置', 'heading_path': ['2 气动点焊机器人应用配置项', '2.11 手动点焊配置'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“手动点焊配置”中，应如何理解或操作“空打压力”？请解释该项的操作方法、用途或结果。', 'anchors': [], 'expected_facts': ['空打压力：用于同步快捷键OP4中的空打功能'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 539, 'line_end': 539, 'excerpt': '**空打压力：用于同步快捷键OP4中的空打功能。**', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}, {'id': 'sample_pneumatic_spot_welding_c2b8069b1b_20', 'category': 'pneumatic_spot_welding', 'kind': 'instruction', 'profile': '', 'revision': 'V5.0 / A-0', 'section': '2.13', 'section_title': '2.13 焊钳压力标定', 'heading_path': ['2 气动点焊机器人应用配置项', '2.13 焊钳压力标定'], 'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'source_version': 'rc4, rc5', 'question': '根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“焊钳压力标定”中“进行压力标定前先在焊钳特性配置中选压力的控制方式(是IO控制还是模拟量控制)→进行压力标定→设置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。', 'anchors': [], 'expected_facts': ['进行压力标定前先在焊钳特性配置中选压力的控制方式(是IO控制还是模拟量控制)→进行压力标定→设置焊接压力→执行焊接指令'], 'representation': 'prose', 'references': [{'markdown': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'line_start': 587, 'line_end': 587, 'excerpt': '进行压力标定前先在焊钳特性配置中选压力的控制方式(是IO控制还是模拟量控制)→进行压力标定→设置焊接压力→执行焊接指令。', 'sha256': '71e7bd39ab1df84a2dab93e6cd9d7dad6f746ea68a04af45ce1f26e48917acad'}]}]
register_sample_bank('pneumatic_spot_welding', PNEUMATIC_SPOT_WELDING_SAMPLE_CASES, [{'source_md': 'assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md', 'seed': 3444384742393339739, 'population': 191, 'authoring': 'local source-derived questions; curated load-analysis tasks', 'selection': 'seeded chapter/kind round-robin', 'selected_lines': [(123, 123), (170, 170), (644, 644), (547, 547), (397, 397), (652, 652), (280, 280), (682, 682), (719, 719), (706, 706), (101, 101), (121, 121), (726, 726), (376, 376), (660, 660), (142, 142), (160, 160), (672, 672), (539, 539), (587, 587)]}])


In [ ]:
# | notest
PNEUMATIC_SPOT_WELDING_SAMPLE_IDS = None
pneumatic_spot_welding_sample_cases = select_cases(PNEUMATIC_SPOT_WELDING_SAMPLE_CASES, PNEUMATIC_SPOT_WELDING_SAMPLE_IDS)
display(pd.DataFrame(pneumatic_spot_welding_sample_cases)[["id", "revision", "kind", "question"]])


In [ ]:
# | notest
await query_embeddings.prepare([case["question"] for case in pneumatic_spot_welding_sample_cases])
pneumatic_spot_welding_sample_metrics = await evaluation.run(pneumatic_spot_welding_sample_cases, concurrency=EVALUATION_CONCURRENCY)
pneumatic_spot_welding_sample_results = {case["id"]: evaluation.load(case) for case in pneumatic_spot_welding_sample_cases}
display(pneumatic_spot_welding_sample_metrics)
# Inspect a complete result: evaluation.inspect(pneumatic_spot_welding_sample_cases[0]["id"])
evaluation.show_answers(pneumatic_spot_welding_sample_cases)


case_id,category,source,revision,kind,representation,corpus_available,retrieval_complete,hits,expected_manual_hit,expected_source_hits,expected_source_context_hits,cross_category_hits,expected_source_in_context,expected_section_hit,expected_section_in_context,anchors_in_matching_hits,anchors_in_context,context_chars,omissions,answer_complete,example_status,retrieval_error,answer_error,assessment
sample_pneumatic_spot_welding_c2b8069b1b_01,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,procedure,prose,True,True,8,True,6,5,2,True,True,True,None,None,21207,2,True,unsupported,,,not_assessable
sample_pneumatic_spot_welding_c2b8069b1b_02,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,prerequisite,prose,True,True,8,True,8,6,0,True,False,False,None,None,6183,0,True,unsupported,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_03,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,instruction,prose,True,True,8,True,8,5,0,True,True,True,False,False,6774,0,True,unsupported,,,abstained
sample_pneumatic_spot_welding_c2b8069b1b_04,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,procedure,prose,True,True,8,True,8,5,0,True,False,False,None,None,5082,0,True,unsupported,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_05,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,instruction,prose,True,True,8,True,8,6,0,True,False,False,False,False,7176,0,True,unsupported,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_06,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,prerequisite,table_row,True,True,8,True,8,5,0,True,True,True,None,None,6708,0,True,included,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_07,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,prerequisite,prose,True,True,8,True,8,6,0,True,False,False,None,None,6731,0,True,unsupported,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_08,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,prerequisite,prose,True,True,8,True,8,5,0,True,True,True,None,None,6519,0,True,included,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_09,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,instruction,prose,True,True,8,True,8,6,0,True,True,True,False,False,6957,0,True,unsupported,,,not reviewed
sample_pneumatic_spot_welding_c2b8069b1b_10,pneumatic_spot_welding,assets/code-docs/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md,V5.0 / A-0,procedure,prose,True,True,8,True,8,5,0,True,True,True,None,None,6909,0,True,included,,,not reviewed


### sample_pneumatic_spot_welding_c2b8069b1b_01

根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），在“焊接流程”中，应如何理解或操作“判断当前焊机是否有报警”？请解释该项的操作方法、用途或结果。

#### Retrieved content summary

- 该行描述了新松机器人气动点焊应用中第6项警报配置，其警报内容为“反转”，对应的信号类型为“输入IO”，当前设定值为“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 211–216; corpus 5.0, revision V5.0 / A-0; object `1717d6be68e3e1e41c9c167b387af763d68350a025f9b775f6b0f343c31a34cd`.

- 文档《SX322018《新松机器人气动点焊应用操作手册》-A-0》在**2025年6月3日**完成**初版**（版本 **A/0**），由**盖梓涵**和**罗晓芳**编制。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 34–39; corpus 4.0, revision V5.0 / A-0; object `9a8f083fb73832e39e9af4467a7f87d36efcf52ac0a9590af69a2e9ab6e210ef`.

- 新松机器人气动点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 11–13; corpus 5.0, revision V5.0 / A-0; object `071d9874fbfbad83d32797d9d2c596f575aad0ebeb7a1f0ab77a0de320527c55`.

- 新松机器人气动点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 11–13; corpus 4.0, revision V5.0 / A-0; object `12acb9f24a0059f0084f3427a63ad803ccfcab1ce50b3ce254049cdc097406f5`.

- 这份表格是新松机器人气动点焊应用（V5.0）的**报警故障排查表**。它定义了从 **10301** 到 **10347** 的一系列报警码，涵盖硬件状态、指令交互、焊接工艺、压力控制、系统逻辑及快捷键状态等五大类故障。

以下是按功能模块划分的详细摘要：

### 1. 基础硬件与环境监控 (10301 - 10303)
涉及外部介质和机械部件的基础状态检查：
*   **10301**：水流量异常（子码 0-2），需检查水流量。
*   **10302**：压缩空气异常（子码 0-1），需检查压缩空气。
*   **10303**：修磨器异常（子码 0），需检查旋转IO反馈。

### 2. 指令交互与逻辑控制 (10304, 10307, 10308, 10311, 10312, 10322)
涉及控制器与上位机/外设之间的指令执行错误，多数需联系开发人员或检查IO配置：
*   **10304 (气动点焊指令集异常)**：
    *   子码 0/2：Init指令异常（联系开发）。
    *   子码 1：Init指令参数超界（检查参数）。
    *   子码 3-5：ProcessCheck指令异常（联系开发）。
*   **10307**：大小开切换指令异常（子码 0-2），联系开发人员。
*   **10308 (气动点焊指令异常)**：
    *   子码 0-10：通用指令异常（联系开发）。
    *   子码 11-12：修磨指令报警（检查行程IO）。
*   **10311**：类型或参数设置错误，联系开发人员。
*   **10312**：焊钳号错误，联系开发人员。
*   **10322 (逻辑IO分配错误)**：
    *   子码 0：IO未分配点焊应用（需在界面配置）。
    *   子码 1：IO分配范围不匹配（需确认范围包含点焊IO）。

### 3. 焊接工艺与质量监控 (10305, 10306, 10315, 10319, 10341, 10342, 10343)
核心关注焊接结果的准确性（少焊/多焊）、过程状态及电极维护：
*   **10305/10306**：焊点**少焊**或**多焊**（子码 0），需人工检查生产记录。
*   **10315 (焊接异常)**：涵盖7种状态（子码 0-7），包括机器人暂停、控制器报警、焊机未就绪、编号错误、超时等，需检查相应IO及焊机状态。
*   **10319**：焊点程序号超界，需修改程序号小于最大值。
*   **10341 (焊点个数检查)**：监控焊枪1-8状态。
    *   子码 0-7：一致（正常）。
    *   子码 8-15：漏焊。
    *   子码 16-23：多焊。
*   **10342/10343**：电极帽管理。
    *   10342：请求更换电极帽。
    *   10343：换帽预警（寿命接近设定值）。

### 4. 压力控制与系统状态 (10313, 10314, 10316-10318, 10320, 10321, 10344-10345)
涉及液压/气动压力标定、超时、通讯及系统数据完整性：
*   **10313 (超时报警)**：
    *   子码 0：切换行程超时。
    *   子码 1-2：加压超时。
    *   子码 3：机器人暂停导致的超时。
*   **10314 (焊钳压力错误)**：7个子码（0-6）分别对应未标定、标定组数错误、范围外、文件号错误、物理量计算错、输出错、通用压力错。处理多为重新标定或联系开发。
*   **10316**：文件操作失败，联系开发。
*   **10317**：大开/小开/加压输出IO为0，需检查IO设置。
*   **10318**：通讯状态异常，检查外设通讯线/模块。
*   **10320**：配置更改导致数据初始化，无需处理。
*   **10321 (断电数据)**：
    *   子码 0：大小超界。
    *   子码 1：校验失败（数据恢复失败）。
*   **10344**：上电情况下执行操作提示。
*   **10345**：通讯屏蔽状态提示，无需处理。

### 5. 状态监控与快捷键反馈 (10309, 10310, 10346, 10347)
反映系统实时运行状态、安全互锁及手动操作（快捷键）的执行结果：
*   **10309**：机器人干涉，检查 Interlock 功能。
*   **10310**：响应安全门请求暂停，检查安全门状态。
*   **10346 (快捷键 - 开始/进行中)**：8个子码（1-8），分别指示小开、大开、空打、焊接开始的等待状态，以及电流/水的开启过程。
*   **10347 (快捷键 - 完成/异常)**：16个子码（10-17, 19-26），分别指示上述动作的**完成状态**（10-17）和**异常状态**（19-26，如小开异常、焊接异常等）。*注：此类状态码主要用于HMI提示，表中未提供具体修复方法。*

### 关键结论
1.  **故障处理层级**：简单物理故障（水、气、IO信号）可自行检查；复杂逻辑错误（指令集、参数计算、压力标定算法）主要依赖开发人员支持。
2.  **安全机制**：系统包含多重安全互锁（机器人干涉、安全门暂停）和工艺保护（焊点数量校验、压力范围校验）。
3.  **维护重点**：电极帽寿命管理（10342/10343）和压力标定（10314）是预防性维护的关键点。
4.  **状态监控**：10346/10347 提供了详细的手动操作流程反馈，便于操作员实时监控各步骤执行情况。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 1050–1490; corpus 4.0, revision V5.0 / A-0; object `e3dadc38890369d017b780a44ab1235a4de416e626eb16886d4cb5cc04f9cc3e`.

- 该表格行是《新松机器人气动点焊应用操作手册》中故障诊断表的**表头行**，定义了以下五列结构：
1. **报警码**
2. **报警信息**
3. **子码**
4. **子码说明**
5. **处理方法**

所有单元格均为独立表头（无合并单元格继承关系），用于规范后续数据行中报警故障的代码、描述及处理方案的呈现方式。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 1052–1058; corpus 5.0, revision V5.0 / A-0; object `133ee46719c2861296c2a12988fb7e381fd41b9d4bee3d7f4d4e493358c73412`.

- 新松机器人弧焊应用操作手册（V5.0，文档 SX322003）报警码表（表320）：含“报警码 / 报警信息 / 解决方法”三列；示例条目包括 20081 焊机通讯断开（检查通讯配置/干扰）、20082 焊枪碰撞（解除碰撞并检查信号）、20083 焊机异常（复位故障）、20084 焊枪碰撞（同上）、20085 压缩空气异常、20086 水流量异常、20087 保护气故障、20088 缺丝（检查送丝信号）、20089 机器人焊枪碰撞且4/5/6轴松抱闸（移至安全位置）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7262–7266; corpus 5.0, revision V5.0 / C-1; object `57d96739305cdd8887a99680e0ec7bfada8f2f941f7eaf294992cd989d6c2e6e`.

- 该行是《新松机器人点焊应用操作手册》（A-2）中报警信息表的**表头行**，定义了五个关键列及其对应关系：

1.  **报警码**：用于标识主要报警代码。
2.  **报警信息**：对应报警码的文字描述。
3.  **子码**：报警的详细子分类代码。
4.  **子码说明**：对子码具体含义的解释。
5.  **处理方法**：针对该报警及子码的故障排查与解决建议。

此结构构成了点焊机器人报警故障诊断的逻辑框架：通过“报警码”和“子码”定位具体故障，查阅“说明”理解原因，并执行“处理方法”进行修复。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322007《新松机器人点焊应用操作手册》（A-2）/SX322007《新松机器人点焊应用操作手册》（A-2）.md`, lines 1664–1670; corpus 4.0, revision V5.0 / A-2; object `22941e4fac6314bf33acea0557a9da7237206ab140d42ed46e4cc9dae1305232`.

#### Answer

根据检索到的手册证据，E1 列出了气动点焊应用的输入 IO 信号表，其中包含

#### Model-assisted source review

Assessment: **not_assessable**; completeness: none; citation support: not_applicable; revision fidelity: faithful.

The generated answer is a fragment that does not address the question about the 'welding process' step '判断当前焊机是否有报警'. It merely states that E1 lists input IO signals. It fails to explain the operation, purpose, or result as requested. The provided evidence (E1, E5, E6) contains alarm codes and IO definitions but does not explicitly contain the procedural step '判断当前焊机是否有报警' within a '焊接流程' section. However, the answer is not a refusal; it is an incomplete, non-responsive fragment. Therefore, it is not a correct answer, nor is it a valid abstention (which would explicitly state inability to answer due to lack of evidence). It is best classified as not_assessable or incorrect due to being a fragment that omits the requested facts.

...

#### Answer

在所提供的检索证据（E1–E8）中，未找到任何关于

### sample_pneumatic_spot_welding_c2b8069b1b_20

根据《SX322018《新松机器人气动点焊应用操作手册》-A-0》（V5.0 / A-0），“焊钳压力标定”中“进行压力标定前先在焊钳特性配置中选压力的控制方式(是IO控制还是模拟量控制)→进行压力标定→设置”的功能或用法是什么？请说明本条记录实际给出的语法、参数或行为，不补写未载明的代码。

#### Retrieved content summary

- 文档《SX322018《新松机器人气动点焊应用操作手册》-A-0》在**2025年6月3日**完成**初版**（版本 **A/0**），由**盖梓涵**和**罗晓芳**编制。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 34–39; corpus 4.0, revision V5.0 / A-0; object `9a8f083fb73832e39e9af4467a7f87d36efcf52ac0a9590af69a2e9ab6e210ef`.

- 《新松机器人气动点焊应用操作手册》A/0版本于2025年6月3日由盖梓涵和罗晓芳完成初版。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 34–39; corpus 5.0, revision V5.0 / A-0; object `1c3a1d4a1e762b79c262394faf4d04607cfd29487b0c3fb05596d75f043ded81`.

- 新松机器人气动点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 11–13; corpus 5.0, revision V5.0 / A-0; object `071d9874fbfbad83d32797d9d2c596f575aad0ebeb7a1f0ab77a0de320527c55`.

- 新松机器人气动点焊应用操作手册
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 11–13; corpus 4.0, revision V5.0 / A-0; object `12acb9f24a0059f0084f3427a63ad803ccfcab1ce50b3ce254049cdc097406f5`.

- 新松机器人气动点焊应用操作手册（V5.0）中，序号 1 的操作模式为“焊接/试验”，对应的信号类型为“输出IO”，数值设置为“1”。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 238–243; corpus 5.0, revision V5.0 / A-0; object `1ee1e3f4ccfa7880500ee8f75c4755a37bd1120f3525fbdc0a83a5095fc42af5`.

- 新松机器人气动点焊参数行：程序 AIRSPOTJ，点 P[1]；VJ=10、ACC=100、CNT=0；WELD 枪号 1、PRS=1；程序号 PROGNo=123456789；GLF=1。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 837–839; corpus 4.0, revision V5.0 / A-0; object `7385c27aeaedb851fa5fcf29d2547a97ec638535af0e371f1bd3defdfdb9bc48`.

- 该行记录了一个气动点焊操作配置项：序号2，动作/事件为“焊接开始”，控制方式为“输出IO”，对应的IO点/值为1。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 244–249; corpus 5.0, revision V5.0 / A-0; object `dc5269b7e1fc02d510714e6e63b5007799c48e76c68da393d3bda60ecbf26e87`.

- 该行为位于《新松机器人气动点焊应用操作手册》（V5.0版本）表格的第5行（索引4），包含4个单元格。所有单元格均为空文本，无跨行/跨列合并，未继承任何值，也不包含图像或子节点。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人气动点焊应用操作手册》-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md`, lines 46–51; corpus 4.0, revision V5.0 / A-0; object `b16f7496a6ecef4bfffe739f7bf17be839effa73d2de8722c5ca5b16a86910a5`.

#### Answer

The retrieved evidence (E1–E8) from SX322018《新松机器人气动点焊应用操作手册》-A-0 (V5.0 / A-0) only contains: document version metadata (E1, E2), the document title (E3, E4), an output-IO configuration table listing signals such as 焊接/试验, 焊接开始, 焊点号确认, 报警复位, 预留, 焊点编号 (E5, E7), and the AIRSPOTJ command syntax with its parameter descriptions (E6). None of these excerpts describe the '焊钳压力标定' (welding-gun pressure calibration) procedure, the '焊钳特性配置' screen, the selection of pressure control mode (IO vs. analog), or the three-step workflow '选压力控制方式 → 进行压力标定 → 设置'. Because the specific calibration workflow, its parameters, and its behavioral description are absent from all supplied evidence, no faithful contextual example can be constructed without inventing undocumented syntax or steps.

#### Model-assisted source review

Assessment: **abstained**; completeness: complete; citation support: supported; revision fidelity: faithful.

The answer correctly identifies that the supplied evidence (E1-E8) does not contain the specific '焊钳压力标定' (pressure calibration) workflow or the '焊钳特性配置' (welding gun characteristic configuration) details requested in the question. The evidence provided consists of version metadata, document titles, IO signal lists, and the AIRSPOTJ command syntax, none of which describe the calibration procedure or the selection of pressure control modes (IO vs. analog). Therefore, the refusal to answer is appropriate and faithful to the provided data.

## Full evaluation and results

Run this cell after defining all banks. It includes the 68 relocated cases and all 760 additional cases. Running a family cell earlier is optional; completed stages are reused. A fresh independent notebook run need not execute the historical manual-suite cells.

In [ ]:
# | notest
ALL_EVALUATION_CASES = GENERAL_OPERATION_CASES + ARC_WELDING_CASES + SAMPLED_CASES
assert len(ALL_EVALUATION_CASES) == len({c["id"] for c in ALL_EVALUATION_CASES}) == 828
await query_embeddings.prepare([case["question"] for case in ALL_EVALUATION_CASES])
evaluation_metrics = await evaluation.run(ALL_EVALUATION_CASES, concurrency=EVALUATION_CONCURRENCY)
display(grouped_metrics(evaluation_metrics, ["category"]))
display(grouped_metrics(evaluation_metrics, ["source", "revision"]))
display(grouped_metrics(evaluation_metrics, ["kind"]))


category,cases,indexed_cases,retrieval_completed,expected_source_hits,expected_source_in_context,answers_completed,generation_errors
adhesive_dispensing,40,20,40,20,20,40,0
arc_welding,70,70,70,60,56,65,5
fds,40,20,40,20,20,39,1
general_operation,98,78,98,78,77,96,2
grinding,40,20,40,20,20,39,1
laser_welding,40,40,40,37,37,38,2
load_analysis,20,20,20,20,20,20,0
material_handling,40,20,40,20,20,38,2
offline_interface,80,40,80,34,34,79,1
palletizing,40,20,40,20,20,40,0


source,revision,cases,indexed_cases,retrieval_completed,expected_source_hits,expected_source_in_context,answers_completed,generation_errors
assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5).docx,B-5,21,21,21,21,21,21,0
assets/code-docs.4.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5).docx,B-5,12,12,12,12,12,11,1
assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(C-1).docx,C-1,17,17,17,17,16,16,1
assets/code-docs.5.0/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1).docx,C-1,18,18,18,18,18,15,3
assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md,V3.0 / V1.4,20,0,20,0,0,19,1
assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md,V3.0 / V1.0,20,0,20,0,0,17,3
assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322008《新松机器人免示教焊接软件操作手册》-V1.0/SX322008《新松机器人免示教焊接软件操作手册》-V1.0.md,V3.0 / V1.0,20,20,20,20,20,20,0
assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md,V3.0 / V1.0,20,0,20,0,0,19,1
assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人负载分析软件使用手册》-V1.0/SX322010《新松机器人负载分析软件使用手册》-V1.0.md,V3.0 / V1.0,20,20,20,20,20,20,0
assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md,V4.0 / V1.0,20,0,20,0,0,20,0


kind,cases,indexed_cases,retrieval_completed,expected_source_hits,expected_source_in_context,answers_completed,generation_errors
alarm,36,22,36,22,22,36,0
instruction,239,158,239,157,155,222,17
parameter,159,97,159,90,89,154,5
prerequisite,91,64,91,59,57,91,0
procedure,303,187,303,180,180,299,4


In [ ]:
# | notest
# Fresh saved results, separate from historical outputs above.
evaluation.show_answers(GENERAL_OPERATION_CASES + ARC_WELDING_CASES)


### general_40_9_1_1_single_output

请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用OUT OT#控制单个数字输出，说明IO号、ON/OFF含义，并给出先置位再复位的例子？

#### Retrieved content summary

- OUT OT# 举例：OUT OT#1 =ON，OUT OT#2 =OFF
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12655–12658; corpus 4.0, revision V5.0 / B-5; object `1a8a05fa04b3c6ecd41ef4898e04478d25278fe2caafad0e633ce0469a256433`.

- **OUT OT# 参数2（说明）**：设置为 `<IO状态>`，其中 OFF 代表断开机器人输出信号，ON 代表接通机器人输出信号。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12651–12654; corpus 4.0, revision V5.0 / B-5; object `8fc02a9902d7114bc456071da455def98c8316c968c52935fab104a14b0fa85c`.

- 该指令为 **OUT OT#**，其格式为 `OUT #<参数1>=<参数2>`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7555–7558; corpus 4.0, revision V5.0 / B-5; object `b3a954aebae14cafa282199d3c78ed562c7796768bc7702c2c89194a2ce01020`.

- 该表格详细定义了机器人 IO 输出指令集，主要涵盖**静态状态输出**、**变量赋值输出**及**路径触发输出**三大类。

**1. 静态状态与常数输出**
*   **OUT OT#**：控制单个 IO（1-4096），状态为 ON/OFF。
*   **OUT OG#**：控制 8 位组 IO（1-8），数值范围 0-255，支持常数赋值。
*   **OUT OGH#**：控制 16 位组 IO（1-256），数值范围 0-65535，支持常数赋值。
*   **OUT OGFA#**：控制 32 位组 IO（1-128），支持直接赋值整数或实数。

**2. 变量赋值输出**
此类指令将变量值写入对应位宽的组 IO：
*   **8 位组 (OG)**：支持整型变量 I (1-520)。
*   **16 位组 (OGH)**：支持整型变量 I (1-520)。
*   **32 位组 (OGFA)**：支持整型变量 I (1-520) 或实型变量 R (1-520)。

**3. 路径触发输出（动态）**
基于机器人运动轨迹，按特定条件触发单个 IO 状态：
*   **OUT_P**：**按距离**触发。参数 D 为距离阈值（-1000 至 1000 mm），正值表示向下一目标行进达到距离时触发，负值表示剩余距离时触发。
*   **OUT_T**：**按时间**触发。参数 T 为时间阈值（-30 至 30 s），正值表示向下一目标行进达到时间时触发，负值表示剩余时间时触发。

所有指令均遵循 `指令 目标#<组/点号> = <值/变量>` 的基本结构，旨在灵活控制机器人数字输出信号的时机与状态。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7549–7801; corpus 4.0, revision V5.0 / B-5; object `18b4d47778f56b34d79647b489f7ff08f5ba1fe5ed5a37f64f1279a68debc905`.

- 该行为新松机器人通用操作手册中 **OUT OG** 功能块的说明行，具体描述其 **参数2** 为 **<组IO数值>**，取值范围为 **0-255** 的常数，且组IO信号可通过界面配置（详见3.1.2节组I/O说明）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7586–7589; corpus 4.0, revision V5.0 / B-5; object `a37debc31b329605f068aa249e535a3d8e7851c50c073ed4e56c481a07eaf1e5`.

- 在“OUT OT#”的“说明”中，“参数2”定义为<IO状态>，其中OFF表示断开机器人输出信号，ON表示接通机器人输出信号。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7564–7567; corpus 4.0, revision V5.0 / B-5; object `aa80b1c233e3422560840f57de8f03ff62225d36907fd103259f50e088075ac0`.

- 该行示例了输出指令（如 `OUT_P OT#1`，所属组包括 `OG`, `OGH` 等）的用法：通过 `MOVL` 定位至 P1（速度 600，加速度 20，计数 100），执行向下一点移动 1mm 并置输出为 1，最后移动至 P2。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 12880–12883; corpus 4.0, revision V5.0 / B-5; object `6db461dbc44754400d20b9a45c0c39fe2000a07378dc87006ba4ecc3c89f9052`.

- 该表行来自《新松机器人通用操作手册》，定义了指令 **OUT OG#** 的语法格式：`OUT OG#<参数1>=I<参数2>`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322001《新松机器人通用操作手册》(B-5)/SX322001《新松机器人通用操作手册》(B-5).md`, lines 7666–7669; corpus 4.0, revision V5.0 / B-5; object `8f4cbe06efb8960bda957154cba375d529494cf068c56f72a4cc95ac4a77f6d2`.

#### Answer

依据《新松机器人通用操作手册》(B-5)第9.1.1节，OUT OT# 用于控制单个数字IO输出。格式为 OUT OT#<IO号> = <IO状态>。参数1为IO号，范围1-4096；参数2为IO状态，ON代表接通机器人输出信号（置位），OFF代表断开机器人输出信号（复位）。先置位再复位的例子即先执行 OUT OT#1 = ON 接通信号，再执行 OUT OT#1 = OFF 断开信号。

### Example — general_operation

机器人在搬运工位需要控制一个电磁阀（IO号1）：先移动到目标位置，然后接通电磁阀（置位），完成动作后再断开电磁阀（复位）。

**Environment and prerequisites:** 前提：机器人处于手动/自动模式，IO号1已在控制器中配置为数字输出。变量：IO号=1（用户指定的输出通道）。初始值：IO#1 当前状态为 OFF。

**Expected result:** 执行 OUT OT#1 = ON 后，IO#1 输出信号接通（高电平）；执行 OUT OT#1 = OFF 后，IO#1 输出信号断开（低电平），完成一次置位-复位循环。

**Manual:** SX322001《新松机器人通用操作手册》(B-5) V5.0 [E1] [E2] [E3] [E5]

```text
MOVL P1 VL=600 ACC=20 CNT=100
OUT OT#1 = ON
OUT OT#1 = OFF
```

#### Model-assisted source review

Assessment: **correct**; completeness: complete; citation support: supported; revision fidelity: faithful.

The answer correctly identifies the OUT OT# instruction for single digital IO output, specifies the format, IO number range (1-4096), and the meaning of ON/OFF (connect/disconnect signal) as per the source. It provides a valid example of setting (ON) then resetting (OFF) the same IO, which aligns with the source examples and the question's request. The added context (MOVL, environment) is reasonable for a procedural example and does not contradict the source.

### general_40_9_1_1_wait_input

请依据《新松机器人通用操作手册》(B-5)第9.1.1节“I/O类指令”回答：怎样用WAIT等待输入信号满足条件，并设置等待时间？请说明输入号、状态值和超时参数。

...

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: supported; revision fidelity: faithful.

The answer is a fragment and does not provide the requested explanation of alarm code 20088, its meaning, or the handling procedure. It only cites the source document. The supplied evidence [E1], [E2], and [E5] clearly contains the necessary information: alarm code 20088 corresponds to '缺丝!' (Wire shortage) and the solution is '送丝异常，检查焊接状态监控中送丝信号状态' (Wire feeding abnormal, check wire feeding signal status in welding state monitoring). The answer fails to extract and present these facts. It is not an abstention because the evidence was sufficient to answer the question. It is not a hallucination because the fragment it does provide is supported, but it is incomplete and therefore incorrect for the requested task.

### arc_50_alarm_20089

请依据《新松机器人弧焊应用操作手册》(C-1)第8章“报警信息→报警信息”表解释报警码20089（该表归类为报警信息）的含义和处理办法；若表中有子码，请结合本条目列出的子码及子码信息解释。请明确区分报警、警告和提示，不要混入其他版本或其他子码的处理办法；未给出的原因、子码或措施请说明缺失，无需生成机器人程序。

#### Retrieved content summary

- 新松机器人弧焊应用操作手册（V5.0，文档 SX322003）报警码表（表320）：含“报警码 / 报警信息 / 解决方法”三列；示例条目包括 20081 焊机通讯断开（检查通讯配置/干扰）、20082 焊枪碰撞（解除碰撞并检查信号）、20083 焊机异常（复位故障）、20084 焊枪碰撞（同上）、20085 压缩空气异常、20086 水流量异常、20087 保护气故障、20088 缺丝（检查送丝信号）、20089 机器人焊枪碰撞且4/5/6轴松抱闸（移至安全位置）。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7262–7266; corpus 5.0, revision V5.0 / C-1; object `57d96739305cdd8887a99680e0ec7bfada8f2f941f7eaf294992cd989d6c2e6e`.

- 报警码 20089：机器人焊枪发生碰撞，4、5、6轴松开抱闸；解决方法：请将机器人移动至安全位置。（源自《新松机器人弧焊应用操作手册》V5.0）
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7307–7311; corpus 5.0, revision V5.0 / C-1; object `4a9ff59727877e72ebb212e4846e6d76fae0f79b5177f8a76221507d7d65338b`.

- 该表格列出了新松机器人弧焊系统（V5.0）中报警码 **20081–20089** 的故障信息及处理方法，主要涵盖以下类别：

1.  **通讯与主机故障**：
    *   **20081**：焊机通讯断开 → 检查通讯配置及干扰。
    *   **20083**：焊机异常 → 复位焊机故障。

2.  **碰撞安全故障**：
    *   **20082/20084**：焊枪碰撞 → 解除碰撞并检查碰撞信号状态。
    *   **20089**：机器人焊枪碰撞且4/5/6轴松抱闸 → **立即移动机器人至安全位置**。

3.  **气体与介质供应故障**（需检查“焊接状态监控”中的相应信号状态）：
    *   **20085**：压缩空气异常（无气）。
    *   **20086**：水流量异常（流量低）。
    *   **20087**：保护气故障（无气）。

4.  **送丝故障**：
    *   **20088**：缺丝/送丝异常 → 检查送丝信号状态。

> **注**：20082 与 20084 的报警信息及解决方法完全一致；20085–20088 均属于外部介质/材料供应问题，需通过监控界面检查对应信号。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7260–7313; corpus 5.0, revision V5.0 / C-1; object `400f55d96b4949d51a44cac8146a26b36a5c4cc61033733e89b100806f6c95e3`.

- 该条目记录了新松机器人弧焊系统报警码 **20087**（保护气故障！）：表示无保护气，需检查焊接状态监控中保护气信号状态。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7297–7301; corpus 5.0, revision V5.0 / C-1; object `e30e5438dbdac9f8cc5b2faee43a905d49214455285a573861b309b79bdc8685`.

- 新松机器人弧焊系统报警码 **6889**（试焊功能）的子码 **1** 表示“试焊结束”，这是快捷键试焊的功能提示，**无需处理**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 19882–19886; corpus 4.0, revision V5.0 / B-5; object `305b5361ef128ab36ab073c93800afafe794dbfff1b0167704ca802adb987e93`.

- 根据您提供的文档数据，这是《新松机器人弧焊应用操作手册》(C-1) 中关于**报警代码、故障描述及处理方法**的表格内容。

为了便于阅读和使用，我已将 JSON 数据整理为标准的 Markdown 表格格式。主要涉及 **传感标定异常**、**断电恢复文件损坏**、**焊接参数配置错误** 等常见问题。

### 新松机器人弧焊应用报警代码表 (部分)

| 报警代码 | 故障描述 | 处理方法 |
| :--- | :--- | :--- |
| **20604** | 传感结果偏移距离大于最大限制 | 请检查传感工艺参数配置 |
| **20632** | TSDF传感开关未打开 | 请阅读报警信息 |
| **20633** | 传感运动序号异常 | 请阅读报警信息 |
| **20634** | 传感运动总数异常 | 请阅读报警信息 |
| **20635** | 断续焊总长度小于起点长度与终点长度之和 | 请阅读报警信息 |
| **20636** | 连续点焊分段数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20637** | 断续焊分段数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20638** | 传感结果数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20639** | 点激光坐标系位置标定点不足 | 请重新标定 |
| **20640** | 点激光坐标系姿态标定点不足 | 请重新标定 |
| **20641** | 点激光坐标系姿态标定点距离过近 | 请重新标定 |
| **20642** | 点激光坐标系姿态标定X轴和Z轴接近共线 | 请重新标定 |
| **20015** | 定点起弧,未找到焊接起点 | 请增加焊接起点 |
| **20016** | 定点起弧,当前作业为空 | 请创建作业 |
| **20017** | ARCON指令后无ARCOFF指令，需成对出现！ | 请与阅读报警信息 |
| **20643** | 二维传感三个点中任意两点距离不足10毫米 | 请重新示教 |
| **20644** | 二维传感三个点接近共线 | 请重新示教 |
| **20645** | 三维传感前三个点中任意两点距离不足10毫米 | 请重新示教 |
| **20646** | 三维传感前三个点接近共线 | 请联系技术人员 |
| **20647** | 三维传感第四点与第五点距离不足10毫米 | 请重新示教 |
| **20648** | 三维传感第四点与第五点所在直线与前三个点所在平面平行 | 请重新示教 |
| **20649** | 三维传感计算时非齐次方程组Ax=b无解 | 请重新示教 |
| **20960** | 摆动基准方向错误，请重新检查 | 请检查摆动基准方向配置 |

### 💡 关键排查建议：

1.  **传感标定类问题 (20639-20642, 20643-20649)**：
    *   大多数报警提示“**请重新标定**”或“**请重新示教**”。
    *   **注意几何约束**：如果提示“点不足”、“距离过近”、“共线”或“平行”，说明示教点选择的位置关系不符合算法要求。例如，二维传感的3个点不能共线且间距需大于10mm；三维传感的点构型需能唯一确定坐标系。
    *   **建议操作**：回到示教模式，严格按照手册要求重新选取特征点进行标定。

2.  **文件/数据完整性问题 (20636-20638)**：
    *   涉及“断电恢复文件损坏”。
    *   **建议操作**：这类问题通常无法通过简单重试解决，需联系新松技术支持获取新的配置文件或重新生成断电恢复策略。

3.  **工艺/程序逻辑问题 (20015-20017, 20635)**：
    *   **20015/20016**：确保在“定点起弧”模式下，作业中确实包含有效的起点和已创建作业。
    *   **20017**：检查用户程序，确保 `ARCON` (起弧) 和 `ARCOFF` (收弧) 指令成对出现，且逻辑顺序正确。
    *   **20635**：检查断续焊参数，确保总长度逻辑上大于或等于起点/终点定义的长度。

4.  **其他**：
    *   **20632**：确认在软件界面或配置文件里是否开启了 TSDF 传感功能。
    *   **20960**：检查摆动焊接（Weave Welding）的基准方向设置是否与当前机器人姿态和焊缝方向匹配。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7369–7373; corpus 5.0, revision V5.0 / C-1; object `7dad3c04b77d02d2639ca388cb8066b6d9e524b5843a5c99ecca9d3a051a7044`.

- 根据您提供的数据片段，其中包含了一条关于新松机器人弧焊应用的报警信息。以下是提取的关键内容：

*   **报警代码**: 20648
*   **报警描述**: 三维传感计算时非齐次方程组Ax=b无解
*   **处理建议**: 请重新示教

这条信息表明，在进行三维传感计算时，数学模型（非齐次方程组）没有解，通常是因为示教点的位置关系不合理（例如共线、共面或距离过近等几何问题），导致无法计算出有效的坐标系或轨迹。因此，建议重新进行示教，确保示教点满足几何要求。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7629–7633; corpus 5.0, revision V5.0 / C-1; object `20502bb57fadcaa38112218df614685e4986e5927f562ed2b0ae606014bd761b`.

- 基于您提供的 JSON 数据片段，这是《新松机器人弧焊应用操作手册》中关于**报警代码（Error Codes）**部分的表格内容。

为了便于查阅和使用，我将这些数据整理为标准的 Markdown 表格格式，分为**报警代码**、**报警描述**和**处理建议/措施**三列。

### 新松机器人弧焊应用报警代码表 (部分)

| 报警代码 | 报警描述 | 处理建议/措施 |
| :--- | :--- | :--- |
| **20603** | 传感超出最大检测次数 | 请检查传感工艺参数配置 |
| **20631** | (数据截断，推测为传感相关) | (数据截断) |
| **20632** | 传感运动序号异常 | 请阅读报警信息 |
| **20633** | 传感运动总数异常 | 请阅读报警信息 |
| **20634** | 断续焊总长度小于起点长度与终点长度之和 | 请阅读报警信息 |
| **20635** | 连续点焊分段数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20636** | 断续焊分段数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20637** | 传感结果数量异常，断电恢复文件可能损坏 | 请联系技术人员重新配置断电恢复文件 |
| **20638** | 点激光坐标系位置标定点不足 | 请重新标定 |
| **20639** | 点激光坐标系姿态标定点不足 | 请重新标定 |
| **20640** | 点激光坐标系姿态标定点距离过近 | 请重新标定 |
| **20641** | 点激光坐标系姿态标定X轴和Z轴接近共线 | 请重新标定 |
| **20015** | 定点起弧,未找到焊接起点 | 请增加焊接起点 |
| **20016** | 定点起弧,当前作业为空 | 请创建作业 |
| **20017** | ARCON指令后无ARCOFF指令，需成对出现！ | 请阅读报警信息 (注：原文为“请与阅读报警信息”，应为笔误) |
| **20642** | 二维传感三个点中任意两点距离不足10毫米 | 请重新示教 |
| **20643** | 二维传感三个点接近共线 | 请重新示教 |
| **20644** | 三维传感前三个点中任意两点距离不足10毫米 | 请重新示教 |
| **20645** | 三维传感前三个点接近共线 | 请联系技术人员 |
| **20646** | 三维传感第四点与第五点距离不足10毫米 | 请重新示教 |
| **20647** | 三维传感第四点与第五点所在直线与前三个点所在平面平行 | 请重新示教 |
| **20648** | 三维传感计算时非齐次方程组Ax=b无解 | 请重新示教 |
| **20649** | 传感距离为0,以至于传感方向无法确定 | 请重新示教 |
| **20960** | 摆动基准方向错误，请重新检查 | 请检查摆动基准方向配置 |

---

### 💡 关键故障处理分类说明

根据上述数据，可以将常见报警归纳为以下几类，便于快速排查：

1.  **标定/示教类问题 (20638-20649, 20642-20647)**
    *   **核心问题**：传感器（激光/视觉）的坐标系标定不合格，或者示教点几何关系不合理（如点太近、共线、共面）。
    *   **主要动作**：**重新标定** 或 **重新示教**。需确保标定点分散且不在同一直线/平面上。

2.  **程序/逻辑配置类问题 (20015-20017, 20634)**
    *   **核心问题**：焊接路径定义错误（如起点缺失、作业为空）、指令配对错误（ARCON/ARCOFF不匹配）、长度逻辑错误。
    *   **主要动作**：检查并修正 **焊接作业创建** 和 **程序指令**。

3.  **传感/工艺参数类问题 (20603, 20632-20633)**
    *   **核心问题**：传感检测次数超限、运动参数异常。
    *   **主要动作**：检查 **传感工艺参数**，确保检测范围、速度等设置合理。

4.  **系统/文件恢复类问题 (20635-20637)**
    *   **核心问题**：断电恢复文件损坏，导致分段数据丢失或不一致。
    *   **主要动作**：此类故障通常无法自行简单复位，需 **联系技术人员** 重建或修复断电恢复文件。

5.  **摆动/配置类问题 (20960)**
    *   **核心问题**：摆动焊接的基准方向配置错误。
    *   **主要动作**：在软件中检查并修正 **摆动基准方向**。

> **注意**：在实际操作中，遇到报警代码时，建议先参考具体报警信息中的详细参数，再对照此表进行排查。若涉及“断电恢复文件损坏”或“方程组无解”等复杂数学/系统错误，建议优先联系新松技术支持。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 7364–7368; corpus 5.0, revision V5.0 / C-1; object `68bfde4c1f30a9ebef269bf3bf5d676e0e3af786633eda5b72f9f41e426110c8`.

#### Answer

依据《新松机器人弧焊应用操作手册》(C-1) V5.0 第8章

#### Model-assisted source review

Assessment: **incorrect**; completeness: partial; citation support: supported; revision fidelity: faithful.

The answer is a fragment and does not provide the requested explanation of alarm code 20089 or its handling procedure. The supplied evidence (E1, E2, E3) clearly contains the alarm code 20089, its description ('机器人焊枪发生碰撞，4、5、6松抱闸'), and the solution ('请移动机器人到安全位置'). The answer fails to include these facts, making it incomplete and incorrect for the requested task.

## Algorithm and document review

**Fresh evaluation status:** 828/828 retrievals and 802/828 answers completed in the published result snapshot. Run identity: `f96c1b3d2c6baf3718305a8a4e28dab22dd63d6bc5af32281f1c7f2e9f48a9a5`. Result files are in `/d/devel/rag/ribosome/db/object_rag_tests/results/f96c1b3d2c6baf37`.

### Corpus availability

The mapped active index contains 38 corpus documents representing **23/38 CSV sources**. Of the new cases, **460/760** have their expected source indexed; **300/760** target absent revisions. Including the 68 legacy cases, the indexed-source denominator is **528/828**. Corpus 4.0/5.0 profiles are not the manual V3.0/V4.0/V5.0 revision labels.

| source | revision |
| --- | --- |
| assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322001《新松机器人通用操作手册》-V1.4/SX322001《新松机器人通用操作手册》-V1.4.md | V3.0 / V1.4 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322005《新松机器人视觉应用操作手册》-V1.0/SX322005《新松机器人视觉应用操作手册》-V1.0.md | V3.0 / V1.0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md | V4.0 / A-1 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V3.0版本/SX322009《新松机器人离线接口库使用手册》-V1.0/SX322009《新松机器人离线接口库使用手册》-V1.0.md | V3.0 / V1.0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322009《新松机器人离线接口库使用手册》(B-4)/SX322009《新松机器人离线接口库使用手册》(B-4).md | V4.0 / B-4 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322002《新松机器人码垛应用操作手册》-V1.0/SX322002《新松机器人码垛应用操作手册》-V1.0.md | V4.0 / V1.0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322006新松机器人打磨应用操作手册（A-1)/SX322006新松机器人打磨应用操作手册（A-1).md | V4.0 / A-1 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322007《新松机器人点焊应用操作手册》-A-0/SX322007《新松机器人点焊应用操作手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322011《新松机器人搬运应用操作手册》-A-0/SX322011《新松机器人搬运应用操作手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322012《新松机器人涂胶功能使用手册》-A-0/SX322012《新松机器人涂胶功能使用手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322013《新松机器人螺柱焊功能使用手册》-A-0/SX322013《新松机器人螺柱焊功能使用手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322014《新松机器人SPR铆接功能使用手册》-A-0/SX322014《新松机器人SPR铆接功能使用手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322015《新松机器人快换应用操作手册》-A-0/SX322015《新松机器人快换应用操作手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322016《新松机器人FDS应用使用手册》-A-0/SX322016《新松机器人FDS应用使用手册》-A-0.md | V4.0 / A-0 |
| assets/code-docs/.md/06 产品手册/软件类/手册/V4.0版本/SX322017《新松机器人跟随应用操作手册》-A-0/SX322017《新松机器人跟随应用操作手册》-A-0.md | V4.0 / A-0 |

### Fresh run metrics

| category | cases | indexed_cases | retrieval_completed | expected_source_hits | expected_source_in_context | answers_completed | generation_errors |
| --- | --- | --- | --- | --- | --- | --- | --- |
| adhesive_dispensing | 40 | 20 | 40 | 20 | 20 | 40 | 0 |
| arc_welding | 70 | 70 | 70 | 60 | 56 | 65 | 5 |
| fds | 40 | 20 | 40 | 20 | 20 | 39 | 1 |
| general_operation | 98 | 78 | 98 | 78 | 77 | 96 | 2 |
| grinding | 40 | 20 | 40 | 20 | 20 | 39 | 1 |
| laser_welding | 40 | 40 | 40 | 37 | 37 | 38 | 2 |
| load_analysis | 20 | 20 | 20 | 20 | 20 | 20 | 0 |
| material_handling | 40 | 20 | 40 | 20 | 20 | 38 | 2 |
| offline_interface | 80 | 40 | 80 | 34 | 34 | 79 | 1 |
| palletizing | 40 | 20 | 40 | 20 | 20 | 40 | 0 |
| pneumatic_spot_welding | 20 | 20 | 20 | 20 | 20 | 19 | 1 |
| spot_welding | 40 | 20 | 40 | 20 | 20 | 39 | 1 |
| spr_riveting | 40 | 20 | 40 | 20 | 20 | 40 | 0 |
| stud_welding | 40 | 20 | 40 | 20 | 20 | 35 | 5 |
| teaching_free_welding | 20 | 20 | 20 | 20 | 20 | 20 | 0 |
| tool_quick_change | 40 | 20 | 40 | 20 | 20 | 40 | 0 |
| tracking | 60 | 40 | 60 | 39 | 39 | 59 | 1 |
| vision | 60 | 20 | 60 | 20 | 20 | 56 | 4 |

Overall and indexed-source-only counts:

| denominator | cases | source_in_context | answers |
| --- | --- | --- | --- |
| all cases | 828 | 503 | 802 |
| indexed sources | 528 | 503 | 512 |

### Review of preserved historical results

The relocated outputs contain 38 saved general-operation answers; the arc-welding answer cell has no saved outputs. These observations describe **historical results only**, and are not accuracy scores for the new 828-case evaluation.

**`general_50_alarm_1507`** — The saved answer explicitly reports that the requested code is absent from its supplied evidence, while the source reference below contains it. This is a retrieval/evidence-coverage failure with an appropriately cautious answer, not evidence that the source manual lacks the code.

Source row: `1507 | 轴内部数据记录初始化:SV | 对应轴号初始化 | 处理方法：联系技术人员。`

Saved answer: 在所提供的检索证据中，C-1 手册（E1、E2、E3）仅包含报警码 1537、1357、1375 的条目；B-5 手册（E4–E7）包含 1514、1307、1757、1755 的条目。没有任何一条证据涉及报警码 1507。因此无法依据现有手册证据说明 1507 的报警信息、报警原因或解决措施，也不能确认该码在 C-1 版本中是否存在。

**`general_50_alarm_6035`** — The saved answer explicitly reports that the requested code is absent from its supplied evidence, while the source reference below contains it. This is a retrieval/evidence-coverage failure with an appropriately cautious answer, not evidence that the source manual lacks the code.

Source row: `6035 | 运动队列运动属运动类别错误 |  | `

Saved answer: 在所提供的检索证据中，C-1版本（V5.0 / C-1）的报警表仅包含报警码6003、6005、6016、6023，未出现报警码6035。B-5版本（V5.0 / B-5）中出现的报警码为8305，亦非6035。因此，现有证据无法说明报警码6035的含义、报警原因及解决措施，无法依据手册给出诊断结论。

**`general_50_alarm_1569`** — The saved answer is only a manual-identification fragment and omits the alarm meaning, cause and remedy. A nonempty explanation or accepted JSON therefore does not establish answer completeness. The actual cause of the fragment cannot be established from saved outputs alone.

Source row: `1569 | 伺服参数内部转换初始化:SV  | 对应轴号初始化 | 处理方法：联系技术人员。`

Saved answer: 依据《新松机器人通用操作手册》(C-1)（V5.0 / C-1）

**Revision fidelity:** `general_40_9_1_1_wait_input` has saved summaries from both B-5 and C-1, whose WAIT examples use different input syntax. Mixed retrieval is expected under the current algorithm; source-specific context and citation checks are needed before judging whether an answer mixed syntax. The presence of both revisions alone does not prove that the answer is wrong.

**Stored summaries:** the saved alarm summaries include a long synthesized troubleshooting guide, whereas the exact source row for 6035 has blank cause/remedy cells. Reviewers must distinguish generated summaries from exact evidence and preserve blanks as unknowns.

### Priorities supported by the evidence

1. **Pack exact alarm rows:** fresh 1507 and 6035 results contain the codes in matching retrieved objects, but lose them when large tables exceed the context budget. Select the specific cited child rows before packing parent tables; lexical/code lookup can help locate those rows. Increasing hit counts alone does not fix this failure.
2. **Answer completeness:** add a task-appropriate fact checklist to evaluation. The fresh 1569 and 20081 fragments show why JSON validity and a nonempty answer are insufficient. Procedural and alarm answers should be assessed without requiring a code example.
3. **Revision-aware evidence:** expose expected source identity separately from RC profile. Retain the mixed-revision baseline while measuring which cited blocks actually reached generation.
4. **Corpus and representation coverage:** the 15 absent CSV revisions are ingestion gaps, not ranking failures. Production selects tables and outer table rows; the load-analysis reference tasks are prose and screenshot-adjacent. The full run is needed to measure the effect of that eligibility rule. No image content has been inferred or OCR added.
5. **Document quality:** keep missing alarm cause/remedy cells explicit (6035), normalize heading paths across renumbered revisions, and provide text equivalents for screenshot-only UI fields. These changes would improve traceability; their retrieval benefit remains unmeasured.

### Assessment method and limits

Canonical source identity, heading matches and anchors are retrieval diagnostics only. Expected facts stay outside generation context. Answer assessments must compare completeness, citation support, revision fidelity and unsupported claims against exact source excerpts. No unreviewed answer is counted as correct. Historical observations above are separately identified; no correctness percentage is claimed for unexecuted cases. Questions include manual revisions and source-derived terms, and supply the category explicitly. These tests do not measure automatic category classification or the full range of unconstrained user queries. No category-boost ablation was run, so this baseline does not estimate the causal benefit of the 0.15 boost. Source grounding was reviewed; controller programs were not executed.

### Fresh model-assisted source assessments

**182/828** cases have a completed review or execution-diagnostic record. The judge uses the configured model separately from production generation, with source-text renderings of the stored exact references and expected facts that are never added to answer prompts. These are model-assisted assessments, **not independently verified accuracy measurements**.

| method | assessment | completeness | cases |
| --- | --- | --- | --- |
| execution-diagnostic | not_assessable | unknown | 8 |
| model-assisted | abstained | complete | 40 |
| model-assisted | abstained | partial | 28 |
| model-assisted | correct | complete | 24 |
| model-assisted | incorrect | partial | 39 |
| model-assisted | not_assessable | complete | 1 |
| model-assisted | not_assessable | none | 11 |
| model-assisted | not_assessable | partial | 10 |
| model-assisted | not_assessable | unknown | 2 |
| model-assisted | partial | partial | 19 |

The qualitative review plan covers **182/828** cases: all 68 legacy cases and three task-diverse cases per CSV source. All 828 answers and retrieval records remain available for inspection; unreviewed cases receive no correctness grade.

**`general_40_9_1_7_tcp_connect` — incorrect.** The answer is a fragment that fails to address the core of the question. It does not explain the parameters NAME, IP, PORT, TIMEOUT, or RET, nor does it explain how to judge the connection result. The provided evidence (E2) contains all the necessary information to answer the question completely, including the parameter definitions and the return value logic (-1 for failure, >=1 for success). The answer's only claim is a citation of the section number, which is not explicitly present in the provided evidence snippets (though it is in the question prompt), and it does not constitute a correct answer to the user's request for an explanation.

Reviewer-view excerpt (HTML formatting removed): `TCPCREATE NAME=<参数1> IP=<参数2> PORT=<参数3> TIMEOUT=<参数4> RET=<参数5>`

**`general_40_9_1_7_substring` — incorrect.** The answer correctly identifies the B-5 revision and the general STRSUB syntax (STR=LS#<param1> BEGIN=<param2> END=<param3> RET=<param4>) supported by [E2] and [E5]. However, it fails to explicitly define the parameters as requested (source string, start position, length/end, target). More critically, it provides a code example using variable names `LS_1` and `LS_2` which are not supported by the evidence. The evidence consistently uses the `LS#<number>` format (e.g., `LS#1`, `LS#2`). The specific assignment values `I2 = 2` and `I3 = 5` are also not present in the provided evidence, which only shows `BEGIN=I2` and `END=I3` as placeholders or variable references. The answer hallucinates specific variable naming conventions and values not found in the source text.

Reviewer-view excerpt (HTML formatting removed): `STRSUB STR=LS#1 BEGIN=I2 END=I3 RET=LS#2`

**`general_40_9_1_3_subjob` — partial.** The answer correctly identifies the CALL and RET syntax and the requirement that RET be the last instruction before END in a sub-job, supported by [E1], [E4], and [E8]. It distinguishes unconditional (CALL SUB1) and conditional (CALL IN#1=1 SUB1) calls. However, it omits the full set of conditional call types (IG, IGH, I, OGH, IGFA, STR) listed in the source, making it incomplete relative to the question's request to distinguish unconditional and input-IO conditional calls. The claim about IN#1 range and state values is directly supported by [E4], so it is not unsupported, but the omission of other condition types reduces completeness.

Reviewer-view excerpt (HTML formatting removed): `IN#<输入信号>=<状态值>，范围1-4096，状态值为ON或OFF`

**`general_40_9_1_6_byte_logic` — partial.** The answer correctly identifies the core distinction (AND vs XOR), the result storage location (parameter 1), and the syntax, all of which are directly supported by the supplied evidence. However, it includes specific binary logic explanations and calculated numerical examples (e.g., 12 AND 10 = 8) that are not present in the provided source text. While these are factually correct in a general computing context, they are unsupported by the specific manual excerpts provided, which only give generic examples like 'AND B1 B2' without initial values or results.

Reviewer-view excerpt (HTML formatting removed): `按位与运算，参数1与参数2按位求与，结果存在参数1中。`

**`general_50_9_1_5_conditional_branch` — abstained.** The answer correctly identifies the IF THEN/ELSE IF/ELSE/ENDIF structure and comparison operators from the supplied evidence (E1, E2). It correctly notes that AND/OR are not present in the provided evidence snippets, which is a valid abstention given the evidence gap. However, the question specifically requested the C-1 revision (E4), but the answer primarily cites B-5 (E1, E2, E3, E5) and only mentions C-1 in passing. The C-1 evidence (E4) is minimal and does not provide the detailed syntax or operators, so the answer's reliance on B-5 is a reasonable fallback but technically a revision mismatch. The abstention regarding AND/OR is appropriate because the supplied evidence does not contain those operators.

Reviewer-view excerpt (HTML formatting removed): `IF THEN <参数1>=<参数2>`

**`general_40_alarm_8438` — abstained.** The question asks for alarm code 8438 from the B-5 manual. The supplied evidence (E2, E3, E4, E6, E7) contains entries for codes 8364, 8374, 8348, 8344, and 8384, but does not contain 8438. The reference provided in the prompt contains the details for 8438, but the instructions require judging the answer against the *supplied evidence* actually provided to the generation process. Since the supplied evidence lacks the specific row for 8438, the model correctly identified that the information was missing from the retrieved context and refused to hallucinate or substitute with other codes. This is a valid abstention due to inadequate supplied evidence.

Reviewer-view excerpt (HTML formatting removed): `8364`

**`general_40_9_1_1_single_output` — correct.** The answer correctly identifies the OUT OT# instruction for single digital IO output, specifies the format, IO number range (1-4096), and the meaning of ON/OFF (connect/disconnect signal) as per the source. It provides a valid example of setting (ON) then resetting (OFF) the same IO, which aligns with the source examples and the question's request. The added context (MOVL, environment) is reasonable for a procedural example and does not contradict the source.

Reviewer-view excerpt (HTML formatting removed): `OUT OT#1 =ON
OUT OT#2 =OFF`

**`general_40_9_1_1_wait_input` — correct.** The answer correctly identifies the WAIT instruction format and parameters from the B-5 revision (E2/E3). It accurately describes parameter 1 (IN#<input number>), parameter 2 (ON/OFF state), and parameter 3 (timeout duration with -1 for infinite wait). The provided example is consistent with the syntax and logic described in the source evidence.

Reviewer-view excerpt (HTML formatting removed): `WAIT <参数1>=<参数2>  T=<参数3>`

**`general_40_9_1_2_conditional_integer` — not_assessable.** The answer correctly identifies the IF I instruction format, parameters, and provides a valid branch example consistent with the B-5 manual. However, it introduces specific scenario details (initial value 150, specific points P2/P3, speed VJ=5) that are not present in the provided evidence for the IF I instruction. While the syntax is correct, the specific values are unsupported by the cited source text for this instruction.

Reviewer-view excerpt (HTML formatting removed): `I<整型变量号>，范围1-520`

**`general_40_9_1_6_integer_assignment` — not_assessable.** The answer cites [E2], [E3], and [E4], but none of these evidence snippets contain the specific text '32位整型用户变量（范围1-520）' or the example logic 'SET I2 10'. [E2] is from revision C-1 (not B-5) and lists 'I[]：32位整型变量' but does not specify the range 1-520 in the provided snippet. [E3] and [E4] are from B-5 but discuss position variables (PR) and a different SET format (SET I<参数1> N=<参数2> I<参数3>), not the simple integer assignment 'SET I1 I2'. The specific example and variable range claims are not supported by the supplied evidence.

Reviewer-view excerpt (HTML formatting removed): `整型 SET | 功能 | 赋值运算，将参数2的值赋给参数1`

6 judge responses needed nonliteral quotations omitted. Raw judge responses and warnings remain available in the review records; an omitted quotation is never presented as an exact source quote.



### Failure-stage diagnostics

| diagnostic | cases | denominator |
| --- | --- | --- |
| Expected CSV/legacy source absent | 300 | 828 |
| Expected source not retrieved (indexed cases) | 20 | 528 |
| Expected anchors absent from matching hits | 62 | 156 |
| Anchors found in hits but lost from context | 4 | 94 |
| Context has budget omissions | 70 | 828 |
| Retrieval exceptions | 0 | 828 |
| Generation exceptions or invalid responses | 26 | 828 |

Counts can overlap. Header-only citation blocks do not count as supplied source evidence. Anchor matches are diagnostics, not answer-correctness labels.


### Source-audited fresh failures

**`general_50_alarm_1507` — context packing.** The matching manual was retrieved, and a retrieved table contains the requested alarm code. That table exceeds the context budget and is omitted; the supplied row evidence lacks the code. The answer appropriately declines to invent the missing diagnosis. This is evidence-packing loss, not a missing source manual.

Exact reference row: `1507 | 轴内部数据记录初始化:SV | 对应轴号初始化 | 处理方法：联系技术人员。`

Recorded answer: 在所提供的检索证据中，C-1 手册（E1、E2、E3）仅包含报警码 1537、1357、1375 的条目；B-5 手册（E4–E7）包含 1514、1307、1757、1755 的条目。没有任何一条证据涉及报警码 1507。因此无法依据现有手册证据说明 1507 的报警信息、报警原因或解决措施，也不能确认该码在 C-1 版本中是否存在。

**`general_50_alarm_6035` — context packing.** The matching manual was retrieved, and a retrieved table contains the requested alarm code. That table exceeds the context budget and is omitted; the supplied row evidence lacks the code. The answer appropriately declines to invent the missing diagnosis. This is evidence-packing loss, not a missing source manual.

Exact reference row: `6035 | 运动队列运动属运动类别错误 |  | `

Recorded answer: 在所提供的检索证据中，C-1版本（V5.0 / C-1）的报警表仅包含报警码6003、6005、6016、6023，未出现报警码6035。B-5版本（V5.0 / B-5）中出现的报警码为8305，亦非6035。因此，现有证据无法说明报警码6035的含义、报警原因及解决措施，无法依据手册给出诊断结论。

**`general_50_alarm_1569` — answer completeness.** The requested alarm code and its source facts reached generation, but the accepted answer is only a manual-identification fragment. Nonempty JSON explanation validation did not ensure the requested meaning and handling were answered.

Exact reference row: `1569 | 伺服参数内部转换初始化:SV  | 对应轴号初始化 | 处理方法：联系技术人员。`

Recorded answer: 依据《新松机器人通用操作手册》(C-1)（V5.0 / C-1）

**`arc_50_alarm_20081` — answer completeness.** The requested alarm code and its source facts reached generation, but the accepted answer is only a manual-identification fragment. Nonempty JSON explanation validation did not ensure the requested meaning and handling were answered.

Exact reference row: `20081 | 焊机通讯断开！ | 请检查通讯配置界面，也可能存在干扰等问题`

Recorded answer: 依据《新松机器人弧焊应用操作手册》(C-1) V5.0 第8章

The 1507/6035 examples show why increasing hit counts alone is insufficient: the relevant identifiers were already in retrieved objects. Select or extract the specific cited child rows before packing large tables. The 1569/20081 examples separately demonstrate answer-completeness failures with facts already supplied. Collect finish reasons and token usage before attributing fragments to any one model or budget cause.


### Local retrieval timing sample

One warm-query sample over 86,811 active objects took 2.575s to load the catalog snapshot and 5.313s for retrieval with a cached query vector. Reusing that snapshot took 1.815s and returned identical evidence in this comparison. These timings exclude query embedding and answer generation and are not corpus-wide latency estimates. The production method scans and decodes the active catalog on each request; a snapshot cache keyed by active generation identities is a concrete optimization to benchmark separately before changing the retriever. The evaluation runner now memoizes these read-only views and checks generation/settings identities on each use; eligibility, ordering, ranking and evidence assembly remain the production methods.


### Indexed content and retrieval eligibility

| type | objects | vectors |
| --- | --- | --- |
| document | 38 | 38 |
| figure | 2737 | 2681 |
| list | 4124 | 3038 |
| list_item | 10738 | 10689 |
| table | 1352 | 652 |
| table_cell | 50804 | 187 |
| table_row | 15505 | 15316 |
| table_row_group | 1513 | 1513 |

The production retriever admits tables and outer table rows. It does not select list, list-item, figure, row-group or cell vectors as independent candidates. In particular, **10,689 list-item vectors already exist** and provide a concrete route to testing broader procedural coverage without regenerating completed summaries.

For example, indexed `list_item` object `5bd5f200dbb9e3e81ebc58680e8d34e48a46cb04c220ceb70de49e69b2a4452e` contains this load-analysis instruction, but that object type is not an eligible candidate:

> 3. **负载数据输入：**此区域可根据用户需要输入末端、手臂负载数据。

Related tables can still be returned. Finding the correct manual therefore does not establish that the requested procedural fact reached the model; the supplied-context and source-review checks address that distinction.


## Cleanup

Run after all selected suites or interruption.

In [ ]:
# | notest
try:
    if "app" in globals(): app.close()
finally:
    for client_name in ("embedding_client", "generation_client"):
        if client_name in globals(): await globals()[client_name]._client.aclose()
